In [ ]:
#@title 一键启动
import sys, json, importlib
from pathlib import Path
PROJECT_DIR = Path("/content/minimax-h3-ref-api")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PACKAGE_FILES = {'preflight.py': '"""Conservative project resource checks; no downloads and no model imports."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport stat\nimport subprocess\n\nGIB = 1024 ** 3\nFIRST_DOWNLOAD_BYTES = 140 * GIB\nMODEL_REVISION = "0919fb684102ef115126612f3fbb84d1027e7317"\n_WEIGHT_COMPONENTS = {"text_encoder": "model", "transformer_ref": "diffusion_pytorch_model",\n                      "vae": "diffusion_pytorch_model", "audio_vae": "diffusion_pytorch_model"}\n_METADATA_LIMIT = 16 * 1024 ** 2\n\n\ndef _hub_cache_path() -> Path:\n    default_home = Path(os.environ.get("XDG_CACHE_HOME", "~/.cache")) / "huggingface"\n    home = Path(os.environ.get("HF_HOME", str(default_home))).expanduser()\n    return Path(os.environ.get("HF_HUB_CACHE", os.environ.get("HUGGINGFACE_HUB_CACHE", str(home / "hub")))).expanduser()\n\n\ndef _cache_file(path, snapshot, blobs):\n    """Accept snapshot files and completed repo/shared content-addressed blobs."""\n    if path.parent.resolve(strict=True) != path.parent:\n        raise ValueError("untrusted cache component directory redirection")\n    resolved = path.resolve(strict=True)\n    # Newer Hub caches share immutable SHA-256 blobs across repositories. Only\n    # an exact snapshot reference may reach this pool; never scan/credit it.\n    shared_pool = blobs.parent.parent / "blobs"\n    shared_bucket = shared_pool / resolved.name[:2]\n    shared_blob = (re.fullmatch(r"[0-9a-f]{64}", resolved.name) is not None\n                   and resolved.parent == shared_bucket\n                   and shared_pool.resolve(strict=True) == shared_pool\n                   and shared_bucket.resolve(strict=True) == shared_bucket)\n    if not (resolved.is_relative_to(snapshot) or\n            (resolved.parent == blobs and re.fullmatch(r"[0-9a-f]{40}|[0-9a-f]{64}", resolved.name))\n            or shared_blob):\n        raise ValueError("cache file escapes its snapshot/repository blobs")\n    info = resolved.stat()\n    if not stat.S_ISREG(info.st_mode) or info.st_size <= 0 or ".incomplete" in resolved.name:\n        raise ValueError("cache file is not a completed regular file")\n    return resolved, info\n\n\ndef _small_json(path, snapshot, blobs):\n    resolved, info = _cache_file(path, snapshot, blobs)\n    if info.st_size > _METADATA_LIMIT:\n        raise ValueError("cache metadata exceeds inspection limit")\n    return json.loads(resolved.read_bytes())\n\n\ndef _weight_metadata(path, snapshot, blobs):\n    """Check only the bounded safetensors header and file size, not weight data."""\n    resolved, before = _cache_file(path, snapshot, blobs)\n    with resolved.open("rb") as source:\n        prefix = source.read(8)\n        if len(prefix) != 8:\n            raise ValueError("incomplete safetensors header")\n        length = int.from_bytes(prefix, "little")\n        if not 2 <= length <= _METADATA_LIMIT or length + 8 > before.st_size:\n            raise ValueError("invalid safetensors header size")\n        header = json.loads(source.read(length))\n    if not isinstance(header, dict):\n        raise ValueError("invalid safetensors header")\n    ranges, names = [], set()\n    sizes = {"BOOL": 1, "U8": 1, "I8": 1, "U16": 2, "I16": 2, "F16": 2, "BF16": 2,\n             "U32": 4, "I32": 4, "F32": 4, "U64": 8, "I64": 8, "F64": 8,\n             "F8_E4M3": 1, "F8_E5M2": 1}\n    for name, entry in header.items():\n        if name == "__metadata__":\n            continue\n        start, end = entry["data_offsets"]\n        shape, element_bytes = entry["shape"], sizes[entry["dtype"]]\n        if not isinstance(shape, list) or len(shape) > 16 or any(type(n) is not int or n < 0 for n in shape):\n            raise ValueError("invalid tensor shape")\n        count = 1\n        for dimension in shape:\n            count *= dimension\n        if type(start) is not int or type(end) is not int or start < 0 or end - start != count * element_bytes:\n            raise ValueError("invalid tensor data range")\n        ranges.append((start, end))\n        names.add(name)\n    cursor = 0\n    for start, end in sorted(ranges):\n        if start != cursor:\n            raise ValueError("tensor data ranges are incomplete or overlap")\n        cursor = end\n    after = resolved.stat()\n    if not names or cursor <= 0 or 8 + length + cursor != before.st_size or (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\n        raise ValueError("weight file is truncated, changing, or has trailing data")\n    identity = (after.st_dev, after.st_ino) if after.st_ino else str(resolved)\n    return names, cursor, after.st_size, identity\n\n\ndef cached_weight_budget(cache: Path) -> dict:\n    """Credit complete weight components, not readiness or model integrity."""\n    revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\n    result = {"model_revision": revision, "reusable_bytes": 0, "verified_components": [],\n              "observation": "no_verified_weights", "scope": "disk_budget_only_not_model_readiness"}\n    if not re.fullmatch(r"[0-9a-f]{40}", revision):\n        result["observation"] = "revision_not_an_exact_commit"\n        return result\n    try:\n        root = cache.resolve(strict=True)\n        repo = root / "models--MiniMaxAI--MiniMax-H3"\n        snapshot = repo / "snapshots" / revision\n        blobs = repo / "blobs"\n        if not snapshot.is_dir():\n            return result\n        # Parent directory symlinks may otherwise redirect an exact snapshot\n        # name into a different revision/repository or an unrelated filesystem.\n        for path in (repo, snapshot.parent, snapshot):\n            if path.resolve(strict=True) != path:\n                raise ValueError("untrusted cache directory redirection")\n        credited = {}\n        for component, stem in _WEIGHT_COMPONENTS.items():\n            directory = snapshot / component\n            single = directory / (stem + ".safetensors")\n            index = directory / (stem + ".safetensors.index.json")\n            if single.exists():\n                files, weight_map, total_size = [single.name], None, None\n            elif index.exists():\n                data = _small_json(index, snapshot, blobs)\n                weight_map = data["weight_map"]\n                if not isinstance(weight_map, dict) or not weight_map:\n                    raise ValueError("invalid weight index")\n                files = sorted(set(weight_map.values()))\n                total_size = data.get("metadata", {}).get("total_size")\n                if total_size is not None and (type(total_size) is not int or total_size <= 0):\n                    raise ValueError("invalid indexed weight size")\n            else:\n                continue\n            if any(not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_-]+\\.safetensors", name) for name in files):\n                raise ValueError("untrusted shard filename")\n            if any(not (directory / name).exists() for name in files):\n                continue  # partially downloaded components receive no credit\n            component_files, data_bytes = {}, 0\n            for filename in files:\n                names, size, disk_bytes, identity = _weight_metadata(directory / filename, snapshot, blobs)\n                if weight_map is not None and names != {key for key, value in weight_map.items() if value == filename}:\n                    raise ValueError("weight index does not match shard tensors")\n                component_files[identity] = disk_bytes\n                data_bytes += size\n            if total_size is not None and data_bytes != total_size:\n                raise ValueError("indexed weight bytes are incomplete")\n            credited.update(component_files)\n            result["verified_components"].append(component)\n        result["reusable_bytes"] = min(sum(credited.values()), FIRST_DOWNLOAD_BYTES)\n        if credited:\n            result["observation"] = "completed_weight_metadata_verified"\n    except FileNotFoundError:\n        # Cache creation/deletion races and missing roots never relax the gate.\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_missing_or_changed")\n    except (OSError, ValueError, TypeError, KeyError, AttributeError, OverflowError, RuntimeError) as error:\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_unverified",\n                      error_type=type(error).__name__)\n    return result\n\n\ndef check_resources(mode: str = "bf16", residency: str = "resident") -> dict:\n    if mode != "bf16":\n        raise ValueError("This project requires native BF16 on an NVIDIA A100; reduced-precision modes are disabled.")\n    if residency not in {"resident", "staged"}:\n        raise ValueError("residency must be resident or staged")\n    if not Path("/proc/meminfo").is_file():\n        raise RuntimeError("Deploy this project on a Linux NVIDIA GPU machine.")\n    memory = {}\n    for line in Path("/proc/meminfo").read_text().splitlines():\n        name, value = line.split(":", 1)\n        memory[name] = int(value.strip().split()[0]) * 1024\n    try:\n        result = subprocess.run(\n            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],\n            text=True, capture_output=True, check=True, timeout=20,\n        )\n        first = result.stdout.strip().splitlines()[0]\n        gpu, memory_mib = first.rsplit(",", 1)\n        vram_gib = float(memory_mib.strip()) / 1024\n    except (OSError, ValueError, IndexError, subprocess.SubprocessError) as exc:\n        raise RuntimeError("No working NVIDIA GPU detected. Select a GPU runtime first.") from exc\n    # The first visible CUDA GPU is used. Report conservatively when device masks are present.\n    visible = os.environ.get("CUDA_VISIBLE_DEVICES")\n    if visible and visible not in {"0", "0,"}:\n        raise RuntimeError("Preflight expects the first NVIDIA GPU. Unset CUDA_VISIBLE_DEVICES or set it to 0.")\n    cache = _hub_cache_path()\n    disk_path = cache\n    while not disk_path.exists():\n        disk_path = disk_path.parent\n    free_disk_bytes = shutil.disk_usage(disk_path).free\n    free_disk = free_disk_bytes / GIB\n    cache_budget = cached_weight_budget(cache)\n    remaining_disk_bytes = max(0, FIRST_DOWNLOAD_BYTES - cache_budget["reusable_bytes"])\n    available_ram = memory["MemAvailable"] / GIB\n    strategy = ("staged_component_residency" if residency == "staged" else\n                "component_cpu_offload" if vram_gib >= 75 else "group_cpu_offload")\n    required_ram_gib = 70 if residency == "staged" else 140\n    report = {\n        "mode": mode, "gpu": gpu.strip(), "vram_gib": round(vram_gib, 1),\n        "available_ram_gib": round(available_ram, 1),\n        "total_ram_gib": round(memory["MemTotal"] / GIB, 1),\n        "cache_free_disk_gib": round(free_disk, 1), "cache_path": str(cache),\n        "cache_reusable_weight_gib": round(cache_budget["reusable_bytes"] / GIB, 2),\n        "cache_remaining_required_gib": round(remaining_disk_bytes / GIB, 2),\n        "cache_weight_observation": cache_budget,\n        "disk_check_skipped": os.environ.get("H3_SKIP_DISK_CHECK") == "1",\n        "a100_variant": "80GB" if vram_gib >= 75 else "40GB",\n        "strategy": strategy,\n        "residency": residency, "required_available_ram_gib": required_ram_gib,\n    }\n    errors = []\n    if "A100" not in gpu.upper():\n        errors.append(f"GPU must be an NVIDIA A100, detected {gpu.strip()}")\n    if available_ram < required_ram_gib:\n        errors.append(f"available host RAM {available_ram:.1f} GiB < {required_ram_gib} GiB ({residency})")\n    if vram_gib < 38:\n        errors.append(f"GPU memory {vram_gib:.1f} GiB < 38 GiB (A100 40GB class required)")\n    if free_disk_bytes < remaining_disk_bytes and not report["disk_check_skipped"]:\n        errors.append(\n            f"cache disk free {free_disk:.1f} GiB < remaining required {remaining_disk_bytes / GIB:.1f} GiB "\n            f"(140 GiB first-download budget minus {cache_budget[\'reusable_bytes\'] / GIB:.1f} GiB "\n            "of completed reusable weights; unverified/partial files receive no credit)"\n        )\n    print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)\n    if errors:\n        raise RuntimeError(\n            "Resource preflight failed before model download: " + "; ".join(errors)\n            + ". These are conservative project gates, not a guarantee against peak-memory OOM."\n        )\n    if residency == "staged":\n        print(\n            "Staged A100 BF16: the text encoder and transformer_ref are loaded one at a time, "\n            "with the previous large component unloaded first. Disk reloads and CPU/GPU transfers add latency. "\n            "The 70 GiB host-RAM gate is a project threshold. The original staged A100 40GB single-image baseline passed; this optimized build still needs GPU acceptance.",\n            flush=True,\n        )\n    elif strategy == "group_cpu_offload":\n        print(\n            "A100 40GB: native BF16 weights are preserved. The 61.7 GB transformer cannot fit in GPU memory, "\n            "so blocks move between CPU RAM and GPU. Throughput depends on PCIe and host memory bandwidth and may be slow.",\n            flush=True,\n        )\n    else:\n        print("A100 80GB: native BF16 with components alternated between CPU RAM and GPU.", flush=True)\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--staged", action="store_true", help="Explicit sequential BF16 residency profile: 70 GiB available host RAM")\n    args = parser.parse_args()\n    check_resources(os.environ.get("H3_MODE", "bf16"), residency="staged" if args.staged else "resident")\n', 'server.py': '"""Authenticated API adapter. Model execution and durable jobs belong to the manager."""\nfrom __future__ import annotations\n\nimport asyncio\nimport errno\nimport hashlib\nimport hmac\nimport ipaddress\nimport json\nimport logging\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport threading\nfrom contextlib import asynccontextmanager\nfrom typing import Annotated, Literal\nimport uuid\nfrom urllib.parse import urlsplit\n\nfrom fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Query, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import FileResponse, JSONResponse\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\nfrom pydantic import BaseModel, ConfigDict, Field, ValidationError\n\nfrom control_protocol import ControlError, rpc_call\nfrom adapter_config import load_adapter_settings\nfrom generation_options import generation_capabilities, validate_generation_options\nfrom runtime_options import RuntimeOptions\n\nLOG = logging.getLogger("minimax_api")\nMIB = 1024 * 1024\nMAX_FILE_BYTES = 32 * MIB\nMAX_TOTAL_BYTES = 64 * MIB\nMAX_REQUEST_BYTES = MAX_TOTAL_BYTES + MIB\nMAX_CONTROL_BYTES = 64 * 1024\nEXTENSIONS = {\n    "image": {".jpg", ".jpeg", ".png", ".webp"},\n    "video": {".mp4", ".mov", ".webm"},\n    "audio": {".wav", ".mp3", ".m4a", ".flac", ".ogg"},\n}\nJOB_ID = re.compile(r"^[a-f0-9]{32}$")\nsecurity = HTTPBearer()\n\n# Public receipts are an explicit projection, not arbitrary owner-provided JSON.\n# Keep this contract independent of the model module: the API must not load it.\nPUBLIC_STAGE_CONDITIONS = ("instance_matches", "attempt_owned", "attempt_live", "model_loaded", "worker_alive",\n    "capability_available", "admission_confirmed", "predecessor_completed", "stage_not_started",\n    "cancel_flag_clear", "cancel_event_clear", "reservation_owned", "budget_grant_held")\nPUBLIC_GATE_CONDITIONS = {\n    "manager.dispatch": ("manager_open", "job_queued", "attempt_unassigned", "model_identity_current",\n        "model_observation_fresh", "model_process_alive", "model_ready", "worker_alive", "worker_accepting", "supervisor_alive"),\n    "model.execute": ("instance_matches", "request_validated", "model_loaded", "worker_alive", "capability_available",\n        "not_draining", "not_stopping", "attempt_available", "job_unowned", "artifact_absent",\n        "generation_options_valid", "resources_reserved"),\n    **{"model.stage." + stage: PUBLIC_STAGE_CONDITIONS for stage in ("prepare", "gpu", "encode_validate")},\n}\n\n\ndef public_gate_receipt(value, expected_gate):\n    """Expose known decision facts; no free-form context, parameters or paths."""\n    if not isinstance(value, dict) or value.get("gate_id") != expected_gate or expected_gate not in PUBLIC_GATE_CONDITIONS:\n        return None\n    required = PUBLIC_GATE_CONDITIONS[expected_gate]\n    raw_conditions = value.get("conditions")\n    raw_conditions = raw_conditions if isinstance(raw_conditions, dict) else {}\n    states = {"true", "false", "unknown", "conflict"}\n    conditions = {name: raw_conditions.get(name) if type(raw_conditions.get(name)) is str\n                  and raw_conditions[name] in states else "unknown" for name in required}\n    result = {"gate_id": expected_gate, "conditions": conditions,\n              "allowed": value.get("allowed") if type(value.get("allowed")) is bool else None}\n    if result["allowed"] is True and any(state != "true" for state in conditions.values()):\n        # Do not publish a positive authorization whose required evidence is\n        # missing, malformed or contradictory; keep uncertainty visible.\n        result["allowed"] = None\n    for name in ("schema_version", "gate_version"):\n        if isinstance(value.get(name), str) and re.fullmatch(r"[0-9]{1,3}\\.[0-9]{1,3}", value[name]):\n            result[name] = value[name]\n    if isinstance(value.get("decision_id"), str) and JOB_ID.fullmatch(value["decision_id"]):\n        result["decision_id"] = value["decision_id"]\n    observed = value.get("observed_at")\n    if type(observed) in {int, float} and math.isfinite(observed) and observed >= 0:\n        result["observed_at"] = observed\n    monotonic = value.get("monotonic_ns")\n    if type(monotonic) is int and 0 <= monotonic < 2 ** 63:\n        result["monotonic_ns"] = monotonic\n    blocked = value.get("blocked")\n    if isinstance(blocked, list):\n        result["blocked"] = [name for name in required if name in blocked]\n    raw = value.get("context")\n    context = {}\n    if isinstance(raw, dict):\n        for name in ("manager_instance_id", "model_instance_id", "instance_id", "job_id", "attempt_id",\n                     "budget_instance_id", "reservation_id", "lease_id", "budget_decision_id"):\n            if isinstance(raw.get(name), str) and JOB_ID.fullmatch(raw[name]):\n                context[name] = raw[name]\n        expected_source = "global_manager.py" if expected_gate == "manager.dispatch" else "model_container.py"\n        if raw.get("source_file") == expected_source:\n            context["source_file"] = expected_source\n        if isinstance(raw.get("source_sha256"), str) and re.fullmatch(r"[a-f0-9]{64}", raw["source_sha256"]):\n            context["source_sha256"] = raw["source_sha256"]\n        if type(raw.get("pipeline")) is bool:\n            context["pipeline"] = raw["pipeline"]\n        if raw.get("budget_scope") in ("stage_budget", "legacy_single_active"):\n            context["budget_scope"] = raw["budget_scope"]\n        if raw.get("stage") in ("prepare", "gpu", "encode_validate"):\n            context["stage"] = raw["stage"]\n    result["context"] = context\n    return result\n\n\ndef public_condition_fields(record):\n    fields = {}\n    if "condition_gate" in record:\n        fields["condition_gate"] = public_gate_receipt(record["condition_gate"], "manager.dispatch")\n    if "execution_condition_gates" in record:\n        raw = record["execution_condition_gates"]\n        fields["execution_condition_gates"] = ({\n            name: public_gate_receipt(raw[name], name) for name in PUBLIC_GATE_CONDITIONS\n            if name.startswith("model.") and name in raw\n        } if isinstance(raw, dict) else None)\n    if "stage_completions" in record:\n        raw = record["stage_completions"]\n        fields["stage_completions"] = ({\n            name: raw.get(name) if type(raw.get(name)) is bool else None\n            for name in ("prepare", "gpu", "encode_validate")\n        } if isinstance(raw, dict) else None)\n    if "cancellation_requested" in record:\n        fields["cancellation_requested"] = (record["cancellation_requested"]\n            if type(record["cancellation_requested"]) is bool else None)\n    return fields\nDEFAULT_CORS_ORIGINS = (\n    "https://zjj1280637679-ship-it.github.io",\n    "http://localhost:8000",\n    "http://127.0.0.1:8000",\n)\n\n\ndef _cors_allowed_origins() -> list[str]:\n    """Explicit browser origins, independent of Bearer authorization.\n\n    H3_CORS_ORIGINS replaces the defaults; an empty value disables cross-origin\n    reads. Public origins require HTTPS, while HTTP is restricted to loopback.\n    """\n    configured = os.environ.get("H3_CORS_ORIGINS")\n    values = DEFAULT_CORS_ORIGINS if configured is None else configured.split(",")\n    origins = []\n    for value in values:\n        value = value.strip()\n        if not value:\n            continue\n        try:\n            if "*" in value or any(char.isspace() or ord(char) < 32 for char in value):\n                raise ValueError()\n            parsed = urlsplit(value)\n            hostname, port = parsed.hostname, parsed.port\n            if (parsed.scheme not in {"https", "http"} or not hostname\n                    or parsed.username is not None or parsed.password is not None\n                    or parsed.path not in {"", "/"} or parsed.query or parsed.fragment\n                    or (port is not None and not 1 <= port <= 65535)):\n                raise ValueError()\n            try:\n                address = ipaddress.ip_address(hostname)\n            except ValueError:\n                address = None\n                hostname = hostname.encode("idna").decode("ascii").lower()\n                if len(hostname) > 253 or not re.fullmatch(\n                    r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?(?:\\.[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?)*",\n                    hostname,\n                ):\n                    raise ValueError()\n            if parsed.scheme == "http" and not (hostname == "localhost" or (address and address.is_loopback)):\n                raise ValueError()\n            host = f"[{address.compressed}]" if address and address.version == 6 else hostname\n            suffix = f":{port}" if port is not None and port != (443 if parsed.scheme == "https" else 80) else ""\n            origin = f"{parsed.scheme}://{host}{suffix}"\n        except (ValueError, UnicodeError):\n            raise RuntimeError("H3_CORS_ORIGINS must contain explicit HTTPS origins or loopback HTTP origins; wildcards, credentials and URL paths are not allowed.") from None\n        if origin not in origins:\n            origins.append(origin)\n    return origins\n\n\nclass RequestGuard:\n    """Authenticate before multipart parsing and bound actual incoming bytes."""\n\n    def __init__(self, app, api_key: str):\n        self.app, self.api_key = app, api_key\n\n    async def __call__(self, scope, receive, send):\n        if scope["type"] != "http" or not scope["path"].startswith("/v1/"):\n            return await self.app(scope, receive, send)\n        headers = dict(scope.get("headers", []))\n        authorization = headers.get(b"authorization", b"").decode("latin1")\n        scheme, _, token = authorization.partition(" ")\n        if scheme.lower() != "bearer" or not hmac.compare_digest(token.encode("latin1"), self.api_key.encode("ascii")):\n            return await JSONResponse({"detail": "Bearer API key required"}, 401,\n                                      headers={"WWW-Authenticate": "Bearer"})(scope, receive, send)\n        limit = MAX_REQUEST_BYTES if scope["path"] == "/v1/jobs" and scope["method"] == "POST" else MAX_CONTROL_BYTES\n        try:\n            content_length = int(headers.get(b"content-length", b"0"))\n            if content_length < 0:\n                raise ValueError\n        except ValueError:\n            return await JSONResponse({"detail": "Invalid Content-Length"}, 400)(scope, receive, send)\n        if content_length > limit:\n            return await JSONResponse({"detail": "Request exceeds body limit"}, 413)(scope, receive, send)\n        received = 0\n\n        async def limited_receive():\n            nonlocal received\n            message = await receive()\n            received += len(message.get("body", b""))\n            if received > limit:\n                raise HTTPException(413, "Request exceeds body limit")\n            return message\n\n        await self.app(scope, limited_receive, send)\n\n\nclass ControlRequest(BaseModel):\n    model_config = ConfigDict(extra="forbid")\n    action: Literal["drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"]\n    force: bool = Field(False, strict=True)\n    command_id: str | None = Field(None, min_length=1, max_length=128, pattern=r"^[A-Za-z0-9._:-]+$")\n\n\nclass DriveArchiveReceipt(BaseModel):\n    model_config = ConfigDict(extra="forbid")\n    file_id: str = Field(min_length=1, max_length=128, pattern=r"^[A-Za-z0-9_-]+$")\n    sha256: str = Field(pattern=r"^[a-f0-9]{64}$")\n    bytes: int = Field(gt=0, strict=True)\n\n\ndef _watch_parent(parent_pid, stop, exit_process=os._exit, process_factory=None, expected_creation=None):\n    # Windows venv redirectors are legitimate intermediate OS parents. Ownership\n    # is the supplied manager process generation, not direct getppid equality.\n    import psutil\n    factory = process_factory or psutil.Process\n    try:\n        if expected_creation is None:\n            expected_creation = factory(parent_pid).create_time()\n    except psutil.Error:\n        exit_process(70)\n        return\n    while not stop.wait(1):\n        try:\n            process = factory(parent_pid)\n            alive = (process.is_running() and process.status() != psutil.STATUS_ZOMBIE\n                     and process.create_time() == expected_creation)\n        except psutil.Error:\n            alive = False\n        if not alive:\n            exit_process(70)\n            return\n\n\ndef create_app(controller=None, api_key: str | None = None, output_dir: Path | None = None):\n    """API creation never initializes CUDA, a model, or a second job-state owner."""\n    token = api_key if api_key is not None else os.environ.get("H3_API_KEY", "")\n    if len(token) < 32 or not token.isascii() or any(c.isspace() for c in token):\n        raise RuntimeError("Set H3_API_KEY to a random ASCII token of at least 32 characters.")\n    instance_id = os.environ.get("H3_INSTANCE_ID") or uuid.uuid4().hex\n    root = Path(output_dir or os.environ.get("H3_OUTPUT_DIR", "./h3_outputs")).resolve()\n    private_key = os.environ.get("H3_CONTROL_KEY", "")\n    if controller is None:\n        manager_url = os.environ.get("H3_MANAGER_URL", "")\n        if not manager_url or len(private_key) < 32:\n            raise RuntimeError("The API requires its manager connection and private control token.")\n        controller = lambda operation, payload: rpc_call(manager_url, private_key, operation, payload)\n    # Only concurrent transport writes are bounded by count; durable jobs/outbox\n    # records have no count limit. Disk bytes govern their storage admission.\n    upload_lock = threading.Lock()\n    upload_state = {"active": 0}\n    disk_reserve_bytes = max(0, int(os.environ.get("H3_DISK_RESERVE_BYTES", 64 * MIB)))\n    stopped = threading.Event()\n\n    def unknown_count():\n        if not root.exists():\n            return 0\n        return sum(1 for directory in root.iterdir()\n                   if JOB_ID.fullmatch(directory.name) and not directory.is_symlink()\n                   and (directory / "submission-uncertain.json").is_file())\n\n    def acquire_upload():\n        with upload_lock:\n            if upload_state["active"] >= 3:\n                return False\n            if shutil.disk_usage(root).free < disk_reserve_bytes + (upload_state["active"] + 1) * MAX_TOTAL_BYTES:\n                raise HTTPException(507, "Insufficient disk space for reference upload")\n            upload_state["active"] += 1\n            return True\n\n    def release_upload():\n        with upload_lock:\n            upload_state["active"] -= 1\n\n    def transport_status():\n        with upload_lock:\n            return {"active_uploads": upload_state["active"], "unknown_submissions": unknown_count(),\n                    "concurrent_upload_limit": 3, "durable_submission_count_limit": None,\n                    "disk_reserve_bytes": disk_reserve_bytes, "max_reference_bytes_per_request": MAX_TOTAL_BYTES}\n\n    def redact(value):\n        if isinstance(value, dict):\n            forbidden = {"api_key", "control_key", "private_key", "internal_key", "token", "authorization", "manager_url"}\n            return {key: redact(item) for key, item in value.items() if key.lower() not in forbidden}\n        if isinstance(value, list):\n            return [redact(item) for item in value]\n        if isinstance(value, str):\n            for secret in (token, private_key):\n                if secret:\n                    value = value.replace(secret, "[redacted]")\n        return value\n\n    def call(operation, payload=None):\n        try:\n            request = dict(payload or {})\n            if operation in {"submit", "cancel", "control", "archive"}:\n                request["expected_instance_id"] = instance_id\n            result = controller(operation, request)\n            if not isinstance(result, dict):\n                raise TypeError("Invalid manager response")\n            return result\n        except ControlError as error:\n            status = getattr(error, "status", 503)\n            status = status if isinstance(status, int) and 400 <= status <= 599 else 503\n            # Transport/internal failures are not evidence of rejection; callers\n            # reconcile uncertain submissions before removing their inputs.\n            if status >= 500 and status != 507:\n                raise HTTPException(503, "Manager unavailable; query the command or job before retrying") from None\n            detail = str(redact(str(error)))[:500]\n            raise HTTPException(status, detail or "Manager rejected the request") from None\n        except Exception as error:\n            LOG.warning("Manager operation failed: %s (%s)", operation, type(error).__name__)\n            raise HTTPException(503, "Manager unavailable; query the command or job before retrying") from None\n\n    def valid_id(job_id):\n        if not JOB_ID.fullmatch(job_id):\n            raise HTTPException(404, "Task does not exist")\n\n    def public_job(record):\n        identity = record.get("id", record.get("job_id"))\n        if not isinstance(identity, str) or not JOB_ID.fullmatch(identity):\n            raise HTTPException(503, "Manager returned an invalid task identity")\n        record_instance = record.get("instance_id")\n        if "instance_id" in record:\n            if record_instance != instance_id:\n                raise HTTPException(503, "Manager returned a task from a different instance")\n        else:\n            # Compatibility with an already running older manager: bind only\n            # after an authenticated current-manager identity readback. Do not\n            # replace an explicit mismatch or infer identity from a job ID.\n            state = call("status")\n            if state.get("instance_id") != instance_id:\n                raise HTTPException(503, "Manager identity is not confirmed for this task")\n        allowed = {"status", "created_at", "started_at", "finished_at", "error", "actual_seconds",\n                   "generation_parameters", "generation_options", "lifecycle", "metrics", "active_phase", "attempt_id", "instance_id",\n                   "cancel_requested", "cleanup_complete", "replayed", "queue_position", "dispatch_state", "dispatch_reason", "waiting_for"}\n        result = {"id": identity, **{key: record[key] for key in allowed if key in record}}\n        result.update(public_condition_fields(record))\n        result["instance_id"] = instance_id\n        if result.get("status") not in {"queued", "running", "cancel_requested", "succeeded", "failed", "cancelled"}:\n            raise HTTPException(503, "Manager returned an invalid task state")\n        if isinstance(record.get("artifact"), dict):\n            result["artifact"] = {key: value for key, value in record["artifact"].items() if key != "path"}\n        result["status_url"] = f"/v1/jobs/{identity}"\n        if result["status"] in {"queued", "running", "cancel_requested"}:\n            result["cancel_url"] = f"/v1/jobs/{identity}/cancel"\n            result["poll_after_seconds"] = 2\n        if result["status"] == "succeeded":\n            result["file_url"] = f"/v1/jobs/{identity}/file"\n            archive = record.get("archive")\n            if isinstance(archive, dict) and archive.get("provider") == "google_drive":\n                try:\n                    receipt = DriveArchiveReceipt.model_validate({k: archive.get(k) for k in DriveArchiveReceipt.model_fields})\n                except ValidationError:\n                    raise HTTPException(503, "Manager returned an invalid archive receipt") from None\n                download_url = f"https://drive.google.com/uc?export=download&id={receipt.file_id}"\n                result["archive"] = {\n                    "provider": "google_drive", **receipt.model_dump(), "download_url": download_url,\n                    "view_url": f"https://drive.google.com/file/d/{receipt.file_id}/view",\n                }\n                result["download_url"] = download_url\n        return redact(result)\n\n    @asynccontextmanager\n    async def lifespan(app):\n        root.mkdir(parents=True, exist_ok=True)\n        watcher = None\n        parent_pid = os.environ.get("H3_PARENT_PID")\n        if parent_pid:\n            parent_pid = int(parent_pid)\n            if parent_pid <= 1:\n                raise RuntimeError("Invalid manager parent process identity")\n            created = os.environ.get("H3_PARENT_CREATED_AT")\n            expected_creation = float(created) if created is not None else None\n            watcher = threading.Thread(target=_watch_parent, args=(parent_pid, stopped),\n                                       kwargs={"expected_creation": expected_creation},\n                                       name="h3-api-owner-watch", daemon=True)\n            watcher.start()\n        try:\n            yield\n        finally:\n            stopped.set()\n            if watcher:\n                watcher.join(timeout=2)\n\n    adapter_settings = load_adapter_settings()\n    runtime_options = RuntimeOptions.from_env()\n    app = FastAPI(title="MiniMax-H3 Ref2VA API", version="0.4.0", lifespan=lifespan,\n                  description="Independent authenticated API adapter; durable scheduling is owned by the global manager.")\n    app.add_middleware(RequestGuard, api_key=token)\n    # Starlette wraps the last-added middleware outermost. Handle valid browser\n    # preflights before RequestGuard, and attach CORS headers even to its 401s.\n    # Every actual /v1 request still traverses the unchanged Bearer guard.\n    app.add_middleware(\n        CORSMiddleware,\n        allow_origins=_cors_allowed_origins(),\n        allow_credentials=False,\n        allow_methods=["GET", "POST"],\n        allow_headers=["Authorization", "Content-Type", "Idempotency-Key", "Range"],\n        expose_headers=["Content-Disposition", "Content-Length", "Content-Range", "Accept-Ranges", "ETag", "Retry-After"],\n        max_age=600,\n    )\n\n    def require_auth(credentials: Annotated[HTTPAuthorizationCredentials, Depends(security)]):\n        if not hmac.compare_digest(credentials.credentials.encode("utf-8"), token.encode("ascii")):\n            raise HTTPException(401, "Invalid API key", headers={"WWW-Authenticate": "Bearer"})\n\n    @app.get("/health/live")\n    def live():\n        return {"api_ready": True, "alive": True, "instance_id": instance_id}\n\n    @app.get("/health")\n    def health():\n        try:\n            state = call("status")\n        except HTTPException:\n            return JSONResponse({"api_ready": True, "manager_ready": False, "model_ready": False,\n                                 "ready": False, "instance_id": instance_id}, 503)\n        same_instance = state.get("instance_id") == instance_id\n        return JSONResponse({"api_ready": True, "manager_ready": same_instance,\n                             "model_ready": same_instance and state.get("model_ready") is True,\n                             "ready": same_instance and state.get("ready") is True,\n                             "instance_id": instance_id}, 200 if same_instance else 503)\n\n    @app.get("/v1/system", dependencies=[Depends(require_auth)])\n    def system():\n        local = transport_status()\n        try:\n            state = redact(call("status"))\n        except HTTPException:\n            return JSONResponse({"instance_id": instance_id, "api_ready": True, "manager_ready": False,\n                                 "ready": False, "model_ready": False, "api_local": local}, 503)\n        return {**state, "api_local": local}\n\n    @app.get("/v1/capabilities", dependencies=[Depends(require_auth)])\n    def capabilities():\n        # Configuration admits queued work while model_ready is still false.\n        # Never infer successful loading from a download percentage.\n        state = call("status")\n        same_instance = state.get("instance_id") == instance_id\n        return {**generation_capabilities(model_ready=same_instance and state.get("model_ready") is True,\n                    settings=adapter_settings, runtime_options=runtime_options), "instance_id": instance_id}\n\n    @app.get("/v1/events", dependencies=[Depends(require_auth)])\n    def events(after: Annotated[int, Query(ge=0)] = 0, limit: Annotated[int, Query(ge=1, le=1000)] = 100):\n        return redact(call("events", {"after": after, "limit": limit}))\n\n    @app.get("/v1/commands/{command_id}", dependencies=[Depends(require_auth)])\n    def command(command_id: str):\n        if not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", command_id):\n            raise HTTPException(404, "Command does not exist")\n        return redact(call("command", {"command_id": command_id}))\n\n    @app.post("/v1/control", status_code=202, dependencies=[Depends(require_auth)])\n    def control(request: ControlRequest):\n        if request.action in {"stop_api", "shutdown"} and request.command_id is None:\n            raise HTTPException(422, "A client command_id is required when stopping this API; query it through the manager")\n        return redact(call("control", request.model_dump(exclude_none=True)))\n\n    @app.post("/v1/jobs", status_code=202, dependencies=[Depends(require_auth)])\n    async def submit(\n        files: Annotated[list[UploadFile], File(description="Reference files in semantic order")],\n        kinds: Annotated[str, Form(description=\'JSON array matching files, e.g. ["image","audio"]\')],\n        prompt: Annotated[str, Form(min_length=1, max_length=4000)],\n        seconds: Annotated[float, Form(ge=5, le=15)] = 5,\n        width: Annotated[int, Form(ge=256, le=1344)] = 960,\n        height: Annotated[int, Form(ge=256, le=1344)] = 544,\n        steps: Annotated[int, Form(ge=2, le=50)] = 50,\n        attention_backend: Annotated[str, Form()] = "auto",\n        acceleration_lora: Annotated[str, Form()] = "none",\n        seed: Annotated[int, Form(ge=0, le=2147483647)] = 42,\n        idempotency_key: Annotated[str | None, Header(alias="Idempotency-Key", max_length=128)] = None,\n    ):\n        directory = None\n        reserved = retain_inputs = False\n        try:\n            if not prompt.strip() or not math.isfinite(seconds):\n                raise HTTPException(422, "Prompt and duration must be valid")\n            try:\n                validate_generation_options({"attention_backend": attention_backend,\n                    "acceleration_lora": acceleration_lora, "steps": steps}, adapter_settings)\n            except ValueError as error:\n                raise HTTPException(422, str(error)) from None\n            if idempotency_key is not None and not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", idempotency_key):\n                raise HTTPException(422, "Invalid Idempotency-Key")\n            if width % 32 or height % 32 or width * height > 1344 * 768:\n                raise HTTPException(422, "Dimensions must be multiples of 32, at most 1,032,192 pixels")\n            frames = 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17)\n            if frames > 360:\n                raise HTTPException(422, "Requested duration rounds beyond 15 seconds; use at most 14.375 seconds")\n            try:\n                modalities = json.loads(kinds)\n            except (ValueError, TypeError):\n                raise HTTPException(422, "kinds must be a JSON array") from None\n            if not isinstance(modalities, list) or not 1 <= len(files) <= 12 or len(modalities) != len(files):\n                raise HTTPException(422, "Supply 1-12 files and one kind per file")\n            if any(not isinstance(kind, str) or kind not in EXTENSIONS for kind in modalities):\n                raise HTTPException(422, "Kinds must be image, video or audio")\n            if modalities.count("image") > 9 or modalities.count("video") > 3 or modalities.count("audio") > 3:\n                raise HTTPException(422, "At most 9 images, 3 videos and 3 audio references")\n            if all(kind == "audio" for kind in modalities):\n                raise HTTPException(422, "At least one image or video reference is required")\n            for upload, kind in zip(files, modalities):\n                if Path(upload.filename or "").suffix.lower() not in EXTENSIONS[kind]:\n                    raise HTTPException(422, f"Unsupported {kind} file extension")\n            reserved = await asyncio.to_thread(acquire_upload)\n            if not reserved:\n                raise HTTPException(429, "Concurrent upload limit reached; retry after uploads finish", headers={"Retry-After": "5"})\n            job_id = uuid.uuid4().hex\n            directory = root / job_id\n            input_dir = directory / "inputs"\n            input_dir.mkdir(parents=True, exist_ok=False)\n            references, content_identity, total = [], [], 0\n            for index, (upload, kind) in enumerate(zip(files, modalities)):\n                suffix = Path(upload.filename or "").suffix.lower()\n                path = input_dir / f"{index:02d}{suffix}"\n                size, digest = 0, hashlib.sha256()\n                with path.open("xb") as destination:\n                    while chunk := await upload.read(MIB):\n                        size += len(chunk)\n                        total += len(chunk)\n                        if size > MAX_FILE_BYTES or total > MAX_TOTAL_BYTES:\n                            raise HTTPException(413, "Each reference <=32 MiB; total references <=64 MiB")\n                        digest.update(chunk)\n                        await asyncio.to_thread(destination.write, chunk)\n                if not size:\n                    raise HTTPException(422, "Reference file is empty")\n                references.append({"kind": kind, "path": str(path)})\n                content_identity.append({"kind": kind, "sha256": digest.hexdigest(), "bytes": size})\n            parameters = dict(prompt=prompt, references=references, seconds=seconds,\n                              width=width, height=height, steps=steps, seed=seed,\n                              attention_backend=attention_backend, acceleration_lora=acceleration_lora)\n            semantic_request = {**parameters, "references": content_identity}\n            fingerprint = hashlib.sha256(json.dumps(semantic_request, sort_keys=True,\n                                                    separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n            payload = {"job_id": job_id, "parameters": parameters, "fingerprint": fingerprint,\n                       "idempotency_key": idempotency_key or job_id}\n            def uncertain_submission():\n                marker = {"job_id": job_id, "idempotency_key": payload["idempotency_key"],\n                          "fingerprint": fingerprint, "state": "submission_outcome_unknown"}\n                (directory / "submission-uncertain.json").write_text(json.dumps(marker), encoding="utf-8")\n                return HTTPException(503, {"message": "Submission outcome unknown; query this job before retrying",\n                                          "job_id": job_id, "status_url": f"/v1/jobs/{job_id}",\n                                          "idempotency_key": payload["idempotency_key"]})\n\n            retain_inputs = True  # A dispatched command may commit before its response arrives.\n            try:\n                record = await asyncio.to_thread(call, "submit", payload)\n            except HTTPException as error:\n                if error.status_code < 500 or error.status_code == 507:\n                    retain_inputs = False\n                    raise\n                # A dropped response does not prove the durable command failed.\n                # Query the identity just sent. Preserve inputs if still unknown.\n                retain_inputs = True\n                try:\n                    record = await asyncio.to_thread(call, "job", {"job_id": job_id})\n                except HTTPException:\n                    raise uncertain_submission() from None\n            try:\n                result = public_job(record)\n            except HTTPException:\n                raise uncertain_submission() from None\n            retain_inputs = result["id"] == job_id\n            result.setdefault("actual_seconds", frames / 24)\n            return result\n        except OSError as error:\n            if error.errno in {errno.ENOSPC, errno.EDQUOT} and not retain_inputs:\n                raise HTTPException(507, "Insufficient disk space for reference upload") from None\n            raise\n        finally:\n            for upload in files:\n                await upload.close()\n            if directory is not None and not retain_inputs:\n                await asyncio.to_thread(shutil.rmtree, directory, ignore_errors=True)\n            if reserved:\n                release_upload()\n\n    def get_record(job_id):\n        valid_id(job_id)\n        record = call("job", {"job_id": job_id})\n        if record.get("id", record.get("job_id")) != job_id:\n            raise HTTPException(503, "Manager returned a different task identity")\n        # A durable manager record resolves this upload\'s uncertain delivery;\n        # only the transport marker is removed, never manager-owned inputs.\n        directory = root / job_id\n        if not directory.is_symlink():\n            try:\n                (directory / "submission-uncertain.json").unlink(missing_ok=True)\n            except OSError:\n                LOG.warning("Could not remove resolved upload marker")\n        return record\n\n    @app.get("/v1/jobs/{job_id}", dependencies=[Depends(require_auth)])\n    def status(job_id: str):\n        return public_job(get_record(job_id))\n\n    @app.post("/v1/jobs/{job_id}/cancel", status_code=202, dependencies=[Depends(require_auth)])\n    def cancel(job_id: str):\n        valid_id(job_id)\n        record = call("cancel", {"job_id": job_id})\n        if record.get("id", record.get("job_id")) != job_id:\n            raise HTTPException(503, "Manager returned a different task identity")\n        return public_job(record)\n\n    @app.post("/v1/jobs/{job_id}/drive-archive", dependencies=[Depends(require_auth)])\n    def register_drive_archive(job_id: str, receipt: DriveArchiveReceipt):\n        """Record the authenticated owner\'s receipt after external upload and readback."""\n        valid_id(job_id)\n        record = call("archive", {"job_id": job_id, **receipt.model_dump()})\n        if record.get("id", record.get("job_id")) != job_id:\n            raise HTTPException(503, "Manager returned a different task identity")\n        return public_job(record)\n\n    @app.get("/v1/jobs/{job_id}/file", dependencies=[Depends(require_auth)])\n    def download(job_id: str):\n        record = get_record(job_id)\n        if record.get("status") != "succeeded":\n            raise HTTPException(409, "Video is not ready")\n        artifact = record.get("artifact")\n        if not isinstance(artifact, dict) or not isinstance(artifact.get("path"), str):\n            raise HTTPException(409, "No verified artifact is published")\n        expected = root / job_id / "result.mp4"\n        candidate = Path(artifact["path"])\n        # The manager chooses output identity; even a malformed/stale response\n        # must not let this public adapter serve arbitrary local files.\n        if candidate != expected or candidate.is_symlink() or expected.parent.is_symlink():\n            raise HTTPException(409, "Artifact ownership does not match the task")\n        try:\n            if candidate.resolve() != expected or not candidate.is_file():\n                raise HTTPException(404, "Result was removed")\n            size = candidate.stat().st_size\n            if not size or artifact.get("bytes") != size:\n                raise HTTPException(409, "Artifact size no longer matches its committed manifest")\n            digest = hashlib.sha256()\n            with candidate.open("rb") as source:\n                for chunk in iter(lambda: source.read(MIB), b""):\n                    digest.update(chunk)\n            committed_hash = artifact.get("sha256")\n            if (not isinstance(committed_hash, str) or not re.fullmatch(r"[a-f0-9]{64}", committed_hash)\n                    or not hmac.compare_digest(digest.hexdigest(), committed_hash)):\n                raise HTTPException(409, "Artifact content no longer matches its committed manifest")\n        except OSError:\n            raise HTTPException(404, "Result was removed or cannot be read") from None\n        return FileResponse(candidate, media_type="video/mp4", filename=f"{job_id}.mp4",\n                            headers={"ETag": \'"\' + digest.hexdigest() + \'"\',\n                                     "Cache-Control": "private, no-store"})\n\n    return app\n', 'model_backend.py': '"""Real MiniMax-H3 ref2va inference. No synthetic output or mock fallback."""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nfrom runtime_options import configure_qwen_offload, measured_load\nfrom pipeline_backend import PipelineBackendSupport\nfrom adapter_config import load_adapter_settings, configure_pipeline, apply_adapters, validate_steps\nfrom download_progress import tracked_model_load\n\nMODEL_ID = "MiniMaxAI/MiniMax-H3"\nMODEL_REVISION = "0919fb684102ef115126612f3fbb84d1027e7317"\n\n\ndef _require_loaded_components(pipe) -> None:\n    """Diffusers may log component-load failures without propagating them."""\n    required = [spec.name for spec in pipe.blocks.expected_components]\n    if not required:\n        raise RuntimeError("Ref2VA workflow declared no required components.")\n    missing = [name for name in required if getattr(pipe, name, None) is None]\n    if missing:\n        raise RuntimeError(\n            "MiniMax-H3 component loading was incomplete: " + ", ".join(missing)\n            + ". Inspect the component loading errors above; API readiness is blocked."\n        )\n\n\ndef _parameter_kept_in_fp32(parameter_name, keep_fp32) -> bool:\n    """Mirror Diffusers 0.43 load_model_dict_into_meta\'s exact path segments.\n\n    Its dtype/device-map helpers use looser substring checks, but they do not\n    define the dtype assigned by the actual weight loader. In particular,\n    ``quant_conv`` must not match ``post_quant_conv``.\n    """\n    if isinstance(keep_fp32, str):\n        keep_fp32 = [keep_fp32]\n    return any(part in parameter_name.split(".") for part in (keep_fp32 or []))\n\n\ndef _audit_parameter_precision(pipe) -> dict:\n    """Inspect loaded parameter metadata, without copying weights or buffers.\n\n    Each official component owns its FP32 exceptions. In particular, MiniMax\'s\n    audio VAE keeps its weights in FP32 even when the pipeline requests BF16.\n    Match the actual loader\'s complete dotted-path segments, not substrings.\n    """\n    from collections import Counter\n    import torch\n\n    components = {}\n    for spec in pipe.blocks.expected_components:\n        name = spec.name\n        module = getattr(pipe, name, None)\n        if not callable(getattr(module, "named_parameters", None)):\n            continue  # tokenizers, schedulers and processors have no weights\n        if (any(getattr(module, flag, False) for flag in\n                ("is_quantized", "is_loaded_in_4bit", "is_loaded_in_8bit"))\n                or getattr(module, "hf_quantizer", None) is not None):\n            raise RuntimeError(f"Quantized component is forbidden: {name}")\n        keep_fp32 = getattr(module, "_keep_in_fp32_modules", None) or []\n        if isinstance(keep_fp32, str):\n            keep_fp32 = [keep_fp32]\n        counts, tensors = Counter(), Counter()\n        for parameter_name, parameter in module.named_parameters():\n            full_name = f"{name}.{parameter_name}"\n            # Quantized tensor subclasses may report their compute dtype as\n            # BF16. The native loader uses ordinary nn.Parameter objects.\n            if type(parameter) is not torch.nn.Parameter or getattr(parameter, "is_quantized", False):\n                raise RuntimeError(f"Non-native or quantized parameter is forbidden: {full_name}")\n            if getattr(parameter, "is_meta", False):\n                raise RuntimeError(f"Parameter was not materialized: {full_name}")\n            expected = torch.float32 if _parameter_kept_in_fp32(parameter_name, keep_fp32) else torch.bfloat16\n            if parameter.dtype != expected:\n                raise RuntimeError(\n                    f"Unexpected precision in {full_name}: {parameter.dtype}, expected {expected}"\n                )\n            dtype = str(parameter.dtype)\n            counts[dtype] += parameter.numel()\n            tensors[dtype] += 1\n        if not tensors:\n            raise RuntimeError(f"No loaded parameters available for precision audit: {name}")\n        record = {"component": name, "status": "passed", "parameter_counts": dict(counts),\n                  "tensor_counts": dict(tensors), "keep_in_fp32_modules": list(keep_fp32)}\n        components[name] = record\n        print("H3_PRECISION_AUDIT " + json.dumps(record, sort_keys=True), flush=True)\n    if not components:\n        raise RuntimeError("No model components available for precision audit")\n    report = {"status": "passed", "scope": "loaded_parameters_only", "components": components}\n    print("H3_PRECISION_AUDIT " + json.dumps(\n        {"status": "passed", "scope": report["scope"], "components": list(components)}, sort_keys=True\n    ), flush=True)\n    return report\n\n\ndef aligned_frames(seconds: float) -> int:\n    frames = 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17)\n    if not 5 <= seconds <= 15 or not 120 <= frames <= 360:\n        raise ValueError("Duration must align to 17*n+5 frames within 5–15s; maximum request is 14.375s.")\n    return frames\n\n\ndef _probe(path: Path, kind: str) -> float:\n    """Reject oversized media before loading all decoded frames into host RAM."""\n    if kind == "image":\n        from PIL import Image\n        with Image.open(path) as im:\n            if im.width * im.height > 16_000_000 or getattr(im, "n_frames", 1) != 1:\n                raise ValueError("Reference images must be static and at most 16 megapixels.")\n            if not .25 <= im.width / im.height <= 4:\n                raise ValueError("Reference image aspect ratio must be between 1:4 and 4:1.")\n            im.verify()\n        return 0\n    process = subprocess.run(\n        ["ffprobe", "-v", "error", "-threads", "1", "-show_format", "-show_streams", "-of", "json", str(path)],\n        capture_output=True, text=True, timeout=30, check=True,\n    )\n    info = json.loads(process.stdout)\n    streams = info.get("streams", [])\n    wanted = [s for s in streams if s.get("codec_type") == kind]\n    if not wanted:\n        raise ValueError(f"No {kind} stream in reference file.")\n    stream = wanted[0]\n    duration = float(stream.get("duration") or info.get("format", {}).get("duration") or 0)\n    if not math.isfinite(duration) or not 2 <= duration <= 15:\n        raise ValueError(f"Reference {kind} duration must be known and between 2 and 15s.")\n    if kind == "video":\n        from fractions import Fraction\n        rate = float(Fraction(stream.get("avg_frame_rate", "0")))\n        if not 0 < rate <= 60 or stream.get("width", 0) * stream.get("height", 0) > 1920 * 1080:\n            raise ValueError("Reference video must be at most 1080p / 60 fps.")\n    for audio in (s for s in streams if s.get("codec_type") == "audio"):\n        if int(audio.get("sample_rate", 0)) > 96000 or int(audio.get("channels", 0)) > 2:\n            raise ValueError("Reference audio must be mono/stereo and at most 96 kHz.")\n    return duration\n\n\ndef _normalize_video(path: Path, target: Path | None = None) -> Path:\n    # Bound decoded memory before MiniMaxH3VideoReference.from_file stacks all frames.\n    target = target or path.with_name(path.stem + "_24fps.mp4")\n    subprocess.run([\n        "ffmpeg", "-nostdin", "-y", "-v", "error", "-threads", "1", "-filter_threads", "1", "-i", str(path),\n        "-map", "0:v:0", "-map", "0:a:0?", "-t", "15",\n        "-vf", "scale=960:544:force_original_aspect_ratio=decrease:force_divisible_by=2,fps=24",\n        "-c:v", "libx264", "-crf", "18", "-preset", "fast", "-pix_fmt", "yuv420p",\n        "-c:a", "aac", "-ac", "2", "-ar", "32000", "-threads", "1", str(target),\n    ], check=True, capture_output=True, timeout=180)\n    return target\n\n\nclass MiniMaxBackend(PipelineBackendSupport):\n    _reference_probe = staticmethod(_probe)\n    _normalize_reference = staticmethod(_normalize_video)\n    _aligned_frames = staticmethod(aligned_frames)\n\n    def __init__(self, runtime_options=None):\n        self._init_runtime(runtime_options)\n        self.adapter_settings = load_adapter_settings()\n        self.pipe = None\n        self.precision_audit = None\n\n    @measured_load\n    @tracked_model_load\n    def load(self) -> None:\n        self.precision_audit = None\n        from preflight import check_resources\n        resources = check_resources()\n        import torch\n        from diffusers import ComponentsManager, ModularPipeline\n\n        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n            raise RuntimeError("A CUDA GPU with native BF16 support is required.")\n        revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\n        torch.backends.cuda.matmul.allow_tf32 = True\n        strategy = resources["strategy"]\n        print(f"A100 inference: native BF16; memory strategy={strategy}", flush=True)\n        if strategy == "component_cpu_offload":\n            if self.runtime_options.denoiser_offload != "stream":\n                raise ValueError("H3_DENOISER_OFFLOAD=sync applies only to group offload; this profile uses component offload.")\n            manager = ComponentsManager()\n            pipe = ModularPipeline.from_pretrained(\n                MODEL_ID, revision=revision, workflow="ref2va", components_manager=manager,\n            )\n            configure_pipeline(pipe, self.adapter_settings)\n            with self._stage("load.components"):\n                pipe.load_components(dtype=torch.bfloat16, revision=revision)\n            _require_loaded_components(pipe)\n            with self._stage("configure.adapters"):\n                apply_adapters(pipe, self.adapter_settings)\n            with self._stage("audit.parameter_precision"):\n                self.precision_audit = _audit_parameter_precision(pipe)\n            self.runtime_options.configure_attention(pipe.transformer_ref)\n            manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="12GB")\n        elif strategy == "group_cpu_offload":\n            pipe = ModularPipeline.from_pretrained(MODEL_ID, revision=revision, workflow="ref2va")\n            configure_pipeline(pipe, self.adapter_settings)\n            with self._stage("load.components"):\n                pipe.load_components(dtype=torch.bfloat16, revision=revision)\n            _require_loaded_components(pipe)\n            pipe.transformer_ref.requires_grad_(False)\n            with self._stage("configure.adapters"):\n                apply_adapters(pipe, self.adapter_settings)\n            with self._stage("audit.parameter_precision"):\n                self.precision_audit = _audit_parameter_precision(pipe)\n            self.runtime_options.configure_attention(pipe.transformer_ref)\n            pipe.transformer_ref.enable_group_offload(**self.runtime_options.group_offload_kwargs(torch))\n            configure_qwen_offload(pipe.text_encoder)\n            # A100 40GB cannot hold the 61.7GB BF16 denoiser as one component.\n            # Stream BF16 blocks without quantization; PCIe bandwidth affects speed.\n            pipe.vae.to("cuda")\n            pipe.audio_vae.to("cuda")\n        else:\n            raise ValueError(f"Unsupported A100 memory strategy: {strategy}")\n        self.pipe = pipe\n\n    def validate_request(self, *, steps, attention_backend=None, acceleration_lora=None):\n        return self.validate_generation_request(steps=steps, attention_backend=attention_backend,\n                                                acceleration_lora=acceleration_lora)\n\n    def _infer_raw(self, prepared):\n        """The original BF16 workflow stays wholly inside the exclusive GPU stage."""\n        if self.pipe is None:\n            raise RuntimeError("The model has not loaded.")\n        import torch\n        p = prepared.parameters\n        return self.pipe(\n            prompt=p["prompt"], references=prepared.references, num_frames=aligned_frames(p["seconds"]),\n            height=p["height"], width=p["width"], num_inference_steps=p["steps"],\n            generator=torch.Generator(device="cpu").manual_seed(p["seed"]),\n            output_type="pil", output=["videos", "audio", "sampling_rate"],\n        )\n', 'staged_backend.py': '"""Sequential component residency: single-image A100 40GB acceptance passed.\n\nVerified on Colab A100 40GB with about 83.47 GiB host RAM: public submission,\npolling, generation, download, and full audiovisual decode. The 512x288 test\nused one image, a 5s request/124 output frames, and 8 steps (7 denoising rounds).\nThe backbone stays BF16 while official _keep_in_fp32_modules layers remain FP32.\nNo quantization or fake output exists. This is an integration test, not a\nquality benchmark or acceptance of video/audio reference combinations.\nOptional runtime modes, adapters, and media scheduling added afterward have not\nbeen GPU-retested; their defaults preserve the accepted inference choices.\n\nTested with Python 3.13.16, PyTorch 2.14.1+cu130, Diffusers 0.43.0, and\nTransformers 5.7.0. The separately pinned PyTorch 2.10 install has not been\nGPU-tested. A100 80GB resident acceptance is recorded separately in README.md;\nit does not establish acceptance of this refactored staged pipeline.\n"""\nfrom __future__ import annotations\n\nimport gc\nimport os\nfrom pathlib import Path\nimport time\n\nfrom model_backend import MODEL_ID, MODEL_REVISION, _normalize_video, _probe, aligned_frames, _parameter_kept_in_fp32, _audit_parameter_precision\nfrom runtime_options import configure_qwen_offload, measured_load\nfrom pipeline_backend import PipelineBackendSupport, GIB\nfrom adapter_config import load_adapter_settings, configure_pipeline, apply_adapters, validate_steps\nfrom download_progress import tracked_model_load\n\n\nclass StagedMiniMaxBackend(PipelineBackendSupport):\n    """Exclusive GPU inference; Qwen and transformer_ref never co-reside."""\n    pipeline_model_peak_ram_bytes = 76 * GIB\n    resource_residency = "staged"\n    _reference_probe = staticmethod(_probe)\n    _normalize_reference = staticmethod(_normalize_video)\n    _aligned_frames = staticmethod(aligned_frames)\n\n    def __init__(self, runtime_options=None):\n        self._init_runtime(runtime_options)\n        self.adapter_settings = load_adapter_settings()\n        self.pipe = None\n        self.revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\n        self._phase = None\n        self.precision_audit = None\n\n    @staticmethod\n    def _log(message):\n        print(f"[{time.strftime(\'%H:%M:%S\')}] BF16 staged: {message}", flush=True)\n\n    def _load_required(self, names):\n        import torch\n        # Diffusers 0.43 load_components logs some failures instead of raising.\n        # Load individually to avoid its temporary dict retaining multiple newly\n        # loaded components before registering them.\n        for name in names:\n            if getattr(self.pipe, name, None) is None:\n                self._log(f"loading {name}")\n                with self._stage(f"load.{name}"):\n                    self.pipe.load_components(names=name, dtype=torch.bfloat16, revision=self.revision)\n            if getattr(self.pipe, name, None) is None:\n                raise RuntimeError(f"Required MiniMax-H3 component failed to load: {name}")\n            if name in {"text_encoder", "transformer_ref"}:\n                from collections import Counter\n                module = getattr(self.pipe, name)\n                keep_fp32 = getattr(module, "_keep_in_fp32_modules", []) or []\n                counts = Counter()\n                for parameter_name, parameter in module.named_parameters():\n                    counts[str(parameter.dtype)] += parameter.numel()\n                    expected = torch.float32 if _parameter_kept_in_fp32(parameter_name, keep_fp32) else torch.bfloat16\n                    if parameter.dtype != expected:\n                        raise RuntimeError(f"Unexpected precision in {name}.{parameter_name}: {parameter.dtype}, expected {expected}")\n                self._log(f"{name} original precision verified, parameter counts={dict(counts)}")\n\n    def _switch(self, target):\n        if self._phase != target:\n            with self._stage(f"switch.{target}"):\n                self._switch_impl(target)\n\n    def _switch_impl(self, target):\n        import torch\n        other = "text_encoder" if target == "denoise" else "transformer_ref"\n        self._phase = None  # a failed load must never leave a stale ready phase\n        self._log(f"unloading {other} before {target}")\n        self.pipe.unload_components(other)\n        gc.collect()\n        torch.cuda.empty_cache()\n        if target == "encode":\n            self._load_required(["text_encoder"])\n            configure_qwen_offload(self.pipe.text_encoder)\n            self._log("Qwen vision/embeddings/rotary on CUDA; decoder layers offloaded as whole BF16 blocks")\n        else:\n            self._load_required(["transformer_ref"])\n            self.pipe.transformer_ref.requires_grad_(False)\n            with self._stage("configure.adapters"):\n                apply_adapters(self.pipe, self.adapter_settings)\n            self.precision_audit = _audit_parameter_precision(self.pipe)\n            self.runtime_options.configure_attention(self.pipe.transformer_ref)\n            selected = getattr(self, "_active_generation_options", None)\n            if selected is not None:\n                self._apply_generation_transformer(selected)\n            self.pipe.transformer_ref.enable_group_offload(**self.runtime_options.group_offload_kwargs(torch))\n            self._log(f"denoiser offload={self.runtime_options.denoiser_offload}; attention={self.runtime_options.attention_backend}")\n        self._phase = target\n        self._log(f"{target} stage loaded; other large component released")\n\n    @measured_load\n    @tracked_model_load\n    def load(self):\n        import torch\n        from diffusers import ModularPipeline\n        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n            raise RuntimeError("Native-BF16 CUDA device is required")\n        if "A100" not in torch.cuda.get_device_name(0).upper():\n            raise RuntimeError("This acceptance build targets an NVIDIA A100")\n        import psutil, shutil\n        if psutil.virtual_memory().available < 70 * 1024**3:\n            raise RuntimeError("Staged BF16 requires at least 70 GiB available host RAM before loading")\n        if torch.cuda.get_device_properties(0).total_memory < 38 * 1024**3:\n            raise RuntimeError("Staged BF16 requires an A100 40 GB or larger")\n        cache = Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser()\n        cache.mkdir(parents=True, exist_ok=True)\n        if shutil.disk_usage(cache).free < 140 * 1024**3 and os.environ.get("H3_SKIP_DISK_CHECK") != "1":\n            raise RuntimeError("Staged BF16 needs 140 GiB free cache disk on the first download")\n        self.pipe = ModularPipeline.from_pretrained(\n            MODEL_ID, revision=self.revision, workflow="ref2va",\n        )\n        configure_pipeline(self.pipe, self.adapter_settings)\n        names = list(self.pipe.blocks.sub_blocks)\n        if names[:3] != ["before_encode", "text_encoder", "vae_encoder"]:\n            raise RuntimeError(f"Unexpected Ref2VA block topology: {names}")\n        if not any(name.startswith("denoise.") for name in names) or not any(name.startswith("decode.") for name in names):\n            raise RuntimeError(f"Unexpected Ref2VA block topology: {names}")\n        self._load_required(["tokenizer", "processor", "scheduler", "audio_scheduler", "vae", "audio_vae"])\n        self.pipe.vae.requires_grad_(False).to("cuda")\n        self.pipe.audio_vae.requires_grad_(False).to("cuda")\n        # Actually load both large components during startup, one at a time. This\n        # also downloads all required weights before the API advertises readiness.\n        self._switch("denoise")\n        self._switch("encode")\n        self._log("both large components loaded successfully in turn; encoder stage ready")\n\n    @staticmethod\n    def _cpu(value):\n        import torch\n        if isinstance(value, torch.Tensor):\n            return value.detach().cpu()\n        if isinstance(value, list):\n            return [StagedMiniMaxBackend._cpu(item) for item in value]\n        if isinstance(value, tuple):\n            return tuple(StagedMiniMaxBackend._cpu(item) for item in value)\n        if isinstance(value, dict):\n            return {key: StagedMiniMaxBackend._cpu(item) for key, item in value.items()}\n        return value\n\n    def _run(self, state, stage):\n        import torch\n        # The resolved workflow has flat leaf keys: before_encode, text_encoder,\n        # vae_encoder, denoise.prepare_layout ... denoise.after_denoise,\n        # decode.video, decode.audio. Calling the documented block contract\n        # (components, PipelineState) avoids a deepcopy of large reference media.\n        with torch.inference_mode():\n            for name, block in self.pipe.blocks.sub_blocks.items():\n                selected = name in {"before_encode", "text_encoder", "vae_encoder"} if stage == "encode" else name.startswith(stage + ".")\n                if selected:\n                    self._log(f"running {name}")\n                    with self._stage(f"block.{name}"):\n                        _, state = block(self.pipe, state)\n        return state\n\n    def validate_request(self, *, steps, attention_backend=None, acceleration_lora=None):\n        return self.validate_generation_request(steps=steps, attention_backend=attention_backend,\n                                                acceleration_lora=acceleration_lora)\n\n    def _infer_raw(self, prepared):\n        """All staged model swapping, text/VAE encode, denoise and decode use GPU lease."""\n        import torch\n        from diffusers.modular_pipelines.modular_pipeline import PipelineState\n        if self.pipe is None:\n            raise RuntimeError("Backend has not loaded")\n        p = prepared.parameters\n        state = None\n        try:\n            self._switch("encode")\n            state = PipelineState(values={\n                "prompt": p["prompt"], "references": prepared.references, "num_frames": aligned_frames(p["seconds"]),\n                "height": p["height"], "width": p["width"], "num_inference_steps": p["steps"],\n                "generator": torch.Generator(device="cpu").manual_seed(p["seed"]),\n                "output_type": "pil",\n            })\n            state = self._run(state, "encode")\n            # prompt_embeds is the one GPU tensor emitted by the encoder; VAE\n            # conditioning latents already live on CPU. Do not retain CUDA state\n            # while swapping 62GB components; preserve generator state unchanged.\n            state.values = self._cpu(state.values)\n            self._switch("denoise")\n            # The denoiser passes prompt_embeds straight through, unlike layout/\n            # condition latents, which their preparation blocks move to device.\n            state.set("prompt_embeds", state.get("prompt_embeds").to("cuda"))\n            state = self._run(state, "denoise")\n            # Free the large denoiser before VAE decode; no longer needed for this\n            # job. Keep the final latents on CUDA for the official decoder blocks.\n            with self._stage("unload.transformer_ref"):\n                self.pipe.unload_components("transformer_ref")\n                self._phase = None\n            state = self._run(state, "decode")\n            return {"videos": state.get("videos"), "audio": state.get("audio"), "sampling_rate": state.get("sampling_rate")}\n        finally:\n            state = None\n', 'staged_server.py': '"""Compatibility API entry point; staging is chosen by the model container owner."""\nfrom server import create_app\n\n\ndef build_app():\n    return create_app()\n', 'smoke_test.py': '"""Exercise a REAL running API and verify its downloaded audiovisual MP4.\n\nThis is a low-step integration smoke test, not a generation-quality benchmark.\nIt has no fake backend, synthetic-video fallback, or GPU/model loader.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom fractions import Fraction\nimport hashlib\nimport json\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport time\nimport uuid\nfrom urllib.parse import urlsplit\n\nPARAMETERS = {\n    "seconds": 5, "width": 512, "height": 288, "steps": 8, "seed": 42,\n}\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef require(condition: bool, message: str) -> None:\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef save_report(outputdir: Path, report: dict) -> None:\n    """Checkpoint the accepted job ID before polling; resuming never resubmits."""\n    target = outputdir / "test_report.json"\n    temporary = target.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(target)\n\n\n@contextmanager\ndef content_history_lock(path):\n    """OS-held writer lock; process exit releases it without stale lock ownership."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.with_suffix(path.suffix + ".lock").open("a+b") as lock:\n        if os.fstat(lock.fileno()).st_size == 0:\n            lock.write(b"0")\n            lock.flush()\n        deadline = time.monotonic() + 5\n        while True:\n            lock.seek(0)\n            try:\n                if os.name == "nt":\n                    import msvcrt\n                    msvcrt.locking(lock.fileno(), msvcrt.LK_NBLCK, 1)\n                else:\n                    import fcntl\n                    fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n                break\n            except OSError:\n                require(time.monotonic() < deadline, "Content history is busy; no generation was submitted")\n                time.sleep(.05)\n        try:\n            yield\n        finally:\n            lock.seek(0)\n            if os.name == "nt":\n                msvcrt.locking(lock.fileno(), msvcrt.LK_UNLCK, 1)\n            else:\n                fcntl.flock(lock.fileno(), fcntl.LOCK_UN)\n\n\ndef register_content(path: Path, reference_hash: str, prompt: str, *, outputdir: Path, api_base_url: str) -> dict:\n    """Reserve fresh content durably before the one paid submission attempt."""\n    canonical = json.dumps({"reference_sha256": reference_hash, "prompt": prompt.strip()},\n                           sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n    fingerprint = hashlib.sha256(canonical.encode("utf-8")).hexdigest()\n    with content_history_lock(path):\n        history = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"schema_version": 1, "entries": {}}\n        require(history.get("schema_version") == 1 and isinstance(history.get("entries"), dict),\n                "Invalid content history; refusing to submit without reliable duplicate detection")\n        require(fingerprint not in history["entries"],\n                "Reference and prompt combination was already registered; provide new content or use --resume-job for query/download only")\n        record = {"fingerprint": fingerprint, "reference_sha256": reference_hash, "prompt": prompt.strip(),\n                  "registered_at": utc_now(), "request_id": uuid.uuid4().hex,\n                  "outputdir": str(outputdir), "api_base_url": api_base_url}\n        history["entries"][fingerprint] = record\n        temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n        try:\n            with temporary.open("x", encoding="utf-8") as output:\n                json.dump(history, output, ensure_ascii=False, indent=2)\n                output.flush()\n                os.fsync(output.fileno())\n            temporary.replace(path)\n            if os.name != "nt":\n                descriptor = os.open(path.parent, os.O_RDONLY)\n                try:\n                    os.fsync(descriptor)\n                finally:\n                    os.close(descriptor)\n        finally:\n            temporary.unlink(missing_ok=True)\n    return {**record, "history_path": str(path)}\n\n\ndef verify_video(path: Path) -> dict:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)],\n        capture_output=True, text=True, timeout=120, check=True,\n    )\n    media = json.loads(probe.stdout)\n    video = next((s for s in media["streams"] if s["codec_type"] == "video"), None)\n    audio = next((s for s in media["streams"] if s["codec_type"] == "audio"), None)\n    require(video is not None and audio is not None, "MP4 must contain both video and audio streams")\n    frames = int(video.get("nb_read_frames") or video.get("nb_frames") or 0)\n    fps = float(Fraction(video.get("avg_frame_rate", "0")))\n    duration = float(video.get("duration") or media["format"]["duration"])\n    require(video.get("codec_name") == "h264", "Expected an H.264 video stream")\n    require(frames >= 120 and 5 <= duration <= 6, "Expected at least 120 frames and a 5–6 second video")\n    require(abs(fps - 24) < 0.01, "Expected 24 fps")\n    require((video["width"], video["height"]) == (512, 288), "Unexpected video dimensions")\n    require(int(audio.get("channels", 0)) == 2, "Expected stereo generated audio")\n    subprocess.run(\n        ["ffmpeg", "-nostdin", "-v", "error", "-xerror", "-i", str(path),\n         "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"],\n        capture_output=True, text=True, timeout=180, check=True,\n    )\n    return {\n        "video_codec": video["codec_name"], "width": video["width"], "height": video["height"],\n        "decoded_frames": frames, "fps": fps, "duration_seconds": duration,\n        "audio_codec": audio.get("codec_name"), "audio_channels": audio["channels"],\n        "audio_sample_rate": audio.get("sample_rate"), "full_decode_passed": True,\n        "bytes": path.stat().st_size, "sha256": sha256(path),\n    }\n\n\ndef run(args, report: dict) -> None:\n    import requests\n\n    reference = None\n    if not args.resume_job:\n        require(bool(getattr(args, "reference", None)), "New generation requires an explicit --reference image")\n        require(isinstance(getattr(args, "prompt", None), str) and bool(args.prompt.strip()),\n                "New generation requires an explicit nonempty --prompt")\n        reference = Path(args.reference).resolve()\n        require(reference.is_file(), "Reference image does not exist")\n        require(reference.stat().st_size <= 8 * 1024 * 1024, "Reference image exceeds smoke-test size limit")\n        from PIL import Image\n        with Image.open(reference) as image:\n            image.verify()\n    parsed = urlsplit(args.url)\n    require(parsed.scheme in {"https", "http"} and bool(parsed.hostname), "Provide an HTTP(S) API base URL")\n    require(not parsed.username and not parsed.password and not parsed.query and not parsed.fragment,\n            "Do not embed credentials, query parameters or fragments in the API URL")\n    key = os.environ.get("H3_API_KEY", "")\n    require(len(key) >= 32, "Set H3_API_KEY to the running server\'s token")\n    for command in ("ffprobe", "ffmpeg"):\n        require(shutil.which(command) is not None, f"{command} is required before submitting a paid GPU job")\n    base = args.url.rstrip("/")\n    report["api_base_url"] = base\n    parameters = {**PARAMETERS, "steps": args.steps,\n                  "attention_backend": getattr(args, "attention_backend", "auto"),\n                  "acceleration_lora": getattr(args, "acceleration_lora", "none")}\n    require(parameters["acceleration_lora"] == "none" or parameters["steps"] == 5,\n            "Turbo Ref2VA requires explicit --steps 5 (four denoiser evaluations)")\n    if not args.resume_job:\n        parameters["prompt"] = args.prompt.strip()\n        report["parameters"] = parameters\n    report["test_scope"] = "real API integration; low-step smoke test, not visual quality acceptance"\n    headers = {"Authorization": f"Bearer {key}"}\n    session = requests.Session()\n    started = time.monotonic()\n    deadline = started + args.startup_timeout\n    print("Waiting for API and manager availability; model loading may continue while jobs queue...", flush=True)\n    while True:\n        try:\n            response = session.get(base + "/health", timeout=10)\n            health = response.json()\n            if not isinstance(health, dict):\n                raise ValueError("Invalid health response")\n            # New APIs expose admission separately from model execution. A legacy\n            # aggregate ready flag is only a fallback when capability keys are\n            # absent, never an override for an explicitly unavailable API/manager.\n            explicit_capabilities = "api_ready" in health or "manager_ready" in health\n            admission_ready = response.status_code == 200 and (\n                health.get("api_ready") is True and health.get("manager_ready") is True\n                if explicit_capabilities else health.get("ready") is True)\n            if admission_ready or (args.resume_job and response.status_code in {200, 503}):\n                report["health"] = {"http_status": response.status_code,\n                    "ready": health.get("ready") is True, "admission_ready": admission_ready,\n                    **{name: health.get(name) if type(health.get(name)) is bool else None\n                       for name in ("api_ready", "manager_ready", "model_ready")},\n                    "observed_at": utc_now()}\n                break\n        except (requests.RequestException, ValueError):\n            pass\n        require(time.monotonic() < deadline, "API and manager did not become available before startup timeout")\n        time.sleep(args.poll_seconds)\n    print(f"/health: HTTP {report[\'health\'][\'http_status\']}, admission_ready={report[\'health\'][\'admission_ready\']}, "\n          f"ready={report[\'health\'][\'ready\']}, model_ready={report[\'health\'][\'model_ready\']}", flush=True)\n\n    if args.resume_job:\n        job_id = args.resume_job\n        report["resumed"] = True\n        report["test_scope"] = "resume existing job: status/download only; no new content registration or generation request"\n        print(f"Resuming status/download for {job_id}; no generation submission", flush=True)\n    else:\n        response = session.post(base + "/v1/jobs", timeout=30)\n        report["unauthorized_submit_http_status"] = response.status_code\n        require(response.status_code in {401, 403}, "Unauthenticated submission was not rejected")\n        print(f"Unauthenticated POST rejected: HTTP {response.status_code}", flush=True)\n        report["reference_sha256"] = sha256(reference)\n        report["reference_source"] = str(reference)\n        history_path = Path(getattr(args, "content_history", None) or args.outputdir.parent / "content-history.json").resolve()\n        registration = register_content(history_path, report["reference_sha256"], parameters["prompt"],\n                                        outputdir=args.outputdir, api_base_url=base)\n        report["content_registration"] = registration\n        save_report(args.outputdir, report)\n        with reference.open("rb") as image:\n            require(sha256(reference) == report["reference_sha256"], "Reference changed after content registration; generation not submitted")\n            response = session.post(\n                base + "/v1/jobs", headers={**headers, "Idempotency-Key": registration["request_id"]}, timeout=120,\n                data={**parameters, "kinds": json.dumps(["image"])},\n                files=[("files", (reference.name, image, mimetypes.guess_type(reference.name)[0] or "image/jpeg"))],\n            )\n        report["submit_http_status"] = response.status_code\n        require(response.status_code == 202, f"Generation submit failed: HTTP {response.status_code}")\n        task = response.json()\n        job_id = task["id"]\n        report["actual_seconds"] = task.get("actual_seconds")\n        print(f"Submitted real job {job_id}: HTTP 202", flush=True)\n    require(re.fullmatch(r"[a-f0-9]{32}", job_id) is not None, "Unexpected job ID format")\n    report["job_id"] = job_id\n    report.setdefault("status_history", [])\n    save_report(args.outputdir, report)\n\n    deadline = time.monotonic() + args.generation_timeout\n    while True:\n        response = session.get(base + f"/v1/jobs/{job_id}", headers=headers, timeout=30)\n        require(response.status_code == 200, f"Status query failed: HTTP {response.status_code}")\n        task = response.json()\n        status = task.get("status")\n        if args.resume_job and task.get("generation_parameters"):\n            report["parameters"] = task["generation_parameters"]\n        report["status_history"].append({"http_status": response.status_code, "status": status,\n                                        "queue_position": task.get("queue_position"),\n                                        "waiting_for": task.get("waiting_for"),\n                                        "active_phase": task.get("active_phase"), "observed_at": utc_now()})\n        save_report(args.outputdir, report)\n        print(f"{utc_now()} GET job: HTTP 200, status={status}, "\n              f"queue_position={task.get(\'queue_position\')}, waiting_for={task.get(\'waiting_for\')}, "\n              f"active_phase={task.get(\'active_phase\')}", flush=True)\n        if status in {"succeeded", "failed", "cancelled"}:\n            report["terminal_status"] = status\n            report["metrics"] = task.get("metrics")\n            report["lifecycle"] = task.get("lifecycle")\n            report["terminal_error"] = task.get("error")\n            report["cleanup_complete"] = task.get("cleanup_complete")\n            break\n        require(status in {"queued", "running", "cancel_requested"}, "Unknown job status")\n        require(time.monotonic() < deadline, "Generation timeout; task may still be running on server")\n        time.sleep(args.poll_seconds)\n    require(status == "succeeded", "Real model generation failed; inspect the server log")\n\n    output = args.outputdir / f"{job_id}.mp4"\n    with session.get(base + f"/v1/jobs/{job_id}/file", headers=headers, timeout=120, stream=True) as response:\n        report["download_http_status"] = response.status_code\n        require(response.status_code == 200, f"Video download failed: HTTP {response.status_code}")\n        size = 0\n        with output.open("wb") as destination:\n            for chunk in response.iter_content(1024 * 1024):\n                size += len(chunk)\n                require(size <= 128 * 1024 * 1024, "Smoke-test output unexpectedly exceeds 128 MiB")\n                destination.write(chunk)\n    report["video"] = verify_video(output)\n    report["video_path"] = str(output.resolve())\n    report["elapsed_seconds"] = round(time.monotonic() - started, 2)\n    report["passed"] = True\n    print("API_SMOKE_TEST_PASSED", flush=True)\n    print(json.dumps(report["video"], indent=2), flush=True)\n    print(f"VIDEO_PATH={output.resolve()}", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--url", required=True, help="Base URL of an already running real server")\n    parser.add_argument("--reference", help="Required for new generation: explicitly chosen local reference image")\n    parser.add_argument("--prompt", help="Required for new generation: fresh explicit prompt; no automatic default")\n    parser.add_argument("--content-history", type=Path,\n                        help="Shared durable content registry; default OUTPUTDIR.parent/content-history.json")\n    parser.add_argument("--steps", type=int, choices=range(2, 51), default=8,\n                        help="Scheduler grid points; Turbo Ref2VA requires 5 for four evaluations")\n    parser.add_argument("--attention-backend", choices=("auto", "native", "_native_flash"), default="auto")\n    parser.add_argument("--acceleration-lora", choices=("none", "lightx2v-ref2va-4step"), default="none")\n    parser.add_argument("--resume-job", help="Existing job ID: query and download only, never submit another generation")\n    parser.add_argument("--outputdir", type=Path, default=Path("smoke_result"))\n    parser.add_argument("--startup-timeout", type=int, default=3600)\n    parser.add_argument("--generation-timeout", type=int, default=2700)\n    parser.add_argument("--poll-seconds", type=int, choices=range(1, 61), default=15, metavar="1..60")\n    args = parser.parse_args()\n    args.outputdir = args.outputdir.resolve()\n    args.outputdir.mkdir(parents=True, exist_ok=True)\n    report = {"started_at": utc_now(), "passed": False}\n    if args.resume_job:\n        require(re.fullmatch(r"[a-f0-9]{32}", args.resume_job) is not None, "Unexpected resume job ID")\n        saved = args.outputdir / "test_report.json"\n        if saved.exists():\n            previous = json.loads(saved.read_text(encoding="utf-8"))\n            require(previous.get("job_id") == args.resume_job, "Output directory belongs to a different job")\n            require(previous.get("api_base_url") == args.url.rstrip("/"), "Output directory belongs to a different API")\n            report.update(previous)\n            report.update(passed=False, resumed_at=utc_now())\n            report.pop("error", None)\n            report.pop("error_type", None)\n    try:\n        run(args, report)\n    except BaseException as error:\n        # No response bodies, headers, credentials, or environment are copied into the report.\n        report["error_type"] = type(error).__name__\n        message = str(error)\n        if os.environ.get("H3_API_KEY"):\n            message = message.replace(os.environ["H3_API_KEY"], "[REDACTED]")\n        report["error"] = message\n        raise\n    finally:\n        report["finished_at"] = utc_now()\n        path = args.outputdir / "test_report.json"\n        save_report(args.outputdir, report)\n        print(f"REPORT_PATH={path}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'runtime_options.py': '"""BF16 runtime controls and lightweight host-wall-time diagnostics.\n\nThe accepted stream/on-demand-pinning path remains the default. ``sync`` and\nexplicit attention choices are diagnostic alternatives, not GPU-validated\nspeedups. No option changes weight precision or downloads attention kernels.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom contextvars import ContextVar\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom functools import wraps\nimport inspect\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport threading\nimport time\n\n\n@dataclass(frozen=True)\nclass RuntimeOptions:\n    denoiser_offload: str = "stream"\n    attention_backend: str = "auto"\n\n    def __post_init__(self):\n        if self.denoiser_offload not in {"stream", "sync"}:\n            raise ValueError("H3_DENOISER_OFFLOAD must be stream or sync (experimental).")\n        if self.attention_backend not in {"auto", "native", "_native_flash"}:\n            raise ValueError("H3_ATTENTION_BACKEND must be auto, native, or _native_flash.")\n\n    @classmethod\n    def from_env(cls):\n        return cls(\n            denoiser_offload=os.environ.get("H3_DENOISER_OFFLOAD", "stream"),\n            attention_backend=os.environ.get("H3_ATTENTION_BACKEND", "auto"),\n        )\n\n    def group_offload_kwargs(self, torch):\n        return {\n            "onload_device": torch.device("cuda"),\n            "offload_device": torch.device("cpu"),\n            "offload_type": "block_level",\n            "num_blocks_per_group": 1,\n            "use_stream": self.denoiser_offload == "stream",\n            "low_cpu_mem_usage": self.denoiser_offload == "stream",\n        }\n\n    def configure_attention(self, transformer):\n        if self.attention_backend != "auto":\n            # Official Diffusers v0.43 API. _native_flash forces PyTorch SDPA\n            # FLASH_ATTENTION; errors must propagate, with no silent fallback.\n            transformer.set_attention_backend(self.attention_backend)\n\n\ndef configure_qwen_offload(text_encoder):\n    """Preserve the A100-tested placement of Qwen\'s directly accessed weights."""\n    import torch\n    from diffusers.hooks import apply_group_offloading\n\n    text_encoder.requires_grad_(False)\n    encoder = text_encoder.model\n    language = encoder.language_model\n    # Qwen reads visual.pos_embed.weight.device outside forward and calls\n    # embed_tokens before language.forward. Rotary uses its buffer directly.\n    encoder.visual.to("cuda")\n    language.embed_tokens.to("cuda")\n    language.norm.to("cuda")\n    language.rotary_emb.to("cuda")\n    for layer in language.layers:\n        apply_group_offloading(\n            layer, onload_device=torch.device("cuda"), offload_device=torch.device("cpu"),\n            offload_type="block_level", num_blocks_per_group=1, use_stream=False,\n        )\n\n\ndef resource_snapshot():\n    """Read counters only: never initialize CUDA, synchronize, or scan weights."""\n    result = {}\n    try:\n        import psutil\n        process = psutil.Process()\n        memory = process.memory_info()\n        cpu = process.cpu_times()\n        host = psutil.virtual_memory()\n        result.update(\n            rss_bytes=memory.rss, cpu_seconds=cpu.user + cpu.system,\n            host_available_bytes=host.available, host_free_bytes=host.free,\n        )\n        for source, key in ((memory, "shared"), (host, "cached")):\n            if hasattr(source, key):\n                result[("process_" if source is memory else "host_") + key + "_bytes"] = getattr(source, key)\n        try:\n            result["read_bytes"] = process.io_counters().read_bytes\n        except (AttributeError, OSError, psutil.Error):\n            pass\n    except Exception:\n        # Counter access can be denied by a container or disappear during exit.\n        # Diagnostics must not abort inference.\n        pass\n    torch = sys.modules.get("torch")\n    if torch is not None:\n        try:\n            if torch.cuda.is_initialized():\n                result["cuda_allocated_bytes"] = torch.cuda.memory_allocated()\n                result["cuda_reserved_bytes"] = torch.cuda.memory_reserved()\n        except (AttributeError, RuntimeError):\n            pass\n    return result\n\n\ndef emit_metric(event):\n    # A leading newline separates our JSON from tqdm\'s carriage-return line.\n    try:\n        print("\\nH3_METRIC " + json.dumps(event, ensure_ascii=False, sort_keys=True), flush=True)\n    except (OSError, ValueError):\n        pass\n\n\nclass RuntimeMetrics:\n    def __init__(self, kind, backend, options):\n        self.started = time.perf_counter()\n        self._owner_thread = threading.get_ident()\n        self._phase_stack = []\n        self._lock = threading.RLock()\n        self.data = {\n            "schema_version": 1, "kind": kind, "backend": backend,\n            "started_at": datetime.now(timezone.utc).isoformat(),\n            "runtime_options": asdict(options),\n            "timing": "host_wall_seconds_without_cuda_synchronize",\n            "resource_counter_scope": "process; overlapping stage deltas are not additive",\n            "resources_before": resource_snapshot(), "stages": [],\n            "status": "running",\n        }\n\n    @contextmanager\n    def stage(self, phase):\n        started = time.perf_counter()\n        # A job may move from its prepare executor to the GPU and encode workers.\n        # Child stages identify themselves explicitly; thread identity is not job identity.\n        is_owner = not phase.startswith("child.")\n        if is_owner:\n            with self._lock:\n                self._phase_stack.append((phase, started))\n        before = resource_snapshot()\n        emit_metric({"event": "stage_start", "phase": phase, "kind": self.data["kind"]})\n        status, error_type = "succeeded", None\n        try:\n            yield\n        except BaseException as error:\n            status, error_type = "failed", type(error).__name__\n            raise\n        finally:\n            after = resource_snapshot()\n            record = {\n                "phase": phase, "status": status,\n                "wall_seconds": round(time.perf_counter() - started, 6),\n                "scope": "main" if is_owner else "child",\n                "resources_before": before, "resources_after": after,\n            }\n            for key in ("read_bytes", "cpu_seconds"):\n                if key in before and key in after:\n                    record[key + "_delta"] = max(0, after[key] - before[key])\n            if error_type:\n                record["error_type"] = error_type\n            with self._lock:\n                self.data["stages"].append(record)\n                if is_owner:\n                    self._phase_stack.pop()\n            emit_metric({"event": "stage_end", "kind": self.data["kind"], **record})\n\n    def phase_snapshot(self):\n        with self._lock:\n            if not self._phase_stack:\n                return None\n            phase, started = self._phase_stack[-1]\n        return {"phase": phase, "elapsed_seconds": round(time.perf_counter() - started, 3), "kind": self.data["kind"]}\n\n    def record_schedule(self, record):\n        """Append path-free scheduling attribution under the metrics lock."""\n        with self._lock:\n            self.data.setdefault("scheduling", []).append(dict(record))\n\n    def finish(self, error=None):\n        self.data.update(\n            status="succeeded" if error is None else "failed",\n            finished_at=datetime.now(timezone.utc).isoformat(),\n            wall_seconds=round(time.perf_counter() - self.started, 6),\n            resources_after=resource_snapshot(),\n        )\n        if error is not None:\n            self.data["error_type"] = type(error).__name__\n        return self.data\n\n\nclass RuntimeSupport:\n    def _init_runtime(self, options=None):\n        self.runtime_options = options or RuntimeOptions.from_env()\n        self._loaded = False\n        self._metrics_context = ContextVar(f"h3_metrics_{id(self)}", default=None)\n        self._last_metrics_context = ContextVar(f"h3_last_metrics_{id(self)}", default=None)\n        self._startup_active_metrics = None\n        self.startup_metrics = None\n        self.last_metrics = None\n\n    @property\n    def _active_metrics(self):\n        return self._metrics_context.get()\n\n    @_active_metrics.setter\n    def _active_metrics(self, value):\n        self._metrics_context.set(value)\n\n    @property\n    def last_metrics(self):\n        return self._last_metrics_context.get()\n\n    @last_metrics.setter\n    def last_metrics(self, value):\n        self._last_metrics_context.set(value)\n\n    @contextmanager\n    def metrics_context(self, metrics):\n        token = self._metrics_context.set(metrics)\n        try:\n            yield metrics\n        finally:\n            self._metrics_context.reset(token)\n\n    @property\n    def current_phase(self):\n        snapshot = self.phase_snapshot()\n        return snapshot["phase"] if snapshot else None\n\n    def phase_snapshot(self, context=None):\n        metrics = getattr(context, "metrics", None) if context is not None else self._active_metrics or self._startup_active_metrics\n        return metrics.phase_snapshot() if metrics is not None else None\n\n    def _new_metrics(self, kind):\n        metrics = RuntimeMetrics(kind, type(self).__name__, self.runtime_options)\n        settings = getattr(self, "adapter_settings", None)\n        if settings is not None:\n            metrics.data["adapters"] = settings.as_dict()\n            metrics.data["adapters_scope"] = "preloaded_deployment; per-job applied choices are in generation_options"\n        return metrics\n\n    @contextmanager\n    def _stage(self, phase):\n        if self._active_metrics is None:\n            yield\n        else:\n            with self._active_metrics.stage(phase):\n                yield\n\n\ndef measured_load(function):\n    @wraps(function)\n    def wrapper(self, *args, **kwargs):\n        if self._loaded:\n            return None\n        metrics = self._new_metrics("startup")\n        self._active_metrics = metrics\n        self._startup_active_metrics = metrics\n        error = None\n        try:\n            with metrics.stage("load.total"):\n                result = function(self, *args, **kwargs)\n            self._loaded = True\n            return result\n        except BaseException as caught:\n            error = caught\n            raise\n        finally:\n            self.startup_metrics = metrics.finish(error)\n            self._active_metrics = None\n            self._startup_active_metrics = None\n    return wrapper\n\n\ndef measured_generation(function):\n    signature = inspect.signature(function)\n\n    @wraps(function)\n    def wrapper(self, *args, **kwargs):\n        self.last_metrics = None  # A failed job must never expose the previous job\'s metrics.\n        arguments = signature.bind(self, *args, **kwargs).arguments\n        output_path = Path(arguments["output_path"])\n        metrics = self._new_metrics("generation")\n        self._active_metrics = metrics\n        error = None\n        try:\n            with metrics.stage("generate.total"):\n                return function(self, *args, **kwargs)\n        except BaseException as caught:\n            error = caught\n            raise\n        finally:\n            self.last_metrics = metrics.finish(error)\n            self._active_metrics = None\n            sidecar = output_path.with_suffix(".metrics.json")\n            temporary = sidecar.with_suffix(sidecar.suffix + ".tmp")\n            try:\n                sidecar.parent.mkdir(parents=True, exist_ok=True)\n                temporary.write_text(json.dumps(self.last_metrics, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n                temporary.replace(sidecar)\n            except OSError as write_error:\n                # Observability must not turn a generated video into a failed job\n                # or replace the original inference exception.\n                self.last_metrics["metrics_write_error"] = type(write_error).__name__\n                emit_metric({"event": "metrics_write_failed", "phase": "metrics", "error_type": type(write_error).__name__})\n    return wrapper\n', 'adapter_config.py': '"""Opt-in, pinned MiniMax-H3 Ref2VA adapters. No GPU acceptance is claimed.\n\nThe baseline is unchanged with both settings ``none``. LoRAs are loaded into\neach freshly loaded transformer_ref BEFORE group-offload hooks are installed.\nOnly reviewed safetensors assets are accepted; no arbitrary remote code, URLs,\npickle files, quantized weights, or silent cross-workflow routing.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport hashlib\nimport math\nimport os\nfrom pathlib import Path\nimport re\nfrom download_progress import adapter_progress, current_progress, tracked_adapter_application\n\n\n@dataclass(frozen=True)\nclass AdapterSpec:\n    key: str\n    repository: str\n    filename: str\n    revision: str\n    sha256: str\n    size: int\n    source_tensors: int\n\n\nTURBO = AdapterSpec(\n    "lightx2v-ref2va-4step", "lightx2v/Minimax-h3-Turbo",\n    "minimax_h3_ref2v_turbo_4step_v0.1_bf16.safetensors",\n    "1a4a150fc6707329409eb52e5d75a0e987ac1928",\n    "9e642fc8749c74f8da5e2382877ab5c7aa37b9a73b7fd0d6d457bd1b3cb1ae99",\n    1383677768, 624,\n)\nREALISM = AdapterSpec(\n    "fal-realism-people", "fal/MiniMax-H3-Realism-People-LoRA",\n    "h3-realism-people-t2v-i2v-r2v.safetensors",\n    "039cc8579d7aa357a882d7f4111b25da4f72dccc",\n    "acc529601d2da117fb81179e76c56e488a3beab1171659d305f04fa3655b787e",\n    131229656, 208,\n)\n\n\n@dataclass(frozen=True)\nclass AdapterSettings:\n    acceleration: str = "none"\n    style: str = "none"\n    style_scale: float = 1.0\n\n    def __post_init__(self):\n        if self.acceleration not in {"none", TURBO.key}:\n            raise ValueError("H3_ACCELERATION_LORA must be none or lightx2v-ref2va-4step")\n        if self.style not in {"none", REALISM.key}:\n            raise ValueError("H3_STYLE_LORA must be none or fal-realism-people; arbitrary/FL2VA adapters are not accepted")\n        if not math.isfinite(self.style_scale) or not 0.0 < self.style_scale <= 1.0:\n            raise ValueError("H3_STYLE_LORA_SCALE must be finite and in (0, 1]")\n\n    @property\n    def turbo(self):\n        return self.acceleration != "none"\n\n    def as_dict(self):\n        return {\n            "acceleration": self.acceleration, "style": self.style,\n            "style_scale": self.style_scale,\n            "reference_image_resize": "match" if self.turbo else "official-2048-short-edge",\n            "required_steps": 5 if self.turbo else None,\n            "denoiser_evaluations": 4 if self.turbo else None,\n            "fused": False,\n            "validation": "not GPU-tested" if self.turbo or self.style != "none" else "baseline",\n        }\n\n\ndef load_adapter_settings(environ=None):\n    env = os.environ if environ is None else environ\n    return AdapterSettings(\n        acceleration=env.get("H3_ACCELERATION_LORA", "none").strip(),\n        style=env.get("H3_STYLE_LORA", "none").strip(),\n        style_scale=float(env.get("H3_STYLE_LORA_SCALE", "1.0")),\n    )\n\n\ndef validate_steps(settings, steps):\n    if settings.turbo and steps != 5:\n        raise ValueError("Ref2VA Turbo requires steps=5 sigma grid points, which gives exactly 4 denoiser evaluations")\n\n\ndef reference_match_size(width, height, target_width, target_height, multiple=32):\n    """ModelTC training policy: cap reference area at canvas area, preserve ratio.\n\n    Rounding follows the author\'s implementation. Small inputs are not scaled\n    up before rounding; a final nearest-32 rounding can increase an axis.\n    """\n    if min(width, height, target_width, target_height, multiple) <= 0:\n        raise ValueError("Image and target dimensions must be positive")\n    if width > 4 * height or height > 4 * width:\n        raise ValueError("Reference image aspect ratio must be between 1:4 and 4:1")\n    scale = min(1.0, math.sqrt(target_width * target_height / (width * height)))\n    return (max(multiple, round(width * scale / multiple) * multiple),\n            max(multiple, round(height * scale / multiple) * multiple))\n\n\ndef configure_pipeline(pipe, settings):\n    """Replace only this pipeline\'s resolved Ref2VA setup when Turbo is enabled.\n\n    The original setup still owns validation and video/audio normalization. Its\n    CPU image resize is replaced from the original pixels before any encoder\n    runs. This intentionally retains a small CPU resize overhead instead of\n    copying upstream\'s full setup implementation or patching process globals.\n    """\n    if not settings.turbo:\n        return\n    from diffusers.modular_pipelines.minimax_h3.before_encoder import MiniMaxH3Ref2VASetupStep\n    from diffusers.modular_pipelines.minimax_h3 import MiniMaxH3ImageReference\n    from PIL import Image\n\n    original = pipe.blocks.sub_blocks.get("before_encode")\n    if getattr(original, "_h3_turbo_match", False):\n        return\n    if not isinstance(original, MiniMaxH3Ref2VASetupStep):\n        raise RuntimeError("Unexpected Ref2VA setup block; refusing to change reference resizing")\n\n    class MatchReferenceSetup(MiniMaxH3Ref2VASetupStep):\n        _h3_turbo_match = True\n\n        def __call__(self, components, state):\n            _, state = original(components, state)\n            normalized = list(state.get("normalized_references"))\n            references = state.get("references")\n            if len(normalized) != len(references):\n                raise RuntimeError("Reference normalization changed the reference count")\n            for index, reference in enumerate(references):\n                if reference.kind != "image":\n                    continue\n                # This API decodes uploaded images with from_file(), which\n                # returns PIL. Reject a future unsupported path explicitly.\n                if not isinstance(reference.image, Image.Image):\n                    raise TypeError("Turbo API requires decoded PIL image references")\n                image = reference.image.convert("RGB")\n                width, height = reference_match_size(\n                    *image.size, state.get("width"), state.get("height"), components.canvas_multiple,\n                )\n                if image.size != (width, height):\n                    image = components.image_processor.resize(image, height=height, width=width)\n                normalized[index] = MiniMaxH3ImageReference(image=image)\n            state.set("normalized_references", normalized)\n            return components, state\n\n    pipe._h3_original_before_encode = original\n    pipe._h3_turbo_before_encode = MatchReferenceSetup()\n    pipe.blocks.sub_blocks["before_encode"] = pipe._h3_turbo_before_encode\n\n\ndef select_reference_setup(pipe, acceleration):\n    """Select an already constructed setup block inside the exclusive GPU lease."""\n    turbo = acceleration == TURBO.key\n    selected = getattr(pipe, "_h3_turbo_before_encode" if turbo else "_h3_original_before_encode", None)\n    if turbo and selected is None:\n        raise RuntimeError("Turbo reference setup was not prepared during model loading")\n    if selected is not None:\n        pipe.blocks.sub_blocks["before_encode"] = selected\n\n\ndef select_loaded_adapters(pipe, settings, acceleration):\n    """Toggle only preloaded, unfused adapters; preserve independent style LoRA.\n\n    This never downloads, loads, fuses, deletes, casts, or changes base weights.\n    Call only while the model execution owns the GPU lease, including restore.\n    """\n    transformer = getattr(pipe, "transformer_ref", None)\n    if transformer is None:\n        raise RuntimeError("Cannot select adapters without transformer_ref")\n    if not settings.turbo and settings.style == "none":\n        if acceleration != "none":\n            raise RuntimeError("Requested acceleration adapter is not loaded")\n        return\n    if getattr(transformer, "_h3_adapter_settings", None) != settings:\n        raise RuntimeError("Loaded adapter identity does not match deployment configuration")\n    names, scales = [], []\n    if acceleration != "none":\n        if acceleration != settings.acceleration:\n            raise RuntimeError("Requested acceleration adapter is not loaded")\n        for name, expected in (("scheduler", 12.0), ("audio_scheduler", 3.0)):\n            if float(getattr(pipe, name).config.shift) != expected:\n                raise RuntimeError(f"Ref2VA Turbo requires {name} shift={expected}")\n        names.append("h3_acceleration")\n        scales.append(1.0)\n    if settings.style != "none":\n        names.append("h3_style")\n        scales.append(settings.style_scale)\n    if names:\n        transformer.set_adapters(names, weights=scales)\n        transformer.enable_lora()\n    else:\n        transformer.disable_lora()\n    # PEFT activation can turn LoRA factors trainable again. Preserve the\n    # inference-only lifetime after enabling, disabling, and restoring.\n    transformer.requires_grad_(False)\n    transformer.eval()\n    if hasattr(transformer, "_reset_stateful_cache"):\n        transformer._reset_stateful_cache()\n\n\n_VERIFIED_FILES = set()\n\n\ndef _download_verified(spec):\n    from huggingface_hub import hf_hub_download\n    progress = current_progress()\n    category = \'acceleration\' if spec == TURBO else \'style\'\n    cached = None\n    try:\n        from huggingface_hub import try_to_load_from_cache\n        cached = try_to_load_from_cache(spec.repository, spec.filename, revision=spec.revision)\n        cached = isinstance(cached, str) and Path(cached).is_file()\n    except (ImportError, OSError, ValueError):\n        cached = None\n    if progress:\n        progress.phase(category, \'verifying\' if cached else \'downloading\',\n                       \'命中缓存，检查文件完整性\' if cached else \'下载所选 LoRA 文件\', total=spec.size, cache_hit=cached)\n    path = Path(hf_hub_download(spec.repository, spec.filename, revision=spec.revision))\n    stat = path.stat()\n    if progress:\n        progress.phase(category, \'verifying\', \'校验文件大小与 SHA-256\', total=spec.size, cache_hit=cached)\n    signature = (str(path.resolve()), stat.st_size, stat.st_mtime_ns, spec.sha256)\n    if signature not in _VERIFIED_FILES:\n        if stat.st_size != spec.size:\n            raise RuntimeError(f"Unexpected adapter file size for {spec.key}")\n        digest = hashlib.sha256()\n        with path.open("rb") as stream:\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                digest.update(chunk)\n        if digest.hexdigest() != spec.sha256:\n            raise RuntimeError(f"SHA-256 mismatch for {spec.key}")\n        _VERIFIED_FILES.add(signature)\n    return path\n\n\ndef _validate_converted(state_dict, transformer, spec):\n    """Verify every converted tensor has a paired, dimension-compatible target."""\n    pairs = {}\n    pattern = re.compile(r"^transformer\\.(.+)\\.lora_([AB])\\.weight$")\n    for key, tensor in state_dict.items():\n        match = pattern.fullmatch(key)\n        if not match or len(tensor.shape) != 2:\n            raise ValueError(f"Unsupported converted LoRA tensor: {key}")\n        name, side = match.groups()\n        if spec == REALISM and not re.fullmatch(r"(?:transformer_blocks|token_refiner\\.refiner_blocks)\\.\\d+\\.attn\\.(to_[qkv]|to_out\\.0)", name):\n            raise ValueError(f"Realism adapter must target only shared attention projections: {name}")\n        pairs.setdefault(name, {})[side] = tensor\n    if not pairs:\n        raise ValueError("Converted adapter has no target tensors")\n    for name, pair in pairs.items():\n        if set(pair) != {"A", "B"}:\n            raise ValueError(f"Unpaired LoRA target: {name}")\n        module = transformer.get_submodule(name)\n        base = module.get_base_layer() if hasattr(module, "get_base_layer") else module\n        a, b = pair["A"], pair["B"]\n        if a.shape[0] != b.shape[1] or tuple(base.weight.shape) != (b.shape[0], a.shape[1]):\n            raise ValueError(f"LoRA shape mismatch for {name}; pruned/other-model adapters are unsupported")\n    return pairs\n\n\n@tracked_adapter_application\ndef apply_adapters(pipe, settings):\n    """Load reviewed adapters into transformer_ref without fusing base weights.\n\n    Both named adapters can be composed, but that combination still needs GPU\n    quality/latency acceptance. Only LoRA factors are cast; keep-FP32 base layers\n    are never changed. Cache state is cleared when supported.\n    """\n    specs = []\n    if settings.turbo:\n        specs.append((TURBO, "h3_acceleration", 1.0))\n    if settings.style != "none":\n        specs.append((REALISM, "h3_style", settings.style_scale))\n    if not specs:\n        return\n    import importlib.util\n    if importlib.util.find_spec("peft") is None:\n        raise RuntimeError("LoRA is enabled but PEFT is missing; install requirements-lora.txt using the existing torch constraints")\n    import torch\n    from safetensors import safe_open\n    transformer = getattr(pipe, "transformer_ref", None)\n    if transformer is None or getattr(pipe, "transformer", None) is not None:\n        raise RuntimeError("Adapters require the ref2va-only pipeline with transformer_ref loaded")\n    if getattr(transformer, "_h3_adapter_settings", None) is not None:\n        if transformer._h3_adapter_settings != settings:\n            raise RuntimeError("Changing adapters requires a freshly loaded transformer_ref")\n        return\n    if settings.turbo:\n        for name, expected in (("scheduler", 12.0), ("audio_scheduler", 3.0)):\n            if float(getattr(pipe, name).config.shift) != expected:\n                raise RuntimeError(f"Ref2VA Turbo requires {name} shift={expected}")\n    base_dtypes = {id(p): p.dtype for p in transformer.parameters()}\n    names, scales = [], []\n    for spec, adapter_name, scale in specs:\n        category = "acceleration" if spec == TURBO else "style"\n        with adapter_progress(category, spec) as progress:\n            path = _download_verified(spec)\n            if progress:\n                progress.phase(category, "loading", "加载并校验 LoRA 张量；不改变主干精度")\n            with safe_open(path, framework="pt", device="cpu") as handle:\n                keys = list(handle.keys())\n                if len(keys) != spec.source_tensors:\n                    raise ValueError(f"Unexpected tensor count for {spec.key}")\n                if any(handle.get_slice(key).get_dtype() != "BF16" for key in keys):\n                    raise ValueError(f"The reviewed {spec.key} asset must contain only BF16 factors")\n                if spec == TURBO and (handle.metadata() or {}).get("alpha") != "8":\n                    raise ValueError("Ref2VA Turbo training alpha must be 8")\n            state_dict, metadata = pipe.lora_state_dict(\n                str(path.parent), weight_name=path.name, use_safetensors=True,\n                local_files_only=True, return_lora_metadata=True,\n            )\n            pairs = _validate_converted(state_dict, transformer, spec)\n            # Use the official ref-specific loader so the file metadata\'s alpha/rank\n            # scaling survives conversion. Generic load_lora_weights(dict) would\n            # lose the original file\'s alpha=8 metadata.\n            pipe.load_lora_into_transformer_ref(\n                state_dict, transformer_ref=transformer, prefix="transformer",\n                adapter_name=adapter_name, metadata=metadata, _pipeline=pipe,\n                low_cpu_mem_usage=True,\n            )\n            del state_dict\n            for name in pairs:\n                layer = transformer.get_submodule(name)\n                if adapter_name not in layer.lora_A or adapter_name not in layer.lora_B:\n                    raise RuntimeError(f"Adapter failed to load into {name}")\n                expected_scale = 8.0 / 128 if spec == TURBO else 1.0\n                if not math.isclose(float(layer.scaling[adapter_name]), expected_scale, rel_tol=1e-6):\n                    raise RuntimeError(f"Unexpected training alpha/rank scale for {spec.key} at {name}")\n                dtype = layer.get_base_layer().weight.dtype\n                if dtype not in {torch.bfloat16, torch.float32}:\n                    raise RuntimeError(f"Unexpected base precision at {name}: {dtype}")\n                layer.lora_A[adapter_name].to(dtype=dtype)\n                layer.lora_B[adapter_name].to(dtype=dtype)\n            del pairs\n            names.append(adapter_name)\n            scales.append(scale)\n            if progress:\n                progress.phase(category, "loaded", "LoRA 已加载并校验", total=spec.size)\n    transformer.set_adapters(names, weights=scales)\n    for parameter in transformer.parameters():\n        if id(parameter) in base_dtypes and parameter.dtype != base_dtypes[id(parameter)]:\n            raise RuntimeError("LoRA loading changed the base-model precision")\n    transformer.requires_grad_(False)\n    transformer.eval()\n    if hasattr(transformer, "_reset_stateful_cache"):\n        transformer._reset_stateful_cache()\n    transformer._h3_adapter_settings = settings\n', 'resource_scheduler.py': '"""Bounded CPU child events; the GPU owner and model stages remain serial.\n\nThe optimizer is exact within floating-point tolerance for its declared cost\nmodel, not a claim of global hardware optimality. Admission uses calibrated\nmemory/thread estimates, not OS-enforced peak limits. Missing calibration selects\none lane. Native codec/library pools are not constrained by the Python executor.\n"""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport threading\nimport time\n\nMIB = 1024 ** 2\nRESERVE_MIB = 2048\n\n\n@dataclass(frozen=True)\nclass Allocation:\n    lanes: tuple[tuple[int, ...], ...]\n    predicted_seconds: float\n    reason: str\n    task_memory_mib: float | None = None\n    threads_per_worker: int | None = None\n    profile_metadata: dict | None = None\n\n\ndef optimal_allocation(costs, io_mib, *, cpu_slots, memory_slots, bandwidth_mib_s,\n                       max_workers=4, overhead_seconds=0.002):\n    """Minimize max(compute makespan, shared IO floor)+lane overhead.\n\n    At most 12 children / 4 lanes: branch-and-bound enumerates legal assignments,\n    removing symmetric empty lanes. Inputs are service estimates from calibration.\n    Equal predicted costs favor fewer workers. No nested GPU concurrency is allowed.\n    """\n    if not costs:\n        return Allocation((), 0.0, "no children")\n    if len(costs) > 12 or any(not math.isfinite(x) or x <= 0 for x in costs):\n        raise ValueError("Require 1..12 finite positive child costs")\n    if len(io_mib) != len(costs) or any(not math.isfinite(x) or x < 0 for x in io_mib):\n        raise ValueError("IO demand must match the child costs")\n    if not math.isfinite(bandwidth_mib_s) or bandwidth_mib_s <= 0:\n        raise ValueError("A positive measured bandwidth is required")\n    if overhead_seconds < 0 or not math.isfinite(overhead_seconds):\n        raise ValueError("Invalid dispatch overhead")\n    if any(isinstance(value, bool) or not isinstance(value, int) for value in (cpu_slots, memory_slots, max_workers)):\n        raise ValueError("CPU, memory, and worker lane caps must be integers")\n    cap = min(4, max_workers, cpu_slots, memory_slots, len(costs))\n    if cap < 1:\n        raise MemoryError("No CPU/memory lane is available")\n    order = sorted(range(len(costs)), key=lambda i: costs[i], reverse=True)\n    io_floor = sum(io_mib) / bandwidth_mib_s\n    winner = None\n    for workers in range(1, cap + 1):\n        loads, bins = [0.0] * workers, [[] for _ in range(workers)]\n        # A valid upper bound from longest-processing-time list scheduling.\n        for index in order:\n            lane = min(range(workers), key=lambda k: loads[k])\n            loads[lane] += costs[index]\n            bins[lane].append(index)\n        best_load, best_bins = max(loads), [row[:] for row in bins]\n        lower_bound = max(sum(costs) / workers, max(costs))\n        loads, bins = [0.0] * workers, [[] for _ in range(workers)]\n\n        def visit(position):\n            nonlocal best_load, best_bins\n            if best_load <= lower_bound + 1e-10:\n                return\n            if position == len(order):\n                best_load, best_bins = max(loads), [row[:] for row in bins]\n                return\n            index = order[position]\n            seen = set()\n            for lane in range(workers):\n                previous = loads[lane]\n                if previous in seen or previous + costs[index] >= best_load - 1e-10:\n                    continue\n                seen.add(previous)\n                loads[lane] += costs[index]\n                bins[lane].append(index)\n                visit(position + 1)\n                bins[lane].pop()\n                loads[lane] = previous\n\n        visit(0)\n        predicted = max(best_load, io_floor) + overhead_seconds * workers\n        candidate = Allocation(tuple(tuple(row) for row in best_bins), predicted,\n                               "minimum predicted makespan under calibrated CPU/IO/memory limits")\n        if winner is None or predicted < winner.predicted_seconds - 1e-10:\n            winner = candidate\n    return winner\n\n\ndef _cpu_slots():\n    """Affinity-visible logical cores; not a cgroup CPU quota or thread limit."""\n    import psutil\n    try:\n        return max(1, len(psutil.Process().cpu_affinity()))\n    except (AttributeError, OSError, psutil.Error):\n        return max(1, os.cpu_count() or 1)\n\n\ndef _phase_table(profile, key, phase):\n    table = profile.get(key, {})\n    if not isinstance(table, dict) or not isinstance(table.get(phase, {}), dict):\n        raise ValueError(f"Calibration {key} must map phase names to modality estimates")\n    return table.get(phase, {})\n\n\nclass MediaController:\n    """Negative feedback applies between batches, never inside a GPU stage.\n\n    A slow/low-memory observation reduces the next concurrency ceiling. Recovery\n    increases it by one only after three healthy observations. A profile is an\n    explicit local calibration artifact, never silently invented from model IO.\n    """\n    def __init__(self):\n        self.ceiling = 4\n        self.healthy = 0\n        self.lock = threading.Lock()\n\n    def plan(self, items, phase):\n        import psutil\n        requested = os.environ.get("H3_MEDIA_WORKERS", "auto")\n        if requested not in {"auto", "1", "2", "3", "4"}:\n            raise ValueError("H3_MEDIA_WORKERS must be auto or 1..4")\n        cpu = _cpu_slots()\n        free = psutil.virtual_memory().available\n        # This coarse guard protects the uncalibrated serial path. It is not a\n        # promise that a codec\'s true peak is bounded by one GiB.\n        if free < (RESERVE_MIB + 1024) * MIB:\n            raise MemoryError("Less than 3 GiB available RAM for reference preparation")\n        with self.lock:\n            ceiling = self.ceiling\n        maximum = min(cpu, ceiling)\n        if requested != "auto":\n            maximum = min(maximum, int(requested))\n        profile_path = os.environ.get("H3_MEDIA_PROFILE", "")\n        if not profile_path:\n            # A manual ceiling does not substitute for measured child costs.\n            return Allocation((tuple(range(len(items))),), 0.0, "uncalibrated: serial; native threads/peak memory not bounded"), 1\n        profile_bytes = Path(profile_path).read_bytes()\n        profile = json.loads(profile_bytes)\n        if not isinstance(profile, dict) or profile.get("schema_version") != 1:\n            raise ValueError("Unsupported media calibration schema")\n        if profile.get("cpu_slots") != cpu:\n            raise ValueError("Media calibration CPU affinity differs from this machine")\n        metadata = {"sha256": hashlib.sha256(profile_bytes).hexdigest(), "schema_version": 1,\n                    "affinity_cpu_slots": cpu, "phase": phase}\n        rates = _phase_table(profile, "seconds_per_mib", phase)\n        if any(item["kind"] not in rates for item in items):\n            return Allocation((tuple(range(len(items))),), 0.0, "missing modality calibration: serial",\n                              profile_metadata=metadata), 1\n        threads = profile.get("threads_per_worker")\n        memory_rates = _phase_table(profile, "task_memory_mib", phase)\n        if threads is None or any(item["kind"] not in memory_rates for item in items):\n            return Allocation((tuple(range(len(items))),), 0.0, "missing thread/memory calibration: serial",\n                              profile_metadata=metadata), 1\n        if isinstance(threads, bool) or not isinstance(threads, int) or threads < 1:\n            raise ValueError("threads_per_worker must be a positive measured integer")\n        demands = [float(memory_rates[item["kind"]]) for item in items]\n        if any(not math.isfinite(value) or value <= 0 for value in demands):\n            raise ValueError("task_memory_mib estimates must be finite and positive")\n        if not items:\n            return Allocation((), 0.0, "no children"), 0\n        task_memory = max(demands)\n        # Prepared objects survive until encoding. Conservatively reserve each\n        # child\'s estimated peak again as its retained-result budget; then admit\n        # temporary concurrent working sets. Estimates need matching input bounds.\n        retained = sum(demands) if phase == "prepare" else 0\n        memory_slots = max(0, math.floor((free / MIB - RESERVE_MIB - retained) / task_memory))\n        cpu_budget = cpu // threads\n        maximum = min(maximum, memory_slots, cpu_budget)\n        if maximum < 1:\n            raise MemoryError("Calibrated reference working sets exceed the available CPU/memory budget")\n        sizes = [max(Path(item["path"]).stat().st_size / MIB, 0.01) for item in items]\n        costs = [float(rates[item["kind"]]) * size for item, size in zip(items, sizes)]\n        allocation = optimal_allocation(costs, sizes, cpu_slots=cpu_budget, memory_slots=memory_slots,\n                                        bandwidth_mib_s=float(profile.get("bandwidth_mib_s", 0)),\n                                        max_workers=maximum,\n                                        overhead_seconds=float(profile.get("overhead_seconds", 0.002)))\n        metadata.update(cpu_budget_slots=cpu_budget, memory_budget_slots=memory_slots,\n                        estimated_child_seconds=costs, estimated_child_memory_mib=demands,\n                        bandwidth_mib_s=float(profile["bandwidth_mib_s"]),\n                        overhead_seconds=float(profile.get("overhead_seconds", 0.002)),\n                        retained_result_reserve_mib=retained)\n        return Allocation(allocation.lanes, allocation.predicted_seconds, allocation.reason,\n                          task_memory, threads, metadata), memory_slots\n\n    def observe(self, allocation, elapsed, available_slots, succeeded):\n        with self.lock:\n            slow = allocation.predicted_seconds > 0 and elapsed > allocation.predicted_seconds * 1.25\n            if not succeeded or slow or available_slots < len(allocation.lanes):\n                self.ceiling = max(1, min(self.ceiling - 1, len(allocation.lanes)))\n                self.healthy = 0\n            else:\n                self.healthy += 1\n                if self.healthy >= 3:\n                    self.ceiling = min(4, self.ceiling + 1)\n                    self.healthy = 0\n\n\nCONTROLLER = MediaController()\n\n\ndef _children(items, function, phase, metrics):\n    if not items:\n        return []\n    allocation, slots = CONTROLLER.plan(items, phase)\n    attribution = _input_attribution(items)\n    attribution["profile"] = allocation.profile_metadata\n    results = [None] * len(items)\n    stop = threading.Event()\n    started, succeeded = time.perf_counter(), False\n    _schedule_log("H3_SCHEDULE", {"phase": phase, "workers": len(allocation.lanes),\n                  "predicted_seconds": allocation.predicted_seconds, "reason": allocation.reason,\n                  "estimated_task_memory_mib": allocation.task_memory_mib,\n                  "estimated_threads_per_worker": allocation.threads_per_worker,\n                  **attribution}, metrics)\n\n    def lane(indices):\n        for index in indices:\n            if stop.is_set():\n                return\n            try:\n                if metrics is None:\n                    results[index] = function(items[index])\n                else:\n                    with metrics.stage(f"child.{phase}.{index}"):\n                        results[index] = function(items[index])\n            except BaseException:\n                stop.set()\n                raise\n\n    try:\n        if len(allocation.lanes) == 1:\n            lane(allocation.lanes[0])\n        else:\n            # Structured lifetime: cancel unstarted work, join started work before\n            # returning or raising. No orphan children overlap the next GPU phase.\n            with ThreadPoolExecutor(max_workers=len(allocation.lanes), thread_name_prefix="h3-media") as pool:\n                futures = [pool.submit(lane, indices) for indices in allocation.lanes]\n                try:\n                    for future in as_completed(futures):\n                        future.result()\n                except BaseException:\n                    stop.set()\n                    for future in futures:\n                        future.cancel()\n                    raise\n        succeeded = True\n        return results\n    finally:\n        elapsed = time.perf_counter() - started\n        if not succeeded:\n            results.clear()  # Release completed child objects before re-raising.\n        try:\n            import psutil\n            free_mib = psutil.virtual_memory().available / MIB\n            estimate = allocation.task_memory_mib or 1024\n            remaining = max(0, math.floor((free_mib - RESERVE_MIB) / estimate))\n            CONTROLLER.observe(allocation, elapsed, remaining, succeeded)\n            _schedule_log("H3_SCHEDULE_FEEDBACK", {"phase": phase, "wall_seconds": elapsed,\n                          "succeeded": succeeded, "next_ceiling": CONTROLLER.ceiling,\n                          "workers": len(allocation.lanes), **attribution}, metrics)\n        except Exception as feedback_error:\n            # A missing/denied counter or telemetry failure must never mask the\n            # original child exception or discard successfully prepared media.\n            _schedule_log("H3_SCHEDULE_FEEDBACK", {"phase": phase, "feedback": "unavailable",\n                          "error_type": type(feedback_error).__name__, "wall_seconds": elapsed,\n                          "succeeded": succeeded, **attribution}, metrics)\n\n\ndef _input_attribution(items):\n    children = []\n    for index, item in enumerate(items):\n        kind = item.get("kind") if isinstance(item, dict) else None\n        size = None\n        if isinstance(item, dict) and "path" in item:\n            try:\n                size = Path(item["path"]).stat().st_size\n            except (OSError, TypeError, ValueError):\n                pass\n        children.append({"index": index, "kind": kind, "input_bytes": size})\n    complete = all(child["input_bytes"] is not None for child in children)\n    return {"child_count": len(children), "children": children,\n            "input_bytes": sum(child["input_bytes"] for child in children) if complete else None}\n\n\ndef _schedule_log(prefix, record, metrics=None):\n    try:\n        print("\\n" + prefix + " " + json.dumps(record), flush=True)\n    except (OSError, ValueError):\n        pass\n    try:\n        recorder = getattr(metrics, "record_schedule", None)\n        if recorder is not None:\n            recorder({"event": prefix, **record})\n    except Exception:\n        pass  # Attribution must not alter the child outcome.\n\n\ndef prepare_references(references, classes, probe, normalize, metrics=None):\n    """Probe barrier -> aggregate validation -> bounded decode barrier, in order."""\n    normalized, refs = [], []\n    lock = threading.Lock()\n    try:\n        durations = _children(references, lambda item: probe(Path(item["path"]), item["kind"]),\n                              "probe", metrics)\n        for kind in ("video", "audio"):\n            if sum(duration for item, duration in zip(references, durations) if item["kind"] == kind) > 15:\n                raise ValueError("Total reference duration per modality must be <=15 seconds")\n\n        def prepare(item):\n            path = Path(item["path"])\n            if item["kind"] == "video":\n                # Known target is registered before conversion, so failed ffmpeg\n                # outputs are cleaned too. normalize uses this same target.\n                target = path.with_name(path.stem + "_24fps.mp4")\n                with lock:\n                    normalized.append(target)\n                path = normalize(path)\n                with lock:\n                    if path not in normalized:\n                        normalized.append(path)\n            return classes[item["kind"]].from_file(str(path))\n\n        refs = _children(references, prepare, "prepare", metrics)\n        return refs, normalized\n    except BaseException:\n        refs.clear()\n        for path in normalized:\n            path.unlink(missing_ok=True)\n        raise\n', 'performance_model.py': '"""CPU-only optimistic physical lower bounds, not a GPU benchmark or simulator.\n\nResource floor = max_r(sum_i(work[i,r]) / peak[r]).\nNode floor = max_r(work[i,r] / peak[r]).\nCritical-path floor = longest dependency path using node floors.\nOverall floor = max(resource floor, critical-path floor).\n\nMissing workload/capacity is reported, never invented. No PyTorch, CUDA,\ndownloads, or live runtime access is used. Numeric inputs use explicit work\nunits; capacities use the same unit per second. Decimal GB means 1e9 bytes.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import deque\nimport json\nimport math\nfrom pathlib import Path\n\n\nDATASHEET = "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\nA100_40_SOURCE = "https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/"\nBACKGROUND = "https://docs.nvidia.com/deeplearning/performance/dl-performance-gpu-background/"\n\n\ndef _number(value, path, *, positive=False):\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\n        raise ValueError(f"{path} must be a finite number")\n    if value < 0 or (positive and value <= 0):\n        raise ValueError(f"{path} must be {\'positive\' if positive else \'non-negative\'}")\n    return float(value)\n\n\ndef hardware_template(profile="a100-sxm-40gb"):\n    """Return capacities only. The caller must supply a profiled workload DAG."""\n    if profile not in {"a100-sxm-40gb", "a100-sxm-80gb"}:\n        raise ValueError("Supported profiles: a100-sxm-40gb, a100-sxm-80gb")\n    hbm = 1555e9 if profile == "a100-sxm-40gb" else 2039e9\n    def resource(unit, capacity, note, source=None):\n        return {"work_unit": unit, "capacity_per_second": capacity, "note": note, "source": source}\n    return {\n        "schema_version": 1,\n        "hardware": profile,\n        "coverage": "partial",\n        "observed_seconds": None,\n        "resources": {\n            "gpu_bf16_dense_flop": resource("FLOP", 312e12,\n                "BF16 dense Tensor Core work only; 624 TFLOP/s sparsity peak does not apply. One multiply-add counts as 2 FLOPs.", DATASHEET),\n            "gpu_fp32_flop": resource("FLOP", None,\n                "Keep FP32-layer work separate. Determine actual instruction path first; do not divide it by the BF16 peak. FP32 CUDA-core peak is 19.5e12 FLOP/s, TF32 is a different path.", DATASHEET),\n            "gpu_hbm_bytes": resource("byte", hbm,\n                "Total HBM traffic, not model-file size or PCIe traffic. Decimal GB/s; theoretical peak, not measured bandwidth.", A100_40_SOURCE if profile.endswith("40gb") else DATASHEET),\n            "host_memory_bytes": resource("byte", None, "Host RAM topology and bandwidth are unknown."),\n            "pcie_host_to_gpu_bytes": resource("byte", None, "Actual host-to-GPU topology/link width unknown. 64 GB/s bidirectional must not be used as one-way capacity."),\n            "pcie_gpu_to_host_bytes": resource("byte", None, "Actual GPU-to-host topology/link width unknown. Keep transfer directions explicit."),\n            "storage_read_bytes": resource("byte", None, "Backing-storage read bandwidth is unknown; distinguish page-cache hits from physical reads."),\n            "network_bytes": resource("byte", None, "External network path and bandwidth are unknown."),\n            "cpu_work": resource("operation", None, "Choose a measured CPU operation definition and matching peak before supplying work."),\n        },\n        "stages": [],\n        "notes": [\n            "This template contains no MiniMax-H3 FLOP or byte estimates and predicts no generation time.",\n            "Add stage objects with id, depends_on, and work entries of {amount, unit}; null amount means unknown.",\n            "Use distinct stages for non-overlappable work and preserve true dependencies. A fused stage assumes ideal overlap among its resources.",\n            "Resource capacities describe shared resource pools. Do not duplicate one physical pool under different names to invent extra capacity.",\n            "Omitted resource work is assumed zero for the declared model. coverage=complete is an explicit assertion by the profiler, not automatic discovery.",\n            "Only use this full-GPU profile when the assigned GPU is the matching full A100, not a MIG slice.",\n            "GPU matrix, memory, launch, decode, I/O and transfer work still need profiling. Available RAM and VRAM capacity are feasibility constraints, not bandwidth.",\n        ],\n        "method_source": BACKGROUND,\n    }\n\n\ndef compute_performance_bounds(document):\n    if not isinstance(document, dict):\n        raise ValueError("Input must be a JSON object")\n    coverage = document.get("coverage", "partial")\n    if coverage not in {"partial", "complete"}:\n        raise ValueError("coverage must be partial or complete")\n    resources = document.get("resources")\n    stages = document.get("stages")\n    if not isinstance(resources, dict) or not resources:\n        raise ValueError("resources must be a nonempty object")\n    if not isinstance(stages, list):\n        raise ValueError("stages must be a list")\n    capacities, units = {}, {}\n    for name, definition in resources.items():\n        if not isinstance(name, str) or not name or not isinstance(definition, dict):\n            raise ValueError("Every resource needs a nonempty name and an object definition")\n        unit = definition.get("work_unit")\n        if not isinstance(unit, str) or not unit.strip():\n            raise ValueError(f"resources.{name}.work_unit must be explicit")\n        units[name] = unit\n        capacity = definition.get("capacity_per_second")\n        capacities[name] = None if capacity is None else _number(capacity, f"resources.{name}.capacity_per_second", positive=True)\n\n    nodes, dependencies, weights, totals, missing = {}, {}, {}, {}, []\n    known_usable_items = 0\n    for index, stage in enumerate(stages):\n        if not isinstance(stage, dict) or not isinstance(stage.get("id"), str) or not stage["id"]:\n            raise ValueError(f"stages[{index}] requires a nonempty string id")\n        name = stage["id"]\n        if name in nodes:\n            raise ValueError(f"Duplicate stage id: {name}")\n        nodes[name] = stage\n        deps = stage.get("depends_on", [])\n        if not isinstance(deps, list) or any(not isinstance(dep, str) for dep in deps) or len(set(deps)) != len(deps):\n            raise ValueError(f"stages.{name}.depends_on must be a list of unique stage ids")\n        dependencies[name] = deps\n        work = stage.get("work")\n        if not isinstance(work, dict):\n            raise ValueError(f"stages.{name}.work must be an object")\n        node_times = []\n        if not work:\n            missing.append({"path": f"stages.{name}.work", "reason": "no workload supplied"})\n        for resource, item in work.items():\n            path = f"stages.{name}.work.{resource}"\n            if resource not in resources:\n                raise ValueError(f"{path} names an undefined resource")\n            if not isinstance(item, dict) or item.get("unit") != units[resource]:\n                raise ValueError(f"{path}.unit must equal {units[resource]!r}; implicit conversion is not performed")\n            if "amount" not in item:\n                raise ValueError(f"{path}.amount must be a number or explicit null")\n            total = totals.setdefault(resource, {"known_amount": 0.0, "unit": units[resource], "has_unknown_work": False})\n            if item["amount"] is None:\n                total["has_unknown_work"] = True\n                missing.append({"path": path + ".amount", "reason": "unknown workload"})\n                continue\n            amount = _number(item["amount"], path + ".amount")\n            total["known_amount"] += amount\n            if capacities[resource] is None:\n                if amount > 0:\n                    missing.append({"path": "resources." + resource + ".capacity_per_second", "reason": "unknown capacity for nonzero work"})\n                else:\n                    node_times.append(0.0)\n                    known_usable_items += 1\n            else:\n                node_times.append(amount / capacities[resource])\n                known_usable_items += 1\n        weights[name] = max(node_times) if node_times else None\n\n    if not stages:\n        missing.append({"path": "stages", "reason": "no workload DAG supplied"})\n    followers = {name: [] for name in nodes}\n    indegree = {name: len(deps) for name, deps in dependencies.items()}\n    for name, deps in dependencies.items():\n        for dep in deps:\n            if dep not in nodes:\n                raise ValueError(f"Stage {name!r} depends on unknown stage {dep!r}")\n            followers[dep].append(name)\n    ready = deque(name for name in nodes if indegree[name] == 0)\n    ends, paths, order = {}, {}, []\n    while ready:\n        name = ready.popleft()\n        order.append(name)\n        predecessor = max(dependencies[name], key=lambda dep: ends[dep], default=None)\n        ends[name] = (ends[predecessor] if predecessor is not None else 0.0) + (weights[name] or 0.0)\n        paths[name] = (paths[predecessor] if predecessor is not None else []) + [name]\n        for successor in followers[name]:\n            indegree[successor] -= 1\n            if indegree[successor] == 0:\n                ready.append(successor)\n    if len(order) != len(nodes):\n        raise ValueError("The stage graph contains a dependency cycle")\n\n    resource_bounds = {\n        name: total["known_amount"] / capacities[name]\n        for name, total in totals.items() if capacities[name] is not None\n    }\n    usable = known_usable_items > 0\n    ideal = max(resource_bounds.values(), default=0.0) if usable else None\n    terminal = max(ends, key=ends.get, default=None)\n    critical = ends[terminal] if terminal is not None and usable else None\n    lower_bound = max(ideal, critical) if ideal is not None and critical is not None else None\n    # Deduplicate shared missing capacity entries while keeping deterministic order.\n    missing = [dict(path=path, reason=reason) for path, reason in dict.fromkeys((entry["path"], entry["reason"]) for entry in missing)]\n    complete = coverage == "complete" and not missing and bool(stages)\n    comparison = None\n    observed = document.get("observed_seconds")\n    if observed is not None:\n        observed = _number(observed, "observed_seconds", positive=True)\n        comparison = {\n            "observed_seconds": observed,\n            "scope": "declared_complete_workload" if complete else "partial_workload_only",\n            "time_over_lower_bound": observed / lower_bound if lower_bound is not None and lower_bound > 0 else None,\n            "lower_bound_fraction_of_observed": lower_bound / observed if lower_bound is not None else None,\n            "inconsistent_observation": lower_bound is not None and observed + 1e-12 < lower_bound,\n            "overall_efficiency": None,\n            "note": "Ratios compare time with an optimistic bound, not measured device utilization or proven achievable efficiency. Partial workloads cannot establish whole-task efficiency.",\n        }\n    return {\n        "schema_version": 1,\n        "coverage": "declared_complete" if complete else "partial",\n        "ideal_full_parallel_lower_bound_seconds": ideal,\n        "critical_path_lower_bound_seconds": critical,\n        "overall_lower_bound_seconds": lower_bound,\n        "resource_lower_bounds_seconds": resource_bounds,\n        "stage_lower_bounds_seconds": weights,\n        "critical_path": paths[terminal] if terminal is not None and usable else [],\n        "total_work_by_resource": totals,\n        "missing": missing,\n        "unavailable_capacities": [name for name, value in capacities.items() if value is None],\n        "comparison": comparison,\n        "interpretation": "Optimistic necessary lower bound under supplied resource pools and DAG, not an achievable runtime prediction. Independent resources and work inside each stage are assumed to overlap perfectly; dependencies cannot overlap.",\n    }\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("input", nargs="?", type=Path, help="UTF-8 JSON workload and resource document")\n    parser.add_argument("--template", choices=("a100-sxm-40gb", "a100-sxm-80gb"), help="Emit capacities only; no fake workload")\n    parser.add_argument("--output", type=Path, help="Write UTF-8 JSON instead of stdout")\n    args = parser.parse_args(argv)\n    if bool(args.input) == bool(args.template):\n        parser.error("Choose exactly one input JSON or --template")\n    try:\n        result = hardware_template(args.template) if args.template else compute_performance_bounds(json.loads(args.input.read_text(encoding="utf-8-sig")))\n    except (ValueError, OSError) as exc:\n        parser.error(str(exc))\n    rendered = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\\n"\n    if args.output:\n        args.output.write_text(rendered, encoding="utf-8")\n    else:\n        print(rendered, end="")\n\n\nif __name__ == "__main__":\n    main()\n', '性能参照模板.json': '{\n  "schema_version": 1,\n  "hardware": "a100-sxm-40gb",\n  "coverage": "partial",\n  "observed_seconds": null,\n  "resources": {\n    "gpu_bf16_dense_flop": {\n      "work_unit": "FLOP",\n      "capacity_per_second": 312000000000000.0,\n      "note": "BF16 dense Tensor Core work only; 624 TFLOP/s sparsity peak does not apply. One multiply-add counts as 2 FLOPs.",\n      "source": "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\n    },\n    "gpu_fp32_flop": {\n      "work_unit": "FLOP",\n      "capacity_per_second": null,\n      "note": "Keep FP32-layer work separate. Determine actual instruction path first; do not divide it by the BF16 peak. FP32 CUDA-core peak is 19.5e12 FLOP/s, TF32 is a different path.",\n      "source": "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\n    },\n    "gpu_hbm_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": 1555000000000.0,\n      "note": "Total HBM traffic, not model-file size or PCIe traffic. Decimal GB/s; theoretical peak, not measured bandwidth.",\n      "source": "https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/"\n    },\n    "host_memory_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": null,\n      "note": "Host RAM topology and bandwidth are unknown.",\n      "source": null\n    },\n    "pcie_host_to_gpu_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": null,\n      "note": "Actual host-to-GPU topology/link width unknown. 64 GB/s bidirectional must not be used as one-way capacity.",\n      "source": null\n    },\n    "pcie_gpu_to_host_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": null,\n      "note": "Actual GPU-to-host topology/link width unknown. Keep transfer directions explicit.",\n      "source": null\n    },\n    "storage_read_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": null,\n      "note": "Backing-storage read bandwidth is unknown; distinguish page-cache hits from physical reads.",\n      "source": null\n    },\n    "network_bytes": {\n      "work_unit": "byte",\n      "capacity_per_second": null,\n      "note": "External network path and bandwidth are unknown.",\n      "source": null\n    },\n    "cpu_work": {\n      "work_unit": "operation",\n      "capacity_per_second": null,\n      "note": "Choose a measured CPU operation definition and matching peak before supplying work.",\n      "source": null\n    }\n  },\n  "stages": [],\n  "notes": [\n    "This template contains no MiniMax-H3 FLOP or byte estimates and predicts no generation time.",\n    "Add stage objects with id, depends_on, and work entries of {amount, unit}; null amount means unknown.",\n    "Use distinct stages for non-overlappable work and preserve true dependencies. A fused stage assumes ideal overlap among its resources.",\n    "Resource capacities describe shared resource pools. Do not duplicate one physical pool under different names to invent extra capacity.",\n    "Omitted resource work is assumed zero for the declared model. coverage=complete is an explicit assertion by the profiler, not automatic discovery.",\n    "Only use this full-GPU profile when the assigned GPU is the matching full A100, not a MIG slice.",\n    "GPU matrix, memory, launch, decode, I/O and transfer work still need profiling. Available RAM and VRAM capacity are feasibility constraints, not bandwidth."\n  ],\n  "method_source": "https://docs.nvidia.com/deeplearning/performance/dl-performance-gpu-background/"\n}\n', 'requirements.txt': '# Linux NVIDIA A100. Native BF16; no quantization dependencies.\ntorch==2.10.0\ntorchvision==0.25.0\ntorchaudio==2.10.0\ndiffusers==0.43.0\ntransformers==5.7.0\naccelerate>=1.10,<2\nsafetensors>=0.6,<1\nhuggingface-hub>=1.0,<2\nnumpy>=1.26,<3\nPillow>=11,<13\neinops>=0.8,<1\nav>=14,<18\nsoundfile>=0.13,<1\nimageio>=2.37,<3\nimageio-ffmpeg>=0.6,<1\nfastapi>=0.115,<1\nuvicorn>=0.34,<1\npython-multipart>=0.0.20,<1\npsutil>=6,<8\nthreadpoolctl>=3,<4\nrequests>=2.32,<3\n', 'requirements-colab.txt': '# Colab: preserve its preinstalled torch / torchvision / torchaudio stack.\n# Notebook adds exact-version constraints for installed members of that stack.\ndiffusers==0.43.0\ntransformers==5.7.0\naccelerate>=1.10,<2\nsafetensors>=0.6,<1\nhuggingface-hub>=1.0,<2\nnumpy>=1.26,<3\nPillow>=11,<13\neinops>=0.8,<1\nav>=14,<18\nsoundfile>=0.13,<1\nimageio>=2.37,<3\nimageio-ffmpeg>=0.6,<1\nfastapi>=0.115,<1\nuvicorn>=0.34,<1\npython-multipart>=0.0.20,<1\n# The notebook host uses psutil while pip runs: retain its preinstalled version.\n# All process ownership/resource APIs used here are available in 5.9.5.\npsutil>=5.9.5,<8\nthreadpoolctl>=3,<4\nrequests>=2.32,<3\n', 'requirements-lora.txt': '# Optional: install only when an explicit acceleration/style adapter is selected.\n# Colab must apply the same exact torch-stack constraints as the base install.\npeft==0.21.2\n', 'run.py': '"""Start/query/control the API, model container and global event manager."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport time\n\nfrom startup import StartupSession\nfrom control_protocol import rpc_call\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--port", type=int, default=8000)\n    parser.add_argument("--staged", action="store_true", help="Explicit sequential BF16 loading for an A100 with >=70 GiB available host RAM; not the default resident profile")\n    parser.add_argument("--public", action="store_true", help="Prepare a temporary tunnel alongside the model; readiness requires their live join")\n    parser.add_argument("--install-cloudflared", action="store_true", help="With --public, fetch an official SHA-256 checked cloudflared release if needed")\n    parser.add_argument("--attention-backend", choices=["auto", "native", "_native_flash"])\n    parser.add_argument("--denoiser-offload", choices=["stream", "sync"])\n    parser.add_argument("--acceleration-lora", choices=["none", "lightx2v-ref2va-4step"])\n    parser.add_argument("--style-lora", choices=["none", "fal-realism-people"])\n    parser.add_argument("--style-lora-scale", type=float)\n    parser.add_argument("--media-workers", choices=["auto", "1", "2", "3", "4"], help="A ceiling; parallel preparation also requires a local measured profile")\n    parser.add_argument("--media-profile", type=Path, help="Local calibration JSON for bounded CPU child scheduling")\n    parser.add_argument("--validate-config", action="store_true", help="Validate configuration without initializing CUDA, downloading, or starting a server")\n    parser.add_argument("--manager-url", help="Existing private loopback controller; use H3_CONTROL_KEY")\n    parser.add_argument("--action", choices=["status", "events", "drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"], default="status")\n    parser.add_argument("--force", action="store_true", help="Explicitly permit stopping an occupied model container")\n    args = parser.parse_args()\n    if args.manager_url:\n        token = os.environ.get("H3_CONTROL_KEY", "")\n        if len(token) < 32:\n            raise ValueError("Set the existing private H3_CONTROL_KEY for controller access")\n        operation = args.action if args.action in {"status", "events"} else "control"\n        payload = {} if operation != "control" else {"action": args.action, "force": args.force}\n        print(json.dumps(rpc_call(args.manager_url, token, operation, payload), ensure_ascii=False, indent=2))\n        return\n    for argument, name in {\n        "attention_backend": "H3_ATTENTION_BACKEND", "denoiser_offload": "H3_DENOISER_OFFLOAD",\n        "acceleration_lora": "H3_ACCELERATION_LORA", "style_lora": "H3_STYLE_LORA",\n        "style_lora_scale": "H3_STYLE_LORA_SCALE", "media_workers": "H3_MEDIA_WORKERS",\n        "media_profile": "H3_MEDIA_PROFILE",\n    }.items():\n        value = getattr(args, argument)\n        if value is not None:\n            os.environ[name] = str(value)\n    from adapter_config import load_adapter_settings\n    from runtime_options import RuntimeOptions\n    runtime_options, adapters = RuntimeOptions.from_env(), load_adapter_settings()\n    if os.environ.get("H3_MEDIA_WORKERS", "auto") not in {"auto", "1", "2", "3", "4"}:\n        raise ValueError("H3_MEDIA_WORKERS must be auto or 1..4")\n    if args.media_profile and not args.media_profile.is_file():\n        raise FileNotFoundError("The media calibration profile does not exist")\n    if args.validate_config:\n        print(json.dumps({"configuration": "valid", "residency": "staged" if args.staged else "resident",\n                          "runtime": runtime_options.__dict__, "adapters": adapters.as_dict(),\n                          "media_workers": os.environ.get("H3_MEDIA_WORKERS", "auto"),\n                          "gpu_started": False}, ensure_ascii=False, indent=2))\n        return\n    residency = "staged" if args.staged else "resident"\n    new_key = not os.environ.get("H3_API_KEY")\n    session = StartupSession(Path(__file__).resolve().parent, port=args.port, staged=args.staged,\n                             public=args.public, install_cloudflared=args.install_cloudflared)\n    if new_key:\n        print("Temporary API key (shown once; keep private): " + session.api_key, flush=True)\n    print(f"BF16 residency profile: {residency}; lifecycle status: {session.state_path}", flush=True)\n    last_state = None\n\n    def show_state(state):\n        nonlocal last_state\n        summary = {name: state[name] for name in (\n            "instance_id", "phase", "ready", "api_ready", "model_ready", "tunnel_connected", "public_ready", "public_url", "error")}\n        if summary != last_state:\n            print(json.dumps(summary, ensure_ascii=False), flush=True)\n            last_state = summary\n\n    try:\n        session.start()\n        print(f"Manager control: {session.manager_url}; API log: {session.api_log}; model log: {session.server_log}", flush=True)\n        try:\n            state = session.wait_ready(on_update=show_state)\n            print(f"Generation ready: http://127.0.0.1:{args.port}/docs", flush=True)\n            if args.public:\n                print(f"Public generation verified for this instance: {state[\'public_url\']}", flush=True)\n        except (RuntimeError, TimeoutError) as error:\n            print(f"Generation unavailable: {error}. Query/control remain available while the manager is alive.", flush=True)\n        while True:\n            state = session.snapshot()\n            show_state(state)\n            if state.get("manager_alive") is False and state["phase"] != "stopped":\n                raise RuntimeError(state["error"])\n            if state["phase"] == "stopped":\n                break\n            time.sleep(1)\n    finally:\n        session.stop()\n\n\nif __name__ == "__main__":\n    signal.signal(signal.SIGTERM, lambda *_: (_ for _ in ()).throw(KeyboardInterrupt()))\n    try:\n        main()\n    except KeyboardInterrupt:\n        print("Server stopped. Release the cloud GPU instance separately to stop provider billing.")\n', 'startup.py': '"""Managed launcher/client: global_manager owns API, model and tunnel lifecycles."""\nfrom __future__ import annotations\nfrom copy import deepcopy\nimport json\nimport os\nfrom pathlib import Path\nimport secrets\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport uuid\nfrom urllib.request import urlopen\nfrom control_protocol import ControlError, rpc_call\nfrom process_owner import ProcessOwner\n\n\ndef assert_port_available(port):\n    with socket.socket() as probe:\n        probe.bind(("127.0.0.1", int(port)))\n\n\ndef _free_port(excluded):\n    while True:\n        with socket.socket() as probe:\n            probe.bind(("127.0.0.1", 0))\n            port = probe.getsockname()[1]\n        if port not in excluded:\n            return port\n\n\ndef probe_ready(url, instance_id, timeout=3):\n    try:\n        with urlopen(url.rstrip("/") + "/health", timeout=timeout) as response:\n            data = json.loads(response.read(16384))\n        return isinstance(data, dict) and data.get("ready") is True and data.get("instance_id") == instance_id\n    except Exception:\n        return False\n\n\nclass StartupSession:\n    def __init__(self, directory, *, port=8000, staged=False, public=False,\n                 install_cloudflared=False, env=None, startup_timeout=3600,\n                 tunnel_timeout=180, poll_interval=1, probe_timeout=3, clock=None,\n                 start_model=True):\n        self.directory = Path(directory).resolve()\n        self.port, self.staged, self.public = int(port), bool(staged), bool(public)\n        if not 1 <= self.port <= 65535 or min(startup_timeout, tunnel_timeout, poll_interval, probe_timeout) <= 0:\n            raise ValueError("Valid ports and positive timeouts are required")\n        self.install_cloudflared = bool(install_cloudflared)\n        self.model_autostart = bool(start_model)\n        self.startup_timeout, self.tunnel_timeout = startup_timeout, tunnel_timeout\n        self.poll_interval, self.probe_timeout = poll_interval, probe_timeout\n        self._clock = clock or time.monotonic\n        self._environment = dict(os.environ if env is None else env)\n        self.api_key = self._environment.get("H3_API_KEY") or secrets.token_urlsafe(32)\n        self._control_key = self._environment.get("H3_CONTROL_KEY") or secrets.token_urlsafe(32)\n        for token in (self.api_key, self._control_key):\n            if len(token) < 32 or not token.isascii() or any(c.isspace() for c in token):\n                raise ValueError("Tokens require at least 32 non-whitespace ASCII characters")\n        if self.api_key == self._control_key:\n            raise ValueError("Public and private control tokens must be distinct")\n        self.instance_id = uuid.uuid4().hex\n        self.root = Path(self._environment.get("H3_OUTPUT_DIR", self.directory / "h3_outputs")).resolve()\n        self.run_directory = self.directory / "runtime" / self.instance_id\n        self.state_path = self.run_directory / "status.json"\n        self.events_path = self.run_directory / "events.jsonl"\n        self.server_log = self.run_directory / "model.log"\n        self.api_log = self.run_directory / "api.log"\n        self.tunnel_log = self.run_directory / "tunnel.log"\n        self.manager_log = self.run_directory / "manager.log"\n        self.manager = self._output = self._started_at = self._manager_owner = None\n        self._state_lock = threading.Lock()\n        self._last = dict(instance_id=self.instance_id, phase="new", ready=False,\n                          api_ready=False, model_ready=False, public_ready=False,\n                          public_requested=self.public, public_url=None,\n                          public_api_ready=False, tunnel_connected=False,\n                          server_alive=False, model_alive=False, tunnel_alive=False,\n                          manager_alive=False, cleanup_complete=True, shutdown_complete=True, error=None)\n        self.manager_url = None\n\n    @staticmethod\n    def _cleanup_confirmed(state):\n        return (state.get("phase") in {"stopped", "failed"} and state.get("cleanup_complete") is True\n                and state.get("shutdown_complete") is True\n                and all(state.get(name) is False for name in ("server_alive", "model_alive", "tunnel_alive"))\n                and not state.get("process_observation_errors"))\n\n    def _remember(self, state):\n        # RPCs can finish out of order while the notebook polls during stop().\n        # A verified terminal receipt is monotonic for this manager instance.\n        with self._state_lock:\n            if not self._cleanup_confirmed(self._last) or self._cleanup_confirmed(state):\n                self._last = deepcopy(state)\n            return deepcopy(self._last)\n\n    def _read_final_receipt(self):\n        try:\n            state = json.loads(self.state_path.read_text(encoding="utf-8"))\n            if isinstance(state, dict) and state.get("instance_id") == self.instance_id and self._cleanup_confirmed(state):\n                self._remember(state)\n        except (OSError, ValueError):\n            pass\n\n    def _manager_tree_alive(self):\n        if self._manager_owner:\n            return self._manager_owner.observe()["alive"]\n        return self.manager is not None and self.manager.poll() is None\n\n    def _owner_observation(self):\n        if self._manager_owner:\n            return self._manager_owner.observe()\n        alive = self.manager is not None and self.manager.poll() is None\n        return {"alive": alive, "cleanup_complete": not alive, "errors": [],\n                "pids": [self.manager.pid] if alive else [], "identities": [],\n                "scope": "manager_launcher_only", "observed_at": time.time()}\n\n    def _unavailable_snapshot(self, owner_state, *, stopped=False):\n        """Keep the last RPC as history; never mix its readiness with current exit evidence."""\n        if owner_state.get("alive") is False:\n            self._read_final_receipt()\n        with self._state_lock:\n            history = deepcopy(self._last)\n        result = deepcopy(history)\n        errors = list(owner_state.get("errors", []))\n        tree_complete = (owner_state.get("cleanup_complete") is True and not errors\n                         and owner_state.get("scope") != "manager_launcher_only")\n        manager_pid = history.get("manager_pid", getattr(self.manager, "pid", None))\n        manager_alive = (None if errors else manager_pid in owner_state.get("pids", []))\n        if owner_state.get("scope") == "manager_launcher_only":\n            manager_alive = owner_state["alive"]\n        confirmed = self._cleanup_confirmed(history) and owner_state.get("cleanup_complete") is True and not errors\n        children_exited = tree_complete or (self._cleanup_confirmed(history) and manager_alive is False\n                                            and owner_state.get("scope") == "manager_launcher_only")\n        child_alive = False if children_exited else None\n        current_time = time.time()\n        owner_observed_at = owner_state.get("observed_at")\n        manager_observed_at = history.get("observed_at")\n        observation_age = lambda timestamp: (round(max(0, current_time - timestamp), 3)\n                                              if isinstance(timestamp, (int, float)) else None)\n        modules = {name: {"alive": child_alive, "ready": False,\n                          "observation_unknown": child_alive is None}\n                   for name in ("api", "model", "tunnel")}\n        modules["manager"] = {"alive": manager_alive, "ready": False,\n                              "supervisor_alive": False if manager_alive is False else None,\n                              "observation_unknown": manager_alive is None}\n        current_owned = {}\n        for name, previous in history.get("owned_processes", {}).items():\n            current_owned[name] = {"alive": child_alive, "cleanup_complete": children_exited,\n                                   "pids": [] if children_exited else None,\n                                   "identities": deepcopy(previous.get("identities", [])),\n                                   "errors": [] if children_exited else ["manager_observation_unavailable"],\n                                   "observed_at": current_time}\n        result.update(last_manager_observation=history, manager_process_tree=deepcopy(owner_state),\n                      manager_observed_at=manager_observed_at, observed_at=current_time,\n                      owner_observed_at=owner_observed_at,\n                      observation_age_seconds=observation_age(owner_observed_at),\n                      manager_observation_age_seconds=observation_age(manager_observed_at),\n                      manager_launcher_pid=getattr(self.manager, "pid", None), manager_alive=manager_alive,\n                      server_alive=child_alive, model_alive=child_alive, tunnel_alive=child_alive,\n                      ready=False, api_ready=False, model_ready=False, public_ready=False,\n                      public_api_ready=False, tunnel_connected=False, supervisor_alive=modules["manager"]["supervisor_alive"],\n                      modules=modules, owned_processes=current_owned,\n                      cleanup_complete=confirmed, observation_stale=True, manager_observation_stale=True,\n                      phase="stopped" if confirmed else "failed" if stopped or manager_alive is False else "degraded")\n        process_errors = deepcopy(history.get("process_observation_errors", {}))\n        if errors:\n            process_errors["session_owner"] = errors\n        result["process_observation_errors"] = process_errors\n        if isinstance(history.get("downloads"), dict):\n            result["downloads"] = {**deepcopy(history["downloads"]), "model_ready": False,\n                                   "observation_stale": True,\n                                   "lifecycle": "stopped" if children_exited else "unavailable"}\n        if not confirmed:\n            result["error"] = ("Owned process cleanup remains unconfirmed" if errors else\n                               "Manager exited before cleanup was confirmed" if manager_alive is False else\n                               "Manager control observation unavailable")\n        return result\n\n    def start(self, *, background=True):\n        if self._started_at is not None:\n            raise RuntimeError("Session already started; query or stop it")\n        assert_port_available(self.port)\n        manager_port = _free_port({self.port})\n        model_port = _free_port({self.port, manager_port})\n        self.manager_url = f"http://127.0.0.1:{manager_port}"\n        self._environment.update(H3_API_KEY=self.api_key, H3_CONTROL_KEY=self._control_key,\n                                 H3_INSTANCE_ID=self.instance_id, H3_OUTPUT_DIR=str(self.root),\n                                 H3_STARTUP_TIMEOUT=str(self.startup_timeout), H3_TUNNEL_TIMEOUT=str(self.tunnel_timeout))\n        self.run_directory.mkdir(parents=True, exist_ok=False)\n        self._started_at = self._clock()\n        self._last.update(phase="preparing", cleanup_complete=False, shutdown_complete=False)\n        command = [sys.executable, "-u", str(self.directory / "global_manager.py"),\n                   "--port", str(manager_port), "--api-port", str(self.port),\n                   "--model-port", str(model_port), "--directory", str(self.directory),\n                   "--instance-id", self.instance_id, "--root", str(self.root), "--parent-pid", str(os.getpid()),\n                   "--startup-timeout", str(self.startup_timeout), "--probe-timeout", str(self.probe_timeout),\n                   "--poll-interval", str(self.poll_interval), "--tunnel-timeout", str(self.tunnel_timeout)]\n        if self.staged:\n            command.append("--staged")\n        if self.public:\n            command.append("--public")\n        if self.install_cloudflared:\n            command.append("--install-cloudflared")\n        if not self.model_autostart:\n            command.append("--no-model-autostart")\n        try:\n            self._output = self.manager_log.open("w", encoding="utf-8")\n            self.manager = subprocess.Popen(command, cwd=self.directory, env=self._environment,\n                                             stdout=self._output, stderr=subprocess.STDOUT,\n                                             start_new_session=os.name != "nt")\n            self._manager_owner = ProcessOwner(self.manager)\n            deadline = self._clock() + 20\n            while self._clock() < deadline:\n                if not self._manager_tree_alive():\n                    raise RuntimeError("Global manager exited during launch; inspect manager.log")\n                try:\n                    state = rpc_call(self.manager_url, self._control_key, "status", timeout=1)\n                    if state.get("instance_id") != self.instance_id:\n                        raise RuntimeError("Manager identity mismatch")\n                    self._remember(state)\n                    self._manager_owner.observe()\n                    return self\n                except ControlError:\n                    time.sleep(.1)\n            raise TimeoutError("Global manager control endpoint did not start")\n        except BaseException:\n            self.stop()\n            raise\n\n    def snapshot(self):\n        if self._started_at is None:\n            return dict(self._last, elapsed_seconds=0)\n        owner_state = self._owner_observation()\n        if owner_state["alive"]:\n            try:\n                state = rpc_call(self.manager_url, self._control_key, "status", timeout=self.probe_timeout)\n                if not isinstance(state, dict) or state.get("instance_id") != self.instance_id:\n                    raise ControlError("Manager identity mismatch", 502)\n                state = self._remember(state)\n                owner_state = self._owner_observation()\n                manager_missing = (owner_state.get("scope") != "manager_launcher_only"\n                                   and state.get("manager_pid") is not None\n                                   and state["manager_pid"] not in owner_state.get("pids", []))\n                if owner_state.get("errors") or not owner_state["alive"] or manager_missing:\n                    return self._unavailable_snapshot(owner_state)\n                return dict(state, manager_alive=True, manager_launcher_pid=self.manager.pid,\n                            manager_process_tree=owner_state)\n            except ControlError:\n                return self._unavailable_snapshot(self._owner_observation())\n        return self._unavailable_snapshot(owner_state)\n\n    def control(self, action, **parameters):\n        if not self.manager or not self._manager_tree_alive():\n            raise ControlError("Manager is not running", 503)\n        return rpc_call(self.manager_url, self._control_key, "control",\n                        {"action": action, "expected_instance_id": self.instance_id, **parameters}, timeout=10)\n\n    def command(self, command_id):\n        return rpc_call(self.manager_url, self._control_key, "command", {"command_id": command_id})\n\n    def events(self, after=0, limit=100):\n        return rpc_call(self.manager_url, self._control_key, "events", {"after": after, "limit": limit})\n\n    def wait_ready(self, on_update=None):\n        while True:\n            state = self.snapshot()\n            if on_update:\n                on_update(state)\n            if state.get("ready"):\n                return state\n            if state.get("phase") in {"failed", "stopped", "stopping"} or state.get("error") and state.get("model_alive") is False:\n                raise RuntimeError(state.get("error") or "Preparation failed; control interface remains available")\n            if self._clock() - self._started_at > self.startup_timeout:\n                raise TimeoutError("Generation readiness deadline exceeded; query or stop the manager")\n            time.sleep(self.poll_interval)\n\n    def stop(self):\n        if self.manager is None:\n            self._last.update(phase="stopped", cleanup_complete=True, manager_alive=False)\n            return dict(self._last)\n        if self._manager_tree_alive():\n            deadline = time.monotonic() + 30\n            requested = False\n            command_id = "shutdown-" + uuid.uuid4().hex\n            while time.monotonic() < deadline and self._manager_tree_alive():\n                try:\n                    if not requested:\n                        self.control("shutdown", command_id=command_id)\n                        requested = True\n                    self.snapshot()\n                except (ControlError, OSError):\n                    # A lost acknowledgement is not permission to interrupt audit\n                    # finalization. Retrying this ID cannot enqueue a second stop.\n                    pass\n                if self.manager.poll() is not None:\n                    break\n                time.sleep(.2)\n            # The manager exits itself only after its final command/audit checkpoint.\n            # A clean child snapshot alone must not trigger an early manager signal.\n        if self._manager_owner:\n            owner_state = self._manager_owner.observe()\n            if owner_state["alive"]:\n                owner_state = self._manager_owner.terminate_all(timeout=15)\n        else:\n            if self.manager.poll() is None:\n                self.manager.terminate()\n                self.manager.wait(timeout=15)\n            owner_state = self._owner_observation()\n        if self._output and not self._output.closed:\n            self._output.close()\n        try:\n            recorded = json.loads(self.state_path.read_text(encoding="utf-8"))\n            if recorded.get("instance_id") == self.instance_id:\n                self._remember(recorded)\n        except (OSError, ValueError):\n            pass\n        return self._unavailable_snapshot(owner_state, stopped=True)\n', 'control_protocol.py': '"""Small authenticated loopback RPC shared by the three managed processes."""\nfrom __future__ import annotations\n\nimport hmac\nimport json\nimport logging\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import urlsplit\nfrom urllib.request import Request, urlopen\n\nMAX_BODY = 2 * 1024 * 1024\nLOG = logging.getLogger("h3.control")\n\n\nclass ControlError(RuntimeError):\n    def __init__(self, message: str, status: int = 409):\n        super().__init__(message)\n        self.status = int(status)\n\n\ndef rpc_call(url, key, operation, payload=None, timeout=10):\n    origin = urlsplit(url)\n    if origin.scheme != "http" or origin.hostname not in {"127.0.0.1", "localhost", "::1"}:\n        raise ControlError("Private control transport must use loopback HTTP", 400)\n    if origin.username or origin.password or origin.query or origin.fragment:\n        raise ControlError("Invalid control origin", 400)\n    body = json.dumps({"operation": operation, "payload": payload or {}}, allow_nan=False).encode()\n    if len(body) > MAX_BODY:\n        raise ControlError("Control payload too large", 413)\n    request = Request(url.rstrip("/") + "/rpc", data=body, method="POST", headers={\n        "Authorization": "Bearer " + key, "Content-Type": "application/json", "Connection": "close"})\n    try:\n        try:\n            response = urlopen(request, timeout=timeout)\n        except HTTPError as error:\n            response = error\n        with response:\n            raw = response.read(MAX_BODY + 1)\n            status = response.status\n        if len(raw) > MAX_BODY:\n            raise ControlError("Control response too large", 502)\n        data = json.loads(raw)\n        if not isinstance(data, dict) or type(data.get("ok")) is not bool:\n            raise ValueError("Invalid response envelope")\n        if data["ok"] is not True or not 200 <= status < 300:\n            raise ControlError(str(data.get("error", "Control operation failed")), data.get("status", status))\n        return data.get("result")\n    except ControlError:\n        raise\n    except (URLError, OSError, ValueError, TimeoutError) as error:\n        raise ControlError("Control transport unavailable or invalid response", 503) from error\n\n\ndef make_server(host, port, key, dispatch):\n    if host not in {"127.0.0.1", "localhost"}:\n        raise ValueError("Private controller must bind IPv4 loopback")\n    if not isinstance(key, str) or len(key) < 32 or not key.isascii() or any(c.isspace() for c in key):\n        raise ValueError("A private control token of at least 32 ASCII characters is required")\n\n    class Handler(BaseHTTPRequestHandler):\n        protocol_version = "HTTP/1.1"\n\n        def setup(self):\n            super().setup()\n            self.connection.settimeout(15)\n\n        def log_message(self, *_):\n            pass  # Never log authentication headers, payloads or query secrets.\n\n        def reply(self, status, value):\n            encoded = json.dumps(value, allow_nan=False).encode()\n            self.send_response(status)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(encoded)))\n            self.send_header("Cache-Control", "no-store")\n            self.send_header("Connection", "close")\n            self.end_headers()\n            self.close_connection = True\n            try:\n                self.wfile.write(encoded)\n            except (BrokenPipeError, ConnectionResetError):\n                pass\n\n        def do_GET(self):\n            if self.path != "/health":\n                return self.reply(404, {"ok": False, "error": "Not found", "status": 404})\n            self.reply(200, {"alive": True})\n\n        def do_POST(self):\n            if self.path != "/rpc":\n                return self.reply(404, {"ok": False, "error": "Not found", "status": 404})\n            auth = self.headers.get("Authorization", "")\n            if not auth.isascii() or not hmac.compare_digest(auth, "Bearer " + key):\n                return self.reply(401, {"ok": False, "error": "Private authorization required", "status": 401})\n            try:\n                if self.headers.get("Transfer-Encoding"):\n                    raise ControlError("Chunked control requests are unsupported", 400)\n                size = int(self.headers.get("Content-Length", "0"))\n                if not 0 < size <= MAX_BODY:\n                    raise ControlError("Invalid control payload size", 413)\n                raw = self.rfile.read(size)\n                if len(raw) != size:\n                    raise ControlError("Incomplete control request", 400)\n                data = json.loads(raw, parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))\n                if not isinstance(data, dict) or not isinstance(data.get("operation"), str):\n                    raise ControlError("Invalid control operation", 400)\n                payload = data.get("payload", {})\n                if not isinstance(payload, dict):\n                    raise ControlError("Control payload must be an object", 400)\n                result = dispatch(data["operation"], payload)\n                self.reply(200, {"ok": True, "result": result})\n            except ControlError as error:\n                self.reply(error.status, {"ok": False, "error": str(error), "status": error.status})\n            except (ValueError, TypeError) as error:\n                self.reply(400, {"ok": False, "error": "Invalid control JSON or arguments", "status": 400})\n            except Exception:\n                LOG.exception("Control dispatch failed")\n                self.reply(500, {"ok": False, "error": "Internal control operation failed; inspect owner log", "status": 500})\n\n    server = ThreadingHTTPServer((host, int(port)), Handler)\n    server.daemon_threads = True\n    return server\n', 'process_owner.py': '"""Track and stop an owned Popen tree by PID and creation identity.\n\nWindows virtual-environment launchers can add a real interpreter descendant.\nAn exited wrapper alone is not the definition of a fully reclaimed process tree.\n"""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor\nimport subprocess\nimport threading\nimport time\n\nimport psutil\n\n\nclass ProcessOwner:\n    def __init__(self, popen):\n        self.process = popen\n        self._identities = {}\n        self._lock = threading.RLock()\n        self._root_unknown = False\n        self._unknown_scopes = set()\n        try:\n            process = psutil.Process(popen.pid)\n            self._identities[process.pid] = process.create_time()\n        except psutil.Error:\n            self._root_unknown = True\n            self._unknown_scopes.add(popen.pid)\n\n    def _same(self, pid, created):\n        try:\n            process = psutil.Process(pid)\n            if abs(process.create_time() - created) >= .001:\n                return None\n            if not process.is_running() or process.status() == psutil.STATUS_ZOMBIE:\n                return None\n            return process\n        except psutil.NoSuchProcess:\n            return None\n\n    def _capture(self):\n        errors = []\n        if self._root_unknown:\n            try:\n                if self.process.poll() is None:\n                    root = psutil.Process(self.process.pid)\n                    self._identities[root.pid] = root.create_time()\n                    self._root_unknown = False\n                # An exited wrapper cannot prove an unobserved subtree exited.\n            except (OSError, psutil.Error) as error:\n                errors.append("root.poll: " + type(error).__name__)\n        for pid, created in list(self._identities.items()):\n            try:\n                process = self._same(pid, created)\n                if process is None:\n                    continue\n                children = process.children(recursive=True)\n                complete = True\n                for child in children:\n                    try:\n                        self._identities.setdefault(child.pid, child.create_time())\n                    except psutil.NoSuchProcess:\n                        pass\n                    except psutil.Error as error:\n                        errors.append("descendant.identity: " + type(error).__name__)\n                        complete = False\n                if complete:\n                    self._unknown_scopes.discard(pid)\n                else:\n                    self._unknown_scopes.add(pid)\n            except (psutil.NoSuchProcess, psutil.ZombieProcess):\n                # A known process can exit between identity validation and\n                # children(). This is an exit observation, not a new unknown\n                # subtree. Previously recorded access/identity uncertainty is\n                # deliberately retained until a successful complete recapture.\n                pass\n            except psutil.Error as error:\n                errors.append(str(pid) + ".children: " + type(error).__name__)\n                self._unknown_scopes.add(pid)\n        errors.extend(str(pid) + ".subtree_exit_unconfirmed" for pid in self._unknown_scopes)\n        return errors\n\n    def observe(self):\n        with self._lock:\n            errors = self._capture()\n            alive = []\n            for pid, created in self._identities.items():\n                try:\n                    if self._same(pid, created) is not None:\n                        alive.append(pid)\n                except psutil.Error as error:\n                    alive.append(pid)  # Unknown exit retains ownership.\n                    errors.append(str(pid) + ".observe: " + type(error).__name__)\n            try:\n                root_running = self.process.poll() is None\n            except OSError as error:\n                root_running = True\n                errors.append("root.poll: " + type(error).__name__)\n            if root_running and self.process.pid not in alive:\n                alive.append(self.process.pid)\n                if self.process.pid not in self._identities:\n                    errors.append("root_identity_unconfirmed")\n            return {"alive": bool(alive) or bool(errors), "cleanup_complete": not alive and not errors,\n                    "pids": sorted(set(alive)), "identities": [{"pid": pid, "created_at": created}\n                     for pid, created in self._identities.items()], "errors": sorted(set(errors)),\n                    "observed_at": time.time()}\n\n    def _signal(self, kill=False):\n        with self._lock:\n            self._capture()\n            root_signalled = False\n            for pid, created in list(self._identities.items()):\n                try:\n                    process = self._same(pid, created)\n                    if process:\n                        process.kill() if kill else process.terminate()\n                        root_signalled |= pid == self.process.pid\n                except psutil.Error:\n                    pass  # Final observation, not signal success, decides cleanup.\n            # The original Popen handle is tied to its original process. This is\n            # also the only safe fallback when its psutil identity was unavailable.\n            try:\n                if not root_signalled and self.process.poll() is None:\n                    self.process.kill() if kill else self.process.terminate()\n            except (OSError, subprocess.SubprocessError):\n                pass\n\n    def request_stop(self):\n        self._signal()\n        return self.observe()\n\n    def wait_stopped(self, timeout=15):\n        deadline = time.monotonic() + max(0, timeout)\n        while True:\n            state = self.observe()\n            if state["cleanup_complete"]:\n                return state\n            if time.monotonic() >= deadline:\n                break\n            time.sleep(min(.05, max(0, deadline - time.monotonic())))\n        self._signal(kill=True)\n        deadline = time.monotonic() + 3\n        while True:\n            state = self.observe()\n            if state["cleanup_complete"] or time.monotonic() >= deadline:\n                return state\n            time.sleep(.025)\n\n    def terminate_all(self, timeout=15):\n        self.request_stop()\n        return self.wait_stopped(timeout)\n\n\ndef stop_many(owners, timeout=15):\n    """Independent stop requests and waits overlap; caller joins actual evidence."""\n    owners = list(owners)\n    for owner in owners:\n        owner.request_stop()\n    if not owners:\n        return []\n    try:\n        with ThreadPoolExecutor(max_workers=len(owners), thread_name_prefix="owned-process-cleanup") as pool:\n            futures = [pool.submit(owner.wait_stopped, timeout) for owner in owners]\n            return [future.result() for future in futures]\n    except (RuntimeError, OSError):\n        # Thread creation failure must not prevent already-requested cleanup.\n        deadline = time.monotonic() + timeout\n        return [owner.wait_stopped(max(0, deadline - time.monotonic())) for owner in owners]\n', 'global_manager.py': '"""Durable main-event owner for independent API and native-BF16 model processes."""\nfrom __future__ import annotations\n\nimport argparse\nfrom copy import deepcopy\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport signal\nimport sqlite3\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.request\nimport uuid\n\nimport psutil\n\nfrom control_protocol import ControlError, make_server, rpc_call\nfrom process_owner import ProcessOwner, stop_many\nfrom generation_options import requested_options, validate_generation_options\nfrom adapter_config import load_adapter_settings\nfrom condition_guard import evaluate_gate\n\nLOG = logging.getLogger("h3_manager")\nTERMINAL = {"succeeded", "failed", "cancelled"}\nNONTERMINAL = {"queued", "running", "cancel_requested"}\nIDENTIFIER = re.compile(r"[a-f0-9]{32}")\nDRIVE_FILE_ID = re.compile(r"[A-Za-z0-9_-]{1,128}")\nSHA256 = re.compile(r"[a-f0-9]{64}")\nGATE_SOURCE_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nDISPATCH_CONDITIONS = ("manager_open", "job_queued", "attempt_unassigned", "model_identity_current",\n                       "model_observation_fresh", "model_process_alive", "model_ready", "worker_alive",\n                       "worker_accepting", "supervisor_alive")\n\n\nclass RootLock:\n    """An OS-held lock, automatically released if the owner process dies."""\n\n    def __init__(self, path):\n        self.file = open(path, "a+b")\n        try:\n            if os.fstat(self.file.fileno()).st_size == 0:\n                self.file.write(b"0")\n                self.file.flush()\n            self.file.seek(0)\n            if os.name == "nt":\n                import msvcrt\n                msvcrt.locking(self.file.fileno(), msvcrt.LK_NBLCK, 1)\n            else:\n                import fcntl\n                fcntl.flock(self.file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except (OSError, IOError) as error:\n            self.file.close()\n            raise RuntimeError("Another manager owns this output directory") from error\n\n    def close(self):\n        if self.file.closed:\n            return\n        self.file.seek(0)\n        if os.name == "nt":\n            import msvcrt\n            msvcrt.locking(self.file.fileno(), msvcrt.LK_UNLCK, 1)\n        else:\n            import fcntl\n            fcntl.flock(self.file.fileno(), fcntl.LOCK_UN)\n        self.file.close()\n\n\ndef probe_api(url, instance_id, timeout=2):\n    try:\n        req = urllib.request.Request(url.rstrip("/") + "/health/live",\n                                     headers={"Cache-Control": "no-cache"})\n        with urllib.request.urlopen(req, timeout=timeout) as response:\n            body = json.loads(response.read(16385))\n        return isinstance(body, dict) and body.get("instance_id") == instance_id and body.get("api_ready") is True\n    except Exception:\n        return False\n\n\nclass GlobalManager:\n    def __init__(self, directory, *, root, instance_id, port, api_port, model_port,\n                 staged=False, public=False, install_cloudflared=False, env=None,\n                 poll_interval=1, probe_timeout=2, startup_timeout=3600, generation_timeout=5400, tunnel_timeout=180,\n                 popen=None, rpc=None, probe=None, clock=None, owner_factory=None, start_model=True):\n        if not IDENTIFIER.fullmatch(instance_id):\n            raise ValueError("Invalid manager instance ID")\n        self.directory, self.root = Path(directory).resolve(), Path(root).resolve()\n        self.instance_id = instance_id\n        self.port, self.api_port, self.model_port = int(port), int(api_port), int(model_port)\n        if len({self.port, self.api_port, self.model_port}) != 3:\n            raise ValueError("Manager, API and model ports must be distinct")\n        self.staged, self.public = bool(staged), bool(public)\n        self.install_cloudflared = bool(install_cloudflared)\n        self.model_autostart = bool(start_model)\n        self.env = dict(os.environ if env is None else env)\n        self.key = self.env.get("H3_CONTROL_KEY", "")\n        if len(self.key) < 32 or len(self.env.get("H3_API_KEY", "")) < 32:\n            raise ValueError("Separate private/public keys of at least 32 characters are required")\n        if self.key == self.env["H3_API_KEY"]:\n            raise ValueError("Private and public tokens must differ")\n        self.poll_interval, self.probe_timeout = poll_interval, probe_timeout\n        self.startup_timeout = startup_timeout\n        self.generation_timeout = generation_timeout\n        self.tunnel_timeout = tunnel_timeout\n        self.disk_reserve_bytes = max(0, int(self.env.get("H3_DISK_RESERVE_BYTES", 64 * 1024 * 1024)))\n        self._popen, self._rpc, self._probe = popen or subprocess.Popen, rpc or rpc_call, probe or probe_api\n        self._owner_factory = owner_factory or ProcessOwner\n        self._clock = clock or time.monotonic\n        self._started = self._clock()\n        self.run_directory = self.directory / "runtime" / instance_id\n        self.run_directory.mkdir(parents=True, exist_ok=True)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.state_path, self.events_path = self.run_directory / "status.json", self.run_directory / "events.jsonl"\n        self._owner = RootLock(self.root / "owner.lock")\n        self._db = sqlite3.connect(self.root / "control.sqlite3", check_same_thread=False)\n        self._db.execute("PRAGMA journal_mode=WAL")\n        self._db.execute("PRAGMA synchronous=FULL")\n        self._db.executescript("""\n            CREATE TABLE IF NOT EXISTS jobs (id TEXT PRIMARY KEY, status TEXT NOT NULL,\n                idempotency_key TEXT UNIQUE, fingerprint TEXT, record TEXT NOT NULL);\n            CREATE INDEX IF NOT EXISTS jobs_status ON jobs(status);\n            CREATE TABLE IF NOT EXISTS events (sequence INTEGER PRIMARY KEY AUTOINCREMENT,\n                event_id TEXT UNIQUE NOT NULL, record TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS commands (id TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,\n                record TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS owned_processes (launch_id TEXT PRIMARY KEY, module TEXT NOT NULL,\n                pid INTEGER NOT NULL, created REAL, exited INTEGER NOT NULL DEFAULT 0);\n        """)\n        self._lock = threading.RLock()\n        self._control_lock = threading.RLock()\n        self._refresh_lock = threading.Lock()\n        self._cancel = threading.Event()\n        self._thread = None\n        self._control_threads = []\n        self._control_admission_closed = False\n        self._control_slots = threading.BoundedSemaphore(4)\n        self.children = {"api": None, "model": None, "tunnel": None}\n        self.owners = {}\n        self._files = []\n        self.model_instance_id = None\n        self._model = {}\n        self._api_ready = self._public_api_ready = self._connected = False\n        self._api_expected = False\n        self._api_expected_elapsed = 0.0\n        self._api_expected_since = None\n        self._api_intent_epoch = 0\n        self._api_lifecycle = "not_started"\n        self._public_url = None\n        self._phase = "new"\n        self._draining = False\n        self._shutdown_started = False\n        self._shutdown_complete = False\n        self._exit_requested = False\n        self._last_observation = None\n        self._model_failure = None\n        self._error = None\n        self._cleanup_errors = []\n        self._signature = None\n        self._condition_gates = {}\n        self._dispatch_gate_signature = None\n        self._export_sequence = 0\n        self._background = False\n        self._ever_ready = False\n        self._model_started = None\n        self._model_unreachable = 0\n        self._model_ever_ready = False\n        self._public_ever_ready = False\n        self._probe_pool = ThreadPoolExecutor(max_workers=2, thread_name_prefix="independent-module-probe")\n        self._closed = False\n        self._recover()\n\n    def _event(self, kind, *, job=None, data=None):\n        event = {"event_id": uuid.uuid4().hex, "instance_id": self.instance_id,\n                 "job_id": job.get("job_id") if job else None,\n                 "attempt_id": job.get("attempt_id") if job else None,\n                 "type": kind, "observed_at": time.time(),\n                 "monotonic_ns": time.monotonic_ns(), "data": data or {}}\n        cursor = self._db.execute("INSERT INTO events(event_id, record) VALUES (?, ?)",\n                                  (event["event_id"], json.dumps(event)))\n        event["sequence"] = cursor.lastrowid\n        return event\n\n    def _recover(self):\n        # OS root ownership is held, but previous child fate may be uncertain.\n        # Never replay an accepted expensive execution after manager replacement.\n        with self._lock, self._db:\n            for (encoded,) in self._db.execute("SELECT record FROM jobs WHERE status IN (\'queued\',\'running\',\'cancel_requested\')").fetchall():\n                job = json.loads(encoded)\n                self._transition(job, "failed", error="manager_restarted; execution not replayed",\n                                 cleanup_complete=False, finished_at=time.time())\n            for command_id, encoded in self._db.execute("SELECT id,record FROM commands").fetchall():\n                command = json.loads(encoded)\n                if command["status"] in {"accepted", "running"}:\n                    command.update(status="failed", error="manager_restarted; command not replayed", finished_at=time.time())\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(command), command_id))\n            self._event("manager.created", data={"recovery_policy": "no automatic execution replay"})\n\n    def _get_job(self, job_id):\n        row = self._db.execute("SELECT record FROM jobs WHERE id=?", (job_id,)).fetchone()\n        if not row:\n            raise ControlError("Job not found", status=404)\n        return json.loads(row[0])\n\n    def _store_job(self, job):\n        self._db.execute("UPDATE jobs SET status=?,record=? WHERE id=?",\n                         (job["status"], json.dumps(job), job["job_id"]))\n\n    def _transition(self, job, target, **updates):\n        allowed = {"queued": {"running", "failed", "cancelled"},\n                   "running": {"queued", "succeeded", "failed", "cancel_requested", "cancelled"},\n                   "cancel_requested": {"cancelled", "failed"}}\n        if target not in allowed.get(job["status"], set()):\n            raise RuntimeError(f"Illegal main event transition {job[\'status\']} -> {target}")\n        job.update(status=target, **updates)\n        job["lifecycle"].append({"state": target, "observed_at": time.time()})\n        self._store_job(job)\n        self._event("job." + target, job=job, data={"error": job.get("error")})\n\n    def _active_jobs(self):\n        rows = self._db.execute("SELECT record FROM jobs WHERE status IN (\'running\',\'cancel_requested\') ORDER BY rowid").fetchall()\n        return [json.loads(row[0]) for row in rows]\n\n    def _queue_summary(self):\n        counts = dict(self._db.execute("SELECT status,COUNT(*) FROM jobs GROUP BY status").fetchall())\n        return {"queued": counts.get("queued", 0), "running": counts.get("running", 0),\n                "cancel_requested": counts.get("cancel_requested", 0),\n                "unfinished": sum(counts.get(s, 0) for s in NONTERMINAL),\n                "job_count_limit": None, "order": "durable_insertion_fifo",\n                "admission_open": not (self._draining or self._shutdown_started or self._model_failure\n                                        or self._phase in {"new", "failed", "stopped"}),\n                "disk_reserve_bytes": self.disk_reserve_bytes}\n\n    def _liveness(self):\n        alive, errors = {}, {}\n        for name, child in self.children.items():\n            try:\n                if name in self.owners:\n                    observation = self.owners[name].observe()\n                    alive[name] = observation["alive"]\n                    if observation["errors"]:\n                        errors[name] = "; ".join(observation["errors"])\n                else:\n                    alive[name] = child is not None and child.poll() is None\n            except OSError as error:\n                alive[name] = True  # Unknown exit retains ownership.\n                errors[name] = type(error).__name__\n        return alive, errors\n\n    def _historical_ownership(self):\n        current = {launch_id for child in self.children.values() if child is not None\n                   for launch_id in getattr(child, "_h3_tree_records", {}).values()}\n        unresolved = []\n        rows = self._db.execute("SELECT launch_id,module,pid,created FROM owned_processes WHERE exited=0").fetchall()\n        for launch_id, module, pid, created in rows:\n            if launch_id in current:\n                continue\n            try:\n                process = psutil.Process(pid)\n                same = created is None or abs(process.create_time() - created) < .001\n                if same and process.is_running() and process.status() != psutil.STATUS_ZOMBIE:\n                    unresolved.append({"module": module, "pid": pid, "identity_known": created is not None})\n            except psutil.NoSuchProcess:\n                pass\n            except psutil.Error:\n                unresolved.append({"module": module, "pid": pid, "identity_known": False})\n        return unresolved\n\n    def snapshot(self):\n        with self._lock:\n            alive, process_errors = self._liveness()\n            historical = self._historical_ownership()\n            age = self._clock() - self._last_observation if self._last_observation is not None else None\n            stale = age is None or age > 3 * self.probe_timeout + self.poll_interval + 2\n            supervised = not self._background or (self._thread is not None and self._thread.is_alive())\n            usable = not stale and supervised and not process_errors and not self._shutdown_started\n            api = bool(usable and alive["api"] and self._api_ready)\n            model = bool(usable and alive["model"] and self._model.get("ready") is True\n                         and self._model.get("worker_alive") is True and not self._model_failure)\n            public_api = bool(api and alive["tunnel"] and self._connected and self._public_api_ready)\n            ready = api and model and not self._draining and (not self.public or public_api)\n            phase = self._phase\n            if phase not in {"new", "stopping", "stopped", "failed"}:\n                phase = "draining" if self._draining else "ready" if ready else "degraded" if self._ever_ready or self._model_failure or self._error else "preparing"\n            downloads = self._model.get("downloads")\n            if (not isinstance(downloads, dict) or downloads.get("model_instance_id") != self.model_instance_id\n                    or not isinstance(downloads.get("items"), dict)):\n                downloads = {"schema_version": 1, "model_instance_id": self.model_instance_id,\n                             "observed_at": None, "observation_stale": True, "items": {},\n                             "error": "progress_not_observed_for_current_model"}\n            else:\n                downloads = deepcopy(downloads)\n            download_lifecycle = (("stopping" if alive["model"] else "stopped") if self._shutdown_started else\n                                  "failed" if self._model_failure and self._model_failure != "model_stopped" else\n                                  "not_started" if self.model_instance_id is None else\n                                  "stopped" if alive["model"] is False else self._model.get("phase", "unknown"))\n            downloads.update(model_ready=model, observation_stale=bool(downloads.get("observation_stale")\n                             or stale or not supervised or alive["model"] is not True or "model" in process_errors),\n                             lifecycle=download_lifecycle)\n            result = {"instance_id": self.instance_id, "phase": phase, "ready": ready,\n                      "api_ready": api, "model_ready": model, "public_requested": self.public,\n                      "public_api_ready": public_api, "public_ready": public_api and model and not self._draining,\n                      "tunnel_connected": bool(alive["tunnel"] and self._connected), "public_url": self._public_url,\n                      "server_alive": alive["api"], "model_alive": alive["model"], "tunnel_alive": alive["tunnel"],\n                      "manager_alive": True, "manager_pid": os.getpid(), "api_pid": getattr(self.children["api"], "pid", None),\n                      "server_pid": getattr(self.children["api"], "pid", None),\n                      "model_pid": getattr(self.children["model"], "pid", None), "tunnel_pid": getattr(self.children["tunnel"], "pid", None),\n                      "model_instance_id": self.model_instance_id,\n                      "downloads": downloads,\n                      "cleanup_complete": not any(alive.values()) and not historical and not process_errors,\n                      "shutdown_complete": self._shutdown_complete,\n                      "historical_ownership": historical, "cleanup_errors": list(self._cleanup_errors),\n                      "error": self._error or self._model_failure, "elapsed_seconds": round(self._clock() - self._started, 3),\n                      "observed_at": time.time(), "observation_age_seconds": round(age, 3) if age is not None else None,\n                      "observation_stale": stale, "supervisor_alive": supervised,\n                      "process_observation_errors": process_errors, "draining": self._draining,\n                      "condition_gates": deepcopy(self._condition_gates),\n                      "queue": self._queue_summary(),\n                      "owned_processes": {name: owner.observe() for name, owner in self.owners.items()},\n                      "state_path": str(self.state_path), "events_path": str(self.events_path),\n                      "modules": {"api": {"alive": alive["api"], "ready": api,\n                                          "expected_running": self._api_expected, "lifecycle": self._api_lifecycle,\n                                          "intent_epoch": self._api_intent_epoch},\n                                  "model": {**self._model, "alive": alive["model"], "ready": model, "downloads": downloads},\n                                  "manager": {"alive": True, "supervisor_alive": supervised}}}\n            return result\n\n    def _checkpoint(self):\n        with self._lock:\n            state = self.snapshot()\n            signature = json.dumps({k: state[k] for k in ("phase", "ready", "api_ready", "model_ready", "public_api_ready", "server_alive", "model_alive", "tunnel_alive", "cleanup_complete", "error")}, sort_keys=True)\n            if signature != self._signature:\n                with self._db:\n                    self._event("manager.state", data=json.loads(signature))\n                self._signature = signature\n            # SQLite remains authoritative if this readable export is interrupted.\n            rows = self._db.execute("SELECT sequence,record FROM events WHERE sequence>? ORDER BY sequence", (self._export_sequence,)).fetchall()\n            if rows:\n                with self.events_path.open("a", encoding="utf-8") as output:\n                    for sequence, record in rows:\n                        event = json.loads(record)\n                        event["sequence"] = sequence\n                        output.write(json.dumps(event) + "\\n")\n                    if self._shutdown_complete:\n                        output.flush()\n                        os.fsync(output.fileno())\n                self._export_sequence = rows[-1][0]\n            # Publish the final receipt only after the matching audit export.\n            temporary = self.state_path.with_suffix(".tmp")\n            with temporary.open("w", encoding="utf-8") as output:\n                output.write(json.dumps(state, indent=2))\n                if self._shutdown_complete:\n                    output.flush()\n                    os.fsync(output.fileno())\n            temporary.replace(self.state_path)\n\n    def _spawn(self, name, command, env):\n        output = (self.run_directory / (name + ".log")).open("a", encoding="utf-8")\n        self._files.append(output)\n        child = self._popen(command, cwd=self.directory, env=env, stdout=output, stderr=subprocess.STDOUT)\n        self.children[name] = child\n        self.owners[name] = self._owner_factory(child)\n        with self._lock, self._db:\n            try:\n                created = psutil.Process(child.pid).create_time()\n            except psutil.Error:\n                created = None\n            launch_id = uuid.uuid4().hex\n            child._h3_launch_id = launch_id\n            child._h3_tree_records = {(child.pid, created): launch_id}\n            self._db.execute("INSERT INTO owned_processes(launch_id,module,pid,created) VALUES (?,?,?,?)",\n                             (launch_id, name, child.pid, created))\n            self._event("module.started", data={"module": name, "pid": child.pid,\n                        "model_instance_id": self.model_instance_id if name == "model" else None})\n        return child\n\n    def _register_process_tree(self):\n        with self._lock, self._db:\n            for name, owner in self.owners.items():\n                child = self.children[name]\n                for identity in owner.observe()["identities"]:\n                    identity_key = (identity["pid"], identity["created_at"])\n                    if identity_key not in child._h3_tree_records:\n                        launch_id = uuid.uuid4().hex\n                        child._h3_tree_records[identity_key] = launch_id\n                        self._db.execute("INSERT INTO owned_processes(launch_id,module,pid,created) VALUES (?,?,?,?)",\n                                         (launch_id, name, *identity_key))\n\n    def _child_env(self):\n        return {**self.env, "H3_PARENT_PID": str(os.getpid()),\n                "H3_PARENT_CREATED_AT": str(psutil.Process(os.getpid()).create_time()), "H3_INSTANCE_ID": self.instance_id,\n                "H3_MANAGER_URL": f"http://127.0.0.1:{self.port}", "H3_OUTPUT_DIR": str(self.root)}\n\n    def _set_api_expectation(self, expected):\n        # Only time in which the API is requested to run consumes its first\n        # public-delivery budget. An explicit stop pauses, rather than resets,\n        # that budget, including when process exit remains unconfirmed.\n        with self._lock:\n            if expected != self._api_expected:\n                now = self._clock()\n                if self._api_expected_since is not None:\n                    self._api_expected_elapsed += max(0.0, now - self._api_expected_since)\n                self._api_expected_since = now if expected else None\n                self._api_expected = expected\n\n    def _launch_api(self):\n        alive, errors = self._liveness()\n        if alive["api"] or "api" in errors:\n            raise ControlError("API ownership has not been released", status=409)\n        with self._lock:\n            self._api_ready = self._public_api_ready = False\n            self._set_api_expectation(True)\n            self._api_intent_epoch += 1\n            self._api_lifecycle = "starting"\n        self._spawn("api", [sys.executable, "-m", "uvicorn", "server:create_app", "--factory", "--host", "127.0.0.1",\n                            "--port", str(self.api_port), "--workers", "1", "--limit-concurrency", "8"], self._child_env())\n\n    def _launch_model(self):\n        alive, errors = self._liveness()\n        if alive["model"] or "model" in errors:\n            raise ControlError("Old model ownership has not been released", status=409)\n        with self._lock:\n            old_processes = self._db.execute("SELECT launch_id,pid,created FROM owned_processes WHERE module=\'model\' AND exited=0").fetchall()\n        for launch_id, pid, created in old_processes:\n            try:\n                process = psutil.Process(pid)\n                same = created is None or abs(process.create_time() - created) < .001\n                if same and process.is_running() and process.status() != psutil.STATUS_ZOMBIE:\n                    raise ControlError("Previous model process exit is unconfirmed", status=409)\n            except psutil.NoSuchProcess:\n                pass\n            except psutil.Error:\n                raise ControlError("Previous model process identity is inaccessible", status=409)\n            with self._lock, self._db:\n                self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (launch_id,))\n        self.model_instance_id = uuid.uuid4().hex\n        self._model = {"phase": "loading", "instance_id": self.model_instance_id, "ready": False}\n        self._model_failure = None\n        self._model_started = self._clock()\n        self._model_unreachable = 0\n        self._model_ever_ready = False\n        command = [sys.executable, "-u", str(self.directory / "model_container.py"), "--port", str(self.model_port),\n                   "--instance-id", self.model_instance_id, "--root", str(self.root), "--parent-pid", str(os.getpid())]\n        if self.staged:\n            command.append("--staged")\n        env = self._child_env()\n        env.update(H3_MODEL_INSTANCE_ID=self.model_instance_id,\n                   H3_DOWNLOAD_PROGRESS_PATH=str(self.run_directory / ("model-" + self.model_instance_id + ".downloads.json")))\n        self._spawn("model", command, env)\n\n    def start(self, *, background=True):\n        with self._control_lock:\n            if self._phase != "new":\n                raise ControlError("Manager already started", status=409)\n            self._phase, self._background = "preparing", bool(background)\n            try:\n                env = self._child_env()\n                self._launch_api()\n                if self.model_autostart:\n                    try:\n                        self._launch_model()\n                    except ControlError:\n                        # Recovery may observe a previous owner child still exiting.\n                        # Keep the API/control plane available; never overlap models.\n                        self._model_failure = "previous_model_exit_unconfirmed"\n                else:\n                    self._model = {"instance_id": None, "phase": "not_started", "ready": False,\n                                   "worker_alive": False, "accepting": False}\n                    with self._lock, self._db:\n                        self._event("model.autostart_skipped", data={"reason": "explicit_configuration"})\n                if self.public:\n                    command = [sys.executable, "-u", str(self.directory / "tunnel.py"), "--port", str(self.api_port),\n                               "--status-file", str(self.run_directory / "tunnel-status.json"), "--instance-id", self.instance_id]\n                    if self.install_cloudflared:\n                        command.append("--install")\n                    self._spawn("tunnel", command, env)\n                if background:\n                    self._thread = threading.Thread(target=self._monitor, name="main-event-supervisor", daemon=True)\n                    self._thread.start()\n                self._checkpoint()\n            except BaseException:\n                self._error = "module_start_failed"\n                self.shutdown()\n                self._phase = "failed"\n                raise\n        return self\n\n    def _transport(self):\n        try:\n            path = self.run_directory / "tunnel-status.json"\n            if path.stat().st_size > 16384:\n                return None, False\n            body = json.loads(path.read_text(encoding="utf-8"))\n            url = body.get("url") if isinstance(body, dict) else None\n            if (body.get("instance_id") != self.instance_id or not isinstance(url, str)\n                    or not re.fullmatch(r"https://[a-z0-9-]+\\.trycloudflare\\.com", url)):\n                return None, False\n            return url, body.get("connected") is True and body.get("phase") not in {"failed", "stopping", "stopped"}\n        except (OSError, ValueError, TypeError, AttributeError):\n            return None, False\n\n    def _fail_jobs_after_exit(self, reason):\n        with self._lock, self._db:\n            for (encoded,) in self._db.execute("SELECT record FROM jobs WHERE status IN (\'queued\',\'running\',\'cancel_requested\')").fetchall():\n                job = json.loads(encoded)\n                target = "cancelled" if job["status"] == "cancel_requested" else "failed"\n                cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=True)\n                self._transition(job, target, error=reason, finished_at=time.time(), cleanup_complete=cleaned,\n                                 cleanup_errors=cleanup_errors, model_lease_released=True)\n\n    def _cleanup_job_files(self, job_id, *, discard_result):\n        errors = []\n        job_root = self.root / job_id\n        if not job_root.resolve().is_relative_to(self.root) or job_root.is_symlink():\n            return False, ["job_directory_identity_invalid"]\n        try:\n            shutil.rmtree(job_root / "inputs")\n        except FileNotFoundError:\n            pass\n        except OSError as error:\n            errors.append("inputs: " + type(error).__name__)\n        if discard_result:\n            for name in ("result.mp4", "result.metrics.json"):\n                try:\n                    (job_root / name).unlink(missing_ok=True)\n                except OSError as error:\n                    errors.append(name + ": " + type(error).__name__)\n        return not errors, errors\n\n    def _model_probe(self, expected_instance_id):\n        try:\n            candidate = self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "status", {}, timeout=self.probe_timeout)\n            if isinstance(candidate, dict) and candidate.get("instance_id") == expected_instance_id:\n                return candidate\n        except Exception:\n            pass\n        return None\n\n    def refresh(self):\n        with self._refresh_lock:\n            if self._shutdown_started:\n                return self.snapshot()\n            with self._control_lock:\n                self._register_process_tree()\n                alive, errors = self._liveness()\n                observed_model_id, observed_child = self.model_instance_id, self.children["model"]\n                observed_api_child = self.children["api"]\n                observed_api_epoch = self._api_intent_epoch\n            api_future = (self._probe_pool.submit(self._probe, f"http://127.0.0.1:{self.api_port}", self.instance_id, self.probe_timeout)\n                          if alive["api"] else None)\n            model_future = (self._probe_pool.submit(self._model_probe, observed_model_id)\n                            if alive["model"] and "model" not in errors else None)\n            api_ready = bool(api_future.result()) if api_future else False\n            model = model_future.result() if model_future else None\n            url, connected = self._transport() if self.public and alive["tunnel"] else (None, False)\n            public_api = bool(api_ready and connected and self._probe(url, self.instance_id, self.probe_timeout))\n            with self._control_lock:\n                if (self._shutdown_started or observed_model_id != self.model_instance_id\n                        or observed_child is not self.children["model"] or observed_api_child is not self.children["api"]\n                        or observed_api_epoch != self._api_intent_epoch):\n                    return self.snapshot()  # Discard a previous model generation\'s observations.\n                if alive["model"] and "model" not in errors:\n                    self._model_unreachable = 0 if model else self._model_unreachable + 1\n                with self._lock:\n                    self._api_ready = bool(api_ready and self._api_expected)\n                    self._public_api_ready = bool(public_api and self._api_expected)\n                    self._connected = connected\n                    if url:\n                        self._public_url = url\n                    previous_downloads = self._model.get("downloads")\n                    self._model = model or {"instance_id": self.model_instance_id, "ready": False,\n                                            "phase": "unreachable" if alive["model"] else\n                                                     "not_started" if self.model_instance_id is None else "stopped"}\n                    if not model and isinstance(previous_downloads, dict):\n                        self._model["downloads"] = {**deepcopy(previous_downloads), "observation_stale": True}\n                    self._last_observation = self._clock()\n                    if model and model.get("ready") is True:\n                        self._model_ever_ready = True\n                    if self._public_api_ready:\n                        self._public_ever_ready = True\n                        if self._error and self._error.startswith("public_"):\n                            self._error = None\n                    if not alive["api"] and self._api_expected:\n                        self._error = "api_process_exited; model and durable jobs retained"\n                        self._api_lifecycle = "failed"\n                    elif api_ready and self._error and self._error.startswith("api_"):\n                        self._error = None\n                    if self._api_expected and api_ready:\n                        self._api_lifecycle = "ready"\n                    if not alive["model"] and self.model_instance_id is not None:\n                        self._model_failure = self._model_failure or "model_process_exited"\n                    elif model and model.get("phase") == "failed":\n                        self._model_failure = "model_load_or_worker_failed"\n                if not alive["model"] and "model" not in errors and self.model_instance_id is not None:\n                    self._fail_jobs_after_exit("model_process_exited")\n                elif model:\n                    self._consume_result(model)\n                    with self._lock:\n                        expired = any(time.time() - job["started_at"] > self.generation_timeout\n                                      for job in self._active_jobs())\n                    # A dead worker can no longer own a useful execution. Confirm\n                    # process exit before releasing the main execution lease.\n                    if expired:\n                        self._stop_model(force=True, reason="generation_deadline_exceeded")\n                    elif self._model_failure or model.get("phase") == "failed" or (model.get("worker_alive") is False and model.get("phase") not in {"starting", "loading"}):\n                        with self._control_lock:\n                            self._stop_model(force=True, reason="model_worker_failed")\n                    else:\n                        self._dispatch_next(model)\n                if alive["model"] and self._model_unreachable >= 3:\n                    with self._control_lock:\n                        self._stop_model(force=True, reason="model_control_unreachable_for_three_cycles")\n                with self._lock:\n                    active = self._active_jobs()\n                if any(time.time() - job["started_at"] > self.generation_timeout for job in active):\n                    with self._control_lock:\n                        self._stop_model(force=True, reason="generation_deadline_exceeded")\n                if self.snapshot()["ready"]:\n                    self._ever_ready = True\n                if (self._model_started is not None and not self._model_ever_ready and self._clock() - self._model_started > self.startup_timeout\n                        and not self._model.get("ready") and not active and alive["model"]):\n                    with self._control_lock:\n                        self._stop_model(force=True, reason="model_startup_deadline_exceeded")\n                public_elapsed = self._api_expected_elapsed\n                if self._api_expected_since is not None:\n                    public_elapsed += max(0.0, self._clock() - self._api_expected_since)\n                if (self.public and self._api_expected and not self._public_ever_ready\n                        and public_elapsed > self.tunnel_timeout):\n                    self._error = "public_preparation_deadline_exceeded; local API and model retained"\n                self._checkpoint()\n                return self.snapshot()\n\n    def _consume_result(self, model):\n        results = model.get("results")\n        if not isinstance(results, list):\n            results = [model.get("last_result")]  # Read old checkpoints without losing their result.\n        acknowledgements = []\n        for result in results:\n            if not isinstance(result, dict) or result.get("status") not in TERMINAL:\n                continue\n            with self._lock, self._db:\n                try:\n                    job = self._get_job(result.get("job_id"))\n                except ControlError:\n                    continue\n                if (result.get("attempt_id") != job["attempt_id"]\n                        or model.get("instance_id") != job["model_instance_id"]\n                        or result.get("instance_id", model["instance_id"]) != job["model_instance_id"]):\n                    continue\n                if job["status"] in TERMINAL:\n                    acknowledgements.append(job["attempt_id"])\n                    continue\n                if job["status"] not in {"running", "cancel_requested"}:\n                    continue\n                if result.get("cleanup_complete") is not True:\n                    self._model_failure = "model_attempt_cleanup_unconfirmed"\n                    job.update(error=self._model_failure, cleanup_complete=False)\n                    self._store_job(job)\n                    self._event("job.cleanup_pending", job=job)\n                    continue  # Retain ownership until process exit is confirmed.\n                target, artifact = result["status"], result.get("artifact")\n                cancellation_requested = job["status"] == "cancel_requested"\n                if cancellation_requested and target != "failed":\n                    target, artifact = "cancelled", None\n                elif target == "succeeded":\n                    artifact = self._verified_artifact(job["job_id"], artifact)\n                    if artifact is None:\n                        target = "failed"\n                cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=target != "succeeded")\n                self._transition(job, target, artifact=artifact if target == "succeeded" else None,\n                                 error=None if target == "succeeded" else result.get("error") or "result_not_committed",\n                                 metrics=result.get("metrics"), finished_at=time.time(), cleanup_complete=cleaned,\n                                 cancellation_requested=cancellation_requested,\n                                 execution_condition_gates=deepcopy(result.get("condition_gates")),\n                                 stage_completions=deepcopy(result.get("stage_completions")),\n                                 generation_options=result.get("generation_options") or job.get("generation_options"),\n                                 cleanup_errors=cleanup_errors, model_lease_released=True, dispatch_state="completed")\n                acknowledgements.append(job["attempt_id"])\n        # The durable transaction must commit before the model may discard a result.\n        if acknowledgements and isinstance(model.get("results"), list):\n            try:\n                self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "ack_results",\n                          {"instance_id": model["instance_id"], "attempt_ids": acknowledgements}, timeout=self.probe_timeout)\n            except Exception:\n                pass  # Next status repeats the durable results; acknowledge idempotently.\n\n    def _verified_artifact(self, job_id, artifact):\n        if not isinstance(artifact, dict):\n            return None\n        try:\n            path = Path(artifact["path"]).resolve()\n            lexical = self.root / job_id / "result.mp4"\n            expected = lexical.resolve()\n            if (path != expected or not path.is_relative_to(self.root) or not path.is_file()\n                    or lexical.is_symlink() or not isinstance(artifact.get("media"), dict)\n                    or artifact["media"].get("full_decode_passed") is not True):\n                return None\n            if path.stat().st_size != artifact["bytes"] or artifact["bytes"] <= 0:\n                return None\n            if not re.fullmatch(r"[a-f0-9]{64}", artifact.get("sha256", "")):\n                return None\n            with path.open("rb") as source:\n                actual = hashlib.file_digest(source, "sha256").hexdigest()\n            if actual != artifact["sha256"]:\n                return None\n            return artifact\n        except (KeyError, OSError, TypeError, ValueError):\n            return None\n\n    def _dispatch_gate(self, model, job, attempt_id):\n        """Called only in the dispatch/cancel arbitration and state transaction.\n\n        The remote observation permits an offer, not remote execution. Container\n        admission and resource ownership are revalidated by their own owners.\n        """\n        model = model if isinstance(model, dict) else {}\n        alive, errors = self._liveness()\n        age = self._clock() - self._last_observation if self._last_observation is not None else None\n        bound = 3 * self.probe_timeout + self.poll_interval + 2\n        identity = self.model_instance_id\n        identities = (identity, model.get("instance_id"), self._model.get("instance_id"))\n        current_identity = (all(value == identity for value in identities)\n                            if all(isinstance(value, str) and value for value in identities) else None)\n        def capability(field):\n            witnesses = [model.get(field), self._model.get(field)]\n            # Missing or malformed fresh evidence cannot borrow a previous True.\n            return witnesses if all(type(value) is bool for value in witnesses) else None\n        observations = {\n            # Draining closes new admission; the already-owned queue still drains.\n            "manager_open": self._shutdown_started is False and self._model_failure is None,\n            "job_queued": job.get("status") == "queued", "attempt_unassigned": job.get("attempt_id") is None,\n            "model_identity_current": current_identity,\n            "model_observation_fresh": None if age is None else 0 <= age <= bound,\n            "model_process_alive": None if "model" in errors else alive.get("model"),\n            "model_ready": capability("ready"),\n            "worker_alive": capability("worker_alive"),\n            "worker_accepting": capability("accepting"),\n            "supervisor_alive": not self._background or (self._thread is not None and self._thread.is_alive()),\n        }\n        receipt = evaluate_gate("manager.dispatch", DISPATCH_CONDITIONS, observations,\n            {"manager_instance_id": self.instance_id, "model_instance_id": identity, "job_id": job["job_id"],\n             "attempt_id": attempt_id, "source_file": "global_manager.py", "source_sha256": GATE_SOURCE_SHA256})\n        # Commit every positive authorization; coalesce unchanged blocked polls.\n        signature = json.dumps({"job_id": job["job_id"], "model_instance_id": identity,\n                                "conditions": receipt["conditions"]}, sort_keys=True)\n        if receipt["allowed"] or signature != self._dispatch_gate_signature:\n            self._dispatch_gate_signature = signature\n            self._condition_gates["manager.dispatch"] = receipt\n            job["condition_gate"] = receipt\n            self._store_job(job)\n            self._event("condition.checked", job=job, data={"receipt": receipt})\n        return receipt\n\n    def _dispatch_next(self, model):\n        with self._control_lock:\n            deadline = time.monotonic() + max(.01, min(.25, self.poll_interval))\n            while not self._shutdown_started and not self._model_failure:\n                with self._lock, self._db:\n                    row = self._db.execute("SELECT record FROM jobs WHERE status=\'queued\' ORDER BY rowid LIMIT 1").fetchone()\n                    if not row:\n                        return\n                    job = json.loads(row[0])\n                    prospective_attempt = uuid.uuid4().hex\n                    receipt = self._dispatch_gate(model, job, prospective_attempt)\n                    if receipt["allowed"] is not True:\n                        return\n                    self._transition(job, "running", attempt_id=prospective_attempt,\n                                     model_instance_id=self.model_instance_id, started_at=time.time(), cleanup_complete=False,\n                                     model_lease_released=False, dispatch_state="offering", dispatch_reason=None)\n                    self._event("job.dispatch_intent", job=job, data={"condition_decision_id": receipt["decision_id"]})\n                # running means this attempt\'s ownership, not ownership of the GPU.\n                # The model stage scheduler alone grants the one GPU lease.\n                payload = {"job_id": job["job_id"], "attempt_id": job["attempt_id"],\n                           "instance_id": self.model_instance_id, "parameters": job["parameters"]}\n                try:\n                    response = self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "execute", payload, timeout=self.probe_timeout)\n                    if not isinstance(response, dict) or response.get("accepted") is not True:\n                        raise ValueError("Invalid execution acknowledgement")\n                    for field in ("job_id", "attempt_id", "instance_id"):\n                        if response.get(field) != payload[field]:\n                            raise ValueError("Execution acknowledgement identity mismatch")\n                except Exception as error:\n                    rejected = None\n                    if isinstance(error, ControlError):\n                        try:\n                            rejected = json.loads(str(error))\n                        except (ValueError, TypeError):\n                            pass\n                    if isinstance(rejected, dict) and rejected.get("guarantees_not_accepted") is True:\n                        with self._lock, self._db:\n                            if error.status in {409, 429}:\n                                self._transition(job, "queued", attempt_id=None, model_instance_id=None,\n                                                 started_at=None, model_lease_released=True, dispatch_state="deferred",\n                                                 dispatch_reason=rejected.get("reason"))\n                                return  # FIFO: do not bypass the head to fill a smaller task.\n                            cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=True)\n                            self._transition(job, "failed", error=str(error), finished_at=time.time(),\n                                             model_lease_released=True, cleanup_complete=cleaned,\n                                             cleanup_errors=cleanup_errors, dispatch_state="rejected")\n                    else:\n                        # A missing acknowledgement is not evidence of rejection.\n                        # Reconcile this exact attempt, never send execute twice.\n                        observed = self._model_probe(self.model_instance_id)\n                        records = ((observed or {}).get("attempts", []) + (observed or {}).get("results", []))\n                        matched = any(isinstance(item, dict) and all(item.get(k) == payload[k]\n                                      for k in ("job_id", "attempt_id", "instance_id")) for item in records)\n                        if not matched:\n                            self._stop_model(force=True, reason="execution_acknowledgement_uncertain")\n                            return\n                        with self._lock, self._db:\n                            job.update(dispatch_state="reconciled")\n                            self._store_job(job)\n                            self._event("job.dispatch_reconciled", job=job)\n                        self._model = observed\n                        self._consume_result(observed)\n                        return\n                else:\n                    with self._lock, self._db:\n                        job.update(dispatch_state="accepted")\n                        self._store_job(job)\n                        self._event("job.dispatch_accepted", job=job)\n                if time.monotonic() >= deadline:\n                    return\n\n    def _monitor(self):\n        try:\n            while not self._cancel.is_set():\n                self.refresh()\n                self._cancel.wait(self.poll_interval)\n        except BaseException as error:\n            LOG.exception("Main event supervision failed")\n            with self._lock:\n                self._error = "supervisor_failed: " + type(error).__name__\n                self._phase, self._draining = "failed", True\n            # Retain the control plane, but end uncertain model ownership.\n            with self._control_lock:\n                self._stop_model(force=True, reason="supervisor_failed")\n            try:\n                self._checkpoint()\n            except Exception:\n                LOG.exception("Could not export final supervisor failure")\n\n    def submit(self, payload):\n        job_id, parameters = payload.get("job_id"), payload.get("parameters")\n        if not isinstance(job_id, str) or not IDENTIFIER.fullmatch(job_id) or not isinstance(parameters, dict):\n            raise ControlError("Invalid job request", status=422)\n        try:\n            options = requested_options(parameters)\n            validate_generation_options(options, load_adapter_settings())\n        except (KeyError, ValueError, TypeError) as error:\n            raise ControlError(f"Invalid generation options: {error}", status=422) from None\n        refs = parameters.get("references")\n        if not isinstance(refs, list) or not 1 <= len(refs) <= 12:\n            raise ControlError("Invalid reference list", status=422)\n        input_root = (self.root / job_id / "inputs").resolve()\n        if not input_root.is_relative_to(self.root):\n            raise ControlError("Input directory escapes output root", status=422)\n        for ref in refs:\n            try:\n                path = Path(ref["path"]).resolve()\n                if not path.is_relative_to(input_root) or not path.is_file() or Path(ref["path"]).is_symlink():\n                    raise ValueError()\n            except (KeyError, TypeError, OSError, ValueError):\n                raise ControlError("References must be owned input files", status=422)\n        idem, fingerprint = payload.get("idempotency_key"), payload.get("fingerprint")\n        if idem is not None and (not isinstance(idem, str) or not 1 <= len(idem) <= 128 or not isinstance(fingerprint, str)):\n            raise ControlError("Invalid idempotency key or fingerprint", status=422)\n        with self._lock, self._db:\n            if idem is not None:\n                row = self._db.execute("SELECT fingerprint,record FROM jobs WHERE idempotency_key=?", (idem,)).fetchone()\n                if row:\n                    if row[0] != fingerprint:\n                        raise ControlError("Idempotency key conflicts with request", status=409)\n                    return self._public_job(json.loads(row[1]))\n            if self._draining or self._shutdown_started or self._phase in {"new", "failed", "stopped"} or self._model_failure:\n                raise ControlError("Generation admission is closed", status=409)\n            if shutil.disk_usage(self.root).free < self.disk_reserve_bytes + 65536:\n                raise ControlError("Insufficient disk space for durable queue metadata", status=507)\n            if self._db.execute("SELECT 1 FROM jobs WHERE id=?", (job_id,)).fetchone():\n                raise ControlError("Job ID already exists", status=409)\n            now = time.time()\n            job = {"job_id": job_id, "status": "queued", "created_at": now, "started_at": None,\n                   "finished_at": None, "attempt_id": None, "model_instance_id": None,\n                   "parameters": parameters, "error": None, "artifact": None, "metrics": None,\n                   "generation_options": {"requested": options, "effective": None},\n                    "cleanup_complete": False, "model_lease_released": True, "dispatch_state": "pending",\n                   "lifecycle": [{"state": "queued", "observed_at": now}]}\n            self._db.execute("INSERT INTO jobs VALUES (?,?,?,?,?)", (job_id, "queued", idem, fingerprint, json.dumps(job)))\n            self._event("job.queued", job=job)\n            return self._public_job(job)\n\n    def _public_job(self, job):\n        result = {k: v for k, v in job.items() if k != "parameters"}\n        # Public task identity belongs to this durable manager/API lifetime;\n        # model_instance_id separately identifies a replaceable GPU process.\n        result["instance_id"] = self.instance_id\n        result["queue_position"] = (self._db.execute("SELECT COUNT(*) FROM jobs WHERE status=\'queued\' AND rowid <= (SELECT rowid FROM jobs WHERE id=?)",\n                                                    (job["job_id"],)).fetchone()[0] if job["status"] == "queued" else None)\n        result["waiting_for"] = None\n        if job["status"] == "queued":\n            # This is a current observation, separate from the last durable\n            # dispatch rejection. Do not make clients infer it from a log line.\n            if self._shutdown_started:\n                result["waiting_for"] = "shutdown"\n            elif self._model_failure:\n                result["waiting_for"] = "model_recovery"\n            elif self._model.get("ready") is not True or self._model.get("worker_alive") is not True:\n                result["waiting_for"] = "model_ready"\n            elif self._model.get("accepting") is not True:\n                result["waiting_for"] = "model_capacity"\n            elif result["queue_position"] > 1:\n                result["waiting_for"] = "earlier_jobs"\n            else:\n                result["waiting_for"] = "scheduler"\n        result["active_phase"] = None\n        if job["status"] in {"running", "cancel_requested"}:\n            for attempt in self._model.get("attempts", []):\n                if attempt.get("attempt_id") == job["attempt_id"] and attempt.get("job_id") == job["job_id"]:\n                    result["active_phase"] = attempt.get("active_phase") or {"phase": attempt.get("phase", attempt.get("status"))}\n                    if isinstance(attempt.get("generation_options"), dict):\n                        result["generation_options"] = deepcopy(attempt["generation_options"])\n                    break\n            if result["active_phase"] is None and job["attempt_id"] == self._model.get("active_attempt_id"):\n                result["active_phase"] = self._model.get("active_phase")\n        return result\n\n    def archive(self, payload):\n        """Record a client-confirmed Drive upload for the committed artifact."""\n        job_id, file_id = payload.get("job_id"), payload.get("file_id")\n        sha256, size = payload.get("sha256"), payload.get("bytes")\n        if (not isinstance(job_id, str) or not IDENTIFIER.fullmatch(job_id)\n                or not isinstance(file_id, str) or not DRIVE_FILE_ID.fullmatch(file_id)\n                or not isinstance(sha256, str) or not SHA256.fullmatch(sha256)\n                or type(size) is not int or size <= 0):\n            raise ControlError("Invalid Google Drive archive receipt", status=422)\n        receipt = {"provider": "google_drive", "file_id": file_id, "sha256": sha256, "bytes": size}\n        with self._lock, self._db:\n            job = self._get_job(job_id)\n            if job["status"] != "succeeded":\n                raise ControlError("Only a succeeded job can be archived", status=409)\n            artifact = job.get("artifact")\n            media = artifact.get("media") if isinstance(artifact, dict) else None\n            if (not isinstance(media, dict) or media.get("full_decode_passed") is not True\n                    or artifact.get("sha256") != sha256 or artifact.get("bytes") != size):\n                raise ControlError("Archive receipt does not match the committed artifact", status=409)\n            previous = job.get("archive")\n            if previous is None:\n                job["archive"] = receipt\n                self._store_job(job)\n                self._event("job.archived", job=job, data={"provider": "google_drive"})\n            elif previous != receipt:\n                raise ControlError("Job already has a different archive receipt", status=409)\n            return self._public_job(job)\n\n    def cancel(self, job_id):\n        with self._control_lock:\n            with self._lock, self._db:\n                job = self._get_job(job_id)\n                if job["status"] in TERMINAL:\n                    return self._public_job(job)\n                if job["status"] == "queued":\n                    cleaned, cleanup_errors = self._cleanup_job_files(job_id, discard_result=True)\n                    self._transition(job, "cancelled", finished_at=time.time(), cleanup_complete=cleaned,\n                                     cleanup_errors=cleanup_errors, model_lease_released=True)\n                    return self._public_job(job)\n                if job["status"] == "running":\n                    self._transition(job, "cancel_requested")\n            try:\n                self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "cancel",\n                          {"job_id": job_id, "attempt_id": job["attempt_id"], "instance_id": job["model_instance_id"]},\n                          timeout=self.probe_timeout)\n            except Exception:\n                pass  # Request remains pending; execution lease is not released.\n            return self._public_job(job)\n\n    def _terminate(self, name, timeout=15):\n        child = self.children[name]\n        if child is None:\n            return True\n        owner = self.owners.get(name)\n        if owner:\n            self._register_process_tree()\n            return self._record_tree_exit(name, owner.terminate_all(timeout))\n        try:\n            if child.poll() is None:\n                child.terminate()\n                try:\n                    child.wait(timeout=timeout)\n                except subprocess.TimeoutExpired:\n                    child.kill()\n                    child.wait(timeout=3)\n            exited = child.poll() is not None\n            if exited and hasattr(child, "_h3_launch_id"):\n                with self._lock, self._db:\n                    self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (child._h3_launch_id,))\n            return exited\n        except (OSError, subprocess.SubprocessError) as error:\n            self._cleanup_errors.append(name + ": " + type(error).__name__)\n            return False\n\n    def _record_tree_exit(self, name, observed):\n        if observed["cleanup_complete"]:\n            with self._lock, self._db:\n                for launch_id in self.children[name]._h3_tree_records.values():\n                    self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (launch_id,))\n            return True\n        self._cleanup_errors.append(name + ".tree: " + "; ".join(observed["errors"] or ["exit unconfirmed"]))\n        return False\n\n    def _stop_model(self, *, force=False, reason="model_stopped"):\n        with self._lock:\n            active = self._active_jobs()\n            queued = self._db.execute("SELECT COUNT(*) FROM jobs WHERE status=\'queued\'").fetchone()[0]\n            if (active or queued) and not force:\n                raise ControlError("Model has unfinished jobs; drain or explicitly force stop", status=409)\n            self._model_failure = reason\n            self._model["ready"] = False\n        if self._terminate("model"):\n            self._fail_jobs_after_exit(reason)\n            downloads = self._model.get("downloads")\n            self._model = {"instance_id": self.model_instance_id, "phase": "stopped", "ready": False,\n                           "worker_alive": False, "cleanup_complete": True}\n            if isinstance(downloads, dict):\n                self._model["downloads"] = {**deepcopy(downloads), "observation_stale": True}\n            with self._lock, self._db:\n                self._event("model.stopped", data={"reason": reason, "cleanup_complete": True})\n        else:\n            raise ControlError("Model exit is unconfirmed; lease retained", status=503)\n\n    def _control_action(self, action, payload):\n        with self._control_lock:\n            if action == "shutdown":\n                state = self.shutdown()\n                if (state.get("cleanup_complete") is not True or state.get("process_observation_errors")\n                        or any(state.get(name) is not False for name in ("server_alive", "model_alive", "tunnel_alive"))):\n                    raise ControlError("Shutdown cleanup is unconfirmed; manager remains available for retry", status=503)\n            elif action == "stop_model":\n                self._stop_model(force=payload.get("force") is True)\n            elif action == "start_model":\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\n                    raise ControlError("Manager is stopping or failed", status=409)\n                self._launch_model()\n            elif action == "stop_api":\n                with self._lock, self._db:\n                    self._set_api_expectation(False)\n                    self._api_intent_epoch += 1\n                    self._api_lifecycle = "stopping"\n                    self._api_ready = self._public_api_ready = False\n                    if self._error and self._error.startswith("api_"):\n                        self._error = None\n                    self._event("api.stop_requested", data={"intent_epoch": self._api_intent_epoch})\n                if not self._terminate("api"):\n                    self._api_lifecycle = "stop_unconfirmed"\n                    raise ControlError("API process exit is unconfirmed", status=503)\n                self._api_lifecycle = "stopped"\n                with self._lock, self._db:\n                    self._event("api.stopped", data={"intent_epoch": self._api_intent_epoch, "cleanup_complete": True})\n            elif action == "start_api":\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\n                    raise ControlError("Manager cannot start API", status=409)\n                self._launch_api()\n            elif action == "drain":\n                self._draining = True  # Existing queue continues to drain.\n            elif action == "resume":\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\n                    raise ControlError("Manager cannot resume", status=409)\n                self._draining = False\n            else:\n                raise ControlError("Unknown control action", status=422)\n            self._checkpoint()\n\n    def control(self, payload):\n        action = payload.get("action")\n        if action not in {"drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"}:\n            raise ControlError("Unknown control action", status=422)\n        command_id = payload.get("command_id") or uuid.uuid4().hex\n        if not isinstance(command_id, str) or not 1 <= len(command_id) <= 128:\n            raise ControlError("Invalid command ID", status=422)\n        fingerprint = json.dumps({"action": action, "force": payload.get("force") is True}, sort_keys=True)\n        record = {"command_id": command_id, "action": action, "status": "accepted", "created_at": time.time(), "error": None}\n\n        def run():\n            try:\n                self._control_action(action, payload)\n                result = {**record, "status": "succeeded", "finished_at": time.time()}\n            except BaseException as error:\n                LOG.exception("Control operation failed")\n                result = {**record, "status": "failed", "error": str(error) if isinstance(error, ControlError) else type(error).__name__,\n                          "finished_at": time.time()}\n            try:\n                with self._lock, self._db:\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(result), command_id))\n                    self._event("command." + result["status"], data={"command_id": command_id, "action": action, "error": result.get("error")})\n                self._checkpoint()\n                if action == "shutdown" and result["status"] == "succeeded":\n                    self._exit_requested = True\n            finally:\n                self._control_slots.release()\n\n        thread = threading.Thread(target=run, name="main-control-" + action, daemon=True)\n        with self._lock:\n            existing = self._db.execute("SELECT fingerprint,record FROM commands WHERE id=?", (command_id,)).fetchone()\n            if existing:\n                if existing[0] != fingerprint:\n                    raise ControlError("Command ID conflicts", status=409)\n                return json.loads(existing[1])\n            if self._control_admission_closed or (self._shutdown_started and action != "shutdown"):\n                raise ControlError("Manager is closing; new control operations are unavailable", status=409)\n            if not self._control_slots.acquire(blocking=False):\n                raise ControlError("Control operation queue is full", status=429)\n            try:\n                with self._db:\n                    self._db.execute("INSERT INTO commands VALUES (?,?,?)", (command_id, fingerprint, json.dumps(record)))\n                    self._event("command.accepted", data={"command_id": command_id, "action": action})\n            except BaseException:\n                self._control_slots.release()\n                raise\n            # Admission, registration and start share close()\'s barrier. An\n            # accepted command can never appear after its worker join snapshot.\n            self._control_threads = [t for t in self._control_threads if t.ident is None or t.is_alive()]\n            self._control_threads.append(thread)\n            try:\n                thread.start()\n            except BaseException:\n                self._control_slots.release()\n                with self._db:\n                    record.update(status="failed", error="command_worker_start_failed", finished_at=time.time())\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(record), command_id))\n                    self._event("command.failed", data={"command_id": command_id, "error": record["error"]})\n                raise\n        return record\n\n    def dispatch(self, operation, payload=None):\n        payload = payload or {}\n        if operation in {"submit", "cancel", "control", "archive"} and payload.get("expected_instance_id", self.instance_id) != self.instance_id:\n            raise ControlError("Manager instance changed; command rejected", status=409)\n        if operation == "status":\n            return self.snapshot()\n        if operation == "submit":\n            return self.submit(payload)\n        if operation == "job":\n            with self._lock:\n                return self._public_job(self._get_job(payload.get("job_id")))\n        if operation == "archive":\n            return self.archive(payload)\n        if operation == "cancel":\n            return self.cancel(payload.get("job_id"))\n        if operation == "control":\n            return self.control(payload)\n        if operation == "command":\n            with self._lock:\n                row = self._db.execute("SELECT record FROM commands WHERE id=?", (payload.get("command_id"),)).fetchone()\n                if not row:\n                    raise ControlError("Command not found", status=404)\n                return json.loads(row[0])\n        if operation == "events":\n            try:\n                after, limit = max(0, int(payload.get("after", 0))), min(500, max(1, int(payload.get("limit", 100))))\n            except (TypeError, ValueError):\n                raise ControlError("Invalid event cursor", status=422)\n            with self._lock:\n                rows = self._db.execute("SELECT sequence,record FROM events WHERE sequence>? ORDER BY sequence LIMIT ?", (after, limit)).fetchall()\n            events = [{**json.loads(encoded), "sequence": seq} for seq, encoded in rows]\n            return {"events": events, "next_cursor": events[-1]["sequence"] if events else after}\n        raise ControlError("Unknown manager operation", status=404)\n\n    def shutdown(self):\n        with self._control_lock:\n            self._shutdown_started, self._draining, self._phase = True, True, "stopping"\n            with self._lock:\n                self._set_api_expectation(False)\n                self._api_intent_epoch += 1\n                self._api_lifecycle = "stopping"\n                self._api_ready = self._public_api_ready = False\n            self._cancel.set()\n            self._register_process_tree()\n            names = [name for name in ("tunnel", "api", "model") if name in self.owners]\n            results = stop_many([self.owners[name] for name in names], timeout=15)\n            for name, observed in zip(names, results):\n                self._record_tree_exit(name, observed)\n            alive, errors = self._liveness()\n            self._api_lifecycle = "stopped" if not alive["api"] and "api" not in errors else "stop_unconfirmed"\n            if not alive["model"] and "model" not in errors:\n                self._fail_jobs_after_exit("manager_stopped")\n            with self._lock:\n                historical = self._historical_ownership()\n            self._phase = "failed" if any(alive.values()) or errors or historical else "stopped"\n            if any(alive.values()) or errors or historical:\n                self._error = "cleanup_incomplete; owned processes remain"\n            elif self._error in {"cleanup_incomplete; owned processes remain", "audit_export_failed_during_shutdown"}:\n                self._error = None\n            try:\n                self._checkpoint()\n            except (OSError, sqlite3.Error):\n                self._error = "audit_export_failed_during_shutdown"\n                LOG.exception("Could not export shutdown audit")\n            return self.snapshot()\n\n    def close(self):\n        with self._lock:\n            if self._closed:\n                return\n            self._control_admission_closed = True\n        self.shutdown()\n        if self._thread and self._thread is not threading.current_thread() and self._thread.ident is not None:\n            self._thread.join(timeout=3 * self.probe_timeout + self.poll_interval + 2)\n            if self._thread.is_alive():\n                raise RuntimeError("Cannot finalize shutdown with a live supervisor")\n        for thread in self._control_threads:\n            if thread is not threading.current_thread() and thread.ident is not None:\n                thread.join(timeout=20)\n                if thread.is_alive():\n                    raise RuntimeError("Cannot finalize shutdown with a live control operation")\n        self._probe_pool.shutdown(wait=True, cancel_futures=True)\n        alive, errors = self._liveness()\n        if any(value is not False for value in alive.values()) or errors or self._historical_ownership():\n            raise RuntimeError("Cannot release root ownership with live children")\n        with self._lock:\n            self._shutdown_complete = True\n            try:\n                with self._db:\n                    self._event("manager.shutdown_complete")\n                self._checkpoint()\n            except BaseException:\n                self._shutdown_complete = False\n                raise\n            self._db.close()\n            self._owner.close()\n            self._closed = True\n        for output in self._files:\n            output.close()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for argument in ("port", "api-port", "model-port"):\n        parser.add_argument("--" + argument, required=True, type=int)\n    parser.add_argument("--directory", required=True, type=Path)\n    parser.add_argument("--root", required=True, type=Path)\n    parser.add_argument("--instance-id", required=True)\n    parser.add_argument("--staged", action="store_true")\n    parser.add_argument("--public", action="store_true")\n    parser.add_argument("--install-cloudflared", action="store_true")\n    parser.add_argument("--no-model-autostart", action="store_true",\n                        help="Start API/public transport only; start_model control can launch the model later")\n    parser.add_argument("--parent-pid", type=int)\n    parser.add_argument("--startup-timeout", type=float, default=3600)\n    parser.add_argument("--generation-timeout", type=float, default=5400)\n    parser.add_argument("--poll-interval", type=float, default=1)\n    parser.add_argument("--probe-timeout", type=float, default=2)\n    parser.add_argument("--tunnel-timeout", type=float, default=180)\n    args = parser.parse_args()\n    manager = GlobalManager(args.directory, root=args.root, instance_id=args.instance_id,\n                            port=args.port, api_port=args.api_port, model_port=args.model_port,\n                            staged=args.staged, public=args.public, install_cloudflared=args.install_cloudflared,\n                            start_model=not args.no_model_autostart,\n                            startup_timeout=args.startup_timeout, generation_timeout=args.generation_timeout,\n                            poll_interval=args.poll_interval, probe_timeout=args.probe_timeout, tunnel_timeout=args.tunnel_timeout)\n    server = make_server("127.0.0.1", args.port, manager.key, manager.dispatch)\n    serve_thread = threading.Thread(target=server.serve_forever, name="manager-control-http", daemon=True)\n    stop_requested = False\n\n    def request_stop(*_):\n        # Python signal handlers can interrupt/re-enter Event.set while its\n        # non-reentrant Condition lock is held. Only assign a flag here.\n        nonlocal stop_requested\n        stop_requested = True\n    parent_identity = None\n    if args.parent_pid:\n        try:\n            parent_identity = float(os.environ.get("H3_PARENT_CREATED_AT") or psutil.Process(args.parent_pid).create_time())\n        except (psutil.Error, ValueError):\n            stop_requested = True\n    signal.signal(signal.SIGTERM, request_stop)\n    signal.signal(signal.SIGINT, request_stop)\n    try:\n        serve_thread.start()\n        if stop_requested:\n            raise RuntimeError("Manager owner identity is unavailable")\n        manager.start()\n        while not stop_requested and not manager._exit_requested:\n            time.sleep(.1)\n            if args.parent_pid:\n                try:\n                    parent = psutil.Process(args.parent_pid)\n                    if parent_identity is None or abs(parent.create_time() - parent_identity) >= .001 or not parent.is_running():\n                        break\n                except psutil.Error:\n                    break\n    finally:\n        manager.close()\n        server.shutdown()\n        server.server_close()\n        serve_thread.join(timeout=3)\n\n\nif __name__ == "__main__":\n    logging.basicConfig(level=logging.INFO)\n    main()\n', 'model_container.py': '"""Private, independently supervised MiniMax-H3 model runtime.\n\nThe HTTP controller never imports torch or loads weights. The supervised owner\nloads after binding its socket. Budgeted CPU stages overlap across requests and\nshare one exclusive GPU lease. Production backends retain native BF16; injected\nbackends are only a test seam.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nfrom contextlib import contextmanager\nfrom fractions import Fraction\nimport gc\nimport hashlib\nimport json\nimport logging\nimport math\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport shutil\nimport signal\nimport sqlite3\nimport subprocess\nimport sys\nimport threading\nimport time\nimport uuid\n\nfrom control_protocol import ControlError, make_server\nfrom condition_guard import evaluate_gate, negate_observation, observation_state\nfrom download_progress import read_progress\nfrom pipeline_scheduler import BudgetUnavailable, PipelineBudget\n\nLOG = logging.getLogger("h3.model_container")\nIDENTIFIER = re.compile(r"[a-f0-9]{32}\\Z")\nTERMINAL = {"succeeded", "failed", "cancelled"}\nSOURCE_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nEXECUTE_CONDITIONS = ("instance_matches", "request_validated", "model_loaded", "worker_alive",\n    "capability_available", "not_draining", "not_stopping", "attempt_available", "job_unowned",\n    "artifact_absent", "generation_options_valid", "resources_reserved")\nSTAGE_CONDITIONS = ("instance_matches", "attempt_owned", "attempt_live", "model_loaded", "worker_alive",\n    "capability_available", "admission_confirmed", "predecessor_completed", "stage_not_started",\n    "cancel_flag_clear", "cancel_event_clear", "reservation_owned", "budget_grant_held")\n\n\nclass ExecutionCancelled(Exception):\n    pass\n\n\nclass ConditionGateBlocked(RuntimeError):\n    """Missing or contradictory evidence is a failed execution, not cancellation."""\n    pass\n\n\ndef _all_observed(values):\n    """Conjoin distinct obligations; a missing obligation is not a spare witness."""\n    states = [observation_state(value) for value in values]\n    if "conflict" in states:\n        return "conflict"\n    if "unknown" in states:\n        return None\n    return all(state == "true" for state in states)\n\n\ndef parent_identity(parent_pid):\n    """Bind ownership to one OS process generation, including on Windows."""\n    import psutil\n    try:\n        process = psutil.Process(parent_pid)\n        # A Windows venv redirector may be the OS parent. The authenticated\n        # launcher specifies the actual owner; bind to that process generation.\n        if not process.is_running() or process.status() == psutil.STATUS_ZOMBIE:\n            raise RuntimeError("The requested model owner is not alive")\n        return process.create_time()\n    except psutil.Error as error:\n        raise RuntimeError("Cannot establish the model parent process identity") from error\n\n\ndef parent_is_alive(parent_pid, created_at):\n    import psutil\n    try:\n        process = psutil.Process(parent_pid)\n        return (process.is_running() and process.create_time() == created_at\n                and process.status() != psutil.STATUS_ZOMBIE)\n    except psutil.Error:\n        # Unverified ownership must not retain a costly unmanaged model worker.\n        return False\n\n\ndef verify_artifact(path: Path, parameters: dict, *, cpu_threads=1) -> dict:\n    """Validate the entire audiovisual artifact before publishing success."""\n    if path.is_symlink() or not path.is_file() or not path.stat().st_size:\n        raise ValueError("Missing model artifact")\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-threads", str(cpu_threads), "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)],\n        capture_output=True, text=True, timeout=120, check=True,\n    )\n    data = json.loads(probe.stdout)\n    streams = data.get("streams", [])\n    video = next((x for x in streams if x.get("codec_type") == "video"), None)\n    audio = next((x for x in streams if x.get("codec_type") == "audio"), None)\n    if video is None or audio is None:\n        raise ValueError("Artifact must contain video and audio")\n    frames = int(video.get("nb_read_frames") or video.get("nb_frames") or 0)\n    fps = float(Fraction(video.get("avg_frame_rate", "0")))\n    duration = float(video.get("duration") or data.get("format", {}).get("duration") or 0)\n    expected_frames = 5 + 17 * math.ceil((round(parameters["seconds"] * 24) - 5) / 17)\n    if ((video.get("width"), video.get("height")) != (parameters["width"], parameters["height"])\n            or video.get("codec_name") != "h264" or frames != expected_frames\n            or abs(fps - 24) > .01 or not math.isfinite(duration)\n            or abs(duration - expected_frames / 24) > .1 or int(audio.get("channels", 0)) != 2):\n        raise ValueError("Artifact media does not match the admitted request")\n    subprocess.run(\n        ["ffmpeg", "-nostdin", "-v", "error", "-xerror", "-threads", str(cpu_threads),\n         "-filter_threads", "1", "-filter_complex_threads", "1", "-i", str(path),\n         "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"],\n        capture_output=True, text=True, timeout=300, check=True,\n    )\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return {"path": str(path), "bytes": path.stat().st_size, "sha256": digest.hexdigest(),\n            "media": {"video_codec": video["codec_name"], "width": video["width"], "height": video["height"],\n                      "decoded_frames": frames, "fps": fps, "duration_seconds": duration,\n                      "audio_codec": audio.get("codec_name"), "audio_channels": audio["channels"],\n                      "audio_sample_rate": audio.get("sample_rate"), "full_decode_passed": True}}\n\n\nclass ModelContainer:\n    def __init__(self, root, instance_id, *, backend=None, staged=False,\n                 artifact_validator=verify_artifact, resource_limits=None, progress_path=None):\n        if not isinstance(instance_id, str) or not IDENTIFIER.fullmatch(instance_id):\n            raise ValueError("Model instance must be a 32-character hexadecimal ID")\n        self.root = Path(root).resolve()\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.instance_id, self.staged = instance_id, bool(staged)\n        self.backend = backend\n        self._injected_backend = backend is not None\n        self._resource_limits = resource_limits\n        self._pipeline = False\n        self._budget = None\n        self._stage_workers = {}\n        self._results = {}\n        self.artifact_validator = artifact_validator\n        self.event_directory = self.root / "_model_runtime" / instance_id\n        if self.event_directory.resolve() != self.event_directory:\n            raise ValueError("Model audit directory must not traverse a symlink")\n        self.event_directory.mkdir(parents=True, exist_ok=True)\n        self.events_path = self.event_directory / "events.jsonl"\n        self.progress_path = Path(progress_path) if progress_path else self.event_directory / "downloads.json"\n        self._lock = threading.RLock()\n        self._queue = queue.Queue()\n        self._worker = None\n        self._done = threading.Event()\n        self._stop_requested = False\n        self._draining = False\n        self._loaded = False\n        self._phase = "loading"\n        self._error = None\n        self._active = None\n        self._last_result = None\n        self._attempts = {}\n        self._sequence = 0\n        self._consecutive_failures = 0\n        self._worker_cleanup_complete = False\n        self._last_condition_gate = None\n        self._ledger_path = self.event_directory / "attempts.sqlite3"\n        ledger = sqlite3.connect(self._ledger_path)\n        try:\n            with ledger:\n                ledger.execute("CREATE TABLE IF NOT EXISTS acknowledged (attempt_id TEXT PRIMARY KEY, record TEXT NOT NULL)")\n        finally:\n            ledger.close()\n\n    def _previous_attempt(self, attempt_id):\n        if attempt_id in self._attempts:\n            return self._attempts[attempt_id]\n        # Open briefly so acknowledgement does not retain a lifetime DB handle.\n        ledger = sqlite3.connect(self._ledger_path)\n        try:\n            row = ledger.execute("SELECT record FROM acknowledged WHERE attempt_id=?", (attempt_id,)).fetchone()\n            return json.loads(row[0]) if row else None\n        finally:\n            ledger.close()\n\n    def _live_attempts(self):\n        return [value for value in self._attempts.values() if value["status"] not in TERMINAL]\n\n    def _event(self, kind, **data):\n        with self._lock:\n            self._sequence += 1\n            event = {"sequence": self._sequence, "event_id": uuid.uuid4().hex,\n                     "instance_id": self.instance_id, "type": kind, "observed_at": time.time(),\n                     "monotonic_ns": time.monotonic_ns(), **data}\n            try:\n                with self.events_path.open("a", encoding="utf-8") as output:\n                    output.write(json.dumps(event, ensure_ascii=False) + "\\n")\n                    output.flush()\n                    os.fsync(output.fileno())\n            except OSError as error:\n                self._loaded = False\n                self._phase = "failed"\n                self._error = "model_audit_unavailable"\n                raise ControlError("Model audit storage unavailable", status=503) from error\n\n    def _safe_event(self, kind, **data):\n        try:\n            self._event(kind, **data)\n        except BaseException:\n            LOG.exception("Could not persist model event %s", kind)\n\n    def _gate(self, gate_id, required, observations, *, job_id, attempt_id, attempt=None, **context):\n        """Caller owns _lock; persist the decision before permitting its effect."""\n        receipt = evaluate_gate(gate_id, required, observations, {\n            "instance_id": self.instance_id, "job_id": job_id, "attempt_id": attempt_id,\n            "source_file": "model_container.py", "source_sha256": SOURCE_SHA256, **context})\n        self._event("condition.gate", job_id=job_id, attempt_id=attempt_id,\n                    gate_id=gate_id, receipt=receipt)\n        self._last_condition_gate = receipt\n        if attempt is not None:\n            attempt.setdefault("condition_gates", {})[gate_id] = receipt\n        return receipt\n\n    def _capability_observation(self, *, admitted=False):\n        if type(self._phase) is not str:\n            return None\n        allowed = {"ready", "running", "draining"} if admitted else {"ready", "running"}\n        if self._phase in allowed:\n            return True\n        if self._phase in {"loading", "starting", "failed", "stopped", "draining"}:\n            return False\n        return None\n\n    def start(self):\n        with self._lock:\n            if self._worker is not None:\n                raise ControlError("Model container has already started", status=409)\n            self._event("model.loading")\n            self._worker = threading.Thread(target=self._work, name="h3-model-worker", daemon=False)\n            try:\n                self._worker.start()\n            except BaseException:\n                self._phase, self._error = "failed", "worker_start_failed"\n                self._loaded = False\n                self._done.set()\n                self._safe_event("model.failed", error=self._error)\n                raise\n        return self\n\n    def status(self):\n        with self._lock:\n            alive = self._worker is not None and self._worker.is_alive()\n            if self._worker is not None and not alive and self._phase not in {"failed", "stopped"}:\n                self._phase, self._error, self._loaded = "failed", "worker_exited", False\n            ready = self._loaded is True and alive and self._capability_observation() is True\n            downloads = read_progress(self.progress_path, expected_instance_id=self.instance_id)\n            downloads.update(model_ready=ready, lifecycle=self._phase,\n                             observation_stale=bool(downloads.get("observation_stale") or not alive))\n            phase = None\n            snapshot = getattr(self.backend, "phase_snapshot", None)\n            if snapshot:\n                try:\n                    phase = snapshot()\n                except Exception:\n                    phase = None\n            active = self._active\n            live = self._live_attempts()\n            # Keep each reply below the private protocol\'s bounded body. Results\n            # remain owned until ack; additional pages surface after that ack.\n            results, result_bytes = [], 0\n            for result in self._results.values():\n                size = len(json.dumps(result).encode())\n                if results and result_bytes + size > 1024 * 1024:\n                    break\n                results.append(result)\n                result_bytes += size\n            return copy.deepcopy({\n                "instance_id": self.instance_id, "phase": self._phase, "ready": ready,\n                "accepting": ready and self._draining is False and self._stop_requested is False and (\n                    self._budget.has_capacity() if self._pipeline else active is None),\n                "pipeline": self._pipeline,\n                "downloads": downloads,\n                "budget": self._budget.snapshot() if self._budget else None,\n                "attempts": [{**self._accepted(item), "phase": item.get("stage", item["status"]),\n                              "generation_options": item.get("generation_options"),\n                              "condition_gates": item.get("condition_gates", {}),\n                              "stage_completions": item.get("stage_completions", {}),\n                              "cancel_requested": item["cancel_requested"]} for item in live],\n                "results": results, "pending_results": len(self._results),\n                "active_job_id": active["job_id"] if active else None,\n                "active_attempt_id": active["attempt_id"] if active else None,\n                "cancel_requested": bool(active and active["cancel_requested"]),\n                "active_phase": phase, "last_result": self._last_result,\n                "worker_alive": alive, "cleanup_complete": False,\n                "worker_cleanup_complete": self._worker_cleanup_complete,\n                "cleanup_scope": "Process exit must be confirmed by the owner before GPU ownership is released",\n                "error": self._error, "observed_at": time.time(), "event_sequence": self._sequence,\n                "last_condition_gate": self._last_condition_gate,\n            })\n\n    def _check_instance(self, payload, required=False):\n        instance = payload.get("instance_id")\n        if instance != self.instance_id and (required or instance is not None):\n            raise ControlError("Model instance does not match this execution generation", status=409)\n\n    def _request(self, payload):\n        self._check_instance(payload, required=True)\n        job_id, attempt_id = payload.get("job_id"), payload.get("attempt_id")\n        if any(not isinstance(x, str) or not IDENTIFIER.fullmatch(x) for x in (job_id, attempt_id)):\n            raise ControlError("Invalid job or attempt identity", status=422)\n        parameters = payload.get("parameters")\n        names = {"prompt", "references", "seconds", "width", "height", "steps", "seed"}\n        optional = {"attention_backend", "acceleration_lora"}\n        if not isinstance(parameters, dict) or not names <= set(parameters) or set(parameters) - names - optional:\n            raise ControlError("Invalid generation parameters", status=422)\n        p = copy.deepcopy(parameters)\n        try:\n            from adapter_config import load_adapter_settings\n            from generation_options import requested_options, validate_generation_options\n            validate_generation_options(requested_options(p), load_adapter_settings())\n        except (TypeError, ValueError) as error:\n            raise ControlError(str(error), status=422) from None\n        if not isinstance(p["prompt"], str) or not p["prompt"].strip() or len(p["prompt"]) > 4000:\n            raise ControlError("Invalid prompt", status=422)\n        if any(isinstance(p[k], bool) or not isinstance(p[k], int) for k in ("width", "height", "steps", "seed")):\n            raise ControlError("Generation dimensions, steps and seed must be integers", status=422)\n        if (not 256 <= p["width"] <= 1344 or not 256 <= p["height"] <= 1344\n                or p["width"] % 32 or p["height"] % 32 or p["width"] * p["height"] > 1344 * 768\n                or not 2 <= p["steps"] <= 50 or not 0 <= p["seed"] <= 2147483647):\n            raise ControlError("Generation parameters outside resource limits", status=422)\n        seconds = p["seconds"]\n        if (isinstance(seconds, bool) or not isinstance(seconds, (int, float)) or not math.isfinite(seconds)\n                or not 5 <= seconds <= 15 or 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17) > 360):\n            raise ControlError("Invalid generation duration", status=422)\n        refs = p["references"]\n        if not isinstance(refs, list) or not 1 <= len(refs) <= 12:\n            raise ControlError("Supply 1-12 references", status=422)\n        directory = self.root / job_id\n        inputs = directory / "inputs"\n        if directory.resolve() != directory or inputs.resolve() != inputs:\n            raise ControlError("Reference directory must stay within its assigned job", status=422)\n        kinds = []\n        total = 0\n        for ref in refs:\n            if not isinstance(ref, dict) or set(ref) != {"kind", "path"} or ref["kind"] not in {"image", "video", "audio"}:\n                raise ControlError("Invalid reference metadata", status=422)\n            try:\n                path = Path(ref["path"])\n                resolved = path.resolve(strict=True)\n                size = resolved.stat().st_size\n                valid = (path.is_absolute() and resolved.is_relative_to(inputs) and path == resolved\n                         and resolved.is_file() and not path.is_symlink() and 0 < size <= 32 * 1024 * 1024)\n            except (OSError, TypeError, ValueError):\n                valid = False\n            if not valid:\n                raise ControlError("Reference path is missing, oversized or outside its assigned inputs", status=422)\n            total += size\n            ref["path"] = str(resolved)\n            kinds.append(ref["kind"])\n        if (total > 64 * 1024 * 1024 or kinds.count("image") > 9 or kinds.count("video") > 3\n                or kinds.count("audio") > 3 or all(k == "audio" for k in kinds)):\n            raise ControlError("Reference combination outside resource limits", status=422)\n        fingerprint = hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()\n        return job_id, attempt_id, p, fingerprint\n\n    def execute(self, payload):\n        # Idempotent replies must remain possible after inputs have been cleaned.\n        self._check_instance(payload, required=True)\n        attempt_id = payload.get("attempt_id")\n        with self._lock:\n            old = self._previous_attempt(attempt_id) if isinstance(attempt_id, str) else None\n            if old:\n                candidate = hashlib.sha256(json.dumps(payload.get("parameters"), sort_keys=True).encode()).hexdigest()\n                if old["job_id"] != payload.get("job_id") or old["fingerprint"] != candidate:\n                    raise ControlError("Attempt identity was reused with different parameters", status=409)\n                return self._accepted(old)\n        try:\n            job_id, attempt_id, parameters, fingerprint = self._request(payload)\n        except ControlError as error:\n            if error.status == 422:\n                raise ControlError(json.dumps({"guarantees_not_accepted": True,\n                                               "reason": "invalid_request", "detail": str(error)}), 422) from None\n            raise\n        # Header probes can perform bounded I/O. Do not hold the controller\'s\n        # state lock while estimating an otherwise unadmitted request.\n        resources = None\n        with self._lock:\n            estimator = self.backend if self._pipeline and self._loaded is True and self._draining is False else None\n        if estimator is not None:\n            try:\n                resources = estimator.estimate_resources(parameters)\n            except ValueError:\n                raise ControlError(json.dumps({"guarantees_not_accepted": True,\n                                               "reason": "invalid_reference_media"}), status=422) from None\n        with self._lock:\n            old = self._previous_attempt(attempt_id)\n            if old:\n                if old["job_id"] != job_id or old["fingerprint"] != fingerprint:\n                    raise ControlError("Attempt identity conflict", status=409)\n                return self._accepted(old)\n            output = self.root / job_id / "result.mp4"\n            validator = getattr(self.backend, "validate_request", None)\n            options_valid = True\n            if validator:\n                try:\n                    validator(steps=parameters["steps"], **{name: parameters[name]\n                        for name in ("attention_backend", "acceleration_lora") if name in parameters})\n                except ValueError:\n                    options_valid = False\n            observations = {\n                "instance_matches": payload.get("instance_id") == self.instance_id,\n                "request_validated": True,\n                "model_loaded": self._loaded,\n                "worker_alive": self._worker.is_alive() if self._worker is not None else False,\n                "capability_available": self._capability_observation(),\n                "not_draining": negate_observation(self._draining),\n                "not_stopping": negate_observation(self._stop_requested),\n                "attempt_available": old is None,\n                "job_unowned": not any(item["job_id"] == job_id for item in self._live_attempts()),\n                "artifact_absent": not (output.exists() or output.is_symlink()),\n                "generation_options_valid": options_valid,\n                "resources_reserved": None,\n            }\n            context = {"pipeline": self._pipeline, "budget_scope": "stage_budget" if self._pipeline else "legacy_single_active"}\n            if any(observation_state(value) != "true" for key, value in observations.items() if key != "resources_reserved"):\n                receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\n                    job_id=job_id, attempt_id=attempt_id, **context)\n                raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": "model_conditions_blocked",\n                    "decision_id": receipt["decision_id"], "blocked": receipt["blocked"]}), 409 if options_valid else 422)\n            attempt = {"job_id": job_id, "attempt_id": attempt_id, "instance_id": self.instance_id,\n                       "status": "accepted", "cancel_requested": False, "parameters": parameters,\n                       "fingerprint": fingerprint, "accepted_at": time.time(), "condition_gates": {},\n                       "stage_started": {"prepare": False, "gpu": False, "encode_validate": False},\n                       "stage_completions": {"prepare": None, "gpu": None, "encode_validate": None}}\n            reserved = False\n            if self._pipeline:\n                if resources is None:\n                    receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\n                        job_id=job_id, attempt_id=attempt_id, **context)\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True,\n                                                   "reason": "model_not_accepting", "decision_id": receipt["decision_id"]}), status=409)\n                try:\n                    self._budget.reserve(attempt_id, resources)\n                except BudgetUnavailable as error:\n                    self._event("reservation.blocked", job_id=job_id, attempt_id=attempt_id,\n                                reservation_witness=self._budget.reservation_witness(attempt_id),\n                                reservation_receipt=self._budget.gate_receipt(attempt_id, "reserve"))\n                    observations["resources_reserved"] = False\n                    receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\n                        job_id=job_id, attempt_id=attempt_id, budget_reason=str(error), **context)\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": str(error),\n                                                   "decision_id": receipt["decision_id"]}),\n                                       status=507 if error.permanent else 429) from None\n                reserved = True\n                attempt["resources"] = resources\n                attempt["_cancel_event"] = threading.Event()\n            try:\n                witness = self._budget.reservation_witness(attempt_id) if self._pipeline else None\n                if self._pipeline:\n                    self._event("reservation.acquired", job_id=job_id, attempt_id=attempt_id,\n                                reservation_witness=witness,\n                                reservation_receipt=self._budget.gate_receipt(attempt_id, "reserve"))\n                observations["resources_reserved"] = (_all_observed([\n                    witness.get("owned"), witness.get("attempt_id") == attempt_id,\n                    witness.get("budget_instance_id") == self._budget.instance_id,\n                    type(witness.get("reservation_id")) is str and bool(witness["reservation_id"])\n                ]) if witness is not None else self._active is None if not self._pipeline else None)\n                receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\n                    job_id=job_id, attempt_id=attempt_id, attempt=attempt,\n                    budget_instance_id=witness.get("budget_instance_id") if witness else None,\n                    reservation_id=witness.get("reservation_id") if witness else None, **context)\n                if receipt["allowed"] is not True:\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": "model_conditions_blocked",\n                        "decision_id": receipt["decision_id"], "blocked": receipt["blocked"]}), 409)\n            except BaseException:\n                if reserved:\n                    self._budget.finish(attempt_id)\n                raise\n            self._attempts[attempt_id] = attempt\n            if not self._pipeline:\n                self._active = attempt\n            self._phase = "running"\n            try:\n                self._event("execution.accepted", job_id=job_id, attempt_id=attempt_id,\n                            resources=attempt.get("resources"), condition_decision_id=receipt["decision_id"])\n                self._queue.put_nowait(attempt)\n            except BaseException:\n                attempt.update(status="failed", error="dispatch_failed")\n                self._active = None\n                if self._budget:\n                    self._budget.finish(attempt_id)\n                self._loaded, self._phase, self._error = False, "failed", "dispatch_failed"\n                raise\n            return self._accepted(attempt)\n\n    def _accepted(self, attempt):\n        return copy.deepcopy({key: attempt[key] for key in ("job_id", "attempt_id", "instance_id", "status")}\n                             | {"accepted": True, "result": attempt.get("result")})\n\n    def cancel(self, payload):\n        self._check_instance(payload, required=True)\n        if not isinstance(payload.get("attempt_id"), str):\n            raise ControlError("Invalid execution attempt", status=422)\n        with self._lock:\n            attempt = self._previous_attempt(payload.get("attempt_id"))\n            if attempt is None or attempt["job_id"] != payload.get("job_id"):\n                raise ControlError("Execution attempt not found", status=404)\n            if attempt["status"] in TERMINAL:\n                return self._accepted(attempt)\n            if not attempt["cancel_requested"]:\n                self._event("execution.cancel_requested", job_id=attempt["job_id"], attempt_id=attempt["attempt_id"])\n                attempt.update(cancel_requested=True, status="cancel_requested")\n                if "_cancel_event" in attempt:\n                    attempt["_cancel_event"].set()\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\n            return self._accepted(attempt)\n\n    def drain(self, payload=None):\n        self._check_instance(payload or {})\n        with self._lock:\n            if not self._draining:\n                self._event("model.drain_requested")\n                self._draining = True\n                if self._phase not in {"failed", "stopped"}:\n                    self._phase = "draining"\n        return self.status()\n\n    def stop(self, payload=None):\n        self._check_instance(payload or {})\n        with self._lock:\n            if not self._stop_requested:\n                self._event("model.stop_requested")\n                self._stop_requested, self._draining = True, True\n                if self._phase not in {"failed", "stopped"}:\n                    self._phase = "draining"\n        return self.status()\n\n    def dispatch(self, operation, payload=None):\n        payload = payload or {}\n        if operation == "status":\n            return self.status()\n        if operation == "execute":\n            return self.execute(payload)\n        if operation == "cancel":\n            return self.cancel(payload)\n        if operation == "drain":\n            return self.drain(payload)\n        if operation == "stop":\n            return self.stop(payload)\n        if operation == "ack_results":\n            return self.ack_results(payload)\n        raise ControlError("Unknown model operation", status=404)\n\n    def ack_results(self, payload):\n        self._check_instance(payload, required=True)\n        identities = payload.get("attempt_ids")\n        if not isinstance(identities, list) or any(not isinstance(x, str) or not IDENTIFIER.fullmatch(x) for x in identities):\n            raise ControlError("Invalid result acknowledgement", status=422)\n        with self._lock:\n            acknowledged = []\n            for identity in identities:\n                attempt = self._attempts.get(identity)\n                if attempt and attempt["status"] not in TERMINAL:\n                    raise ControlError("Cannot acknowledge an unfinished execution", status=409)\n                if identity not in self._results:\n                    if self._previous_attempt(identity):\n                        acknowledged.append(identity)\n                    continue\n                record = {key: value for key, value in attempt.items() if not key.startswith("_")}\n                ledger = sqlite3.connect(self._ledger_path)\n                try:\n                    with ledger:\n                        ledger.execute("INSERT OR REPLACE INTO acknowledged VALUES (?,?)", (identity, json.dumps(record)))\n                finally:\n                    ledger.close()\n                self._results.pop(identity)\n                self._attempts.pop(identity)\n                if self._budget:\n                    self._budget.acknowledge(identity)\n                acknowledged.append(identity)\n            return {"instance_id": self.instance_id, "acknowledged": acknowledged}\n\n    def _revoke_pipeline(self, error, *, current=None):\n        with self._lock:\n            self._loaded, self._phase, self._error = False, "failed", error\n            for attempt in self._live_attempts():\n                if attempt is not current:\n                    attempt["cancel_requested"] = True\n                    attempt["_cancel_event"].set()\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\n\n    @contextmanager\n    def _pipeline_stage(self, attempt, name, lease):\n        if name not in {"prepare", "gpu", "encode_validate"} or lease != ("gpu" if name == "gpu" else "cpu"):\n            raise ValueError("Unknown pipeline stage or resource lease")\n        cancellation = attempt["_cancel_event"]\n        identity, job_id = attempt["attempt_id"], attempt["job_id"]\n        witness = None\n        # Never hold the model lock while waiting for a budget grant. Budget\n        # callbacks read Event only and never acquire this model\'s lock.\n        try:\n            with self._budget.stage(identity, lease, cpu_threads=attempt["resources"]["cpu_threads"],\n                                    cancellation_event=cancellation,\n                                    context={"model_instance_id": self.instance_id, "job_id": job_id,\n                                             "stage": name}) as acquired:\n                with self._lock:\n                    witness = self._budget.lease_witness(identity, lease)\n                    reservation = self._budget.reservation_witness(identity)\n                    if witness is not None:\n                        self._event("lease.acquired", job_id=job_id, attempt_id=identity, stage=name,\n                                    lease_id=witness.get("lease_id"), budget_instance_id=witness.get("budget_instance_id"),\n                                    lease_witness=witness)\n                    admission = attempt.get("condition_gates", {}).get("model.execute", {})\n                    admission_context = admission.get("context", {})\n                    previous = {"prepare": None, "gpu": "prepare", "encode_validate": "gpu"}[name]\n                    live_state = attempt.get("status")\n                    live = (None if type(live_state) is not str else False if live_state in TERMINAL else True if live_state in {\n                        "accepted", "preparing", "gpu_wait", "gpu_running", "encoding", "validating", "cancel_requested"} else None)\n                    observations = {\n                        "instance_matches": attempt.get("instance_id") == self.instance_id,\n                        "attempt_owned": (self._attempts.get(identity) is attempt\n                            and admission_context.get("instance_id") == self.instance_id\n                            and admission_context.get("job_id") == job_id and admission_context.get("attempt_id") == identity),\n                        "attempt_live": live, "model_loaded": self._loaded,\n                        "worker_alive": self._worker.is_alive() if self._worker is not None else False,\n                        "capability_available": self._capability_observation(admitted=True),\n                        "admission_confirmed": admission.get("allowed"),\n                        "predecessor_completed": admission.get("allowed") if previous is None else attempt.get("stage_completions", {}).get(previous),\n                        "stage_not_started": negate_observation(attempt.get("stage_started", {}).get(name)),\n                        "cancel_flag_clear": negate_observation(attempt.get("cancel_requested")),\n                        "cancel_event_clear": negate_observation(cancellation.is_set()),\n                        "reservation_owned": _all_observed([\n                            reservation.get("owned"), reservation.get("attempt_id") == identity,\n                            reservation.get("budget_instance_id") == self._budget.instance_id,\n                            type(reservation.get("reservation_id")) is str and bool(reservation["reservation_id"])\n                        ]) if reservation else None,\n                        "budget_grant_held": False if acquired is False else _all_observed([acquired,\n                            witness.get("owned") if witness else (False if acquired is False else None),\n                            witness.get("attempt_id") == identity if witness else None,\n                            witness.get("kind") == lease if witness else None,\n                            witness.get("budget_instance_id") == self._budget.instance_id if witness else None,\n                            witness.get("reservation_id") == reservation.get("reservation_id") if witness and reservation else None]),\n                    }\n                    receipt = self._gate("model.stage." + name, STAGE_CONDITIONS, observations,\n                        job_id=job_id, attempt_id=identity, attempt=attempt, stage=name,\n                        lease_id=witness.get("lease_id") if witness else None,\n                        budget_instance_id=witness.get("budget_instance_id") if witness else None,\n                        budget_decision_id=witness.get("decision_id") if witness else None)\n                    if receipt["allowed"] is not True:\n                        # A confirmed cancellation is distinct from unknown or\n                        # contradictory ownership/capability/precedence evidence.\n                        non_cancel_unknown = any(state in {"unknown", "conflict"} for state in receipt["conditions"].values())\n                        if (not non_cancel_unknown and attempt.get("cancel_requested") is True\n                                and cancellation.is_set() is True):\n                            raise ExecutionCancelled()\n                        raise ConditionGateBlocked("Pipeline stage condition evidence is blocked")\n                    attempt["stage"] = name\n                    attempt["stage_started"][name] = True\n                    attempt["status"] = {"prepare": "preparing", "gpu": "gpu_running", "encode_validate": "encoding"}[name]\n                    stage_resources = dict(attempt["resources"], cpu_threads=(\n                        self._budget.gpu_host_cpu_threads if lease == "gpu" else attempt["resources"]["cpu_threads"]))\n                    event_context = {"condition_decision_id": receipt["decision_id"], "lease_id": witness["lease_id"],\n                                     "budget_instance_id": witness["budget_instance_id"], "lease_witness": witness}\n                    self._event("stage.started", job_id=job_id, attempt_id=identity, stage=name,\n                                resources=stage_resources, **event_context)\n                succeeded = False\n                try:\n                    yield\n                    succeeded = True\n                except BaseException:\n                    if lease == "gpu":\n                        self._revoke_pipeline("gpu_execution_failed", current=attempt)\n                    raise\n                finally:\n                    with self._lock:\n                        # A successfully persisted finish precedes the completion\n                        # witness. An audit failure cannot unlock the next stage.\n                        self._event("stage.finished", job_id=job_id, attempt_id=identity,\n                                    stage=name, succeeded=succeeded, resources=stage_resources, **event_context)\n                        attempt["stage_completions"][name] = succeeded\n        finally:\n            if witness is not None:\n                with self._lock:\n                    released = self._budget.release_witness(witness["lease_id"])\n                    self._event("lease.released", job_id=job_id, attempt_id=identity, stage=name,\n                                lease_id=witness["lease_id"], budget_instance_id=witness["budget_instance_id"],\n                                lease_witness=released)\n                    if released is None:\n                        raise ConditionGateBlocked("Lease release evidence is unavailable")\n\n    def _run_pipeline_attempt(self, attempt):\n        identity = attempt["attempt_id"]\n        path = self.root / attempt["job_id"] / "result.mp4"\n        result = {"job_id": attempt["job_id"], "attempt_id": identity, "instance_id": self.instance_id,\n                  "status": "failed", "error": None, "artifact": None, "metrics": None,\n                  "started_at": time.time(), "cleanup_complete": False}\n        prepared = cpu_result = None\n        try:\n            parameters = copy.deepcopy(attempt["parameters"])\n            for reference in parameters["references"]:\n                reference["path"] = Path(reference["path"])\n            with self._pipeline_stage(attempt, "prepare", "cpu"):\n                prepared = self.backend.prepare(cpu_threads=attempt["resources"]["cpu_threads"], **parameters)\n                data = getattr(getattr(prepared, "metrics", None), "data", {})\n                if isinstance(data.get("generation_options"), dict):\n                    with self._lock:\n                        attempt["generation_options"] = data["generation_options"]\n            with self._lock:\n                if not attempt["cancel_requested"]:\n                    attempt["status"], attempt["stage"] = "gpu_wait", "gpu_wait"\n            with self._pipeline_stage(attempt, "gpu", "gpu"):\n                cpu_result = self.backend.infer(prepared)\n                if getattr(cpu_result, "cpu_handoff_complete", False) is not True:\n                    raise RuntimeError("Backend did not confirm CPU handoff and CUDA synchronization")\n                if isinstance(attempt.get("generation_options"), dict):\n                    self._event("generation.configured", job_id=attempt["job_id"], attempt_id=identity,\n                                generation_options=copy.deepcopy(attempt["generation_options"]))\n            # CPU encoding and full-file decoding consume their CPU budget, but\n            # no GPU lease. Another admitted job may now perform its inference.\n            with self._pipeline_stage(attempt, "encode_validate", "cpu"):\n                metrics = self.backend.encode(cpu_result, output_path=path)\n                if isinstance(metrics, dict):\n                    result["metrics"] = metrics\n                with self._lock:\n                    if attempt["cancel_requested"]:\n                        raise ExecutionCancelled()\n                    attempt["status"] = "validating"\n                if self.artifact_validator is verify_artifact:\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"],\n                        cpu_threads=attempt["resources"]["cpu_threads"])\n                else:\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"])\n            result["status"] = "succeeded"\n            with self._lock:\n                self._consecutive_failures = 0\n        except ExecutionCancelled:\n            result["status"] = "cancelled"\n        except BaseException as error:\n            LOG.exception("Pipeline attempt failed: %s/%s", attempt["job_id"], identity)\n            result["error"] = ("condition_gate_blocked" if isinstance(error, ConditionGateBlocked)\n                               else "generation_failed; inspect model log")\n            with self._lock:\n                if not isinstance(error, ValueError):\n                    self._consecutive_failures += 1\n                fatal = not isinstance(error, Exception) or self._consecutive_failures >= 2\n            if fatal:\n                self._revoke_pipeline("model_execution_unavailable", current=attempt)\n        finally:\n            try:\n                if isinstance(attempt.get("generation_options"), dict):\n                    result["generation_options"] = copy.deepcopy(attempt["generation_options"])\n                result["condition_gates"] = copy.deepcopy(attempt.get("condition_gates", {}))\n                result["stage_completions"] = copy.deepcopy(attempt.get("stage_completions", {}))\n                # This operation is strictly CPU cleanup; it must not empty the\n                # CUDA cache while the next attempt owns the GPU lease.\n                self.backend.cleanup(prepared=prepared, result=cpu_result)\n                prepared = cpu_result = None\n                if result["metrics"] is not None:\n                    try:\n                        encoded = json.dumps(result["metrics"], allow_nan=False)\n                        result["metrics"] = json.loads(encoded) if len(encoded.encode()) <= 128 * 1024 else None\n                    except (TypeError, ValueError):\n                        result["metrics"] = None\n                        LOG.warning("Pipeline metrics were not finite JSON; omitting diagnostics")\n                inputs = self.root / attempt["job_id"] / "inputs"\n                if inputs.is_symlink() or inputs.resolve() != inputs:\n                    raise ValueError("Input ownership changed during pipeline execution")\n                if inputs.exists():\n                    shutil.rmtree(inputs)\n                result["cleanup_complete"] = True\n            except BaseException:\n                LOG.exception("Pipeline attempt cleanup failed")\n                result.update(status="failed", artifact=None, cleanup_complete=False,\n                              error="execution_cleanup_failed; inspect model log")\n                self._revoke_pipeline("execution_cleanup_failed", current=attempt)\n            with self._lock:\n                if attempt["cancel_requested"] and result["status"] == "succeeded":\n                    result["status"] = "cancelled"\n                if result["status"] != "succeeded":\n                    result["artifact"] = None\n                    try:\n                        path.unlink(missing_ok=True)\n                        path.with_suffix(".metrics.json").unlink(missing_ok=True)\n                        path.with_suffix(".metrics.json.tmp").unlink(missing_ok=True)\n                    except OSError:\n                        result.update(status="failed", cleanup_complete=False, error="cancel_cleanup_failed")\n                        self._revoke_pipeline("cancel_cleanup_failed", current=attempt)\n                result["finished_at"] = time.time()\n                attempt.update(status=result["status"], result=result)\n                attempt.pop("parameters", None)\n                self._results[identity] = self._last_result = result\n                if result["cleanup_complete"]:\n                    self._budget.finish(identity, len(json.dumps(result).encode()))\n                if self._phase != "failed":\n                    self._phase = "draining" if self._draining else ("running" if self._live_attempts() else "ready")\n                self._safe_event("execution." + result["status"], job_id=attempt["job_id"],\n                                 attempt_id=identity, result=result)\n\n    def _run_attempt(self, attempt):\n        path = self.root / attempt["job_id"] / "result.mp4"\n        result = {"job_id": attempt["job_id"], "attempt_id": attempt["attempt_id"],\n                  "instance_id": self.instance_id, "status": "failed", "error": None,\n                  "artifact": None, "metrics": None, "started_at": time.time(),\n                  "cleanup_complete": False}\n        fatal = None\n        generated = False\n        try:\n            with self._lock:\n                if not attempt["cancel_requested"]:\n                    attempt["status"] = "running"\n                self._event("execution.running", job_id=attempt["job_id"], attempt_id=attempt["attempt_id"])\n            if not attempt["cancel_requested"]:\n                params = copy.deepcopy(attempt["parameters"])\n                for reference in params["references"]:\n                    reference["path"] = Path(reference["path"])\n                self.backend.generate(output_path=path, **params)\n                generated = True\n                if not attempt["cancel_requested"]:\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"])\n            with self._lock:\n                result["status"] = "cancelled" if attempt["cancel_requested"] else "succeeded"\n            self._consecutive_failures = 0\n        except BaseException as error:\n            LOG.exception("Model attempt failed: %s/%s", attempt["job_id"], attempt["attempt_id"])\n            result["error"] = "generation_failed; inspect model log"\n            # Bad input media may be a request error. Invalid generated output\n            # is an execution failure even when the validator raises ValueError.\n            if generated or not isinstance(error, ValueError):\n                self._consecutive_failures += 1\n            if not isinstance(error, Exception) or self._consecutive_failures >= 2:\n                fatal = error\n                with self._lock:\n                    self._loaded, self._phase, self._error = False, "failed", "model_execution_unavailable"\n        finally:\n            try:\n                # Keep the lease until execution and its local cleanup have ended.\n                metrics = getattr(self.backend, "last_metrics", None)\n                if isinstance(metrics, dict):\n                    try:\n                        encoded = json.dumps(metrics, allow_nan=False)\n                        if len(encoded.encode("utf-8")) <= 128 * 1024:\n                            result["metrics"] = json.loads(encoded)\n                    except (TypeError, ValueError):\n                        LOG.warning("Model metrics were not finite JSON; omitting diagnostics")\n                if result["status"] != "succeeded":\n                    result["artifact"] = None\n                    path.unlink(missing_ok=True)\n                inputs = self.root / attempt["job_id"] / "inputs"\n                if inputs.is_symlink() or inputs.resolve() != inputs:\n                    raise ValueError("Input directory ownership changed during execution")\n                if inputs.exists():\n                    shutil.rmtree(inputs)\n                # Attempt-local files are reclaimed; this does not release the\n                # model process\'s persistent CPU/GPU resource ownership.\n                result["cleanup_complete"] = True\n            except BaseException as error:\n                LOG.exception("Attempt cleanup failed")\n                result.update(status="failed", artifact=None, cleanup_complete=False,\n                              error="execution_cleanup_failed; inspect model log")\n                fatal = error\n                with self._lock:\n                    self._loaded, self._phase, self._error = False, "failed", "execution_cleanup_failed"\n            with self._lock:\n                # A cancel racing with validation/cleanup still wins before commit.\n                if attempt["cancel_requested"] and result["status"] == "succeeded":\n                    result.update(status="cancelled", artifact=None)\n                    try:\n                        path.unlink(missing_ok=True)\n                    except OSError as error:\n                        result.update(status="failed", error="cancel_cleanup_failed", cleanup_complete=False)\n                        fatal = error\n                        self._loaded, self._phase, self._error = False, "failed", "cancel_cleanup_failed"\n                result["finished_at"] = time.time()\n                attempt.update(status=result["status"], result=result)\n                attempt.pop("parameters", None)\n                self._last_result = result\n                self._results[attempt["attempt_id"]] = result\n                self._active = None\n                if self._phase != "failed":\n                    self._phase = "draining" if self._draining else "ready"\n                self._event("execution." + result["status"], job_id=attempt["job_id"],\n                            attempt_id=attempt["attempt_id"], result=result)\n        if fatal is not None:\n            raise fatal\n\n    def _work(self):\n        try:\n            if self.backend is None:\n                if self.staged:\n                    from staged_backend import StagedMiniMaxBackend\n                    self.backend = StagedMiniMaxBackend()\n                else:\n                    from model_backend import MiniMaxBackend\n                    self.backend = MiniMaxBackend()\n            self._pipeline = all(callable(getattr(self.backend, name, None)) for name in\n                                 ("prepare", "infer", "encode", "cleanup", "estimate_resources"))\n            if not self._pipeline and not self._injected_backend:\n                raise RuntimeError("Production backend must implement the staged pipeline contract")\n            if self._pipeline:\n                self._budget = PipelineBudget(self.root, limits=self._resource_limits,\n                    model_peak_ram_bytes=getattr(self.backend, "pipeline_model_peak_ram_bytes", 144 * 1024 ** 3))\n                configure = getattr(self.backend, "configure_cpu_budget", None)\n                if callable(configure):\n                    self._budget.native_cpu_status = configure(self._budget.gpu_host_cpu_threads)\n                elif not self._injected_backend:\n                    raise RuntimeError("Production backend must configure its native CPU thread budget before loading")\n                else:\n                    self._budget.native_cpu_status = {"configured": False, "scope": "injected CPU test backend"}\n            self.backend.load()\n            if self._pipeline:\n                native_snapshot = getattr(self.backend, "cpu_budget_snapshot", None)\n                if callable(native_snapshot):\n                    # Model loading can import additional BLAS/OpenMP runtimes.\n                    # Read back their limits before publishing readiness; never\n                    # reconfigure process-wide pools while jobs are running.\n                    self._budget.native_cpu_status = native_snapshot()\n                elif not self._injected_backend:\n                    raise RuntimeError("Production backend must verify native CPU budgets after loading")\n            with self._lock:\n                self._loaded = True\n                self._phase = "draining" if self._draining else "ready"\n                self._event("model.ready", accepting=not self._draining)\n            while True:\n                with self._lock:\n                    if self._pipeline:\n                        for identity, worker in list(self._stage_workers.items()):\n                            if not worker.is_alive():\n                                worker.join()\n                                del self._stage_workers[identity]\n                        if self._phase == "failed":\n                            self._revoke_pipeline(self._error or "pipeline_unavailable")\n                        if (self._stop_requested or self._phase == "failed") and not self._live_attempts() and not self._stage_workers:\n                            break\n                    else:\n                        if self._stop_requested and self._active is None:\n                            break\n                        if self._phase == "failed":\n                            break\n                try:\n                    attempt = self._queue.get(timeout=.1)\n                except queue.Empty:\n                    continue\n                try:\n                    if self._pipeline:\n                        worker = threading.Thread(target=self._run_pipeline_attempt, args=(attempt,),\n                                                  name="h3-stage-" + attempt["attempt_id"][:8], daemon=False)\n                        with self._lock:\n                            self._stage_workers[attempt["attempt_id"]] = worker\n                            try:\n                                worker.start()\n                            except BaseException:\n                                self._stage_workers.pop(attempt["attempt_id"], None)\n                                self._revoke_pipeline("pipeline_worker_start_failed")\n                                self._run_pipeline_attempt(attempt)\n                                raise\n                    else:\n                        self._run_attempt(attempt)\n                finally:\n                    self._queue.task_done()\n        except BaseException as error:\n            LOG.exception("Model worker exited")\n            with self._lock:\n                self._loaded, self._phase = False, "failed"\n                self._error = self._error or "model_load_or_worker_failed"\n                if self._active is not None:\n                    attempt = self._active\n                    result = {"job_id": attempt["job_id"], "attempt_id": attempt["attempt_id"],\n                              "instance_id": self.instance_id, "status": "failed", "artifact": None,\n                              "metrics": None, "error": "model_worker_exited", "finished_at": time.time(),\n                              "cleanup_complete": False}\n                    attempt.update(status="failed", result=result)\n                    self._results[attempt["attempt_id"]] = result\n                    self._last_result, self._active = result, None\n                self._safe_event("model.failed", error=self._error, error_type=type(error).__name__)\n        finally:\n            with self._lock:\n                self._loaded = False\n            if self._pipeline:\n                # Never unload model state while prepare, CUDA inference, CPU\n                # encoding, validation or local cleanup still refers to it.\n                for worker in list(self._stage_workers.values()):\n                    if worker.ident is not None:\n                        worker.join()\n                for attempt in self._live_attempts():\n                    attempt["cancel_requested"] = True\n                    attempt["_cancel_event"].set()\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\n                    self._run_pipeline_attempt(attempt)\n            try:\n                backend, self.backend = self.backend, None\n                unload = getattr(backend, "unload", None)\n                if unload:\n                    unload()\n                if hasattr(backend, "pipe"):\n                    backend.pipe = None\n                del backend\n                gc.collect()\n                torch = sys.modules.get("torch")\n                if torch is not None and torch.cuda.is_initialized():\n                    torch.cuda.empty_cache()\n                self._worker_cleanup_complete = True\n            except BaseException:\n                LOG.exception("Model worker cleanup failed")\n                self._phase, self._error = "failed", "model_unload_failed"\n            with self._lock:\n                if self._phase != "failed":\n                    self._phase = "stopped"\n                self._safe_event("model.worker_exited", phase=self._phase,\n                                 worker_cleanup_complete=self._worker_cleanup_complete,\n                                 budget_snapshot=self._budget.snapshot() if self._budget is not None else None)\n            self._done.set()\n\n    def join(self, timeout=None):\n        if self._worker is not None and self._worker.ident is not None:\n            self._worker.join(timeout)\n        return self._done.is_set()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--port", type=int, required=True)\n    parser.add_argument("--instance-id", required=True)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--staged", action="store_true")\n    parser.add_argument("--parent-pid", type=int)\n    args = parser.parse_args()\n    owner_created_at = parent_identity(args.parent_pid) if args.parent_pid else None\n    key = os.environ.get("H3_CONTROL_KEY", "")\n    if len(key) < 32:\n        raise RuntimeError("A private H3_CONTROL_KEY is required")\n    container = ModelContainer(args.root, args.instance_id, staged=args.staged,\n                               progress_path=os.environ.get("H3_DOWNLOAD_PROGRESS_PATH"))\n    server = make_server("127.0.0.1", args.port, key, container.dispatch)\n    http = threading.Thread(target=server.serve_forever, name="h3-model-control", daemon=True)\n    http.start()\n    stop_pending = False\n    stop_applied = False\n\n    def request_stop(*_):\n        # Python signal handlers may interrupt this thread while it holds an\n        # RLock or is writing an event. Only record intent here; even Event.set\n        # takes a lock and must not run in a reentrant signal callback.\n        nonlocal stop_pending\n        stop_pending = True\n\n    def apply_stop():\n        nonlocal stop_applied\n        if stop_applied:\n            return\n        stop_applied = True\n        try:\n            container.stop()\n        except BaseException:\n            LOG.exception("Could not record model stop")\n            with container._lock:\n                container._stop_requested = container._draining = True\n                container._loaded = False\n\n    signal.signal(signal.SIGTERM, request_stop)\n    signal.signal(signal.SIGINT, request_stop)\n    try:\n        container.start()\n        while True:\n            if args.parent_pid and not parent_is_alive(args.parent_pid, owner_created_at):\n                # An ownerless CUDA worker must not outlive its resource lease.\n                apply_stop()\n                container.join(2)\n                server.shutdown()\n                server.server_close()\n                os._exit(70)\n            if stop_pending:\n                apply_stop()\n            if container._stop_requested:\n                if container.join(.2):\n                    break\n            else:\n                # Failed loading leaves the private controller available for diagnosis.\n                time.sleep(.2)\n    finally:\n        apply_stop()\n        container.join(2)\n        server.shutdown()\n        server.server_close()\n\n\nif __name__ == "__main__":\n    logging.basicConfig(level=logging.INFO)\n    main()\n', 'tunnel.py': '"""Prepare an explicit temporary tunnel independently of model readiness."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport signal\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport threading\n\n\ndef watch_owner(parent_pid):\n    """Exit through the signal handler so the connector is also reclaimed."""\n    if not parent_pid:\n        return\n    import psutil\n    try:\n        owner = psutil.Process(int(parent_pid))\n        created = owner.create_time()\n    except psutil.Error:\n        raise RuntimeError("Tunnel owner is already unavailable")\n\n    def observe():\n        while True:\n            time.sleep(1)\n            try:\n                alive = (owner.is_running() and owner.create_time() == created\n                         and owner.status() != psutil.STATUS_ZOMBIE)\n            except psutil.Error:\n                alive = False\n            if not alive:\n                if os.name == "nt":\n                    import _thread\n                    _thread.interrupt_main()\n                else:\n                    os.kill(os.getpid(), signal.SIGTERM)\n                return\n    threading.Thread(target=observe, name="tunnel-owner-watch", daemon=True).start()\nimport urllib.request\n\n\ndef ready(port: int) -> bool:\n    try:\n        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as response:\n            return json.load(response).get("ready") is True\n    except Exception:\n        return False\n\n\ndef cloudflared_binary(install: bool = False) -> str:\n    found = shutil.which("cloudflared")\n    if found:\n        return found\n    cached = Path(__file__).resolve().parent / ".cache" / "cloudflared"\n    if cached.is_file():\n        return str(cached)\n    if not install:\n        raise RuntimeError("cloudflared is not installed. Use --install to fetch an official release with SHA-256 verification.")\n    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "amd64"}:\n        raise RuntimeError("Automatic cloudflared install supports Linux x86_64 only; otherwise install it from Cloudflare first.")\n    request = urllib.request.Request(\n        "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",\n        headers={"User-Agent": "minimax-h3-ref-api", "Accept": "application/vnd.github+json"},\n    )\n    with urllib.request.urlopen(request, timeout=30) as response:\n        release = json.load(response)\n    asset = next(a for a in release["assets"] if a["name"] == "cloudflared-linux-amd64")\n    digest = asset.get("digest", "")\n    if not digest.startswith("sha256:"):\n        raise RuntimeError("Official release metadata lacks a SHA-256 digest; install cloudflared manually from Cloudflare.")\n    url = asset["browser_download_url"]\n    if not url.startswith("https://github.com/cloudflare/cloudflared/releases/download/"):\n        raise RuntimeError("Unexpected cloudflared release origin")\n    cached.parent.mkdir(parents=True, exist_ok=True)\n    temporary = cached.with_suffix(".download")\n    hasher = hashlib.sha256()\n    try:\n        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as output:\n            while chunk := response.read(1024 * 1024):\n                hasher.update(chunk)\n                output.write(chunk)\n        if hasher.hexdigest() != digest.split(":", 1)[1]:\n            raise RuntimeError("cloudflared SHA-256 mismatch; binary was not installed")\n        temporary.chmod(0o700)\n        temporary.replace(cached)\n    finally:\n        temporary.unlink(missing_ok=True)\n    print(f"Installed official cloudflared {release[\'tag_name\']} (SHA-256 verified).", flush=True)\n    return str(cached)\n\n\ndef run_tunnel(port: int = 8000, install: bool = False, *, status_file: Path | None = None,\n               instance_id: str | None = None) -> None:\n    """Register the transport; a candidate URL never implies application readiness."""\n    if status_file and (not instance_id or not re.fullmatch(r"[a-f0-9]{32}", instance_id)):\n        raise ValueError("Managed tunnel status requires its owner\'s 32-character instance ID")\n    state = {"instance_id": instance_id, "phase": "preparing", "url": None,\n             "connected": False, "pid": None, "error": None}\n\n    def publish(**changes):\n        state.update(changes, observed_at=time.time())\n        if status_file:\n            path = Path(status_file)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            temporary = path.with_suffix(".tmp")\n            temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")\n            temporary.replace(path)\n\n    process = None\n    registered = set()\n    publish()\n    try:\n        # A loading origin may not be listening yet. Transport installation and\n        # registration do not depend on its /health response.\n        binary = cloudflared_binary(install)\n        process = subprocess.Popen(\n            [binary, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,\n        )\n        publish(pid=process.pid)\n        for line in process.stdout:\n            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", line)\n            if match and not state["url"]:\n                publish(url=match.group(0))\n                print(f"PUBLIC_API_URL={match.group(0)}", flush=True)\n                print("Candidate URL allocated; application readiness still requires the live join.", flush=True)\n            connection = re.search(r"connIndex=(\\d+)", line)\n            connection_id = connection.group(1) if connection else "default"\n            if "Registered tunnel connection" in line:\n                registered.add(connection_id)\n                publish(phase="registered", connected=True)\n            elif any(marker in line for marker in (\n                "Unregistered tunnel connection", "Connection terminated",\n                "failed to serve tunnel connection", "Serve tunnel error",\n            )):\n                if connection:\n                    registered.discard(connection_id)\n                else:\n                    registered.clear()  # No connection identity: revoke conservatively.\n                publish(phase="registered" if registered else "reconnecting", connected=bool(registered))\n            print(line, end="", flush=True)\n        if process.wait() != 0:\n            raise RuntimeError("cloudflared stopped; see the tunnel log above")\n    except KeyboardInterrupt:\n        publish(phase="stopping", connected=False)\n        raise\n    except BaseException as error:\n        publish(phase="failed", connected=False, error=type(error).__name__)\n        raise\n    finally:\n        if process is not None and process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        publish(phase="failed" if state["phase"] == "failed" else "stopped", connected=False)\n\n\nif __name__ == "__main__":\n    signal.signal(signal.SIGTERM, lambda *_: (_ for _ in ()).throw(KeyboardInterrupt()))\n    watch_owner(os.environ.get("H3_PARENT_PID"))\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--port", type=int, default=8000)\n    parser.add_argument("--install", action="store_true", help="Download official cloudflared if missing, checking GitHub release SHA-256")\n    parser.add_argument("--status-file", type=Path, help="Atomic transport state checkpoint owned by the startup supervisor")\n    parser.add_argument("--instance-id", help="Owning startup session ID; never an API credential")\n    args = parser.parse_args()\n    try:\n        run_tunnel(args.port, args.install, status_file=args.status_file, instance_id=args.instance_id)\n    except KeyboardInterrupt:\n        print("Tunnel stopped. Local GPU server remains running until you stop it.")\n', 'client.py': '"""Submit ordered references to the ref2va API, poll, and download the MP4."""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import ExitStack\nimport json\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport time\nimport requests\n\n\nclass ReferenceAction(argparse.Action):\n    def __call__(self, parser, namespace, value, option_string=None):\n        items = list(getattr(namespace, self.dest, None) or [])\n        items.append((option_string.removeprefix("--"), Path(value)))\n        setattr(namespace, self.dest, items)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("url", help="API base URL, e.g. https://example.trycloudflare.com")\n    parser.add_argument("--prompt", required=True)\n    for kind in ("image", "video", "audio"):\n        parser.add_argument("--" + kind, dest="references", action=ReferenceAction, metavar="FILE")\n    parser.add_argument("--output", type=Path, default=Path("result.mp4"))\n    parser.add_argument("--seconds", type=float, default=5)\n    parser.add_argument("--width", type=int, default=960)\n    parser.add_argument("--height", type=int, default=544)\n    parser.add_argument("--steps", type=int, default=50)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--timeout", type=int, default=7200, help="Maximum poll time in seconds")\n    args = parser.parse_args()\n    key = os.environ.get("H3_API_KEY")\n    if not key:\n        parser.error("Set H3_API_KEY in the caller\'s environment first")\n    if not args.references:\n        parser.error("At least one --image or --video reference is required")\n    if args.output.exists():\n        parser.error("Output file already exists; choose another --output path")\n    base = args.url.rstrip("/")\n    if not (base.startswith("https://") or base.startswith("http://127.0.0.1:")):\n        parser.error("Use HTTPS for public endpoints, or HTTP on 127.0.0.1 for local testing")\n    headers = {"Authorization": "Bearer " + key}\n    with ExitStack() as stack:\n        files = [("files", (path.name, stack.enter_context(path.open("rb")), mimetypes.guess_type(path.name)[0] or "application/octet-stream"))\n                 for _, path in args.references]\n        response = requests.post(\n            base + "/v1/jobs", headers=headers, files=files,\n            data={"kinds": json.dumps([kind for kind, _ in args.references]), "prompt": args.prompt,\n                  "seconds": args.seconds, "width": args.width, "height": args.height,\n                  "steps": args.steps, "seed": args.seed}, timeout=120,\n        )\n    response.raise_for_status()\n    job_id = response.json()["id"]\n    print("Job:", job_id, flush=True)\n    deadline = time.monotonic() + args.timeout\n    while True:\n        response = requests.get(base + f"/v1/jobs/{job_id}", headers=headers, timeout=30)\n        response.raise_for_status()\n        job = response.json()\n        print(job["status"], flush=True)\n        if job["status"] == "failed":\n            raise RuntimeError(json.dumps(job, ensure_ascii=False))\n        if job["status"] == "succeeded":\n            break\n        if time.monotonic() >= deadline:\n            raise TimeoutError(f"Polling timed out. Job {job_id} may still be running; query it later.")\n        time.sleep(min(30, max(0, deadline - time.monotonic())))\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    partial = args.output.with_suffix(args.output.suffix + ".part")\n    created_partial = False\n    try:\n        with requests.get(base + f"/v1/jobs/{job_id}/file", headers=headers, timeout=120, stream=True) as response:\n            response.raise_for_status()\n            with partial.open("xb") as output:\n                created_partial = True\n                for chunk in response.iter_content(1024 * 1024):\n                    output.write(chunk)\n        partial.replace(args.output)\n    finally:\n        if created_partial:\n            partial.unlink(missing_ok=True)\n    print("Saved:", args.output.resolve())\n\n\nif __name__ == "__main__":\n    main()\n', 'README.md': '# MiniMax-H3 ref2va 参考运行时 · API 0.4.0\n\n这里保存已验收的专用实现：总管理、API 后台、大模型容器三个受管进程，独立的可选公网隧道与 Colab 控制台。模型容器是模型运行时进程，不把 GPU 硬件本身称为容器。\n\n- 当前模型：MiniMax-H3，仅 ref2va；至少图片或视频参考，可附加音频。\n- A100 原生 BF16 主干与 LoRA，保留官方 FP32 例外，不启用权重量化。\n- 任务分为 CPU 预处理、独占 GPU 推理、CPU 编码校验；按真实依赖及资源预算调度。\n- 队列无固定条数上限，可指定 ID 取消；资源不足、未知提交和运行中取消各自有真实状态。\n- API 可先于模型就绪接收排队。生成、产物完整校验、任务清理、服务进程退出和云平台释放分别确认。\n- 每任务控制 attention_backend、acceleration_lora、steps；Turbo 要显式 steps=5，对应4次模型评估。预加载不等于当前任务启用。\n\n仓库内的[运行说明](../../docs/operations.md)、[架构](../../docs/architecture.md)、[验证边界](../../docs/validation.md)提供当前入口。此 README 进入自包含 notebook 时，仓库相对链接仅用于源码导航；云端实际操作使用面板与日志。\n\n```bash\npython run.py --help\npython smoke_test.py --help\n```\n\n历史 A100 实测不是当前机器的一次新运行。源码来源与精度/配置/任务/释放证据见仓库 evidence。真实生成验收每次必须是全新内容；未知结果按原 ID 对账，不自动重新生成。\n', 'notebook_ui.py': '"""Read-only dashboard and separate diagnostics for the notebook control panel."""\nfrom __future__ import annotations\n\nimport html\nimport json\nfrom collections import deque\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nimport re\nimport subprocess\nimport time\n\nSTYLE = """\n<style>\n.h3-card{--bg:#f8fafc;--ink:#172033;--muted:#627086;--line:#dbe3ec;--accent:#187c72;box-sizing:border-box;background:var(--bg);color:var(--ink);border:1px solid var(--line);border-left:4px solid var(--accent);border-radius:12px;padding:17px 20px;margin:8px 0;font:14px/1.55 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;max-width:980px}\n.h3-card *{box-sizing:border-box}.h3-card[data-tone="error"]{--accent:#ba4242}.h3-card[data-tone="idle"]{--accent:#7b8797}.h3-card[data-tone="running"]{--accent:#bd7c16}\n.h3-head{display:flex;gap:12px;justify-content:space-between;align-items:center;flex-wrap:wrap}.h3-title{font-size:17px;font-weight:650;letter-spacing:-.2px}.h3-badge{font-size:11px;font-weight:650;letter-spacing:.3px;color:var(--accent);border:1px solid var(--line);padding:3px 9px;border-radius:99px;white-space:nowrap}.h3-note{color:var(--muted);margin:5px 0 0}\n.h3-fields{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px 18px;margin-top:13px}.h3-label{display:block;color:var(--muted);font-size:11px;letter-spacing:.25px;margin-bottom:2px}.h3-value{font-size:14px;font-weight:550;overflow-wrap:anywhere}.h3-card details{margin-top:12px;border-top:1px solid var(--line);padding-top:10px}.h3-card summary{cursor:pointer;color:var(--muted);font-size:12px}.h3-card pre{white-space:pre-wrap;overflow-wrap:anywhere;font:12px/1.55 ui-monospace,SFMono-Regular,Consolas,monospace;margin:9px 0 0;max-height:280px;overflow:auto}.h3-card a{color:var(--accent);text-decoration:none}.h3-card a:hover{text-decoration:underline}\n.h3-board{border-left-width:1px;padding:22px 24px}.h3-board h2{font-size:23px;line-height:1.3;margin:4px 0 8px}.h3-kicker{font-size:12px;color:var(--muted)}.h3-next{margin:12px 0 21px;padding:11px 14px;border-radius:8px;background:color-mix(in srgb,var(--accent) 8%,var(--bg));font-size:15px}.h3-modules{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:12px 0 19px}.h3-module{padding:12px 14px;border:1px solid var(--line);border-radius:8px}.h3-module strong{font-weight:650;display:block;font-size:16px;margin-top:4px}.h3-subtle{color:var(--muted);font-size:12px;margin-top:4px}.h3-summary{display:flex;flex-wrap:wrap;gap:8px 22px;margin:12px 0}.h3-activity{border-top:1px solid var(--line);margin-top:18px;padding-top:13px}.h3-activity .h3-title{font-size:14px}.h3-activity .h3-fields{margin-top:8px;gap:8px 18px}.h3-stamp{font-size:11px;color:var(--muted);margin-top:17px}.h3-logs{border-left-width:1px}.h3-logs pre{max-height:360px;margin:14px 0 0}.h3-logs pre:focus{outline:2px solid var(--accent);outline-offset:3px}\n@media(max-width:620px){.h3-board{padding:17px 15px}.h3-modules{grid-template-columns:1fr;gap:8px}.h3-module{padding:9px 12px;display:grid;grid-template-columns:95px 1fr;align-items:center}.h3-module strong{margin:0}.h3-module .h3-subtle{grid-column:2}.h3-board h2{font-size:21px}.h3-fields{grid-template-columns:1fr}}\n@media(prefers-color-scheme:dark){.h3-card{--bg:#17202d;--ink:#e4ebf4;--muted:#a5b3c6;--line:#344358;--accent:#67cdbb}.h3-card[data-tone="error"]{--accent:#ff9696}.h3-card[data-tone="running"]{--accent:#e4ba6d}.h3-card[data-tone="idle"]{--accent:#b5bfcb}}\n</style>\n"""\n\n_dashboard = None\n_diagnostics = deque(maxlen=20)\n_known_secrets = ()\n_process_observer = None\n_DIAGNOSTIC_LIMIT = 12000\n\n\ndef redact_text(text, secrets=()):\n    """Redact known credentials and common authorization/key assignments before display."""\n    text = str(text)\n    for secret in sorted({str(s) for s in (*_known_secrets, *secrets) if s}, key=len, reverse=True):\n        text = text.replace(secret, \'[已隐藏]\')\n    text = re.sub(r\'(?i)(\\bBearer\\s+)[^\\s\\"\\\'<>;,]+\', r\'\\1[已隐藏]\', text)\n    text = re.sub(r\'\'\'(?ix)(["\']?(?:H3_API_KEY|API_KEY|api[-_]?key|access_token|authorization)["\']?\\s*[:=]\\s*)(["\'])(.*?)(\\2)\'\'\',\n                  lambda match: match.group(1) + match.group(2) + \'[已隐藏]\' + match.group(2), text)\n    text = re.sub(r\'(?i)(\\b(?:H3_API_KEY|API_KEY|api[-_]?key|access_token)\\s*[:=]\\s*)[^\\s\\"\\\'<>;,]+\', r\'\\1[已隐藏]\', text)\n    return text\n\n\ndef _remember(title, text):\n    if text:\n        _diagnostics.append(redact_text(str(title) + \'\\n\' + str(text))[-_DIAGNOSTIC_LIMIT:])\n\n\ndef last_diagnostics():\n    return \'\\n\\n\'.join(_diagnostics)[-_DIAGNOSTIC_LIMIT:]\n\n\ndef diagnostics_html(title, text, *, secrets=()):\n    """A bounded, keyboard-scrollable log view; never embedded in dashboard cards."""\n    value = redact_text(text, secrets=secrets)\n    truncated = len(value) > _DIAGNOSTIC_LIMIT\n    value = value[-_DIAGNOSTIC_LIMIT:]\n    note = \'仅显示最后 12,000 个字符；完整记录保留在日志文件。\' if truncated else \'只读日志 · 敏感凭据已隐藏\'\n    return (STYLE + \'<section class="h3-card h3-logs" aria-label="日志"><div class="h3-title">\'\n            + html.escape(redact_text(title), quote=True) + \'</div><p class="h3-note">\' + note\n            + \'</p><pre tabindex="0" aria-label="日志内容">\' + html.escape(value or \'暂无记录。\', quote=True)\n            + \'</pre></section>\')\n\n\ndef _short(value, maximum=160):\n    text = redact_text(value).replace(\'\\n\', \' \').replace(\'\\r\', \' \')\n    return text if len(text) <= maximum else text[:maximum - 1] + \'…\'\n\n\ndef _activity(title, kwargs):\n    _remember(title, kwargs.get(\'details\'))\n    fields = [(label, value) for label, value in kwargs.get(\'fields\', ())\n              if not any(word in str(label) for word in (\'日志\', \'路径\', \'密钥\', \'当前记录\'))]\n    return {\'title\': _short(title, 70), \'badge\': _short(kwargs.get(\'badge\', \'\'), 35),\n            \'note\': _short(kwargs.get(\'note\', \'\')), \'tone\': kwargs.get(\'tone\', \'idle\'),\n            \'fields\': [(str(k), _short(v, 95)) for k, v in fields[:4]]}\n\n\ndef card_html(title, *, badge="", note="", fields=(), tone="idle", details=None, details_label="查看详情"):\n    escape = lambda value: html.escape(_short(value), quote=True)\n    if tone not in {"idle", "running", "success", "error"}:\n        tone = "idle"\n    values = "".join(\n        \'<div><span class="h3-label">\' + escape(label) + \'</span><span class="h3-value">\'\n        + escape(value) + \'</span></div>\' for label, value in fields\n    )\n    return (STYLE + \'<section class="h3-card" data-tone="\' + tone + \'"><div class="h3-head">\'\n            + \'<div class="h3-title">\' + escape(title) + \'</div><span class="h3-badge">\'\n            + escape(badge) + \'</span></div>\' + (\'<p class="h3-note">\' + escape(note) + \'</p>\' if note else "")\n            + (\'<div class="h3-fields">\' + values + \'</div>\' if fields else "") + "</section>")\n\n\ndef render_dashboard(context, activity=None):\n    """Render supplied observations only. This pure function performs no host I/O."""\n    escape = lambda value: html.escape(_short(value), quote=True)\n    state = context.get(\'state\') or {}\n    stale = bool(state.get(\'observation_stale\'))\n    failed_read = bool(context.get(\'observation_error\'))\n    phase = state.get(\'phase\')\n    if context.get(\'placeholder\'):\n        headline, next_step, tone = \'等待首次读取\', \'运行「刷新看板」查看本次状态；此处是界面占位，没有实时观察。\', \'idle\'\n    elif failed_read:\n        headline, next_step, tone = \'状态读取失败\', \'重新刷新看板；如仍失败，在日志区查看原因。\', \'error\'\n    elif stale:\n        headline, next_step, tone = \'状态需要刷新\', \'当前观察已过期；刷新后再判断可用能力。\', \'running\'\n    elif phase == \'stopped\' and state.get(\'cleanup_complete\'):\n        headline, next_step, tone = \'本次服务已停止\', \'确认结果已保存后，可在控制面板释放 Colab 实例。\', \'idle\'\n    elif phase == \'stopping\':\n        headline, next_step, tone = \'正在停止与回收\', \'等待受管进程退出并确认清理完成；此时不要重复启动。\', \'running\'\n    elif state.get(\'error\') or phase in {\'failed\', \'degraded\'}:\n        headline, next_step, tone = \'有模块需要处理\', \'先看下方可用能力；具体原因在日志区，控制面板仍可查询或停止。\', \'error\'\n    elif state.get(\'draining\'):\n        headline, next_step, tone = \'已暂停接收新任务\', \'已接收的队列继续执行；可恢复接收，也可停止服务。\', \'idle\'\n    elif state.get(\'ready\'):\n        headline, next_step, tone = \'可以生成\', \'在控制面板提交新内容，或按任务 ID 查询已有任务。\', \'success\'\n    elif state.get(\'api_ready\'):\n        accepting = (state.get(\'queue\') or {}).get(\'admission_open\') is True\n        headline, next_step, tone = (\'API 可用 · 生成尚未就绪\',\n            (\'可先查询或排队；\' if accepting else \'可先查询；\') + \'模型、公网的当前能力分别见下方。\', \'running\')\n    elif state:\n        headline, next_step, tone = \'服务正在准备\', \'刷新看板观察进展；不要重复启动。\', \'running\'\n    elif not context.get(\'preflight_ok\'):\n        headline, next_step, tone = \'尚未检查资源\', \'在控制面板先运行「资源与驻留方案」。\', \'idle\'\n    elif not context.get(\'install_ok\'):\n        headline, next_step, tone = \'资源检查通过\', \'在控制面板运行「安装与检查」。\', \'idle\'\n    else:\n        headline, next_step, tone = \'准备工作已完成\', \'在控制面板选择是否开放公网，再启动服务。\', \'idle\'\n\n    def module(name, alive_key, ready_key=None):\n        alive, ready = state.get(alive_key), state.get(ready_key) if ready_key else None\n        if not state:\n            status, detail = (\'尚未读取\' if context.get(\'placeholder\') else \'未启动\'), \'尚无运行观察\'\n        elif ready is True:\n            status, detail = \'可用\', \'进程运行中\' if alive is True else \'存活状态未确认\'\n        elif alive is True:\n            status, detail = (\'运行中\', \'调度与控制\') if ready_key is None else (\'准备中\', \'进程运行，能力未就绪\')\n        elif alive is False:\n            status, detail = \'已停止\', \'进程未运行\'\n        else:\n            status, detail = \'未确认\', \'缺少当前观察\'\n        if stale or failed_read:\n            status, detail = \'待刷新\', \'不据旧状态判断就绪\'\n        return (\'<div class="h3-module"><span class="h3-label">\' + escape(name) + \'</span><strong>\'\n                + escape(status) + \'</strong><div class="h3-subtle">\' + escape(detail) + \'</div></div>\')\n\n    if stale or failed_read:\n        public = \'待刷新\'\n    elif not state:\n        public = \'尚未观察\'\n    elif not state.get(\'public_requested\'):\n        public = \'未启用\'\n    elif state.get(\'public_api_ready\'):\n        public = \'API 探测通过\'\n    elif state.get(\'tunnel_connected\'):\n        public = \'隧道已连接 · API 待验证\'\n    else:\n        public = \'准备中\'\n    queue = state.get(\'queue\') or {}\n    summary = [(\'公网\', public)]\n    if queue and not stale and not failed_read:\n        summary += [(\'任务\', f"排队 {queue.get(\'queued\', \'—\')} · 执行 {queue.get(\'running\', \'—\')}")]\n        if queue.get(\'admission_open\') is False:\n            summary.append((\'接收任务\', \'已暂停\'))\n    resources = context.get(\'resource_report\') or {}\n    if resources:\n        summary += [(\'GPU\', resources.get(\'gpu\', \'未记录\')),\n                    (\'可用内存\', str(resources.get(\'available_ram_gib\', \'—\')) + \' GiB（预检）\')]\n    resource_text = \'\'.join(\'<span><span class="h3-label">\' + escape(k) + \'</span>\' + escape(v) + \'</span>\' for k, v in summary)\n    stamp = state.get(\'observed_at\')\n    if isinstance(stamp, (int, float)):\n        try:\n            stamp = datetime.fromtimestamp(stamp, timezone(timedelta(hours=8))).strftime(\'%m-%d %H:%M:%S 北京时间\')\n        except (OverflowError, OSError, ValueError):\n            stamp = \'未记录\'\n    else:\n        stamp = \'尚无服务快照\'\n    body = (STYLE + \'<section class="h3-card h3-board" data-tone="\' + tone + \'" aria-label="运行看板">\'\n            \'<div class="h3-kicker">当前状态</div><h2>\' + escape(headline) + \'</h2>\'\n            \'<div class="h3-next"><b>下一步</b>\u3000\' + escape(next_step) + \'</div><div class="h3-modules">\'\n            + module(\'总管理\', \'manager_alive\') + module(\'API 后台\', \'server_alive\', \'api_ready\')\n            + module(\'大模型容器\', \'model_alive\', \'model_ready\') + \'</div>\'\n            \'<div class="h3-summary">\' + resource_text + \'</div>\')\n    if activity:\n        activity_tone = activity.get(\'tone\', \'idle\')\n        if activity_tone not in {\'idle\', \'running\', \'success\', \'error\'}:\n            activity_tone = \'idle\'\n        body += (\'<div class="h3-activity" data-tone="\' + activity_tone + \'"><div class="h3-label">最近操作</div>\'\n                 \'<div class="h3-head"><span class="h3-title">\' + escape(activity.get(\'title\', \'\'))\n                 + \'</span><span class="h3-badge">\' + escape(activity.get(\'badge\', \'\')) + \'</span></div>\')\n        if activity.get(\'note\'):\n            body += \'<div class="h3-note">\' + escape(activity[\'note\']) + \'</div>\'\n        fields = activity.get(\'fields\') or ()\n        if fields:\n            body += \'<div class="h3-fields">\' + \'\'.join(\'<div><span class="h3-label">\' + escape(k)\n                + \'</span><span class="h3-value">\' + escape(v) + \'</span></div>\' for k, v in fields[:4]) + \'</div>\'\n        body += \'</div>\'\n    return body + \'<div class="h3-stamp">服务观察：\' + escape(stamp) + \' · 按需刷新，不执行控制命令</div></section>\'\n\n\nclass Dashboard:\n    def __init__(self, context_reader):\n        self.context_reader = context_reader\n        self.activity = None\n        self._handle = None\n\n    def refresh(self):\n        from IPython.display import HTML, display\n        global _known_secrets\n        context = dict(self.context_reader())\n        host = context.pop(\'host\', None)\n        if host is not None:\n            key = getattr(host, \'api_key\', None)\n            if key:\n                _known_secrets = tuple(set((*_known_secrets[-3:], str(key))))\n            try:\n                context[\'state\'] = host.snapshot()\n            except Exception as error:\n                context[\'state\'] = {}\n                context[\'observation_error\'] = True\n                _remember(\'看板状态读取失败\', str(error))\n        rendered = HTML(render_dashboard(context, self.activity))\n        if self._handle is None:\n            self._handle = display(rendered, display_id=\'h3-dashboard\')\n        else:\n            self._handle.update(rendered)\n        return context\n\n\ndef mount_dashboard(context_reader):\n    global _dashboard\n    _dashboard = Dashboard(context_reader)\n    _dashboard.refresh()\n    return _dashboard\n\n\nclass StatusCard:\n    def __init__(self, title, **kwargs):\n        self._handle = None\n        self.update(title, **kwargs)\n\n    def update(self, title, **kwargs):\n        from IPython.display import HTML, display\n        activity = _activity(title, kwargs)\n        if _dashboard is not None:\n            _dashboard.activity = activity\n            _dashboard.refresh()\n        elif self._handle is None:\n            self._handle = display(HTML(card_html(**activity)), display_id=True)\n        else:\n            self._handle.update(HTML(card_html(**activity)))\n\n\ndef elapsed_text(seconds):\n    seconds = max(0, int(seconds))\n    return f"{seconds // 60}分{seconds % 60:02d}秒" if seconds >= 60 else f"{seconds}秒"\n\n\ndef log_tail(path, limit=6000):\n    path = Path(path)\n    if not path.is_file():\n        return ""\n    with path.open("rb") as source:\n        source.seek(max(0, path.stat().st_size - limit))\n        text = source.read().decode("utf-8", errors="replace")\n    return re.sub(r"\\x1b\\[[0-?]*[ -/]*[@-~]", "", text).replace("\\r", "\\n")\n\n\ndef latest_stage(path):\n    lines = [line.strip() for line in log_tail(path).splitlines() if line.strip()]\n    for line in reversed(lines):\n        if "BF16 staged:" in line:\n            return line.split("BF16 staged:", 1)[1].strip()[:145]\n        if "status=" in line or "API_SMOKE_TEST_PASSED" in line:\n            return line[:145]\n    return lines[-1][:145] if lines else "进程已启动，等待首条日志"\n\n\ndef run_logged(command, log_path, *, title, cwd=None, env=None, card=None, status_reader=None):\n    """Run a real child process; persist logs and update a bounded status card."""\n    path = Path(log_path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    status = card or StatusCard(title, badge="执行中", tone="running", note="完整输出写入日志。")\n    started = time.monotonic()\n    with path.open("a", encoding="utf-8") as output:\n        output.write("\\n--- " + title + " ---\\n")\n        output.flush()\n        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)\n        observer = _process_observer\n        if observer is not None:\n            observer.process_started(process)\n        try:\n            while process.poll() is None:\n                if observer is not None:\n                    observer.process_progress(process)\n                current = []\n                if status_reader is not None:\n                    try:\n                        current = status_reader()\n                    except (OSError, ValueError):\n                        pass\n                status.update(title, badge="执行中", tone="running", fields=\n                    [("已用时间", elapsed_text(time.monotonic() - started))]\n                    + current)\n                time.sleep(2)\n        except BaseException:\n            process.terminate()\n            try:\n                process.wait(timeout=15)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n            status.update(title, badge="已中断", tone="error", note="操作进程已停止；已提交的远程任务可能仍在执行。", fields=[("日志", str(path))])\n            raise\n        finally:\n            if observer is not None:\n                observer.process_finished(process)\n    if process.returncode:\n        status.update(title, badge="未完成", tone="error", note="查看日志中的实际错误后再重试。",\n                      fields=[("日志", str(path))], details=log_tail(path), details_label="展开最后的错误日志")\n        raise RuntimeError(f"{title}退出码 {process.returncode}；完整日志：{path}")\n    status.update(title, badge="完成", tone="success", fields=[\n        ("已用时间", elapsed_text(time.monotonic() - started)), ("完整日志", str(path)),\n    ])\n    return process.returncode\n\n\ndef json_probe(command, log_path, *, cwd=None, env=None, card=None):\n    """Capture a short JSON-producing check without dumping full notebook output."""\n    process = subprocess.Popen(command, cwd=cwd, env=env, text=True,\n                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    observer = _process_observer\n    try:\n        if observer is not None:\n            observer.process_started(process)\n        stdout, stderr = process.communicate(timeout=180)\n    except BaseException:\n        if process.poll() is None:\n            process.kill()\n            process.communicate(timeout=10)\n        raise\n    finally:\n        if observer is not None:\n            observer.process_finished(process)\n    with Path(log_path).open("a", encoding="utf-8") as log:\n        log.write(stdout + "\\n" + stderr + "\\n")\n    if process.returncode:\n        if card is not None:\n            card.update("依赖检查未通过", badge="需要处理", tone="error", note="检查日志中的版本或导入错误后再继续。",\n                        fields=[("日志", str(log_path))], details=stderr[-4000:], details_label="展开检查错误")\n        raise RuntimeError(f"依赖检查失败；请在日志区查看：{log_path}")\n    return json.loads(stdout.strip().splitlines()[-1])\n', 'pipeline_backend.py': '"""CPU prepare -> exclusive GPU inference -> CPU encoding, with per-job ownership.\n\nReference/codec behavior follows Diffusers v0.43 references.py/export_utils.py.\nIts encode_video has no thread-budget argument; this local PyAV writer sets\ncodec thread_count explicitly and avoids stacking every output frame again.\nResource numbers are conservative admission estimates, not calibrated optima.\n"""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom contextlib import contextmanager, nullcontext\nfrom dataclasses import dataclass\nfrom fractions import Fraction\nimport json\nimport gc\nimport math\nimport os\nfrom pathlib import Path\n\nfrom runtime_options import RuntimeSupport, measured_generation\nfrom generation_options import requested_options, validate_generation_options, effective_options\nfrom adapter_config import select_reference_setup, select_loaded_adapters\n\nMIB, GIB = 1024**2, 1024**3\n\n\n@dataclass\nclass Prepared:\n    parameters: dict\n    references: list\n    normalized: list\n    metrics: object\n    cpu_threads: int = 1\n    cleaned: bool = False\n\n\n@dataclass\nclass CPUResult:\n    video: list\n    audio: object\n    sampling_rate: int\n    metrics: object\n    cpu_threads: int\n    cpu_handoff_complete: bool = False\n    cleaned: bool = False\n\n\ndef _audio_from_container(av, container, stream):\n    import numpy as np\n    import torch\n    stream.codec_context.thread_count = 1\n    rate = int(stream.codec_context.sample_rate)\n    if not 0 < rate <= 96000 or len(stream.layout.channels) > 2:\n        raise ValueError("Reference soundtrack exceeds 96 kHz / stereo")\n    resampler = av.AudioResampler(format="fltp", layout=stream.layout, rate=rate)\n    chunks, samples = [], 0\n    def append(frames):\n        nonlocal samples\n        for item in frames:\n            value = item.to_ndarray()\n            samples += value.shape[-1]\n            if samples > math.ceil(15.1 * rate):\n                raise ValueError("Decoded reference soundtrack exceeds 15 seconds")\n            chunks.append(value)\n    for frame in container.decode(stream):\n        append(resampler.resample(frame))\n    append(resampler.resample(None))\n    if not chunks:\n        raise ValueError("Reference soundtrack is empty")\n    return torch.from_numpy(np.concatenate(chunks, axis=-1).astype(np.float32, copy=False)), rate\n\n\ndef decode_reference(kind, path, classes):\n    """CPU-only decoding with one native codec thread and decoded-size guards."""\n    if kind == "image":\n        return classes[kind].from_file(str(path))\n    import av\n    import numpy as np\n    with av.open(str(path)) as container:\n        if kind == "audio":\n            audio, rate = _audio_from_container(av, container, container.streams.audio[0])\n            return classes[kind](audio=audio, sample_rate=rate)\n        stream = container.streams.video[0]\n        stream.codec_context.thread_count = 1\n        frames = []\n        for frame in container.decode(stream):\n            if len(frames) >= 362 or frame.width * frame.height > 960 * 544:\n                raise ValueError("Normalized reference exceeds bounded decoded geometry")\n            frames.append(frame.to_ndarray(format="rgb24"))\n        if not frames:\n            raise ValueError("Reference video is empty")\n        fps = float(stream.average_rate or stream.guessed_rate)\n        audio, rate = None, None\n        if container.streams.audio:\n            container.seek(0)\n            audio, rate = _audio_from_container(av, container, container.streams.audio[0])\n        return classes[kind](frames=np.stack(frames), fps=fps, audio=audio, sample_rate=rate)\n\n\ndef cpu_media(video, audio, sampling_rate, torch):\n    """Copy outputs into CPU-owned numpy buffers; never retain PipelineState."""\n    import numpy as np\n    def array(value):\n        if isinstance(value, torch.Tensor):\n            value = value.detach().to(device="cpu", copy=True)\n            if value.is_floating_point():\n                value = value.float()\n            value = value.numpy()\n        return np.array(value, copy=True)\n    frames = []\n    for frame in video:\n        pixels = array(frame)\n        if pixels.ndim != 3 or pixels.shape[-1] != 3:\n            raise ValueError("Expected HWC RGB video output")\n        if pixels.dtype != np.uint8:\n            if not np.isfinite(pixels).all() or pixels.min() < 0 or pixels.max() > 1:\n                raise ValueError("Float video output must be finite and normalized")\n            pixels = (pixels * 255).round().astype(np.uint8)\n        frames.append(pixels)\n    waveform = array(audio).astype(np.float32, copy=False)\n    if waveform.ndim != 2 or waveform.shape[0] != 2 or not np.isfinite(waveform).all():\n        raise ValueError("Expected finite stereo audio output")\n    if not frames or not 0 < int(sampling_rate) <= 96000:\n        raise ValueError("Invalid output media")\n    return frames, waveform, int(sampling_rate)\n\n\ndef encode_cpu_media(result, output_path):\n    """Same H.264/AAC formats as official exporter, explicitly bounded threads."""\n    import av\n    import numpy as np\n    if not result.cpu_handoff_complete or result.cleaned:\n        raise RuntimeError("Encoding requires a completed CPU ownership handoff")\n    with av.open(str(output_path), mode="w") as container:\n        stream = container.add_stream("libx264", rate=24)\n        stream.width, stream.height = result.video[0].shape[1], result.video[0].shape[0]\n        stream.pix_fmt = "yuv420p"\n        stream.codec_context.thread_count = result.cpu_threads\n        audio_stream = container.add_stream("aac", rate=result.sampling_rate)\n        cc = audio_stream.codec_context\n        cc.sample_rate, cc.layout = result.sampling_rate, "stereo"\n        cc.time_base, cc.thread_count = Fraction(1, result.sampling_rate), 1\n        for pixels in result.video:\n            frame = av.VideoFrame.from_ndarray(pixels, format="rgb24")\n            for packet in stream.encode(frame):\n                container.mux(packet)\n        for packet in stream.encode():\n            container.mux(packet)\n        samples = (np.clip(result.audio.T, -1, 1) * 32767).astype(np.int16)\n        audio_frame = av.AudioFrame.from_ndarray(np.ascontiguousarray(samples).reshape(1, -1), format="s16", layout="stereo")\n        audio_frame.sample_rate = result.sampling_rate\n        resampler = av.AudioResampler(format=cc.format or "fltp", layout=cc.layout or "stereo", rate=cc.sample_rate)\n        next_pts = 0\n        for frame in resampler.resample(audio_frame) + resampler.resample(None):\n            if frame.pts is None:\n                frame.pts = next_pts\n            next_pts += frame.samples\n            frame.sample_rate = result.sampling_rate\n            for packet in audio_stream.encode(frame):\n                container.mux(packet)\n        for packet in audio_stream.encode():\n            container.mux(packet)\n    if not Path(output_path).is_file() or not Path(output_path).stat().st_size:\n        raise RuntimeError("Encoder produced no video")\n\n\nclass PipelineBackendSupport(RuntimeSupport):\n    pipeline_model_peak_ram_bytes = 144 * GIB\n    resource_residency = "resident"\n\n    def validate_generation_request(self, *, steps, attention_backend=None, acceleration_lora=None):\n        values = {"steps": steps}\n        if attention_backend is not None:\n            values["attention_backend"] = attention_backend\n        if acceleration_lora is not None:\n            values["acceleration_lora"] = acceleration_lora\n        return validate_generation_options(requested_options(values), self.adapter_settings)\n\n    def _apply_generation_transformer(self, effective, *, audit=True):\n        transformer = getattr(self.pipe, "transformer_ref", None)\n        if transformer is None:\n            raise RuntimeError("The generation transformer is unavailable")\n        # Always set native explicitly, so forced Flash never leaks into auto.\n        transformer.set_attention_backend(effective["attention_backend"])\n        select_loaded_adapters(self.pipe, self.adapter_settings, effective["acceleration_lora"])\n        if audit and self._active_metrics is not None:\n            self._active_metrics.data["generation_options"]["effective"] = dict(effective)\n\n    @contextmanager\n    def generation_context(self, prepared):\n        """Mutate and restore shared model state only inside infer\'s GPU lease.\n\n        CPU preparation can overlap other jobs; it must not change adapters,\n        attention processors, reference setup, or scheduler state. Staged\n        residency applies the same choices after its transformer is loaded.\n        """\n        if getattr(self, "_active_generation_options", None) is not None:\n            raise RuntimeError("Generation configuration already owns the GPU lease")\n        requested = requested_options(prepared.parameters)\n        validate_generation_options(requested, self.adapter_settings)\n        effective = effective_options(requested, self.adapter_settings, self.runtime_options)\n        prepared.metrics.data.setdefault("generation_options", {"requested": dict(requested), "effective": None})\n        original = self.pipe.blocks.sub_blocks["before_encode"]\n        self._active_generation_options = effective\n        try:\n            select_reference_setup(self.pipe, requested["acceleration_lora"])\n            if getattr(self.pipe, "transformer_ref", None) is not None:\n                self._apply_generation_transformer(effective)\n            yield\n        finally:\n            try:\n                self.pipe.blocks.sub_blocks["before_encode"] = original\n                if getattr(self.pipe, "transformer_ref", None) is not None:\n                    defaults = effective_options({"attention_backend": "auto",\n                        "acceleration_lora": self.adapter_settings.acceleration, "steps": requested["steps"]},\n                        self.adapter_settings, self.runtime_options)\n                    self._apply_generation_transformer(defaults, audit=False)\n            finally:\n                self._active_generation_options = None\n\n    def configure_cpu_budget(self, gpu_threads):\n        """Configure native CPU pools exactly once, before any model is loaded.\n\n        GPU work also occupies host CPU capacity. CPU preparation/encoding use\n        bounded codecs and numpy copies, not multithreaded torch/BLAS operators.\n        Environment defaults cover libraries imported later; the caller can\n        audit again after load without changing any process-wide setting.\n        """\n        if isinstance(gpu_threads, bool) or not isinstance(gpu_threads, int) or gpu_threads < 1:\n            raise ValueError("GPU host CPU reservation must be a positive integer")\n        previous = getattr(self, "_native_cpu_requested", None)\n        if previous is not None:\n            if previous != gpu_threads:\n                raise RuntimeError("Native CPU pools cannot be reconfigured during this model lifetime")\n            return self.cpu_budget_snapshot()\n        if self._loaded or self.pipe is not None:\n            raise RuntimeError("Native CPU pools must be configured before model loading")\n        # Keep the hardware gate before torch imports/downloads, including the\n        # real Windows CPU integration\'s explicit Linux/NVIDIA rejection.\n        from preflight import check_resources\n        check_resources(residency=self.resource_residency)\n        environment = {name: str(gpu_threads) for name in (\n            "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",\n            "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS")}\n        environment.update(OMP_MAX_ACTIVE_LEVELS="1", OMP_DYNAMIC="FALSE", MKL_DYNAMIC="FALSE",\n                           TOKENIZERS_PARALLELISM="false", RAYON_NUM_THREADS=str(gpu_threads))\n        os.environ.update(environment)\n        import numpy  # Load the CPU pools we use before applying/readback.\n        import torch\n        from threadpoolctl import threadpool_limits\n        torch.set_num_threads(gpu_threads)\n        interop_note = "set_before_load"\n        try:\n            torch.set_num_interop_threads(1)\n        except RuntimeError:\n            if torch.get_num_interop_threads() != 1:\n                raise RuntimeError("Cannot enforce one torch interop thread before model loading") from None\n            interop_note = "already_initialized_at_one"\n        self._native_cpu_limiter = threadpool_limits(limits=gpu_threads)\n        self._native_cpu_requested = gpu_threads\n        self._native_cpu_environment = environment\n        self._native_cpu_interop_note = interop_note\n        try:\n            return self.cpu_budget_snapshot()\n        except BaseException:\n            self._native_cpu_requested = None\n            raise\n\n    def cpu_budget_snapshot(self):\n        """Read and validate real pool limits; never change them in running jobs."""\n        requested = getattr(self, "_native_cpu_requested", None)\n        if requested is None:\n            raise RuntimeError("Native CPU budget has not been configured")\n        import torch\n        from threadpoolctl import threadpool_info\n        pools = [{key: pool.get(key) for key in ("user_api", "internal_api", "prefix", "version", "num_threads")}\n                 for pool in threadpool_info()]\n        actual = torch.get_num_threads()\n        interop = torch.get_num_interop_threads()\n        if actual != requested or interop != 1:\n            raise RuntimeError(f"Torch CPU budget differs from reservation: intra={actual}, interop={interop}, reserved={requested}")\n        for pool in pools:\n            threads = pool["num_threads"]\n            if isinstance(threads, bool) or not isinstance(threads, int) or not 1 <= threads <= requested:\n                raise RuntimeError(f"Native pool CPU limit is not confirmed: {pool[\'internal_api\']}={threads}, reserved={requested}")\n        return {"configured": True, "configured_before_load": True, "gpu_host_cpu_threads": requested,\n                "torch_num_threads": actual, "torch_num_interop_threads": interop,\n                "interop_configuration": self._native_cpu_interop_note,\n                "native_pools": pools, "environment_defaults": dict(self._native_cpu_environment),\n                "scope": "observed native CPU pools; codec lanes have separate explicit thread budgets"}\n\n    @property\n    def pipeline_model_headroom_bytes(self):\n        import psutil\n        return max(0, self.pipeline_model_peak_ram_bytes - psutil.Process().memory_info().rss)\n\n    def estimate_resources(self, parameters):\n        """Bounds retained media + decode/transfer/encode copies; excludes model."""\n        from PIL import Image\n        references = parameters["references"]\n        durations, retained, source_bytes, normalized_bytes, pipeline_media = [], 0, 0, 0, 0\n        frames = self._aligned_frames(parameters["seconds"])\n        for item in references:\n            path, kind = Path(item["path"]), item["kind"]\n            duration = self._reference_probe(path, kind)\n            durations.append((kind, duration))\n            source_bytes += path.stat().st_size\n            if kind == "image":\n                with Image.open(path) as image:\n                    retained += image.width * image.height * 4\n                    # Official conditioning upscales even tiny images to a 2048\n                    # short edge. Reserve resized RGB plus float/temporary copies.\n                    ratio = max(image.width, image.height) / min(image.width, image.height)\n                    pipeline_media += 2048 * (math.ceil(2048 * ratio / 32) * 32) * 20\n            elif kind == "video":\n                size = (math.ceil(duration * 24) + 2) * 960 * 544 * 3 + math.ceil(duration * 32000) * 2 * 4\n                retained += size\n                normalized_bytes += size + MIB\n                # The GPU workflow can resize reference video again to the\n                # official 1344*768 canvas; it is not a CPU preparation stage.\n                pipeline_media += min(frames, math.ceil(duration * 24) + 2) * 1344 * 768 * 3 * 8\n            elif kind == "audio":\n                retained += math.ceil(duration * 96000) * 2 * 4\n            else:\n                raise ValueError("Unknown reference modality")\n        for kind in ("video", "audio"):\n            if sum(d for k, d in durations if k == kind) > 15:\n                raise ValueError("Total reference duration per modality must be <=15 seconds")\n        output = frames * parameters["width"] * parameters["height"] * 3 + math.ceil(frames / 24 * 96000) * 2 * 4\n        return {"ram_bytes": 3 * retained + 4 * output + pipeline_media + 256 * MIB,\n                "cpu_threads": 1, "disk_bytes": source_bytes + normalized_bytes + output + 4 * MIB,\n                "reference_retained_bytes": retained, "output_retained_bytes": output,\n                "pipeline_reference_workspace_bytes": pipeline_media,\n                "estimate_kind": "conservative decoded-media admission estimate; not calibrated physical optimum"}\n\n    def prepare(self, *, prompt, references, seconds, width, height, steps, seed, cpu_threads=1,\n                attention_backend=None, acceleration_lora=None):\n        options = {}\n        if attention_backend is not None:\n            options["attention_backend"] = attention_backend\n        if acceleration_lora is not None:\n            options["acceleration_lora"] = acceleration_lora\n        self.validate_request(steps=steps, **options)\n        if self.pipe is None:\n            raise RuntimeError("The model has not loaded")\n        if isinstance(cpu_threads, bool) or not isinstance(cpu_threads, int) or cpu_threads < 1:\n            raise ValueError("cpu_threads must be a positive reserved CPU budget")\n        from diffusers.modular_pipelines.minimax_h3 import MiniMaxH3ImageReference, MiniMaxH3VideoReference, MiniMaxH3AudioReference\n        classes = {"image": MiniMaxH3ImageReference, "video": MiniMaxH3VideoReference, "audio": MiniMaxH3AudioReference}\n        parameters = dict(prompt=prompt, seconds=seconds, width=width, height=height, steps=steps, seed=seed, **options)\n        metrics = self._active_metrics or self._new_metrics("generation")\n        metrics.data["timing"] = "host_wall_seconds; inference synchronizes at CPU handoff"\n        metrics.data["pipeline_stage_contract"] = "cpu_prepare / exclusive_gpu_infer / cpu_encode_validate"\n        metrics.data["generation_options"] = {"requested": requested_options(parameters), "effective": None}\n        prepared = Prepared(parameters, [], [], metrics, cpu_threads)\n        try:\n            with self.metrics_context(metrics), self._stage("references.prepare"):\n                durations = [self._reference_probe(Path(item["path"]), item["kind"]) for item in references]\n                for kind in ("video", "audio"):\n                    if sum(d for item, d in zip(references, durations) if item["kind"] == kind) > 15:\n                        raise ValueError("Total reference duration per modality must be <=15 seconds")\n                normalized_targets = {}\n                for index, item in enumerate(references):\n                    if item["kind"] == "video":\n                        path = Path(item["path"])\n                        target = path.with_name(path.stem + f"_ref{index}_24fps.mp4")\n                        prepared.normalized.append(target)\n                        normalized_targets[index] = target\n                def decode(index):\n                    item = references[index]\n                    with metrics.stage(f"child.prepare.{index}"):\n                        path = Path(item["path"])\n                        if item["kind"] == "video":\n                            path = self._normalize_reference(path, target=normalized_targets[index])\n                        return decode_reference(item["kind"], path, classes)\n                prepared.references = [None] * len(references)\n                lanes = min(cpu_threads, len(references))\n                metrics.record_schedule({"phase": "prepare", "workers": lanes, "child_count": len(references),\n                                         "reason": "explicit CPU budget; each native decoder/normalizer uses one thread"})\n                if lanes <= 1:\n                    for index in range(len(references)):\n                        prepared.references[index] = decode(index)\n                else:\n                    with ThreadPoolExecutor(max_workers=lanes, thread_name_prefix="h3-reference") as pool:\n                        futures = {pool.submit(decode, i): i for i in range(len(references))}\n                        for future in as_completed(futures):\n                            prepared.references[futures[future]] = future.result()\n            return prepared\n        except BaseException as error:\n            metrics.finish(error)\n            self.cleanup(prepared=prepared)\n            raise\n\n    def infer(self, prepared):\n        import torch\n        if prepared.cleaned:\n            raise RuntimeError("Prepared references have already been released")\n        raw = None\n        try:\n            with self.metrics_context(prepared.metrics), self._stage("pipeline"), torch.inference_mode():\n                configuration = self.generation_context(prepared) if hasattr(self, "adapter_settings") else nullcontext()\n                with configuration:\n                    try:\n                        raw = self._infer_raw(prepared)\n                        video, audio, rate = cpu_media(raw["videos"][0], raw["audio"][0], raw["sampling_rate"], torch)\n                        result = CPUResult(video, audio, rate, prepared.metrics, prepared.cpu_threads)\n                    finally:\n                        raw = None\n                        prepared.references.clear()\n                        # Synchronize before restoring adapters and attention.\n                        # CPU encoding never changes model or CUDA state.\n                        gc.collect()\n                        torch.cuda.synchronize()\n                        torch.cuda.empty_cache()\n                result.cpu_handoff_complete = True\n                return result\n        except BaseException as error:\n            prepared.metrics.finish(error)\n            raise\n        finally:\n            raw = None\n\n    def encode(self, result, output_path):\n        path = Path(output_path)\n        error = None\n        try:\n            with self.metrics_context(result.metrics), self._stage("encode_mp4"):\n                encode_cpu_media(result, path)\n        except BaseException as caught:\n            error = caught\n            raise\n        finally:\n            metrics = result.metrics.finish(error)\n            self.last_metrics = metrics\n            sidecar = path.with_suffix(".metrics.json")\n            temporary = sidecar.with_suffix(sidecar.suffix + ".tmp")\n            try:\n                temporary.write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n                temporary.replace(sidecar)\n            except OSError as write_error:\n                metrics["metrics_write_error"] = type(write_error).__name__\n        return metrics\n\n    def cleanup(self, prepared=None, result=None):\n        """CPU ownership only: never collect globally or touch CUDA from this lane."""\n        if prepared is not None and not prepared.cleaned:\n            prepared.references.clear()\n            for path in prepared.normalized:\n                Path(path).unlink(missing_ok=True)\n            prepared.normalized.clear()\n            prepared.cleaned = True\n        if result is not None and not result.cleaned:\n            result.video.clear()\n            result.audio = None\n            result.cleaned = True\n\n    @measured_generation\n    def generate(self, prompt, references, output_path, seconds, width, height, steps, seed,\n                 attention_backend=None, acceleration_lora=None):\n        """Standalone compatibility: the same stages, called sequentially."""\n        prepared = result = None\n        try:\n            prepared = self.prepare(prompt=prompt, references=references, seconds=seconds,\n                                    width=width, height=height, steps=steps, seed=seed,\n                                    attention_backend=attention_backend, acceleration_lora=acceleration_lora)\n            result = self.infer(prepared)\n            return self.encode(result, output_path)\n        finally:\n            self.cleanup(prepared=prepared, result=result)\n', 'pipeline_scheduler.py': '"""Per-job working-set admission and stage leases; no global job-count cap."""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom collections import OrderedDict, deque\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport threading\nimport time\nimport uuid\n\nimport psutil\n\nfrom condition_guard import evaluate_gate, negate_observation\n\nGiB = 1024 ** 3\n\n\ndef cgroup_cpu_quotas(root=Path(\'/sys/fs/cgroup\'), membership=Path(\'/proc/self/cgroup\')):\n    """Read v1/v2 quota limits, including stricter visible parent cgroups."""\n    root = Path(root).resolve()\n    candidates = {(root, \'v2\'), (root / \'cpu\', \'v1\'), (root / \'cpu,cpuacct\', \'v1\')}\n    try:\n        records = Path(membership).read_text().splitlines()\n    except OSError:\n        records = []\n    for record in records:\n        parts = record.split(\':\', 2)\n        if len(parts) != 3:\n            continue\n        controllers, relative = parts[1], parts[2].lstrip(\'/\')\n        bases = [(root, \'v2\')] if not controllers else (\n            [(root / \'cpu\', \'v1\'), (root / \'cpu,cpuacct\', \'v1\')] if \'cpu\' in controllers.split(\',\') else [])\n        for base, version in bases:\n            leaf = (base / relative).resolve()\n            if not leaf.is_relative_to(base):\n                continue\n            for directory in (leaf, *leaf.parents):\n                if not directory.is_relative_to(base):\n                    break\n                candidates.add((directory, version))\n    limits = []\n    for directory, version in sorted(candidates):\n        try:\n            if version == \'v2\':\n                filename = directory / \'cpu.max\'\n                quota, period = filename.read_text().split()\n                if quota == \'max\':\n                    continue\n            else:\n                filename = directory / \'cpu.cfs_quota_us\'\n                quota = filename.read_text().strip()\n                period = (directory / \'cpu.cfs_period_us\').read_text().strip()\n            quota, period = int(quota), int(period)\n            if quota > 0 and period > 0:\n                limits.append({\'source\': str(filename), \'cores\': quota / period})\n        except (OSError, ValueError):\n            continue\n    return limits\n\n\ndef detect_cpu_capacity():\n    logical = max(1, os.cpu_count() or 1)\n    affinity = None\n    try:\n        affinity = len(os.sched_getaffinity(0))\n    except (AttributeError, OSError):\n        try:\n            affinity = len(psutil.Process().cpu_affinity())\n        except (AttributeError, psutil.Error):\n            pass\n    quota_limits = cgroup_cpu_quotas() if sys.platform.startswith(\'linux\') else []\n    effective = min([logical] + ([affinity] if affinity else []) + [value[\'cores\'] for value in quota_limits])\n    return {\'logical_cpus\': logical, \'affinity_cpus\': affinity, \'cgroup_quotas\': quota_limits,\n            \'effective_cpu_cores\': effective, \'usable_cpu_threads\': max(1, math.floor(effective)),\n            \'fractional_quota_note\': \'A quota below one CPU uses one throttled lane, not one full CPU\'}\n\n\nclass BudgetUnavailable(Exception):\n    def __init__(self, reason, permanent=False):\n        super().__init__(reason)\n        self.permanent = permanent\n\n\nclass PipelineBudget:\n    def __init__(self, root, *, limits=None, model_peak_ram_bytes=0):\n        self.root = Path(root)\n        self._condition = threading.Condition()\n        self._reservations = {}\n        self._reservation_ids = {}\n        self._cancellations = {}\n        self._result_bytes = {}\n        self._cpu_used = 0\n        self._gpu_owner = None\n        self._model_peak = int(model_peak_ram_bytes)\n        self._system_checks = limits is None\n        self._cpu_capacity = detect_cpu_capacity()\n        self._os_reserve = int(os.environ.get("H3_PIPELINE_OS_RESERVE_MIB", "2048")) * 1024 ** 2\n        defaults = {\n            "ram_bytes": int(os.environ.get("H3_PIPELINE_RAM_MIB", "8192")) * 1024 ** 2,\n            "disk_bytes": max(0, shutil.disk_usage(self.root).free - GiB),\n            "cpu_threads": int(os.environ.get("H3_PIPELINE_CPU_THREADS", str(self._cpu_capacity[\'usable_cpu_threads\']))),\n        }\n        self.limits = defaults if limits is None else {**defaults, **limits}\n        if any(type(value) is not int or value <= 0 for value in self.limits.values()):\n            raise ValueError("Pipeline budgets must be positive integers")\n        self._requested_cpu_threads = self.limits[\'cpu_threads\']\n        self.limits[\'cpu_threads\'] = min(self.limits[\'cpu_threads\'], self._cpu_capacity[\'usable_cpu_threads\'])\n        self._requested_gpu_cpu_threads = int(os.environ.get(\'H3_PIPELINE_GPU_CPU_THREADS\', \'1\'))\n        if self._requested_gpu_cpu_threads < 1:\n            raise ValueError(\'GPU host CPU reservation must be positive\')\n        self.gpu_host_cpu_threads = min(self._requested_gpu_cpu_threads, self.limits[\'cpu_threads\'])\n        self.native_cpu_status = {\'configured\': False, \'reason\': \'Backend native CPU configuration pending\'}\n        self.instance_id = uuid.uuid4().hex\n        self._source_sha256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n        self._last_gates = {}\n        self._attempt_gates = OrderedDict()\n        self._active_leases = {}\n        self._released_leases = OrderedDict()\n        self._lease_history = deque(maxlen=512)\n        self._lease_event_count = 0\n\n    def _capacity_version(self):\n        payload = {\'limits\': self.limits, \'gpu_host_cpu_threads\': self.gpu_host_cpu_threads}\n        return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n    def _gate(self, identity, kind, observations, context=None):\n        # Called with our Condition held. No callbacks into other ownership domains.\n        receipt = evaluate_gate(\'pipeline.\' + kind, list(observations), observations,\n                                {**(context or {}), \'budget_instance_id\': self.instance_id,\n                                 \'attempt_id\': identity, \'resource_kind\': kind,\n                                 \'reservation_id\': self._reservation_ids.get(identity),\n                                 \'capacity_version\': self._capacity_version(),\n                                 \'source_file\': \'pipeline_scheduler.py\',\n                                 \'source_sha256\': self._source_sha256})\n        self._last_gates[receipt[\'gate_id\']] = receipt\n        key = (identity, kind)\n        self._attempt_gates[key] = receipt\n        self._attempt_gates.move_to_end(key)\n        while len(self._attempt_gates) > 128:\n            self._attempt_gates.popitem(last=False)\n        return receipt\n\n    def gate_receipt(self, identity, kind):\n        with self._condition:\n            return copy.deepcopy(self._attempt_gates.get((identity, kind)))\n\n    def reservation_witness(self, identity):\n        with self._condition:\n            return {\'owned\': identity in self._reservations,\n                    \'budget_instance_id\': self.instance_id, \'attempt_id\': identity,\n                    \'reservation_id\': self._reservation_ids.get(identity),\n                    \'resources\': copy.deepcopy(self._reservations.get(identity)),\n                    \'capacity_version\': self._capacity_version()}\n\n    def lease_witness(self, identity, kind):\n        with self._condition:\n            return copy.deepcopy(next((lease for lease in self._active_leases.values()\n                                       if lease[\'attempt_id\'] == identity and lease[\'kind\'] == kind), None))\n\n    def release_witness(self, lease_id):\n        with self._condition:\n            return copy.deepcopy(self._released_leases.get(lease_id))\n\n    def set_cancelled(self, identity, observation):\n        """Update cancellation evidence in the same domain as budget grants.\n\n        Model cancellation also owns its Event and rechecks under its own lock.\n        Updating this ledger does not interrupt a lease already granted.\n        """\n        with self._condition:\n            if identity not in self._reservations:\n                return False\n            self._cancellations[identity] = copy.deepcopy(observation)\n            self._condition.notify_all()\n            return True\n\n    def _lease_event(self, event_type, lease):\n        self._lease_event_count += 1\n        event = {key: copy.deepcopy(lease[key]) for key in\n                 (\'lease_id\', \'budget_instance_id\', \'attempt_id\', \'kind\', \'cpu_threads\',\n                  \'capacity_version\', \'decision_id\', \'reservation_id\')}\n        event.update(event_id=uuid.uuid4().hex, sequence=self._lease_event_count,\n                     type=event_type, observed_at=time.time(), monotonic_ns=time.monotonic_ns(),\n                     cpu_threads_in_use=self._cpu_used, gpu_attempt_id=self._gpu_owner,\n                     limits=dict(self.limits))\n        self._lease_history.append(event)\n        return event\n\n    def _used(self, name):\n        return sum(value[name] for value in self._reservations.values()) + (\n            sum(self._result_bytes.values()) if name == "ram_bytes" else 0)\n\n    def reserve(self, identity, resources):\n        if type(identity) is not str or not 1 <= len(identity) <= 256:\n            raise BudgetUnavailable(\'invalid_attempt_identity\', permanent=True)\n        charge = {name: resources[name] for name in ("ram_bytes", "disk_bytes", "cpu_threads")}\n        if any(type(value) is not int or value <= 0 for value in charge.values()):\n            raise BudgetUnavailable("invalid_backend_resource_estimate", permanent=True)\n        with self._condition:\n            if identity in self._reservations:\n                same = self._reservations[identity] == charge\n                self._gate(identity, \'reserve.existing\', {\'reservation_matches\': same})\n                if not same:\n                    raise BudgetUnavailable(\'reservation_identity_conflict\', permanent=True)\n                return\n            fits = all(charge[name] <= self.limits[name] for name in charge)\n            room = all(self._used(name) + charge[name] <= self.limits[name]\n                       for name in (\'ram_bytes\', \'disk_bytes\'))\n            host_available = disk_available = True\n            if self._system_checks:\n                try:\n                    # Existing task memory is not part of the resident model.\n                    rss = psutil.Process().memory_info().rss\n                    resident_model = max(0, rss - self._used(\'ram_bytes\'))\n                    future_model = max(0, self._model_peak - resident_model)\n                    host_available = psutil.virtual_memory().available >= (\n                        future_model + self._os_reserve + self._used(\'ram_bytes\') + charge[\'ram_bytes\'])\n                    disk_available = shutil.disk_usage(self.root).free >= GiB + self._used(\'disk_bytes\') + charge[\'disk_bytes\']\n                except (OSError, psutil.Error):\n                    host_available = disk_available = None\n            receipt = self._gate(identity, \'reserve\', {\n                \'estimate_within_limits\': fits, \'working_set_available\': room,\n                \'host_memory_available\': host_available, \'disk_available\': disk_available,\n                \'result_identity_unused\': identity not in self._result_bytes})\n            if not fits:\n                raise BudgetUnavailable("request_exceeds_working_set_budget", permanent=True)\n            if not room:\n                raise BudgetUnavailable("resource_budget_exhausted")\n            if host_available is False:\n                raise BudgetUnavailable(\'live_host_memory_pressure\')\n            if disk_available is False:\n                raise BudgetUnavailable(\'live_disk_pressure\')\n            if not receipt[\'allowed\']:\n                raise BudgetUnavailable(\'reservation_conditions_unconfirmed\')\n            self._reservations[identity] = charge\n            self._reservation_ids[identity] = uuid.uuid4().hex\n            self._cancellations[identity] = False\n\n    def finish(self, identity, result_bytes=0):\n        with self._condition:\n            if any(lease[\'attempt_id\'] == identity for lease in self._active_leases.values()):\n                raise BudgetUnavailable(\'attempt_still_holds_stage_lease\')\n            self._reservations.pop(identity, None)\n            self._reservation_ids.pop(identity, None)\n            self._cancellations.pop(identity, None)\n            if result_bytes:\n                self._result_bytes[identity] = result_bytes\n            self._condition.notify_all()\n\n    def acknowledge(self, identity):\n        with self._condition:\n            self._result_bytes.pop(identity, None)\n            self._condition.notify_all()\n\n    @contextmanager\n    def stage(self, identity, kind, *, cpu_threads=1, cancelled=lambda: False,\n              cancellation_event=None, context=None):\n        if kind not in {\'cpu\', \'gpu\'}:\n            raise ValueError(\'Unknown resource stage\')\n        charge = self.gpu_host_cpu_threads if kind == \'gpu\' else cpu_threads\n        if type(charge) is not int or not 1 <= charge <= self.limits[\'cpu_threads\']:\n            raise ValueError(\'Stage CPU requirement exceeds the available CPU budget\')\n        acquired = False\n        lease_id = None\n        with self._condition:\n            reservation_id = self._reservation_ids.get(identity)\n        while True:\n            # Legacy arbitrary callbacks may take a model lock, so never invoke\n            # them while holding the budget lock. Their sampled state does not\n            # claim global atomicity; the model rechecks before actual effects.\n            cancellation = None\n            if cancellation_event is None:\n                try:\n                    cancellation = cancelled()\n                except Exception:\n                    pass\n            with self._condition:\n                if cancellation_event is not None:\n                    # Event.is_set is a local flag read, never a foreign callback.\n                    cancellation = (threading.Event.is_set(cancellation_event)\n                                    if type(cancellation_event) is threading.Event else None)\n                observations = {\n                    \'reservation_owned\': reservation_id is not None and self._reservation_ids.get(identity) == reservation_id,\n                    \'cancellation_clear\': negate_observation(self._cancellations.get(identity)),\n                    \'not_cancelled\': negate_observation(cancellation),\n                    \'attempt_stage_idle\': not any(lease[\'attempt_id\'] == identity for lease in self._active_leases.values()),\n                    \'cpu_grant_available\': self._cpu_used + charge <= self.limits[\'cpu_threads\'],\n                }\n                if kind == \'gpu\':\n                    observations[\'gpu_grant_available\'] = self._gpu_owner is None\n                receipt = self._gate(identity, \'stage.\' + kind, observations, context)\n                if receipt[\'allowed\']:\n                    # The declaration and its resource effect share this lock.\n                    self._cpu_used += charge\n                    if kind == \'gpu\':\n                        self._gpu_owner = identity\n                    lease_id = uuid.uuid4().hex\n                    lease = {\'owned\': True, \'lease_id\': lease_id, \'attempt_id\': identity, \'kind\': kind,\n                             \'budget_instance_id\': self.instance_id, \'reservation_id\': reservation_id,\n                             \'cpu_threads\': charge, \'capacity_version\': receipt[\'context\'][\'capacity_version\'],\n                             \'decision_id\': receipt[\'decision_id\'], \'grant_receipt\': receipt,\n                             \'granted_at\': time.time(), \'granted_monotonic_ns\': time.monotonic_ns()}\n                    self._active_leases[lease_id] = lease\n                    self._lease_event(\'lease.acquired\', lease)\n                    acquired = True\n                    break\n                if any(receipt[\'conditions\'][name] != \'true\' for name in\n                       (\'reservation_owned\', \'cancellation_clear\', \'not_cancelled\')):\n                    break\n                # Only actual contention waits; wait releases the lock and thus\n                # cannot serialize unrelated CPU/GPU releases or admissions.\n                self._condition.wait(.05)\n        try:\n            yield acquired\n        finally:\n            if acquired:\n                with self._condition:\n                    lease = self._active_leases.get(lease_id)\n                    if lease is None or lease[\'attempt_id\'] != identity or lease[\'kind\'] != kind:\n                        raise RuntimeError(\'Stage lease ownership was lost\')\n                    if kind == "gpu":\n                        if self._gpu_owner != identity:\n                            raise RuntimeError(\'GPU lease ownership was lost\')\n                        self._gpu_owner = None\n                    self._cpu_used -= charge\n                    self._active_leases.pop(lease_id)\n                    lease = {**lease, \'owned\': False, \'released_at\': time.time(),\n                             \'released_monotonic_ns\': time.monotonic_ns()}\n                    self._released_leases[lease_id] = lease\n                    while len(self._released_leases) > 256:\n                        self._released_leases.popitem(last=False)\n                    self._lease_event(\'lease.released\', lease)\n                    self._condition.notify_all()\n\n    def snapshot(self):\n        with self._condition:\n            used = {name: self._used(name) for name in ("ram_bytes", "disk_bytes")}\n            return {"limits": dict(self.limits), "reserved": used,\n                    \'budget_instance_id\': self.instance_id,\n                    \'capacity_version\': self._capacity_version(),\n                    \'last_gate_receipts\': copy.deepcopy(self._last_gates),\n                    \'active_leases\': copy.deepcopy(list(self._active_leases.values())),\n                    \'lease_history\': {\'capacity\': self._lease_history.maxlen,\n                                      \'events\': copy.deepcopy(list(self._lease_history)),\n                                      \'count\': self._lease_event_count,\n                                      \'dropped\': self._lease_event_count - len(self._lease_history),\n                                      \'history_complete\': self._lease_event_count == len(self._lease_history)},\n                    "free": {name: self.limits[name] - used[name] for name in used},\n                    "cpu_threads_in_use": self._cpu_used, "gpu_attempt_id": self._gpu_owner,\n                    "gpu_host_cpu_threads_in_use": self.gpu_host_cpu_threads if self._gpu_owner else 0,\n                    "cpu_stage_threads_in_use": self._cpu_used - (self.gpu_host_cpu_threads if self._gpu_owner else 0),\n                    "cpu_count_scope": "Reserved native execution lanes; not sampled CPU utilization",\n                    "gpu_host_cpu_threads": self.gpu_host_cpu_threads,\n                    "requested_cpu_threads": self._requested_cpu_threads,\n                    "requested_gpu_host_cpu_threads": self._requested_gpu_cpu_threads,\n                    "cpu_capacity": self._cpu_capacity, \'native_cpu\': self.native_cpu_status,\n                    "io_budget": {"scope": "disk_capacity_only", "bandwidth_calibrated": False,\n                                  "bytes_per_second_limit": None,\n                                  "optimality": "Conservative admission estimates; no mathematical optimum claimed"},\n                    "admitted_attempts": len(self._reservations),\n                    "unacknowledged_result_bytes": sum(self._result_bytes.values()),\n                    "model_peak_ram_bytes": self._model_peak,\n                    "os_reserve_bytes": self._os_reserve,\n                    "live_pressure_checks": self._system_checks}\n\n    def has_capacity(self):\n        state = self.snapshot()\n        return all(value > 0 for value in state["free"].values())\n', 'pipeline_smoke_test.py': '"""Real public-API queue/cancel/two-content acceptance. No model or fake video.\n\nStart while the model is loading: accept eight jobs, cancel six queued jobs,\nthen fetch the two distinct survivors. --resume only queries existing identities.\nStage overlap and GPU-release proofs are checked separately from the cloud audit.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport uuid\nfrom urllib.parse import urlsplit\n\n\ndef require(ok, message):\n    if not ok:\n        raise RuntimeError(message)\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef checkpoint(folder, report):\n    temporary = folder / \'pipeline_report.json.tmp\'\n    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temporary.replace(folder / \'pipeline_report.json\')\n\n\ndef download_ranges(base, key, job, folder):\n    import requests\n    artifact = job[\'artifact\']\n    size = artifact[\'bytes\']\n    require(isinstance(size, int) and 0 < size <= 128 * 1024**2, \'Unexpected output size\')\n    require(artifact.get(\'media\', {}).get(\'full_decode_passed\') is True, \'Server did not validate the complete video\')\n    url = base + \'/v1/jobs/\' + job[\'id\'] + \'/file\'\n    def part(bounds):\n        start, end = bounds\n        records = []\n        for number in range(1, 4):\n            record = {\'attempt\': number, \'start\': start, \'end\': end}\n            records.append(record)\n            try:\n                with requests.get(url, headers={\'Authorization\': \'Bearer \'+key,\n                        \'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\'},\n                        timeout=(15, 25), allow_redirects=False, stream=True) as response:\n                    record.update(http_status=response.status_code,\n                                  content_range=response.headers.get(\'Content-Range\'))\n                    require(response.status_code == 206, \'Range request was not accepted\')\n                    require(record[\'content_range\'] == f\'bytes {start}-{end}/{size}\', \'Incorrect Content-Range\')\n                    data = response.raw.read(end-start+2)\n                    record[\'bytes\'] = len(data)\n                    require(len(data) == end-start+1, \'Truncated range\')\n                record[\'passed\'] = True\n                return data, records\n            except Exception as error:\n                record.update(passed=False, error_type=type(error).__name__)\n        return None, records\n    bounds = [(start, min(size-1, start+65535)) for start in range(0, size, 65536)]\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        parts = list(pool.map(part, bounds))\n    result = {\'job_id\': job[\'id\'], \'ranges\': [records for _, records in parts], \'passed\': False}\n    (folder / (job[\'id\']+\'-download.json\')).write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    require(all(data is not None for data,_ in parts), \'File transfer exhausted bounded range retries; resume without generating again\')\n    content = b\'\'.join(data for data,_ in parts)\n    actual = hashlib.sha256(content).hexdigest()\n    require(len(content)==size and actual==artifact[\'sha256\'], \'External file differs from committed artifact\')\n    path = folder / (job[\'id\']+\'.mp4\')\n    path.write_bytes(content)\n    result.update(passed=True, bytes=size, sha256=actual, path=str(path),\n                  full_decode_basis=\'Server full audiovisual decode, with exact external byte count and SHA256 match\')\n    (folder / (job[\'id\']+\'-download.json\')).write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    return result\n\n\ndef get_json(base, key, route, *, authenticated=True, records=None):\n    """Bounded read retries never submit or replace an existing generation."""\n    import requests\n    for attempt in range(1, 4):\n        record = {\'route\': route, \'attempt\': attempt, \'observed_at\': now()}\n        if records is not None:\n            records.append(record)\n        try:\n            response = requests.get(base+route,\n                headers={\'Authorization\': \'Bearer \'+key} if authenticated else {},\n                timeout=(10,20), allow_redirects=False)\n            record[\'http_status\'] = response.status_code\n            if response.status_code == 200:\n                return response.json()\n            require(response.status_code in (429,502,503,504),\n                    \'GET \'+route+\' returned HTTP \'+str(response.status_code))\n        except requests.RequestException as error:\n            record[\'error_type\'] = type(error).__name__\n        if attempt < 3:\n            time.sleep(attempt)\n    raise RuntimeError(\'GET \'+route+\' exhausted bounded read retries; resume existing task identities\')\n\n\ndef _run(args):\n    import requests\n    key = os.environ.get(\'H3_API_KEY\', \'\')\n    require(len(key)>=32, \'Set H3_API_KEY without placing it in command-line arguments\')\n    base = args.url.rstrip(\'/\')\n    parsed = urlsplit(base)\n    require(parsed.scheme in {\'http\',\'https\'} and parsed.hostname and not any(\n        (parsed.username, parsed.password, parsed.query, parsed.fragment)), \'Invalid API base URL\')\n    args.outputdir.mkdir(parents=True, exist_ok=True)\n    target = args.outputdir/\'pipeline_report.json\'\n    headers = {\'Authorization\':\'Bearer \'+key}\n    reads = []\n    def get(route, authenticated=True):\n        return get_json(base,key,route,authenticated=authenticated,records=reads)\n    if args.resume:\n        report = json.loads(target.read_text(encoding=\'utf-8\'))\n        require(report[\'api_base_url\']==base and len(report.get(\'jobs\',[]))==8,\n                \'Resume needs the original API URL and all eight recorded job identities\')\n        require(all(job.get(\'id\') for job in report[\'jobs\']), \'Resolve uncertain submission identities before resume\')\n        reads = report.setdefault(\'read_attempts\', [])\n        report[\'resumed_at\'] = now()\n        args._active_report = True\n    else:\n        require(not target.exists(), \'Existing test report: use --resume; never resubmit an existing run\')\n        require(args.reference_a and args.reference_b and args.prompt_a and args.prompt_b,\n                \'Two fresh reference images and distinct explicit prompts are required\')\n        refs = [args.reference_a.resolve(),args.reference_b.resolve()]\n        require(all(p.is_file() for p in refs), \'Reference files do not exist\')\n        hashes = [digest(p) for p in refs]\n        require(hashes[0]!=hashes[1] and args.prompt_a.strip()!=args.prompt_b.strip(),\n                \'The two survivors must use different references and prompts\')\n        report = {\'schema_version\':1, \'run_id\':uuid.uuid4().hex, \'started_at\':now(), \'api_base_url\':base,\n                  \'passed\':False, \'jobs\':[], \'downloads\':[], \'observations\':[], \'read_attempts\':reads,\n                  \'scope\':\'Eight durable queued jobs; six queued cancellations; two new-content real generations. Stage overlap and shutdown are independently audited.\'}\n        checkpoint(args.outputdir,report)\n        args._active_report = True\n        deadline=time.monotonic()+args.startup_timeout\n        while True:\n            try:\n                system=get(\'/v1/system\')\n                if system.get(\'api_ready\') and not system.get(\'observation_stale\'):\n                    break\n            except Exception:\n                pass\n            require(time.monotonic()<deadline, \'API did not become queryable\')\n            time.sleep(3)\n        require(system.get(\'model_ready\') is False, \'Run queue admission/cancel test during cold loading, before any GPU job can start\')\n        report[\'instance_id\']=system[\'instance_id\']\n        report[\'system_before_submission\']=system\n        from smoke_test import register_content\n        registry=args.content_history or args.outputdir.parent/\'content-history.json\'\n        report[\'content_reservations\']=[]\n        for index,prompt in enumerate((args.prompt_a,args.prompt_b)):\n            report[\'content_reservations\'].append(register_content(registry,hashes[index],prompt,\n                outputdir=args.outputdir,api_base_url=base))\n            checkpoint(args.outputdir,report)\n        unauth=requests.get(base+\'/v1/system\',timeout=15,allow_redirects=False)\n        report[\'unauthorized_system_http_status\']=unauth.status_code\n        require(unauth.status_code==401,\'Unauthorized query was not rejected\')\n        for index in range(8):\n            survivor=index in (0,2)\n            content_index=1 if index==2 else 0\n            prompt=(args.prompt_a,args.prompt_b)[content_index]\n            if not survivor:\n                prompt+=f\' Alternate cancelled queue test {index}: a new viewpoint with slow lateral camera motion.\'\n            entry={\'index\':index,\'survivor\':survivor,\'reference\':str(refs[content_index]),\n                   \'reference_sha256\':hashes[content_index], \'prompt\':prompt,\n                   \'idempotency_key\':report[\'run_id\']+\'-\'+str(index), \'submission_attempts\':[]}\n            report[\'jobs\'].append(entry)\n            checkpoint(args.outputdir,report)\n            for attempt in range(1,4):\n                try:\n                    with refs[content_index].open(\'rb\') as media:\n                        response=requests.post(base+\'/v1/jobs\',headers={**headers,\'Idempotency-Key\':entry[\'idempotency_key\']},\n                            data={\'prompt\':prompt,\'kinds\':\'["image"]\',\'seconds\':5,\'width\':512,\'height\':288,\'steps\':8,\'seed\':42+index},\n                            files=[(\'files\',(refs[content_index].name,media,\'image/png\'))],\n                            timeout=(15,90),allow_redirects=False)\n                    entry[\'submission_attempts\'].append({\'attempt\':attempt,\'http_status\':response.status_code})\n                    if response.status_code==202:\n                        value=response.json()\n                        entry.update(id=value[\'id\'],submitted_status=value[\'status\'],receipt=value)\n                        break\n                    require(response.status_code in (429,502,503,504), \'Submission definitively rejected: HTTP \'+str(response.status_code))\n                except requests.RequestException as error:\n                    entry[\'submission_attempts\'].append({\'attempt\':attempt,\'error_type\':type(error).__name__})\n                finally:\n                    checkpoint(args.outputdir,report)\n                time.sleep(1)\n            require(entry.get(\'id\'), \'Submission uncertain: keep this report and idempotency key; do not start a new run\')\n            require(entry[\'submitted_status\']==\'queued\',\'Model started before queue cancellation acceptance window\')\n            print(\'accepted\',index,entry[\'id\'],flush=True)\n        report[\'all_eight_accepted_before_cancellation\']=True\n        checkpoint(args.outputdir,report)\n    try:\n        require(get(\'/v1/system\')[\'instance_id\']==report[\'instance_id\'],\'Manager instance changed; do not replay generation\')\n        # Repeating cancellation for the recorded identities is idempotent, including on resume.\n        for entry in report[\'jobs\']:\n            if not entry[\'survivor\']:\n                before=get(\'/v1/jobs/\'+entry[\'id\'])\n                require(before[\'status\'] in {\'queued\',\'cancelled\'},\'Cancellation target already began executing; preserve failure evidence\')\n                response=requests.post(base+\'/v1/jobs/\'+entry[\'id\']+\'/cancel\',headers=headers,timeout=30,allow_redirects=False)\n                require(response.status_code==202,\'Cancellation request was not accepted\')\n                after=get(\'/v1/jobs/\'+entry[\'id\'])\n                entry[\'cancel_receipt\']=response.json()\n                entry[\'terminal\']=after\n                require(after[\'status\']==\'cancelled\' and after.get(\'started_at\') is None,\n                        \'Queued cancellation did not complete before dispatch\')\n                checkpoint(args.outputdir,report)\n        deadline=time.monotonic()+args.generation_timeout\n        previous=None\n        while True:\n            survivors=[entry for entry in report[\'jobs\'] if entry[\'survivor\']]\n            for entry in survivors:\n                entry[\'terminal\']=get(\'/v1/jobs/\'+entry[\'id\'])\n            summary=[{\'id\':entry[\'id\'],\'status\':entry[\'terminal\'][\'status\'],\n                      \'stage\':entry[\'terminal\'].get(\'active_phase\')} for entry in survivors]\n            signature=json.dumps(summary,sort_keys=True)\n            if signature!=previous:\n                observation={\'observed_at\':now(),\'jobs\':summary}\n                report[\'observations\'].append(observation)\n                print(json.dumps(observation),flush=True)\n                previous=signature\n            checkpoint(args.outputdir,report)\n            require(all(entry[\'terminal\'][\'status\'] not in {\'failed\',\'cancelled\'} for entry in survivors),\n                    \'A generation did not succeed; inspect the retained report and model log\')\n            if all(entry[\'terminal\'][\'status\']==\'succeeded\' for entry in survivors):\n                break\n            require(time.monotonic()<deadline,\'Generation deadline exceeded; query or cancel existing IDs without resubmitting\')\n            time.sleep(args.poll_seconds)\n        for entry in survivors:\n            report[\'downloads\'].append(download_ranges(base,key,entry[\'terminal\'],args.outputdir))\n            checkpoint(args.outputdir,report)\n        report.update(passed=True,finished_at=now())\n        checkpoint(args.outputdir,report)\n        print(\'QUEUE_CANCEL_TWO_GENERATIONS_PASSED\',flush=True)\n        return report\n    except Exception as error:\n        report.update(passed=False,error_type=type(error).__name__,error=str(error),last_updated_at=now())\n        checkpoint(args.outputdir,report)\n        raise\n\n\ndef run(args):\n    """Checkpoint failures during admission as well as during generation polling."""\n    args._active_report = False\n    try:\n        return _run(args)\n    except Exception as error:\n        target = args.outputdir/\'pipeline_report.json\'\n        if args._active_report and target.is_file():\n            try:\n                report = json.loads(target.read_text(encoding=\'utf-8\'))\n                message = str(error)\n                key = os.environ.get(\'H3_API_KEY\')\n                if key:\n                    message = message.replace(key, \'[REDACTED]\')\n                report.update(passed=False,error_type=type(error).__name__,error=message,last_updated_at=now())\n                checkpoint(args.outputdir,report)\n            except (OSError, ValueError, TypeError):\n                pass  # Preserve the original error and any earlier durable identities.\n        raise\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--url\',required=True)\n    parser.add_argument(\'--outputdir\',type=Path,required=True)\n    parser.add_argument(\'--reference-a\',type=Path)\n    parser.add_argument(\'--reference-b\',type=Path)\n    parser.add_argument(\'--prompt-a\')\n    parser.add_argument(\'--prompt-b\')\n    parser.add_argument(\'--resume\',action=\'store_true\')\n    parser.add_argument(\'--content-history\',type=Path)\n    parser.add_argument(\'--startup-timeout\',type=float,default=300)\n    parser.add_argument(\'--generation-timeout\',type=float,default=5400)\n    parser.add_argument(\'--poll-seconds\',type=float,default=10)\n    run(parser.parse_args())\n\n\nif __name__==\'__main__\':\n    main()\n', 'notebook_console.py': '"""Owned notebook UI bridge. It observes the manager; it is not a GPU scheduler."""\nfrom __future__ import annotations\n\nimport ast\nimport base64\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nimport os\nimport secrets\nfrom pathlib import Path\nimport threading\nimport sys\nimport time\nimport traceback\nimport uuid\n\nimport notebook_ui as ui\n\n\nACTION_TITLES = {\'bootstrap\': \'一键启动\', \'preflight\': \'资源检查\', \'install\': \'安装依赖\', \'start\': \'启动服务\',\n                 \'generate\': \'生成验收\', \'stop\': \'停止服务\', \'release\': \'释放实例\', \'control\': \'组件控制\'}\nSTARTUP_LABELS = {\n    \'preflight\': \'检查资源\',\n    \'install\': \'安装依赖\',\n    \'start\': \'启动独立服务\',\n    \'awaiting_api\': \'等待 API 及所选公网可达\',\n    \'awaiting_model\': \'API 已可接收排队任务；等待模型完整加载\',\n    \'ready\': \'模型与 API 已就绪；排队任务由总管理自动调度\',\n    \'failed\': \'启动未完成，查看实际错误；已启动的后台仍可查询和停止\',\n    \'cancelled\': \'启动已取消\',\n}\nLOG_NAMES = {\'model\': \'server_log\', \'api\': \'api_log\', \'manager\': \'manager_log\', \'tunnel\': \'tunnel_log\'}\nBOOLS = {\'staged\', \'cached\', \'public\', \'force\'}\nSTRINGS = {\'reference_file\', \'prompt\', \'resume_job_id\', \'endpoint\', \'acceleration\', \'style\',\n           \'attention\', \'offload\', \'action\', \'command_id\', \'request_id\', \'source\', \'credential_id\'}\nCHOICES = {\'endpoint\': {\'public\', \'local\'}, \'acceleration\': {\'none\', \'lightx2v-ref2va-4step\'},\n           \'style\': {\'none\', \'fal-realism-people\'}, \'attention\': {\'auto\', \'native\', \'_native_flash\'},\n           \'offload\': {\'stream\', \'sync\'}}\nSAFE_STATE = {\'instance_id\', \'phase\', \'ready\', \'api_ready\', \'model_ready\', \'public_requested\',\n              \'public_api_ready\', \'public_ready\', \'tunnel_connected\', \'server_alive\', \'model_alive\',\n              \'manager_alive\', \'tunnel_alive\', \'cleanup_complete\', \'observed_at\', \'observation_stale\',\n              \'elapsed_seconds\', \'draining\', \'queue\'}\n\n\nclass OperationCancelled(RuntimeError):\n    pass\n\n\ndef validated(payload):\n    if not isinstance(payload, dict) or len(json.dumps(payload)) > 65536:\n        raise ValueError(\'控制参数必须是有限的对象。\')\n    result = {}\n    for key, value in payload.items():\n        if key in BOOLS:\n            if type(value) is not bool:\n                raise ValueError(f\'{key} 必须为布尔值。\')\n        elif key in STRINGS:\n            if not isinstance(value, str) or len(value) > (6000 if key == \'prompt\' else 2048):\n                raise ValueError(f\'{key} 不是有效的短文本。\')\n        elif key == \'style_scale\':\n            if type(value) not in (int, float) or not .1 <= value <= 1:\n                raise ValueError(\'风格强度应在 0.1 与 1 之间。\')\n        elif key in {\'cursor\'}:\n            continue\n        else:\n            raise ValueError(f\'未知控制字段：{key}\')\n        if key in CHOICES and value not in CHOICES[key]:\n            raise ValueError(f\'{key} 选项无效。\')\n        result[key] = value\n    return result\n\n\ndef action_code(source, parameters):\n    """Replace only known top-level notebook defaults, never execute browser-supplied code."""\n    tree = ast.parse(source)\n    found = set()\n    for node in tree.body:\n        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):\n            name = node.targets[0].id\n            if name in parameters:\n                node.value = ast.Constant(parameters[name])\n                found.add(name)\n    if found != set(parameters):\n        raise ValueError(\'操作参数与冻结的操作代码不匹配。\')\n\n    class NoAutomaticDownloads(ast.NodeTransformer):\n        def visit_Expr(self, node):\n            call = node.value\n            if (isinstance(call, ast.Call) and isinstance(call.func, ast.Attribute)\n                    and isinstance(call.func.value, ast.Name) and call.func.value.id == \'files\'\n                    and call.func.attr == \'download\'):\n                return ast.copy_location(ast.Pass(), node)\n            return self.generic_visit(node)\n    tree = NoAutomaticDownloads().visit(tree)\n    return compile(ast.fix_missing_locations(tree), \'owned-notebook-operation\', \'exec\')\n\n\nclass ConsoleController:\n    def __init__(self, directory, programs, scope=None, release_callback=None):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.programs = dict(programs)\n        self.scope = scope if scope is not None else {}\n        self.scope[\'PROJECT_DIR\'] = self.directory\n        self.scope[\'dashboard\'] = self\n        self.scope[\'print\'] = self._print\n        self.scope.update(StatusCard=ui.StatusCard, elapsed_text=ui.elapsed_text, log_tail=ui.log_tail,\n                          run_logged=ui.run_logged, json_probe=ui.json_probe, os=os, sys=sys, json=json, Path=Path)\n        self.lock = threading.RLock()\n        self.executor = ThreadPoolExecutor(max_workers=3, thread_name_prefix=\'h3-console\')\n        self.operations = {}\n        self.active = {}\n        self.finished = {}\n        self.cancel_startup = threading.Event()\n        self.startup_stage = None\n        self.startup_delivery = None\n        self._bootstrap_hosts = {}\n        self.requested_options = {}\n        self.processes = {}\n        self.process_owners = {}\n        self.closed = False\n        self.epoch = None\n        self.activity = None\n        self._last_activity = None\n        self.notice = None\n        self.release_callback = release_callback\n        self.audit_path = self.directory / \'console-events.jsonl\'\n        self.log_path = self.directory / \'console.log\'\n        ui._dashboard = self\n        ui._process_observer = self\n\n    def _secrets(self):\n        host = self.scope.get(\'startup_host\')\n        return [getattr(host, \'api_key\', \'\'), getattr(host, \'_control_key\', \'\'), self.scope.get(\'API_KEY\', \'\') or \'\',\n                self.scope.get(\'BOOTSTRAP_API_KEY\', \'\') or \'\']\n\n    def _clean(self, value):\n        return ui.redact_text(str(value), secrets=self._secrets())\n\n    def _sanitize(self, value):\n        if isinstance(value, str):\n            return self._clean(value)\n        if isinstance(value, dict):\n            return {key: self._sanitize(item) for key, item in value.items()}\n        if isinstance(value, (list, tuple)):\n            return [self._sanitize(item) for item in value]\n        return value\n\n    def _event(self, kind, **values):\n        record = {\'type\': kind, \'observed_at\': time.time(), **values}\n        with self.lock:\n            with self.audit_path.open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(self._clean(json.dumps(record, ensure_ascii=False)) + \'\\n\')\n\n    def _print(self, *values, sep=\' \', end=\'\\n\', **kwargs):\n        text = self._clean(sep.join(str(v) for v in values))\n        stamp = time.strftime(\'%m-%d %H:%M:%S\', time.gmtime(time.time() + 8 * 3600))\n        with self.lock:\n            with self.log_path.open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(\'[\' + stamp + \' 北京时间] \' + text + end)\n\n    def refresh(self):\n        """StatusCard sink: update data and log, never create a notebook output."""\n        item = self.activity or {}\n        marker = json.dumps(item, sort_keys=True, ensure_ascii=False)\n        if marker != self._last_activity:\n            self._last_activity = marker\n            self._print(item.get(\'title\', \'\'), item.get(\'badge\', \'\'), item.get(\'note\', \'\'))\n\n    def process_started(self, process):\n        owner = None\n        if type(getattr(process, \'pid\', None)) is int:\n            from process_owner import ProcessOwner\n            owner = ProcessOwner(process)\n        with self.lock:\n            self.processes[threading.get_ident()] = process\n            if owner is not None:\n                self.process_owners[id(process)] = (owner, threading.get_ident())\n            cancelled = any(record.get(\'thread_id\') == threading.get_ident()\n                            and (record.get(\'cancel_monitor\') or record.get(\'cancel_requested\'))\n                            for record in self.active.values())\n        if cancelled:\n            self._terminate_client_process(process)\n\n    def process_progress(self, process):\n        with self.lock:\n            entry = self.process_owners.get(id(process))\n        if entry:\n            entry[0].observe()  # Capture descendants before wrappers exit.\n\n    def process_finished(self, process):\n        with self.lock:\n            entry = self.process_owners.get(id(process))\n        if entry:\n            state = entry[0].observe()\n            if not state[\'cleanup_complete\']:\n                state = entry[0].terminate_all(timeout=10)\n            self._event(\'client.process_cleanup\', **state)\n            if not state[\'cleanup_complete\']:\n                raise RuntimeError(\'安装或客户端的子进程树尚未确认退出；请停止后检查日志。\')\n        with self.lock:\n            self.process_owners.pop(id(process), None)\n            if self.processes.get(threading.get_ident()) is process:\n                self.processes.pop(threading.get_ident(), None)\n\n    def _terminate_client_process(self, process):\n        with self.lock:\n            entry = self.process_owners.get(id(process))\n        if entry:\n            state = entry[0].terminate_all(timeout=10)\n            self._event(\'client.process_cleanup\', **state)\n            if not state[\'cleanup_complete\']:\n                raise RuntimeError(\'子进程树仍未退出；未声明停止或释放完成。\')\n        elif process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=10)\n            except Exception:\n                process.kill()\n                process.wait(timeout=5)\n\n    def _stop_client_monitor(self):\n        with self.lock:\n            active = self.active.get(\'generate\')\n            if active:\n                active[\'cancel_monitor\'] = True\n            process = self.processes.get(active.get(\'thread_id\')) if active else None\n        if process is not None and process.poll() is None:\n            self._terminate_client_process(process)\n            self._event(\'client.monitor_stopped\', request_id=active[\'request_id\'])\n\n    def _execute(self, name, parameters=None):\n        exec(action_code(self.programs[name], parameters or {}), self.scope)\n\n    def _configuration(self, payload):\n        self.requested_options = {name: payload.get(name, \'none\') for name in (\'acceleration\', \'style\')}\n        self._execute(\'advanced\', {\'ACCELERATION_LORA\': payload.get(\'acceleration\', \'none\'),\n            \'STYLE_LORA\': payload.get(\'style\', \'none\'), \'STYLE_LORA_SCALE\': payload.get(\'style_scale\', 1.0),\n            \'ATTENTION_BACKEND\': payload.get(\'attention\', \'auto\'), \'DENOISER_OFFLOAD\': payload.get(\'offload\', \'stream\')})\n\n    def _check_startup_cancelled(self):\n        if self.cancel_startup.is_set():\n            raise OperationCancelled(\'启动已取消；不再进入后续阶段。\')\n\n    def _startup_stage(self, stage):\n        if self.startup_stage != stage:\n            self.startup_stage = stage\n            self._event(\'startup.stage\', stage=stage)\n            self._print(STARTUP_LABELS.get(stage, stage))\n\n    def _await_bootstrap(self, host, public):\n        deadline = time.monotonic() + float(getattr(host, \'startup_timeout\', 3600))\n        interval = max(.01, min(2, float(getattr(host, \'poll_interval\', 1))))\n        while True:\n            self._check_startup_cancelled()\n            state = host.snapshot()\n            fresh = not state.get(\'observation_stale\', False)\n            if state.get(\'phase\') in {\'failed\', \'stopped\', \'stopping\'} or (fresh and state.get(\'error\')):\n                raise RuntimeError(state.get(\'error\') or \'服务准备未完成；API 可用时仍可查询与取消任务。\')\n            api = fresh and state.get(\'api_ready\') is True and (not public or state.get(\'public_api_ready\') is True)\n            if api and state.get(\'model_ready\') is True and state.get(\'ready\') is True:\n                self._startup_stage(\'ready\')\n                return\n            self._startup_stage(\'awaiting_model\' if api else \'awaiting_api\')\n            if time.monotonic() >= deadline:\n                raise TimeoutError(\'等待服务就绪超时；请查看日志。已有任务和后台保留，可查询、取消或停止。\')\n            self.cancel_startup.wait(interval)\n\n    def _bootstrap(self, payload):\n        payload = dict(payload, public=payload.get(\'public\', True))\n        self._check_startup_cancelled()\n        host = self.scope.get(\'startup_host\')\n        state = host.snapshot() if host else {}\n        public = payload.get(\'public\', True)\n        if host and any(state.get(name) for name in (\'manager_alive\', \'server_alive\', \'model_alive\', \'tunnel_alive\')):\n            # A repeated start observes the owned instance; it cannot reinstall or rotate its token.\n            if bool(getattr(host, \'public\', False)) != public or bool(getattr(host, \'staged\', False)) != payload.get(\'staged\', False):\n                raise RuntimeError(\'现有实例配置不同；请先停止，再以新配置启动。\')\n            self._bind_bootstrap_host(payload[\'request_id\'], host)\n            return self._await_bootstrap(host, public)\n        clean_terminal = (state.get(\'phase\') in {\'failed\', \'stopped\'} and state.get(\'cleanup_complete\') is True\n                          and all(state.get(name) is False for name in (\'manager_alive\', \'server_alive\', \'model_alive\', \'tunnel_alive\'))\n                          and not state.get(\'process_observation_errors\'))\n        if host and not clean_terminal and (state.get(\'observation_stale\') or state.get(\'phase\') != \'new\'):\n            raise RuntimeError(\'旧生命周期尚未确认清理完成；先停止并检查状态。\')\n        self.scope.pop(\'startup_host\', None)\n        self.scope.update(PREFLIGHT_OK=False, INSTALL_OK=False, RESOURCE_REPORT={})\n        self.requested_options = {name: payload.get(name, \'none\') for name in (\'acceleration\', \'style\')}\n        self.scope[\'BOOTSTRAP_API_KEY\'] = secrets.token_urlsafe(32)\n        self.scope[\'API_KEY\'] = self.scope[\'BOOTSTRAP_API_KEY\']\n        self.scope[\'BOOTSTRAP_PUBLIC\'] = public\n        self._event(\'credential.generated\', purpose=\'public_api\')\n        for stage in (\'preflight\', \'install\', \'start\'):\n            self._check_startup_cancelled()\n            self._startup_stage(stage)\n            self._operate(stage, payload)\n            self._check_startup_cancelled()\n            if stage == \'preflight\' and self.scope.get(\'PREFLIGHT_OK\') is not True:\n                raise RuntimeError(\'资源检查没有提供通过凭据，停止启动。\')\n            if stage == \'install\' and self.scope.get(\'INSTALL_OK\') is not True:\n                raise RuntimeError(\'依赖检查没有提供通过凭据，停止启动。\')\n        host = self.scope.get(\'startup_host\')\n        if not host:\n            raise RuntimeError(\'总管理未创建，停止启动。\')\n        self._bind_bootstrap_host(payload[\'request_id\'], host)\n        return self._await_bootstrap(host, public)\n\n    def _bind_bootstrap_host(self, request_id, host):\n        """Bind observations to the owner accepted by this bootstrap request."""\n        instance_id = getattr(host, \'instance_id\', None)\n        with self.lock:\n            operation = self.operations.get(request_id)\n            if not instance_id or not operation or operation.get(\'action\') != \'bootstrap\':\n                raise RuntimeError(\'启动请求与实例绑定无效。\')\n            previous = self._bootstrap_hosts.get(request_id)\n            if previous and (previous[0] is not host or previous[1] != instance_id):\n                raise RuntimeError(\'启动请求不能更换已绑定实例。\')\n            self._bootstrap_hosts[request_id] = (host, instance_id)\n            operation[\'instance_id\'] = instance_id\n        self._event(\'startup.host_bound\', request_id=request_id, instance_id=instance_id)\n\n    def _cancel_bootstrap_and_wait(self):\n        with self.lock:\n            record = self.active.get(\'setup\')\n            if not record or record[\'action\'] != \'bootstrap\':\n                return\n            record[\'cancel_requested\'] = True\n            self.cancel_startup.set()\n            process = self.processes.get(record.get(\'thread_id\'))\n            finished = self.finished[record[\'request_id\']]\n        self._event(\'startup.cancel_requested\', request_id=record[\'request_id\'])\n        if process is not None:\n            self._terminate_client_process(process)\n        # Join before stopping the owner: a late start must not resurrect a released service.\n        if not finished.wait(60):\n            raise TimeoutError(\'启动步骤尚未退出；未声明停止或释放完成，请检查日志后重试。\')\n\n    def begin_startup_delivery(self, request_id, timeout):\n        """Observe delivery on the calling cell; never schedule dependent work."""\n        if isinstance(timeout, bool) or not isinstance(timeout, (int, float)) or not math.isfinite(timeout) or not 0 < timeout <= 600:\n            raise ValueError(\'启动交付观察时间须为 0 到 600 秒之间的有限数值。\')\n        with self.lock:\n            self.startup_delivery = {\'request_id\': request_id, \'status\': \'observing\', \'active\': True,\n                \'started_at\': time.time(), \'deadline_at\': time.time() + timeout, \'timeout_seconds\': timeout,\n                \'instance_id\': None, \'last_observed_at\': None, \'api_ready\': False, \'model_ready\': False,\n                \'cleanup_complete\': None}\n        self._event(\'startup.delivery_started\', request_id=request_id, timeout_seconds=timeout,\n                    dependency=\'current_instance_api_and_selected_public_api; model_ready_not_required\')\n        self._print(\'正在交付独立 API；最多观察\', timeout,\n                    \'秒，不等待模型。观察期间请用 Colab 本单元的停止按钮取消，程序会回收已启动进程。\')\n\n    def wait_startup_delivery(self, request_id, *, publish=None):\n        """A bounded foreground handoff, independent from model preparation.\n\n        Expiry only ends observation and retains the worker/owner. Interrupting\n        the cell explicitly cancels startup, joins it, then reclaims its owned\n        processes through the same audited stop path as the control panel.\n        """\n        delivery = self.startup_delivery\n        if not delivery or delivery[\'request_id\'] != request_id or not delivery[\'active\']:\n            raise RuntimeError(\'启动交付观察身份不匹配。\')\n        deadline = time.monotonic() + delivery[\'timeout_seconds\']\n        status, error, last_publish = \'observation_failed\', None, -float(\'inf\')\n        def update():\n            nonlocal last_publish\n            if publish:\n                try:\n                    publish()\n                except Exception as caught:\n                    self._print(\'启动快照显示未更新：\', type(caught).__name__)\n            last_publish = time.monotonic()\n        try:\n            while True:\n                with self.lock:\n                    record = dict(self.operations.get(request_id) or {})\n                    binding = self._bootstrap_hosts.get(request_id)\n                if self.cancel_startup.is_set() or record.get(\'status\') == \'cancelled\':\n                    status = \'cancelled\'\n                    break\n                host, expected = binding if binding else (None, None)\n                # An upgraded controller can still hold a clean, stopped host\n                # while its new worker validates and replaces it. That previous\n                # owner\'s terminal state cannot finish this request\'s handoff.\n                if (host is not None and self.scope.get(\'startup_host\') is host\n                        and getattr(host, \'instance_id\', None) == expected):\n                    state = host.snapshot()\n                    same = bool(expected and state.get(\'instance_id\') == expected)\n                    fresh = same and state.get(\'observation_stale\') is False\n                    selected_public = bool(getattr(host, \'public\', False))\n                    with self.lock:\n                        delivery.update(instance_id=expected, last_observed_at=state.get(\'observed_at\'),\n                                        api_ready=bool(fresh and state.get(\'api_ready\') is True),\n                                        model_ready=bool(fresh and state.get(\'model_ready\') is True))\n                    if fresh and state.get(\'api_ready\') is True and (not selected_public or state.get(\'public_api_ready\') is True):\n                        status = \'api_ready\'\n                        break\n                    if same and state.get(\'phase\') in {\'failed\', \'stopped\', \'stopping\'}:\n                        status, error = \'failed\', state.get(\'error\') or \'启动生命周期已结束\'\n                        break\n                if record.get(\'status\') == \'failed\':\n                    status, error = \'failed\', record.get(\'error\') or \'启动未完成\'\n                    break\n                if time.monotonic() >= deadline:\n                    status = \'observation_expired\'\n                    break\n                if time.monotonic() - last_publish >= 2:\n                    update()\n                self.cancel_startup.wait(min(.2, max(0, deadline - time.monotonic())))\n        except KeyboardInterrupt:\n            status, error = self._interrupt_delivery(delivery, publish=update)\n            raise\n        except Exception as caught:\n            error = str(caught)\n            self._print(\'启动交付观察未完成；后台所有权保留：\', caught)\n        finally:\n            with self.lock:\n                delivery.update(status=status, active=False, finished_at=time.time())\n                if error:\n                    delivery[\'error\'] = self._clean(error)[:500]\n            self._event(\'startup.delivery_finished\', **dict(delivery))\n            messages = {\'api_ready\': \'API 交付已确认；模型可继续下载，任务按实际就绪条件处理。\',\n                        \'observation_expired\': \'前台观察到期；后台仍受管，面板可继续查询、取消启动或停止。\',\n                        \'cancelled\': \'启动交付观察已取消；进程清理以停止回执为准。\',\n                        \'failed\': \'启动未完成；控制器及已有后台保留，查看日志后处理。\',\n                        \'interrupted\': \'启动已中断，所拥有进程清理已确认。\',\n                        \'interrupted_cleanup_unconfirmed\': \'启动已中断，但清理未确认；继续保留所有权。\'}\n            self._print(messages.get(status, \'前台观察已结束；继续通过面板查询实际状态。\'))\n            update()\n        return dict(delivery)\n\n    def _interrupt_delivery(self, delivery, *, publish=None):\n        self.cancel_startup.set()\n        with self.lock:\n            delivery.update(status=\'cancelling\', deadline_at=time.time() + 90)\n        self._event(\'startup.delivery_interrupted\', request_id=delivery[\'request_id\'])\n        self._print(\'收到停止本单元请求，正在取消启动并回收已拥有的进程。\')\n        try:\n            if publish:\n                publish()\n        except BaseException:\n            pass  # Another interrupt/display failure must not skip cleanup.\n        try:\n            self._operate(\'stop\', {})\n            with self.lock:\n                delivery[\'cleanup_complete\'] = True\n            return \'interrupted\', None\n        except BaseException as caught:\n            with self.lock:\n                delivery[\'cleanup_complete\'] = False\n            self._print(\'启动中断后的清理尚未确认；保留控制器，请继续查询或停止：\', caught)\n            return \'interrupted_cleanup_unconfirmed\', str(caught)\n\n    def _operate(self, action, payload):\n        if action == \'bootstrap\':\n            return self._bootstrap(payload)\n        if action == \'preflight\':\n            return self._execute(\'preflight\', {\'STAGED_BF16\': payload.get(\'staged\', False),\n                                             \'MODEL_ALREADY_CACHED\': payload.get(\'cached\', False)})\n        if action == \'install\':\n            self._configuration(payload)\n            return self._execute(\'install\')\n        if action == \'start\':\n            self._configuration(payload)\n            return self._execute(\'start\', {\'START_GPU_API\': True, \'OPEN_PUBLIC_API\': payload.get(\'public\', False)})\n        if action == \'generate\':\n            host = self.scope.get(\'startup_host\')\n            state = host.snapshot() if host else {}\n            self.scope[\'PUBLIC_API_URL\'] = state.get(\'public_url\') if state.get(\'public_api_ready\') else None\n            return self._execute(\'generate\', {\'RUN_SMOKE_TEST\': True,\n                \'验收接口\': \'本机API\' if payload.get(\'endpoint\') == \'local\' else \'临时公网\',\n                \'RESUME_JOB_ID\': payload.get(\'resume_job_id\', \'\').strip(),\n                \'TEST_REFERENCE_FILE\': payload.get(\'reference_file\', \'\'), \'TEST_PROMPT\': payload.get(\'prompt\', \'\')})\n        if action in {\'stop\', \'release\'}:\n            self._cancel_bootstrap_and_wait()\n            self._stop_client_monitor()\n            with self.lock:\n                remaining = list(self.process_owners.values())\n            for owner, _ in remaining:\n                state = owner.terminate_all(timeout=10)\n                self._event(\'client.process_cleanup\', **state)\n                if not state[\'cleanup_complete\']:\n                    raise RuntimeError(\'仍有客户端子进程，不能声明停止或释放完成。\')\n                with self.lock:\n                    self.process_owners.pop(id(owner.process), None)\n            self._execute(\'stop\', {\'STOP_SERVER\': True, \'RELEASE_COLAB_RUNTIME\': False})\n            if action == \'release\':\n                self._event(\'runtime.release_requested\')\n                if self.release_callback:\n                    self.release_callback()\n                else:\n                    from google.colab import runtime\n                    runtime.unassign()\n            return\n        if action == \'control\':\n            host = self.scope.get(\'startup_host\')\n            if not host:\n                raise RuntimeError(\'尚未启动总管理。\')\n            command = payload.get(\'action\')\n            if command == \'query\':\n                if not payload.get(\'command_id\', \'\').strip():\n                    raise ValueError(\'先填写命令 ID。\')\n                result = host.command(payload[\'command_id\'].strip())\n            elif command in {\'drain\', \'resume\', \'stop_model\', \'start_model\', \'stop_api\', \'start_api\'}:\n                result = host.control(command, force=payload.get(\'force\', False))\n            else:\n                raise ValueError(\'组件控制动作无效。\')\n            self._print(json.dumps(result, ensure_ascii=False))\n            return result\n        raise ValueError(\'未知操作。\')\n\n    def _worker(self, record, lane, action, payload):\n        with self.lock:\n            record.update(status=\'running\', started_at=time.time(), thread_id=threading.get_ident())\n        self._event(\'operation.started\', action=action, request_id=record[\'request_id\'])\n        try:\n            result = self._operate(action, payload)\n            if action == \'bootstrap\':\n                self._check_startup_cancelled()\n            with self.lock:\n                record.update(status=\'succeeded\', finished_at=time.time())\n                if isinstance(result, dict):\n                    record[\'command\'] = {k: result[k] for k in (\'command_id\', \'status\') if k in result}\n                message = (\'已发起平台释放；是否完成以 Colab 断开为准。\' if action == \'release\'\n                           else \'操作已完成。\' if action != \'control\' else \'命令回执已记录；接收不等于执行完成。\')\n                self.notice = None\n                self._print(ACTION_TITLES[action], message)\n        except BaseException as error:\n            cancelled = action == \'bootstrap\' and self.cancel_startup.is_set()\n            if action == \'bootstrap\':\n                self._startup_stage(\'cancelled\' if cancelled else \'failed\')\n            self._print(str(error) if cancelled else traceback.format_exc())\n            with self.lock:\n                record.update(status=\'cancelled\' if cancelled else \'failed\', finished_at=time.time(), error=self._clean(error)[:500])\n                if not cancelled:\n                    self.notice = {\'id\': record[\'request_id\'], \'kind\': \'error\', \'title\': ACTION_TITLES[action] + \'未完成\',\n                                   \'message\': self._clean(error)[:500]}\n        finally:\n            self._event(\'operation.finished\', action=action, request_id=record[\'request_id\'], status=record[\'status\'])\n            with self.lock:\n                self.active.pop(lane, None)\n                self.processes.pop(threading.get_ident(), None)\n                self.finished[record[\'request_id\']].set()\n\n    def submit(self, action, payload):\n        request_id = payload.get(\'request_id\')\n        if not request_id or len(request_id) > 128:\n            raise ValueError(\'操作必须携带唯一 request_id。\')\n        fingerprint = hashlib.sha256(json.dumps([action, payload], sort_keys=True).encode()).hexdigest()\n        lane = \'generate\' if action == \'generate\' else \'control\' if action in {\'stop\', \'release\', \'control\'} else \'setup\'\n        with self.lock:\n            if self.closed:\n                raise RuntimeError(\'控制台已关闭。\')\n            previous = self.operations.get(request_id)\n            if previous:\n                if previous[\'fingerprint\'] != fingerprint:\n                    raise ValueError(\'同一请求 ID 的内容发生变化。\')\n                return dict(previous)\n            setup = self.active.get(\'setup\')\n            setup_allows_control = setup and setup[\'action\'] == \'bootstrap\' and (\n                action in {\'stop\', \'release\'} or self.startup_stage in {\'awaiting_api\', \'awaiting_model\', \'ready\'})\n            if lane in self.active or (setup and not setup_allows_control) or (lane == \'setup\' and self.active):\n                raise RuntimeError(\'相关操作尚在执行；可继续观察日志，不重复提交。\')\n            if action == \'bootstrap\':\n                self.cancel_startup.clear()\n                self.notice = None\n            record = {\'request_id\': request_id, \'action\': action, \'status\': \'accepted\',\n                      \'accepted_at\': time.time(), \'fingerprint\': fingerprint}\n            self.operations[request_id] = record\n            self.active[lane] = record\n            self.finished[request_id] = threading.Event()\n            self._event(\'operation.accepted\', action=action, request_id=request_id, fingerprint=fingerprint)\n            self.executor.submit(self._worker, record, lane, action, payload)\n            return dict(record)\n\n    def _connection_context(self, state=None):\n        host = self.scope.get(\'startup_host\')\n        with self.lock:\n            preparing = (self.startup_stage in {\'preflight\', \'install\', \'start\'}\n                         and any(r[\'action\'] == \'bootstrap\' for r in self.active.values()))\n        if preparing:\n            host, state = None, {}\n        if state is None:\n            state = host.snapshot() if host else {}\n        if state.get(\'instance_id\') and state[\'instance_id\'] != getattr(host, \'instance_id\', None):\n            state = {}  # A read raced with replacement of the owned lifecycle.\n        key = getattr(host, \'api_key\', None) or self.scope.get(\'BOOTSTRAP_API_KEY\')\n        public = bool(state.get(\'public_requested\', self.scope.get(\'BOOTSTRAP_PUBLIC\', False)))\n        fresh = not state.get(\'observation_stale\', False)\n        api_ready = state.get(\'api_ready\') is True and fresh\n        public_ready = api_ready and state.get(\'public_api_ready\') is True\n        local = \'http://127.0.0.1:\' + str(getattr(host, \'port\', 8000)) if api_ready else None\n        candidate = state.get(\'public_url\') if public else None\n        url = candidate if public_ready else local if not public else None\n        connection = {\'url\': url, \'candidate_url\': candidate, \'local_url\': local,\n                      \'api_ready\': api_ready, \'public_requested\': public,\n                      \'public_api_ready\': public_ready,\n                      \'model_ready\': state.get(\'model_ready\') is True and fresh,\n                      \'observation_stale\': not fresh, \'key_available\': bool(key),\n                      \'credential_id\': hashlib.sha256(key.encode()).hexdigest()[:16] if key else None,\n                      \'instance_id\': getattr(host, \'instance_id\', None)}\n        return connection, key\n\n    def _downloads(self, state):\n        progress = state.get(\'downloads\')\n        if isinstance(progress, dict) and progress.get(\'schema_version\') == 1:\n            return progress\n        items = {}\n        for name, variable in ((\'model\', None), (\'acceleration\', \'ACCELERATION_LORA\'), (\'style\', \'STYLE_LORA\')):\n            selection = self.requested_options.get(name, self.scope.get(variable, \'none\')) if variable else None\n            phase = \'waiting\' if name == \'model\' or selection != \'none\' else \'disabled\'\n            items[name] = {\'phase\': phase, \'bytes_completed\': 0, \'bytes_total\': None, \'percent\': None,\n                           \'files_completed\': 0, \'files_total\': None, \'active_files\': 0,\n                           \'scope\': \'observed_transfers\' if name == \'model\' else \'asset\',\n                           \'detail\': \'等待模型容器提供真实进度\' if phase == \'waiting\' else \'未启用\'}\n        return {\'schema_version\': 1, \'model_instance_id\': None, \'observed_at\': None,\n                \'observation_stale\': False, \'model_ready\': False, \'lifecycle\': \'not_started\', \'items\': items}\n\n    def snapshot(self, source=\'diagnostics\', *, observe_host=True):\n        host = self.scope.get(\'startup_host\')\n        for key in self._secrets():\n            if key and key not in ui._known_secrets:\n                ui._known_secrets = (*ui._known_secrets[-4:], key)\n        try:\n            # Mounting the panel must not depend on an API/model RPC response.\n            # The browser requests a fresh observation after the cell returns.\n            state = (host.snapshot() if observe_host else {\'observation_stale\': True}) if host else {}\n        except Exception as error:\n            state = {\'observation_stale\': True, \'phase\': \'unavailable\'}\n            self._print(\'状态读取失败：\', error)\n        safe = {k: state[k] for k in SAFE_STATE if k in state}\n        if state.get(\'error\'):\n            safe[\'error\'] = self._clean(state[\'error\'])[:300]\n        if source == \'events\':\n            text = json.dumps(host.events(limit=35), ensure_ascii=False, indent=2) if host else \'尚无运行实例。\'\n        elif source in LOG_NAMES:\n            text = ui.log_tail(getattr(host, LOG_NAMES[source]), 16000) if host else \'尚无运行实例。\'\n        elif source == \'install\':\n            text = ui.log_tail(self.directory / \'install.log\', 16000)\n        elif source == \'preflight\':\n            text = ui.log_tail(self.directory / \'preflight.log\', 16000)\n        elif source == \'state\':\n            text = json.dumps(self._sanitize(safe), ensure_ascii=False, indent=2, sort_keys=True) if host else \'尚无运行实例。\'\n        elif source == \'smoke\':\n            smoke_log = self.scope.get(\'smoke_log\')\n            text = (ui.log_tail(smoke_log, 16000) if smoke_log else \'\') or \'暂无生成验收日志。\'\n        else:\n            text = ui.log_tail(self.log_path, 10000) + \'\\n\' + ui.last_diagnostics()[-4000:]\n        with self.lock:\n            active = [{k: v for k, v in record.items() if k not in {\'fingerprint\', \'thread_id\'}} for record in self.active.values()]\n            notice = dict(self.notice) if self.notice else None\n        resource = self.scope.get(\'RESOURCE_REPORT\') or {}\n        bundle = self.scope.get(\'bundle\')\n        connection, _ = self._connection_context(state)\n        return self._sanitize({\'state\': safe, \'resources\': {k: resource[k] for k in (\'gpu\', \'available_ram_gib\', \'residency\') if k in resource},\n                \'connection\': connection,\n                \'downloads\': self._downloads(state),\n                \'ui\': {\'connected\': not self.closed, \'closed\': self.closed, \'busy\': bool(active),\n                       \'operation\': \', \'.join(ACTION_TITLES.get(r[\'action\'], r[\'action\']) for r in active),\n                       \'operations\': active, \'preflight_ok\': self.scope.get(\'PREFLIGHT_OK\', False),\n                       \'install_ok\': self.scope.get(\'INSTALL_OK\', False),\n                       \'startup_stage\': self.startup_stage,\n                       \'startup_delivery\': dict(self.startup_delivery) if self.startup_delivery else None,\n                       \'bootstrap_running\': any(r[\'action\'] == \'bootstrap\' for r in active),\n                       \'connection_available\': connection[\'key_available\']},\n                \'logs\': {\'text\': self._clean(text)[-16000:], \'source\': source, \'observed_at\': time.time()},\n                \'notice\': notice, \'result\': {\'available\': bool(bundle and Path(bundle).is_file())}})\n\n    def _download(self, action):\n        if action == \'export_connection\':\n            connection, key = self._connection_context()\n            if not key:\n                raise RuntimeError(\'密钥尚未生成；先执行一键启动。\')\n            data = json.dumps({**connection, \'api_key\': key,\n                               \'note\': \'url 为空时尚不可连接；API 可达后重新导出。模型未就绪时任务只排队。\'}, ensure_ascii=False).encode()\n            name, mime = \'api-connection.private.json\', \'application/json\'\n        else:\n            path = self.scope.get(\'bundle\')\n            if not path or not Path(path).is_file():\n                raise RuntimeError(\'本次尚无已验证结果包。\')\n            path = Path(path)\n            if path.stat().st_size > 32 * 1024**2:\n                raise ValueError(\'结果包超过内联下载上限；请使用 Colab 文件栏下载。\')\n            data, name, mime = path.read_bytes(), path.name, \'application/zip\'\n        return {\'name\': name, \'mime\': mime, \'data_base64\': base64.b64encode(data).decode(\'ascii\')}\n\n    def dispatch(self, action, payload=None):\n        try:\n            payload = validated(payload or {})\n            if self.closed:\n                return {\'ui\': {\'connected\': False, \'closed\': True}, \'error\': \'控制台已关闭。\'}\n            if action in {\'snapshot\', \'refresh_board\'}:\n                return self.snapshot(payload.get(\'source\', \'diagnostics\'))\n            if action == \'reveal_connection\':\n                result = self.snapshot(payload.get(\'source\', \'diagnostics\'))\n                connection, key = self._connection_context()\n                if not key:\n                    raise RuntimeError(\'密钥尚未生成；先执行一键启动。\')\n                if payload.get(\'credential_id\') != connection[\'credential_id\']:\n                    raise RuntimeError(\'连接凭据已改变；请刷新看板后重新显示或复制。\')\n                result[\'connection\'] = self._sanitize(connection)\n                # This dedicated user-triggered response is never embedded into saved HTML or logs.\n                result[\'connection_secret\'] = {\'api_key\': key, \'credential_id\': connection[\'credential_id\']}\n                self._event(\'credential.revealed\', credential_id=connection[\'credential_id\'])\n                return result\n            if action in {\'export_connection\', \'download_result\'}:\n                result = self.snapshot(payload.get(\'source\', \'diagnostics\'))\n                result[\'download\'] = self._download(action)\n                self._event(\'download.prepared\', action=action)\n                return result\n            if action not in ACTION_TITLES:\n                raise ValueError(\'未知控制动作。\')\n            receipt = self.submit(action, payload)\n            return {\'accepted\': True, \'operation\': {k: v for k, v in receipt.items() if k not in {\'fingerprint\', \'thread_id\'}}}\n        except Exception as error:\n            return {\'error\': self._clean(error)[:500], \'notice\': {\'id\': uuid.uuid4().hex,\n                \'kind\': \'error\', \'title\': \'操作未执行\', \'message\': self._clean(error)[:500]}}\n\n    def close(self):\n        with self.lock:\n            if self.active or self.process_owners:\n                raise RuntimeError(\'仍有受管操作，不能丢弃控制器。\')\n            self.closed = True\n            self.epoch = None\n        self.executor.shutdown(wait=True, cancel_futures=True)\n        if ui._dashboard is self:\n            ui._dashboard = None\n        if getattr(ui, \'_process_observer\', None) is self:\n            ui._process_observer = None\n        self._event(\'console.closed\')\n\n\ndef render_console(template, callback_name=\'\', initial=None):\n    state = initial or {\'ui\': {\'connected\': False, \'offline\': True}, \'state\': {}, \'resources\': {},\n                        \'logs\': {\'text\': \'尚未连接控制台。运行本单元后，日志在这里持续更新。\', \'source\': \'diagnostics\'}}\n    safe_json = lambda data: json.dumps(data, ensure_ascii=False).replace(\'<\', \'\\\\u003c\').replace(\'>\', \'\\\\u003e\').replace(\'&\', \'\\\\u0026\')\n    return template.replace(\'__CALLBACK_NAME__\', safe_json(callback_name)).replace(\'__INITIAL_STATE__\', safe_json(state))\n\n\ndef mount_console(directory, programs, scope=None, *, auto_start=False, bootstrap_payload=None,\n                  foreground_bootstrap_timeout=None):\n    """Display the unchanged panel, optionally holding the cell until API delivery.\n\n    The owner still runs independent startup branches concurrently. The finite\n    foreground observer never waits for the model or treats expiry as failure.\n    """\n    from google.colab import output\n    from IPython.display import HTML, JSON, display\n    scope = scope if scope is not None else {}\n    controller = scope.get(\'h3_console\')\n    created = False\n    if not isinstance(controller, ConsoleController) or controller.closed:\n        controller = ConsoleController(directory, programs, scope)\n        scope[\'h3_console\'] = controller\n        created = True\n    controller.epoch = uuid.uuid4().hex\n    epoch = controller.epoch\n    name = \'h3.console.\' + epoch\n    def callback(action, payload=None):\n        if controller.epoch != epoch:\n            return JSON({\'ui\': {\'connected\': False, \'closed\': True}, \'error\': \'旧面板已停用。\'})\n        return JSON(controller.dispatch(action, payload))\n    output.register_callback(name, callback)\n    template = (Path(directory) / \'console.html\').read_text(encoding=\'utf-8\')\n    request_id = \'mount-\' + epoch\n    foreground = created and auto_start and foreground_bootstrap_timeout is not None\n    active_delivery = lambda: foreground and bool(controller.startup_delivery and controller.startup_delivery.get(\'active\'))\n    try:\n        if foreground:\n            controller.begin_startup_delivery(request_id, foreground_bootstrap_timeout)\n        initial = HTML(render_console(template, name, controller.snapshot(observe_host=False)))\n        handle = display(initial, display_id=True) if foreground else display(initial)\n        if created and auto_start:\n            payload = {\'public\': True, **(bootstrap_payload or {}), \'request_id\': request_id}\n            receipt = controller.dispatch(\'bootstrap\', payload)\n            if foreground:\n                def publish():\n                    if callable(getattr(handle, \'update\', None)):\n                        handle.update(HTML(render_console(template, name, controller.snapshot())))\n                if receipt.get(\'accepted\') is not True:\n                    with controller.lock:\n                        controller.startup_delivery.update(active=False, status=\'failed\',\n                            error=receipt.get(\'error\', \'启动未被接收\'), finished_at=time.time())\n                    publish()\n                else:\n                    controller.wait_startup_delivery(request_id, publish=publish)\n    except KeyboardInterrupt:\n        # Cover the display/dispatch window before wait_startup_delivery owns\n        # interrupts. Its own handler sets active=False, preventing double stop.\n        if active_delivery():\n            status, error = controller._interrupt_delivery(controller.startup_delivery)\n            controller.startup_delivery.update(status=status, error=error, active=False, finished_at=time.time())\n            controller._event(\'startup.delivery_finished\', **dict(controller.startup_delivery))\n        raise\n    finally:\n        if active_delivery():\n            controller.startup_delivery.update(status=\'observation_failed\', active=False, finished_at=time.time())\n            controller._event(\'startup.delivery_finished\', **dict(controller.startup_delivery))\n    return controller\n', 'console.html': '<!doctype html>\n<html lang="zh-CN">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>MiniMax-H3 控制台</title>\n<style>\n:root {\n  color-scheme: light dark;\n  --canvas: #f1f2f5;\n  --panel: #fff;\n  --surface: #f5f5f7;\n  --input: #f7f7f9;\n  --toolbar: #fafafc;\n  --ink: #1d1d1f;\n  --muted: #5f636d;\n  --line: #dedfe5;\n  --edge: #2226341a;\n  --accent: #0065d8;\n  --accent-hover: #005bc4;\n  --tint: #eaf2ff;\n  --ready: #187641;\n  --danger: #b32634;\n  --warn: #80540e;\n  --log: #f7f8fb;\n  --scroll-thumb: #bfc1ca;\n  --shadow: 0 1px 2px #20283a08, 0 5px 14px #20283a0b;\n  --mono: ui-monospace, "SFMono-Regular", Consolas, "Microsoft YaHei", monospace;\n}\n\n@media (prefers-color-scheme: dark) {\n  :root {\n    --canvas: #16171b;\n    --panel: #222327;\n    --surface: #2b2c31;\n    --input: #292a30;\n    --toolbar: #25262b;\n    --ink: #f2f2f5;\n    --muted: #b3b5bf;\n    --line: #3c3e46;\n    --edge: #ffffff20;\n    --accent: #78b6ff;\n    --accent-hover: #9ac9ff;\n    --tint: #263953;\n    --ready: #73d69a;\n    --danger: #ff9398;\n    --warn: #edc278;\n    --log: #1c1d22;\n    --scroll-thumb: #646773;\n    --shadow: 0 1px 2px #0004, 0 5px 14px #0003;\n  }\n}\n\n* { box-sizing: border-box; }\nbody {\n  margin: 0;\n  color: var(--ink);\n  font: 13px/1.5 -apple-system, BlinkMacSystemFont, "Segoe UI Variable Text", "Segoe UI", "PingFang SC", "Microsoft YaHei", system-ui, sans-serif;\n}\nh1, h2, p { margin: 0; }\nbutton, input, textarea, select { font: inherit; }\nbutton, input[type="checkbox"], select, summary { cursor: pointer; }\nbutton {\n  min-height: 33px;\n  padding: 5px 10px;\n  border: 1px solid var(--edge);\n  border-radius: 9px;\n  background: var(--panel);\n  color: var(--ink);\n  font-weight: 600;\n  overflow-wrap: anywhere;\n}\nbutton:hover:not(:disabled) { background: var(--surface); }\nbutton:active:not(:disabled) { box-shadow: inset 0 1px 3px #0002; }\nbutton:disabled { cursor: default; color: var(--muted); opacity: .8; }\nbutton.primary { border-color: transparent; background: var(--accent); color: light-dark(#fff, #102640); }\nbutton.primary:hover:not(:disabled) { background: var(--accent-hover); }\nbutton.primary:disabled { background: var(--tint); color: var(--accent); opacity: 1; }\nbutton.danger { color: var(--danger); }\n:is(button, input, textarea, select, summary, a, .board, .control, .log-viewport):focus-visible {\n  outline: 2px solid var(--accent);\n  outline-offset: 2px;\n}\n:is(.board, .control, .log-viewport):focus-visible { outline-offset: -3px; }\ninput:not([type="checkbox"]), textarea, select {\n  width: 100%;\n  min-width: 0;\n  min-height: 34px;\n  padding: 5px 8px;\n  border: 1px solid var(--line);\n  border-radius: 8px;\n  background: var(--input);\n  color: var(--ink);\n}\ninput::placeholder, textarea::placeholder { color: var(--muted); opacity: 1; }\ntextarea { min-height: 52px; max-height: 112px; resize: vertical; }\ninput[type="checkbox"] {\n  width: 15px;\n  height: 15px;\n  flex: 0 0 auto;\n  margin: 0;\n  accent-color: var(--accent);\n}\nlabel { display: block; color: var(--muted); font-size: 12px; }\nlabel > :is(input, textarea, select) { margin-top: 4px; }\n\n.app {\n  display: grid;\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n  gap: 12px;\n  width: 100%;\n  height: 686px;\n  padding: 10px;\n  border-radius: 22px;\n  background: var(--canvas);\n}\n.left {\n  display: grid;\n  grid-template-rows: 392px minmax(0, 1fr);\n  gap: 12px;\n  min-width: 0;\n  min-height: 0;\n}\n.panel {\n  min-width: 0;\n  min-height: 0;\n  overflow: hidden;\n  border: 1px solid var(--edge);\n  border-radius: 16px;\n  background: var(--panel);\n  box-shadow: var(--shadow);\n}\n.panel-head {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n  flex-shrink: 0;\n  padding: 11px 15px;\n  border-bottom: 1px solid var(--line);\n}\n.panel-head h2 { font-size: 16px; line-height: 1.3; font-weight: 650; }\n.eyebrow { color: var(--muted); font-size: 11px; font-weight: 550; }\n:is(.board, .control, .log-viewport, .notice-zone) {\n  scrollbar-width: thin;\n  scrollbar-color: var(--scroll-thumb) transparent;\n}\n\n.board {\n  display: flex;\n  flex-direction: column;\n  padding: 12px 15px 10px;\n  overflow-y: auto;\n  overscroll-behavior: contain;\n  scrollbar-gutter: stable;\n}\n.board-top { display: flex; align-items: flex-start; justify-content: space-between; gap: 8px; }\n.board-top > div { min-width: 0; }\n.board h1 { margin-top: 1px; font-size: 22px; line-height: 1.25; font-weight: 650; overflow-wrap: anywhere; }\n.board[data-tone="error"] h1 { color: var(--danger); }\n.board[data-tone="warn"] h1 { color: var(--warn); }\n.board[data-tone="ready"] h1 { color: var(--ready); }\n.board-refresh { min-height: 26px; padding: 3px 8px; flex-shrink: 0; font-size: 11px; }\n.board-message { color: var(--muted); font-size: 11px; overflow-wrap: anywhere; }\n.capabilities {\n  display: grid;\n  grid-template-columns: repeat(4, minmax(0, 1fr));\n  gap: 1px;\n  flex-shrink: 0;\n  margin: 9px 0;\n  overflow: hidden;\n  border: 1px solid var(--line);\n  border-radius: 10px;\n  background: var(--line);\n}\n.capability { min-width: 0; padding: 5px 8px; background: var(--surface); overflow-wrap: anywhere; }\n.capability span { display: block; color: var(--muted); font-size: 10px; line-height: 1.4; }\n.capability strong { display: block; font-size: 12px; line-height: 1.4; font-weight: 650; }\n.connection-box { flex-shrink: 0; }\n.connection-row {\n  display: grid;\n  grid-template-columns: 53px minmax(0, 1fr) auto;\n  gap: 5px;\n  align-items: center;\n  margin-bottom: 4px;\n}\n.connection-row.key-row { grid-template-columns: 53px minmax(0, 1fr) auto auto; }\n.connection-row label { font-size: 11px; white-space: nowrap; }\n.connection-row input { height: 27px; min-height: 27px; padding: 3px 7px; font: 11px/1.3 var(--mono); }\n.connection-row button { min-height: 27px; padding: 3px 7px; font-size: 11px; }\n.connection-hint { color: var(--muted); font-size: 10px; overflow-wrap: anywhere; }\n.transfers { flex-shrink: 0; margin: 7px 0 6px; border-top: 1px solid var(--line); }\n.transfer-row {\n  position: relative;\n  display: grid;\n  grid-template-columns: 69px minmax(0, 1fr) 40px;\n  gap: 0 5px;\n  min-height: 34px;\n  padding: 4px 0;\n  border-bottom: 1px solid var(--line);\n  overflow-wrap: anywhere;\n}\n.transfer-name { grid-row: 1 / 3; align-self: center; font-size: 11px; font-weight: 650; }\n.transfer-phase { font-size: 11px; line-height: 1.35; }\n.transfer-number { color: var(--muted); font-size: 10px; text-align: right; font-variant-numeric: tabular-nums; }\n.transfer-detail { grid-column: 2 / 4; color: var(--muted); font-size: 10px; line-height: 1.35; }\n.transfer-row progress {\n  appearance: none;\n  position: absolute;\n  inset: auto 0 0;\n  width: 100%;\n  height: 3px;\n  overflow: hidden;\n  border: 0;\n  border-radius: 99px;\n  background: var(--line);\n}\n.transfer-row progress::-webkit-progress-bar { background: var(--line); }\n.transfer-row progress::-webkit-progress-value { background: var(--accent); }\n.transfer-row progress::-moz-progress-bar { background: var(--accent); }\n.transfer-row[data-phase="failed"] .transfer-phase { color: var(--danger); }\n.transfer-row[data-phase="disabled"] .transfer-phase { color: var(--muted); }\n#resources { font-size: 10px; overflow-wrap: anywhere; }\n.board-bottom {\n  display: flex;\n  justify-content: space-between;\n  align-items: center;\n  flex: 0 0 auto;\n  flex-wrap: wrap;\n  gap: 4px 8px;\n  margin-top: auto;\n  padding-top: 5px;\n  font-size: 10px;\n}\n.queue { padding: 2px 7px; border-radius: 99px; background: var(--surface); font-variant-numeric: tabular-nums; }\n.stamp { color: var(--muted); }\n\n.control {\n  overflow: auto;\n  overscroll-behavior: contain;\n  scrollbar-gutter: stable;\n}\n.control-actions {\n  display: grid;\n  gap: 6px;\n  padding: 8px 12px;\n}\n.control-actions button { min-height: 31px; padding: 4px 6px; font-size: 12px; }\n.control-body { padding: 4px 12px 13px; }\n.button-row { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 6px; }\n.button-row.two { grid-template-columns: repeat(2, minmax(0, 1fr)); }\n.checkline { display: flex; align-items: center; gap: 8px; margin: 3px 0 10px; color: var(--ink); }\n.checkline input { margin: 0; }\n#public {\n  appearance: none;\n  position: relative;\n  width: 31px;\n  height: 18px;\n  border: 1px solid var(--line);\n  border-radius: 99px;\n  background: var(--surface);\n}\n#public::before {\n  position: absolute;\n  inset: 2px auto 2px 2px;\n  aspect-ratio: 1;\n  border-radius: 50%;\n  background: #fff;\n  box-shadow: 0 1px 3px #0004;\n  content: "";\n}\n#public:checked { border-color: transparent; background: var(--accent); }\n#public:checked::before { inset-inline: auto 2px; }\n@media (forced-colors: active) {\n  #public { appearance: auto; width: 15px; height: 15px; }\n  #public::before { content: none; }\n}\n.form-grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 8px; }\n.full { grid-column: 1 / -1; }\n.field-note { margin-top: 5px; color: var(--muted); font-size: 11px; overflow-wrap: anywhere; }\n#download-result { margin-top: 9px; }\n.advanced { margin-top: 10px; border-top: 1px solid var(--line); }\n.advanced summary { padding: 8px 0 4px; font-size: 12px; font-weight: 650; }\n.advanced h3 { margin: 10px 0 6px; font-size: 12px; font-weight: 650; }\n.advanced .checkline { margin: 8px 0; }\n\n.right { display: flex; flex-direction: column; }\n.right .panel-head > span { padding: 3px 8px; border-radius: 99px; background: var(--surface); }\n.status-dot { display: inline-block; width: 7px; height: 7px; margin-right: 6px; border-radius: 50%; background: var(--muted); }\n.status-dot.live { background: var(--ready); }\n.log-controls {\n  display: flex;\n  flex: 0 0 auto;\n  flex-wrap: wrap;\n  align-items: center;\n  gap: 6px;\n  padding: 8px 12px;\n  border-bottom: 1px solid var(--line);\n  background: var(--toolbar);\n}\n.log-controls :is(button, select) { min-height: 30px; padding: 4px 8px; font-size: 12px; }\n.log-controls .log-source { flex: 1 1 9rem; min-width: 0; max-width: 16rem; }\n.log-controls .log-lines { flex: 0 1 auto; width: auto; max-width: 100%; }\n.follow { display: flex; align-items: center; gap: 5px; margin-left: auto; white-space: nowrap; font-size: 11px; }\n.follow input { margin: 0; }\n.notice-zone { flex: 0 0 auto; max-height: 150px; padding: 0 12px; overflow: auto; overscroll-behavior: contain; }\n.notice { margin-top: 9px; padding: 9px 11px; border: 1px solid var(--line); border-left: 3px solid var(--accent); border-radius: 9px; background: var(--tint); }\n.notice[data-kind="error"] { border-left-color: var(--danger); background: var(--surface); }\n.notice[data-kind="warning"] { border-left-color: var(--warn); background: var(--surface); }\n.notice-title { font-size: 12px; font-weight: 650; }\n.notice-message { margin-top: 4px; color: var(--muted); font-size: 12px; white-space: pre-wrap; overflow-wrap: anywhere; }\n.notice-actions { display: flex; align-items: center; flex-wrap: wrap; gap: 7px; margin-top: 7px; }\n.notice-actions :is(button, a) { min-height: 30px; padding: 4px 9px; font-size: 12px; }\n.notice-actions a { border: 1px solid var(--line); border-radius: 8px; color: var(--accent); background: var(--panel); text-decoration: none; }\n.notice-actions a:hover { text-decoration: underline; }\n.log-viewport {\n  flex: 1;\n  min-height: 0;\n  margin: 10px 12px 0;\n  overflow: auto;\n  overscroll-behavior: contain;\n  scrollbar-gutter: stable;\n  border: 1px solid var(--line);\n  border-radius: 11px;\n  background: var(--log);\n}\n.log-intro { padding: 12px 13px; border-bottom: 1px solid var(--line); background: var(--toolbar); font-size: 12px; overflow-wrap: anywhere; }\n.log-intro h3 { margin: 0 0 4px; font-size: 14px; font-weight: 650; }\n.log-intro p { color: var(--muted); }\n.log-viewport pre { margin: 0; padding: 11px 13px; color: var(--muted); font: 12px/1.65 var(--mono); white-space: pre-wrap; overflow-wrap: anywhere; }\n.log-footer { display: flex; justify-content: space-between; flex: 0 0 auto; flex-wrap: wrap; gap: 4px 8px; padding: 8px 13px; color: var(--muted); font-size: 10px; }\n.sr-only { position: absolute; width: 1px; height: 1px; padding: 0; margin: -1px; overflow: hidden; clip: rect(0, 0, 0, 0); white-space: nowrap; border: 0; }\n\n@media (max-width: 920px) {\n  .follow { margin-left: 0; }\n  .capability { padding-inline: 5px; }\n}\n@media (max-width: 759px) {\n  .app { height: auto; grid-template-columns: minmax(0, 1fr); padding: 8px; gap: 10px; }\n  .left { grid-template-rows: auto 390px; gap: 10px; }\n  .board { scrollbar-gutter: auto; }\n  .right { height: 440px; }\n  .log-controls .log-source { max-width: none; }\n}\n@media (max-width: 379px) {\n  .capabilities, .control-actions .button-row, .advanced .button-row { grid-template-columns: repeat(2, minmax(0, 1fr)); }\n  .form-grid { grid-template-columns: minmax(0, 1fr); }\n}\n</style>\n</head>\n<body>\n<main class="app" id="h3-console" aria-label="MiniMax-H3 控制台">\n  <div class="left">\n    <section class="panel board" id="board" aria-labelledby="headline" tabindex="0">\n      <div class="board-top"><div><p class="eyebrow">消息看板</p><h1 id="headline">尚未读取状态</h1></div><button type="button" class="board-refresh" id="refresh-board" title="只刷新状态，不执行控制命令">刷新</button></div>\n      <p class="board-message" id="next-step">运行控制台单元后显示本次观察。</p>\n      <div class="capabilities" aria-label="独立能力">\n        <div class="capability"><span>总管理</span><strong id="manager-state">尚未读取</strong></div>\n        <div class="capability"><span>API 后台</span><strong id="api-state">尚未读取</strong></div>\n        <div class="capability"><span>模型容器</span><strong id="model-state">尚未读取</strong></div>\n        <div class="capability"><span>公网 API</span><strong id="public-state">尚未读取</strong></div>\n      </div>\n      <div class="connection-box" aria-label="API 连接信息">\n        <div class="connection-row"><label for="api-url">API URL</label><input id="api-url" readonly value="尚未分配" autocomplete="off" spellcheck="false" aria-describedby="connection-hint"><button type="button" id="copy-url" disabled aria-label="复制 API URL">复制</button></div>\n        <div class="connection-row key-row"><label for="api-key">API 密钥</label><input id="api-key" readonly value="尚未生成" autocomplete="off" spellcheck="false" aria-label="API 密钥"><button type="button" id="toggle-key" disabled aria-label="显示 API 密钥">显示</button><button type="button" id="copy-key" disabled aria-label="复制 API 密钥">复制</button></div>\n        <p class="connection-hint" id="connection-hint">API 可达后可复制地址；密钥按需显示。</p>\n      </div>\n      <div class="transfers" aria-label="模型与 LoRA 下载进度">\n        <div class="transfer-row" id="transfer-model"><span class="transfer-name">模型</span><span class="transfer-phase" id="model-download-phase">尚未观察</span><span class="transfer-number" id="model-download-number">—</span><span class="transfer-detail" id="model-download-detail">下载与加载分别显示</span><progress id="model-download-progress" max="100" aria-label="模型下载进度"></progress></div>\n        <div class="transfer-row" id="transfer-acceleration"><span class="transfer-name">加速 LoRA</span><span class="transfer-phase" id="acceleration-download-phase">尚未观察</span><span class="transfer-number" id="acceleration-download-number">—</span><span class="transfer-detail" id="acceleration-download-detail">未启用时不下载</span><progress id="acceleration-download-progress" max="100" aria-label="加速 LoRA 下载进度"></progress></div>\n        <div class="transfer-row" id="transfer-style"><span class="transfer-name">风格 LoRA</span><span class="transfer-phase" id="style-download-phase">尚未观察</span><span class="transfer-number" id="style-download-number">—</span><span class="transfer-detail" id="style-download-detail">未启用时不下载</span><progress id="style-download-progress" max="100" aria-label="风格 LoRA 下载进度"></progress></div>\n      </div>\n      <p class="eyebrow" id="resources">配置：A100 · 原生 BF16</p>\n      <div class="board-bottom"><span class="queue" id="queue-state">排队 —\u3000执行 —</span><span class="stamp" id="board-stamp">尚无服务观察</span></div>\n    </section>\n    <section class="panel control" id="control-scroll" aria-labelledby="control-heading" tabindex="0">\n      <header class="panel-head"><h2 id="control-heading">常用控制</h2><span class="eyebrow">启动后自动推进</span></header>\n      <div class="control-actions">\n        <div class="button-row two"><button type="button" data-action="bootstrap" class="primary">一键启动</button><button type="button" data-action="export_connection">导出 API 连接</button></div>\n        <div class="button-row"><button type="button" data-action="generate">提交 / 查询任务</button><button type="button" data-action="stop">停止服务</button><button type="button" class="danger" data-action="release">释放云实例</button></div>\n      </div>\n      <div class="control-body">\n        <label class="checkline"><input id="public" type="checkbox" checked>开放临时公网 API</label>\n        <div class="form-grid">\n          <label class="full">本次参考图路径<input id="reference_file" placeholder="/content/本次新参考图.png" autocomplete="off"></label>\n          <label class="full">本次视频描述<textarea id="prompt" rows="2" placeholder="描述本次全新内容；按已有任务 ID 恢复时可留空"></textarea></label>\n          <label>已有任务 ID<input id="resume_job_id" placeholder="留空则创建新任务" autocomplete="off"></label>\n          <label>验收接口<select id="endpoint"><option value="public">临时公网</option><option value="local">本机 API</option></select></label>\n        </div>\n        <button type="button" data-action="download_result" id="download-result" disabled>下载最近结果</button>\n        <details class="advanced" id="advanced-controls"><summary>高级控制 · 不常用时收起</summary><div class="advanced-body">\n          <h3>分步维护</h3>\n          <div class="button-row"><button type="button" data-action="preflight">检查资源</button><button type="button" data-action="install">安装依赖</button><button type="button" data-action="start">单独启动</button></div>\n          <p class="field-note">正常使用只需一键启动；检查、安装或启动失败时在日志区处理。</p>\n          <label class="checkline"><input id="staged" type="checkbox">使用 40GB 分阶段驻留</label>\n          <label class="checkline"><input id="cached" type="checkbox">权重已完整缓存</label>\n          <h3>性能与 LoRA</h3>\n          <div class="form-grid">\n            <label>加速 LoRA<select id="acceleration"><option value="none">不启用</option><option value="lightx2v-ref2va-4step">Ref2VA Turbo · 实验</option></select></label>\n            <label>风格 LoRA<select id="style"><option value="none">不启用</option><option value="fal-realism-people">人物写实 · 实验</option></select></label>\n            <label>风格强度<input id="style_scale" type="number" min="0.1" max="1" step="0.1" value="1"></label>\n            <label>注意力后端<select id="attention"><option value="auto">自动</option><option value="native">原生</option><option value="_native_flash">原生 Flash · 实验</option></select></label>\n            <label>去噪卸载<select id="offload"><option value="stream">流式</option><option value="sync">同步 · 实验</option></select></label>\n          </div><p class="field-note">高级参数在安装与启动前设置；保持 BF16，LoRA 默认关闭。</p>\n          <h3>组件控制</h3>\n          <div class="form-grid"><label>操作<select id="component-action"><option value="drain">暂停接收新任务</option><option value="resume">恢复接收新任务</option><option value="stop_model">停止模型</option><option value="start_model">启动模型</option><option value="stop_api">停止 API</option><option value="start_api">启动 API</option></select></label><label>命令 ID<input id="command_id" placeholder="查询已有命令" autocomplete="off"></label></div>\n          <label class="checkline"><input id="force" type="checkbox">强制停止模型</label>\n          <div class="button-row two"><button type="button" id="send-control">执行组件操作</button><button type="button" id="query-command">查询命令</button></div>\n          <p class="field-note">暂停接收时，已有队列继续执行。命令接收不等于执行完成。</p>\n          <h3>客户端连接</h3><p class="field-note">启动时自动生成 API 密钥，可用上方按钮导出。地址为空表示尚不可连接；API 可达后重新导出，密钥不变。模型准备期间接受排队，完整就绪后自动处理。</p>\n        </div></details>\n      </div>\n    </section>\n  </div>\n  <section class="panel right" aria-labelledby="logs-heading">\n    <header class="panel-head"><h2 id="logs-heading">动态日志</h2><span class="eyebrow"><i class="status-dot" id="live-dot" aria-hidden="true"></i><span id="connection-state">等待连接</span></span></header>\n    <div class="log-controls">\n      <label class="sr-only" for="log-source">日志来源</label>\n      <select id="log-source" class="log-source">\n        <option value="diagnostics">操作与诊断</option><option value="preflight">资源检查</option>\n        <option value="install">安装依赖</option><option value="model">模型容器</option>\n        <option value="api">API 后台</option><option value="manager">全局管理</option>\n        <option value="tunnel">公网隧道</option><option value="events">调度事件</option>\n        <option value="state">状态详情</option><option value="smoke">最近验收</option>\n      </select>\n      <label class="sr-only" for="log-lines">最新日志行数</label>\n      <select id="log-lines" class="log-lines">\n        <option value="20">20 行</option><option value="40">40 行</option><option value="60">60 行</option>\n        <option value="80">80 行</option><option value="100" selected>100 行</option><option value="120">120 行</option>\n        <option value="140">140 行</option><option value="160">160 行</option><option value="180">180 行</option>\n        <option value="200">200 行</option>\n      </select>\n      <button type="button" id="toggle-logs">暂停</button><button type="button" id="refresh-logs">刷新</button>\n      <label class="follow"><input type="checkbox" id="follow-logs" checked>跟随新日志</label>\n    </div>\n    <div class="notice-zone" id="notice-zone" aria-live="polite" aria-atomic="false"></div>\n    <div class="log-viewport" id="log-viewport" tabindex="0" aria-label="日志内容，可独立滚动">\n      <header class="log-intro" id="log-intro">\n        <h3>MiniMax-H3 · 参考生视频</h3>\n        <p>A100 原生 BF16 · 仅 ref2va。点击 Colab「一键启动」单元的运行箭头后，自动检查资源、安装依赖并启动 API、公网和模型。API 可达后即可排队，模型就绪后自动处理。</p>\n      </header>\n      <pre id="log-text">尚无本次运行日志。</pre>\n    </div>\n    <footer class="log-footer"><span id="log-stamp">只读 · 有限尾部 · 凭据脱敏</span><span id="log-mode">每 2 秒读取</span></footer>\n  </section>\n</main>\n<script>\n(() => {\n  \'use strict\';\n  const CALLBACK_NAME = __CALLBACK_NAME__;\n  const INITIAL_STATE = __INITIAL_STATE__;\n  const $ = id => document.getElementById(id);\n  const actionButtons = [...document.querySelectorAll(\'[data-action]\')];\n  const activeActions = new Set();\n  const uncertainRequests = new Map();\n  const knownNotices = new Set();\n  const blobUrls = new Set();\n  let current = INITIAL_STATE && typeof INITIAL_STATE === \'object\' ? INITIAL_STATE : {};\n  let timer = null, polling = false, paused = false, disposed = false;\n  let connectionLost = false, localNotice = null, shownNoticeId = null, failures = 0;\n  let lastText = \'\', lastSource = \'\', lastLogs = null, pendingConfirmation = null;\n  let revealedKey = \'\', keyVisible = false, secretBusy = false, secretEpoch = 0;\n  let observedCredentialId = null, observedInstanceId = null;\n  const hasTransport = () => typeof window.H3_TRANSPORT === \'function\' || typeof window.google?.colab?.kernel?.invokeFunction === \'function\';\n  const offline = () => current.ui?.offline === true || !hasTransport();\n  const closed = () => current.ui?.closed === true;\n  const delivering = () => current.ui?.startup_delivery?.active === true &&\n    Date.now() < Number(current.ui.startup_delivery.deadline_at || 0) * 1000 + 5000;\n  const text = (id, value) => { $(id).textContent = String(value ?? \'—\'); };\n  const nowText = value => {\n    if (!value) return \'未记录\';\n    const date = typeof value === \'number\' ? new Date(value * 1000) : new Date(value);\n    if (!Number.isFinite(date.getTime())) return \'未记录\';\n    return new Intl.DateTimeFormat(\'zh-CN\', {timeZone:\'Asia/Shanghai\', hour:\'2-digit\', minute:\'2-digit\', second:\'2-digit\', hour12:false}).format(date) + \' 北京时间\';\n  };\n  const safeText = value => String(value ?? \'\').replace(/\\x1b\\[[0-?]*[ -/]*[@-~]/g, \'\').replace(/\\bBearer\\s+[^\\s"\'<>;,]+/gi,\'Bearer [已隐藏]\').replace(/(["\']?(?:H3_API_KEY|API_KEY|api[-_]?key|access_token)["\']?\\s*[:=]\\s*)(["\'])(.*?)\\2/gi,\'$1$2[已隐藏]$2\');\n  const payload = () => ({staged:$(\'staged\').checked,cached:$(\'cached\').checked,public:$(\'public\').checked,\n    reference_file:$(\'reference_file\').value.trim(),prompt:$(\'prompt\').value.trim(),resume_job_id:$(\'resume_job_id\').value.trim(),\n    endpoint:$(\'endpoint\').value,acceleration:$(\'acceleration\').value,style:$(\'style\').value,style_scale:Number($(\'style_scale\').value),\n    attention:$(\'attention\').value,offload:$(\'offload\').value});\n  function uuid() {\n    if (window.crypto?.randomUUID) return window.crypto.randomUUID();\n    const bytes = new Uint8Array(16); window.crypto.getRandomValues(bytes);\n    return [...bytes].map(value=>value.toString(16).padStart(2,\'0\')).join(\'\');\n  }\n  function setButtons() {\n    const unavailable = offline() || closed() || delivering();\n    const busy = current.ui?.busy === true;\n    const operations = current.ui?.operations || [];\n    const generationBusy = operations.some(item=>item.action===\'generate\');\n    const stopping = current.state?.phase===\'stopping\' || operations.some(item=>[\'stop\',\'release\'].includes(item.action));\n    const apiUsable = current.state?.api_ready===true && current.state?.observation_stale!==true && !connectionLost;\n    const connectionAvailable = current.ui?.connection_available===true || apiUsable;\n    for (const button of actionButtons) {\n      const action = button.dataset.action;\n      const setupBlocked = (busy || current.ui?.bootstrap_running===true) && [\'bootstrap\',\'preflight\',\'install\',\'start\'].includes(action);\n      const generationBlocked = action===\'generate\' && (stopping || generationBusy || (busy && !apiUsable));\n      button.disabled = unavailable || activeActions.has(action) || setupBlocked || generationBlocked || (action===\'export_connection\' && !connectionAvailable) || (action===\'download_result\' && !current.result?.available);\n    }\n    $(\'send-control\').disabled = unavailable || activeActions.has(\'control\');\n    $(\'query-command\').disabled = unavailable || activeActions.has(\'control\');\n    $(\'refresh-board\').disabled = unavailable || polling;\n    $(\'refresh-logs\').disabled = unavailable || polling;\n    $(\'toggle-logs\').disabled = unavailable;\n    const connection=current.connection||{};\n    const secretUnavailable=unavailable || connectionLost || document.hidden || !connection.key_available || !connection.credential_id;\n    $(\'toggle-key\').disabled=secretUnavailable || secretBusy;\n    $(\'copy-key\').disabled=secretUnavailable || secretBusy;\n    $(\'copy-url\').disabled=unavailable || connectionLost || current.state?.observation_stale===true || !usableUrl(connection.url);\n  }\n  function usableUrl(value) {\n    if(typeof value!==\'string\'||!value)return null;\n    try {const url=new URL(value);return [\'http:\',\'https:\'].includes(url.protocol)&&!url.username&&!url.password?value:null;}catch{return null;}\n  }\n  function clearSecret() {\n    revealedKey=\'\';keyVisible=false;secretEpoch++;\n    $(\'api-key\').value=current.connection?.key_available?\'••••••••••••••••\':\'尚未生成\';\n    text(\'toggle-key\',\'显示\');$(\'toggle-key\').setAttribute(\'aria-label\',\'显示 API 密钥\');\n  }\n  function renderConnection() {\n    const connection=current.connection||{};\n    const credentialId=connection.credential_id||null, instanceId=current.state?.instance_id||null;\n    if(credentialId!==observedCredentialId || instanceId!==observedInstanceId || offline() || closed() || connectionLost || document.hidden){\n      clearSecret();observedCredentialId=credentialId;observedInstanceId=instanceId;\n    }\n    const stale=current.state?.observation_stale===true;\n    const readyUrl=stale?null:usableUrl(connection.url), candidate=usableUrl(connection.candidate_url)||(stale?usableUrl(connection.url):null);\n    const local=usableUrl(connection.local_url);\n    $(\'api-url\').value=readyUrl||candidate||(offline()?\'尚未连接运行时\':\'等待分配地址\');\n    $(\'api-url\').title=readyUrl||candidate||\'\';\n    let hint=\'密钥在启动初期生成；按需显示或复制。\';\n    if(offline()||closed())hint=\'当前为离线界面，没有本次连接或密钥。\';\n    else if(connectionLost)hint=\'连接中断；上次地址不代表当前可用，密钥已隐藏。\';\n    else if(readyUrl)hint=(connection.public_requested?\'公网 API 已通过探测\':\'本机 API 已可达\')+\' · 密钥默认隐藏\';\n    else if(candidate)hint=\'地址已分配 · 尚未验证／当前不可用\'+(local?\' · 本机 API 可用\':\'\');\n    else if(connection.public_requested&&local)hint=\'公网地址准备中 · 本机 API 已可用\';\n    else if(current.ui?.bootstrap_running)hint=\'正在准备 API；密钥生成后即可显示或复制。\';\n    text(\'connection-hint\',hint);$(\'connection-hint\').title=hint+(local?\'\\n本机 API：\'+local:\'\');\n    $(\'api-key\').value=keyVisible&&revealedKey?revealedKey:connection.key_available?\'••••••••••••••••\':\'尚未生成\';\n    text(\'toggle-key\',keyVisible?\'隐藏\':\'显示\');\n    $(\'toggle-key\').setAttribute(\'aria-label\',keyVisible?\'隐藏 API 密钥\':\'显示 API 密钥\');\n  }\n  function bytesText(value) {\n    if(typeof value!==\'number\'||!Number.isFinite(value)||value<0)return \'—\';\n    const units=[\'B\',\'KiB\',\'MiB\',\'GiB\',\'TiB\'];let amount=value,index=0;\n    while(amount>=1024&&index<units.length-1){amount/=1024;index++;}\n    return (index?amount.toFixed(1):Math.round(amount))+\' \'+units[index];\n  }\n  function renderTransfers() {\n    const downloads=current.downloads||{}, items=downloads.items||{};\n    const stale=downloads.observation_stale===true || connectionLost || current.state?.observation_stale===true;\n    const phases={waiting:\'等待下载\',downloading:\'下载中\',verifying:\'校验中\',loading:\'加载中\',loaded:\'已加载\',disabled:\'未启用\',failed:\'准备失败\'};\n    for(const name of [\'model\',\'acceleration\',\'style\']) {\n      const item=items[name], row=$(\'transfer-\'+name), progress=$(name+\'-download-progress\');\n      let phase=\'尚未观察\', detail=name===\'model\'?\'下载与加载分别显示\':\'未启用时不下载\', percent=null;\n      if(item&&typeof item===\'object\') {\n        phase=phases[item.phase]||\'等待可测进度\';\n        if(name===\'model\'&&item.phase===\'loaded\')phase=\'已加载 · 等待就绪检查\';\n        if(name===\'model\'&&current.state?.model_ready===true&&!stale)phase=\'可推理\';\n        const bytesKnown=typeof item.bytes_completed===\'number\'&&Number.isFinite(item.bytes_completed)&&item.bytes_completed>=0;\n        const totalKnown=typeof item.bytes_total===\'number\'&&Number.isFinite(item.bytes_total)&&item.bytes_total>0;\n        const parts=[];\n        if(bytesKnown)parts.push((item.scope===\'observed_transfers\'?\'已观察 \':\'\')+bytesText(item.bytes_completed)+(totalKnown?\' / \'+bytesText(item.bytes_total):\' · 总量未知\'));\n        if(Number.isInteger(item.files_completed)&&item.files_completed>=0)parts.push(\'文件 \'+item.files_completed+\' / \'+(Number.isInteger(item.files_total)&&item.files_total>=0?item.files_total:\'?\'));\n        detail=parts.join(\' · \')||item.detail||\'等待可测进度\';\n        if(bytesKnown&&item.scope===\'observed_transfers\')detail+=\'（含续传）\';\n        if(item.phase===\'disabled\')detail=\'不下载，不占用加载资源\';\n        else if(item.detail&&[\'loaded\',\'loading\',\'verifying\',\'failed\'].includes(item.phase))detail=item.detail;\n        if(!stale&&totalKnown&&typeof item.percent===\'number\'&&Number.isFinite(item.percent)&&item.percent>=0&&item.percent<=100)percent=item.percent;\n        if([\'stopped\',\'stopping\',\'failed\'].includes(downloads.lifecycle)&&item.phase!==\'disabled\'){\n          phase=({stopped:\'已停止\',stopping:\'正在停止\',failed:\'已失败\'})[downloads.lifecycle]+\' · 上次\'+phase;\n          detail=\'保留记录：\'+detail;percent=null;\n        } else if(stale&&item.phase!==\'disabled\'){phase=\'待刷新 · 上次\'+phase;detail=\'上次观察：\'+detail;}\n        row.dataset.phase=item.phase||\'waiting\';\n      } else {row.dataset.phase=\'waiting\';if(!offline()&&current.ui?.bootstrap_running)detail=\'下载观察尚未开始\';}\n      if(offline()||closed()){phase=\'尚未观察\';detail=name===\'model\'?\'连接后显示真实下载与加载状态\':\'连接后显示实际启用状态\';percent=null;}\n      text(name+\'-download-phase\',phase);$(name+\'-download-phase\').title=phase;\n      text(name+\'-download-detail\',safeText(detail));\n      $(name+\'-download-detail\').title=safeText([detail,item?.detail].filter(Boolean).join(\'\\n\'));\n      text(name+\'-download-number\',percent===null?\'—\':percent.toFixed(percent===100?0:1)+\'%\');\n      progress.hidden=percent===null;\n      if(percent===null)progress.removeAttribute(\'value\');else progress.value=percent;\n    }\n  }\n  function renderBoard() {\n    const state=current.state||{}, ui=current.ui||{}, resources=current.resources||{};\n    const cleanStop=!ui.bootstrap_running && state.phase===\'stopped\' && state.cleanup_complete===true && !state.manager_alive && !state.server_alive && !state.model_alive && !state.tunnel_alive;\n    const stale=(state.observation_stale===true || connectionLost || ui.observation_error===true) && !cleanStop;\n    let headline=\'等待一键启动\', next=\'一键完成检查、安装与启动；API 就绪后可排队，模型就绪后自动处理。\', tone=\'\';\n    if (closed()) {headline=\'控制台连接已关闭\';next=\'需要继续操作时，重新运行控制台单元。\';}\n    else if (offline()) {headline=\'未连接运行时\';next=\'运行本格即自动启动；正常无需再次点击。此处尚无实时状态。\';}\n    else if(delivering()){headline=\'一键启动 · 交付 API\';next=\'主单元正有限观察，状态由宿主推送；如需取消，请停止本单元，程序会回收已启动进程。\';}\n    else if(cleanStop){headline=\'服务已停止\';next=\'进程清理已确认；释放云实例需另行操作。\';}\n    else if(stale){headline=\'状态需要刷新\';next=\'当前观察无法确认；查看右侧日志，不据旧状态判断可用能力。\';tone=\'warn\';}\n    else if(state.phase===\'stopping\'){headline=\'正在停止与回收\';next=\'等待进程退出与清理确认，请勿重复启动。\';tone=\'warn\';}\n    else if(state.error || [\'failed\',\'degraded\'].includes(state.phase)){headline=\'有模块需要处理\';next=\'仍可使用已就绪的能力；原因与处理提示在右侧日志。\';tone=\'error\';}\n    else if(ui.startup_stage===\'failed\'){headline=\'启动未完成\';next=\'原因在右侧日志；处理后可再次一键启动，已就绪的模块仍可查询。\';tone=\'error\';}\n    else if(ui.startup_stage===\'cancelled\'){headline=\'自动启动已取消\';next=\'查看各模块的真实状态；需要继续时可再次一键启动。\';}\n    else if(state.draining){headline=\'已暂停接收新任务\';next=\'已有队列继续执行；高级控制中可以恢复接收。\';}\n    else if(state.ready){headline=\'可以生成\';next=\'排队任务自动处理；API 可查询进度、失败原因和视频结果。\';tone=\'ready\';}\n    else if(state.api_ready && state.model_ready){headline=\'模型已就绪\';next=state.public_requested&&!state.public_api_ready?\'本机 API 已可处理任务；公网仍在准备，请以公网探测结果为准。\':\'任务自动处理；API 返回进度与视频结果。\';tone=\'ready\';}\n    else if(state.api_ready){headline=state.queue?.admission_open?\'可排队 · 等待模型\':\'API 可查询 · 等待模型\';next=state.queue?.admission_open?\'已接收任务保持排队；模型完成加载并通过就绪检查后自动处理。\':\'API 可查询；任务接收状态以当前后台为准。\';}\n    else if(ui.bootstrap_running){const stages={preflight:\'检查资源\',install:\'安装依赖\',start:\'启动后台\',awaiting_api:\'等待 API\',awaiting_model:\'等待模型\'};headline=\'一键启动 · \'+(stages[ui.startup_stage]||\'准备中\');next=\'正在自动推进；API 密钥可用后可导出，异常会在右侧日志显示。\';}\n    else if(ui.busy){headline=ui.operation || \'操作进行中\';next=\'详细过程只在右侧日志更新；不重复提交当前操作。\';}\n    else if(state.phase && state.phase!==\'new\'){headline=\'服务正在准备\';next=\'右侧日志持续更新；无需等待模型就绪才观察 API。\';}\n    else if(ui.install_ok){headline=\'依赖已就绪\';next=\'点击一键启动继续；需要分步维护时再展开高级控制。\';}\n    else if(ui.preflight_ok){headline=\'资源检查通过\';next=\'点击一键启动继续安装和启动，无需逐步点击。\';}\n    text(\'headline\',headline);text(\'next-step\',next);$(\'board\').dataset.tone=tone;\n    const module=(alive,ready)=>{\n      if(offline() || closed() || !Object.keys(state).length) return \'尚未读取\';\n      if(cleanStop) return \'已停止\';\n      if(stale) return \'待刷新\';\n      if(ready===true) return \'可用\';\n      if(alive===true) return ready===undefined?\'运行中\':\'准备中\';\n      return alive===false?\'已停止\':\'未确认\';\n    };\n    text(\'manager-state\',module(state.manager_alive));text(\'api-state\',module(state.server_alive,state.api_ready));text(\'model-state\',module(state.model_alive,state.model_ready));\n    text(\'public-state\', offline()||closed()||!Object.keys(state).length?\'尚未读取\':cleanStop?\'已关闭\':stale?\'待刷新\':!state.public_requested?\'未启用\':state.public_api_ready?\'探测通过\':state.tunnel_connected?\'等待探测\':\'准备中\');\n    const queue=state.queue||{};text(\'queue-state\',stale||offline()?\'排队 —\u3000执行 —\':`排队 ${queue.queued??\'—\'}\u3000执行 ${queue.running??\'—\'}`);\n    const gpu=resources.gpu?String(resources.gpu).replace(/^NVIDIA\\s+/,\'\'):\'A100（配置）\';\n    text(\'resources\',`${gpu} · 原生 BF16${resources.available_ram_gib!==undefined?\' · 可用内存 \'+resources.available_ram_gib+\' GiB（预检）\':\'\'}`);\n    text(\'board-stamp\',state.observed_at?\'观察 \'+nowText(state.observed_at):\'尚无服务观察\');\n    renderConnection();renderTransfers();\n  }\n  function connectionLabel() {\n    const value=closed()?\'连接已关闭\':offline()?\'离线占位\':delivering()?\'启动交付 · 宿主推送\':connectionLost?\'连接待确认\':paused?\'已暂停读取\':document.hidden?\'页面隐藏 · 暂停\':\'持续读取\';\n    text(\'connection-state\',value);$(\'live-dot\').classList.toggle(\'live\',!offline()&&!closed()&&!connectionLost&&!paused&&!document.hidden);\n    text(\'log-mode\',delivering()?\'宿主推送 · 停止本单元可取消\':paused?\'日志保持静止\':offline()||closed()?\'未自动读取\':\'每 2 秒读取\');\n  }\n  function clearNotice() { $(\'notice-zone\').replaceChildren(); shownNoticeId=null; }\n  function notice(value,buttons=[]) {\n    if(!value){clearNotice();return;}\n    clearNotice();\n    const card=document.createElement(\'section\');card.className=\'notice\';card.dataset.kind=value.kind||\'info\';\n    const heading=document.createElement(\'div\');heading.className=\'notice-title\';heading.textContent=safeText(value.title||\'提示\');\n    const message=document.createElement(\'div\');message.className=\'notice-message\';message.textContent=safeText(value.message||\'\').slice(0,3000);\n    card.append(heading,message);\n    const actions=document.createElement(\'div\');actions.className=\'notice-actions\';\n    for(const item of buttons){const button=document.createElement(\'button\');button.type=\'button\';button.textContent=item.label;if(item.primary)button.className=\'primary\';button.addEventListener(\'click\',item.click);actions.append(button);}\n    if(!buttons.length){const dismiss=document.createElement(\'button\');dismiss.type=\'button\';dismiss.textContent=\'收起\';dismiss.addEventListener(\'click\',()=>{if(value.id)knownNotices.add(String(value.id));localNotice=null;clearNotice();});actions.append(dismiss);}\n    card.append(actions);$(\'notice-zone\').append(card);shownNoticeId=value.id?String(value.id):null;\n    return {card,actions};\n  }\n  function showLocal(value){localNotice=value;notice(value);}\n  function renderLogs(logs) {\n    if(!logs || typeof logs.text!==\'string\')return;\n    const selectedSource=$(\'log-source\').value;\n    const source=logs.source||selectedSource;\n    if(source!==selectedSource)return;\n    lastLogs=logs;\n    const viewport=$(\'log-viewport\');const nearBottom=viewport.scrollHeight-viewport.scrollTop-viewport.clientHeight<38;\n    const scrollTop=viewport.scrollTop;const changedSource=source!==lastSource;\n    const fullText=safeText(logs.text).slice(-64000);\n    const tail=source===\'state\'?fullText:fullText.replace(/\\r?\\n$/, \'\').split(/\\r?\\n/).slice(-Number($(\'log-lines\').value)).join(\'\\n\');\n    const next=tail||\'该来源暂无日志。\';\n    if(next!==lastText || changedSource){$(\'log-text\').textContent=next;lastText=next;lastSource=source;\n      if($(\'follow-logs\').checked&&(nearBottom||changedSource))viewport.scrollTop=viewport.scrollHeight;\n      else viewport.scrollTop=scrollTop;\n    }\n    text(\'log-stamp\',`只读 · 已脱敏 · ${logs.observed_at?nowText(logs.observed_at):\'最新读取\'}`);\n  }\n  function apply(data) {\n    if(!data || typeof data!==\'object\')return;\n    for(const key of [\'state\',\'resources\',\'ui\',\'result\',\'connection\',\'downloads\'])if(data[key]!==undefined)current[key]=data[key];\n    renderBoard();renderLogs(data.logs);connectionLabel();setButtons();\n    if(data.notice&&!pendingConfirmation&&!localNotice){const id=String(data.notice.id||JSON.stringify(data.notice));if(!knownNotices.has(id)&&shownNoticeId!==id)notice({...data.notice,id});}\n    if(closed()){clearTimeout(timer);timer=null;}\n  }\n  async function rpc(action,data) {\n    if(offline()||closed())throw new Error(\'当前未连接到控制台后台。\');\n    if(delivering())throw new Error(\'启动交付观察期间由宿主推送状态；如需取消，请停止本单元。\');\n    const request=typeof window.H3_TRANSPORT===\'function\'?window.H3_TRANSPORT(action,data):window.google.colab.kernel.invokeFunction(CALLBACK_NAME,[action,data],{});\n    let timeout;\n    try {\n      const raw=await Promise.race([Promise.resolve(request),new Promise((_,reject)=>{timeout=setTimeout(()=>reject(new Error(\'连接超时，操作结果尚未确认。\')),20000);})]);\n      let value=raw?.data?.[\'application/json\']??raw;\n      if(typeof value===\'string\')value=JSON.parse(value);\n      if(!value||typeof value!==\'object\')throw new Error(\'后台没有返回可识别的结果。\');\n      if(value.accepted===false || value.ok===false || (value.error && ![\'snapshot\',\'refresh_board\'].includes(action)) || (value.error && !value.state)){\n        const error=new Error(typeof value.error===\'string\'?value.error:\'后台拒绝了本次操作。\');error.confirmed=true;throw error;\n      }\n      return value;\n    } finally {clearTimeout(timeout);}\n  }\n  async function requestSecret() {\n    const credentialId=current.connection?.credential_id, epoch=secretEpoch;\n    if(!credentialId||!current.connection?.key_available||connectionLost||document.hidden)throw new Error(\'当前密钥不可读取；先刷新连接状态。\');\n    if(keyVisible&&revealedKey&&credentialId===observedCredentialId)return revealedKey;\n    const data=await rpc(\'reveal_connection\',{credential_id:credentialId});\n    const secret=data.connection_secret;\n    if(epoch!==secretEpoch||credentialId!==current.connection?.credential_id||secret?.credential_id!==credentialId||offline()||closed()||connectionLost||document.hidden){\n      throw new Error(\'连接或页面状态已变化，密钥未显示；请重新读取。\');\n    }\n    if(typeof secret.api_key!==\'string\'||secret.api_key.length<32||secret.api_key.length>4096)throw new Error(\'没有收到有效的 API 密钥。\');\n    return secret.api_key;\n  }\n  async function toggleSecret() {\n    if(secretBusy)return;\n    if(keyVisible){clearSecret();renderConnection();setButtons();return;}\n    secretBusy=true;setButtons();\n    try {revealedKey=await requestSecret();keyVisible=true;renderConnection();}\n    catch {clearSecret();showLocal({kind:\'warning\',title:\'密钥未显示\',message:\'当前连接无法确认或密钥已变化。刷新看板后可重新显示；密钥不会写入日志。\'});}\n    finally{secretBusy=false;setButtons();}\n  }\n  async function copyConnection(kind) {\n    if(kind===\'key\'&&secretBusy)return;\n    let value=\'\';const button=kind===\'key\'?$(\'copy-key\'):$(\'copy-url\');\n    if(kind===\'key\'){secretBusy=true;setButtons();}\n    try {\n      value=kind===\'key\'?await requestSecret():usableUrl(current.connection?.url);\n      if(!value)throw new Error(\'No confirmed value\');\n      if(typeof navigator.clipboard?.writeText!==\'function\')throw new Error(\'Clipboard unavailable\');\n      await navigator.clipboard.writeText(value);\n      text(button.id,\'已复制\');setTimeout(()=>{if(!disposed)text(button.id,\'复制\');},1500);\n    } catch {\n      showLocal({kind:\'warning\',title:\'暂时无法复制\',message:kind===\'key\'?\'浏览器未允许复制或连接已变化。可先点击「显示」，再手动复制密钥。\':\'尚无已验证地址或浏览器未允许复制。确认 API 可用后，可从地址栏手动复制。\'});\n    } finally {value=\'\';if(kind===\'key\')secretBusy=false;setButtons();}\n  }\n  function schedule() {\n    clearTimeout(timer);timer=null;\n    if(disposed||offline()||closed()||paused||document.hidden)return;\n    if(delivering()){\n      const remaining=Number(current.ui.startup_delivery.deadline_at)*1000+5100-Date.now();\n      timer=setTimeout(()=>{connectionLabel();setButtons();readSnapshot(false);},Math.max(100,remaining));return;\n    }\n    timer=setTimeout(()=>readSnapshot(false),Math.min(10000,2000*Math.max(1,failures)));\n  }\n  async function readSnapshot(manual=false,action=\'snapshot\') {\n    if(polling||disposed||offline()||closed()||delivering())return;\n    if(!manual&&(paused||document.hidden))return;\n    const source=$(\'log-source\').value;\n    clearTimeout(timer);timer=null;polling=true;setButtons();\n    try {const data=await rpc(action,{source});connectionLost=false;failures=0;if(localNotice?.id===\'connection-lost\'){localNotice=null;clearNotice();}apply(data);}\n    catch(error){connectionLost=true;failures++;renderBoard();if(manual||failures===1)showLocal({id:\'connection-lost\',kind:\'warning\',title:\'暂时无法读取状态\',message:safeText(error.message)+\'\\n日志保留在原位；不会自动重发业务操作。\'});}\n    finally{polling=false;connectionLabel();setButtons();if(source!==$(\'log-source\').value)readSnapshot(true);else schedule();}\n  }\n  function offerDownload(download) {\n    if(!download || typeof download.data_base64!==\'string\')return;\n    if(download.data_base64.length>Math.ceil(32*1024*1024*4/3)+8)throw new Error(\'下载文件超过控制台的 32 MiB 上限。\');\n    const binary=atob(download.data_base64);const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);\n    const url=URL.createObjectURL(new Blob([bytes],{type:download.mime||\'application/octet-stream\'}));blobUrls.add(url);\n    localNotice={kind:\'info\',title:\'文件已准备\',message:\'点击保存文件。若为私密连接文件，请单独保管。\'};\n    const view=notice(localNotice);const link=document.createElement(\'a\');link.href=url;link.download=String(download.name||\'result.bin\').replace(/[\\\\/]/g,\'_\');link.textContent=\'保存文件\';view.actions.prepend(link);\n    link.addEventListener(\'click\',()=>{setTimeout(()=>{URL.revokeObjectURL(url);blobUrls.delete(url);},60000);});\n  }\n  async function dispatch(action,values) {\n    if(activeActions.has(action))return;\n    if(delivering()){showLocal({kind:\'info\',title:\'启动交付观察中\',message:\'当前单元仍在执行，后台保持并行准备。如需取消，请使用 Colab 本单元的停止按钮，程序会回收已启动进程。\'});return;}\n    if(offline()||closed()){showLocal({kind:\'info\',title:\'尚未连接\',message:\'连接运行时并运行控制台单元后再操作。\'});return;}\n    const fingerprint=action+\':\'+JSON.stringify(values);const requestId=uncertainRequests.get(fingerprint)||uuid();\n    activeActions.add(action);setButtons();localNotice=null;pendingConfirmation=null;clearNotice();\n    try {\n      const data=await rpc(action,{...values,request_id:requestId});uncertainRequests.delete(fingerprint);connectionLost=false;apply(data);\n      if(data.download)offerDownload(data.download);\n      else if(data.notice){localNotice=data.notice;notice(data.notice);}\n      // Ordinary accepted operations update the log and board without another notice card.\n    } catch(error) {\n      if(error.confirmed)uncertainRequests.delete(fingerprint);else uncertainRequests.set(fingerprint,requestId);\n      showLocal({kind:\'error\',title:error.confirmed?\'操作未完成\':\'操作结果尚未确认\',message:safeText(error.message)+(error.confirmed?\'\\n请根据日志处理后再操作。\':\'\\n未自动重发。先查看状态；再次提交相同操作时会沿用本次请求编号。\')});\n    } finally {activeActions.delete(action);setButtons();schedule();}\n  }\n  function requestAction(action,values) {\n    if(action===\'release\'||(action===\'control\'&&values.action===\'stop_model\'&&values.force)){\n      pendingConfirmation={action,values};localNotice=null;\n      notice({kind:\'warning\',title:action===\'release\'?\'释放当前云实例？\':\'强制停止模型？\',message:action===\'release\'?\'释放后，Colab 临时文件将丢失。请确认已保存结果。\':\'这会中断模型进程中的任务。排队取消不需要强制停止。\'},[\n        {label:\'取消\',click:()=>{pendingConfirmation=null;clearNotice();}},\n        {label:action===\'release\'?\'确认释放\':\'确认强制停止\',primary:true,click:()=>{const pending=pendingConfirmation;if(pending){pendingConfirmation=null;dispatch(pending.action,pending.values);}}}\n      ]);return;\n    }\n    dispatch(action,values);\n  }\n  for(const button of actionButtons)button.addEventListener(\'click\',()=>requestAction(button.dataset.action,payload()));\n  $(\'send-control\').addEventListener(\'click\',()=>requestAction(\'control\',{action:$(\'component-action\').value,force:$(\'force\').checked}));\n  $(\'query-command\').addEventListener(\'click\',()=>{const id=$(\'command_id\').value.trim();if(!id){showLocal({kind:\'info\',title:\'填写命令 ID\',message:\'在高级控制中填写需要查询的命令 ID。\'});return;}dispatch(\'control\',{action:\'query\',command_id:id});});\n  $(\'refresh-board\').addEventListener(\'click\',()=>readSnapshot(true,\'refresh_board\'));\n  $(\'toggle-key\').addEventListener(\'click\',toggleSecret);\n  $(\'copy-key\').addEventListener(\'click\',()=>copyConnection(\'key\'));\n  $(\'copy-url\').addEventListener(\'click\',()=>copyConnection(\'url\'));\n  $(\'refresh-logs\').addEventListener(\'click\',()=>readSnapshot(true));\n  $(\'log-source\').addEventListener(\'change\',()=>{\n    lastSource=\'\';\n    $(\'log-lines\').disabled=$(\'log-source\').value===\'state\';\n    readSnapshot(true);\n  });\n  $(\'log-lines\').addEventListener(\'change\',()=>{if(!offline())renderLogs(lastLogs);});\n  $(\'toggle-logs\').addEventListener(\'click\',()=>{paused=!paused;clearTimeout(timer);timer=null;text(\'toggle-logs\',paused?\'继续\':\'暂停\');connectionLabel();if(!paused)readSnapshot(true);});\n  $(\'follow-logs\').addEventListener(\'change\',()=>{if($(\'follow-logs\').checked)$(\'log-viewport\').scrollTop=$(\'log-viewport\').scrollHeight;});\n  document.addEventListener(\'visibilitychange\',()=>{clearTimeout(timer);timer=null;if(document.hidden)clearSecret();connectionLabel();setButtons();if(!document.hidden&&!paused)readSnapshot(false);});\n  function dispose(){disposed=true;clearSecret();clearTimeout(timer);timer=null;for(const url of blobUrls)URL.revokeObjectURL(url);blobUrls.clear();}\n  window.addEventListener(\'pagehide\',dispose,{once:true});window.addEventListener(\'beforeunload\',dispose,{once:true});\n  apply(current);if(offline())text(\'log-text\',\'尚未连接运行时，没有实时日志。\\n连接后，提示、确认和下载入口只在此区域出现。\');else schedule();\n})();\n</script>\n</body>\n</html>\n', 'download_progress.py': '"""Observed Hugging Face transfer progress, separate from model readiness.\n\nNo extra downloads, repository scans, or scheduling are introduced. An owned\nload temporarily supplies a progress-bar observer to HF\'s existing HTTP/Xet\ntransfer functions. Unsupported signatures degrade to unknown progress. The\ndownload algorithm, retries, validation, caching and worker counts stay in HF.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom copy import deepcopy\nimport functools\nimport hashlib\nimport importlib\nimport inspect\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport threading\nimport time\nimport uuid\n\nMAX_STATE_BYTES = 256 * 1024\nPHASES = {\'waiting\', \'downloading\', \'verifying\', \'loading\', \'loaded\', \'disabled\', \'failed\'}\n_ACTIVE = None\n_RECORDER = None\n_HOOK_LOCK = threading.RLock()\n\n\ndef _number(value):\n    return type(value) in (int, float) and math.isfinite(value) and value >= 0\n\n\ndef read_progress(path, expected_instance_id=None):\n    unavailable = {\'schema_version\': 1, \'model_instance_id\': expected_instance_id,\n                   \'observed_at\': None, \'observation_stale\': True, \'items\': {}}\n    try:\n        path = Path(path)\n        if path.stat().st_size > MAX_STATE_BYTES:\n            raise ValueError(\'oversized\')\n        with path.open(\'r\', encoding=\'utf-8\') as stream:\n            raw = stream.read(MAX_STATE_BYTES + 1)\n        if len(raw.encode(\'utf-8\')) > MAX_STATE_BYTES:\n            raise ValueError(\'oversized\')\n        data = json.loads(raw)\n        if data.get(\'schema_version\') != 1 or not isinstance(data.get(\'items\'), dict):\n            raise ValueError(\'invalid_schema\')\n        if expected_instance_id is not None and data.get(\'model_instance_id\') != expected_instance_id:\n            raise ValueError(\'generation_mismatch\')\n        if not _number(data.get(\'observed_at\')):\n            raise ValueError(\'invalid_timestamp\')\n        items = {}\n        for key in (\'model\', \'acceleration\', \'style\'):\n            item = data[\'items\'].get(key)\n            if item is None:\n                continue\n            if not isinstance(item, dict) or item.get(\'phase\') not in PHASES:\n                raise ValueError(\'invalid_item\')\n            clean = {\'phase\': item[\'phase\']}\n            for field in (\'bytes_completed\', \'bytes_total\', \'percent\', \'files_completed\', \'files_total\', \'active_files\'):\n                value = item.get(field)\n                if value is not None and not _number(value):\n                    raise ValueError(\'invalid_counter\')\n                if field == \'percent\' and value is not None and value > 100:\n                    raise ValueError(\'invalid_percent\')\n                clean[field] = value\n            for field in (\'scope\', \'detail\'):\n                clean[field] = str(item.get(field, \'\'))[:240]\n            clean[\'cache_hit\'] = item.get(\'cache_hit\') if type(item.get(\'cache_hit\')) is bool else None\n            items[key] = clean\n        return {\'schema_version\': 1, \'model_instance_id\': data.get(\'model_instance_id\'),\n                \'observed_at\': data[\'observed_at\'], \'observation_stale\': False, \'items\': items}\n    except (OSError, ValueError, TypeError, AttributeError) as error:\n        return dict(unavailable, error=str(error) if isinstance(error, ValueError) and str(error) in {\n            \'oversized\', \'invalid_schema\', \'generation_mismatch\', \'invalid_timestamp\', \'invalid_item\',\n            \'invalid_counter\', \'invalid_percent\'} else \'progress_unavailable\')\n\n\nclass DownloadProgress:\n    def __init__(self, path=None, *, instance_id=None, settings=None, interval=.5, clock=time.monotonic):\n        self.path = Path(path) if path else None\n        self.instance_id = instance_id\n        self.lock = threading.RLock()\n        self.interval, self.clock, self.last_write = interval, clock, -math.inf\n        self.category = \'model\'\n        self.transfers = {}\n        self.transfer_identities = {}\n        self.measurement_available = None\n        self.items = {}\n        self._phase_details = {}\n        for name in (\'model\', \'acceleration\', \'style\'):\n            enabled = name == \'model\' or (settings is not None and getattr(settings, name, \'none\') != \'none\')\n            self.items[name] = dict(phase=\'waiting\' if enabled else \'disabled\', bytes_completed=0,\n                bytes_total=None, percent=None, files_completed=0, files_total=None,\n                active_files=0, scope=\'observed_transfers\' if name == \'model\' else \'asset\',\n                detail=\'尚未开始\' if enabled else \'未启用\', cache_hit=None)\n        self._write(force=True)\n\n    def _document(self):\n        return {\'schema_version\': 1, \'model_instance_id\': self.instance_id,\n                \'observed_at\': time.time(), \'observation_stale\': False, \'items\': deepcopy(self.items)}\n\n    def snapshot(self):\n        with self.lock:\n            return self._document()\n\n    def _write(self, *, force=False, event=None):\n        if not self.path:\n            return\n        now = self.clock()\n        if not force and now - self.last_write < self.interval:\n            return\n        try:\n            self.path.parent.mkdir(parents=True, exist_ok=True)\n            temporary = self.path.with_name(self.path.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n            try:\n                temporary.write_text(json.dumps(self._document(), ensure_ascii=False), encoding=\'utf-8\')\n                os.replace(temporary, self.path)\n            finally:\n                temporary.unlink(missing_ok=True)\n            if event:\n                with self.path.with_suffix(\'.events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                    stream.write(json.dumps({\'observed_at\': time.time(), \'model_instance_id\': self.instance_id,\n                                             **event}, ensure_ascii=False) + \'\\n\')\n            self.last_write = now\n        except OSError:\n            # UI telemetry must never abort or retry an otherwise valid download.\n            pass\n\n    def phase(self, category, phase, detail=\'\', *, total=None, cache_hit=None):\n        if phase not in PHASES or category not in self.items:\n            raise ValueError(\'Invalid progress phase/category\')\n        with self.lock:\n            item = self.items[category]\n            if self.measurement_available is False and phase in {\'loading\', \'loaded\'}:\n                detail += \'；当前下载库未提供兼容进度回调\'\n            item.update(phase=phase, detail=detail[:240])\n            self._phase_details[category] = (phase, detail[:240])\n            if total is not None and category != \'model\':\n                item.update(bytes_total=int(total), files_total=1)\n            if cache_hit is not None:\n                item[\'cache_hit\'] = bool(cache_hit)\n            if phase in {\'verifying\', \'loaded\'} and total is not None and category != \'model\':\n                item.update(bytes_completed=int(total), percent=100, files_completed=1)\n            self._write(force=True, event={\'type\': \'phase\', \'category\': category, \'phase\': phase})\n\n    def begin_transfer(self, category, total=None, initial=0, identity=None):\n        with self.lock:\n            identity = (category, identity) if identity is not None else None\n            transfer = self.transfer_identities.get(identity) if identity is not None else None\n            if transfer is None:\n                transfer = uuid.uuid4().hex\n                self.transfers[transfer] = {\'category\': category, \'n\': max(0, initial),\n                                           \'total\': total, \'finished\': False, \'active\': 0}\n                if identity is not None:\n                    self.transfer_identities[identity] = transfer\n            entry = self.transfers[transfer]\n            entry.update(n=max(entry[\'n\'], initial), finished=False, active=entry[\'active\'] + 1)\n            self._update_item(category)\n            self._write(force=True, event={\'type\': \'transfer.started\', \'category\': category,\n                                          \'transfer_id\': transfer, \'total\': total, \'initial\': initial})\n        return transfer\n\n    def update_transfer(self, transfer, completed):\n        with self.lock:\n            entry = self.transfers[transfer]\n            # Callback retries cannot decrease or double-count a transfer\'s absolute progress.\n            entry[\'n\'] = max(entry[\'n\'], completed)\n            self._update_item(entry[\'category\'])\n            self._write()\n\n    def finish_transfer(self, transfer, *, success):\n        with self.lock:\n            entry = self.transfers[transfer]\n            entry[\'active\'] = max(0, entry[\'active\'] - 1)\n            entry.update(finished=entry[\'active\'] == 0, success=success)\n            self._update_item(entry[\'category\'])\n            self._write(force=True, event={\'type\': \'transfer.finished\', \'category\': entry[\'category\'],\n                                          \'transfer_id\': transfer, \'success\': success})\n\n    def _update_item(self, category):\n        entries = [e for e in self.transfers.values() if e[\'category\'] == category]\n        item = self.items[category]\n        item[\'active_files\'] = sum(not e[\'finished\'] for e in entries)\n        item[\'files_completed\'] = sum(e.get(\'success\') is True and e[\'finished\'] for e in entries)\n        value = sum(e[\'n\'] for e in entries)\n        item[\'bytes_completed\'] = max(item[\'bytes_completed\'], value)\n        if category != \'model\' and item[\'bytes_total\']:\n            item[\'bytes_completed\'] = min(item[\'bytes_total\'], item[\'bytes_completed\'])\n            item[\'percent\'] = round(100 * item[\'bytes_completed\'] / item[\'bytes_total\'], 1)\n        if item[\'active_files\']:\n            item.update(phase=\'downloading\', detail=\'已观察文件量（含续传与重建）；整模型总量未知\' if category == \'model\'\n                        else \'正在下载/重建已选 LoRA 文件\')\n        elif item[\'phase\'] == \'downloading\':\n            phase, detail = self._phase_details.get(category, (\'loading\', \'继续加载；下载完成不代表模型就绪\'))\n            item.update(phase=phase, detail=detail)\n\n\nclass _ProgressBar:\n    """Protocol proxy used by HF HTTP and Xet reconstruction callbacks."""\n    def __init__(self, recorder, transfer, *, initial=0, total=None, delegate=None):\n        self.recorder, self.transfer, self.delegate = recorder, transfer, delegate\n        self.n, self.total = initial, total\n        self.disable = False\n        self.lock = threading.RLock()\n\n    def update(self, amount=1):\n        with self.lock:\n            self.n = max(0, self.n + (amount or 0))\n            try:\n                self.recorder.update_transfer(self.transfer, self.n)\n            except Exception:\n                pass  # Observation cannot fail an otherwise valid payload transfer.\n            if self.delegate is not None:\n                return self.delegate.update(amount)\n\n    def __getattr__(self, name):\n        if self.delegate is not None:\n            return getattr(self.delegate, name)\n        if name == \'format_dict\':\n            return {\'n\': self.n, \'total\': self.total, \'rate\': None}\n        if name in {\'refresh\', \'set_description\', \'set_description_str\', \'set_postfix_str\', \'close\'}:\n            return lambda *a, **kw: None\n        raise AttributeError(name)\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *args):\n        return False\n\n\ndef _wrap_transfer(original, recorder):\n    try:\n        signature = inspect.signature(original)\n    except (ValueError, TypeError):\n        return original\n    if \'_tqdm_bar\' not in signature.parameters:\n        return original\n\n    @functools.wraps(original)\n    def observed(*args, **kwargs):\n        bound = signature.bind_partial(*args, **kwargs)\n        existing = bound.arguments.get(\'_tqdm_bar\')\n        if isinstance(existing, _ProgressBar):\n            return original(*args, **kwargs)  # HF\'s nested HTTP retry owns the same byte counter.\n        total, initial = bound.arguments.get(\'expected_size\'), bound.arguments.get(\'resume_size\', 0)\n        total = int(total) if _number(total) else None\n        initial = int(initial) if _number(initial) else 0\n        # Hash an owned temporary-file identity, never persist a signed URL or local path.\n        path = bound.arguments.get(\'incomplete_path\') or bound.arguments.get(\'downloaded_file\')\n        if path is None:\n            path = getattr(bound.arguments.get(\'temp_file\'), \'name\', None)\n        identity = hashlib.sha256(os.fsencode(path)).hexdigest() if isinstance(path, (str, bytes, os.PathLike)) else None\n        transfer = recorder.begin_transfer(recorder.category, total, initial, identity=identity)\n        bound.arguments[\'_tqdm_bar\'] = _ProgressBar(recorder, transfer, initial=initial, total=total, delegate=existing)\n        try:\n            result = original(*bound.args, **bound.kwargs)\n        except BaseException:\n            recorder.finish_transfer(transfer, success=False)\n            raise\n        recorder.finish_transfer(transfer, success=True)\n        return result\n    return observed\n\n\n@contextmanager\ndef observe_hub_downloads(recorder):\n    """A model process owns these hooks; no thread pool or concurrency limit changes."""\n    global _ACTIVE\n    with _HOOK_LOCK:\n        if _ACTIVE is recorder:\n            yield recorder\n            return\n        previous, patched = _ACTIVE, []\n        _ACTIVE = recorder\n        try:\n            try:\n                module = importlib.import_module(\'huggingface_hub.file_download\')\n                for name in (\'http_get\', \'xet_get\'):\n                    original = getattr(module, name, None)\n                    if callable(original):\n                        wrapper = _wrap_transfer(original, recorder)\n                        if wrapper is not original:\n                            setattr(module, name, wrapper)\n                            patched.append((module, name, original, wrapper))\n            except (ImportError, AttributeError, TypeError, ValueError):\n                pass\n            recorder.measurement_available = bool(patched)\n            yield recorder\n        finally:\n            for module, name, original, wrapper in reversed(patched):\n                if getattr(module, name) is wrapper:\n                    setattr(module, name, original)\n            _ACTIVE = previous\n\n\ndef current_progress():\n    return _ACTIVE or _RECORDER\n\n\n@contextmanager\ndef adapter_progress(category, spec):\n    recorder = current_progress()\n    if recorder is None:\n        yield None\n        return\n    with observe_hub_downloads(recorder):\n        previous = recorder.category\n        recorder.category = category\n        try:\n            recorder.phase(category, \'loading\', \'准备所选 LoRA\', total=spec.size)\n            yield recorder\n        except BaseException:\n            recorder.phase(category, \'failed\', \'LoRA 下载、校验或加载失败；详见模型日志\')\n            raise\n        finally:\n            recorder.category = previous\n\n\ndef tracked_model_load(function):\n    @functools.wraps(function)\n    def tracked(self, *args, **kwargs):\n        global _RECORDER\n        path = os.environ.get(\'H3_DOWNLOAD_PROGRESS_PATH\')\n        if not path:\n            return function(self, *args, **kwargs)\n        recorder = DownloadProgress(path, instance_id=os.environ.get(\'H3_MODEL_INSTANCE_ID\'),\n                                    settings=getattr(self, \'adapter_settings\', None))\n        _RECORDER = recorder\n        with observe_hub_downloads(recorder):\n            recorder.phase(\'model\', \'loading\', \'检查缓存并加载所需组件；整模型总量未知\')\n            try:\n                result = function(self, *args, **kwargs)\n            except BaseException:\n                recorder.phase(\'model\', \'failed\', \'下载或加载失败；详见模型日志\')\n                raise\n            no_transfers = not any(e[\'category\'] == \'model\' for e in recorder.transfers.values())\n            recorder.phase(\'model\', \'loaded\', \'组件加载完成；等待容器就绪检查\' +\n                           (\'。未观察到传输：缓存命中或当前库未提供回调\' if no_transfers else \'\'))\n            return result\n    return tracked\n\n\ndef tracked_adapter_application(function):\n    @functools.wraps(function)\n    def tracked(pipe, settings, *args, **kwargs):\n        try:\n            return function(pipe, settings, *args, **kwargs)\n        except BaseException:\n            recorder = current_progress()\n            if recorder:\n                for category in (\'acceleration\', \'style\'):\n                    if getattr(settings, category, \'none\') != \'none\':\n                        recorder.phase(category, \'failed\', \'LoRA 下载、验证或组合加载失败；详见模型日志\')\n            raise\n    return tracked\n', 'generation_options.py': '"""Auditable per-job BF16 choices; no arbitrary kernels or adapter downloads.\n\nConfiguration permits queue admission before readiness. Only the model\'s GPU\nlease may apply these choices; CPU preparation only validates immutable data.\n"""\nfrom __future__ import annotations\n\nfrom adapter_config import AdapterSettings, TURBO, load_adapter_settings, validate_steps\nfrom runtime_options import RuntimeOptions\n\nATTENTION_BACKENDS = ("auto", "native", "_native_flash")\nACCELERATION_LORAS = ("none", TURBO.key)\n\n\ndef requested_options(parameters):\n    """The same defaults apply at public API, queue, and model boundaries."""\n    return {\n        "attention_backend": parameters.get("attention_backend", "auto"),\n        "acceleration_lora": parameters.get("acceleration_lora", "none"),\n        "steps": parameters["steps"],\n    }\n\n\ndef validate_generation_options(requested, settings):\n    if requested["attention_backend"] not in ATTENTION_BACKENDS:\n        raise ValueError("attention_backend must be auto, native or _native_flash; quantized/external attention kernels are unsupported")\n    selected = requested["acceleration_lora"]\n    if selected not in ACCELERATION_LORAS:\n        raise ValueError("acceleration_lora must be none or lightx2v-ref2va-4step; arbitrary adapters are unsupported")\n    steps = requested["steps"]\n    if isinstance(steps, bool) or not isinstance(steps, int) or not 2 <= steps <= 50:\n        raise ValueError("steps must be an integer between 2 and 50")\n    if selected != "none" and selected != settings.acceleration:\n        raise ValueError("Requested acceleration LoRA is not configured for preloading; restart with H3_ACCELERATION_LORA=lightx2v-ref2va-4step")\n    validate_steps(AdapterSettings(acceleration=selected), steps)\n    return requested\n\n\ndef effective_options(requested, settings, runtime_options):\n    attention = requested["attention_backend"]\n    if attention == "auto":\n        attention = runtime_options.attention_backend\n    # Explicitly restore PyTorch dispatch after a previous forced-Flash job.\n    if attention == "auto":\n        attention = "native"\n    return {**requested, "attention_backend": attention, "attention_scope": "transformer_ref",\n            "precision": "bf16", "denoiser_evaluations": requested["steps"] - 1,\n            "style_lora": settings.style, "style_scale": settings.style_scale}\n\n\ndef generation_capabilities(*, model_ready=False, settings=None, runtime_options=None):\n    settings = settings or load_adapter_settings()\n    runtime_options = runtime_options or RuntimeOptions.from_env()\n    default_attention = runtime_options.attention_backend\n    attention = []\n    for identity, label in (("auto", "自动 · PyTorch SDPA"), ("native", "PyTorch SDPA"),\n                            ("_native_flash", "PyTorch Flash SDPA")):\n        value = {"id": identity, "label": label, "supported": True, "configured": True,\n                 "ready": bool(model_ready), "scope": "transformer_ref", "precision": "bf16"}\n        if identity == "auto":\n            value["resolves_to"] = "native" if default_attention == "auto" else default_attention\n        if identity == "_native_flash":\n            value["reason"] = "强制 PyTorch Flash SDPA；内核不兼容时明确失败，不静默回退"\n        attention.append(value)\n    configured = settings.acceleration == TURBO.key\n    return {"schema_version": 1, "workflow": "ref2va", "precision": "bf16",\n            "model_ready": bool(model_ready),\n            "defaults": {"attention_backend": "auto", "acceleration_lora": "none", "steps": 50},\n            "options": {"attention_backends": attention, "acceleration_loras": [\n                {"id": "none", "label": "关闭", "supported": True, "configured": True,\n                 "ready": bool(model_ready), "steps": {"min": 2, "max": 50, "default": 50}},\n                {"id": TURBO.key, "label": "LightX2V Ref2VA Turbo · 4 次评估", "supported": True,\n                 "configured": configured, "ready": bool(configured and model_ready),\n                 "steps": {"min": 5, "max": 5, "required": 5, "denoiser_evaluations": 4},\n                 "reason": "已配置预加载；模型就绪前可排队" if configured else "未在启动配置中预加载",\n                 "repository": TURBO.repository, "revision": TURBO.revision, "sha256": TURBO.sha256,\n                 "precision": "bf16", "validation": "GPU acceptance is reported separately; configuration is not a speed claim"},\n            ]},\n            "queue_semantics": "supported && configured permits admission before model_ready; readiness never means completion",\n            "step_semantics": "steps counts sigma grid points; actual denoiser evaluations = steps - 1",\n            "style_lora": {"id": settings.style, "scale": settings.style_scale, "control": "deployment"},\n            "quantization": False}\n', 'lora_preflight.py': '"""Fresh-process LoRA dependency check before model downloads or GPU loading.\n\nPEFT 0.21.2 visits its optional torchao dispatcher even for ordinary BF16\nLinear layers. An installed torchao < 0.16 raises before the normal dispatcher.\nThe dedicated Colab runtime may explicitly remove that unused optional package;\nthis helper never installs or upgrades packages, or changes the PyTorch stack.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom importlib import metadata\nimport importlib.util\nimport json\nimport subprocess\nimport sys\n\nfrom packaging.version import Version\n\n\nSTACK_PACKAGES = ("torch", "torchvision", "torchaudio", "psutil")\n\n\ndef distribution_version(name):\n    try:\n        return metadata.version(name)\n    except metadata.PackageNotFoundError:\n        return None\n\n\ndef prepare_optional_torchao(*, repair=False):\n    """Run before importing torch/PEFT; only the explicit old optional package changes."""\n    before = {name: distribution_version(name) for name in STACK_PACKAGES}\n    old_version = distribution_version("torchao")\n    removed = None\n    if old_version is not None and Version(old_version) < Version("0.16.0"):\n        if not repair:\n            raise RuntimeError(\n                f"Unused torchao {old_version} blocks PEFT BF16 LoRA dispatch. "\n                "In this dedicated runtime run lora_preflight.py --repair-optional-torchao; "\n                "do not upgrade the existing PyTorch stack."\n            )\n        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],\n                       check=True, capture_output=True, text=True)\n        importlib.invalidate_caches()\n        removed = old_version\n    after = {name: distribution_version(name) for name in STACK_PACKAGES}\n    if before != after:\n        raise RuntimeError("LoRA optional dependency preparation changed the protected PyTorch/psutil stack")\n    current = distribution_version("torchao")\n    if removed is not None and (current is not None or importlib.util.find_spec("torchao") is not None):\n        raise RuntimeError("torchao remains importable after uninstall; use a clean model process before loading LoRA")\n    return {"optional_torchao": {"before": old_version, "after": current, "removed_version": removed},\n            "stack_before": before, "stack_after": after, "stack_unchanged": True}\n\n\ndef bf16_lora_smoke():\n    """Exercise PEFT\'s actual dispatch and injection, which import-only checks miss.\n\n    Tiny CPU tensors avoid GPU initialization, large model downloads, and changes\n    to the model container. Native BF16 GPU support is checked by normal preflight.\n    """\n    import torch\n    import peft\n    from peft import LoraConfig, get_peft_model\n    from peft.import_utils import is_torchao_available\n\n    base = torch.nn.Sequential(torch.nn.Linear(4, 4, bias=False, dtype=torch.bfloat16))\n    model = get_peft_model(base, LoraConfig(r=2, lora_alpha=2, target_modules=["0"]),\n                           autocast_adapter_dtype=False)\n    model.requires_grad_(False)\n    model.eval()\n    value = torch.ones((1, 4), dtype=torch.bfloat16)\n    with torch.no_grad():\n        enabled = model(value)\n        with model.disable_adapter():\n            disabled = model(value)\n        restored = model(value)\n    if any(parameter.dtype != torch.bfloat16 for parameter in model.parameters()):\n        raise RuntimeError("Tiny LoRA injection changed BF16 parameter precision")\n    if any(output.dtype != torch.bfloat16 or not torch.isfinite(output).all().item()\n           for output in (enabled, disabled, restored)):\n        raise RuntimeError("Tiny BF16 LoRA enable/disable/restore forward check failed")\n    if not any("lora_A" in name for name, _ in model.named_parameters()):\n        raise RuntimeError("Tiny LoRA test did not inject an adapter")\n    return {"peft": peft.__version__, "torch": torch.__version__,\n            "torchao_available": is_torchao_available(), "device": "cpu", "precision": "bf16",\n            "injected": True, "enable_disable_restore_forward": True,\n            "parameter_elements": sum(parameter.numel() for parameter in model.parameters()),\n            "real_model_loaded": False, "gpu_generation_verified": False}\n\n\ndef run(*, repair=False):\n    report = prepare_optional_torchao(repair=repair)\n    report["smoke"] = bf16_lora_smoke()\n    report["status"] = "passed"\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repair-optional-torchao", action="store_true")\n    args = parser.parse_args()\n    print(json.dumps(run(repair=args.repair_optional_torchao), ensure_ascii=False))\n', 'condition_guard.py': '"""Bounded, side-effect-free four-valued admission decisions.\n\nThe caller owns observation identity, freshness and the lock that makes a decision\nactionable. This helper performs no I/O and acquires no locks. Its timestamp is\nthe decision time, not a claim that an external observation was refreshed then.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nimport uuid\n\nSCHEMA_VERSION = "1.0"\nGATE_VERSION = "1.0"\nMAX_CONDITIONS = 64\nMAX_WITNESSES = 64\nMAX_CONTEXT_BYTES = 8192\n\n\ndef observation_state(value):\n    """Only native booleans assert facts; missing witnesses do not negate facts."""\n    if type(value) is bool:\n        return "true" if value else "false"\n    if type(value) is str and value in {"unknown", "conflict"}:\n        return value\n    if type(value) is list:\n        if len(value) > MAX_WITNESSES:\n            return "unknown"\n        # Lists are one level only: nested structures cannot introduce assertions.\n        states = {observation_state(item) if type(item) is not list else "unknown"\n                  for item in value}\n        if "conflict" in states or {"true", "false"} <= states:\n            return "conflict"\n        if "true" in states:\n            return "true"\n        if "false" in states:\n            return "false"\n    return "unknown"\n\n\ndef negate_observation(value):\n    """Negate known booleans without turning unknown/conflict into a true fact."""\n    state = observation_state(value)\n    return {"true": False, "false": True, "unknown": None, "conflict": "conflict"}[state]\n\n\ndef _context_copy(context):\n    if type(context) is not dict:\n        raise ValueError("Gate context must be a JSON object")\n    def check(value, depth=0):\n        if depth > 5:\n            raise ValueError("Gate context is too deeply nested")\n        if value is None or type(value) in {bool, int, str}:\n            return\n        if type(value) is float and math.isfinite(value):\n            return\n        if type(value) in {list, dict} and len(value) <= 64:\n            if type(value) is dict:\n                if any(type(key) is not str for key in value):\n                    raise ValueError("Gate context keys must be strings")\n                value = value.values()\n            for item in value:\n                check(item, depth + 1)\n            return\n        raise ValueError("Gate context must contain bounded finite JSON values")\n    check(context)\n    encoded = json.dumps(context, allow_nan=False, separators=(",", ":"), ensure_ascii=True)\n    if len(encoded.encode("utf-8")) > MAX_CONTEXT_BYTES:\n        raise ValueError("Gate context exceeds the receipt budget")\n    return json.loads(encoded)\n\n\ndef evaluate_gate(gate_id, required, observations, context):\n    """Return a decision receipt; malformed declarations fail before admission.\n\nExplicit false facts differ from unknown and contradictory facts. In a witness\nlist, a native boolean supplies evidence even if another witness is missing;\nopposite boolean witnesses produce conflict. No string \'true\' or numeric 1 is a\ntrue assertion. Inputs outside the declared required set cannot enable a gate.\n"""\n    if type(gate_id) is not str or not 1 <= len(gate_id) <= 128:\n        raise ValueError("Gate identity must be a bounded nonempty string")\n    if (type(required) not in {list, tuple} or not 1 <= len(required) <= MAX_CONDITIONS\n            or any(type(name) is not str or not 1 <= len(name) <= 128 for name in required)\n            or len(required) != len(set(required))):\n        raise ValueError("Gate requires a bounded, nonempty set of unique condition names")\n    if type(observations) is not dict:\n        raise ValueError("Gate observations must be a mapping")\n    context = _context_copy(context)\n    conditions = {name: observation_state(observations.get(name)) for name in required}\n    blocked = [name for name, state in conditions.items() if state != "true"]\n    return {"schema_version": SCHEMA_VERSION, "gate_version": GATE_VERSION,\n            "gate_id": gate_id, "decision_id": uuid.uuid4().hex,\n            "observed_at": time.time(), "monotonic_ns": time.monotonic_ns(),\n            "context": context, "conditions": conditions,\n            "allowed": not blocked, "blocked": blocked}\n'}
CONSOLE_PROGRAMS = {'preflight': '\x23@title 资源与驻留方案\n\x23@markdown 只读取硬件与空间；不下载权重，不开启公网。默认A100 80GB常驻；40GB需显式选择分阶段。\nif not globals().get("dashboard"):\n    raise RuntimeError("先运行顶部的刷新看板，再执行资源检查。")\nimport os, sys\nimport contextlib, io\nfrom pathlib import Path\nPROJECT_DIR = Path("/content/minimax-h3-ref-api")\nPROJECT_DIR.mkdir(parents=True, exist_ok=True)\n使用40GB分阶段方案 = False \x23@param {type:"boolean"}\nSTAGED_BF16 = 使用40GB分阶段方案\n权重已完整缓存 = False \x23@param {type:"boolean"}\nMODEL_ALREADY_CACHED = 权重已完整缓存\nRESIDENCY = "staged" if STAGED_BF16 else "resident"\nos.environ["H3_MODE"] = "bf16"\nos.environ["H3_SKIP_DISK_CHECK"] = "1" if MODEL_ALREADY_CACHED else "0"\nPREFLIGHT_OK = False\n(PROJECT_DIR / "preflight.py").write_text(\'"""Conservative project resource checks; no downloads and no model imports."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport stat\\nimport subprocess\\n\\nGIB = 1024 ** 3\\nFIRST_DOWNLOAD_BYTES = 140 * GIB\\nMODEL_REVISION = "0919fb684102ef115126612f3fbb84d1027e7317"\\n_WEIGHT_COMPONENTS = {"text_encoder": "model", "transformer_ref": "diffusion_pytorch_model",\\n                      "vae": "diffusion_pytorch_model", "audio_vae": "diffusion_pytorch_model"}\\n_METADATA_LIMIT = 16 * 1024 ** 2\\n\\n\\ndef _hub_cache_path() -> Path:\\n    default_home = Path(os.environ.get("XDG_CACHE_HOME", "~/.cache")) / "huggingface"\\n    home = Path(os.environ.get("HF_HOME", str(default_home))).expanduser()\\n    return Path(os.environ.get("HF_HUB_CACHE", os.environ.get("HUGGINGFACE_HUB_CACHE", str(home / "hub")))).expanduser()\\n\\n\\ndef _cache_file(path, snapshot, blobs):\\n    """Accept snapshot files and completed repo/shared content-addressed blobs."""\\n    if path.parent.resolve(strict=True) != path.parent:\\n        raise ValueError("untrusted cache component directory redirection")\\n    resolved = path.resolve(strict=True)\\n    # Newer Hub caches share immutable SHA-256 blobs across repositories. Only\\n    # an exact snapshot reference may reach this pool; never scan/credit it.\\n    shared_pool = blobs.parent.parent / "blobs"\\n    shared_bucket = shared_pool / resolved.name[:2]\\n    shared_blob = (re.fullmatch(r"[0-9a-f]{64}", resolved.name) is not None\\n                   and resolved.parent == shared_bucket\\n                   and shared_pool.resolve(strict=True) == shared_pool\\n                   and shared_bucket.resolve(strict=True) == shared_bucket)\\n    if not (resolved.is_relative_to(snapshot) or\\n            (resolved.parent == blobs and re.fullmatch(r"[0-9a-f]{40}|[0-9a-f]{64}", resolved.name))\\n            or shared_blob):\\n        raise ValueError("cache file escapes its snapshot/repository blobs")\\n    info = resolved.stat()\\n    if not stat.S_ISREG(info.st_mode) or info.st_size <= 0 or ".incomplete" in resolved.name:\\n        raise ValueError("cache file is not a completed regular file")\\n    return resolved, info\\n\\n\\ndef _small_json(path, snapshot, blobs):\\n    resolved, info = _cache_file(path, snapshot, blobs)\\n    if info.st_size > _METADATA_LIMIT:\\n        raise ValueError("cache metadata exceeds inspection limit")\\n    return json.loads(resolved.read_bytes())\\n\\n\\ndef _weight_metadata(path, snapshot, blobs):\\n    """Check only the bounded safetensors header and file size, not weight data."""\\n    resolved, before = _cache_file(path, snapshot, blobs)\\n    with resolved.open("rb") as source:\\n        prefix = source.read(8)\\n        if len(prefix) != 8:\\n            raise ValueError("incomplete safetensors header")\\n        length = int.from_bytes(prefix, "little")\\n        if not 2 <= length <= _METADATA_LIMIT or length + 8 > before.st_size:\\n            raise ValueError("invalid safetensors header size")\\n        header = json.loads(source.read(length))\\n    if not isinstance(header, dict):\\n        raise ValueError("invalid safetensors header")\\n    ranges, names = [], set()\\n    sizes = {"BOOL": 1, "U8": 1, "I8": 1, "U16": 2, "I16": 2, "F16": 2, "BF16": 2,\\n             "U32": 4, "I32": 4, "F32": 4, "U64": 8, "I64": 8, "F64": 8,\\n             "F8_E4M3": 1, "F8_E5M2": 1}\\n    for name, entry in header.items():\\n        if name == "__metadata__":\\n            continue\\n        start, end = entry["data_offsets"]\\n        shape, element_bytes = entry["shape"], sizes[entry["dtype"]]\\n        if not isinstance(shape, list) or len(shape) > 16 or any(type(n) is not int or n < 0 for n in shape):\\n            raise ValueError("invalid tensor shape")\\n        count = 1\\n        for dimension in shape:\\n            count *= dimension\\n        if type(start) is not int or type(end) is not int or start < 0 or end - start != count * element_bytes:\\n            raise ValueError("invalid tensor data range")\\n        ranges.append((start, end))\\n        names.add(name)\\n    cursor = 0\\n    for start, end in sorted(ranges):\\n        if start != cursor:\\n            raise ValueError("tensor data ranges are incomplete or overlap")\\n        cursor = end\\n    after = resolved.stat()\\n    if not names or cursor <= 0 or 8 + length + cursor != before.st_size or (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\\n        raise ValueError("weight file is truncated, changing, or has trailing data")\\n    identity = (after.st_dev, after.st_ino) if after.st_ino else str(resolved)\\n    return names, cursor, after.st_size, identity\\n\\n\\ndef cached_weight_budget(cache: Path) -> dict:\\n    """Credit complete weight components, not readiness or model integrity."""\\n    revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\\n    result = {"model_revision": revision, "reusable_bytes": 0, "verified_components": [],\\n              "observation": "no_verified_weights", "scope": "disk_budget_only_not_model_readiness"}\\n    if not re.fullmatch(r"[0-9a-f]{40}", revision):\\n        result["observation"] = "revision_not_an_exact_commit"\\n        return result\\n    try:\\n        root = cache.resolve(strict=True)\\n        repo = root / "models--MiniMaxAI--MiniMax-H3"\\n        snapshot = repo / "snapshots" / revision\\n        blobs = repo / "blobs"\\n        if not snapshot.is_dir():\\n            return result\\n        # Parent directory symlinks may otherwise redirect an exact snapshot\\n        # name into a different revision/repository or an unrelated filesystem.\\n        for path in (repo, snapshot.parent, snapshot):\\n            if path.resolve(strict=True) != path:\\n                raise ValueError("untrusted cache directory redirection")\\n        credited = {}\\n        for component, stem in _WEIGHT_COMPONENTS.items():\\n            directory = snapshot / component\\n            single = directory / (stem + ".safetensors")\\n            index = directory / (stem + ".safetensors.index.json")\\n            if single.exists():\\n                files, weight_map, total_size = [single.name], None, None\\n            elif index.exists():\\n                data = _small_json(index, snapshot, blobs)\\n                weight_map = data["weight_map"]\\n                if not isinstance(weight_map, dict) or not weight_map:\\n                    raise ValueError("invalid weight index")\\n                files = sorted(set(weight_map.values()))\\n                total_size = data.get("metadata", {}).get("total_size")\\n                if total_size is not None and (type(total_size) is not int or total_size <= 0):\\n                    raise ValueError("invalid indexed weight size")\\n            else:\\n                continue\\n            if any(not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_-]+\\\\.safetensors", name) for name in files):\\n                raise ValueError("untrusted shard filename")\\n            if any(not (directory / name).exists() for name in files):\\n                continue  # partially downloaded components receive no credit\\n            component_files, data_bytes = {}, 0\\n            for filename in files:\\n                names, size, disk_bytes, identity = _weight_metadata(directory / filename, snapshot, blobs)\\n                if weight_map is not None and names != {key for key, value in weight_map.items() if value == filename}:\\n                    raise ValueError("weight index does not match shard tensors")\\n                component_files[identity] = disk_bytes\\n                data_bytes += size\\n            if total_size is not None and data_bytes != total_size:\\n                raise ValueError("indexed weight bytes are incomplete")\\n            credited.update(component_files)\\n            result["verified_components"].append(component)\\n        result["reusable_bytes"] = min(sum(credited.values()), FIRST_DOWNLOAD_BYTES)\\n        if credited:\\n            result["observation"] = "completed_weight_metadata_verified"\\n    except FileNotFoundError:\\n        # Cache creation/deletion races and missing roots never relax the gate.\\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_missing_or_changed")\\n    except (OSError, ValueError, TypeError, KeyError, AttributeError, OverflowError, RuntimeError) as error:\\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_unverified",\\n                      error_type=type(error).__name__)\\n    return result\\n\\n\\ndef check_resources(mode: str = "bf16", residency: str = "resident") -> dict:\\n    if mode != "bf16":\\n        raise ValueError("This project requires native BF16 on an NVIDIA A100; reduced-precision modes are disabled.")\\n    if residency not in {"resident", "staged"}:\\n        raise ValueError("residency must be resident or staged")\\n    if not Path("/proc/meminfo").is_file():\\n        raise RuntimeError("Deploy this project on a Linux NVIDIA GPU machine.")\\n    memory = {}\\n    for line in Path("/proc/meminfo").read_text().splitlines():\\n        name, value = line.split(":", 1)\\n        memory[name] = int(value.strip().split()[0]) * 1024\\n    try:\\n        result = subprocess.run(\\n            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],\\n            text=True, capture_output=True, check=True, timeout=20,\\n        )\\n        first = result.stdout.strip().splitlines()[0]\\n        gpu, memory_mib = first.rsplit(",", 1)\\n        vram_gib = float(memory_mib.strip()) / 1024\\n    except (OSError, ValueError, IndexError, subprocess.SubprocessError) as exc:\\n        raise RuntimeError("No working NVIDIA GPU detected. Select a GPU runtime first.") from exc\\n    # The first visible CUDA GPU is used. Report conservatively when device masks are present.\\n    visible = os.environ.get("CUDA_VISIBLE_DEVICES")\\n    if visible and visible not in {"0", "0,"}:\\n        raise RuntimeError("Preflight expects the first NVIDIA GPU. Unset CUDA_VISIBLE_DEVICES or set it to 0.")\\n    cache = _hub_cache_path()\\n    disk_path = cache\\n    while not disk_path.exists():\\n        disk_path = disk_path.parent\\n    free_disk_bytes = shutil.disk_usage(disk_path).free\\n    free_disk = free_disk_bytes / GIB\\n    cache_budget = cached_weight_budget(cache)\\n    remaining_disk_bytes = max(0, FIRST_DOWNLOAD_BYTES - cache_budget["reusable_bytes"])\\n    available_ram = memory["MemAvailable"] / GIB\\n    strategy = ("staged_component_residency" if residency == "staged" else\\n                "component_cpu_offload" if vram_gib >= 75 else "group_cpu_offload")\\n    required_ram_gib = 70 if residency == "staged" else 140\\n    report = {\\n        "mode": mode, "gpu": gpu.strip(), "vram_gib": round(vram_gib, 1),\\n        "available_ram_gib": round(available_ram, 1),\\n        "total_ram_gib": round(memory["MemTotal"] / GIB, 1),\\n        "cache_free_disk_gib": round(free_disk, 1), "cache_path": str(cache),\\n        "cache_reusable_weight_gib": round(cache_budget["reusable_bytes"] / GIB, 2),\\n        "cache_remaining_required_gib": round(remaining_disk_bytes / GIB, 2),\\n        "cache_weight_observation": cache_budget,\\n        "disk_check_skipped": os.environ.get("H3_SKIP_DISK_CHECK") == "1",\\n        "a100_variant": "80GB" if vram_gib >= 75 else "40GB",\\n        "strategy": strategy,\\n        "residency": residency, "required_available_ram_gib": required_ram_gib,\\n    }\\n    errors = []\\n    if "A100" not in gpu.upper():\\n        errors.append(f"GPU must be an NVIDIA A100, detected {gpu.strip()}")\\n    if available_ram < required_ram_gib:\\n        errors.append(f"available host RAM {available_ram:.1f} GiB < {required_ram_gib} GiB ({residency})")\\n    if vram_gib < 38:\\n        errors.append(f"GPU memory {vram_gib:.1f} GiB < 38 GiB (A100 40GB class required)")\\n    if free_disk_bytes < remaining_disk_bytes and not report["disk_check_skipped"]:\\n        errors.append(\\n            f"cache disk free {free_disk:.1f} GiB < remaining required {remaining_disk_bytes / GIB:.1f} GiB "\\n            f"(140 GiB first-download budget minus {cache_budget[\\\'reusable_bytes\\\'] / GIB:.1f} GiB "\\n            "of completed reusable weights; unverified/partial files receive no credit)"\\n        )\\n    print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)\\n    if errors:\\n        raise RuntimeError(\\n            "Resource preflight failed before model download: " + "; ".join(errors)\\n            + ". These are conservative project gates, not a guarantee against peak-memory OOM."\\n        )\\n    if residency == "staged":\\n        print(\\n            "Staged A100 BF16: the text encoder and transformer_ref are loaded one at a time, "\\n            "with the previous large component unloaded first. Disk reloads and CPU/GPU transfers add latency. "\\n            "The 70 GiB host-RAM gate is a project threshold. The original staged A100 40GB single-image baseline passed; this optimized build still needs GPU acceptance.",\\n            flush=True,\\n        )\\n    elif strategy == "group_cpu_offload":\\n        print(\\n            "A100 40GB: native BF16 weights are preserved. The 61.7 GB transformer cannot fit in GPU memory, "\\n            "so blocks move between CPU RAM and GPU. Throughput depends on PCIe and host memory bandwidth and may be slow.",\\n            flush=True,\\n        )\\n    else:\\n        print("A100 80GB: native BF16 with components alternated between CPU RAM and GPU.", flush=True)\\n    return report\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--staged", action="store_true", help="Explicit sequential BF16 residency profile: 70 GiB available host RAM")\\n    args = parser.parse_args()\\n    check_resources(os.environ.get("H3_MODE", "bf16"), residency="staged" if args.staged else "resident")\\n\', encoding="utf-8")\n(PROJECT_DIR / "notebook_ui.py").write_text(\'"""Read-only dashboard and separate diagnostics for the notebook control panel."""\\nfrom __future__ import annotations\\n\\nimport html\\nimport json\\nfrom collections import deque\\nfrom datetime import datetime, timedelta, timezone\\nfrom pathlib import Path\\nimport re\\nimport subprocess\\nimport time\\n\\nSTYLE = """\\n<style>\\n.h3-card{--bg:#f8fafc;--ink:#172033;--muted:#627086;--line:#dbe3ec;--accent:#187c72;box-sizing:border-box;background:var(--bg);color:var(--ink);border:1px solid var(--line);border-left:4px solid var(--accent);border-radius:12px;padding:17px 20px;margin:8px 0;font:14px/1.55 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;max-width:980px}\\n.h3-card *{box-sizing:border-box}.h3-card[data-tone="error"]{--accent:#ba4242}.h3-card[data-tone="idle"]{--accent:#7b8797}.h3-card[data-tone="running"]{--accent:#bd7c16}\\n.h3-head{display:flex;gap:12px;justify-content:space-between;align-items:center;flex-wrap:wrap}.h3-title{font-size:17px;font-weight:650;letter-spacing:-.2px}.h3-badge{font-size:11px;font-weight:650;letter-spacing:.3px;color:var(--accent);border:1px solid var(--line);padding:3px 9px;border-radius:99px;white-space:nowrap}.h3-note{color:var(--muted);margin:5px 0 0}\\n.h3-fields{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px 18px;margin-top:13px}.h3-label{display:block;color:var(--muted);font-size:11px;letter-spacing:.25px;margin-bottom:2px}.h3-value{font-size:14px;font-weight:550;overflow-wrap:anywhere}.h3-card details{margin-top:12px;border-top:1px solid var(--line);padding-top:10px}.h3-card summary{cursor:pointer;color:var(--muted);font-size:12px}.h3-card pre{white-space:pre-wrap;overflow-wrap:anywhere;font:12px/1.55 ui-monospace,SFMono-Regular,Consolas,monospace;margin:9px 0 0;max-height:280px;overflow:auto}.h3-card a{color:var(--accent);text-decoration:none}.h3-card a:hover{text-decoration:underline}\\n.h3-board{border-left-width:1px;padding:22px 24px}.h3-board h2{font-size:23px;line-height:1.3;margin:4px 0 8px}.h3-kicker{font-size:12px;color:var(--muted)}.h3-next{margin:12px 0 21px;padding:11px 14px;border-radius:8px;background:color-mix(in srgb,var(--accent) 8%,var(--bg));font-size:15px}.h3-modules{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:12px 0 19px}.h3-module{padding:12px 14px;border:1px solid var(--line);border-radius:8px}.h3-module strong{font-weight:650;display:block;font-size:16px;margin-top:4px}.h3-subtle{color:var(--muted);font-size:12px;margin-top:4px}.h3-summary{display:flex;flex-wrap:wrap;gap:8px 22px;margin:12px 0}.h3-activity{border-top:1px solid var(--line);margin-top:18px;padding-top:13px}.h3-activity .h3-title{font-size:14px}.h3-activity .h3-fields{margin-top:8px;gap:8px 18px}.h3-stamp{font-size:11px;color:var(--muted);margin-top:17px}.h3-logs{border-left-width:1px}.h3-logs pre{max-height:360px;margin:14px 0 0}.h3-logs pre:focus{outline:2px solid var(--accent);outline-offset:3px}\\n@media(max-width:620px){.h3-board{padding:17px 15px}.h3-modules{grid-template-columns:1fr;gap:8px}.h3-module{padding:9px 12px;display:grid;grid-template-columns:95px 1fr;align-items:center}.h3-module strong{margin:0}.h3-module .h3-subtle{grid-column:2}.h3-board h2{font-size:21px}.h3-fields{grid-template-columns:1fr}}\\n@media(prefers-color-scheme:dark){.h3-card{--bg:#17202d;--ink:#e4ebf4;--muted:#a5b3c6;--line:#344358;--accent:#67cdbb}.h3-card[data-tone="error"]{--accent:#ff9696}.h3-card[data-tone="running"]{--accent:#e4ba6d}.h3-card[data-tone="idle"]{--accent:#b5bfcb}}\\n</style>\\n"""\\n\\n_dashboard = None\\n_diagnostics = deque(maxlen=20)\\n_known_secrets = ()\\n_process_observer = None\\n_DIAGNOSTIC_LIMIT = 12000\\n\\n\\ndef redact_text(text, secrets=()):\\n    """Redact known credentials and common authorization/key assignments before display."""\\n    text = str(text)\\n    for secret in sorted({str(s) for s in (*_known_secrets, *secrets) if s}, key=len, reverse=True):\\n        text = text.replace(secret, \\\'[已隐藏]\\\')\\n    text = re.sub(r\\\'(?i)(\\\\bBearer\\\\s+)[^\\\\s\\\\"\\\\\\\'<>;,]+\\\', r\\\'\\\\1[已隐藏]\\\', text)\\n    text = re.sub(r\\\'\\\'\\\'(?ix)(["\\\']?(?:H3_API_KEY|API_KEY|api[-_]?key|access_token|authorization)["\\\']?\\\\s*[:=]\\\\s*)(["\\\'])(.*?)(\\\\2)\\\'\\\'\\\',\\n                  lambda match: match.group(1) + match.group(2) + \\\'[已隐藏]\\\' + match.group(2), text)\\n    text = re.sub(r\\\'(?i)(\\\\b(?:H3_API_KEY|API_KEY|api[-_]?key|access_token)\\\\s*[:=]\\\\s*)[^\\\\s\\\\"\\\\\\\'<>;,]+\\\', r\\\'\\\\1[已隐藏]\\\', text)\\n    return text\\n\\n\\ndef _remember(title, text):\\n    if text:\\n        _diagnostics.append(redact_text(str(title) + \\\'\\\\n\\\' + str(text))[-_DIAGNOSTIC_LIMIT:])\\n\\n\\ndef last_diagnostics():\\n    return \\\'\\\\n\\\\n\\\'.join(_diagnostics)[-_DIAGNOSTIC_LIMIT:]\\n\\n\\ndef diagnostics_html(title, text, *, secrets=()):\\n    """A bounded, keyboard-scrollable log view; never embedded in dashboard cards."""\\n    value = redact_text(text, secrets=secrets)\\n    truncated = len(value) > _DIAGNOSTIC_LIMIT\\n    value = value[-_DIAGNOSTIC_LIMIT:]\\n    note = \\\'仅显示最后 12,000 个字符；完整记录保留在日志文件。\\\' if truncated else \\\'只读日志 · 敏感凭据已隐藏\\\'\\n    return (STYLE + \\\'<section class="h3-card h3-logs" aria-label="日志"><div class="h3-title">\\\'\\n            + html.escape(redact_text(title), quote=True) + \\\'</div><p class="h3-note">\\\' + note\\n            + \\\'</p><pre tabindex="0" aria-label="日志内容">\\\' + html.escape(value or \\\'暂无记录。\\\', quote=True)\\n            + \\\'</pre></section>\\\')\\n\\n\\ndef _short(value, maximum=160):\\n    text = redact_text(value).replace(\\\'\\\\n\\\', \\\' \\\').replace(\\\'\\\\r\\\', \\\' \\\')\\n    return text if len(text) <= maximum else text[:maximum - 1] + \\\'…\\\'\\n\\n\\ndef _activity(title, kwargs):\\n    _remember(title, kwargs.get(\\\'details\\\'))\\n    fields = [(label, value) for label, value in kwargs.get(\\\'fields\\\', ())\\n              if not any(word in str(label) for word in (\\\'日志\\\', \\\'路径\\\', \\\'密钥\\\', \\\'当前记录\\\'))]\\n    return {\\\'title\\\': _short(title, 70), \\\'badge\\\': _short(kwargs.get(\\\'badge\\\', \\\'\\\'), 35),\\n            \\\'note\\\': _short(kwargs.get(\\\'note\\\', \\\'\\\')), \\\'tone\\\': kwargs.get(\\\'tone\\\', \\\'idle\\\'),\\n            \\\'fields\\\': [(str(k), _short(v, 95)) for k, v in fields[:4]]}\\n\\n\\ndef card_html(title, *, badge="", note="", fields=(), tone="idle", details=None, details_label="查看详情"):\\n    escape = lambda value: html.escape(_short(value), quote=True)\\n    if tone not in {"idle", "running", "success", "error"}:\\n        tone = "idle"\\n    values = "".join(\\n        \\\'<div><span class="h3-label">\\\' + escape(label) + \\\'</span><span class="h3-value">\\\'\\n        + escape(value) + \\\'</span></div>\\\' for label, value in fields\\n    )\\n    return (STYLE + \\\'<section class="h3-card" data-tone="\\\' + tone + \\\'"><div class="h3-head">\\\'\\n            + \\\'<div class="h3-title">\\\' + escape(title) + \\\'</div><span class="h3-badge">\\\'\\n            + escape(badge) + \\\'</span></div>\\\' + (\\\'<p class="h3-note">\\\' + escape(note) + \\\'</p>\\\' if note else "")\\n            + (\\\'<div class="h3-fields">\\\' + values + \\\'</div>\\\' if fields else "") + "</section>")\\n\\n\\ndef render_dashboard(context, activity=None):\\n    """Render supplied observations only. This pure function performs no host I/O."""\\n    escape = lambda value: html.escape(_short(value), quote=True)\\n    state = context.get(\\\'state\\\') or {}\\n    stale = bool(state.get(\\\'observation_stale\\\'))\\n    failed_read = bool(context.get(\\\'observation_error\\\'))\\n    phase = state.get(\\\'phase\\\')\\n    if context.get(\\\'placeholder\\\'):\\n        headline, next_step, tone = \\\'等待首次读取\\\', \\\'运行「刷新看板」查看本次状态；此处是界面占位，没有实时观察。\\\', \\\'idle\\\'\\n    elif failed_read:\\n        headline, next_step, tone = \\\'状态读取失败\\\', \\\'重新刷新看板；如仍失败，在日志区查看原因。\\\', \\\'error\\\'\\n    elif stale:\\n        headline, next_step, tone = \\\'状态需要刷新\\\', \\\'当前观察已过期；刷新后再判断可用能力。\\\', \\\'running\\\'\\n    elif phase == \\\'stopped\\\' and state.get(\\\'cleanup_complete\\\'):\\n        headline, next_step, tone = \\\'本次服务已停止\\\', \\\'确认结果已保存后，可在控制面板释放 Colab 实例。\\\', \\\'idle\\\'\\n    elif phase == \\\'stopping\\\':\\n        headline, next_step, tone = \\\'正在停止与回收\\\', \\\'等待受管进程退出并确认清理完成；此时不要重复启动。\\\', \\\'running\\\'\\n    elif state.get(\\\'error\\\') or phase in {\\\'failed\\\', \\\'degraded\\\'}:\\n        headline, next_step, tone = \\\'有模块需要处理\\\', \\\'先看下方可用能力；具体原因在日志区，控制面板仍可查询或停止。\\\', \\\'error\\\'\\n    elif state.get(\\\'draining\\\'):\\n        headline, next_step, tone = \\\'已暂停接收新任务\\\', \\\'已接收的队列继续执行；可恢复接收，也可停止服务。\\\', \\\'idle\\\'\\n    elif state.get(\\\'ready\\\'):\\n        headline, next_step, tone = \\\'可以生成\\\', \\\'在控制面板提交新内容，或按任务 ID 查询已有任务。\\\', \\\'success\\\'\\n    elif state.get(\\\'api_ready\\\'):\\n        accepting = (state.get(\\\'queue\\\') or {}).get(\\\'admission_open\\\') is True\\n        headline, next_step, tone = (\\\'API 可用 · 生成尚未就绪\\\',\\n            (\\\'可先查询或排队；\\\' if accepting else \\\'可先查询；\\\') + \\\'模型、公网的当前能力分别见下方。\\\', \\\'running\\\')\\n    elif state:\\n        headline, next_step, tone = \\\'服务正在准备\\\', \\\'刷新看板观察进展；不要重复启动。\\\', \\\'running\\\'\\n    elif not context.get(\\\'preflight_ok\\\'):\\n        headline, next_step, tone = \\\'尚未检查资源\\\', \\\'在控制面板先运行「资源与驻留方案」。\\\', \\\'idle\\\'\\n    elif not context.get(\\\'install_ok\\\'):\\n        headline, next_step, tone = \\\'资源检查通过\\\', \\\'在控制面板运行「安装与检查」。\\\', \\\'idle\\\'\\n    else:\\n        headline, next_step, tone = \\\'准备工作已完成\\\', \\\'在控制面板选择是否开放公网，再启动服务。\\\', \\\'idle\\\'\\n\\n    def module(name, alive_key, ready_key=None):\\n        alive, ready = state.get(alive_key), state.get(ready_key) if ready_key else None\\n        if not state:\\n            status, detail = (\\\'尚未读取\\\' if context.get(\\\'placeholder\\\') else \\\'未启动\\\'), \\\'尚无运行观察\\\'\\n        elif ready is True:\\n            status, detail = \\\'可用\\\', \\\'进程运行中\\\' if alive is True else \\\'存活状态未确认\\\'\\n        elif alive is True:\\n            status, detail = (\\\'运行中\\\', \\\'调度与控制\\\') if ready_key is None else (\\\'准备中\\\', \\\'进程运行，能力未就绪\\\')\\n        elif alive is False:\\n            status, detail = \\\'已停止\\\', \\\'进程未运行\\\'\\n        else:\\n            status, detail = \\\'未确认\\\', \\\'缺少当前观察\\\'\\n        if stale or failed_read:\\n            status, detail = \\\'待刷新\\\', \\\'不据旧状态判断就绪\\\'\\n        return (\\\'<div class="h3-module"><span class="h3-label">\\\' + escape(name) + \\\'</span><strong>\\\'\\n                + escape(status) + \\\'</strong><div class="h3-subtle">\\\' + escape(detail) + \\\'</div></div>\\\')\\n\\n    if stale or failed_read:\\n        public = \\\'待刷新\\\'\\n    elif not state:\\n        public = \\\'尚未观察\\\'\\n    elif not state.get(\\\'public_requested\\\'):\\n        public = \\\'未启用\\\'\\n    elif state.get(\\\'public_api_ready\\\'):\\n        public = \\\'API 探测通过\\\'\\n    elif state.get(\\\'tunnel_connected\\\'):\\n        public = \\\'隧道已连接 · API 待验证\\\'\\n    else:\\n        public = \\\'准备中\\\'\\n    queue = state.get(\\\'queue\\\') or {}\\n    summary = [(\\\'公网\\\', public)]\\n    if queue and not stale and not failed_read:\\n        summary += [(\\\'任务\\\', f"排队 {queue.get(\\\'queued\\\', \\\'—\\\')} · 执行 {queue.get(\\\'running\\\', \\\'—\\\')}")]\\n        if queue.get(\\\'admission_open\\\') is False:\\n            summary.append((\\\'接收任务\\\', \\\'已暂停\\\'))\\n    resources = context.get(\\\'resource_report\\\') or {}\\n    if resources:\\n        summary += [(\\\'GPU\\\', resources.get(\\\'gpu\\\', \\\'未记录\\\')),\\n                    (\\\'可用内存\\\', str(resources.get(\\\'available_ram_gib\\\', \\\'—\\\')) + \\\' GiB（预检）\\\')]\\n    resource_text = \\\'\\\'.join(\\\'<span><span class="h3-label">\\\' + escape(k) + \\\'</span>\\\' + escape(v) + \\\'</span>\\\' for k, v in summary)\\n    stamp = state.get(\\\'observed_at\\\')\\n    if isinstance(stamp, (int, float)):\\n        try:\\n            stamp = datetime.fromtimestamp(stamp, timezone(timedelta(hours=8))).strftime(\\\'%m-%d %H:%M:%S 北京时间\\\')\\n        except (OverflowError, OSError, ValueError):\\n            stamp = \\\'未记录\\\'\\n    else:\\n        stamp = \\\'尚无服务快照\\\'\\n    body = (STYLE + \\\'<section class="h3-card h3-board" data-tone="\\\' + tone + \\\'" aria-label="运行看板">\\\'\\n            \\\'<div class="h3-kicker">当前状态</div><h2>\\\' + escape(headline) + \\\'</h2>\\\'\\n            \\\'<div class="h3-next"><b>下一步</b>\\u3000\\\' + escape(next_step) + \\\'</div><div class="h3-modules">\\\'\\n            + module(\\\'总管理\\\', \\\'manager_alive\\\') + module(\\\'API 后台\\\', \\\'server_alive\\\', \\\'api_ready\\\')\\n            + module(\\\'大模型容器\\\', \\\'model_alive\\\', \\\'model_ready\\\') + \\\'</div>\\\'\\n            \\\'<div class="h3-summary">\\\' + resource_text + \\\'</div>\\\')\\n    if activity:\\n        activity_tone = activity.get(\\\'tone\\\', \\\'idle\\\')\\n        if activity_tone not in {\\\'idle\\\', \\\'running\\\', \\\'success\\\', \\\'error\\\'}:\\n            activity_tone = \\\'idle\\\'\\n        body += (\\\'<div class="h3-activity" data-tone="\\\' + activity_tone + \\\'"><div class="h3-label">最近操作</div>\\\'\\n                 \\\'<div class="h3-head"><span class="h3-title">\\\' + escape(activity.get(\\\'title\\\', \\\'\\\'))\\n                 + \\\'</span><span class="h3-badge">\\\' + escape(activity.get(\\\'badge\\\', \\\'\\\')) + \\\'</span></div>\\\')\\n        if activity.get(\\\'note\\\'):\\n            body += \\\'<div class="h3-note">\\\' + escape(activity[\\\'note\\\']) + \\\'</div>\\\'\\n        fields = activity.get(\\\'fields\\\') or ()\\n        if fields:\\n            body += \\\'<div class="h3-fields">\\\' + \\\'\\\'.join(\\\'<div><span class="h3-label">\\\' + escape(k)\\n                + \\\'</span><span class="h3-value">\\\' + escape(v) + \\\'</span></div>\\\' for k, v in fields[:4]) + \\\'</div>\\\'\\n        body += \\\'</div>\\\'\\n    return body + \\\'<div class="h3-stamp">服务观察：\\\' + escape(stamp) + \\\' · 按需刷新，不执行控制命令</div></section>\\\'\\n\\n\\nclass Dashboard:\\n    def __init__(self, context_reader):\\n        self.context_reader = context_reader\\n        self.activity = None\\n        self._handle = None\\n\\n    def refresh(self):\\n        from IPython.display import HTML, display\\n        global _known_secrets\\n        context = dict(self.context_reader())\\n        host = context.pop(\\\'host\\\', None)\\n        if host is not None:\\n            key = getattr(host, \\\'api_key\\\', None)\\n            if key:\\n                _known_secrets = tuple(set((*_known_secrets[-3:], str(key))))\\n            try:\\n                context[\\\'state\\\'] = host.snapshot()\\n            except Exception as error:\\n                context[\\\'state\\\'] = {}\\n                context[\\\'observation_error\\\'] = True\\n                _remember(\\\'看板状态读取失败\\\', str(error))\\n        rendered = HTML(render_dashboard(context, self.activity))\\n        if self._handle is None:\\n            self._handle = display(rendered, display_id=\\\'h3-dashboard\\\')\\n        else:\\n            self._handle.update(rendered)\\n        return context\\n\\n\\ndef mount_dashboard(context_reader):\\n    global _dashboard\\n    _dashboard = Dashboard(context_reader)\\n    _dashboard.refresh()\\n    return _dashboard\\n\\n\\nclass StatusCard:\\n    def __init__(self, title, **kwargs):\\n        self._handle = None\\n        self.update(title, **kwargs)\\n\\n    def update(self, title, **kwargs):\\n        from IPython.display import HTML, display\\n        activity = _activity(title, kwargs)\\n        if _dashboard is not None:\\n            _dashboard.activity = activity\\n            _dashboard.refresh()\\n        elif self._handle is None:\\n            self._handle = display(HTML(card_html(**activity)), display_id=True)\\n        else:\\n            self._handle.update(HTML(card_html(**activity)))\\n\\n\\ndef elapsed_text(seconds):\\n    seconds = max(0, int(seconds))\\n    return f"{seconds // 60}分{seconds % 60:02d}秒" if seconds >= 60 else f"{seconds}秒"\\n\\n\\ndef log_tail(path, limit=6000):\\n    path = Path(path)\\n    if not path.is_file():\\n        return ""\\n    with path.open("rb") as source:\\n        source.seek(max(0, path.stat().st_size - limit))\\n        text = source.read().decode("utf-8", errors="replace")\\n    return re.sub(r"\\\\x1b\\\\[[0-?]*[ -/]*[@-~]", "", text).replace("\\\\r", "\\\\n")\\n\\n\\ndef latest_stage(path):\\n    lines = [line.strip() for line in log_tail(path).splitlines() if line.strip()]\\n    for line in reversed(lines):\\n        if "BF16 staged:" in line:\\n            return line.split("BF16 staged:", 1)[1].strip()[:145]\\n        if "status=" in line or "API_SMOKE_TEST_PASSED" in line:\\n            return line[:145]\\n    return lines[-1][:145] if lines else "进程已启动，等待首条日志"\\n\\n\\ndef run_logged(command, log_path, *, title, cwd=None, env=None, card=None, status_reader=None):\\n    """Run a real child process; persist logs and update a bounded status card."""\\n    path = Path(log_path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    status = card or StatusCard(title, badge="执行中", tone="running", note="完整输出写入日志。")\\n    started = time.monotonic()\\n    with path.open("a", encoding="utf-8") as output:\\n        output.write("\\\\n--- " + title + " ---\\\\n")\\n        output.flush()\\n        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)\\n        observer = _process_observer\\n        if observer is not None:\\n            observer.process_started(process)\\n        try:\\n            while process.poll() is None:\\n                if observer is not None:\\n                    observer.process_progress(process)\\n                current = []\\n                if status_reader is not None:\\n                    try:\\n                        current = status_reader()\\n                    except (OSError, ValueError):\\n                        pass\\n                status.update(title, badge="执行中", tone="running", fields=\\n                    [("已用时间", elapsed_text(time.monotonic() - started))]\\n                    + current)\\n                time.sleep(2)\\n        except BaseException:\\n            process.terminate()\\n            try:\\n                process.wait(timeout=15)\\n            except subprocess.TimeoutExpired:\\n                process.kill()\\n                process.wait()\\n            status.update(title, badge="已中断", tone="error", note="操作进程已停止；已提交的远程任务可能仍在执行。", fields=[("日志", str(path))])\\n            raise\\n        finally:\\n            if observer is not None:\\n                observer.process_finished(process)\\n    if process.returncode:\\n        status.update(title, badge="未完成", tone="error", note="查看日志中的实际错误后再重试。",\\n                      fields=[("日志", str(path))], details=log_tail(path), details_label="展开最后的错误日志")\\n        raise RuntimeError(f"{title}退出码 {process.returncode}；完整日志：{path}")\\n    status.update(title, badge="完成", tone="success", fields=[\\n        ("已用时间", elapsed_text(time.monotonic() - started)), ("完整日志", str(path)),\\n    ])\\n    return process.returncode\\n\\n\\ndef json_probe(command, log_path, *, cwd=None, env=None, card=None):\\n    """Capture a short JSON-producing check without dumping full notebook output."""\\n    process = subprocess.Popen(command, cwd=cwd, env=env, text=True,\\n                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)\\n    observer = _process_observer\\n    try:\\n        if observer is not None:\\n            observer.process_started(process)\\n        stdout, stderr = process.communicate(timeout=180)\\n    except BaseException:\\n        if process.poll() is None:\\n            process.kill()\\n            process.communicate(timeout=10)\\n        raise\\n    finally:\\n        if observer is not None:\\n            observer.process_finished(process)\\n    with Path(log_path).open("a", encoding="utf-8") as log:\\n        log.write(stdout + "\\\\n" + stderr + "\\\\n")\\n    if process.returncode:\\n        if card is not None:\\n            card.update("依赖检查未通过", badge="需要处理", tone="error", note="检查日志中的版本或导入错误后再继续。",\\n                        fields=[("日志", str(log_path))], details=stderr[-4000:], details_label="展开检查错误")\\n        raise RuntimeError(f"依赖检查失败；请在日志区查看：{log_path}")\\n    return json.loads(stdout.strip().splitlines()[-1])\\n\', encoding="utf-8")\nif str(PROJECT_DIR) not in sys.path:\n    sys.path.insert(0, str(PROJECT_DIR))\nfrom preflight import check_resources\nfrom notebook_ui import StatusCard, elapsed_text, latest_stage, log_tail, run_logged, json_probe\nresource_card = StatusCard("检查资源", badge="检查中", tone="running")\ncaptured = io.StringIO()\ntry:\n    with contextlib.redirect_stdout(captured):\n        RESOURCE_REPORT = check_resources(residency=RESIDENCY)\nexcept Exception as error:\n    resource_card.update("资源检查未通过", badge="需要处理", tone="error", note=str(error),\n                         details=captured.getvalue(), details_label="展开资源详情")\n    raise\nfinally:\n    (PROJECT_DIR / "preflight.log").write_text(captured.getvalue(), encoding="utf-8")\nPREFLIGHT_OK = True\nresource_card.update("资源检查通过", badge="可继续", tone="success", note="本格未下载模型；已有缓存会在加载时检查。",\n    fields=[("GPU", RESOURCE_REPORT["gpu"]), ("可用 RAM", str(RESOURCE_REPORT["available_ram_gib"]) + " GiB"),\n            ("空余磁盘", str(RESOURCE_REPORT["cache_free_disk_gib"]) + " GiB"), ("驻留方案", RESIDENCY)],\n    details=captured.getvalue(), details_label="展开完整检查")\n', 'install': '\x23@title 安装与检查\n\x23@markdown 保留 Colab 预装 PyTorch 栈。完整安装记录写入 install.log，界面只显示当前状态。\nimport subprocess, shutil, json\nINSTALL_OK = False\nif not globals().get("PREFLIGHT_OK"):\n    raise RuntimeError("先运行资源检查；资源检查不通过时不安装或下载。")\nexisting_host = globals().get("startup_host")\nif existing_host:\n    existing_state = existing_host.snapshot()\n    if (existing_state["server_alive"] or existing_state["tunnel_alive"] or\n            existing_state.get("model_alive") or existing_state.get("manager_alive") or\n            (existing_state["phase"] != "new" and\n             (existing_state["phase"] not in {"failed", "stopped"} or\n              not existing_state.get("cleanup_complete", False)))):\n        raise RuntimeError("旧生命周期仍在运行或尚未完成清理，不能安装依赖；先在停止与释放 停止并确认子进程已退出。")\nPYTORCH_STACK_CHECK = \'import importlib.metadata as metadata, json, platform, torch, psutil\\nversions = {}\\nfor name in ("torch", "torchvision", "torchaudio"):\\n    try:\\n        versions[name] = metadata.version(name)\\n    except metadata.PackageNotFoundError:\\n        pass\\nreport = {"python": platform.python_version(), "pytorch_stack": versions,\\n          "host_runtime": {"psutil": metadata.version("psutil")},\\n          "torch_cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available(),\\n          "native_bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),\\n          "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\\nprint(json.dumps(report, ensure_ascii=False))\\nif psutil.__version__ != report["host_runtime"]["psutil"]:\\n    raise RuntimeError("psutil imported module and installed distribution versions do not match")\\nif not report["native_bf16"]:\\n    raise RuntimeError("The preinstalled PyTorch stack has no native-BF16 CUDA support")\\n\'\nDEPENDENCY_CHECK = \'import json, os, platform, torch, diffusers, transformers, fastapi, uvicorn, av, PIL, psutil\\nfrom diffusers import ModularPipeline, MiniMaxH3Transformer3DModel\\nfrom diffusers.modular_pipelines.minimax_h3 import MiniMaxH3ImageReference, MiniMaxH3VideoReference, MiniMaxH3AudioReference\\nfrom transformers import Qwen3VLForConditionalGeneration\\nfrom diffusers.utils.export_utils import encode_video\\nfrom adapter_config import load_adapter_settings\\nsettings = load_adapter_settings()\\nif settings.acceleration != "none" or settings.style != "none":\\n    try:\\n        import peft\\n    except ImportError as exc:\\n        raise RuntimeError("LoRA enabled: rerun notebook step 2 to install requirements-lora.txt") from exc\\nif not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\\n    raise RuntimeError("Native-BF16 CUDA support is required")\\nprint(json.dumps({"fresh_process_imports": "passed", "python": platform.python_version(),\\n                  "torch": torch.__version__, "cuda": torch.version.cuda,\\n                  "diffusers": diffusers.__version__, "transformers": transformers.__version__,\\n                  "host_runtime": {"psutil": psutil.__version__},\\n                  "native_bf16": True}, ensure_ascii=False))\\n\'\nPACKAGE_FILES = {\'preflight.py\': \'"""Conservative project resource checks; no downloads and no model imports."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport stat\\nimport subprocess\\n\\nGIB = 1024 ** 3\\nFIRST_DOWNLOAD_BYTES = 140 * GIB\\nMODEL_REVISION = "0919fb684102ef115126612f3fbb84d1027e7317"\\n_WEIGHT_COMPONENTS = {"text_encoder": "model", "transformer_ref": "diffusion_pytorch_model",\\n                      "vae": "diffusion_pytorch_model", "audio_vae": "diffusion_pytorch_model"}\\n_METADATA_LIMIT = 16 * 1024 ** 2\\n\\n\\ndef _hub_cache_path() -> Path:\\n    default_home = Path(os.environ.get("XDG_CACHE_HOME", "~/.cache")) / "huggingface"\\n    home = Path(os.environ.get("HF_HOME", str(default_home))).expanduser()\\n    return Path(os.environ.get("HF_HUB_CACHE", os.environ.get("HUGGINGFACE_HUB_CACHE", str(home / "hub")))).expanduser()\\n\\n\\ndef _cache_file(path, snapshot, blobs):\\n    """Accept snapshot files and completed repo/shared content-addressed blobs."""\\n    if path.parent.resolve(strict=True) != path.parent:\\n        raise ValueError("untrusted cache component directory redirection")\\n    resolved = path.resolve(strict=True)\\n    # Newer Hub caches share immutable SHA-256 blobs across repositories. Only\\n    # an exact snapshot reference may reach this pool; never scan/credit it.\\n    shared_pool = blobs.parent.parent / "blobs"\\n    shared_bucket = shared_pool / resolved.name[:2]\\n    shared_blob = (re.fullmatch(r"[0-9a-f]{64}", resolved.name) is not None\\n                   and resolved.parent == shared_bucket\\n                   and shared_pool.resolve(strict=True) == shared_pool\\n                   and shared_bucket.resolve(strict=True) == shared_bucket)\\n    if not (resolved.is_relative_to(snapshot) or\\n            (resolved.parent == blobs and re.fullmatch(r"[0-9a-f]{40}|[0-9a-f]{64}", resolved.name))\\n            or shared_blob):\\n        raise ValueError("cache file escapes its snapshot/repository blobs")\\n    info = resolved.stat()\\n    if not stat.S_ISREG(info.st_mode) or info.st_size <= 0 or ".incomplete" in resolved.name:\\n        raise ValueError("cache file is not a completed regular file")\\n    return resolved, info\\n\\n\\ndef _small_json(path, snapshot, blobs):\\n    resolved, info = _cache_file(path, snapshot, blobs)\\n    if info.st_size > _METADATA_LIMIT:\\n        raise ValueError("cache metadata exceeds inspection limit")\\n    return json.loads(resolved.read_bytes())\\n\\n\\ndef _weight_metadata(path, snapshot, blobs):\\n    """Check only the bounded safetensors header and file size, not weight data."""\\n    resolved, before = _cache_file(path, snapshot, blobs)\\n    with resolved.open("rb") as source:\\n        prefix = source.read(8)\\n        if len(prefix) != 8:\\n            raise ValueError("incomplete safetensors header")\\n        length = int.from_bytes(prefix, "little")\\n        if not 2 <= length <= _METADATA_LIMIT or length + 8 > before.st_size:\\n            raise ValueError("invalid safetensors header size")\\n        header = json.loads(source.read(length))\\n    if not isinstance(header, dict):\\n        raise ValueError("invalid safetensors header")\\n    ranges, names = [], set()\\n    sizes = {"BOOL": 1, "U8": 1, "I8": 1, "U16": 2, "I16": 2, "F16": 2, "BF16": 2,\\n             "U32": 4, "I32": 4, "F32": 4, "U64": 8, "I64": 8, "F64": 8,\\n             "F8_E4M3": 1, "F8_E5M2": 1}\\n    for name, entry in header.items():\\n        if name == "__metadata__":\\n            continue\\n        start, end = entry["data_offsets"]\\n        shape, element_bytes = entry["shape"], sizes[entry["dtype"]]\\n        if not isinstance(shape, list) or len(shape) > 16 or any(type(n) is not int or n < 0 for n in shape):\\n            raise ValueError("invalid tensor shape")\\n        count = 1\\n        for dimension in shape:\\n            count *= dimension\\n        if type(start) is not int or type(end) is not int or start < 0 or end - start != count * element_bytes:\\n            raise ValueError("invalid tensor data range")\\n        ranges.append((start, end))\\n        names.add(name)\\n    cursor = 0\\n    for start, end in sorted(ranges):\\n        if start != cursor:\\n            raise ValueError("tensor data ranges are incomplete or overlap")\\n        cursor = end\\n    after = resolved.stat()\\n    if not names or cursor <= 0 or 8 + length + cursor != before.st_size or (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\\n        raise ValueError("weight file is truncated, changing, or has trailing data")\\n    identity = (after.st_dev, after.st_ino) if after.st_ino else str(resolved)\\n    return names, cursor, after.st_size, identity\\n\\n\\ndef cached_weight_budget(cache: Path) -> dict:\\n    """Credit complete weight components, not readiness or model integrity."""\\n    revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\\n    result = {"model_revision": revision, "reusable_bytes": 0, "verified_components": [],\\n              "observation": "no_verified_weights", "scope": "disk_budget_only_not_model_readiness"}\\n    if not re.fullmatch(r"[0-9a-f]{40}", revision):\\n        result["observation"] = "revision_not_an_exact_commit"\\n        return result\\n    try:\\n        root = cache.resolve(strict=True)\\n        repo = root / "models--MiniMaxAI--MiniMax-H3"\\n        snapshot = repo / "snapshots" / revision\\n        blobs = repo / "blobs"\\n        if not snapshot.is_dir():\\n            return result\\n        # Parent directory symlinks may otherwise redirect an exact snapshot\\n        # name into a different revision/repository or an unrelated filesystem.\\n        for path in (repo, snapshot.parent, snapshot):\\n            if path.resolve(strict=True) != path:\\n                raise ValueError("untrusted cache directory redirection")\\n        credited = {}\\n        for component, stem in _WEIGHT_COMPONENTS.items():\\n            directory = snapshot / component\\n            single = directory / (stem + ".safetensors")\\n            index = directory / (stem + ".safetensors.index.json")\\n            if single.exists():\\n                files, weight_map, total_size = [single.name], None, None\\n            elif index.exists():\\n                data = _small_json(index, snapshot, blobs)\\n                weight_map = data["weight_map"]\\n                if not isinstance(weight_map, dict) or not weight_map:\\n                    raise ValueError("invalid weight index")\\n                files = sorted(set(weight_map.values()))\\n                total_size = data.get("metadata", {}).get("total_size")\\n                if total_size is not None and (type(total_size) is not int or total_size <= 0):\\n                    raise ValueError("invalid indexed weight size")\\n            else:\\n                continue\\n            if any(not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_-]+\\\\.safetensors", name) for name in files):\\n                raise ValueError("untrusted shard filename")\\n            if any(not (directory / name).exists() for name in files):\\n                continue  # partially downloaded components receive no credit\\n            component_files, data_bytes = {}, 0\\n            for filename in files:\\n                names, size, disk_bytes, identity = _weight_metadata(directory / filename, snapshot, blobs)\\n                if weight_map is not None and names != {key for key, value in weight_map.items() if value == filename}:\\n                    raise ValueError("weight index does not match shard tensors")\\n                component_files[identity] = disk_bytes\\n                data_bytes += size\\n            if total_size is not None and data_bytes != total_size:\\n                raise ValueError("indexed weight bytes are incomplete")\\n            credited.update(component_files)\\n            result["verified_components"].append(component)\\n        result["reusable_bytes"] = min(sum(credited.values()), FIRST_DOWNLOAD_BYTES)\\n        if credited:\\n            result["observation"] = "completed_weight_metadata_verified"\\n    except FileNotFoundError:\\n        # Cache creation/deletion races and missing roots never relax the gate.\\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_missing_or_changed")\\n    except (OSError, ValueError, TypeError, KeyError, AttributeError, OverflowError, RuntimeError) as error:\\n        result.update(reusable_bytes=0, verified_components=[], observation="cache_unverified",\\n                      error_type=type(error).__name__)\\n    return result\\n\\n\\ndef check_resources(mode: str = "bf16", residency: str = "resident") -> dict:\\n    if mode != "bf16":\\n        raise ValueError("This project requires native BF16 on an NVIDIA A100; reduced-precision modes are disabled.")\\n    if residency not in {"resident", "staged"}:\\n        raise ValueError("residency must be resident or staged")\\n    if not Path("/proc/meminfo").is_file():\\n        raise RuntimeError("Deploy this project on a Linux NVIDIA GPU machine.")\\n    memory = {}\\n    for line in Path("/proc/meminfo").read_text().splitlines():\\n        name, value = line.split(":", 1)\\n        memory[name] = int(value.strip().split()[0]) * 1024\\n    try:\\n        result = subprocess.run(\\n            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],\\n            text=True, capture_output=True, check=True, timeout=20,\\n        )\\n        first = result.stdout.strip().splitlines()[0]\\n        gpu, memory_mib = first.rsplit(",", 1)\\n        vram_gib = float(memory_mib.strip()) / 1024\\n    except (OSError, ValueError, IndexError, subprocess.SubprocessError) as exc:\\n        raise RuntimeError("No working NVIDIA GPU detected. Select a GPU runtime first.") from exc\\n    # The first visible CUDA GPU is used. Report conservatively when device masks are present.\\n    visible = os.environ.get("CUDA_VISIBLE_DEVICES")\\n    if visible and visible not in {"0", "0,"}:\\n        raise RuntimeError("Preflight expects the first NVIDIA GPU. Unset CUDA_VISIBLE_DEVICES or set it to 0.")\\n    cache = _hub_cache_path()\\n    disk_path = cache\\n    while not disk_path.exists():\\n        disk_path = disk_path.parent\\n    free_disk_bytes = shutil.disk_usage(disk_path).free\\n    free_disk = free_disk_bytes / GIB\\n    cache_budget = cached_weight_budget(cache)\\n    remaining_disk_bytes = max(0, FIRST_DOWNLOAD_BYTES - cache_budget["reusable_bytes"])\\n    available_ram = memory["MemAvailable"] / GIB\\n    strategy = ("staged_component_residency" if residency == "staged" else\\n                "component_cpu_offload" if vram_gib >= 75 else "group_cpu_offload")\\n    required_ram_gib = 70 if residency == "staged" else 140\\n    report = {\\n        "mode": mode, "gpu": gpu.strip(), "vram_gib": round(vram_gib, 1),\\n        "available_ram_gib": round(available_ram, 1),\\n        "total_ram_gib": round(memory["MemTotal"] / GIB, 1),\\n        "cache_free_disk_gib": round(free_disk, 1), "cache_path": str(cache),\\n        "cache_reusable_weight_gib": round(cache_budget["reusable_bytes"] / GIB, 2),\\n        "cache_remaining_required_gib": round(remaining_disk_bytes / GIB, 2),\\n        "cache_weight_observation": cache_budget,\\n        "disk_check_skipped": os.environ.get("H3_SKIP_DISK_CHECK") == "1",\\n        "a100_variant": "80GB" if vram_gib >= 75 else "40GB",\\n        "strategy": strategy,\\n        "residency": residency, "required_available_ram_gib": required_ram_gib,\\n    }\\n    errors = []\\n    if "A100" not in gpu.upper():\\n        errors.append(f"GPU must be an NVIDIA A100, detected {gpu.strip()}")\\n    if available_ram < required_ram_gib:\\n        errors.append(f"available host RAM {available_ram:.1f} GiB < {required_ram_gib} GiB ({residency})")\\n    if vram_gib < 38:\\n        errors.append(f"GPU memory {vram_gib:.1f} GiB < 38 GiB (A100 40GB class required)")\\n    if free_disk_bytes < remaining_disk_bytes and not report["disk_check_skipped"]:\\n        errors.append(\\n            f"cache disk free {free_disk:.1f} GiB < remaining required {remaining_disk_bytes / GIB:.1f} GiB "\\n            f"(140 GiB first-download budget minus {cache_budget[\\\'reusable_bytes\\\'] / GIB:.1f} GiB "\\n            "of completed reusable weights; unverified/partial files receive no credit)"\\n        )\\n    print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)\\n    if errors:\\n        raise RuntimeError(\\n            "Resource preflight failed before model download: " + "; ".join(errors)\\n            + ". These are conservative project gates, not a guarantee against peak-memory OOM."\\n        )\\n    if residency == "staged":\\n        print(\\n            "Staged A100 BF16: the text encoder and transformer_ref are loaded one at a time, "\\n            "with the previous large component unloaded first. Disk reloads and CPU/GPU transfers add latency. "\\n            "The 70 GiB host-RAM gate is a project threshold. The original staged A100 40GB single-image baseline passed; this optimized build still needs GPU acceptance.",\\n            flush=True,\\n        )\\n    elif strategy == "group_cpu_offload":\\n        print(\\n            "A100 40GB: native BF16 weights are preserved. The 61.7 GB transformer cannot fit in GPU memory, "\\n            "so blocks move between CPU RAM and GPU. Throughput depends on PCIe and host memory bandwidth and may be slow.",\\n            flush=True,\\n        )\\n    else:\\n        print("A100 80GB: native BF16 with components alternated between CPU RAM and GPU.", flush=True)\\n    return report\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--staged", action="store_true", help="Explicit sequential BF16 residency profile: 70 GiB available host RAM")\\n    args = parser.parse_args()\\n    check_resources(os.environ.get("H3_MODE", "bf16"), residency="staged" if args.staged else "resident")\\n\', \'server.py\': \'"""Authenticated API adapter. Model execution and durable jobs belong to the manager."""\\nfrom __future__ import annotations\\n\\nimport asyncio\\nimport errno\\nimport hashlib\\nimport hmac\\nimport ipaddress\\nimport json\\nimport logging\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport threading\\nfrom contextlib import asynccontextmanager\\nfrom typing import Annotated, Literal\\nimport uuid\\nfrom urllib.parse import urlsplit\\n\\nfrom fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Query, UploadFile\\nfrom fastapi.middleware.cors import CORSMiddleware\\nfrom fastapi.responses import FileResponse, JSONResponse\\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\\nfrom pydantic import BaseModel, ConfigDict, Field, ValidationError\\n\\nfrom control_protocol import ControlError, rpc_call\\nfrom adapter_config import load_adapter_settings\\nfrom generation_options import generation_capabilities, validate_generation_options\\nfrom runtime_options import RuntimeOptions\\n\\nLOG = logging.getLogger("minimax_api")\\nMIB = 1024 * 1024\\nMAX_FILE_BYTES = 32 * MIB\\nMAX_TOTAL_BYTES = 64 * MIB\\nMAX_REQUEST_BYTES = MAX_TOTAL_BYTES + MIB\\nMAX_CONTROL_BYTES = 64 * 1024\\nEXTENSIONS = {\\n    "image": {".jpg", ".jpeg", ".png", ".webp"},\\n    "video": {".mp4", ".mov", ".webm"},\\n    "audio": {".wav", ".mp3", ".m4a", ".flac", ".ogg"},\\n}\\nJOB_ID = re.compile(r"^[a-f0-9]{32}$")\\nsecurity = HTTPBearer()\\n\\n# Public receipts are an explicit projection, not arbitrary owner-provided JSON.\\n# Keep this contract independent of the model module: the API must not load it.\\nPUBLIC_STAGE_CONDITIONS = ("instance_matches", "attempt_owned", "attempt_live", "model_loaded", "worker_alive",\\n    "capability_available", "admission_confirmed", "predecessor_completed", "stage_not_started",\\n    "cancel_flag_clear", "cancel_event_clear", "reservation_owned", "budget_grant_held")\\nPUBLIC_GATE_CONDITIONS = {\\n    "manager.dispatch": ("manager_open", "job_queued", "attempt_unassigned", "model_identity_current",\\n        "model_observation_fresh", "model_process_alive", "model_ready", "worker_alive", "worker_accepting", "supervisor_alive"),\\n    "model.execute": ("instance_matches", "request_validated", "model_loaded", "worker_alive", "capability_available",\\n        "not_draining", "not_stopping", "attempt_available", "job_unowned", "artifact_absent",\\n        "generation_options_valid", "resources_reserved"),\\n    **{"model.stage." + stage: PUBLIC_STAGE_CONDITIONS for stage in ("prepare", "gpu", "encode_validate")},\\n}\\n\\n\\ndef public_gate_receipt(value, expected_gate):\\n    """Expose known decision facts; no free-form context, parameters or paths."""\\n    if not isinstance(value, dict) or value.get("gate_id") != expected_gate or expected_gate not in PUBLIC_GATE_CONDITIONS:\\n        return None\\n    required = PUBLIC_GATE_CONDITIONS[expected_gate]\\n    raw_conditions = value.get("conditions")\\n    raw_conditions = raw_conditions if isinstance(raw_conditions, dict) else {}\\n    states = {"true", "false", "unknown", "conflict"}\\n    conditions = {name: raw_conditions.get(name) if type(raw_conditions.get(name)) is str\\n                  and raw_conditions[name] in states else "unknown" for name in required}\\n    result = {"gate_id": expected_gate, "conditions": conditions,\\n              "allowed": value.get("allowed") if type(value.get("allowed")) is bool else None}\\n    if result["allowed"] is True and any(state != "true" for state in conditions.values()):\\n        # Do not publish a positive authorization whose required evidence is\\n        # missing, malformed or contradictory; keep uncertainty visible.\\n        result["allowed"] = None\\n    for name in ("schema_version", "gate_version"):\\n        if isinstance(value.get(name), str) and re.fullmatch(r"[0-9]{1,3}\\\\.[0-9]{1,3}", value[name]):\\n            result[name] = value[name]\\n    if isinstance(value.get("decision_id"), str) and JOB_ID.fullmatch(value["decision_id"]):\\n        result["decision_id"] = value["decision_id"]\\n    observed = value.get("observed_at")\\n    if type(observed) in {int, float} and math.isfinite(observed) and observed >= 0:\\n        result["observed_at"] = observed\\n    monotonic = value.get("monotonic_ns")\\n    if type(monotonic) is int and 0 <= monotonic < 2 ** 63:\\n        result["monotonic_ns"] = monotonic\\n    blocked = value.get("blocked")\\n    if isinstance(blocked, list):\\n        result["blocked"] = [name for name in required if name in blocked]\\n    raw = value.get("context")\\n    context = {}\\n    if isinstance(raw, dict):\\n        for name in ("manager_instance_id", "model_instance_id", "instance_id", "job_id", "attempt_id",\\n                     "budget_instance_id", "reservation_id", "lease_id", "budget_decision_id"):\\n            if isinstance(raw.get(name), str) and JOB_ID.fullmatch(raw[name]):\\n                context[name] = raw[name]\\n        expected_source = "global_manager.py" if expected_gate == "manager.dispatch" else "model_container.py"\\n        if raw.get("source_file") == expected_source:\\n            context["source_file"] = expected_source\\n        if isinstance(raw.get("source_sha256"), str) and re.fullmatch(r"[a-f0-9]{64}", raw["source_sha256"]):\\n            context["source_sha256"] = raw["source_sha256"]\\n        if type(raw.get("pipeline")) is bool:\\n            context["pipeline"] = raw["pipeline"]\\n        if raw.get("budget_scope") in ("stage_budget", "legacy_single_active"):\\n            context["budget_scope"] = raw["budget_scope"]\\n        if raw.get("stage") in ("prepare", "gpu", "encode_validate"):\\n            context["stage"] = raw["stage"]\\n    result["context"] = context\\n    return result\\n\\n\\ndef public_condition_fields(record):\\n    fields = {}\\n    if "condition_gate" in record:\\n        fields["condition_gate"] = public_gate_receipt(record["condition_gate"], "manager.dispatch")\\n    if "execution_condition_gates" in record:\\n        raw = record["execution_condition_gates"]\\n        fields["execution_condition_gates"] = ({\\n            name: public_gate_receipt(raw[name], name) for name in PUBLIC_GATE_CONDITIONS\\n            if name.startswith("model.") and name in raw\\n        } if isinstance(raw, dict) else None)\\n    if "stage_completions" in record:\\n        raw = record["stage_completions"]\\n        fields["stage_completions"] = ({\\n            name: raw.get(name) if type(raw.get(name)) is bool else None\\n            for name in ("prepare", "gpu", "encode_validate")\\n        } if isinstance(raw, dict) else None)\\n    if "cancellation_requested" in record:\\n        fields["cancellation_requested"] = (record["cancellation_requested"]\\n            if type(record["cancellation_requested"]) is bool else None)\\n    return fields\\nDEFAULT_CORS_ORIGINS = (\\n    "https://zjj1280637679-ship-it.github.io",\\n    "http://localhost:8000",\\n    "http://127.0.0.1:8000",\\n)\\n\\n\\ndef _cors_allowed_origins() -> list[str]:\\n    """Explicit browser origins, independent of Bearer authorization.\\n\\n    H3_CORS_ORIGINS replaces the defaults; an empty value disables cross-origin\\n    reads. Public origins require HTTPS, while HTTP is restricted to loopback.\\n    """\\n    configured = os.environ.get("H3_CORS_ORIGINS")\\n    values = DEFAULT_CORS_ORIGINS if configured is None else configured.split(",")\\n    origins = []\\n    for value in values:\\n        value = value.strip()\\n        if not value:\\n            continue\\n        try:\\n            if "*" in value or any(char.isspace() or ord(char) < 32 for char in value):\\n                raise ValueError()\\n            parsed = urlsplit(value)\\n            hostname, port = parsed.hostname, parsed.port\\n            if (parsed.scheme not in {"https", "http"} or not hostname\\n                    or parsed.username is not None or parsed.password is not None\\n                    or parsed.path not in {"", "/"} or parsed.query or parsed.fragment\\n                    or (port is not None and not 1 <= port <= 65535)):\\n                raise ValueError()\\n            try:\\n                address = ipaddress.ip_address(hostname)\\n            except ValueError:\\n                address = None\\n                hostname = hostname.encode("idna").decode("ascii").lower()\\n                if len(hostname) > 253 or not re.fullmatch(\\n                    r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?(?:\\\\.[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?)*",\\n                    hostname,\\n                ):\\n                    raise ValueError()\\n            if parsed.scheme == "http" and not (hostname == "localhost" or (address and address.is_loopback)):\\n                raise ValueError()\\n            host = f"[{address.compressed}]" if address and address.version == 6 else hostname\\n            suffix = f":{port}" if port is not None and port != (443 if parsed.scheme == "https" else 80) else ""\\n            origin = f"{parsed.scheme}://{host}{suffix}"\\n        except (ValueError, UnicodeError):\\n            raise RuntimeError("H3_CORS_ORIGINS must contain explicit HTTPS origins or loopback HTTP origins; wildcards, credentials and URL paths are not allowed.") from None\\n        if origin not in origins:\\n            origins.append(origin)\\n    return origins\\n\\n\\nclass RequestGuard:\\n    """Authenticate before multipart parsing and bound actual incoming bytes."""\\n\\n    def __init__(self, app, api_key: str):\\n        self.app, self.api_key = app, api_key\\n\\n    async def __call__(self, scope, receive, send):\\n        if scope["type"] != "http" or not scope["path"].startswith("/v1/"):\\n            return await self.app(scope, receive, send)\\n        headers = dict(scope.get("headers", []))\\n        authorization = headers.get(b"authorization", b"").decode("latin1")\\n        scheme, _, token = authorization.partition(" ")\\n        if scheme.lower() != "bearer" or not hmac.compare_digest(token.encode("latin1"), self.api_key.encode("ascii")):\\n            return await JSONResponse({"detail": "Bearer API key required"}, 401,\\n                                      headers={"WWW-Authenticate": "Bearer"})(scope, receive, send)\\n        limit = MAX_REQUEST_BYTES if scope["path"] == "/v1/jobs" and scope["method"] == "POST" else MAX_CONTROL_BYTES\\n        try:\\n            content_length = int(headers.get(b"content-length", b"0"))\\n            if content_length < 0:\\n                raise ValueError\\n        except ValueError:\\n            return await JSONResponse({"detail": "Invalid Content-Length"}, 400)(scope, receive, send)\\n        if content_length > limit:\\n            return await JSONResponse({"detail": "Request exceeds body limit"}, 413)(scope, receive, send)\\n        received = 0\\n\\n        async def limited_receive():\\n            nonlocal received\\n            message = await receive()\\n            received += len(message.get("body", b""))\\n            if received > limit:\\n                raise HTTPException(413, "Request exceeds body limit")\\n            return message\\n\\n        await self.app(scope, limited_receive, send)\\n\\n\\nclass ControlRequest(BaseModel):\\n    model_config = ConfigDict(extra="forbid")\\n    action: Literal["drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"]\\n    force: bool = Field(False, strict=True)\\n    command_id: str | None = Field(None, min_length=1, max_length=128, pattern=r"^[A-Za-z0-9._:-]+$")\\n\\n\\nclass DriveArchiveReceipt(BaseModel):\\n    model_config = ConfigDict(extra="forbid")\\n    file_id: str = Field(min_length=1, max_length=128, pattern=r"^[A-Za-z0-9_-]+$")\\n    sha256: str = Field(pattern=r"^[a-f0-9]{64}$")\\n    bytes: int = Field(gt=0, strict=True)\\n\\n\\ndef _watch_parent(parent_pid, stop, exit_process=os._exit, process_factory=None, expected_creation=None):\\n    # Windows venv redirectors are legitimate intermediate OS parents. Ownership\\n    # is the supplied manager process generation, not direct getppid equality.\\n    import psutil\\n    factory = process_factory or psutil.Process\\n    try:\\n        if expected_creation is None:\\n            expected_creation = factory(parent_pid).create_time()\\n    except psutil.Error:\\n        exit_process(70)\\n        return\\n    while not stop.wait(1):\\n        try:\\n            process = factory(parent_pid)\\n            alive = (process.is_running() and process.status() != psutil.STATUS_ZOMBIE\\n                     and process.create_time() == expected_creation)\\n        except psutil.Error:\\n            alive = False\\n        if not alive:\\n            exit_process(70)\\n            return\\n\\n\\ndef create_app(controller=None, api_key: str | None = None, output_dir: Path | None = None):\\n    """API creation never initializes CUDA, a model, or a second job-state owner."""\\n    token = api_key if api_key is not None else os.environ.get("H3_API_KEY", "")\\n    if len(token) < 32 or not token.isascii() or any(c.isspace() for c in token):\\n        raise RuntimeError("Set H3_API_KEY to a random ASCII token of at least 32 characters.")\\n    instance_id = os.environ.get("H3_INSTANCE_ID") or uuid.uuid4().hex\\n    root = Path(output_dir or os.environ.get("H3_OUTPUT_DIR", "./h3_outputs")).resolve()\\n    private_key = os.environ.get("H3_CONTROL_KEY", "")\\n    if controller is None:\\n        manager_url = os.environ.get("H3_MANAGER_URL", "")\\n        if not manager_url or len(private_key) < 32:\\n            raise RuntimeError("The API requires its manager connection and private control token.")\\n        controller = lambda operation, payload: rpc_call(manager_url, private_key, operation, payload)\\n    # Only concurrent transport writes are bounded by count; durable jobs/outbox\\n    # records have no count limit. Disk bytes govern their storage admission.\\n    upload_lock = threading.Lock()\\n    upload_state = {"active": 0}\\n    disk_reserve_bytes = max(0, int(os.environ.get("H3_DISK_RESERVE_BYTES", 64 * MIB)))\\n    stopped = threading.Event()\\n\\n    def unknown_count():\\n        if not root.exists():\\n            return 0\\n        return sum(1 for directory in root.iterdir()\\n                   if JOB_ID.fullmatch(directory.name) and not directory.is_symlink()\\n                   and (directory / "submission-uncertain.json").is_file())\\n\\n    def acquire_upload():\\n        with upload_lock:\\n            if upload_state["active"] >= 3:\\n                return False\\n            if shutil.disk_usage(root).free < disk_reserve_bytes + (upload_state["active"] + 1) * MAX_TOTAL_BYTES:\\n                raise HTTPException(507, "Insufficient disk space for reference upload")\\n            upload_state["active"] += 1\\n            return True\\n\\n    def release_upload():\\n        with upload_lock:\\n            upload_state["active"] -= 1\\n\\n    def transport_status():\\n        with upload_lock:\\n            return {"active_uploads": upload_state["active"], "unknown_submissions": unknown_count(),\\n                    "concurrent_upload_limit": 3, "durable_submission_count_limit": None,\\n                    "disk_reserve_bytes": disk_reserve_bytes, "max_reference_bytes_per_request": MAX_TOTAL_BYTES}\\n\\n    def redact(value):\\n        if isinstance(value, dict):\\n            forbidden = {"api_key", "control_key", "private_key", "internal_key", "token", "authorization", "manager_url"}\\n            return {key: redact(item) for key, item in value.items() if key.lower() not in forbidden}\\n        if isinstance(value, list):\\n            return [redact(item) for item in value]\\n        if isinstance(value, str):\\n            for secret in (token, private_key):\\n                if secret:\\n                    value = value.replace(secret, "[redacted]")\\n        return value\\n\\n    def call(operation, payload=None):\\n        try:\\n            request = dict(payload or {})\\n            if operation in {"submit", "cancel", "control", "archive"}:\\n                request["expected_instance_id"] = instance_id\\n            result = controller(operation, request)\\n            if not isinstance(result, dict):\\n                raise TypeError("Invalid manager response")\\n            return result\\n        except ControlError as error:\\n            status = getattr(error, "status", 503)\\n            status = status if isinstance(status, int) and 400 <= status <= 599 else 503\\n            # Transport/internal failures are not evidence of rejection; callers\\n            # reconcile uncertain submissions before removing their inputs.\\n            if status >= 500 and status != 507:\\n                raise HTTPException(503, "Manager unavailable; query the command or job before retrying") from None\\n            detail = str(redact(str(error)))[:500]\\n            raise HTTPException(status, detail or "Manager rejected the request") from None\\n        except Exception as error:\\n            LOG.warning("Manager operation failed: %s (%s)", operation, type(error).__name__)\\n            raise HTTPException(503, "Manager unavailable; query the command or job before retrying") from None\\n\\n    def valid_id(job_id):\\n        if not JOB_ID.fullmatch(job_id):\\n            raise HTTPException(404, "Task does not exist")\\n\\n    def public_job(record):\\n        identity = record.get("id", record.get("job_id"))\\n        if not isinstance(identity, str) or not JOB_ID.fullmatch(identity):\\n            raise HTTPException(503, "Manager returned an invalid task identity")\\n        record_instance = record.get("instance_id")\\n        if "instance_id" in record:\\n            if record_instance != instance_id:\\n                raise HTTPException(503, "Manager returned a task from a different instance")\\n        else:\\n            # Compatibility with an already running older manager: bind only\\n            # after an authenticated current-manager identity readback. Do not\\n            # replace an explicit mismatch or infer identity from a job ID.\\n            state = call("status")\\n            if state.get("instance_id") != instance_id:\\n                raise HTTPException(503, "Manager identity is not confirmed for this task")\\n        allowed = {"status", "created_at", "started_at", "finished_at", "error", "actual_seconds",\\n                   "generation_parameters", "generation_options", "lifecycle", "metrics", "active_phase", "attempt_id", "instance_id",\\n                   "cancel_requested", "cleanup_complete", "replayed", "queue_position", "dispatch_state", "dispatch_reason", "waiting_for"}\\n        result = {"id": identity, **{key: record[key] for key in allowed if key in record}}\\n        result.update(public_condition_fields(record))\\n        result["instance_id"] = instance_id\\n        if result.get("status") not in {"queued", "running", "cancel_requested", "succeeded", "failed", "cancelled"}:\\n            raise HTTPException(503, "Manager returned an invalid task state")\\n        if isinstance(record.get("artifact"), dict):\\n            result["artifact"] = {key: value for key, value in record["artifact"].items() if key != "path"}\\n        result["status_url"] = f"/v1/jobs/{identity}"\\n        if result["status"] in {"queued", "running", "cancel_requested"}:\\n            result["cancel_url"] = f"/v1/jobs/{identity}/cancel"\\n            result["poll_after_seconds"] = 2\\n        if result["status"] == "succeeded":\\n            result["file_url"] = f"/v1/jobs/{identity}/file"\\n            archive = record.get("archive")\\n            if isinstance(archive, dict) and archive.get("provider") == "google_drive":\\n                try:\\n                    receipt = DriveArchiveReceipt.model_validate({k: archive.get(k) for k in DriveArchiveReceipt.model_fields})\\n                except ValidationError:\\n                    raise HTTPException(503, "Manager returned an invalid archive receipt") from None\\n                download_url = f"https://drive.google.com/uc?export=download&id={receipt.file_id}"\\n                result["archive"] = {\\n                    "provider": "google_drive", **receipt.model_dump(), "download_url": download_url,\\n                    "view_url": f"https://drive.google.com/file/d/{receipt.file_id}/view",\\n                }\\n                result["download_url"] = download_url\\n        return redact(result)\\n\\n    @asynccontextmanager\\n    async def lifespan(app):\\n        root.mkdir(parents=True, exist_ok=True)\\n        watcher = None\\n        parent_pid = os.environ.get("H3_PARENT_PID")\\n        if parent_pid:\\n            parent_pid = int(parent_pid)\\n            if parent_pid <= 1:\\n                raise RuntimeError("Invalid manager parent process identity")\\n            created = os.environ.get("H3_PARENT_CREATED_AT")\\n            expected_creation = float(created) if created is not None else None\\n            watcher = threading.Thread(target=_watch_parent, args=(parent_pid, stopped),\\n                                       kwargs={"expected_creation": expected_creation},\\n                                       name="h3-api-owner-watch", daemon=True)\\n            watcher.start()\\n        try:\\n            yield\\n        finally:\\n            stopped.set()\\n            if watcher:\\n                watcher.join(timeout=2)\\n\\n    adapter_settings = load_adapter_settings()\\n    runtime_options = RuntimeOptions.from_env()\\n    app = FastAPI(title="MiniMax-H3 Ref2VA API", version="0.4.0", lifespan=lifespan,\\n                  description="Independent authenticated API adapter; durable scheduling is owned by the global manager.")\\n    app.add_middleware(RequestGuard, api_key=token)\\n    # Starlette wraps the last-added middleware outermost. Handle valid browser\\n    # preflights before RequestGuard, and attach CORS headers even to its 401s.\\n    # Every actual /v1 request still traverses the unchanged Bearer guard.\\n    app.add_middleware(\\n        CORSMiddleware,\\n        allow_origins=_cors_allowed_origins(),\\n        allow_credentials=False,\\n        allow_methods=["GET", "POST"],\\n        allow_headers=["Authorization", "Content-Type", "Idempotency-Key", "Range"],\\n        expose_headers=["Content-Disposition", "Content-Length", "Content-Range", "Accept-Ranges", "ETag", "Retry-After"],\\n        max_age=600,\\n    )\\n\\n    def require_auth(credentials: Annotated[HTTPAuthorizationCredentials, Depends(security)]):\\n        if not hmac.compare_digest(credentials.credentials.encode("utf-8"), token.encode("ascii")):\\n            raise HTTPException(401, "Invalid API key", headers={"WWW-Authenticate": "Bearer"})\\n\\n    @app.get("/health/live")\\n    def live():\\n        return {"api_ready": True, "alive": True, "instance_id": instance_id}\\n\\n    @app.get("/health")\\n    def health():\\n        try:\\n            state = call("status")\\n        except HTTPException:\\n            return JSONResponse({"api_ready": True, "manager_ready": False, "model_ready": False,\\n                                 "ready": False, "instance_id": instance_id}, 503)\\n        same_instance = state.get("instance_id") == instance_id\\n        return JSONResponse({"api_ready": True, "manager_ready": same_instance,\\n                             "model_ready": same_instance and state.get("model_ready") is True,\\n                             "ready": same_instance and state.get("ready") is True,\\n                             "instance_id": instance_id}, 200 if same_instance else 503)\\n\\n    @app.get("/v1/system", dependencies=[Depends(require_auth)])\\n    def system():\\n        local = transport_status()\\n        try:\\n            state = redact(call("status"))\\n        except HTTPException:\\n            return JSONResponse({"instance_id": instance_id, "api_ready": True, "manager_ready": False,\\n                                 "ready": False, "model_ready": False, "api_local": local}, 503)\\n        return {**state, "api_local": local}\\n\\n    @app.get("/v1/capabilities", dependencies=[Depends(require_auth)])\\n    def capabilities():\\n        # Configuration admits queued work while model_ready is still false.\\n        # Never infer successful loading from a download percentage.\\n        state = call("status")\\n        same_instance = state.get("instance_id") == instance_id\\n        return {**generation_capabilities(model_ready=same_instance and state.get("model_ready") is True,\\n                    settings=adapter_settings, runtime_options=runtime_options), "instance_id": instance_id}\\n\\n    @app.get("/v1/events", dependencies=[Depends(require_auth)])\\n    def events(after: Annotated[int, Query(ge=0)] = 0, limit: Annotated[int, Query(ge=1, le=1000)] = 100):\\n        return redact(call("events", {"after": after, "limit": limit}))\\n\\n    @app.get("/v1/commands/{command_id}", dependencies=[Depends(require_auth)])\\n    def command(command_id: str):\\n        if not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", command_id):\\n            raise HTTPException(404, "Command does not exist")\\n        return redact(call("command", {"command_id": command_id}))\\n\\n    @app.post("/v1/control", status_code=202, dependencies=[Depends(require_auth)])\\n    def control(request: ControlRequest):\\n        if request.action in {"stop_api", "shutdown"} and request.command_id is None:\\n            raise HTTPException(422, "A client command_id is required when stopping this API; query it through the manager")\\n        return redact(call("control", request.model_dump(exclude_none=True)))\\n\\n    @app.post("/v1/jobs", status_code=202, dependencies=[Depends(require_auth)])\\n    async def submit(\\n        files: Annotated[list[UploadFile], File(description="Reference files in semantic order")],\\n        kinds: Annotated[str, Form(description=\\\'JSON array matching files, e.g. ["image","audio"]\\\')],\\n        prompt: Annotated[str, Form(min_length=1, max_length=4000)],\\n        seconds: Annotated[float, Form(ge=5, le=15)] = 5,\\n        width: Annotated[int, Form(ge=256, le=1344)] = 960,\\n        height: Annotated[int, Form(ge=256, le=1344)] = 544,\\n        steps: Annotated[int, Form(ge=2, le=50)] = 50,\\n        attention_backend: Annotated[str, Form()] = "auto",\\n        acceleration_lora: Annotated[str, Form()] = "none",\\n        seed: Annotated[int, Form(ge=0, le=2147483647)] = 42,\\n        idempotency_key: Annotated[str | None, Header(alias="Idempotency-Key", max_length=128)] = None,\\n    ):\\n        directory = None\\n        reserved = retain_inputs = False\\n        try:\\n            if not prompt.strip() or not math.isfinite(seconds):\\n                raise HTTPException(422, "Prompt and duration must be valid")\\n            try:\\n                validate_generation_options({"attention_backend": attention_backend,\\n                    "acceleration_lora": acceleration_lora, "steps": steps}, adapter_settings)\\n            except ValueError as error:\\n                raise HTTPException(422, str(error)) from None\\n            if idempotency_key is not None and not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", idempotency_key):\\n                raise HTTPException(422, "Invalid Idempotency-Key")\\n            if width % 32 or height % 32 or width * height > 1344 * 768:\\n                raise HTTPException(422, "Dimensions must be multiples of 32, at most 1,032,192 pixels")\\n            frames = 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17)\\n            if frames > 360:\\n                raise HTTPException(422, "Requested duration rounds beyond 15 seconds; use at most 14.375 seconds")\\n            try:\\n                modalities = json.loads(kinds)\\n            except (ValueError, TypeError):\\n                raise HTTPException(422, "kinds must be a JSON array") from None\\n            if not isinstance(modalities, list) or not 1 <= len(files) <= 12 or len(modalities) != len(files):\\n                raise HTTPException(422, "Supply 1-12 files and one kind per file")\\n            if any(not isinstance(kind, str) or kind not in EXTENSIONS for kind in modalities):\\n                raise HTTPException(422, "Kinds must be image, video or audio")\\n            if modalities.count("image") > 9 or modalities.count("video") > 3 or modalities.count("audio") > 3:\\n                raise HTTPException(422, "At most 9 images, 3 videos and 3 audio references")\\n            if all(kind == "audio" for kind in modalities):\\n                raise HTTPException(422, "At least one image or video reference is required")\\n            for upload, kind in zip(files, modalities):\\n                if Path(upload.filename or "").suffix.lower() not in EXTENSIONS[kind]:\\n                    raise HTTPException(422, f"Unsupported {kind} file extension")\\n            reserved = await asyncio.to_thread(acquire_upload)\\n            if not reserved:\\n                raise HTTPException(429, "Concurrent upload limit reached; retry after uploads finish", headers={"Retry-After": "5"})\\n            job_id = uuid.uuid4().hex\\n            directory = root / job_id\\n            input_dir = directory / "inputs"\\n            input_dir.mkdir(parents=True, exist_ok=False)\\n            references, content_identity, total = [], [], 0\\n            for index, (upload, kind) in enumerate(zip(files, modalities)):\\n                suffix = Path(upload.filename or "").suffix.lower()\\n                path = input_dir / f"{index:02d}{suffix}"\\n                size, digest = 0, hashlib.sha256()\\n                with path.open("xb") as destination:\\n                    while chunk := await upload.read(MIB):\\n                        size += len(chunk)\\n                        total += len(chunk)\\n                        if size > MAX_FILE_BYTES or total > MAX_TOTAL_BYTES:\\n                            raise HTTPException(413, "Each reference <=32 MiB; total references <=64 MiB")\\n                        digest.update(chunk)\\n                        await asyncio.to_thread(destination.write, chunk)\\n                if not size:\\n                    raise HTTPException(422, "Reference file is empty")\\n                references.append({"kind": kind, "path": str(path)})\\n                content_identity.append({"kind": kind, "sha256": digest.hexdigest(), "bytes": size})\\n            parameters = dict(prompt=prompt, references=references, seconds=seconds,\\n                              width=width, height=height, steps=steps, seed=seed,\\n                              attention_backend=attention_backend, acceleration_lora=acceleration_lora)\\n            semantic_request = {**parameters, "references": content_identity}\\n            fingerprint = hashlib.sha256(json.dumps(semantic_request, sort_keys=True,\\n                                                    separators=(",", ":"), allow_nan=False).encode()).hexdigest()\\n            payload = {"job_id": job_id, "parameters": parameters, "fingerprint": fingerprint,\\n                       "idempotency_key": idempotency_key or job_id}\\n            def uncertain_submission():\\n                marker = {"job_id": job_id, "idempotency_key": payload["idempotency_key"],\\n                          "fingerprint": fingerprint, "state": "submission_outcome_unknown"}\\n                (directory / "submission-uncertain.json").write_text(json.dumps(marker), encoding="utf-8")\\n                return HTTPException(503, {"message": "Submission outcome unknown; query this job before retrying",\\n                                          "job_id": job_id, "status_url": f"/v1/jobs/{job_id}",\\n                                          "idempotency_key": payload["idempotency_key"]})\\n\\n            retain_inputs = True  # A dispatched command may commit before its response arrives.\\n            try:\\n                record = await asyncio.to_thread(call, "submit", payload)\\n            except HTTPException as error:\\n                if error.status_code < 500 or error.status_code == 507:\\n                    retain_inputs = False\\n                    raise\\n                # A dropped response does not prove the durable command failed.\\n                # Query the identity just sent. Preserve inputs if still unknown.\\n                retain_inputs = True\\n                try:\\n                    record = await asyncio.to_thread(call, "job", {"job_id": job_id})\\n                except HTTPException:\\n                    raise uncertain_submission() from None\\n            try:\\n                result = public_job(record)\\n            except HTTPException:\\n                raise uncertain_submission() from None\\n            retain_inputs = result["id"] == job_id\\n            result.setdefault("actual_seconds", frames / 24)\\n            return result\\n        except OSError as error:\\n            if error.errno in {errno.ENOSPC, errno.EDQUOT} and not retain_inputs:\\n                raise HTTPException(507, "Insufficient disk space for reference upload") from None\\n            raise\\n        finally:\\n            for upload in files:\\n                await upload.close()\\n            if directory is not None and not retain_inputs:\\n                await asyncio.to_thread(shutil.rmtree, directory, ignore_errors=True)\\n            if reserved:\\n                release_upload()\\n\\n    def get_record(job_id):\\n        valid_id(job_id)\\n        record = call("job", {"job_id": job_id})\\n        if record.get("id", record.get("job_id")) != job_id:\\n            raise HTTPException(503, "Manager returned a different task identity")\\n        # A durable manager record resolves this upload\\\'s uncertain delivery;\\n        # only the transport marker is removed, never manager-owned inputs.\\n        directory = root / job_id\\n        if not directory.is_symlink():\\n            try:\\n                (directory / "submission-uncertain.json").unlink(missing_ok=True)\\n            except OSError:\\n                LOG.warning("Could not remove resolved upload marker")\\n        return record\\n\\n    @app.get("/v1/jobs/{job_id}", dependencies=[Depends(require_auth)])\\n    def status(job_id: str):\\n        return public_job(get_record(job_id))\\n\\n    @app.post("/v1/jobs/{job_id}/cancel", status_code=202, dependencies=[Depends(require_auth)])\\n    def cancel(job_id: str):\\n        valid_id(job_id)\\n        record = call("cancel", {"job_id": job_id})\\n        if record.get("id", record.get("job_id")) != job_id:\\n            raise HTTPException(503, "Manager returned a different task identity")\\n        return public_job(record)\\n\\n    @app.post("/v1/jobs/{job_id}/drive-archive", dependencies=[Depends(require_auth)])\\n    def register_drive_archive(job_id: str, receipt: DriveArchiveReceipt):\\n        """Record the authenticated owner\\\'s receipt after external upload and readback."""\\n        valid_id(job_id)\\n        record = call("archive", {"job_id": job_id, **receipt.model_dump()})\\n        if record.get("id", record.get("job_id")) != job_id:\\n            raise HTTPException(503, "Manager returned a different task identity")\\n        return public_job(record)\\n\\n    @app.get("/v1/jobs/{job_id}/file", dependencies=[Depends(require_auth)])\\n    def download(job_id: str):\\n        record = get_record(job_id)\\n        if record.get("status") != "succeeded":\\n            raise HTTPException(409, "Video is not ready")\\n        artifact = record.get("artifact")\\n        if not isinstance(artifact, dict) or not isinstance(artifact.get("path"), str):\\n            raise HTTPException(409, "No verified artifact is published")\\n        expected = root / job_id / "result.mp4"\\n        candidate = Path(artifact["path"])\\n        # The manager chooses output identity; even a malformed/stale response\\n        # must not let this public adapter serve arbitrary local files.\\n        if candidate != expected or candidate.is_symlink() or expected.parent.is_symlink():\\n            raise HTTPException(409, "Artifact ownership does not match the task")\\n        try:\\n            if candidate.resolve() != expected or not candidate.is_file():\\n                raise HTTPException(404, "Result was removed")\\n            size = candidate.stat().st_size\\n            if not size or artifact.get("bytes") != size:\\n                raise HTTPException(409, "Artifact size no longer matches its committed manifest")\\n            digest = hashlib.sha256()\\n            with candidate.open("rb") as source:\\n                for chunk in iter(lambda: source.read(MIB), b""):\\n                    digest.update(chunk)\\n            committed_hash = artifact.get("sha256")\\n            if (not isinstance(committed_hash, str) or not re.fullmatch(r"[a-f0-9]{64}", committed_hash)\\n                    or not hmac.compare_digest(digest.hexdigest(), committed_hash)):\\n                raise HTTPException(409, "Artifact content no longer matches its committed manifest")\\n        except OSError:\\n            raise HTTPException(404, "Result was removed or cannot be read") from None\\n        return FileResponse(candidate, media_type="video/mp4", filename=f"{job_id}.mp4",\\n                            headers={"ETag": \\\'"\\\' + digest.hexdigest() + \\\'"\\\',\\n                                     "Cache-Control": "private, no-store"})\\n\\n    return app\\n\', \'model_backend.py\': \'"""Real MiniMax-H3 ref2va inference. No synthetic output or mock fallback."""\\nfrom __future__ import annotations\\n\\nimport gc\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nfrom runtime_options import configure_qwen_offload, measured_load\\nfrom pipeline_backend import PipelineBackendSupport\\nfrom adapter_config import load_adapter_settings, configure_pipeline, apply_adapters, validate_steps\\nfrom download_progress import tracked_model_load\\n\\nMODEL_ID = "MiniMaxAI/MiniMax-H3"\\nMODEL_REVISION = "0919fb684102ef115126612f3fbb84d1027e7317"\\n\\n\\ndef _require_loaded_components(pipe) -> None:\\n    """Diffusers may log component-load failures without propagating them."""\\n    required = [spec.name for spec in pipe.blocks.expected_components]\\n    if not required:\\n        raise RuntimeError("Ref2VA workflow declared no required components.")\\n    missing = [name for name in required if getattr(pipe, name, None) is None]\\n    if missing:\\n        raise RuntimeError(\\n            "MiniMax-H3 component loading was incomplete: " + ", ".join(missing)\\n            + ". Inspect the component loading errors above; API readiness is blocked."\\n        )\\n\\n\\ndef _parameter_kept_in_fp32(parameter_name, keep_fp32) -> bool:\\n    """Mirror Diffusers 0.43 load_model_dict_into_meta\\\'s exact path segments.\\n\\n    Its dtype/device-map helpers use looser substring checks, but they do not\\n    define the dtype assigned by the actual weight loader. In particular,\\n    ``quant_conv`` must not match ``post_quant_conv``.\\n    """\\n    if isinstance(keep_fp32, str):\\n        keep_fp32 = [keep_fp32]\\n    return any(part in parameter_name.split(".") for part in (keep_fp32 or []))\\n\\n\\ndef _audit_parameter_precision(pipe) -> dict:\\n    """Inspect loaded parameter metadata, without copying weights or buffers.\\n\\n    Each official component owns its FP32 exceptions. In particular, MiniMax\\\'s\\n    audio VAE keeps its weights in FP32 even when the pipeline requests BF16.\\n    Match the actual loader\\\'s complete dotted-path segments, not substrings.\\n    """\\n    from collections import Counter\\n    import torch\\n\\n    components = {}\\n    for spec in pipe.blocks.expected_components:\\n        name = spec.name\\n        module = getattr(pipe, name, None)\\n        if not callable(getattr(module, "named_parameters", None)):\\n            continue  # tokenizers, schedulers and processors have no weights\\n        if (any(getattr(module, flag, False) for flag in\\n                ("is_quantized", "is_loaded_in_4bit", "is_loaded_in_8bit"))\\n                or getattr(module, "hf_quantizer", None) is not None):\\n            raise RuntimeError(f"Quantized component is forbidden: {name}")\\n        keep_fp32 = getattr(module, "_keep_in_fp32_modules", None) or []\\n        if isinstance(keep_fp32, str):\\n            keep_fp32 = [keep_fp32]\\n        counts, tensors = Counter(), Counter()\\n        for parameter_name, parameter in module.named_parameters():\\n            full_name = f"{name}.{parameter_name}"\\n            # Quantized tensor subclasses may report their compute dtype as\\n            # BF16. The native loader uses ordinary nn.Parameter objects.\\n            if type(parameter) is not torch.nn.Parameter or getattr(parameter, "is_quantized", False):\\n                raise RuntimeError(f"Non-native or quantized parameter is forbidden: {full_name}")\\n            if getattr(parameter, "is_meta", False):\\n                raise RuntimeError(f"Parameter was not materialized: {full_name}")\\n            expected = torch.float32 if _parameter_kept_in_fp32(parameter_name, keep_fp32) else torch.bfloat16\\n            if parameter.dtype != expected:\\n                raise RuntimeError(\\n                    f"Unexpected precision in {full_name}: {parameter.dtype}, expected {expected}"\\n                )\\n            dtype = str(parameter.dtype)\\n            counts[dtype] += parameter.numel()\\n            tensors[dtype] += 1\\n        if not tensors:\\n            raise RuntimeError(f"No loaded parameters available for precision audit: {name}")\\n        record = {"component": name, "status": "passed", "parameter_counts": dict(counts),\\n                  "tensor_counts": dict(tensors), "keep_in_fp32_modules": list(keep_fp32)}\\n        components[name] = record\\n        print("H3_PRECISION_AUDIT " + json.dumps(record, sort_keys=True), flush=True)\\n    if not components:\\n        raise RuntimeError("No model components available for precision audit")\\n    report = {"status": "passed", "scope": "loaded_parameters_only", "components": components}\\n    print("H3_PRECISION_AUDIT " + json.dumps(\\n        {"status": "passed", "scope": report["scope"], "components": list(components)}, sort_keys=True\\n    ), flush=True)\\n    return report\\n\\n\\ndef aligned_frames(seconds: float) -> int:\\n    frames = 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17)\\n    if not 5 <= seconds <= 15 or not 120 <= frames <= 360:\\n        raise ValueError("Duration must align to 17*n+5 frames within 5–15s; maximum request is 14.375s.")\\n    return frames\\n\\n\\ndef _probe(path: Path, kind: str) -> float:\\n    """Reject oversized media before loading all decoded frames into host RAM."""\\n    if kind == "image":\\n        from PIL import Image\\n        with Image.open(path) as im:\\n            if im.width * im.height > 16_000_000 or getattr(im, "n_frames", 1) != 1:\\n                raise ValueError("Reference images must be static and at most 16 megapixels.")\\n            if not .25 <= im.width / im.height <= 4:\\n                raise ValueError("Reference image aspect ratio must be between 1:4 and 4:1.")\\n            im.verify()\\n        return 0\\n    process = subprocess.run(\\n        ["ffprobe", "-v", "error", "-threads", "1", "-show_format", "-show_streams", "-of", "json", str(path)],\\n        capture_output=True, text=True, timeout=30, check=True,\\n    )\\n    info = json.loads(process.stdout)\\n    streams = info.get("streams", [])\\n    wanted = [s for s in streams if s.get("codec_type") == kind]\\n    if not wanted:\\n        raise ValueError(f"No {kind} stream in reference file.")\\n    stream = wanted[0]\\n    duration = float(stream.get("duration") or info.get("format", {}).get("duration") or 0)\\n    if not math.isfinite(duration) or not 2 <= duration <= 15:\\n        raise ValueError(f"Reference {kind} duration must be known and between 2 and 15s.")\\n    if kind == "video":\\n        from fractions import Fraction\\n        rate = float(Fraction(stream.get("avg_frame_rate", "0")))\\n        if not 0 < rate <= 60 or stream.get("width", 0) * stream.get("height", 0) > 1920 * 1080:\\n            raise ValueError("Reference video must be at most 1080p / 60 fps.")\\n    for audio in (s for s in streams if s.get("codec_type") == "audio"):\\n        if int(audio.get("sample_rate", 0)) > 96000 or int(audio.get("channels", 0)) > 2:\\n            raise ValueError("Reference audio must be mono/stereo and at most 96 kHz.")\\n    return duration\\n\\n\\ndef _normalize_video(path: Path, target: Path | None = None) -> Path:\\n    # Bound decoded memory before MiniMaxH3VideoReference.from_file stacks all frames.\\n    target = target or path.with_name(path.stem + "_24fps.mp4")\\n    subprocess.run([\\n        "ffmpeg", "-nostdin", "-y", "-v", "error", "-threads", "1", "-filter_threads", "1", "-i", str(path),\\n        "-map", "0:v:0", "-map", "0:a:0?", "-t", "15",\\n        "-vf", "scale=960:544:force_original_aspect_ratio=decrease:force_divisible_by=2,fps=24",\\n        "-c:v", "libx264", "-crf", "18", "-preset", "fast", "-pix_fmt", "yuv420p",\\n        "-c:a", "aac", "-ac", "2", "-ar", "32000", "-threads", "1", str(target),\\n    ], check=True, capture_output=True, timeout=180)\\n    return target\\n\\n\\nclass MiniMaxBackend(PipelineBackendSupport):\\n    _reference_probe = staticmethod(_probe)\\n    _normalize_reference = staticmethod(_normalize_video)\\n    _aligned_frames = staticmethod(aligned_frames)\\n\\n    def __init__(self, runtime_options=None):\\n        self._init_runtime(runtime_options)\\n        self.adapter_settings = load_adapter_settings()\\n        self.pipe = None\\n        self.precision_audit = None\\n\\n    @measured_load\\n    @tracked_model_load\\n    def load(self) -> None:\\n        self.precision_audit = None\\n        from preflight import check_resources\\n        resources = check_resources()\\n        import torch\\n        from diffusers import ComponentsManager, ModularPipeline\\n\\n        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\\n            raise RuntimeError("A CUDA GPU with native BF16 support is required.")\\n        revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\\n        torch.backends.cuda.matmul.allow_tf32 = True\\n        strategy = resources["strategy"]\\n        print(f"A100 inference: native BF16; memory strategy={strategy}", flush=True)\\n        if strategy == "component_cpu_offload":\\n            if self.runtime_options.denoiser_offload != "stream":\\n                raise ValueError("H3_DENOISER_OFFLOAD=sync applies only to group offload; this profile uses component offload.")\\n            manager = ComponentsManager()\\n            pipe = ModularPipeline.from_pretrained(\\n                MODEL_ID, revision=revision, workflow="ref2va", components_manager=manager,\\n            )\\n            configure_pipeline(pipe, self.adapter_settings)\\n            with self._stage("load.components"):\\n                pipe.load_components(dtype=torch.bfloat16, revision=revision)\\n            _require_loaded_components(pipe)\\n            with self._stage("configure.adapters"):\\n                apply_adapters(pipe, self.adapter_settings)\\n            with self._stage("audit.parameter_precision"):\\n                self.precision_audit = _audit_parameter_precision(pipe)\\n            self.runtime_options.configure_attention(pipe.transformer_ref)\\n            manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="12GB")\\n        elif strategy == "group_cpu_offload":\\n            pipe = ModularPipeline.from_pretrained(MODEL_ID, revision=revision, workflow="ref2va")\\n            configure_pipeline(pipe, self.adapter_settings)\\n            with self._stage("load.components"):\\n                pipe.load_components(dtype=torch.bfloat16, revision=revision)\\n            _require_loaded_components(pipe)\\n            pipe.transformer_ref.requires_grad_(False)\\n            with self._stage("configure.adapters"):\\n                apply_adapters(pipe, self.adapter_settings)\\n            with self._stage("audit.parameter_precision"):\\n                self.precision_audit = _audit_parameter_precision(pipe)\\n            self.runtime_options.configure_attention(pipe.transformer_ref)\\n            pipe.transformer_ref.enable_group_offload(**self.runtime_options.group_offload_kwargs(torch))\\n            configure_qwen_offload(pipe.text_encoder)\\n            # A100 40GB cannot hold the 61.7GB BF16 denoiser as one component.\\n            # Stream BF16 blocks without quantization; PCIe bandwidth affects speed.\\n            pipe.vae.to("cuda")\\n            pipe.audio_vae.to("cuda")\\n        else:\\n            raise ValueError(f"Unsupported A100 memory strategy: {strategy}")\\n        self.pipe = pipe\\n\\n    def validate_request(self, *, steps, attention_backend=None, acceleration_lora=None):\\n        return self.validate_generation_request(steps=steps, attention_backend=attention_backend,\\n                                                acceleration_lora=acceleration_lora)\\n\\n    def _infer_raw(self, prepared):\\n        """The original BF16 workflow stays wholly inside the exclusive GPU stage."""\\n        if self.pipe is None:\\n            raise RuntimeError("The model has not loaded.")\\n        import torch\\n        p = prepared.parameters\\n        return self.pipe(\\n            prompt=p["prompt"], references=prepared.references, num_frames=aligned_frames(p["seconds"]),\\n            height=p["height"], width=p["width"], num_inference_steps=p["steps"],\\n            generator=torch.Generator(device="cpu").manual_seed(p["seed"]),\\n            output_type="pil", output=["videos", "audio", "sampling_rate"],\\n        )\\n\', \'staged_backend.py\': \'"""Sequential component residency: single-image A100 40GB acceptance passed.\\n\\nVerified on Colab A100 40GB with about 83.47 GiB host RAM: public submission,\\npolling, generation, download, and full audiovisual decode. The 512x288 test\\nused one image, a 5s request/124 output frames, and 8 steps (7 denoising rounds).\\nThe backbone stays BF16 while official _keep_in_fp32_modules layers remain FP32.\\nNo quantization or fake output exists. This is an integration test, not a\\nquality benchmark or acceptance of video/audio reference combinations.\\nOptional runtime modes, adapters, and media scheduling added afterward have not\\nbeen GPU-retested; their defaults preserve the accepted inference choices.\\n\\nTested with Python 3.13.16, PyTorch 2.14.1+cu130, Diffusers 0.43.0, and\\nTransformers 5.7.0. The separately pinned PyTorch 2.10 install has not been\\nGPU-tested. A100 80GB resident acceptance is recorded separately in README.md;\\nit does not establish acceptance of this refactored staged pipeline.\\n"""\\nfrom __future__ import annotations\\n\\nimport gc\\nimport os\\nfrom pathlib import Path\\nimport time\\n\\nfrom model_backend import MODEL_ID, MODEL_REVISION, _normalize_video, _probe, aligned_frames, _parameter_kept_in_fp32, _audit_parameter_precision\\nfrom runtime_options import configure_qwen_offload, measured_load\\nfrom pipeline_backend import PipelineBackendSupport, GIB\\nfrom adapter_config import load_adapter_settings, configure_pipeline, apply_adapters, validate_steps\\nfrom download_progress import tracked_model_load\\n\\n\\nclass StagedMiniMaxBackend(PipelineBackendSupport):\\n    """Exclusive GPU inference; Qwen and transformer_ref never co-reside."""\\n    pipeline_model_peak_ram_bytes = 76 * GIB\\n    resource_residency = "staged"\\n    _reference_probe = staticmethod(_probe)\\n    _normalize_reference = staticmethod(_normalize_video)\\n    _aligned_frames = staticmethod(aligned_frames)\\n\\n    def __init__(self, runtime_options=None):\\n        self._init_runtime(runtime_options)\\n        self.adapter_settings = load_adapter_settings()\\n        self.pipe = None\\n        self.revision = os.environ.get("H3_MODEL_REVISION", MODEL_REVISION)\\n        self._phase = None\\n        self.precision_audit = None\\n\\n    @staticmethod\\n    def _log(message):\\n        print(f"[{time.strftime(\\\'%H:%M:%S\\\')}] BF16 staged: {message}", flush=True)\\n\\n    def _load_required(self, names):\\n        import torch\\n        # Diffusers 0.43 load_components logs some failures instead of raising.\\n        # Load individually to avoid its temporary dict retaining multiple newly\\n        # loaded components before registering them.\\n        for name in names:\\n            if getattr(self.pipe, name, None) is None:\\n                self._log(f"loading {name}")\\n                with self._stage(f"load.{name}"):\\n                    self.pipe.load_components(names=name, dtype=torch.bfloat16, revision=self.revision)\\n            if getattr(self.pipe, name, None) is None:\\n                raise RuntimeError(f"Required MiniMax-H3 component failed to load: {name}")\\n            if name in {"text_encoder", "transformer_ref"}:\\n                from collections import Counter\\n                module = getattr(self.pipe, name)\\n                keep_fp32 = getattr(module, "_keep_in_fp32_modules", []) or []\\n                counts = Counter()\\n                for parameter_name, parameter in module.named_parameters():\\n                    counts[str(parameter.dtype)] += parameter.numel()\\n                    expected = torch.float32 if _parameter_kept_in_fp32(parameter_name, keep_fp32) else torch.bfloat16\\n                    if parameter.dtype != expected:\\n                        raise RuntimeError(f"Unexpected precision in {name}.{parameter_name}: {parameter.dtype}, expected {expected}")\\n                self._log(f"{name} original precision verified, parameter counts={dict(counts)}")\\n\\n    def _switch(self, target):\\n        if self._phase != target:\\n            with self._stage(f"switch.{target}"):\\n                self._switch_impl(target)\\n\\n    def _switch_impl(self, target):\\n        import torch\\n        other = "text_encoder" if target == "denoise" else "transformer_ref"\\n        self._phase = None  # a failed load must never leave a stale ready phase\\n        self._log(f"unloading {other} before {target}")\\n        self.pipe.unload_components(other)\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n        if target == "encode":\\n            self._load_required(["text_encoder"])\\n            configure_qwen_offload(self.pipe.text_encoder)\\n            self._log("Qwen vision/embeddings/rotary on CUDA; decoder layers offloaded as whole BF16 blocks")\\n        else:\\n            self._load_required(["transformer_ref"])\\n            self.pipe.transformer_ref.requires_grad_(False)\\n            with self._stage("configure.adapters"):\\n                apply_adapters(self.pipe, self.adapter_settings)\\n            self.precision_audit = _audit_parameter_precision(self.pipe)\\n            self.runtime_options.configure_attention(self.pipe.transformer_ref)\\n            selected = getattr(self, "_active_generation_options", None)\\n            if selected is not None:\\n                self._apply_generation_transformer(selected)\\n            self.pipe.transformer_ref.enable_group_offload(**self.runtime_options.group_offload_kwargs(torch))\\n            self._log(f"denoiser offload={self.runtime_options.denoiser_offload}; attention={self.runtime_options.attention_backend}")\\n        self._phase = target\\n        self._log(f"{target} stage loaded; other large component released")\\n\\n    @measured_load\\n    @tracked_model_load\\n    def load(self):\\n        import torch\\n        from diffusers import ModularPipeline\\n        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\\n            raise RuntimeError("Native-BF16 CUDA device is required")\\n        if "A100" not in torch.cuda.get_device_name(0).upper():\\n            raise RuntimeError("This acceptance build targets an NVIDIA A100")\\n        import psutil, shutil\\n        if psutil.virtual_memory().available < 70 * 1024**3:\\n            raise RuntimeError("Staged BF16 requires at least 70 GiB available host RAM before loading")\\n        if torch.cuda.get_device_properties(0).total_memory < 38 * 1024**3:\\n            raise RuntimeError("Staged BF16 requires an A100 40 GB or larger")\\n        cache = Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser()\\n        cache.mkdir(parents=True, exist_ok=True)\\n        if shutil.disk_usage(cache).free < 140 * 1024**3 and os.environ.get("H3_SKIP_DISK_CHECK") != "1":\\n            raise RuntimeError("Staged BF16 needs 140 GiB free cache disk on the first download")\\n        self.pipe = ModularPipeline.from_pretrained(\\n            MODEL_ID, revision=self.revision, workflow="ref2va",\\n        )\\n        configure_pipeline(self.pipe, self.adapter_settings)\\n        names = list(self.pipe.blocks.sub_blocks)\\n        if names[:3] != ["before_encode", "text_encoder", "vae_encoder"]:\\n            raise RuntimeError(f"Unexpected Ref2VA block topology: {names}")\\n        if not any(name.startswith("denoise.") for name in names) or not any(name.startswith("decode.") for name in names):\\n            raise RuntimeError(f"Unexpected Ref2VA block topology: {names}")\\n        self._load_required(["tokenizer", "processor", "scheduler", "audio_scheduler", "vae", "audio_vae"])\\n        self.pipe.vae.requires_grad_(False).to("cuda")\\n        self.pipe.audio_vae.requires_grad_(False).to("cuda")\\n        # Actually load both large components during startup, one at a time. This\\n        # also downloads all required weights before the API advertises readiness.\\n        self._switch("denoise")\\n        self._switch("encode")\\n        self._log("both large components loaded successfully in turn; encoder stage ready")\\n\\n    @staticmethod\\n    def _cpu(value):\\n        import torch\\n        if isinstance(value, torch.Tensor):\\n            return value.detach().cpu()\\n        if isinstance(value, list):\\n            return [StagedMiniMaxBackend._cpu(item) for item in value]\\n        if isinstance(value, tuple):\\n            return tuple(StagedMiniMaxBackend._cpu(item) for item in value)\\n        if isinstance(value, dict):\\n            return {key: StagedMiniMaxBackend._cpu(item) for key, item in value.items()}\\n        return value\\n\\n    def _run(self, state, stage):\\n        import torch\\n        # The resolved workflow has flat leaf keys: before_encode, text_encoder,\\n        # vae_encoder, denoise.prepare_layout ... denoise.after_denoise,\\n        # decode.video, decode.audio. Calling the documented block contract\\n        # (components, PipelineState) avoids a deepcopy of large reference media.\\n        with torch.inference_mode():\\n            for name, block in self.pipe.blocks.sub_blocks.items():\\n                selected = name in {"before_encode", "text_encoder", "vae_encoder"} if stage == "encode" else name.startswith(stage + ".")\\n                if selected:\\n                    self._log(f"running {name}")\\n                    with self._stage(f"block.{name}"):\\n                        _, state = block(self.pipe, state)\\n        return state\\n\\n    def validate_request(self, *, steps, attention_backend=None, acceleration_lora=None):\\n        return self.validate_generation_request(steps=steps, attention_backend=attention_backend,\\n                                                acceleration_lora=acceleration_lora)\\n\\n    def _infer_raw(self, prepared):\\n        """All staged model swapping, text/VAE encode, denoise and decode use GPU lease."""\\n        import torch\\n        from diffusers.modular_pipelines.modular_pipeline import PipelineState\\n        if self.pipe is None:\\n            raise RuntimeError("Backend has not loaded")\\n        p = prepared.parameters\\n        state = None\\n        try:\\n            self._switch("encode")\\n            state = PipelineState(values={\\n                "prompt": p["prompt"], "references": prepared.references, "num_frames": aligned_frames(p["seconds"]),\\n                "height": p["height"], "width": p["width"], "num_inference_steps": p["steps"],\\n                "generator": torch.Generator(device="cpu").manual_seed(p["seed"]),\\n                "output_type": "pil",\\n            })\\n            state = self._run(state, "encode")\\n            # prompt_embeds is the one GPU tensor emitted by the encoder; VAE\\n            # conditioning latents already live on CPU. Do not retain CUDA state\\n            # while swapping 62GB components; preserve generator state unchanged.\\n            state.values = self._cpu(state.values)\\n            self._switch("denoise")\\n            # The denoiser passes prompt_embeds straight through, unlike layout/\\n            # condition latents, which their preparation blocks move to device.\\n            state.set("prompt_embeds", state.get("prompt_embeds").to("cuda"))\\n            state = self._run(state, "denoise")\\n            # Free the large denoiser before VAE decode; no longer needed for this\\n            # job. Keep the final latents on CUDA for the official decoder blocks.\\n            with self._stage("unload.transformer_ref"):\\n                self.pipe.unload_components("transformer_ref")\\n                self._phase = None\\n            state = self._run(state, "decode")\\n            return {"videos": state.get("videos"), "audio": state.get("audio"), "sampling_rate": state.get("sampling_rate")}\\n        finally:\\n            state = None\\n\', \'staged_server.py\': \'"""Compatibility API entry point; staging is chosen by the model container owner."""\\nfrom server import create_app\\n\\n\\ndef build_app():\\n    return create_app()\\n\', \'smoke_test.py\': \'"""Exercise a REAL running API and verify its downloaded audiovisual MP4.\\n\\nThis is a low-step integration smoke test, not a generation-quality benchmark.\\nIt has no fake backend, synthetic-video fallback, or GPU/model loader.\\n"""\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom contextlib import contextmanager\\nfrom datetime import datetime, timezone\\nfrom fractions import Fraction\\nimport hashlib\\nimport json\\nimport mimetypes\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport subprocess\\nimport time\\nimport uuid\\nfrom urllib.parse import urlsplit\\n\\nPARAMETERS = {\\n    "seconds": 5, "width": 512, "height": 288, "steps": 8, "seed": 42,\\n}\\n\\n\\ndef utc_now() -> str:\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef require(condition: bool, message: str) -> None:\\n    if not condition:\\n        raise RuntimeError(message)\\n\\n\\ndef sha256(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open("rb") as source:\\n        for block in iter(lambda: source.read(1024 * 1024), b""):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef save_report(outputdir: Path, report: dict) -> None:\\n    """Checkpoint the accepted job ID before polling; resuming never resubmits."""\\n    target = outputdir / "test_report.json"\\n    temporary = target.with_suffix(".json.tmp")\\n    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\\n    temporary.replace(target)\\n\\n\\n@contextmanager\\ndef content_history_lock(path):\\n    """OS-held writer lock; process exit releases it without stale lock ownership."""\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with path.with_suffix(path.suffix + ".lock").open("a+b") as lock:\\n        if os.fstat(lock.fileno()).st_size == 0:\\n            lock.write(b"0")\\n            lock.flush()\\n        deadline = time.monotonic() + 5\\n        while True:\\n            lock.seek(0)\\n            try:\\n                if os.name == "nt":\\n                    import msvcrt\\n                    msvcrt.locking(lock.fileno(), msvcrt.LK_NBLCK, 1)\\n                else:\\n                    import fcntl\\n                    fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\\n                break\\n            except OSError:\\n                require(time.monotonic() < deadline, "Content history is busy; no generation was submitted")\\n                time.sleep(.05)\\n        try:\\n            yield\\n        finally:\\n            lock.seek(0)\\n            if os.name == "nt":\\n                msvcrt.locking(lock.fileno(), msvcrt.LK_UNLCK, 1)\\n            else:\\n                fcntl.flock(lock.fileno(), fcntl.LOCK_UN)\\n\\n\\ndef register_content(path: Path, reference_hash: str, prompt: str, *, outputdir: Path, api_base_url: str) -> dict:\\n    """Reserve fresh content durably before the one paid submission attempt."""\\n    canonical = json.dumps({"reference_sha256": reference_hash, "prompt": prompt.strip()},\\n                           sort_keys=True, ensure_ascii=False, separators=(",", ":"))\\n    fingerprint = hashlib.sha256(canonical.encode("utf-8")).hexdigest()\\n    with content_history_lock(path):\\n        history = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"schema_version": 1, "entries": {}}\\n        require(history.get("schema_version") == 1 and isinstance(history.get("entries"), dict),\\n                "Invalid content history; refusing to submit without reliable duplicate detection")\\n        require(fingerprint not in history["entries"],\\n                "Reference and prompt combination was already registered; provide new content or use --resume-job for query/download only")\\n        record = {"fingerprint": fingerprint, "reference_sha256": reference_hash, "prompt": prompt.strip(),\\n                  "registered_at": utc_now(), "request_id": uuid.uuid4().hex,\\n                  "outputdir": str(outputdir), "api_base_url": api_base_url}\\n        history["entries"][fingerprint] = record\\n        temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\\n        try:\\n            with temporary.open("x", encoding="utf-8") as output:\\n                json.dump(history, output, ensure_ascii=False, indent=2)\\n                output.flush()\\n                os.fsync(output.fileno())\\n            temporary.replace(path)\\n            if os.name != "nt":\\n                descriptor = os.open(path.parent, os.O_RDONLY)\\n                try:\\n                    os.fsync(descriptor)\\n                finally:\\n                    os.close(descriptor)\\n        finally:\\n            temporary.unlink(missing_ok=True)\\n    return {**record, "history_path": str(path)}\\n\\n\\ndef verify_video(path: Path) -> dict:\\n    probe = subprocess.run(\\n        ["ffprobe", "-v", "error", "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)],\\n        capture_output=True, text=True, timeout=120, check=True,\\n    )\\n    media = json.loads(probe.stdout)\\n    video = next((s for s in media["streams"] if s["codec_type"] == "video"), None)\\n    audio = next((s for s in media["streams"] if s["codec_type"] == "audio"), None)\\n    require(video is not None and audio is not None, "MP4 must contain both video and audio streams")\\n    frames = int(video.get("nb_read_frames") or video.get("nb_frames") or 0)\\n    fps = float(Fraction(video.get("avg_frame_rate", "0")))\\n    duration = float(video.get("duration") or media["format"]["duration"])\\n    require(video.get("codec_name") == "h264", "Expected an H.264 video stream")\\n    require(frames >= 120 and 5 <= duration <= 6, "Expected at least 120 frames and a 5–6 second video")\\n    require(abs(fps - 24) < 0.01, "Expected 24 fps")\\n    require((video["width"], video["height"]) == (512, 288), "Unexpected video dimensions")\\n    require(int(audio.get("channels", 0)) == 2, "Expected stereo generated audio")\\n    subprocess.run(\\n        ["ffmpeg", "-nostdin", "-v", "error", "-xerror", "-i", str(path),\\n         "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"],\\n        capture_output=True, text=True, timeout=180, check=True,\\n    )\\n    return {\\n        "video_codec": video["codec_name"], "width": video["width"], "height": video["height"],\\n        "decoded_frames": frames, "fps": fps, "duration_seconds": duration,\\n        "audio_codec": audio.get("codec_name"), "audio_channels": audio["channels"],\\n        "audio_sample_rate": audio.get("sample_rate"), "full_decode_passed": True,\\n        "bytes": path.stat().st_size, "sha256": sha256(path),\\n    }\\n\\n\\ndef run(args, report: dict) -> None:\\n    import requests\\n\\n    reference = None\\n    if not args.resume_job:\\n        require(bool(getattr(args, "reference", None)), "New generation requires an explicit --reference image")\\n        require(isinstance(getattr(args, "prompt", None), str) and bool(args.prompt.strip()),\\n                "New generation requires an explicit nonempty --prompt")\\n        reference = Path(args.reference).resolve()\\n        require(reference.is_file(), "Reference image does not exist")\\n        require(reference.stat().st_size <= 8 * 1024 * 1024, "Reference image exceeds smoke-test size limit")\\n        from PIL import Image\\n        with Image.open(reference) as image:\\n            image.verify()\\n    parsed = urlsplit(args.url)\\n    require(parsed.scheme in {"https", "http"} and bool(parsed.hostname), "Provide an HTTP(S) API base URL")\\n    require(not parsed.username and not parsed.password and not parsed.query and not parsed.fragment,\\n            "Do not embed credentials, query parameters or fragments in the API URL")\\n    key = os.environ.get("H3_API_KEY", "")\\n    require(len(key) >= 32, "Set H3_API_KEY to the running server\\\'s token")\\n    for command in ("ffprobe", "ffmpeg"):\\n        require(shutil.which(command) is not None, f"{command} is required before submitting a paid GPU job")\\n    base = args.url.rstrip("/")\\n    report["api_base_url"] = base\\n    parameters = {**PARAMETERS, "steps": args.steps,\\n                  "attention_backend": getattr(args, "attention_backend", "auto"),\\n                  "acceleration_lora": getattr(args, "acceleration_lora", "none")}\\n    require(parameters["acceleration_lora"] == "none" or parameters["steps"] == 5,\\n            "Turbo Ref2VA requires explicit --steps 5 (four denoiser evaluations)")\\n    if not args.resume_job:\\n        parameters["prompt"] = args.prompt.strip()\\n        report["parameters"] = parameters\\n    report["test_scope"] = "real API integration; low-step smoke test, not visual quality acceptance"\\n    headers = {"Authorization": f"Bearer {key}"}\\n    session = requests.Session()\\n    started = time.monotonic()\\n    deadline = started + args.startup_timeout\\n    print("Waiting for API and manager availability; model loading may continue while jobs queue...", flush=True)\\n    while True:\\n        try:\\n            response = session.get(base + "/health", timeout=10)\\n            health = response.json()\\n            if not isinstance(health, dict):\\n                raise ValueError("Invalid health response")\\n            # New APIs expose admission separately from model execution. A legacy\\n            # aggregate ready flag is only a fallback when capability keys are\\n            # absent, never an override for an explicitly unavailable API/manager.\\n            explicit_capabilities = "api_ready" in health or "manager_ready" in health\\n            admission_ready = response.status_code == 200 and (\\n                health.get("api_ready") is True and health.get("manager_ready") is True\\n                if explicit_capabilities else health.get("ready") is True)\\n            if admission_ready or (args.resume_job and response.status_code in {200, 503}):\\n                report["health"] = {"http_status": response.status_code,\\n                    "ready": health.get("ready") is True, "admission_ready": admission_ready,\\n                    **{name: health.get(name) if type(health.get(name)) is bool else None\\n                       for name in ("api_ready", "manager_ready", "model_ready")},\\n                    "observed_at": utc_now()}\\n                break\\n        except (requests.RequestException, ValueError):\\n            pass\\n        require(time.monotonic() < deadline, "API and manager did not become available before startup timeout")\\n        time.sleep(args.poll_seconds)\\n    print(f"/health: HTTP {report[\\\'health\\\'][\\\'http_status\\\']}, admission_ready={report[\\\'health\\\'][\\\'admission_ready\\\']}, "\\n          f"ready={report[\\\'health\\\'][\\\'ready\\\']}, model_ready={report[\\\'health\\\'][\\\'model_ready\\\']}", flush=True)\\n\\n    if args.resume_job:\\n        job_id = args.resume_job\\n        report["resumed"] = True\\n        report["test_scope"] = "resume existing job: status/download only; no new content registration or generation request"\\n        print(f"Resuming status/download for {job_id}; no generation submission", flush=True)\\n    else:\\n        response = session.post(base + "/v1/jobs", timeout=30)\\n        report["unauthorized_submit_http_status"] = response.status_code\\n        require(response.status_code in {401, 403}, "Unauthenticated submission was not rejected")\\n        print(f"Unauthenticated POST rejected: HTTP {response.status_code}", flush=True)\\n        report["reference_sha256"] = sha256(reference)\\n        report["reference_source"] = str(reference)\\n        history_path = Path(getattr(args, "content_history", None) or args.outputdir.parent / "content-history.json").resolve()\\n        registration = register_content(history_path, report["reference_sha256"], parameters["prompt"],\\n                                        outputdir=args.outputdir, api_base_url=base)\\n        report["content_registration"] = registration\\n        save_report(args.outputdir, report)\\n        with reference.open("rb") as image:\\n            require(sha256(reference) == report["reference_sha256"], "Reference changed after content registration; generation not submitted")\\n            response = session.post(\\n                base + "/v1/jobs", headers={**headers, "Idempotency-Key": registration["request_id"]}, timeout=120,\\n                data={**parameters, "kinds": json.dumps(["image"])},\\n                files=[("files", (reference.name, image, mimetypes.guess_type(reference.name)[0] or "image/jpeg"))],\\n            )\\n        report["submit_http_status"] = response.status_code\\n        require(response.status_code == 202, f"Generation submit failed: HTTP {response.status_code}")\\n        task = response.json()\\n        job_id = task["id"]\\n        report["actual_seconds"] = task.get("actual_seconds")\\n        print(f"Submitted real job {job_id}: HTTP 202", flush=True)\\n    require(re.fullmatch(r"[a-f0-9]{32}", job_id) is not None, "Unexpected job ID format")\\n    report["job_id"] = job_id\\n    report.setdefault("status_history", [])\\n    save_report(args.outputdir, report)\\n\\n    deadline = time.monotonic() + args.generation_timeout\\n    while True:\\n        response = session.get(base + f"/v1/jobs/{job_id}", headers=headers, timeout=30)\\n        require(response.status_code == 200, f"Status query failed: HTTP {response.status_code}")\\n        task = response.json()\\n        status = task.get("status")\\n        if args.resume_job and task.get("generation_parameters"):\\n            report["parameters"] = task["generation_parameters"]\\n        report["status_history"].append({"http_status": response.status_code, "status": status,\\n                                        "queue_position": task.get("queue_position"),\\n                                        "waiting_for": task.get("waiting_for"),\\n                                        "active_phase": task.get("active_phase"), "observed_at": utc_now()})\\n        save_report(args.outputdir, report)\\n        print(f"{utc_now()} GET job: HTTP 200, status={status}, "\\n              f"queue_position={task.get(\\\'queue_position\\\')}, waiting_for={task.get(\\\'waiting_for\\\')}, "\\n              f"active_phase={task.get(\\\'active_phase\\\')}", flush=True)\\n        if status in {"succeeded", "failed", "cancelled"}:\\n            report["terminal_status"] = status\\n            report["metrics"] = task.get("metrics")\\n            report["lifecycle"] = task.get("lifecycle")\\n            report["terminal_error"] = task.get("error")\\n            report["cleanup_complete"] = task.get("cleanup_complete")\\n            break\\n        require(status in {"queued", "running", "cancel_requested"}, "Unknown job status")\\n        require(time.monotonic() < deadline, "Generation timeout; task may still be running on server")\\n        time.sleep(args.poll_seconds)\\n    require(status == "succeeded", "Real model generation failed; inspect the server log")\\n\\n    output = args.outputdir / f"{job_id}.mp4"\\n    with session.get(base + f"/v1/jobs/{job_id}/file", headers=headers, timeout=120, stream=True) as response:\\n        report["download_http_status"] = response.status_code\\n        require(response.status_code == 200, f"Video download failed: HTTP {response.status_code}")\\n        size = 0\\n        with output.open("wb") as destination:\\n            for chunk in response.iter_content(1024 * 1024):\\n                size += len(chunk)\\n                require(size <= 128 * 1024 * 1024, "Smoke-test output unexpectedly exceeds 128 MiB")\\n                destination.write(chunk)\\n    report["video"] = verify_video(output)\\n    report["video_path"] = str(output.resolve())\\n    report["elapsed_seconds"] = round(time.monotonic() - started, 2)\\n    report["passed"] = True\\n    print("API_SMOKE_TEST_PASSED", flush=True)\\n    print(json.dumps(report["video"], indent=2), flush=True)\\n    print(f"VIDEO_PATH={output.resolve()}", flush=True)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--url", required=True, help="Base URL of an already running real server")\\n    parser.add_argument("--reference", help="Required for new generation: explicitly chosen local reference image")\\n    parser.add_argument("--prompt", help="Required for new generation: fresh explicit prompt; no automatic default")\\n    parser.add_argument("--content-history", type=Path,\\n                        help="Shared durable content registry; default OUTPUTDIR.parent/content-history.json")\\n    parser.add_argument("--steps", type=int, choices=range(2, 51), default=8,\\n                        help="Scheduler grid points; Turbo Ref2VA requires 5 for four evaluations")\\n    parser.add_argument("--attention-backend", choices=("auto", "native", "_native_flash"), default="auto")\\n    parser.add_argument("--acceleration-lora", choices=("none", "lightx2v-ref2va-4step"), default="none")\\n    parser.add_argument("--resume-job", help="Existing job ID: query and download only, never submit another generation")\\n    parser.add_argument("--outputdir", type=Path, default=Path("smoke_result"))\\n    parser.add_argument("--startup-timeout", type=int, default=3600)\\n    parser.add_argument("--generation-timeout", type=int, default=2700)\\n    parser.add_argument("--poll-seconds", type=int, choices=range(1, 61), default=15, metavar="1..60")\\n    args = parser.parse_args()\\n    args.outputdir = args.outputdir.resolve()\\n    args.outputdir.mkdir(parents=True, exist_ok=True)\\n    report = {"started_at": utc_now(), "passed": False}\\n    if args.resume_job:\\n        require(re.fullmatch(r"[a-f0-9]{32}", args.resume_job) is not None, "Unexpected resume job ID")\\n        saved = args.outputdir / "test_report.json"\\n        if saved.exists():\\n            previous = json.loads(saved.read_text(encoding="utf-8"))\\n            require(previous.get("job_id") == args.resume_job, "Output directory belongs to a different job")\\n            require(previous.get("api_base_url") == args.url.rstrip("/"), "Output directory belongs to a different API")\\n            report.update(previous)\\n            report.update(passed=False, resumed_at=utc_now())\\n            report.pop("error", None)\\n            report.pop("error_type", None)\\n    try:\\n        run(args, report)\\n    except BaseException as error:\\n        # No response bodies, headers, credentials, or environment are copied into the report.\\n        report["error_type"] = type(error).__name__\\n        message = str(error)\\n        if os.environ.get("H3_API_KEY"):\\n            message = message.replace(os.environ["H3_API_KEY"], "[REDACTED]")\\n        report["error"] = message\\n        raise\\n    finally:\\n        report["finished_at"] = utc_now()\\n        path = args.outputdir / "test_report.json"\\n        save_report(args.outputdir, report)\\n        print(f"REPORT_PATH={path}", flush=True)\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'runtime_options.py\': \'"""BF16 runtime controls and lightweight host-wall-time diagnostics.\\n\\nThe accepted stream/on-demand-pinning path remains the default. ``sync`` and\\nexplicit attention choices are diagnostic alternatives, not GPU-validated\\nspeedups. No option changes weight precision or downloads attention kernels.\\n"""\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nfrom contextvars import ContextVar\\nfrom dataclasses import asdict, dataclass\\nfrom datetime import datetime, timezone\\nfrom functools import wraps\\nimport inspect\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport sys\\nimport threading\\nimport time\\n\\n\\n@dataclass(frozen=True)\\nclass RuntimeOptions:\\n    denoiser_offload: str = "stream"\\n    attention_backend: str = "auto"\\n\\n    def __post_init__(self):\\n        if self.denoiser_offload not in {"stream", "sync"}:\\n            raise ValueError("H3_DENOISER_OFFLOAD must be stream or sync (experimental).")\\n        if self.attention_backend not in {"auto", "native", "_native_flash"}:\\n            raise ValueError("H3_ATTENTION_BACKEND must be auto, native, or _native_flash.")\\n\\n    @classmethod\\n    def from_env(cls):\\n        return cls(\\n            denoiser_offload=os.environ.get("H3_DENOISER_OFFLOAD", "stream"),\\n            attention_backend=os.environ.get("H3_ATTENTION_BACKEND", "auto"),\\n        )\\n\\n    def group_offload_kwargs(self, torch):\\n        return {\\n            "onload_device": torch.device("cuda"),\\n            "offload_device": torch.device("cpu"),\\n            "offload_type": "block_level",\\n            "num_blocks_per_group": 1,\\n            "use_stream": self.denoiser_offload == "stream",\\n            "low_cpu_mem_usage": self.denoiser_offload == "stream",\\n        }\\n\\n    def configure_attention(self, transformer):\\n        if self.attention_backend != "auto":\\n            # Official Diffusers v0.43 API. _native_flash forces PyTorch SDPA\\n            # FLASH_ATTENTION; errors must propagate, with no silent fallback.\\n            transformer.set_attention_backend(self.attention_backend)\\n\\n\\ndef configure_qwen_offload(text_encoder):\\n    """Preserve the A100-tested placement of Qwen\\\'s directly accessed weights."""\\n    import torch\\n    from diffusers.hooks import apply_group_offloading\\n\\n    text_encoder.requires_grad_(False)\\n    encoder = text_encoder.model\\n    language = encoder.language_model\\n    # Qwen reads visual.pos_embed.weight.device outside forward and calls\\n    # embed_tokens before language.forward. Rotary uses its buffer directly.\\n    encoder.visual.to("cuda")\\n    language.embed_tokens.to("cuda")\\n    language.norm.to("cuda")\\n    language.rotary_emb.to("cuda")\\n    for layer in language.layers:\\n        apply_group_offloading(\\n            layer, onload_device=torch.device("cuda"), offload_device=torch.device("cpu"),\\n            offload_type="block_level", num_blocks_per_group=1, use_stream=False,\\n        )\\n\\n\\ndef resource_snapshot():\\n    """Read counters only: never initialize CUDA, synchronize, or scan weights."""\\n    result = {}\\n    try:\\n        import psutil\\n        process = psutil.Process()\\n        memory = process.memory_info()\\n        cpu = process.cpu_times()\\n        host = psutil.virtual_memory()\\n        result.update(\\n            rss_bytes=memory.rss, cpu_seconds=cpu.user + cpu.system,\\n            host_available_bytes=host.available, host_free_bytes=host.free,\\n        )\\n        for source, key in ((memory, "shared"), (host, "cached")):\\n            if hasattr(source, key):\\n                result[("process_" if source is memory else "host_") + key + "_bytes"] = getattr(source, key)\\n        try:\\n            result["read_bytes"] = process.io_counters().read_bytes\\n        except (AttributeError, OSError, psutil.Error):\\n            pass\\n    except Exception:\\n        # Counter access can be denied by a container or disappear during exit.\\n        # Diagnostics must not abort inference.\\n        pass\\n    torch = sys.modules.get("torch")\\n    if torch is not None:\\n        try:\\n            if torch.cuda.is_initialized():\\n                result["cuda_allocated_bytes"] = torch.cuda.memory_allocated()\\n                result["cuda_reserved_bytes"] = torch.cuda.memory_reserved()\\n        except (AttributeError, RuntimeError):\\n            pass\\n    return result\\n\\n\\ndef emit_metric(event):\\n    # A leading newline separates our JSON from tqdm\\\'s carriage-return line.\\n    try:\\n        print("\\\\nH3_METRIC " + json.dumps(event, ensure_ascii=False, sort_keys=True), flush=True)\\n    except (OSError, ValueError):\\n        pass\\n\\n\\nclass RuntimeMetrics:\\n    def __init__(self, kind, backend, options):\\n        self.started = time.perf_counter()\\n        self._owner_thread = threading.get_ident()\\n        self._phase_stack = []\\n        self._lock = threading.RLock()\\n        self.data = {\\n            "schema_version": 1, "kind": kind, "backend": backend,\\n            "started_at": datetime.now(timezone.utc).isoformat(),\\n            "runtime_options": asdict(options),\\n            "timing": "host_wall_seconds_without_cuda_synchronize",\\n            "resource_counter_scope": "process; overlapping stage deltas are not additive",\\n            "resources_before": resource_snapshot(), "stages": [],\\n            "status": "running",\\n        }\\n\\n    @contextmanager\\n    def stage(self, phase):\\n        started = time.perf_counter()\\n        # A job may move from its prepare executor to the GPU and encode workers.\\n        # Child stages identify themselves explicitly; thread identity is not job identity.\\n        is_owner = not phase.startswith("child.")\\n        if is_owner:\\n            with self._lock:\\n                self._phase_stack.append((phase, started))\\n        before = resource_snapshot()\\n        emit_metric({"event": "stage_start", "phase": phase, "kind": self.data["kind"]})\\n        status, error_type = "succeeded", None\\n        try:\\n            yield\\n        except BaseException as error:\\n            status, error_type = "failed", type(error).__name__\\n            raise\\n        finally:\\n            after = resource_snapshot()\\n            record = {\\n                "phase": phase, "status": status,\\n                "wall_seconds": round(time.perf_counter() - started, 6),\\n                "scope": "main" if is_owner else "child",\\n                "resources_before": before, "resources_after": after,\\n            }\\n            for key in ("read_bytes", "cpu_seconds"):\\n                if key in before and key in after:\\n                    record[key + "_delta"] = max(0, after[key] - before[key])\\n            if error_type:\\n                record["error_type"] = error_type\\n            with self._lock:\\n                self.data["stages"].append(record)\\n                if is_owner:\\n                    self._phase_stack.pop()\\n            emit_metric({"event": "stage_end", "kind": self.data["kind"], **record})\\n\\n    def phase_snapshot(self):\\n        with self._lock:\\n            if not self._phase_stack:\\n                return None\\n            phase, started = self._phase_stack[-1]\\n        return {"phase": phase, "elapsed_seconds": round(time.perf_counter() - started, 3), "kind": self.data["kind"]}\\n\\n    def record_schedule(self, record):\\n        """Append path-free scheduling attribution under the metrics lock."""\\n        with self._lock:\\n            self.data.setdefault("scheduling", []).append(dict(record))\\n\\n    def finish(self, error=None):\\n        self.data.update(\\n            status="succeeded" if error is None else "failed",\\n            finished_at=datetime.now(timezone.utc).isoformat(),\\n            wall_seconds=round(time.perf_counter() - self.started, 6),\\n            resources_after=resource_snapshot(),\\n        )\\n        if error is not None:\\n            self.data["error_type"] = type(error).__name__\\n        return self.data\\n\\n\\nclass RuntimeSupport:\\n    def _init_runtime(self, options=None):\\n        self.runtime_options = options or RuntimeOptions.from_env()\\n        self._loaded = False\\n        self._metrics_context = ContextVar(f"h3_metrics_{id(self)}", default=None)\\n        self._last_metrics_context = ContextVar(f"h3_last_metrics_{id(self)}", default=None)\\n        self._startup_active_metrics = None\\n        self.startup_metrics = None\\n        self.last_metrics = None\\n\\n    @property\\n    def _active_metrics(self):\\n        return self._metrics_context.get()\\n\\n    @_active_metrics.setter\\n    def _active_metrics(self, value):\\n        self._metrics_context.set(value)\\n\\n    @property\\n    def last_metrics(self):\\n        return self._last_metrics_context.get()\\n\\n    @last_metrics.setter\\n    def last_metrics(self, value):\\n        self._last_metrics_context.set(value)\\n\\n    @contextmanager\\n    def metrics_context(self, metrics):\\n        token = self._metrics_context.set(metrics)\\n        try:\\n            yield metrics\\n        finally:\\n            self._metrics_context.reset(token)\\n\\n    @property\\n    def current_phase(self):\\n        snapshot = self.phase_snapshot()\\n        return snapshot["phase"] if snapshot else None\\n\\n    def phase_snapshot(self, context=None):\\n        metrics = getattr(context, "metrics", None) if context is not None else self._active_metrics or self._startup_active_metrics\\n        return metrics.phase_snapshot() if metrics is not None else None\\n\\n    def _new_metrics(self, kind):\\n        metrics = RuntimeMetrics(kind, type(self).__name__, self.runtime_options)\\n        settings = getattr(self, "adapter_settings", None)\\n        if settings is not None:\\n            metrics.data["adapters"] = settings.as_dict()\\n            metrics.data["adapters_scope"] = "preloaded_deployment; per-job applied choices are in generation_options"\\n        return metrics\\n\\n    @contextmanager\\n    def _stage(self, phase):\\n        if self._active_metrics is None:\\n            yield\\n        else:\\n            with self._active_metrics.stage(phase):\\n                yield\\n\\n\\ndef measured_load(function):\\n    @wraps(function)\\n    def wrapper(self, *args, **kwargs):\\n        if self._loaded:\\n            return None\\n        metrics = self._new_metrics("startup")\\n        self._active_metrics = metrics\\n        self._startup_active_metrics = metrics\\n        error = None\\n        try:\\n            with metrics.stage("load.total"):\\n                result = function(self, *args, **kwargs)\\n            self._loaded = True\\n            return result\\n        except BaseException as caught:\\n            error = caught\\n            raise\\n        finally:\\n            self.startup_metrics = metrics.finish(error)\\n            self._active_metrics = None\\n            self._startup_active_metrics = None\\n    return wrapper\\n\\n\\ndef measured_generation(function):\\n    signature = inspect.signature(function)\\n\\n    @wraps(function)\\n    def wrapper(self, *args, **kwargs):\\n        self.last_metrics = None  # A failed job must never expose the previous job\\\'s metrics.\\n        arguments = signature.bind(self, *args, **kwargs).arguments\\n        output_path = Path(arguments["output_path"])\\n        metrics = self._new_metrics("generation")\\n        self._active_metrics = metrics\\n        error = None\\n        try:\\n            with metrics.stage("generate.total"):\\n                return function(self, *args, **kwargs)\\n        except BaseException as caught:\\n            error = caught\\n            raise\\n        finally:\\n            self.last_metrics = metrics.finish(error)\\n            self._active_metrics = None\\n            sidecar = output_path.with_suffix(".metrics.json")\\n            temporary = sidecar.with_suffix(sidecar.suffix + ".tmp")\\n            try:\\n                sidecar.parent.mkdir(parents=True, exist_ok=True)\\n                temporary.write_text(json.dumps(self.last_metrics, ensure_ascii=False, indent=2) + "\\\\n", encoding="utf-8")\\n                temporary.replace(sidecar)\\n            except OSError as write_error:\\n                # Observability must not turn a generated video into a failed job\\n                # or replace the original inference exception.\\n                self.last_metrics["metrics_write_error"] = type(write_error).__name__\\n                emit_metric({"event": "metrics_write_failed", "phase": "metrics", "error_type": type(write_error).__name__})\\n    return wrapper\\n\', \'adapter_config.py\': \'"""Opt-in, pinned MiniMax-H3 Ref2VA adapters. No GPU acceptance is claimed.\\n\\nThe baseline is unchanged with both settings ``none``. LoRAs are loaded into\\neach freshly loaded transformer_ref BEFORE group-offload hooks are installed.\\nOnly reviewed safetensors assets are accepted; no arbitrary remote code, URLs,\\npickle files, quantized weights, or silent cross-workflow routing.\\n"""\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport re\\nfrom download_progress import adapter_progress, current_progress, tracked_adapter_application\\n\\n\\n@dataclass(frozen=True)\\nclass AdapterSpec:\\n    key: str\\n    repository: str\\n    filename: str\\n    revision: str\\n    sha256: str\\n    size: int\\n    source_tensors: int\\n\\n\\nTURBO = AdapterSpec(\\n    "lightx2v-ref2va-4step", "lightx2v/Minimax-h3-Turbo",\\n    "minimax_h3_ref2v_turbo_4step_v0.1_bf16.safetensors",\\n    "1a4a150fc6707329409eb52e5d75a0e987ac1928",\\n    "9e642fc8749c74f8da5e2382877ab5c7aa37b9a73b7fd0d6d457bd1b3cb1ae99",\\n    1383677768, 624,\\n)\\nREALISM = AdapterSpec(\\n    "fal-realism-people", "fal/MiniMax-H3-Realism-People-LoRA",\\n    "h3-realism-people-t2v-i2v-r2v.safetensors",\\n    "039cc8579d7aa357a882d7f4111b25da4f72dccc",\\n    "acc529601d2da117fb81179e76c56e488a3beab1171659d305f04fa3655b787e",\\n    131229656, 208,\\n)\\n\\n\\n@dataclass(frozen=True)\\nclass AdapterSettings:\\n    acceleration: str = "none"\\n    style: str = "none"\\n    style_scale: float = 1.0\\n\\n    def __post_init__(self):\\n        if self.acceleration not in {"none", TURBO.key}:\\n            raise ValueError("H3_ACCELERATION_LORA must be none or lightx2v-ref2va-4step")\\n        if self.style not in {"none", REALISM.key}:\\n            raise ValueError("H3_STYLE_LORA must be none or fal-realism-people; arbitrary/FL2VA adapters are not accepted")\\n        if not math.isfinite(self.style_scale) or not 0.0 < self.style_scale <= 1.0:\\n            raise ValueError("H3_STYLE_LORA_SCALE must be finite and in (0, 1]")\\n\\n    @property\\n    def turbo(self):\\n        return self.acceleration != "none"\\n\\n    def as_dict(self):\\n        return {\\n            "acceleration": self.acceleration, "style": self.style,\\n            "style_scale": self.style_scale,\\n            "reference_image_resize": "match" if self.turbo else "official-2048-short-edge",\\n            "required_steps": 5 if self.turbo else None,\\n            "denoiser_evaluations": 4 if self.turbo else None,\\n            "fused": False,\\n            "validation": "not GPU-tested" if self.turbo or self.style != "none" else "baseline",\\n        }\\n\\n\\ndef load_adapter_settings(environ=None):\\n    env = os.environ if environ is None else environ\\n    return AdapterSettings(\\n        acceleration=env.get("H3_ACCELERATION_LORA", "none").strip(),\\n        style=env.get("H3_STYLE_LORA", "none").strip(),\\n        style_scale=float(env.get("H3_STYLE_LORA_SCALE", "1.0")),\\n    )\\n\\n\\ndef validate_steps(settings, steps):\\n    if settings.turbo and steps != 5:\\n        raise ValueError("Ref2VA Turbo requires steps=5 sigma grid points, which gives exactly 4 denoiser evaluations")\\n\\n\\ndef reference_match_size(width, height, target_width, target_height, multiple=32):\\n    """ModelTC training policy: cap reference area at canvas area, preserve ratio.\\n\\n    Rounding follows the author\\\'s implementation. Small inputs are not scaled\\n    up before rounding; a final nearest-32 rounding can increase an axis.\\n    """\\n    if min(width, height, target_width, target_height, multiple) <= 0:\\n        raise ValueError("Image and target dimensions must be positive")\\n    if width > 4 * height or height > 4 * width:\\n        raise ValueError("Reference image aspect ratio must be between 1:4 and 4:1")\\n    scale = min(1.0, math.sqrt(target_width * target_height / (width * height)))\\n    return (max(multiple, round(width * scale / multiple) * multiple),\\n            max(multiple, round(height * scale / multiple) * multiple))\\n\\n\\ndef configure_pipeline(pipe, settings):\\n    """Replace only this pipeline\\\'s resolved Ref2VA setup when Turbo is enabled.\\n\\n    The original setup still owns validation and video/audio normalization. Its\\n    CPU image resize is replaced from the original pixels before any encoder\\n    runs. This intentionally retains a small CPU resize overhead instead of\\n    copying upstream\\\'s full setup implementation or patching process globals.\\n    """\\n    if not settings.turbo:\\n        return\\n    from diffusers.modular_pipelines.minimax_h3.before_encoder import MiniMaxH3Ref2VASetupStep\\n    from diffusers.modular_pipelines.minimax_h3 import MiniMaxH3ImageReference\\n    from PIL import Image\\n\\n    original = pipe.blocks.sub_blocks.get("before_encode")\\n    if getattr(original, "_h3_turbo_match", False):\\n        return\\n    if not isinstance(original, MiniMaxH3Ref2VASetupStep):\\n        raise RuntimeError("Unexpected Ref2VA setup block; refusing to change reference resizing")\\n\\n    class MatchReferenceSetup(MiniMaxH3Ref2VASetupStep):\\n        _h3_turbo_match = True\\n\\n        def __call__(self, components, state):\\n            _, state = original(components, state)\\n            normalized = list(state.get("normalized_references"))\\n            references = state.get("references")\\n            if len(normalized) != len(references):\\n                raise RuntimeError("Reference normalization changed the reference count")\\n            for index, reference in enumerate(references):\\n                if reference.kind != "image":\\n                    continue\\n                # This API decodes uploaded images with from_file(), which\\n                # returns PIL. Reject a future unsupported path explicitly.\\n                if not isinstance(reference.image, Image.Image):\\n                    raise TypeError("Turbo API requires decoded PIL image references")\\n                image = reference.image.convert("RGB")\\n                width, height = reference_match_size(\\n                    *image.size, state.get("width"), state.get("height"), components.canvas_multiple,\\n                )\\n                if image.size != (width, height):\\n                    image = components.image_processor.resize(image, height=height, width=width)\\n                normalized[index] = MiniMaxH3ImageReference(image=image)\\n            state.set("normalized_references", normalized)\\n            return components, state\\n\\n    pipe._h3_original_before_encode = original\\n    pipe._h3_turbo_before_encode = MatchReferenceSetup()\\n    pipe.blocks.sub_blocks["before_encode"] = pipe._h3_turbo_before_encode\\n\\n\\ndef select_reference_setup(pipe, acceleration):\\n    """Select an already constructed setup block inside the exclusive GPU lease."""\\n    turbo = acceleration == TURBO.key\\n    selected = getattr(pipe, "_h3_turbo_before_encode" if turbo else "_h3_original_before_encode", None)\\n    if turbo and selected is None:\\n        raise RuntimeError("Turbo reference setup was not prepared during model loading")\\n    if selected is not None:\\n        pipe.blocks.sub_blocks["before_encode"] = selected\\n\\n\\ndef select_loaded_adapters(pipe, settings, acceleration):\\n    """Toggle only preloaded, unfused adapters; preserve independent style LoRA.\\n\\n    This never downloads, loads, fuses, deletes, casts, or changes base weights.\\n    Call only while the model execution owns the GPU lease, including restore.\\n    """\\n    transformer = getattr(pipe, "transformer_ref", None)\\n    if transformer is None:\\n        raise RuntimeError("Cannot select adapters without transformer_ref")\\n    if not settings.turbo and settings.style == "none":\\n        if acceleration != "none":\\n            raise RuntimeError("Requested acceleration adapter is not loaded")\\n        return\\n    if getattr(transformer, "_h3_adapter_settings", None) != settings:\\n        raise RuntimeError("Loaded adapter identity does not match deployment configuration")\\n    names, scales = [], []\\n    if acceleration != "none":\\n        if acceleration != settings.acceleration:\\n            raise RuntimeError("Requested acceleration adapter is not loaded")\\n        for name, expected in (("scheduler", 12.0), ("audio_scheduler", 3.0)):\\n            if float(getattr(pipe, name).config.shift) != expected:\\n                raise RuntimeError(f"Ref2VA Turbo requires {name} shift={expected}")\\n        names.append("h3_acceleration")\\n        scales.append(1.0)\\n    if settings.style != "none":\\n        names.append("h3_style")\\n        scales.append(settings.style_scale)\\n    if names:\\n        transformer.set_adapters(names, weights=scales)\\n        transformer.enable_lora()\\n    else:\\n        transformer.disable_lora()\\n    # PEFT activation can turn LoRA factors trainable again. Preserve the\\n    # inference-only lifetime after enabling, disabling, and restoring.\\n    transformer.requires_grad_(False)\\n    transformer.eval()\\n    if hasattr(transformer, "_reset_stateful_cache"):\\n        transformer._reset_stateful_cache()\\n\\n\\n_VERIFIED_FILES = set()\\n\\n\\ndef _download_verified(spec):\\n    from huggingface_hub import hf_hub_download\\n    progress = current_progress()\\n    category = \\\'acceleration\\\' if spec == TURBO else \\\'style\\\'\\n    cached = None\\n    try:\\n        from huggingface_hub import try_to_load_from_cache\\n        cached = try_to_load_from_cache(spec.repository, spec.filename, revision=spec.revision)\\n        cached = isinstance(cached, str) and Path(cached).is_file()\\n    except (ImportError, OSError, ValueError):\\n        cached = None\\n    if progress:\\n        progress.phase(category, \\\'verifying\\\' if cached else \\\'downloading\\\',\\n                       \\\'命中缓存，检查文件完整性\\\' if cached else \\\'下载所选 LoRA 文件\\\', total=spec.size, cache_hit=cached)\\n    path = Path(hf_hub_download(spec.repository, spec.filename, revision=spec.revision))\\n    stat = path.stat()\\n    if progress:\\n        progress.phase(category, \\\'verifying\\\', \\\'校验文件大小与 SHA-256\\\', total=spec.size, cache_hit=cached)\\n    signature = (str(path.resolve()), stat.st_size, stat.st_mtime_ns, spec.sha256)\\n    if signature not in _VERIFIED_FILES:\\n        if stat.st_size != spec.size:\\n            raise RuntimeError(f"Unexpected adapter file size for {spec.key}")\\n        digest = hashlib.sha256()\\n        with path.open("rb") as stream:\\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\\n                digest.update(chunk)\\n        if digest.hexdigest() != spec.sha256:\\n            raise RuntimeError(f"SHA-256 mismatch for {spec.key}")\\n        _VERIFIED_FILES.add(signature)\\n    return path\\n\\n\\ndef _validate_converted(state_dict, transformer, spec):\\n    """Verify every converted tensor has a paired, dimension-compatible target."""\\n    pairs = {}\\n    pattern = re.compile(r"^transformer\\\\.(.+)\\\\.lora_([AB])\\\\.weight$")\\n    for key, tensor in state_dict.items():\\n        match = pattern.fullmatch(key)\\n        if not match or len(tensor.shape) != 2:\\n            raise ValueError(f"Unsupported converted LoRA tensor: {key}")\\n        name, side = match.groups()\\n        if spec == REALISM and not re.fullmatch(r"(?:transformer_blocks|token_refiner\\\\.refiner_blocks)\\\\.\\\\d+\\\\.attn\\\\.(to_[qkv]|to_out\\\\.0)", name):\\n            raise ValueError(f"Realism adapter must target only shared attention projections: {name}")\\n        pairs.setdefault(name, {})[side] = tensor\\n    if not pairs:\\n        raise ValueError("Converted adapter has no target tensors")\\n    for name, pair in pairs.items():\\n        if set(pair) != {"A", "B"}:\\n            raise ValueError(f"Unpaired LoRA target: {name}")\\n        module = transformer.get_submodule(name)\\n        base = module.get_base_layer() if hasattr(module, "get_base_layer") else module\\n        a, b = pair["A"], pair["B"]\\n        if a.shape[0] != b.shape[1] or tuple(base.weight.shape) != (b.shape[0], a.shape[1]):\\n            raise ValueError(f"LoRA shape mismatch for {name}; pruned/other-model adapters are unsupported")\\n    return pairs\\n\\n\\n@tracked_adapter_application\\ndef apply_adapters(pipe, settings):\\n    """Load reviewed adapters into transformer_ref without fusing base weights.\\n\\n    Both named adapters can be composed, but that combination still needs GPU\\n    quality/latency acceptance. Only LoRA factors are cast; keep-FP32 base layers\\n    are never changed. Cache state is cleared when supported.\\n    """\\n    specs = []\\n    if settings.turbo:\\n        specs.append((TURBO, "h3_acceleration", 1.0))\\n    if settings.style != "none":\\n        specs.append((REALISM, "h3_style", settings.style_scale))\\n    if not specs:\\n        return\\n    import importlib.util\\n    if importlib.util.find_spec("peft") is None:\\n        raise RuntimeError("LoRA is enabled but PEFT is missing; install requirements-lora.txt using the existing torch constraints")\\n    import torch\\n    from safetensors import safe_open\\n    transformer = getattr(pipe, "transformer_ref", None)\\n    if transformer is None or getattr(pipe, "transformer", None) is not None:\\n        raise RuntimeError("Adapters require the ref2va-only pipeline with transformer_ref loaded")\\n    if getattr(transformer, "_h3_adapter_settings", None) is not None:\\n        if transformer._h3_adapter_settings != settings:\\n            raise RuntimeError("Changing adapters requires a freshly loaded transformer_ref")\\n        return\\n    if settings.turbo:\\n        for name, expected in (("scheduler", 12.0), ("audio_scheduler", 3.0)):\\n            if float(getattr(pipe, name).config.shift) != expected:\\n                raise RuntimeError(f"Ref2VA Turbo requires {name} shift={expected}")\\n    base_dtypes = {id(p): p.dtype for p in transformer.parameters()}\\n    names, scales = [], []\\n    for spec, adapter_name, scale in specs:\\n        category = "acceleration" if spec == TURBO else "style"\\n        with adapter_progress(category, spec) as progress:\\n            path = _download_verified(spec)\\n            if progress:\\n                progress.phase(category, "loading", "加载并校验 LoRA 张量；不改变主干精度")\\n            with safe_open(path, framework="pt", device="cpu") as handle:\\n                keys = list(handle.keys())\\n                if len(keys) != spec.source_tensors:\\n                    raise ValueError(f"Unexpected tensor count for {spec.key}")\\n                if any(handle.get_slice(key).get_dtype() != "BF16" for key in keys):\\n                    raise ValueError(f"The reviewed {spec.key} asset must contain only BF16 factors")\\n                if spec == TURBO and (handle.metadata() or {}).get("alpha") != "8":\\n                    raise ValueError("Ref2VA Turbo training alpha must be 8")\\n            state_dict, metadata = pipe.lora_state_dict(\\n                str(path.parent), weight_name=path.name, use_safetensors=True,\\n                local_files_only=True, return_lora_metadata=True,\\n            )\\n            pairs = _validate_converted(state_dict, transformer, spec)\\n            # Use the official ref-specific loader so the file metadata\\\'s alpha/rank\\n            # scaling survives conversion. Generic load_lora_weights(dict) would\\n            # lose the original file\\\'s alpha=8 metadata.\\n            pipe.load_lora_into_transformer_ref(\\n                state_dict, transformer_ref=transformer, prefix="transformer",\\n                adapter_name=adapter_name, metadata=metadata, _pipeline=pipe,\\n                low_cpu_mem_usage=True,\\n            )\\n            del state_dict\\n            for name in pairs:\\n                layer = transformer.get_submodule(name)\\n                if adapter_name not in layer.lora_A or adapter_name not in layer.lora_B:\\n                    raise RuntimeError(f"Adapter failed to load into {name}")\\n                expected_scale = 8.0 / 128 if spec == TURBO else 1.0\\n                if not math.isclose(float(layer.scaling[adapter_name]), expected_scale, rel_tol=1e-6):\\n                    raise RuntimeError(f"Unexpected training alpha/rank scale for {spec.key} at {name}")\\n                dtype = layer.get_base_layer().weight.dtype\\n                if dtype not in {torch.bfloat16, torch.float32}:\\n                    raise RuntimeError(f"Unexpected base precision at {name}: {dtype}")\\n                layer.lora_A[adapter_name].to(dtype=dtype)\\n                layer.lora_B[adapter_name].to(dtype=dtype)\\n            del pairs\\n            names.append(adapter_name)\\n            scales.append(scale)\\n            if progress:\\n                progress.phase(category, "loaded", "LoRA 已加载并校验", total=spec.size)\\n    transformer.set_adapters(names, weights=scales)\\n    for parameter in transformer.parameters():\\n        if id(parameter) in base_dtypes and parameter.dtype != base_dtypes[id(parameter)]:\\n            raise RuntimeError("LoRA loading changed the base-model precision")\\n    transformer.requires_grad_(False)\\n    transformer.eval()\\n    if hasattr(transformer, "_reset_stateful_cache"):\\n        transformer._reset_stateful_cache()\\n    transformer._h3_adapter_settings = settings\\n\', \'resource_scheduler.py\': \'"""Bounded CPU child events; the GPU owner and model stages remain serial.\\n\\nThe optimizer is exact within floating-point tolerance for its declared cost\\nmodel, not a claim of global hardware optimality. Admission uses calibrated\\nmemory/thread estimates, not OS-enforced peak limits. Missing calibration selects\\none lane. Native codec/library pools are not constrained by the Python executor.\\n"""\\nfrom __future__ import annotations\\n\\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport threading\\nimport time\\n\\nMIB = 1024 ** 2\\nRESERVE_MIB = 2048\\n\\n\\n@dataclass(frozen=True)\\nclass Allocation:\\n    lanes: tuple[tuple[int, ...], ...]\\n    predicted_seconds: float\\n    reason: str\\n    task_memory_mib: float | None = None\\n    threads_per_worker: int | None = None\\n    profile_metadata: dict | None = None\\n\\n\\ndef optimal_allocation(costs, io_mib, *, cpu_slots, memory_slots, bandwidth_mib_s,\\n                       max_workers=4, overhead_seconds=0.002):\\n    """Minimize max(compute makespan, shared IO floor)+lane overhead.\\n\\n    At most 12 children / 4 lanes: branch-and-bound enumerates legal assignments,\\n    removing symmetric empty lanes. Inputs are service estimates from calibration.\\n    Equal predicted costs favor fewer workers. No nested GPU concurrency is allowed.\\n    """\\n    if not costs:\\n        return Allocation((), 0.0, "no children")\\n    if len(costs) > 12 or any(not math.isfinite(x) or x <= 0 for x in costs):\\n        raise ValueError("Require 1..12 finite positive child costs")\\n    if len(io_mib) != len(costs) or any(not math.isfinite(x) or x < 0 for x in io_mib):\\n        raise ValueError("IO demand must match the child costs")\\n    if not math.isfinite(bandwidth_mib_s) or bandwidth_mib_s <= 0:\\n        raise ValueError("A positive measured bandwidth is required")\\n    if overhead_seconds < 0 or not math.isfinite(overhead_seconds):\\n        raise ValueError("Invalid dispatch overhead")\\n    if any(isinstance(value, bool) or not isinstance(value, int) for value in (cpu_slots, memory_slots, max_workers)):\\n        raise ValueError("CPU, memory, and worker lane caps must be integers")\\n    cap = min(4, max_workers, cpu_slots, memory_slots, len(costs))\\n    if cap < 1:\\n        raise MemoryError("No CPU/memory lane is available")\\n    order = sorted(range(len(costs)), key=lambda i: costs[i], reverse=True)\\n    io_floor = sum(io_mib) / bandwidth_mib_s\\n    winner = None\\n    for workers in range(1, cap + 1):\\n        loads, bins = [0.0] * workers, [[] for _ in range(workers)]\\n        # A valid upper bound from longest-processing-time list scheduling.\\n        for index in order:\\n            lane = min(range(workers), key=lambda k: loads[k])\\n            loads[lane] += costs[index]\\n            bins[lane].append(index)\\n        best_load, best_bins = max(loads), [row[:] for row in bins]\\n        lower_bound = max(sum(costs) / workers, max(costs))\\n        loads, bins = [0.0] * workers, [[] for _ in range(workers)]\\n\\n        def visit(position):\\n            nonlocal best_load, best_bins\\n            if best_load <= lower_bound + 1e-10:\\n                return\\n            if position == len(order):\\n                best_load, best_bins = max(loads), [row[:] for row in bins]\\n                return\\n            index = order[position]\\n            seen = set()\\n            for lane in range(workers):\\n                previous = loads[lane]\\n                if previous in seen or previous + costs[index] >= best_load - 1e-10:\\n                    continue\\n                seen.add(previous)\\n                loads[lane] += costs[index]\\n                bins[lane].append(index)\\n                visit(position + 1)\\n                bins[lane].pop()\\n                loads[lane] = previous\\n\\n        visit(0)\\n        predicted = max(best_load, io_floor) + overhead_seconds * workers\\n        candidate = Allocation(tuple(tuple(row) for row in best_bins), predicted,\\n                               "minimum predicted makespan under calibrated CPU/IO/memory limits")\\n        if winner is None or predicted < winner.predicted_seconds - 1e-10:\\n            winner = candidate\\n    return winner\\n\\n\\ndef _cpu_slots():\\n    """Affinity-visible logical cores; not a cgroup CPU quota or thread limit."""\\n    import psutil\\n    try:\\n        return max(1, len(psutil.Process().cpu_affinity()))\\n    except (AttributeError, OSError, psutil.Error):\\n        return max(1, os.cpu_count() or 1)\\n\\n\\ndef _phase_table(profile, key, phase):\\n    table = profile.get(key, {})\\n    if not isinstance(table, dict) or not isinstance(table.get(phase, {}), dict):\\n        raise ValueError(f"Calibration {key} must map phase names to modality estimates")\\n    return table.get(phase, {})\\n\\n\\nclass MediaController:\\n    """Negative feedback applies between batches, never inside a GPU stage.\\n\\n    A slow/low-memory observation reduces the next concurrency ceiling. Recovery\\n    increases it by one only after three healthy observations. A profile is an\\n    explicit local calibration artifact, never silently invented from model IO.\\n    """\\n    def __init__(self):\\n        self.ceiling = 4\\n        self.healthy = 0\\n        self.lock = threading.Lock()\\n\\n    def plan(self, items, phase):\\n        import psutil\\n        requested = os.environ.get("H3_MEDIA_WORKERS", "auto")\\n        if requested not in {"auto", "1", "2", "3", "4"}:\\n            raise ValueError("H3_MEDIA_WORKERS must be auto or 1..4")\\n        cpu = _cpu_slots()\\n        free = psutil.virtual_memory().available\\n        # This coarse guard protects the uncalibrated serial path. It is not a\\n        # promise that a codec\\\'s true peak is bounded by one GiB.\\n        if free < (RESERVE_MIB + 1024) * MIB:\\n            raise MemoryError("Less than 3 GiB available RAM for reference preparation")\\n        with self.lock:\\n            ceiling = self.ceiling\\n        maximum = min(cpu, ceiling)\\n        if requested != "auto":\\n            maximum = min(maximum, int(requested))\\n        profile_path = os.environ.get("H3_MEDIA_PROFILE", "")\\n        if not profile_path:\\n            # A manual ceiling does not substitute for measured child costs.\\n            return Allocation((tuple(range(len(items))),), 0.0, "uncalibrated: serial; native threads/peak memory not bounded"), 1\\n        profile_bytes = Path(profile_path).read_bytes()\\n        profile = json.loads(profile_bytes)\\n        if not isinstance(profile, dict) or profile.get("schema_version") != 1:\\n            raise ValueError("Unsupported media calibration schema")\\n        if profile.get("cpu_slots") != cpu:\\n            raise ValueError("Media calibration CPU affinity differs from this machine")\\n        metadata = {"sha256": hashlib.sha256(profile_bytes).hexdigest(), "schema_version": 1,\\n                    "affinity_cpu_slots": cpu, "phase": phase}\\n        rates = _phase_table(profile, "seconds_per_mib", phase)\\n        if any(item["kind"] not in rates for item in items):\\n            return Allocation((tuple(range(len(items))),), 0.0, "missing modality calibration: serial",\\n                              profile_metadata=metadata), 1\\n        threads = profile.get("threads_per_worker")\\n        memory_rates = _phase_table(profile, "task_memory_mib", phase)\\n        if threads is None or any(item["kind"] not in memory_rates for item in items):\\n            return Allocation((tuple(range(len(items))),), 0.0, "missing thread/memory calibration: serial",\\n                              profile_metadata=metadata), 1\\n        if isinstance(threads, bool) or not isinstance(threads, int) or threads < 1:\\n            raise ValueError("threads_per_worker must be a positive measured integer")\\n        demands = [float(memory_rates[item["kind"]]) for item in items]\\n        if any(not math.isfinite(value) or value <= 0 for value in demands):\\n            raise ValueError("task_memory_mib estimates must be finite and positive")\\n        if not items:\\n            return Allocation((), 0.0, "no children"), 0\\n        task_memory = max(demands)\\n        # Prepared objects survive until encoding. Conservatively reserve each\\n        # child\\\'s estimated peak again as its retained-result budget; then admit\\n        # temporary concurrent working sets. Estimates need matching input bounds.\\n        retained = sum(demands) if phase == "prepare" else 0\\n        memory_slots = max(0, math.floor((free / MIB - RESERVE_MIB - retained) / task_memory))\\n        cpu_budget = cpu // threads\\n        maximum = min(maximum, memory_slots, cpu_budget)\\n        if maximum < 1:\\n            raise MemoryError("Calibrated reference working sets exceed the available CPU/memory budget")\\n        sizes = [max(Path(item["path"]).stat().st_size / MIB, 0.01) for item in items]\\n        costs = [float(rates[item["kind"]]) * size for item, size in zip(items, sizes)]\\n        allocation = optimal_allocation(costs, sizes, cpu_slots=cpu_budget, memory_slots=memory_slots,\\n                                        bandwidth_mib_s=float(profile.get("bandwidth_mib_s", 0)),\\n                                        max_workers=maximum,\\n                                        overhead_seconds=float(profile.get("overhead_seconds", 0.002)))\\n        metadata.update(cpu_budget_slots=cpu_budget, memory_budget_slots=memory_slots,\\n                        estimated_child_seconds=costs, estimated_child_memory_mib=demands,\\n                        bandwidth_mib_s=float(profile["bandwidth_mib_s"]),\\n                        overhead_seconds=float(profile.get("overhead_seconds", 0.002)),\\n                        retained_result_reserve_mib=retained)\\n        return Allocation(allocation.lanes, allocation.predicted_seconds, allocation.reason,\\n                          task_memory, threads, metadata), memory_slots\\n\\n    def observe(self, allocation, elapsed, available_slots, succeeded):\\n        with self.lock:\\n            slow = allocation.predicted_seconds > 0 and elapsed > allocation.predicted_seconds * 1.25\\n            if not succeeded or slow or available_slots < len(allocation.lanes):\\n                self.ceiling = max(1, min(self.ceiling - 1, len(allocation.lanes)))\\n                self.healthy = 0\\n            else:\\n                self.healthy += 1\\n                if self.healthy >= 3:\\n                    self.ceiling = min(4, self.ceiling + 1)\\n                    self.healthy = 0\\n\\n\\nCONTROLLER = MediaController()\\n\\n\\ndef _children(items, function, phase, metrics):\\n    if not items:\\n        return []\\n    allocation, slots = CONTROLLER.plan(items, phase)\\n    attribution = _input_attribution(items)\\n    attribution["profile"] = allocation.profile_metadata\\n    results = [None] * len(items)\\n    stop = threading.Event()\\n    started, succeeded = time.perf_counter(), False\\n    _schedule_log("H3_SCHEDULE", {"phase": phase, "workers": len(allocation.lanes),\\n                  "predicted_seconds": allocation.predicted_seconds, "reason": allocation.reason,\\n                  "estimated_task_memory_mib": allocation.task_memory_mib,\\n                  "estimated_threads_per_worker": allocation.threads_per_worker,\\n                  **attribution}, metrics)\\n\\n    def lane(indices):\\n        for index in indices:\\n            if stop.is_set():\\n                return\\n            try:\\n                if metrics is None:\\n                    results[index] = function(items[index])\\n                else:\\n                    with metrics.stage(f"child.{phase}.{index}"):\\n                        results[index] = function(items[index])\\n            except BaseException:\\n                stop.set()\\n                raise\\n\\n    try:\\n        if len(allocation.lanes) == 1:\\n            lane(allocation.lanes[0])\\n        else:\\n            # Structured lifetime: cancel unstarted work, join started work before\\n            # returning or raising. No orphan children overlap the next GPU phase.\\n            with ThreadPoolExecutor(max_workers=len(allocation.lanes), thread_name_prefix="h3-media") as pool:\\n                futures = [pool.submit(lane, indices) for indices in allocation.lanes]\\n                try:\\n                    for future in as_completed(futures):\\n                        future.result()\\n                except BaseException:\\n                    stop.set()\\n                    for future in futures:\\n                        future.cancel()\\n                    raise\\n        succeeded = True\\n        return results\\n    finally:\\n        elapsed = time.perf_counter() - started\\n        if not succeeded:\\n            results.clear()  # Release completed child objects before re-raising.\\n        try:\\n            import psutil\\n            free_mib = psutil.virtual_memory().available / MIB\\n            estimate = allocation.task_memory_mib or 1024\\n            remaining = max(0, math.floor((free_mib - RESERVE_MIB) / estimate))\\n            CONTROLLER.observe(allocation, elapsed, remaining, succeeded)\\n            _schedule_log("H3_SCHEDULE_FEEDBACK", {"phase": phase, "wall_seconds": elapsed,\\n                          "succeeded": succeeded, "next_ceiling": CONTROLLER.ceiling,\\n                          "workers": len(allocation.lanes), **attribution}, metrics)\\n        except Exception as feedback_error:\\n            # A missing/denied counter or telemetry failure must never mask the\\n            # original child exception or discard successfully prepared media.\\n            _schedule_log("H3_SCHEDULE_FEEDBACK", {"phase": phase, "feedback": "unavailable",\\n                          "error_type": type(feedback_error).__name__, "wall_seconds": elapsed,\\n                          "succeeded": succeeded, **attribution}, metrics)\\n\\n\\ndef _input_attribution(items):\\n    children = []\\n    for index, item in enumerate(items):\\n        kind = item.get("kind") if isinstance(item, dict) else None\\n        size = None\\n        if isinstance(item, dict) and "path" in item:\\n            try:\\n                size = Path(item["path"]).stat().st_size\\n            except (OSError, TypeError, ValueError):\\n                pass\\n        children.append({"index": index, "kind": kind, "input_bytes": size})\\n    complete = all(child["input_bytes"] is not None for child in children)\\n    return {"child_count": len(children), "children": children,\\n            "input_bytes": sum(child["input_bytes"] for child in children) if complete else None}\\n\\n\\ndef _schedule_log(prefix, record, metrics=None):\\n    try:\\n        print("\\\\n" + prefix + " " + json.dumps(record), flush=True)\\n    except (OSError, ValueError):\\n        pass\\n    try:\\n        recorder = getattr(metrics, "record_schedule", None)\\n        if recorder is not None:\\n            recorder({"event": prefix, **record})\\n    except Exception:\\n        pass  # Attribution must not alter the child outcome.\\n\\n\\ndef prepare_references(references, classes, probe, normalize, metrics=None):\\n    """Probe barrier -> aggregate validation -> bounded decode barrier, in order."""\\n    normalized, refs = [], []\\n    lock = threading.Lock()\\n    try:\\n        durations = _children(references, lambda item: probe(Path(item["path"]), item["kind"]),\\n                              "probe", metrics)\\n        for kind in ("video", "audio"):\\n            if sum(duration for item, duration in zip(references, durations) if item["kind"] == kind) > 15:\\n                raise ValueError("Total reference duration per modality must be <=15 seconds")\\n\\n        def prepare(item):\\n            path = Path(item["path"])\\n            if item["kind"] == "video":\\n                # Known target is registered before conversion, so failed ffmpeg\\n                # outputs are cleaned too. normalize uses this same target.\\n                target = path.with_name(path.stem + "_24fps.mp4")\\n                with lock:\\n                    normalized.append(target)\\n                path = normalize(path)\\n                with lock:\\n                    if path not in normalized:\\n                        normalized.append(path)\\n            return classes[item["kind"]].from_file(str(path))\\n\\n        refs = _children(references, prepare, "prepare", metrics)\\n        return refs, normalized\\n    except BaseException:\\n        refs.clear()\\n        for path in normalized:\\n            path.unlink(missing_ok=True)\\n        raise\\n\', \'performance_model.py\': \'"""CPU-only optimistic physical lower bounds, not a GPU benchmark or simulator.\\n\\nResource floor = max_r(sum_i(work[i,r]) / peak[r]).\\nNode floor = max_r(work[i,r] / peak[r]).\\nCritical-path floor = longest dependency path using node floors.\\nOverall floor = max(resource floor, critical-path floor).\\n\\nMissing workload/capacity is reported, never invented. No PyTorch, CUDA,\\ndownloads, or live runtime access is used. Numeric inputs use explicit work\\nunits; capacities use the same unit per second. Decimal GB means 1e9 bytes.\\n"""\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom collections import deque\\nimport json\\nimport math\\nfrom pathlib import Path\\n\\n\\nDATASHEET = "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\\nA100_40_SOURCE = "https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/"\\nBACKGROUND = "https://docs.nvidia.com/deeplearning/performance/dl-performance-gpu-background/"\\n\\n\\ndef _number(value, path, *, positive=False):\\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\\n        raise ValueError(f"{path} must be a finite number")\\n    if value < 0 or (positive and value <= 0):\\n        raise ValueError(f"{path} must be {\\\'positive\\\' if positive else \\\'non-negative\\\'}")\\n    return float(value)\\n\\n\\ndef hardware_template(profile="a100-sxm-40gb"):\\n    """Return capacities only. The caller must supply a profiled workload DAG."""\\n    if profile not in {"a100-sxm-40gb", "a100-sxm-80gb"}:\\n        raise ValueError("Supported profiles: a100-sxm-40gb, a100-sxm-80gb")\\n    hbm = 1555e9 if profile == "a100-sxm-40gb" else 2039e9\\n    def resource(unit, capacity, note, source=None):\\n        return {"work_unit": unit, "capacity_per_second": capacity, "note": note, "source": source}\\n    return {\\n        "schema_version": 1,\\n        "hardware": profile,\\n        "coverage": "partial",\\n        "observed_seconds": None,\\n        "resources": {\\n            "gpu_bf16_dense_flop": resource("FLOP", 312e12,\\n                "BF16 dense Tensor Core work only; 624 TFLOP/s sparsity peak does not apply. One multiply-add counts as 2 FLOPs.", DATASHEET),\\n            "gpu_fp32_flop": resource("FLOP", None,\\n                "Keep FP32-layer work separate. Determine actual instruction path first; do not divide it by the BF16 peak. FP32 CUDA-core peak is 19.5e12 FLOP/s, TF32 is a different path.", DATASHEET),\\n            "gpu_hbm_bytes": resource("byte", hbm,\\n                "Total HBM traffic, not model-file size or PCIe traffic. Decimal GB/s; theoretical peak, not measured bandwidth.", A100_40_SOURCE if profile.endswith("40gb") else DATASHEET),\\n            "host_memory_bytes": resource("byte", None, "Host RAM topology and bandwidth are unknown."),\\n            "pcie_host_to_gpu_bytes": resource("byte", None, "Actual host-to-GPU topology/link width unknown. 64 GB/s bidirectional must not be used as one-way capacity."),\\n            "pcie_gpu_to_host_bytes": resource("byte", None, "Actual GPU-to-host topology/link width unknown. Keep transfer directions explicit."),\\n            "storage_read_bytes": resource("byte", None, "Backing-storage read bandwidth is unknown; distinguish page-cache hits from physical reads."),\\n            "network_bytes": resource("byte", None, "External network path and bandwidth are unknown."),\\n            "cpu_work": resource("operation", None, "Choose a measured CPU operation definition and matching peak before supplying work."),\\n        },\\n        "stages": [],\\n        "notes": [\\n            "This template contains no MiniMax-H3 FLOP or byte estimates and predicts no generation time.",\\n            "Add stage objects with id, depends_on, and work entries of {amount, unit}; null amount means unknown.",\\n            "Use distinct stages for non-overlappable work and preserve true dependencies. A fused stage assumes ideal overlap among its resources.",\\n            "Resource capacities describe shared resource pools. Do not duplicate one physical pool under different names to invent extra capacity.",\\n            "Omitted resource work is assumed zero for the declared model. coverage=complete is an explicit assertion by the profiler, not automatic discovery.",\\n            "Only use this full-GPU profile when the assigned GPU is the matching full A100, not a MIG slice.",\\n            "GPU matrix, memory, launch, decode, I/O and transfer work still need profiling. Available RAM and VRAM capacity are feasibility constraints, not bandwidth.",\\n        ],\\n        "method_source": BACKGROUND,\\n    }\\n\\n\\ndef compute_performance_bounds(document):\\n    if not isinstance(document, dict):\\n        raise ValueError("Input must be a JSON object")\\n    coverage = document.get("coverage", "partial")\\n    if coverage not in {"partial", "complete"}:\\n        raise ValueError("coverage must be partial or complete")\\n    resources = document.get("resources")\\n    stages = document.get("stages")\\n    if not isinstance(resources, dict) or not resources:\\n        raise ValueError("resources must be a nonempty object")\\n    if not isinstance(stages, list):\\n        raise ValueError("stages must be a list")\\n    capacities, units = {}, {}\\n    for name, definition in resources.items():\\n        if not isinstance(name, str) or not name or not isinstance(definition, dict):\\n            raise ValueError("Every resource needs a nonempty name and an object definition")\\n        unit = definition.get("work_unit")\\n        if not isinstance(unit, str) or not unit.strip():\\n            raise ValueError(f"resources.{name}.work_unit must be explicit")\\n        units[name] = unit\\n        capacity = definition.get("capacity_per_second")\\n        capacities[name] = None if capacity is None else _number(capacity, f"resources.{name}.capacity_per_second", positive=True)\\n\\n    nodes, dependencies, weights, totals, missing = {}, {}, {}, {}, []\\n    known_usable_items = 0\\n    for index, stage in enumerate(stages):\\n        if not isinstance(stage, dict) or not isinstance(stage.get("id"), str) or not stage["id"]:\\n            raise ValueError(f"stages[{index}] requires a nonempty string id")\\n        name = stage["id"]\\n        if name in nodes:\\n            raise ValueError(f"Duplicate stage id: {name}")\\n        nodes[name] = stage\\n        deps = stage.get("depends_on", [])\\n        if not isinstance(deps, list) or any(not isinstance(dep, str) for dep in deps) or len(set(deps)) != len(deps):\\n            raise ValueError(f"stages.{name}.depends_on must be a list of unique stage ids")\\n        dependencies[name] = deps\\n        work = stage.get("work")\\n        if not isinstance(work, dict):\\n            raise ValueError(f"stages.{name}.work must be an object")\\n        node_times = []\\n        if not work:\\n            missing.append({"path": f"stages.{name}.work", "reason": "no workload supplied"})\\n        for resource, item in work.items():\\n            path = f"stages.{name}.work.{resource}"\\n            if resource not in resources:\\n                raise ValueError(f"{path} names an undefined resource")\\n            if not isinstance(item, dict) or item.get("unit") != units[resource]:\\n                raise ValueError(f"{path}.unit must equal {units[resource]!r}; implicit conversion is not performed")\\n            if "amount" not in item:\\n                raise ValueError(f"{path}.amount must be a number or explicit null")\\n            total = totals.setdefault(resource, {"known_amount": 0.0, "unit": units[resource], "has_unknown_work": False})\\n            if item["amount"] is None:\\n                total["has_unknown_work"] = True\\n                missing.append({"path": path + ".amount", "reason": "unknown workload"})\\n                continue\\n            amount = _number(item["amount"], path + ".amount")\\n            total["known_amount"] += amount\\n            if capacities[resource] is None:\\n                if amount > 0:\\n                    missing.append({"path": "resources." + resource + ".capacity_per_second", "reason": "unknown capacity for nonzero work"})\\n                else:\\n                    node_times.append(0.0)\\n                    known_usable_items += 1\\n            else:\\n                node_times.append(amount / capacities[resource])\\n                known_usable_items += 1\\n        weights[name] = max(node_times) if node_times else None\\n\\n    if not stages:\\n        missing.append({"path": "stages", "reason": "no workload DAG supplied"})\\n    followers = {name: [] for name in nodes}\\n    indegree = {name: len(deps) for name, deps in dependencies.items()}\\n    for name, deps in dependencies.items():\\n        for dep in deps:\\n            if dep not in nodes:\\n                raise ValueError(f"Stage {name!r} depends on unknown stage {dep!r}")\\n            followers[dep].append(name)\\n    ready = deque(name for name in nodes if indegree[name] == 0)\\n    ends, paths, order = {}, {}, []\\n    while ready:\\n        name = ready.popleft()\\n        order.append(name)\\n        predecessor = max(dependencies[name], key=lambda dep: ends[dep], default=None)\\n        ends[name] = (ends[predecessor] if predecessor is not None else 0.0) + (weights[name] or 0.0)\\n        paths[name] = (paths[predecessor] if predecessor is not None else []) + [name]\\n        for successor in followers[name]:\\n            indegree[successor] -= 1\\n            if indegree[successor] == 0:\\n                ready.append(successor)\\n    if len(order) != len(nodes):\\n        raise ValueError("The stage graph contains a dependency cycle")\\n\\n    resource_bounds = {\\n        name: total["known_amount"] / capacities[name]\\n        for name, total in totals.items() if capacities[name] is not None\\n    }\\n    usable = known_usable_items > 0\\n    ideal = max(resource_bounds.values(), default=0.0) if usable else None\\n    terminal = max(ends, key=ends.get, default=None)\\n    critical = ends[terminal] if terminal is not None and usable else None\\n    lower_bound = max(ideal, critical) if ideal is not None and critical is not None else None\\n    # Deduplicate shared missing capacity entries while keeping deterministic order.\\n    missing = [dict(path=path, reason=reason) for path, reason in dict.fromkeys((entry["path"], entry["reason"]) for entry in missing)]\\n    complete = coverage == "complete" and not missing and bool(stages)\\n    comparison = None\\n    observed = document.get("observed_seconds")\\n    if observed is not None:\\n        observed = _number(observed, "observed_seconds", positive=True)\\n        comparison = {\\n            "observed_seconds": observed,\\n            "scope": "declared_complete_workload" if complete else "partial_workload_only",\\n            "time_over_lower_bound": observed / lower_bound if lower_bound is not None and lower_bound > 0 else None,\\n            "lower_bound_fraction_of_observed": lower_bound / observed if lower_bound is not None else None,\\n            "inconsistent_observation": lower_bound is not None and observed + 1e-12 < lower_bound,\\n            "overall_efficiency": None,\\n            "note": "Ratios compare time with an optimistic bound, not measured device utilization or proven achievable efficiency. Partial workloads cannot establish whole-task efficiency.",\\n        }\\n    return {\\n        "schema_version": 1,\\n        "coverage": "declared_complete" if complete else "partial",\\n        "ideal_full_parallel_lower_bound_seconds": ideal,\\n        "critical_path_lower_bound_seconds": critical,\\n        "overall_lower_bound_seconds": lower_bound,\\n        "resource_lower_bounds_seconds": resource_bounds,\\n        "stage_lower_bounds_seconds": weights,\\n        "critical_path": paths[terminal] if terminal is not None and usable else [],\\n        "total_work_by_resource": totals,\\n        "missing": missing,\\n        "unavailable_capacities": [name for name, value in capacities.items() if value is None],\\n        "comparison": comparison,\\n        "interpretation": "Optimistic necessary lower bound under supplied resource pools and DAG, not an achievable runtime prediction. Independent resources and work inside each stage are assumed to overlap perfectly; dependencies cannot overlap.",\\n    }\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("input", nargs="?", type=Path, help="UTF-8 JSON workload and resource document")\\n    parser.add_argument("--template", choices=("a100-sxm-40gb", "a100-sxm-80gb"), help="Emit capacities only; no fake workload")\\n    parser.add_argument("--output", type=Path, help="Write UTF-8 JSON instead of stdout")\\n    args = parser.parse_args(argv)\\n    if bool(args.input) == bool(args.template):\\n        parser.error("Choose exactly one input JSON or --template")\\n    try:\\n        result = hardware_template(args.template) if args.template else compute_performance_bounds(json.loads(args.input.read_text(encoding="utf-8-sig")))\\n    except (ValueError, OSError) as exc:\\n        parser.error(str(exc))\\n    rendered = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\\\\n"\\n    if args.output:\\n        args.output.write_text(rendered, encoding="utf-8")\\n    else:\\n        print(rendered, end="")\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'性能参照模板.json\': \'{\\n  "schema_version": 1,\\n  "hardware": "a100-sxm-40gb",\\n  "coverage": "partial",\\n  "observed_seconds": null,\\n  "resources": {\\n    "gpu_bf16_dense_flop": {\\n      "work_unit": "FLOP",\\n      "capacity_per_second": 312000000000000.0,\\n      "note": "BF16 dense Tensor Core work only; 624 TFLOP/s sparsity peak does not apply. One multiply-add counts as 2 FLOPs.",\\n      "source": "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\\n    },\\n    "gpu_fp32_flop": {\\n      "work_unit": "FLOP",\\n      "capacity_per_second": null,\\n      "note": "Keep FP32-layer work separate. Determine actual instruction path first; do not divide it by the BF16 peak. FP32 CUDA-core peak is 19.5e12 FLOP/s, TF32 is a different path.",\\n      "source": "https://images.nvidia.com/data-center/a100/a100-datasheet.pdf"\\n    },\\n    "gpu_hbm_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": 1555000000000.0,\\n      "note": "Total HBM traffic, not model-file size or PCIe traffic. Decimal GB/s; theoretical peak, not measured bandwidth.",\\n      "source": "https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/"\\n    },\\n    "host_memory_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": null,\\n      "note": "Host RAM topology and bandwidth are unknown.",\\n      "source": null\\n    },\\n    "pcie_host_to_gpu_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": null,\\n      "note": "Actual host-to-GPU topology/link width unknown. 64 GB/s bidirectional must not be used as one-way capacity.",\\n      "source": null\\n    },\\n    "pcie_gpu_to_host_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": null,\\n      "note": "Actual GPU-to-host topology/link width unknown. Keep transfer directions explicit.",\\n      "source": null\\n    },\\n    "storage_read_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": null,\\n      "note": "Backing-storage read bandwidth is unknown; distinguish page-cache hits from physical reads.",\\n      "source": null\\n    },\\n    "network_bytes": {\\n      "work_unit": "byte",\\n      "capacity_per_second": null,\\n      "note": "External network path and bandwidth are unknown.",\\n      "source": null\\n    },\\n    "cpu_work": {\\n      "work_unit": "operation",\\n      "capacity_per_second": null,\\n      "note": "Choose a measured CPU operation definition and matching peak before supplying work.",\\n      "source": null\\n    }\\n  },\\n  "stages": [],\\n  "notes": [\\n    "This template contains no MiniMax-H3 FLOP or byte estimates and predicts no generation time.",\\n    "Add stage objects with id, depends_on, and work entries of {amount, unit}; null amount means unknown.",\\n    "Use distinct stages for non-overlappable work and preserve true dependencies. A fused stage assumes ideal overlap among its resources.",\\n    "Resource capacities describe shared resource pools. Do not duplicate one physical pool under different names to invent extra capacity.",\\n    "Omitted resource work is assumed zero for the declared model. coverage=complete is an explicit assertion by the profiler, not automatic discovery.",\\n    "Only use this full-GPU profile when the assigned GPU is the matching full A100, not a MIG slice.",\\n    "GPU matrix, memory, launch, decode, I/O and transfer work still need profiling. Available RAM and VRAM capacity are feasibility constraints, not bandwidth."\\n  ],\\n  "method_source": "https://docs.nvidia.com/deeplearning/performance/dl-performance-gpu-background/"\\n}\\n\', \'requirements.txt\': \'# Linux NVIDIA A100. Native BF16; no quantization dependencies.\\ntorch==2.10.0\\ntorchvision==0.25.0\\ntorchaudio==2.10.0\\ndiffusers==0.43.0\\ntransformers==5.7.0\\naccelerate>=1.10,<2\\nsafetensors>=0.6,<1\\nhuggingface-hub>=1.0,<2\\nnumpy>=1.26,<3\\nPillow>=11,<13\\neinops>=0.8,<1\\nav>=14,<18\\nsoundfile>=0.13,<1\\nimageio>=2.37,<3\\nimageio-ffmpeg>=0.6,<1\\nfastapi>=0.115,<1\\nuvicorn>=0.34,<1\\npython-multipart>=0.0.20,<1\\npsutil>=6,<8\\nthreadpoolctl>=3,<4\\nrequests>=2.32,<3\\n\', \'requirements-colab.txt\': \'# Colab: preserve its preinstalled torch / torchvision / torchaudio stack.\\n# Notebook adds exact-version constraints for installed members of that stack.\\ndiffusers==0.43.0\\ntransformers==5.7.0\\naccelerate>=1.10,<2\\nsafetensors>=0.6,<1\\nhuggingface-hub>=1.0,<2\\nnumpy>=1.26,<3\\nPillow>=11,<13\\neinops>=0.8,<1\\nav>=14,<18\\nsoundfile>=0.13,<1\\nimageio>=2.37,<3\\nimageio-ffmpeg>=0.6,<1\\nfastapi>=0.115,<1\\nuvicorn>=0.34,<1\\npython-multipart>=0.0.20,<1\\n# The notebook host uses psutil while pip runs: retain its preinstalled version.\\n# All process ownership/resource APIs used here are available in 5.9.5.\\npsutil>=5.9.5,<8\\nthreadpoolctl>=3,<4\\nrequests>=2.32,<3\\n\', \'requirements-lora.txt\': \'# Optional: install only when an explicit acceleration/style adapter is selected.\\n# Colab must apply the same exact torch-stack constraints as the base install.\\npeft==0.21.2\\n\', \'run.py\': \'"""Start/query/control the API, model container and global event manager."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport signal\\nimport time\\n\\nfrom startup import StartupSession\\nfrom control_protocol import rpc_call\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--port", type=int, default=8000)\\n    parser.add_argument("--staged", action="store_true", help="Explicit sequential BF16 loading for an A100 with >=70 GiB available host RAM; not the default resident profile")\\n    parser.add_argument("--public", action="store_true", help="Prepare a temporary tunnel alongside the model; readiness requires their live join")\\n    parser.add_argument("--install-cloudflared", action="store_true", help="With --public, fetch an official SHA-256 checked cloudflared release if needed")\\n    parser.add_argument("--attention-backend", choices=["auto", "native", "_native_flash"])\\n    parser.add_argument("--denoiser-offload", choices=["stream", "sync"])\\n    parser.add_argument("--acceleration-lora", choices=["none", "lightx2v-ref2va-4step"])\\n    parser.add_argument("--style-lora", choices=["none", "fal-realism-people"])\\n    parser.add_argument("--style-lora-scale", type=float)\\n    parser.add_argument("--media-workers", choices=["auto", "1", "2", "3", "4"], help="A ceiling; parallel preparation also requires a local measured profile")\\n    parser.add_argument("--media-profile", type=Path, help="Local calibration JSON for bounded CPU child scheduling")\\n    parser.add_argument("--validate-config", action="store_true", help="Validate configuration without initializing CUDA, downloading, or starting a server")\\n    parser.add_argument("--manager-url", help="Existing private loopback controller; use H3_CONTROL_KEY")\\n    parser.add_argument("--action", choices=["status", "events", "drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"], default="status")\\n    parser.add_argument("--force", action="store_true", help="Explicitly permit stopping an occupied model container")\\n    args = parser.parse_args()\\n    if args.manager_url:\\n        token = os.environ.get("H3_CONTROL_KEY", "")\\n        if len(token) < 32:\\n            raise ValueError("Set the existing private H3_CONTROL_KEY for controller access")\\n        operation = args.action if args.action in {"status", "events"} else "control"\\n        payload = {} if operation != "control" else {"action": args.action, "force": args.force}\\n        print(json.dumps(rpc_call(args.manager_url, token, operation, payload), ensure_ascii=False, indent=2))\\n        return\\n    for argument, name in {\\n        "attention_backend": "H3_ATTENTION_BACKEND", "denoiser_offload": "H3_DENOISER_OFFLOAD",\\n        "acceleration_lora": "H3_ACCELERATION_LORA", "style_lora": "H3_STYLE_LORA",\\n        "style_lora_scale": "H3_STYLE_LORA_SCALE", "media_workers": "H3_MEDIA_WORKERS",\\n        "media_profile": "H3_MEDIA_PROFILE",\\n    }.items():\\n        value = getattr(args, argument)\\n        if value is not None:\\n            os.environ[name] = str(value)\\n    from adapter_config import load_adapter_settings\\n    from runtime_options import RuntimeOptions\\n    runtime_options, adapters = RuntimeOptions.from_env(), load_adapter_settings()\\n    if os.environ.get("H3_MEDIA_WORKERS", "auto") not in {"auto", "1", "2", "3", "4"}:\\n        raise ValueError("H3_MEDIA_WORKERS must be auto or 1..4")\\n    if args.media_profile and not args.media_profile.is_file():\\n        raise FileNotFoundError("The media calibration profile does not exist")\\n    if args.validate_config:\\n        print(json.dumps({"configuration": "valid", "residency": "staged" if args.staged else "resident",\\n                          "runtime": runtime_options.__dict__, "adapters": adapters.as_dict(),\\n                          "media_workers": os.environ.get("H3_MEDIA_WORKERS", "auto"),\\n                          "gpu_started": False}, ensure_ascii=False, indent=2))\\n        return\\n    residency = "staged" if args.staged else "resident"\\n    new_key = not os.environ.get("H3_API_KEY")\\n    session = StartupSession(Path(__file__).resolve().parent, port=args.port, staged=args.staged,\\n                             public=args.public, install_cloudflared=args.install_cloudflared)\\n    if new_key:\\n        print("Temporary API key (shown once; keep private): " + session.api_key, flush=True)\\n    print(f"BF16 residency profile: {residency}; lifecycle status: {session.state_path}", flush=True)\\n    last_state = None\\n\\n    def show_state(state):\\n        nonlocal last_state\\n        summary = {name: state[name] for name in (\\n            "instance_id", "phase", "ready", "api_ready", "model_ready", "tunnel_connected", "public_ready", "public_url", "error")}\\n        if summary != last_state:\\n            print(json.dumps(summary, ensure_ascii=False), flush=True)\\n            last_state = summary\\n\\n    try:\\n        session.start()\\n        print(f"Manager control: {session.manager_url}; API log: {session.api_log}; model log: {session.server_log}", flush=True)\\n        try:\\n            state = session.wait_ready(on_update=show_state)\\n            print(f"Generation ready: http://127.0.0.1:{args.port}/docs", flush=True)\\n            if args.public:\\n                print(f"Public generation verified for this instance: {state[\\\'public_url\\\']}", flush=True)\\n        except (RuntimeError, TimeoutError) as error:\\n            print(f"Generation unavailable: {error}. Query/control remain available while the manager is alive.", flush=True)\\n        while True:\\n            state = session.snapshot()\\n            show_state(state)\\n            if state.get("manager_alive") is False and state["phase"] != "stopped":\\n                raise RuntimeError(state["error"])\\n            if state["phase"] == "stopped":\\n                break\\n            time.sleep(1)\\n    finally:\\n        session.stop()\\n\\n\\nif __name__ == "__main__":\\n    signal.signal(signal.SIGTERM, lambda *_: (_ for _ in ()).throw(KeyboardInterrupt()))\\n    try:\\n        main()\\n    except KeyboardInterrupt:\\n        print("Server stopped. Release the cloud GPU instance separately to stop provider billing.")\\n\', \'startup.py\': \'"""Managed launcher/client: global_manager owns API, model and tunnel lifecycles."""\\nfrom __future__ import annotations\\nfrom copy import deepcopy\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport secrets\\nimport signal\\nimport socket\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nimport uuid\\nfrom urllib.request import urlopen\\nfrom control_protocol import ControlError, rpc_call\\nfrom process_owner import ProcessOwner\\n\\n\\ndef assert_port_available(port):\\n    with socket.socket() as probe:\\n        probe.bind(("127.0.0.1", int(port)))\\n\\n\\ndef _free_port(excluded):\\n    while True:\\n        with socket.socket() as probe:\\n            probe.bind(("127.0.0.1", 0))\\n            port = probe.getsockname()[1]\\n        if port not in excluded:\\n            return port\\n\\n\\ndef probe_ready(url, instance_id, timeout=3):\\n    try:\\n        with urlopen(url.rstrip("/") + "/health", timeout=timeout) as response:\\n            data = json.loads(response.read(16384))\\n        return isinstance(data, dict) and data.get("ready") is True and data.get("instance_id") == instance_id\\n    except Exception:\\n        return False\\n\\n\\nclass StartupSession:\\n    def __init__(self, directory, *, port=8000, staged=False, public=False,\\n                 install_cloudflared=False, env=None, startup_timeout=3600,\\n                 tunnel_timeout=180, poll_interval=1, probe_timeout=3, clock=None,\\n                 start_model=True):\\n        self.directory = Path(directory).resolve()\\n        self.port, self.staged, self.public = int(port), bool(staged), bool(public)\\n        if not 1 <= self.port <= 65535 or min(startup_timeout, tunnel_timeout, poll_interval, probe_timeout) <= 0:\\n            raise ValueError("Valid ports and positive timeouts are required")\\n        self.install_cloudflared = bool(install_cloudflared)\\n        self.model_autostart = bool(start_model)\\n        self.startup_timeout, self.tunnel_timeout = startup_timeout, tunnel_timeout\\n        self.poll_interval, self.probe_timeout = poll_interval, probe_timeout\\n        self._clock = clock or time.monotonic\\n        self._environment = dict(os.environ if env is None else env)\\n        self.api_key = self._environment.get("H3_API_KEY") or secrets.token_urlsafe(32)\\n        self._control_key = self._environment.get("H3_CONTROL_KEY") or secrets.token_urlsafe(32)\\n        for token in (self.api_key, self._control_key):\\n            if len(token) < 32 or not token.isascii() or any(c.isspace() for c in token):\\n                raise ValueError("Tokens require at least 32 non-whitespace ASCII characters")\\n        if self.api_key == self._control_key:\\n            raise ValueError("Public and private control tokens must be distinct")\\n        self.instance_id = uuid.uuid4().hex\\n        self.root = Path(self._environment.get("H3_OUTPUT_DIR", self.directory / "h3_outputs")).resolve()\\n        self.run_directory = self.directory / "runtime" / self.instance_id\\n        self.state_path = self.run_directory / "status.json"\\n        self.events_path = self.run_directory / "events.jsonl"\\n        self.server_log = self.run_directory / "model.log"\\n        self.api_log = self.run_directory / "api.log"\\n        self.tunnel_log = self.run_directory / "tunnel.log"\\n        self.manager_log = self.run_directory / "manager.log"\\n        self.manager = self._output = self._started_at = self._manager_owner = None\\n        self._state_lock = threading.Lock()\\n        self._last = dict(instance_id=self.instance_id, phase="new", ready=False,\\n                          api_ready=False, model_ready=False, public_ready=False,\\n                          public_requested=self.public, public_url=None,\\n                          public_api_ready=False, tunnel_connected=False,\\n                          server_alive=False, model_alive=False, tunnel_alive=False,\\n                          manager_alive=False, cleanup_complete=True, shutdown_complete=True, error=None)\\n        self.manager_url = None\\n\\n    @staticmethod\\n    def _cleanup_confirmed(state):\\n        return (state.get("phase") in {"stopped", "failed"} and state.get("cleanup_complete") is True\\n                and state.get("shutdown_complete") is True\\n                and all(state.get(name) is False for name in ("server_alive", "model_alive", "tunnel_alive"))\\n                and not state.get("process_observation_errors"))\\n\\n    def _remember(self, state):\\n        # RPCs can finish out of order while the notebook polls during stop().\\n        # A verified terminal receipt is monotonic for this manager instance.\\n        with self._state_lock:\\n            if not self._cleanup_confirmed(self._last) or self._cleanup_confirmed(state):\\n                self._last = deepcopy(state)\\n            return deepcopy(self._last)\\n\\n    def _read_final_receipt(self):\\n        try:\\n            state = json.loads(self.state_path.read_text(encoding="utf-8"))\\n            if isinstance(state, dict) and state.get("instance_id") == self.instance_id and self._cleanup_confirmed(state):\\n                self._remember(state)\\n        except (OSError, ValueError):\\n            pass\\n\\n    def _manager_tree_alive(self):\\n        if self._manager_owner:\\n            return self._manager_owner.observe()["alive"]\\n        return self.manager is not None and self.manager.poll() is None\\n\\n    def _owner_observation(self):\\n        if self._manager_owner:\\n            return self._manager_owner.observe()\\n        alive = self.manager is not None and self.manager.poll() is None\\n        return {"alive": alive, "cleanup_complete": not alive, "errors": [],\\n                "pids": [self.manager.pid] if alive else [], "identities": [],\\n                "scope": "manager_launcher_only", "observed_at": time.time()}\\n\\n    def _unavailable_snapshot(self, owner_state, *, stopped=False):\\n        """Keep the last RPC as history; never mix its readiness with current exit evidence."""\\n        if owner_state.get("alive") is False:\\n            self._read_final_receipt()\\n        with self._state_lock:\\n            history = deepcopy(self._last)\\n        result = deepcopy(history)\\n        errors = list(owner_state.get("errors", []))\\n        tree_complete = (owner_state.get("cleanup_complete") is True and not errors\\n                         and owner_state.get("scope") != "manager_launcher_only")\\n        manager_pid = history.get("manager_pid", getattr(self.manager, "pid", None))\\n        manager_alive = (None if errors else manager_pid in owner_state.get("pids", []))\\n        if owner_state.get("scope") == "manager_launcher_only":\\n            manager_alive = owner_state["alive"]\\n        confirmed = self._cleanup_confirmed(history) and owner_state.get("cleanup_complete") is True and not errors\\n        children_exited = tree_complete or (self._cleanup_confirmed(history) and manager_alive is False\\n                                            and owner_state.get("scope") == "manager_launcher_only")\\n        child_alive = False if children_exited else None\\n        current_time = time.time()\\n        owner_observed_at = owner_state.get("observed_at")\\n        manager_observed_at = history.get("observed_at")\\n        observation_age = lambda timestamp: (round(max(0, current_time - timestamp), 3)\\n                                              if isinstance(timestamp, (int, float)) else None)\\n        modules = {name: {"alive": child_alive, "ready": False,\\n                          "observation_unknown": child_alive is None}\\n                   for name in ("api", "model", "tunnel")}\\n        modules["manager"] = {"alive": manager_alive, "ready": False,\\n                              "supervisor_alive": False if manager_alive is False else None,\\n                              "observation_unknown": manager_alive is None}\\n        current_owned = {}\\n        for name, previous in history.get("owned_processes", {}).items():\\n            current_owned[name] = {"alive": child_alive, "cleanup_complete": children_exited,\\n                                   "pids": [] if children_exited else None,\\n                                   "identities": deepcopy(previous.get("identities", [])),\\n                                   "errors": [] if children_exited else ["manager_observation_unavailable"],\\n                                   "observed_at": current_time}\\n        result.update(last_manager_observation=history, manager_process_tree=deepcopy(owner_state),\\n                      manager_observed_at=manager_observed_at, observed_at=current_time,\\n                      owner_observed_at=owner_observed_at,\\n                      observation_age_seconds=observation_age(owner_observed_at),\\n                      manager_observation_age_seconds=observation_age(manager_observed_at),\\n                      manager_launcher_pid=getattr(self.manager, "pid", None), manager_alive=manager_alive,\\n                      server_alive=child_alive, model_alive=child_alive, tunnel_alive=child_alive,\\n                      ready=False, api_ready=False, model_ready=False, public_ready=False,\\n                      public_api_ready=False, tunnel_connected=False, supervisor_alive=modules["manager"]["supervisor_alive"],\\n                      modules=modules, owned_processes=current_owned,\\n                      cleanup_complete=confirmed, observation_stale=True, manager_observation_stale=True,\\n                      phase="stopped" if confirmed else "failed" if stopped or manager_alive is False else "degraded")\\n        process_errors = deepcopy(history.get("process_observation_errors", {}))\\n        if errors:\\n            process_errors["session_owner"] = errors\\n        result["process_observation_errors"] = process_errors\\n        if isinstance(history.get("downloads"), dict):\\n            result["downloads"] = {**deepcopy(history["downloads"]), "model_ready": False,\\n                                   "observation_stale": True,\\n                                   "lifecycle": "stopped" if children_exited else "unavailable"}\\n        if not confirmed:\\n            result["error"] = ("Owned process cleanup remains unconfirmed" if errors else\\n                               "Manager exited before cleanup was confirmed" if manager_alive is False else\\n                               "Manager control observation unavailable")\\n        return result\\n\\n    def start(self, *, background=True):\\n        if self._started_at is not None:\\n            raise RuntimeError("Session already started; query or stop it")\\n        assert_port_available(self.port)\\n        manager_port = _free_port({self.port})\\n        model_port = _free_port({self.port, manager_port})\\n        self.manager_url = f"http://127.0.0.1:{manager_port}"\\n        self._environment.update(H3_API_KEY=self.api_key, H3_CONTROL_KEY=self._control_key,\\n                                 H3_INSTANCE_ID=self.instance_id, H3_OUTPUT_DIR=str(self.root),\\n                                 H3_STARTUP_TIMEOUT=str(self.startup_timeout), H3_TUNNEL_TIMEOUT=str(self.tunnel_timeout))\\n        self.run_directory.mkdir(parents=True, exist_ok=False)\\n        self._started_at = self._clock()\\n        self._last.update(phase="preparing", cleanup_complete=False, shutdown_complete=False)\\n        command = [sys.executable, "-u", str(self.directory / "global_manager.py"),\\n                   "--port", str(manager_port), "--api-port", str(self.port),\\n                   "--model-port", str(model_port), "--directory", str(self.directory),\\n                   "--instance-id", self.instance_id, "--root", str(self.root), "--parent-pid", str(os.getpid()),\\n                   "--startup-timeout", str(self.startup_timeout), "--probe-timeout", str(self.probe_timeout),\\n                   "--poll-interval", str(self.poll_interval), "--tunnel-timeout", str(self.tunnel_timeout)]\\n        if self.staged:\\n            command.append("--staged")\\n        if self.public:\\n            command.append("--public")\\n        if self.install_cloudflared:\\n            command.append("--install-cloudflared")\\n        if not self.model_autostart:\\n            command.append("--no-model-autostart")\\n        try:\\n            self._output = self.manager_log.open("w", encoding="utf-8")\\n            self.manager = subprocess.Popen(command, cwd=self.directory, env=self._environment,\\n                                             stdout=self._output, stderr=subprocess.STDOUT,\\n                                             start_new_session=os.name != "nt")\\n            self._manager_owner = ProcessOwner(self.manager)\\n            deadline = self._clock() + 20\\n            while self._clock() < deadline:\\n                if not self._manager_tree_alive():\\n                    raise RuntimeError("Global manager exited during launch; inspect manager.log")\\n                try:\\n                    state = rpc_call(self.manager_url, self._control_key, "status", timeout=1)\\n                    if state.get("instance_id") != self.instance_id:\\n                        raise RuntimeError("Manager identity mismatch")\\n                    self._remember(state)\\n                    self._manager_owner.observe()\\n                    return self\\n                except ControlError:\\n                    time.sleep(.1)\\n            raise TimeoutError("Global manager control endpoint did not start")\\n        except BaseException:\\n            self.stop()\\n            raise\\n\\n    def snapshot(self):\\n        if self._started_at is None:\\n            return dict(self._last, elapsed_seconds=0)\\n        owner_state = self._owner_observation()\\n        if owner_state["alive"]:\\n            try:\\n                state = rpc_call(self.manager_url, self._control_key, "status", timeout=self.probe_timeout)\\n                if not isinstance(state, dict) or state.get("instance_id") != self.instance_id:\\n                    raise ControlError("Manager identity mismatch", 502)\\n                state = self._remember(state)\\n                owner_state = self._owner_observation()\\n                manager_missing = (owner_state.get("scope") != "manager_launcher_only"\\n                                   and state.get("manager_pid") is not None\\n                                   and state["manager_pid"] not in owner_state.get("pids", []))\\n                if owner_state.get("errors") or not owner_state["alive"] or manager_missing:\\n                    return self._unavailable_snapshot(owner_state)\\n                return dict(state, manager_alive=True, manager_launcher_pid=self.manager.pid,\\n                            manager_process_tree=owner_state)\\n            except ControlError:\\n                return self._unavailable_snapshot(self._owner_observation())\\n        return self._unavailable_snapshot(owner_state)\\n\\n    def control(self, action, **parameters):\\n        if not self.manager or not self._manager_tree_alive():\\n            raise ControlError("Manager is not running", 503)\\n        return rpc_call(self.manager_url, self._control_key, "control",\\n                        {"action": action, "expected_instance_id": self.instance_id, **parameters}, timeout=10)\\n\\n    def command(self, command_id):\\n        return rpc_call(self.manager_url, self._control_key, "command", {"command_id": command_id})\\n\\n    def events(self, after=0, limit=100):\\n        return rpc_call(self.manager_url, self._control_key, "events", {"after": after, "limit": limit})\\n\\n    def wait_ready(self, on_update=None):\\n        while True:\\n            state = self.snapshot()\\n            if on_update:\\n                on_update(state)\\n            if state.get("ready"):\\n                return state\\n            if state.get("phase") in {"failed", "stopped", "stopping"} or state.get("error") and state.get("model_alive") is False:\\n                raise RuntimeError(state.get("error") or "Preparation failed; control interface remains available")\\n            if self._clock() - self._started_at > self.startup_timeout:\\n                raise TimeoutError("Generation readiness deadline exceeded; query or stop the manager")\\n            time.sleep(self.poll_interval)\\n\\n    def stop(self):\\n        if self.manager is None:\\n            self._last.update(phase="stopped", cleanup_complete=True, manager_alive=False)\\n            return dict(self._last)\\n        if self._manager_tree_alive():\\n            deadline = time.monotonic() + 30\\n            requested = False\\n            command_id = "shutdown-" + uuid.uuid4().hex\\n            while time.monotonic() < deadline and self._manager_tree_alive():\\n                try:\\n                    if not requested:\\n                        self.control("shutdown", command_id=command_id)\\n                        requested = True\\n                    self.snapshot()\\n                except (ControlError, OSError):\\n                    # A lost acknowledgement is not permission to interrupt audit\\n                    # finalization. Retrying this ID cannot enqueue a second stop.\\n                    pass\\n                if self.manager.poll() is not None:\\n                    break\\n                time.sleep(.2)\\n            # The manager exits itself only after its final command/audit checkpoint.\\n            # A clean child snapshot alone must not trigger an early manager signal.\\n        if self._manager_owner:\\n            owner_state = self._manager_owner.observe()\\n            if owner_state["alive"]:\\n                owner_state = self._manager_owner.terminate_all(timeout=15)\\n        else:\\n            if self.manager.poll() is None:\\n                self.manager.terminate()\\n                self.manager.wait(timeout=15)\\n            owner_state = self._owner_observation()\\n        if self._output and not self._output.closed:\\n            self._output.close()\\n        try:\\n            recorded = json.loads(self.state_path.read_text(encoding="utf-8"))\\n            if recorded.get("instance_id") == self.instance_id:\\n                self._remember(recorded)\\n        except (OSError, ValueError):\\n            pass\\n        return self._unavailable_snapshot(owner_state, stopped=True)\\n\', \'control_protocol.py\': \'"""Small authenticated loopback RPC shared by the three managed processes."""\\nfrom __future__ import annotations\\n\\nimport hmac\\nimport json\\nimport logging\\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\nfrom urllib.error import HTTPError, URLError\\nfrom urllib.parse import urlsplit\\nfrom urllib.request import Request, urlopen\\n\\nMAX_BODY = 2 * 1024 * 1024\\nLOG = logging.getLogger("h3.control")\\n\\n\\nclass ControlError(RuntimeError):\\n    def __init__(self, message: str, status: int = 409):\\n        super().__init__(message)\\n        self.status = int(status)\\n\\n\\ndef rpc_call(url, key, operation, payload=None, timeout=10):\\n    origin = urlsplit(url)\\n    if origin.scheme != "http" or origin.hostname not in {"127.0.0.1", "localhost", "::1"}:\\n        raise ControlError("Private control transport must use loopback HTTP", 400)\\n    if origin.username or origin.password or origin.query or origin.fragment:\\n        raise ControlError("Invalid control origin", 400)\\n    body = json.dumps({"operation": operation, "payload": payload or {}}, allow_nan=False).encode()\\n    if len(body) > MAX_BODY:\\n        raise ControlError("Control payload too large", 413)\\n    request = Request(url.rstrip("/") + "/rpc", data=body, method="POST", headers={\\n        "Authorization": "Bearer " + key, "Content-Type": "application/json", "Connection": "close"})\\n    try:\\n        try:\\n            response = urlopen(request, timeout=timeout)\\n        except HTTPError as error:\\n            response = error\\n        with response:\\n            raw = response.read(MAX_BODY + 1)\\n            status = response.status\\n        if len(raw) > MAX_BODY:\\n            raise ControlError("Control response too large", 502)\\n        data = json.loads(raw)\\n        if not isinstance(data, dict) or type(data.get("ok")) is not bool:\\n            raise ValueError("Invalid response envelope")\\n        if data["ok"] is not True or not 200 <= status < 300:\\n            raise ControlError(str(data.get("error", "Control operation failed")), data.get("status", status))\\n        return data.get("result")\\n    except ControlError:\\n        raise\\n    except (URLError, OSError, ValueError, TimeoutError) as error:\\n        raise ControlError("Control transport unavailable or invalid response", 503) from error\\n\\n\\ndef make_server(host, port, key, dispatch):\\n    if host not in {"127.0.0.1", "localhost"}:\\n        raise ValueError("Private controller must bind IPv4 loopback")\\n    if not isinstance(key, str) or len(key) < 32 or not key.isascii() or any(c.isspace() for c in key):\\n        raise ValueError("A private control token of at least 32 ASCII characters is required")\\n\\n    class Handler(BaseHTTPRequestHandler):\\n        protocol_version = "HTTP/1.1"\\n\\n        def setup(self):\\n            super().setup()\\n            self.connection.settimeout(15)\\n\\n        def log_message(self, *_):\\n            pass  # Never log authentication headers, payloads or query secrets.\\n\\n        def reply(self, status, value):\\n            encoded = json.dumps(value, allow_nan=False).encode()\\n            self.send_response(status)\\n            self.send_header("Content-Type", "application/json")\\n            self.send_header("Content-Length", str(len(encoded)))\\n            self.send_header("Cache-Control", "no-store")\\n            self.send_header("Connection", "close")\\n            self.end_headers()\\n            self.close_connection = True\\n            try:\\n                self.wfile.write(encoded)\\n            except (BrokenPipeError, ConnectionResetError):\\n                pass\\n\\n        def do_GET(self):\\n            if self.path != "/health":\\n                return self.reply(404, {"ok": False, "error": "Not found", "status": 404})\\n            self.reply(200, {"alive": True})\\n\\n        def do_POST(self):\\n            if self.path != "/rpc":\\n                return self.reply(404, {"ok": False, "error": "Not found", "status": 404})\\n            auth = self.headers.get("Authorization", "")\\n            if not auth.isascii() or not hmac.compare_digest(auth, "Bearer " + key):\\n                return self.reply(401, {"ok": False, "error": "Private authorization required", "status": 401})\\n            try:\\n                if self.headers.get("Transfer-Encoding"):\\n                    raise ControlError("Chunked control requests are unsupported", 400)\\n                size = int(self.headers.get("Content-Length", "0"))\\n                if not 0 < size <= MAX_BODY:\\n                    raise ControlError("Invalid control payload size", 413)\\n                raw = self.rfile.read(size)\\n                if len(raw) != size:\\n                    raise ControlError("Incomplete control request", 400)\\n                data = json.loads(raw, parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))\\n                if not isinstance(data, dict) or not isinstance(data.get("operation"), str):\\n                    raise ControlError("Invalid control operation", 400)\\n                payload = data.get("payload", {})\\n                if not isinstance(payload, dict):\\n                    raise ControlError("Control payload must be an object", 400)\\n                result = dispatch(data["operation"], payload)\\n                self.reply(200, {"ok": True, "result": result})\\n            except ControlError as error:\\n                self.reply(error.status, {"ok": False, "error": str(error), "status": error.status})\\n            except (ValueError, TypeError) as error:\\n                self.reply(400, {"ok": False, "error": "Invalid control JSON or arguments", "status": 400})\\n            except Exception:\\n                LOG.exception("Control dispatch failed")\\n                self.reply(500, {"ok": False, "error": "Internal control operation failed; inspect owner log", "status": 500})\\n\\n    server = ThreadingHTTPServer((host, int(port)), Handler)\\n    server.daemon_threads = True\\n    return server\\n\', \'process_owner.py\': \'"""Track and stop an owned Popen tree by PID and creation identity.\\n\\nWindows virtual-environment launchers can add a real interpreter descendant.\\nAn exited wrapper alone is not the definition of a fully reclaimed process tree.\\n"""\\nfrom __future__ import annotations\\n\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport subprocess\\nimport threading\\nimport time\\n\\nimport psutil\\n\\n\\nclass ProcessOwner:\\n    def __init__(self, popen):\\n        self.process = popen\\n        self._identities = {}\\n        self._lock = threading.RLock()\\n        self._root_unknown = False\\n        self._unknown_scopes = set()\\n        try:\\n            process = psutil.Process(popen.pid)\\n            self._identities[process.pid] = process.create_time()\\n        except psutil.Error:\\n            self._root_unknown = True\\n            self._unknown_scopes.add(popen.pid)\\n\\n    def _same(self, pid, created):\\n        try:\\n            process = psutil.Process(pid)\\n            if abs(process.create_time() - created) >= .001:\\n                return None\\n            if not process.is_running() or process.status() == psutil.STATUS_ZOMBIE:\\n                return None\\n            return process\\n        except psutil.NoSuchProcess:\\n            return None\\n\\n    def _capture(self):\\n        errors = []\\n        if self._root_unknown:\\n            try:\\n                if self.process.poll() is None:\\n                    root = psutil.Process(self.process.pid)\\n                    self._identities[root.pid] = root.create_time()\\n                    self._root_unknown = False\\n                # An exited wrapper cannot prove an unobserved subtree exited.\\n            except (OSError, psutil.Error) as error:\\n                errors.append("root.poll: " + type(error).__name__)\\n        for pid, created in list(self._identities.items()):\\n            try:\\n                process = self._same(pid, created)\\n                if process is None:\\n                    continue\\n                children = process.children(recursive=True)\\n                complete = True\\n                for child in children:\\n                    try:\\n                        self._identities.setdefault(child.pid, child.create_time())\\n                    except psutil.NoSuchProcess:\\n                        pass\\n                    except psutil.Error as error:\\n                        errors.append("descendant.identity: " + type(error).__name__)\\n                        complete = False\\n                if complete:\\n                    self._unknown_scopes.discard(pid)\\n                else:\\n                    self._unknown_scopes.add(pid)\\n            except (psutil.NoSuchProcess, psutil.ZombieProcess):\\n                # A known process can exit between identity validation and\\n                # children(). This is an exit observation, not a new unknown\\n                # subtree. Previously recorded access/identity uncertainty is\\n                # deliberately retained until a successful complete recapture.\\n                pass\\n            except psutil.Error as error:\\n                errors.append(str(pid) + ".children: " + type(error).__name__)\\n                self._unknown_scopes.add(pid)\\n        errors.extend(str(pid) + ".subtree_exit_unconfirmed" for pid in self._unknown_scopes)\\n        return errors\\n\\n    def observe(self):\\n        with self._lock:\\n            errors = self._capture()\\n            alive = []\\n            for pid, created in self._identities.items():\\n                try:\\n                    if self._same(pid, created) is not None:\\n                        alive.append(pid)\\n                except psutil.Error as error:\\n                    alive.append(pid)  # Unknown exit retains ownership.\\n                    errors.append(str(pid) + ".observe: " + type(error).__name__)\\n            try:\\n                root_running = self.process.poll() is None\\n            except OSError as error:\\n                root_running = True\\n                errors.append("root.poll: " + type(error).__name__)\\n            if root_running and self.process.pid not in alive:\\n                alive.append(self.process.pid)\\n                if self.process.pid not in self._identities:\\n                    errors.append("root_identity_unconfirmed")\\n            return {"alive": bool(alive) or bool(errors), "cleanup_complete": not alive and not errors,\\n                    "pids": sorted(set(alive)), "identities": [{"pid": pid, "created_at": created}\\n                     for pid, created in self._identities.items()], "errors": sorted(set(errors)),\\n                    "observed_at": time.time()}\\n\\n    def _signal(self, kill=False):\\n        with self._lock:\\n            self._capture()\\n            root_signalled = False\\n            for pid, created in list(self._identities.items()):\\n                try:\\n                    process = self._same(pid, created)\\n                    if process:\\n                        process.kill() if kill else process.terminate()\\n                        root_signalled |= pid == self.process.pid\\n                except psutil.Error:\\n                    pass  # Final observation, not signal success, decides cleanup.\\n            # The original Popen handle is tied to its original process. This is\\n            # also the only safe fallback when its psutil identity was unavailable.\\n            try:\\n                if not root_signalled and self.process.poll() is None:\\n                    self.process.kill() if kill else self.process.terminate()\\n            except (OSError, subprocess.SubprocessError):\\n                pass\\n\\n    def request_stop(self):\\n        self._signal()\\n        return self.observe()\\n\\n    def wait_stopped(self, timeout=15):\\n        deadline = time.monotonic() + max(0, timeout)\\n        while True:\\n            state = self.observe()\\n            if state["cleanup_complete"]:\\n                return state\\n            if time.monotonic() >= deadline:\\n                break\\n            time.sleep(min(.05, max(0, deadline - time.monotonic())))\\n        self._signal(kill=True)\\n        deadline = time.monotonic() + 3\\n        while True:\\n            state = self.observe()\\n            if state["cleanup_complete"] or time.monotonic() >= deadline:\\n                return state\\n            time.sleep(.025)\\n\\n    def terminate_all(self, timeout=15):\\n        self.request_stop()\\n        return self.wait_stopped(timeout)\\n\\n\\ndef stop_many(owners, timeout=15):\\n    """Independent stop requests and waits overlap; caller joins actual evidence."""\\n    owners = list(owners)\\n    for owner in owners:\\n        owner.request_stop()\\n    if not owners:\\n        return []\\n    try:\\n        with ThreadPoolExecutor(max_workers=len(owners), thread_name_prefix="owned-process-cleanup") as pool:\\n            futures = [pool.submit(owner.wait_stopped, timeout) for owner in owners]\\n            return [future.result() for future in futures]\\n    except (RuntimeError, OSError):\\n        # Thread creation failure must not prevent already-requested cleanup.\\n        deadline = time.monotonic() + timeout\\n        return [owner.wait_stopped(max(0, deadline - time.monotonic())) for owner in owners]\\n\', \'global_manager.py\': \'"""Durable main-event owner for independent API and native-BF16 model processes."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom copy import deepcopy\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport hashlib\\nimport json\\nimport logging\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport signal\\nimport sqlite3\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nimport urllib.request\\nimport uuid\\n\\nimport psutil\\n\\nfrom control_protocol import ControlError, make_server, rpc_call\\nfrom process_owner import ProcessOwner, stop_many\\nfrom generation_options import requested_options, validate_generation_options\\nfrom adapter_config import load_adapter_settings\\nfrom condition_guard import evaluate_gate\\n\\nLOG = logging.getLogger("h3_manager")\\nTERMINAL = {"succeeded", "failed", "cancelled"}\\nNONTERMINAL = {"queued", "running", "cancel_requested"}\\nIDENTIFIER = re.compile(r"[a-f0-9]{32}")\\nDRIVE_FILE_ID = re.compile(r"[A-Za-z0-9_-]{1,128}")\\nSHA256 = re.compile(r"[a-f0-9]{64}")\\nGATE_SOURCE_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\nDISPATCH_CONDITIONS = ("manager_open", "job_queued", "attempt_unassigned", "model_identity_current",\\n                       "model_observation_fresh", "model_process_alive", "model_ready", "worker_alive",\\n                       "worker_accepting", "supervisor_alive")\\n\\n\\nclass RootLock:\\n    """An OS-held lock, automatically released if the owner process dies."""\\n\\n    def __init__(self, path):\\n        self.file = open(path, "a+b")\\n        try:\\n            if os.fstat(self.file.fileno()).st_size == 0:\\n                self.file.write(b"0")\\n                self.file.flush()\\n            self.file.seek(0)\\n            if os.name == "nt":\\n                import msvcrt\\n                msvcrt.locking(self.file.fileno(), msvcrt.LK_NBLCK, 1)\\n            else:\\n                import fcntl\\n                fcntl.flock(self.file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\\n        except (OSError, IOError) as error:\\n            self.file.close()\\n            raise RuntimeError("Another manager owns this output directory") from error\\n\\n    def close(self):\\n        if self.file.closed:\\n            return\\n        self.file.seek(0)\\n        if os.name == "nt":\\n            import msvcrt\\n            msvcrt.locking(self.file.fileno(), msvcrt.LK_UNLCK, 1)\\n        else:\\n            import fcntl\\n            fcntl.flock(self.file.fileno(), fcntl.LOCK_UN)\\n        self.file.close()\\n\\n\\ndef probe_api(url, instance_id, timeout=2):\\n    try:\\n        req = urllib.request.Request(url.rstrip("/") + "/health/live",\\n                                     headers={"Cache-Control": "no-cache"})\\n        with urllib.request.urlopen(req, timeout=timeout) as response:\\n            body = json.loads(response.read(16385))\\n        return isinstance(body, dict) and body.get("instance_id") == instance_id and body.get("api_ready") is True\\n    except Exception:\\n        return False\\n\\n\\nclass GlobalManager:\\n    def __init__(self, directory, *, root, instance_id, port, api_port, model_port,\\n                 staged=False, public=False, install_cloudflared=False, env=None,\\n                 poll_interval=1, probe_timeout=2, startup_timeout=3600, generation_timeout=5400, tunnel_timeout=180,\\n                 popen=None, rpc=None, probe=None, clock=None, owner_factory=None, start_model=True):\\n        if not IDENTIFIER.fullmatch(instance_id):\\n            raise ValueError("Invalid manager instance ID")\\n        self.directory, self.root = Path(directory).resolve(), Path(root).resolve()\\n        self.instance_id = instance_id\\n        self.port, self.api_port, self.model_port = int(port), int(api_port), int(model_port)\\n        if len({self.port, self.api_port, self.model_port}) != 3:\\n            raise ValueError("Manager, API and model ports must be distinct")\\n        self.staged, self.public = bool(staged), bool(public)\\n        self.install_cloudflared = bool(install_cloudflared)\\n        self.model_autostart = bool(start_model)\\n        self.env = dict(os.environ if env is None else env)\\n        self.key = self.env.get("H3_CONTROL_KEY", "")\\n        if len(self.key) < 32 or len(self.env.get("H3_API_KEY", "")) < 32:\\n            raise ValueError("Separate private/public keys of at least 32 characters are required")\\n        if self.key == self.env["H3_API_KEY"]:\\n            raise ValueError("Private and public tokens must differ")\\n        self.poll_interval, self.probe_timeout = poll_interval, probe_timeout\\n        self.startup_timeout = startup_timeout\\n        self.generation_timeout = generation_timeout\\n        self.tunnel_timeout = tunnel_timeout\\n        self.disk_reserve_bytes = max(0, int(self.env.get("H3_DISK_RESERVE_BYTES", 64 * 1024 * 1024)))\\n        self._popen, self._rpc, self._probe = popen or subprocess.Popen, rpc or rpc_call, probe or probe_api\\n        self._owner_factory = owner_factory or ProcessOwner\\n        self._clock = clock or time.monotonic\\n        self._started = self._clock()\\n        self.run_directory = self.directory / "runtime" / instance_id\\n        self.run_directory.mkdir(parents=True, exist_ok=True)\\n        self.root.mkdir(parents=True, exist_ok=True)\\n        self.state_path, self.events_path = self.run_directory / "status.json", self.run_directory / "events.jsonl"\\n        self._owner = RootLock(self.root / "owner.lock")\\n        self._db = sqlite3.connect(self.root / "control.sqlite3", check_same_thread=False)\\n        self._db.execute("PRAGMA journal_mode=WAL")\\n        self._db.execute("PRAGMA synchronous=FULL")\\n        self._db.executescript("""\\n            CREATE TABLE IF NOT EXISTS jobs (id TEXT PRIMARY KEY, status TEXT NOT NULL,\\n                idempotency_key TEXT UNIQUE, fingerprint TEXT, record TEXT NOT NULL);\\n            CREATE INDEX IF NOT EXISTS jobs_status ON jobs(status);\\n            CREATE TABLE IF NOT EXISTS events (sequence INTEGER PRIMARY KEY AUTOINCREMENT,\\n                event_id TEXT UNIQUE NOT NULL, record TEXT NOT NULL);\\n            CREATE TABLE IF NOT EXISTS commands (id TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,\\n                record TEXT NOT NULL);\\n            CREATE TABLE IF NOT EXISTS owned_processes (launch_id TEXT PRIMARY KEY, module TEXT NOT NULL,\\n                pid INTEGER NOT NULL, created REAL, exited INTEGER NOT NULL DEFAULT 0);\\n        """)\\n        self._lock = threading.RLock()\\n        self._control_lock = threading.RLock()\\n        self._refresh_lock = threading.Lock()\\n        self._cancel = threading.Event()\\n        self._thread = None\\n        self._control_threads = []\\n        self._control_admission_closed = False\\n        self._control_slots = threading.BoundedSemaphore(4)\\n        self.children = {"api": None, "model": None, "tunnel": None}\\n        self.owners = {}\\n        self._files = []\\n        self.model_instance_id = None\\n        self._model = {}\\n        self._api_ready = self._public_api_ready = self._connected = False\\n        self._api_expected = False\\n        self._api_expected_elapsed = 0.0\\n        self._api_expected_since = None\\n        self._api_intent_epoch = 0\\n        self._api_lifecycle = "not_started"\\n        self._public_url = None\\n        self._phase = "new"\\n        self._draining = False\\n        self._shutdown_started = False\\n        self._shutdown_complete = False\\n        self._exit_requested = False\\n        self._last_observation = None\\n        self._model_failure = None\\n        self._error = None\\n        self._cleanup_errors = []\\n        self._signature = None\\n        self._condition_gates = {}\\n        self._dispatch_gate_signature = None\\n        self._export_sequence = 0\\n        self._background = False\\n        self._ever_ready = False\\n        self._model_started = None\\n        self._model_unreachable = 0\\n        self._model_ever_ready = False\\n        self._public_ever_ready = False\\n        self._probe_pool = ThreadPoolExecutor(max_workers=2, thread_name_prefix="independent-module-probe")\\n        self._closed = False\\n        self._recover()\\n\\n    def _event(self, kind, *, job=None, data=None):\\n        event = {"event_id": uuid.uuid4().hex, "instance_id": self.instance_id,\\n                 "job_id": job.get("job_id") if job else None,\\n                 "attempt_id": job.get("attempt_id") if job else None,\\n                 "type": kind, "observed_at": time.time(),\\n                 "monotonic_ns": time.monotonic_ns(), "data": data or {}}\\n        cursor = self._db.execute("INSERT INTO events(event_id, record) VALUES (?, ?)",\\n                                  (event["event_id"], json.dumps(event)))\\n        event["sequence"] = cursor.lastrowid\\n        return event\\n\\n    def _recover(self):\\n        # OS root ownership is held, but previous child fate may be uncertain.\\n        # Never replay an accepted expensive execution after manager replacement.\\n        with self._lock, self._db:\\n            for (encoded,) in self._db.execute("SELECT record FROM jobs WHERE status IN (\\\'queued\\\',\\\'running\\\',\\\'cancel_requested\\\')").fetchall():\\n                job = json.loads(encoded)\\n                self._transition(job, "failed", error="manager_restarted; execution not replayed",\\n                                 cleanup_complete=False, finished_at=time.time())\\n            for command_id, encoded in self._db.execute("SELECT id,record FROM commands").fetchall():\\n                command = json.loads(encoded)\\n                if command["status"] in {"accepted", "running"}:\\n                    command.update(status="failed", error="manager_restarted; command not replayed", finished_at=time.time())\\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(command), command_id))\\n            self._event("manager.created", data={"recovery_policy": "no automatic execution replay"})\\n\\n    def _get_job(self, job_id):\\n        row = self._db.execute("SELECT record FROM jobs WHERE id=?", (job_id,)).fetchone()\\n        if not row:\\n            raise ControlError("Job not found", status=404)\\n        return json.loads(row[0])\\n\\n    def _store_job(self, job):\\n        self._db.execute("UPDATE jobs SET status=?,record=? WHERE id=?",\\n                         (job["status"], json.dumps(job), job["job_id"]))\\n\\n    def _transition(self, job, target, **updates):\\n        allowed = {"queued": {"running", "failed", "cancelled"},\\n                   "running": {"queued", "succeeded", "failed", "cancel_requested", "cancelled"},\\n                   "cancel_requested": {"cancelled", "failed"}}\\n        if target not in allowed.get(job["status"], set()):\\n            raise RuntimeError(f"Illegal main event transition {job[\\\'status\\\']} -> {target}")\\n        job.update(status=target, **updates)\\n        job["lifecycle"].append({"state": target, "observed_at": time.time()})\\n        self._store_job(job)\\n        self._event("job." + target, job=job, data={"error": job.get("error")})\\n\\n    def _active_jobs(self):\\n        rows = self._db.execute("SELECT record FROM jobs WHERE status IN (\\\'running\\\',\\\'cancel_requested\\\') ORDER BY rowid").fetchall()\\n        return [json.loads(row[0]) for row in rows]\\n\\n    def _queue_summary(self):\\n        counts = dict(self._db.execute("SELECT status,COUNT(*) FROM jobs GROUP BY status").fetchall())\\n        return {"queued": counts.get("queued", 0), "running": counts.get("running", 0),\\n                "cancel_requested": counts.get("cancel_requested", 0),\\n                "unfinished": sum(counts.get(s, 0) for s in NONTERMINAL),\\n                "job_count_limit": None, "order": "durable_insertion_fifo",\\n                "admission_open": not (self._draining or self._shutdown_started or self._model_failure\\n                                        or self._phase in {"new", "failed", "stopped"}),\\n                "disk_reserve_bytes": self.disk_reserve_bytes}\\n\\n    def _liveness(self):\\n        alive, errors = {}, {}\\n        for name, child in self.children.items():\\n            try:\\n                if name in self.owners:\\n                    observation = self.owners[name].observe()\\n                    alive[name] = observation["alive"]\\n                    if observation["errors"]:\\n                        errors[name] = "; ".join(observation["errors"])\\n                else:\\n                    alive[name] = child is not None and child.poll() is None\\n            except OSError as error:\\n                alive[name] = True  # Unknown exit retains ownership.\\n                errors[name] = type(error).__name__\\n        return alive, errors\\n\\n    def _historical_ownership(self):\\n        current = {launch_id for child in self.children.values() if child is not None\\n                   for launch_id in getattr(child, "_h3_tree_records", {}).values()}\\n        unresolved = []\\n        rows = self._db.execute("SELECT launch_id,module,pid,created FROM owned_processes WHERE exited=0").fetchall()\\n        for launch_id, module, pid, created in rows:\\n            if launch_id in current:\\n                continue\\n            try:\\n                process = psutil.Process(pid)\\n                same = created is None or abs(process.create_time() - created) < .001\\n                if same and process.is_running() and process.status() != psutil.STATUS_ZOMBIE:\\n                    unresolved.append({"module": module, "pid": pid, "identity_known": created is not None})\\n            except psutil.NoSuchProcess:\\n                pass\\n            except psutil.Error:\\n                unresolved.append({"module": module, "pid": pid, "identity_known": False})\\n        return unresolved\\n\\n    def snapshot(self):\\n        with self._lock:\\n            alive, process_errors = self._liveness()\\n            historical = self._historical_ownership()\\n            age = self._clock() - self._last_observation if self._last_observation is not None else None\\n            stale = age is None or age > 3 * self.probe_timeout + self.poll_interval + 2\\n            supervised = not self._background or (self._thread is not None and self._thread.is_alive())\\n            usable = not stale and supervised and not process_errors and not self._shutdown_started\\n            api = bool(usable and alive["api"] and self._api_ready)\\n            model = bool(usable and alive["model"] and self._model.get("ready") is True\\n                         and self._model.get("worker_alive") is True and not self._model_failure)\\n            public_api = bool(api and alive["tunnel"] and self._connected and self._public_api_ready)\\n            ready = api and model and not self._draining and (not self.public or public_api)\\n            phase = self._phase\\n            if phase not in {"new", "stopping", "stopped", "failed"}:\\n                phase = "draining" if self._draining else "ready" if ready else "degraded" if self._ever_ready or self._model_failure or self._error else "preparing"\\n            downloads = self._model.get("downloads")\\n            if (not isinstance(downloads, dict) or downloads.get("model_instance_id") != self.model_instance_id\\n                    or not isinstance(downloads.get("items"), dict)):\\n                downloads = {"schema_version": 1, "model_instance_id": self.model_instance_id,\\n                             "observed_at": None, "observation_stale": True, "items": {},\\n                             "error": "progress_not_observed_for_current_model"}\\n            else:\\n                downloads = deepcopy(downloads)\\n            download_lifecycle = (("stopping" if alive["model"] else "stopped") if self._shutdown_started else\\n                                  "failed" if self._model_failure and self._model_failure != "model_stopped" else\\n                                  "not_started" if self.model_instance_id is None else\\n                                  "stopped" if alive["model"] is False else self._model.get("phase", "unknown"))\\n            downloads.update(model_ready=model, observation_stale=bool(downloads.get("observation_stale")\\n                             or stale or not supervised or alive["model"] is not True or "model" in process_errors),\\n                             lifecycle=download_lifecycle)\\n            result = {"instance_id": self.instance_id, "phase": phase, "ready": ready,\\n                      "api_ready": api, "model_ready": model, "public_requested": self.public,\\n                      "public_api_ready": public_api, "public_ready": public_api and model and not self._draining,\\n                      "tunnel_connected": bool(alive["tunnel"] and self._connected), "public_url": self._public_url,\\n                      "server_alive": alive["api"], "model_alive": alive["model"], "tunnel_alive": alive["tunnel"],\\n                      "manager_alive": True, "manager_pid": os.getpid(), "api_pid": getattr(self.children["api"], "pid", None),\\n                      "server_pid": getattr(self.children["api"], "pid", None),\\n                      "model_pid": getattr(self.children["model"], "pid", None), "tunnel_pid": getattr(self.children["tunnel"], "pid", None),\\n                      "model_instance_id": self.model_instance_id,\\n                      "downloads": downloads,\\n                      "cleanup_complete": not any(alive.values()) and not historical and not process_errors,\\n                      "shutdown_complete": self._shutdown_complete,\\n                      "historical_ownership": historical, "cleanup_errors": list(self._cleanup_errors),\\n                      "error": self._error or self._model_failure, "elapsed_seconds": round(self._clock() - self._started, 3),\\n                      "observed_at": time.time(), "observation_age_seconds": round(age, 3) if age is not None else None,\\n                      "observation_stale": stale, "supervisor_alive": supervised,\\n                      "process_observation_errors": process_errors, "draining": self._draining,\\n                      "condition_gates": deepcopy(self._condition_gates),\\n                      "queue": self._queue_summary(),\\n                      "owned_processes": {name: owner.observe() for name, owner in self.owners.items()},\\n                      "state_path": str(self.state_path), "events_path": str(self.events_path),\\n                      "modules": {"api": {"alive": alive["api"], "ready": api,\\n                                          "expected_running": self._api_expected, "lifecycle": self._api_lifecycle,\\n                                          "intent_epoch": self._api_intent_epoch},\\n                                  "model": {**self._model, "alive": alive["model"], "ready": model, "downloads": downloads},\\n                                  "manager": {"alive": True, "supervisor_alive": supervised}}}\\n            return result\\n\\n    def _checkpoint(self):\\n        with self._lock:\\n            state = self.snapshot()\\n            signature = json.dumps({k: state[k] for k in ("phase", "ready", "api_ready", "model_ready", "public_api_ready", "server_alive", "model_alive", "tunnel_alive", "cleanup_complete", "error")}, sort_keys=True)\\n            if signature != self._signature:\\n                with self._db:\\n                    self._event("manager.state", data=json.loads(signature))\\n                self._signature = signature\\n            # SQLite remains authoritative if this readable export is interrupted.\\n            rows = self._db.execute("SELECT sequence,record FROM events WHERE sequence>? ORDER BY sequence", (self._export_sequence,)).fetchall()\\n            if rows:\\n                with self.events_path.open("a", encoding="utf-8") as output:\\n                    for sequence, record in rows:\\n                        event = json.loads(record)\\n                        event["sequence"] = sequence\\n                        output.write(json.dumps(event) + "\\\\n")\\n                    if self._shutdown_complete:\\n                        output.flush()\\n                        os.fsync(output.fileno())\\n                self._export_sequence = rows[-1][0]\\n            # Publish the final receipt only after the matching audit export.\\n            temporary = self.state_path.with_suffix(".tmp")\\n            with temporary.open("w", encoding="utf-8") as output:\\n                output.write(json.dumps(state, indent=2))\\n                if self._shutdown_complete:\\n                    output.flush()\\n                    os.fsync(output.fileno())\\n            temporary.replace(self.state_path)\\n\\n    def _spawn(self, name, command, env):\\n        output = (self.run_directory / (name + ".log")).open("a", encoding="utf-8")\\n        self._files.append(output)\\n        child = self._popen(command, cwd=self.directory, env=env, stdout=output, stderr=subprocess.STDOUT)\\n        self.children[name] = child\\n        self.owners[name] = self._owner_factory(child)\\n        with self._lock, self._db:\\n            try:\\n                created = psutil.Process(child.pid).create_time()\\n            except psutil.Error:\\n                created = None\\n            launch_id = uuid.uuid4().hex\\n            child._h3_launch_id = launch_id\\n            child._h3_tree_records = {(child.pid, created): launch_id}\\n            self._db.execute("INSERT INTO owned_processes(launch_id,module,pid,created) VALUES (?,?,?,?)",\\n                             (launch_id, name, child.pid, created))\\n            self._event("module.started", data={"module": name, "pid": child.pid,\\n                        "model_instance_id": self.model_instance_id if name == "model" else None})\\n        return child\\n\\n    def _register_process_tree(self):\\n        with self._lock, self._db:\\n            for name, owner in self.owners.items():\\n                child = self.children[name]\\n                for identity in owner.observe()["identities"]:\\n                    identity_key = (identity["pid"], identity["created_at"])\\n                    if identity_key not in child._h3_tree_records:\\n                        launch_id = uuid.uuid4().hex\\n                        child._h3_tree_records[identity_key] = launch_id\\n                        self._db.execute("INSERT INTO owned_processes(launch_id,module,pid,created) VALUES (?,?,?,?)",\\n                                         (launch_id, name, *identity_key))\\n\\n    def _child_env(self):\\n        return {**self.env, "H3_PARENT_PID": str(os.getpid()),\\n                "H3_PARENT_CREATED_AT": str(psutil.Process(os.getpid()).create_time()), "H3_INSTANCE_ID": self.instance_id,\\n                "H3_MANAGER_URL": f"http://127.0.0.1:{self.port}", "H3_OUTPUT_DIR": str(self.root)}\\n\\n    def _set_api_expectation(self, expected):\\n        # Only time in which the API is requested to run consumes its first\\n        # public-delivery budget. An explicit stop pauses, rather than resets,\\n        # that budget, including when process exit remains unconfirmed.\\n        with self._lock:\\n            if expected != self._api_expected:\\n                now = self._clock()\\n                if self._api_expected_since is not None:\\n                    self._api_expected_elapsed += max(0.0, now - self._api_expected_since)\\n                self._api_expected_since = now if expected else None\\n                self._api_expected = expected\\n\\n    def _launch_api(self):\\n        alive, errors = self._liveness()\\n        if alive["api"] or "api" in errors:\\n            raise ControlError("API ownership has not been released", status=409)\\n        with self._lock:\\n            self._api_ready = self._public_api_ready = False\\n            self._set_api_expectation(True)\\n            self._api_intent_epoch += 1\\n            self._api_lifecycle = "starting"\\n        self._spawn("api", [sys.executable, "-m", "uvicorn", "server:create_app", "--factory", "--host", "127.0.0.1",\\n                            "--port", str(self.api_port), "--workers", "1", "--limit-concurrency", "8"], self._child_env())\\n\\n    def _launch_model(self):\\n        alive, errors = self._liveness()\\n        if alive["model"] or "model" in errors:\\n            raise ControlError("Old model ownership has not been released", status=409)\\n        with self._lock:\\n            old_processes = self._db.execute("SELECT launch_id,pid,created FROM owned_processes WHERE module=\\\'model\\\' AND exited=0").fetchall()\\n        for launch_id, pid, created in old_processes:\\n            try:\\n                process = psutil.Process(pid)\\n                same = created is None or abs(process.create_time() - created) < .001\\n                if same and process.is_running() and process.status() != psutil.STATUS_ZOMBIE:\\n                    raise ControlError("Previous model process exit is unconfirmed", status=409)\\n            except psutil.NoSuchProcess:\\n                pass\\n            except psutil.Error:\\n                raise ControlError("Previous model process identity is inaccessible", status=409)\\n            with self._lock, self._db:\\n                self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (launch_id,))\\n        self.model_instance_id = uuid.uuid4().hex\\n        self._model = {"phase": "loading", "instance_id": self.model_instance_id, "ready": False}\\n        self._model_failure = None\\n        self._model_started = self._clock()\\n        self._model_unreachable = 0\\n        self._model_ever_ready = False\\n        command = [sys.executable, "-u", str(self.directory / "model_container.py"), "--port", str(self.model_port),\\n                   "--instance-id", self.model_instance_id, "--root", str(self.root), "--parent-pid", str(os.getpid())]\\n        if self.staged:\\n            command.append("--staged")\\n        env = self._child_env()\\n        env.update(H3_MODEL_INSTANCE_ID=self.model_instance_id,\\n                   H3_DOWNLOAD_PROGRESS_PATH=str(self.run_directory / ("model-" + self.model_instance_id + ".downloads.json")))\\n        self._spawn("model", command, env)\\n\\n    def start(self, *, background=True):\\n        with self._control_lock:\\n            if self._phase != "new":\\n                raise ControlError("Manager already started", status=409)\\n            self._phase, self._background = "preparing", bool(background)\\n            try:\\n                env = self._child_env()\\n                self._launch_api()\\n                if self.model_autostart:\\n                    try:\\n                        self._launch_model()\\n                    except ControlError:\\n                        # Recovery may observe a previous owner child still exiting.\\n                        # Keep the API/control plane available; never overlap models.\\n                        self._model_failure = "previous_model_exit_unconfirmed"\\n                else:\\n                    self._model = {"instance_id": None, "phase": "not_started", "ready": False,\\n                                   "worker_alive": False, "accepting": False}\\n                    with self._lock, self._db:\\n                        self._event("model.autostart_skipped", data={"reason": "explicit_configuration"})\\n                if self.public:\\n                    command = [sys.executable, "-u", str(self.directory / "tunnel.py"), "--port", str(self.api_port),\\n                               "--status-file", str(self.run_directory / "tunnel-status.json"), "--instance-id", self.instance_id]\\n                    if self.install_cloudflared:\\n                        command.append("--install")\\n                    self._spawn("tunnel", command, env)\\n                if background:\\n                    self._thread = threading.Thread(target=self._monitor, name="main-event-supervisor", daemon=True)\\n                    self._thread.start()\\n                self._checkpoint()\\n            except BaseException:\\n                self._error = "module_start_failed"\\n                self.shutdown()\\n                self._phase = "failed"\\n                raise\\n        return self\\n\\n    def _transport(self):\\n        try:\\n            path = self.run_directory / "tunnel-status.json"\\n            if path.stat().st_size > 16384:\\n                return None, False\\n            body = json.loads(path.read_text(encoding="utf-8"))\\n            url = body.get("url") if isinstance(body, dict) else None\\n            if (body.get("instance_id") != self.instance_id or not isinstance(url, str)\\n                    or not re.fullmatch(r"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com", url)):\\n                return None, False\\n            return url, body.get("connected") is True and body.get("phase") not in {"failed", "stopping", "stopped"}\\n        except (OSError, ValueError, TypeError, AttributeError):\\n            return None, False\\n\\n    def _fail_jobs_after_exit(self, reason):\\n        with self._lock, self._db:\\n            for (encoded,) in self._db.execute("SELECT record FROM jobs WHERE status IN (\\\'queued\\\',\\\'running\\\',\\\'cancel_requested\\\')").fetchall():\\n                job = json.loads(encoded)\\n                target = "cancelled" if job["status"] == "cancel_requested" else "failed"\\n                cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=True)\\n                self._transition(job, target, error=reason, finished_at=time.time(), cleanup_complete=cleaned,\\n                                 cleanup_errors=cleanup_errors, model_lease_released=True)\\n\\n    def _cleanup_job_files(self, job_id, *, discard_result):\\n        errors = []\\n        job_root = self.root / job_id\\n        if not job_root.resolve().is_relative_to(self.root) or job_root.is_symlink():\\n            return False, ["job_directory_identity_invalid"]\\n        try:\\n            shutil.rmtree(job_root / "inputs")\\n        except FileNotFoundError:\\n            pass\\n        except OSError as error:\\n            errors.append("inputs: " + type(error).__name__)\\n        if discard_result:\\n            for name in ("result.mp4", "result.metrics.json"):\\n                try:\\n                    (job_root / name).unlink(missing_ok=True)\\n                except OSError as error:\\n                    errors.append(name + ": " + type(error).__name__)\\n        return not errors, errors\\n\\n    def _model_probe(self, expected_instance_id):\\n        try:\\n            candidate = self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "status", {}, timeout=self.probe_timeout)\\n            if isinstance(candidate, dict) and candidate.get("instance_id") == expected_instance_id:\\n                return candidate\\n        except Exception:\\n            pass\\n        return None\\n\\n    def refresh(self):\\n        with self._refresh_lock:\\n            if self._shutdown_started:\\n                return self.snapshot()\\n            with self._control_lock:\\n                self._register_process_tree()\\n                alive, errors = self._liveness()\\n                observed_model_id, observed_child = self.model_instance_id, self.children["model"]\\n                observed_api_child = self.children["api"]\\n                observed_api_epoch = self._api_intent_epoch\\n            api_future = (self._probe_pool.submit(self._probe, f"http://127.0.0.1:{self.api_port}", self.instance_id, self.probe_timeout)\\n                          if alive["api"] else None)\\n            model_future = (self._probe_pool.submit(self._model_probe, observed_model_id)\\n                            if alive["model"] and "model" not in errors else None)\\n            api_ready = bool(api_future.result()) if api_future else False\\n            model = model_future.result() if model_future else None\\n            url, connected = self._transport() if self.public and alive["tunnel"] else (None, False)\\n            public_api = bool(api_ready and connected and self._probe(url, self.instance_id, self.probe_timeout))\\n            with self._control_lock:\\n                if (self._shutdown_started or observed_model_id != self.model_instance_id\\n                        or observed_child is not self.children["model"] or observed_api_child is not self.children["api"]\\n                        or observed_api_epoch != self._api_intent_epoch):\\n                    return self.snapshot()  # Discard a previous model generation\\\'s observations.\\n                if alive["model"] and "model" not in errors:\\n                    self._model_unreachable = 0 if model else self._model_unreachable + 1\\n                with self._lock:\\n                    self._api_ready = bool(api_ready and self._api_expected)\\n                    self._public_api_ready = bool(public_api and self._api_expected)\\n                    self._connected = connected\\n                    if url:\\n                        self._public_url = url\\n                    previous_downloads = self._model.get("downloads")\\n                    self._model = model or {"instance_id": self.model_instance_id, "ready": False,\\n                                            "phase": "unreachable" if alive["model"] else\\n                                                     "not_started" if self.model_instance_id is None else "stopped"}\\n                    if not model and isinstance(previous_downloads, dict):\\n                        self._model["downloads"] = {**deepcopy(previous_downloads), "observation_stale": True}\\n                    self._last_observation = self._clock()\\n                    if model and model.get("ready") is True:\\n                        self._model_ever_ready = True\\n                    if self._public_api_ready:\\n                        self._public_ever_ready = True\\n                        if self._error and self._error.startswith("public_"):\\n                            self._error = None\\n                    if not alive["api"] and self._api_expected:\\n                        self._error = "api_process_exited; model and durable jobs retained"\\n                        self._api_lifecycle = "failed"\\n                    elif api_ready and self._error and self._error.startswith("api_"):\\n                        self._error = None\\n                    if self._api_expected and api_ready:\\n                        self._api_lifecycle = "ready"\\n                    if not alive["model"] and self.model_instance_id is not None:\\n                        self._model_failure = self._model_failure or "model_process_exited"\\n                    elif model and model.get("phase") == "failed":\\n                        self._model_failure = "model_load_or_worker_failed"\\n                if not alive["model"] and "model" not in errors and self.model_instance_id is not None:\\n                    self._fail_jobs_after_exit("model_process_exited")\\n                elif model:\\n                    self._consume_result(model)\\n                    with self._lock:\\n                        expired = any(time.time() - job["started_at"] > self.generation_timeout\\n                                      for job in self._active_jobs())\\n                    # A dead worker can no longer own a useful execution. Confirm\\n                    # process exit before releasing the main execution lease.\\n                    if expired:\\n                        self._stop_model(force=True, reason="generation_deadline_exceeded")\\n                    elif self._model_failure or model.get("phase") == "failed" or (model.get("worker_alive") is False and model.get("phase") not in {"starting", "loading"}):\\n                        with self._control_lock:\\n                            self._stop_model(force=True, reason="model_worker_failed")\\n                    else:\\n                        self._dispatch_next(model)\\n                if alive["model"] and self._model_unreachable >= 3:\\n                    with self._control_lock:\\n                        self._stop_model(force=True, reason="model_control_unreachable_for_three_cycles")\\n                with self._lock:\\n                    active = self._active_jobs()\\n                if any(time.time() - job["started_at"] > self.generation_timeout for job in active):\\n                    with self._control_lock:\\n                        self._stop_model(force=True, reason="generation_deadline_exceeded")\\n                if self.snapshot()["ready"]:\\n                    self._ever_ready = True\\n                if (self._model_started is not None and not self._model_ever_ready and self._clock() - self._model_started > self.startup_timeout\\n                        and not self._model.get("ready") and not active and alive["model"]):\\n                    with self._control_lock:\\n                        self._stop_model(force=True, reason="model_startup_deadline_exceeded")\\n                public_elapsed = self._api_expected_elapsed\\n                if self._api_expected_since is not None:\\n                    public_elapsed += max(0.0, self._clock() - self._api_expected_since)\\n                if (self.public and self._api_expected and not self._public_ever_ready\\n                        and public_elapsed > self.tunnel_timeout):\\n                    self._error = "public_preparation_deadline_exceeded; local API and model retained"\\n                self._checkpoint()\\n                return self.snapshot()\\n\\n    def _consume_result(self, model):\\n        results = model.get("results")\\n        if not isinstance(results, list):\\n            results = [model.get("last_result")]  # Read old checkpoints without losing their result.\\n        acknowledgements = []\\n        for result in results:\\n            if not isinstance(result, dict) or result.get("status") not in TERMINAL:\\n                continue\\n            with self._lock, self._db:\\n                try:\\n                    job = self._get_job(result.get("job_id"))\\n                except ControlError:\\n                    continue\\n                if (result.get("attempt_id") != job["attempt_id"]\\n                        or model.get("instance_id") != job["model_instance_id"]\\n                        or result.get("instance_id", model["instance_id"]) != job["model_instance_id"]):\\n                    continue\\n                if job["status"] in TERMINAL:\\n                    acknowledgements.append(job["attempt_id"])\\n                    continue\\n                if job["status"] not in {"running", "cancel_requested"}:\\n                    continue\\n                if result.get("cleanup_complete") is not True:\\n                    self._model_failure = "model_attempt_cleanup_unconfirmed"\\n                    job.update(error=self._model_failure, cleanup_complete=False)\\n                    self._store_job(job)\\n                    self._event("job.cleanup_pending", job=job)\\n                    continue  # Retain ownership until process exit is confirmed.\\n                target, artifact = result["status"], result.get("artifact")\\n                cancellation_requested = job["status"] == "cancel_requested"\\n                if cancellation_requested and target != "failed":\\n                    target, artifact = "cancelled", None\\n                elif target == "succeeded":\\n                    artifact = self._verified_artifact(job["job_id"], artifact)\\n                    if artifact is None:\\n                        target = "failed"\\n                cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=target != "succeeded")\\n                self._transition(job, target, artifact=artifact if target == "succeeded" else None,\\n                                 error=None if target == "succeeded" else result.get("error") or "result_not_committed",\\n                                 metrics=result.get("metrics"), finished_at=time.time(), cleanup_complete=cleaned,\\n                                 cancellation_requested=cancellation_requested,\\n                                 execution_condition_gates=deepcopy(result.get("condition_gates")),\\n                                 stage_completions=deepcopy(result.get("stage_completions")),\\n                                 generation_options=result.get("generation_options") or job.get("generation_options"),\\n                                 cleanup_errors=cleanup_errors, model_lease_released=True, dispatch_state="completed")\\n                acknowledgements.append(job["attempt_id"])\\n        # The durable transaction must commit before the model may discard a result.\\n        if acknowledgements and isinstance(model.get("results"), list):\\n            try:\\n                self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "ack_results",\\n                          {"instance_id": model["instance_id"], "attempt_ids": acknowledgements}, timeout=self.probe_timeout)\\n            except Exception:\\n                pass  # Next status repeats the durable results; acknowledge idempotently.\\n\\n    def _verified_artifact(self, job_id, artifact):\\n        if not isinstance(artifact, dict):\\n            return None\\n        try:\\n            path = Path(artifact["path"]).resolve()\\n            lexical = self.root / job_id / "result.mp4"\\n            expected = lexical.resolve()\\n            if (path != expected or not path.is_relative_to(self.root) or not path.is_file()\\n                    or lexical.is_symlink() or not isinstance(artifact.get("media"), dict)\\n                    or artifact["media"].get("full_decode_passed") is not True):\\n                return None\\n            if path.stat().st_size != artifact["bytes"] or artifact["bytes"] <= 0:\\n                return None\\n            if not re.fullmatch(r"[a-f0-9]{64}", artifact.get("sha256", "")):\\n                return None\\n            with path.open("rb") as source:\\n                actual = hashlib.file_digest(source, "sha256").hexdigest()\\n            if actual != artifact["sha256"]:\\n                return None\\n            return artifact\\n        except (KeyError, OSError, TypeError, ValueError):\\n            return None\\n\\n    def _dispatch_gate(self, model, job, attempt_id):\\n        """Called only in the dispatch/cancel arbitration and state transaction.\\n\\n        The remote observation permits an offer, not remote execution. Container\\n        admission and resource ownership are revalidated by their own owners.\\n        """\\n        model = model if isinstance(model, dict) else {}\\n        alive, errors = self._liveness()\\n        age = self._clock() - self._last_observation if self._last_observation is not None else None\\n        bound = 3 * self.probe_timeout + self.poll_interval + 2\\n        identity = self.model_instance_id\\n        identities = (identity, model.get("instance_id"), self._model.get("instance_id"))\\n        current_identity = (all(value == identity for value in identities)\\n                            if all(isinstance(value, str) and value for value in identities) else None)\\n        def capability(field):\\n            witnesses = [model.get(field), self._model.get(field)]\\n            # Missing or malformed fresh evidence cannot borrow a previous True.\\n            return witnesses if all(type(value) is bool for value in witnesses) else None\\n        observations = {\\n            # Draining closes new admission; the already-owned queue still drains.\\n            "manager_open": self._shutdown_started is False and self._model_failure is None,\\n            "job_queued": job.get("status") == "queued", "attempt_unassigned": job.get("attempt_id") is None,\\n            "model_identity_current": current_identity,\\n            "model_observation_fresh": None if age is None else 0 <= age <= bound,\\n            "model_process_alive": None if "model" in errors else alive.get("model"),\\n            "model_ready": capability("ready"),\\n            "worker_alive": capability("worker_alive"),\\n            "worker_accepting": capability("accepting"),\\n            "supervisor_alive": not self._background or (self._thread is not None and self._thread.is_alive()),\\n        }\\n        receipt = evaluate_gate("manager.dispatch", DISPATCH_CONDITIONS, observations,\\n            {"manager_instance_id": self.instance_id, "model_instance_id": identity, "job_id": job["job_id"],\\n             "attempt_id": attempt_id, "source_file": "global_manager.py", "source_sha256": GATE_SOURCE_SHA256})\\n        # Commit every positive authorization; coalesce unchanged blocked polls.\\n        signature = json.dumps({"job_id": job["job_id"], "model_instance_id": identity,\\n                                "conditions": receipt["conditions"]}, sort_keys=True)\\n        if receipt["allowed"] or signature != self._dispatch_gate_signature:\\n            self._dispatch_gate_signature = signature\\n            self._condition_gates["manager.dispatch"] = receipt\\n            job["condition_gate"] = receipt\\n            self._store_job(job)\\n            self._event("condition.checked", job=job, data={"receipt": receipt})\\n        return receipt\\n\\n    def _dispatch_next(self, model):\\n        with self._control_lock:\\n            deadline = time.monotonic() + max(.01, min(.25, self.poll_interval))\\n            while not self._shutdown_started and not self._model_failure:\\n                with self._lock, self._db:\\n                    row = self._db.execute("SELECT record FROM jobs WHERE status=\\\'queued\\\' ORDER BY rowid LIMIT 1").fetchone()\\n                    if not row:\\n                        return\\n                    job = json.loads(row[0])\\n                    prospective_attempt = uuid.uuid4().hex\\n                    receipt = self._dispatch_gate(model, job, prospective_attempt)\\n                    if receipt["allowed"] is not True:\\n                        return\\n                    self._transition(job, "running", attempt_id=prospective_attempt,\\n                                     model_instance_id=self.model_instance_id, started_at=time.time(), cleanup_complete=False,\\n                                     model_lease_released=False, dispatch_state="offering", dispatch_reason=None)\\n                    self._event("job.dispatch_intent", job=job, data={"condition_decision_id": receipt["decision_id"]})\\n                # running means this attempt\\\'s ownership, not ownership of the GPU.\\n                # The model stage scheduler alone grants the one GPU lease.\\n                payload = {"job_id": job["job_id"], "attempt_id": job["attempt_id"],\\n                           "instance_id": self.model_instance_id, "parameters": job["parameters"]}\\n                try:\\n                    response = self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "execute", payload, timeout=self.probe_timeout)\\n                    if not isinstance(response, dict) or response.get("accepted") is not True:\\n                        raise ValueError("Invalid execution acknowledgement")\\n                    for field in ("job_id", "attempt_id", "instance_id"):\\n                        if response.get(field) != payload[field]:\\n                            raise ValueError("Execution acknowledgement identity mismatch")\\n                except Exception as error:\\n                    rejected = None\\n                    if isinstance(error, ControlError):\\n                        try:\\n                            rejected = json.loads(str(error))\\n                        except (ValueError, TypeError):\\n                            pass\\n                    if isinstance(rejected, dict) and rejected.get("guarantees_not_accepted") is True:\\n                        with self._lock, self._db:\\n                            if error.status in {409, 429}:\\n                                self._transition(job, "queued", attempt_id=None, model_instance_id=None,\\n                                                 started_at=None, model_lease_released=True, dispatch_state="deferred",\\n                                                 dispatch_reason=rejected.get("reason"))\\n                                return  # FIFO: do not bypass the head to fill a smaller task.\\n                            cleaned, cleanup_errors = self._cleanup_job_files(job["job_id"], discard_result=True)\\n                            self._transition(job, "failed", error=str(error), finished_at=time.time(),\\n                                             model_lease_released=True, cleanup_complete=cleaned,\\n                                             cleanup_errors=cleanup_errors, dispatch_state="rejected")\\n                    else:\\n                        # A missing acknowledgement is not evidence of rejection.\\n                        # Reconcile this exact attempt, never send execute twice.\\n                        observed = self._model_probe(self.model_instance_id)\\n                        records = ((observed or {}).get("attempts", []) + (observed or {}).get("results", []))\\n                        matched = any(isinstance(item, dict) and all(item.get(k) == payload[k]\\n                                      for k in ("job_id", "attempt_id", "instance_id")) for item in records)\\n                        if not matched:\\n                            self._stop_model(force=True, reason="execution_acknowledgement_uncertain")\\n                            return\\n                        with self._lock, self._db:\\n                            job.update(dispatch_state="reconciled")\\n                            self._store_job(job)\\n                            self._event("job.dispatch_reconciled", job=job)\\n                        self._model = observed\\n                        self._consume_result(observed)\\n                        return\\n                else:\\n                    with self._lock, self._db:\\n                        job.update(dispatch_state="accepted")\\n                        self._store_job(job)\\n                        self._event("job.dispatch_accepted", job=job)\\n                if time.monotonic() >= deadline:\\n                    return\\n\\n    def _monitor(self):\\n        try:\\n            while not self._cancel.is_set():\\n                self.refresh()\\n                self._cancel.wait(self.poll_interval)\\n        except BaseException as error:\\n            LOG.exception("Main event supervision failed")\\n            with self._lock:\\n                self._error = "supervisor_failed: " + type(error).__name__\\n                self._phase, self._draining = "failed", True\\n            # Retain the control plane, but end uncertain model ownership.\\n            with self._control_lock:\\n                self._stop_model(force=True, reason="supervisor_failed")\\n            try:\\n                self._checkpoint()\\n            except Exception:\\n                LOG.exception("Could not export final supervisor failure")\\n\\n    def submit(self, payload):\\n        job_id, parameters = payload.get("job_id"), payload.get("parameters")\\n        if not isinstance(job_id, str) or not IDENTIFIER.fullmatch(job_id) or not isinstance(parameters, dict):\\n            raise ControlError("Invalid job request", status=422)\\n        try:\\n            options = requested_options(parameters)\\n            validate_generation_options(options, load_adapter_settings())\\n        except (KeyError, ValueError, TypeError) as error:\\n            raise ControlError(f"Invalid generation options: {error}", status=422) from None\\n        refs = parameters.get("references")\\n        if not isinstance(refs, list) or not 1 <= len(refs) <= 12:\\n            raise ControlError("Invalid reference list", status=422)\\n        input_root = (self.root / job_id / "inputs").resolve()\\n        if not input_root.is_relative_to(self.root):\\n            raise ControlError("Input directory escapes output root", status=422)\\n        for ref in refs:\\n            try:\\n                path = Path(ref["path"]).resolve()\\n                if not path.is_relative_to(input_root) or not path.is_file() or Path(ref["path"]).is_symlink():\\n                    raise ValueError()\\n            except (KeyError, TypeError, OSError, ValueError):\\n                raise ControlError("References must be owned input files", status=422)\\n        idem, fingerprint = payload.get("idempotency_key"), payload.get("fingerprint")\\n        if idem is not None and (not isinstance(idem, str) or not 1 <= len(idem) <= 128 or not isinstance(fingerprint, str)):\\n            raise ControlError("Invalid idempotency key or fingerprint", status=422)\\n        with self._lock, self._db:\\n            if idem is not None:\\n                row = self._db.execute("SELECT fingerprint,record FROM jobs WHERE idempotency_key=?", (idem,)).fetchone()\\n                if row:\\n                    if row[0] != fingerprint:\\n                        raise ControlError("Idempotency key conflicts with request", status=409)\\n                    return self._public_job(json.loads(row[1]))\\n            if self._draining or self._shutdown_started or self._phase in {"new", "failed", "stopped"} or self._model_failure:\\n                raise ControlError("Generation admission is closed", status=409)\\n            if shutil.disk_usage(self.root).free < self.disk_reserve_bytes + 65536:\\n                raise ControlError("Insufficient disk space for durable queue metadata", status=507)\\n            if self._db.execute("SELECT 1 FROM jobs WHERE id=?", (job_id,)).fetchone():\\n                raise ControlError("Job ID already exists", status=409)\\n            now = time.time()\\n            job = {"job_id": job_id, "status": "queued", "created_at": now, "started_at": None,\\n                   "finished_at": None, "attempt_id": None, "model_instance_id": None,\\n                   "parameters": parameters, "error": None, "artifact": None, "metrics": None,\\n                   "generation_options": {"requested": options, "effective": None},\\n                    "cleanup_complete": False, "model_lease_released": True, "dispatch_state": "pending",\\n                   "lifecycle": [{"state": "queued", "observed_at": now}]}\\n            self._db.execute("INSERT INTO jobs VALUES (?,?,?,?,?)", (job_id, "queued", idem, fingerprint, json.dumps(job)))\\n            self._event("job.queued", job=job)\\n            return self._public_job(job)\\n\\n    def _public_job(self, job):\\n        result = {k: v for k, v in job.items() if k != "parameters"}\\n        # Public task identity belongs to this durable manager/API lifetime;\\n        # model_instance_id separately identifies a replaceable GPU process.\\n        result["instance_id"] = self.instance_id\\n        result["queue_position"] = (self._db.execute("SELECT COUNT(*) FROM jobs WHERE status=\\\'queued\\\' AND rowid <= (SELECT rowid FROM jobs WHERE id=?)",\\n                                                    (job["job_id"],)).fetchone()[0] if job["status"] == "queued" else None)\\n        result["waiting_for"] = None\\n        if job["status"] == "queued":\\n            # This is a current observation, separate from the last durable\\n            # dispatch rejection. Do not make clients infer it from a log line.\\n            if self._shutdown_started:\\n                result["waiting_for"] = "shutdown"\\n            elif self._model_failure:\\n                result["waiting_for"] = "model_recovery"\\n            elif self._model.get("ready") is not True or self._model.get("worker_alive") is not True:\\n                result["waiting_for"] = "model_ready"\\n            elif self._model.get("accepting") is not True:\\n                result["waiting_for"] = "model_capacity"\\n            elif result["queue_position"] > 1:\\n                result["waiting_for"] = "earlier_jobs"\\n            else:\\n                result["waiting_for"] = "scheduler"\\n        result["active_phase"] = None\\n        if job["status"] in {"running", "cancel_requested"}:\\n            for attempt in self._model.get("attempts", []):\\n                if attempt.get("attempt_id") == job["attempt_id"] and attempt.get("job_id") == job["job_id"]:\\n                    result["active_phase"] = attempt.get("active_phase") or {"phase": attempt.get("phase", attempt.get("status"))}\\n                    if isinstance(attempt.get("generation_options"), dict):\\n                        result["generation_options"] = deepcopy(attempt["generation_options"])\\n                    break\\n            if result["active_phase"] is None and job["attempt_id"] == self._model.get("active_attempt_id"):\\n                result["active_phase"] = self._model.get("active_phase")\\n        return result\\n\\n    def archive(self, payload):\\n        """Record a client-confirmed Drive upload for the committed artifact."""\\n        job_id, file_id = payload.get("job_id"), payload.get("file_id")\\n        sha256, size = payload.get("sha256"), payload.get("bytes")\\n        if (not isinstance(job_id, str) or not IDENTIFIER.fullmatch(job_id)\\n                or not isinstance(file_id, str) or not DRIVE_FILE_ID.fullmatch(file_id)\\n                or not isinstance(sha256, str) or not SHA256.fullmatch(sha256)\\n                or type(size) is not int or size <= 0):\\n            raise ControlError("Invalid Google Drive archive receipt", status=422)\\n        receipt = {"provider": "google_drive", "file_id": file_id, "sha256": sha256, "bytes": size}\\n        with self._lock, self._db:\\n            job = self._get_job(job_id)\\n            if job["status"] != "succeeded":\\n                raise ControlError("Only a succeeded job can be archived", status=409)\\n            artifact = job.get("artifact")\\n            media = artifact.get("media") if isinstance(artifact, dict) else None\\n            if (not isinstance(media, dict) or media.get("full_decode_passed") is not True\\n                    or artifact.get("sha256") != sha256 or artifact.get("bytes") != size):\\n                raise ControlError("Archive receipt does not match the committed artifact", status=409)\\n            previous = job.get("archive")\\n            if previous is None:\\n                job["archive"] = receipt\\n                self._store_job(job)\\n                self._event("job.archived", job=job, data={"provider": "google_drive"})\\n            elif previous != receipt:\\n                raise ControlError("Job already has a different archive receipt", status=409)\\n            return self._public_job(job)\\n\\n    def cancel(self, job_id):\\n        with self._control_lock:\\n            with self._lock, self._db:\\n                job = self._get_job(job_id)\\n                if job["status"] in TERMINAL:\\n                    return self._public_job(job)\\n                if job["status"] == "queued":\\n                    cleaned, cleanup_errors = self._cleanup_job_files(job_id, discard_result=True)\\n                    self._transition(job, "cancelled", finished_at=time.time(), cleanup_complete=cleaned,\\n                                     cleanup_errors=cleanup_errors, model_lease_released=True)\\n                    return self._public_job(job)\\n                if job["status"] == "running":\\n                    self._transition(job, "cancel_requested")\\n            try:\\n                self._rpc(f"http://127.0.0.1:{self.model_port}", self.key, "cancel",\\n                          {"job_id": job_id, "attempt_id": job["attempt_id"], "instance_id": job["model_instance_id"]},\\n                          timeout=self.probe_timeout)\\n            except Exception:\\n                pass  # Request remains pending; execution lease is not released.\\n            return self._public_job(job)\\n\\n    def _terminate(self, name, timeout=15):\\n        child = self.children[name]\\n        if child is None:\\n            return True\\n        owner = self.owners.get(name)\\n        if owner:\\n            self._register_process_tree()\\n            return self._record_tree_exit(name, owner.terminate_all(timeout))\\n        try:\\n            if child.poll() is None:\\n                child.terminate()\\n                try:\\n                    child.wait(timeout=timeout)\\n                except subprocess.TimeoutExpired:\\n                    child.kill()\\n                    child.wait(timeout=3)\\n            exited = child.poll() is not None\\n            if exited and hasattr(child, "_h3_launch_id"):\\n                with self._lock, self._db:\\n                    self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (child._h3_launch_id,))\\n            return exited\\n        except (OSError, subprocess.SubprocessError) as error:\\n            self._cleanup_errors.append(name + ": " + type(error).__name__)\\n            return False\\n\\n    def _record_tree_exit(self, name, observed):\\n        if observed["cleanup_complete"]:\\n            with self._lock, self._db:\\n                for launch_id in self.children[name]._h3_tree_records.values():\\n                    self._db.execute("UPDATE owned_processes SET exited=1 WHERE launch_id=?", (launch_id,))\\n            return True\\n        self._cleanup_errors.append(name + ".tree: " + "; ".join(observed["errors"] or ["exit unconfirmed"]))\\n        return False\\n\\n    def _stop_model(self, *, force=False, reason="model_stopped"):\\n        with self._lock:\\n            active = self._active_jobs()\\n            queued = self._db.execute("SELECT COUNT(*) FROM jobs WHERE status=\\\'queued\\\'").fetchone()[0]\\n            if (active or queued) and not force:\\n                raise ControlError("Model has unfinished jobs; drain or explicitly force stop", status=409)\\n            self._model_failure = reason\\n            self._model["ready"] = False\\n        if self._terminate("model"):\\n            self._fail_jobs_after_exit(reason)\\n            downloads = self._model.get("downloads")\\n            self._model = {"instance_id": self.model_instance_id, "phase": "stopped", "ready": False,\\n                           "worker_alive": False, "cleanup_complete": True}\\n            if isinstance(downloads, dict):\\n                self._model["downloads"] = {**deepcopy(downloads), "observation_stale": True}\\n            with self._lock, self._db:\\n                self._event("model.stopped", data={"reason": reason, "cleanup_complete": True})\\n        else:\\n            raise ControlError("Model exit is unconfirmed; lease retained", status=503)\\n\\n    def _control_action(self, action, payload):\\n        with self._control_lock:\\n            if action == "shutdown":\\n                state = self.shutdown()\\n                if (state.get("cleanup_complete") is not True or state.get("process_observation_errors")\\n                        or any(state.get(name) is not False for name in ("server_alive", "model_alive", "tunnel_alive"))):\\n                    raise ControlError("Shutdown cleanup is unconfirmed; manager remains available for retry", status=503)\\n            elif action == "stop_model":\\n                self._stop_model(force=payload.get("force") is True)\\n            elif action == "start_model":\\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\\n                    raise ControlError("Manager is stopping or failed", status=409)\\n                self._launch_model()\\n            elif action == "stop_api":\\n                with self._lock, self._db:\\n                    self._set_api_expectation(False)\\n                    self._api_intent_epoch += 1\\n                    self._api_lifecycle = "stopping"\\n                    self._api_ready = self._public_api_ready = False\\n                    if self._error and self._error.startswith("api_"):\\n                        self._error = None\\n                    self._event("api.stop_requested", data={"intent_epoch": self._api_intent_epoch})\\n                if not self._terminate("api"):\\n                    self._api_lifecycle = "stop_unconfirmed"\\n                    raise ControlError("API process exit is unconfirmed", status=503)\\n                self._api_lifecycle = "stopped"\\n                with self._lock, self._db:\\n                    self._event("api.stopped", data={"intent_epoch": self._api_intent_epoch, "cleanup_complete": True})\\n            elif action == "start_api":\\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\\n                    raise ControlError("Manager cannot start API", status=409)\\n                self._launch_api()\\n            elif action == "drain":\\n                self._draining = True  # Existing queue continues to drain.\\n            elif action == "resume":\\n                if self._shutdown_started or self._phase in {"failed", "stopped"}:\\n                    raise ControlError("Manager cannot resume", status=409)\\n                self._draining = False\\n            else:\\n                raise ControlError("Unknown control action", status=422)\\n            self._checkpoint()\\n\\n    def control(self, payload):\\n        action = payload.get("action")\\n        if action not in {"drain", "resume", "start_model", "stop_model", "start_api", "stop_api", "shutdown"}:\\n            raise ControlError("Unknown control action", status=422)\\n        command_id = payload.get("command_id") or uuid.uuid4().hex\\n        if not isinstance(command_id, str) or not 1 <= len(command_id) <= 128:\\n            raise ControlError("Invalid command ID", status=422)\\n        fingerprint = json.dumps({"action": action, "force": payload.get("force") is True}, sort_keys=True)\\n        record = {"command_id": command_id, "action": action, "status": "accepted", "created_at": time.time(), "error": None}\\n\\n        def run():\\n            try:\\n                self._control_action(action, payload)\\n                result = {**record, "status": "succeeded", "finished_at": time.time()}\\n            except BaseException as error:\\n                LOG.exception("Control operation failed")\\n                result = {**record, "status": "failed", "error": str(error) if isinstance(error, ControlError) else type(error).__name__,\\n                          "finished_at": time.time()}\\n            try:\\n                with self._lock, self._db:\\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(result), command_id))\\n                    self._event("command." + result["status"], data={"command_id": command_id, "action": action, "error": result.get("error")})\\n                self._checkpoint()\\n                if action == "shutdown" and result["status"] == "succeeded":\\n                    self._exit_requested = True\\n            finally:\\n                self._control_slots.release()\\n\\n        thread = threading.Thread(target=run, name="main-control-" + action, daemon=True)\\n        with self._lock:\\n            existing = self._db.execute("SELECT fingerprint,record FROM commands WHERE id=?", (command_id,)).fetchone()\\n            if existing:\\n                if existing[0] != fingerprint:\\n                    raise ControlError("Command ID conflicts", status=409)\\n                return json.loads(existing[1])\\n            if self._control_admission_closed or (self._shutdown_started and action != "shutdown"):\\n                raise ControlError("Manager is closing; new control operations are unavailable", status=409)\\n            if not self._control_slots.acquire(blocking=False):\\n                raise ControlError("Control operation queue is full", status=429)\\n            try:\\n                with self._db:\\n                    self._db.execute("INSERT INTO commands VALUES (?,?,?)", (command_id, fingerprint, json.dumps(record)))\\n                    self._event("command.accepted", data={"command_id": command_id, "action": action})\\n            except BaseException:\\n                self._control_slots.release()\\n                raise\\n            # Admission, registration and start share close()\\\'s barrier. An\\n            # accepted command can never appear after its worker join snapshot.\\n            self._control_threads = [t for t in self._control_threads if t.ident is None or t.is_alive()]\\n            self._control_threads.append(thread)\\n            try:\\n                thread.start()\\n            except BaseException:\\n                self._control_slots.release()\\n                with self._db:\\n                    record.update(status="failed", error="command_worker_start_failed", finished_at=time.time())\\n                    self._db.execute("UPDATE commands SET record=? WHERE id=?", (json.dumps(record), command_id))\\n                    self._event("command.failed", data={"command_id": command_id, "error": record["error"]})\\n                raise\\n        return record\\n\\n    def dispatch(self, operation, payload=None):\\n        payload = payload or {}\\n        if operation in {"submit", "cancel", "control", "archive"} and payload.get("expected_instance_id", self.instance_id) != self.instance_id:\\n            raise ControlError("Manager instance changed; command rejected", status=409)\\n        if operation == "status":\\n            return self.snapshot()\\n        if operation == "submit":\\n            return self.submit(payload)\\n        if operation == "job":\\n            with self._lock:\\n                return self._public_job(self._get_job(payload.get("job_id")))\\n        if operation == "archive":\\n            return self.archive(payload)\\n        if operation == "cancel":\\n            return self.cancel(payload.get("job_id"))\\n        if operation == "control":\\n            return self.control(payload)\\n        if operation == "command":\\n            with self._lock:\\n                row = self._db.execute("SELECT record FROM commands WHERE id=?", (payload.get("command_id"),)).fetchone()\\n                if not row:\\n                    raise ControlError("Command not found", status=404)\\n                return json.loads(row[0])\\n        if operation == "events":\\n            try:\\n                after, limit = max(0, int(payload.get("after", 0))), min(500, max(1, int(payload.get("limit", 100))))\\n            except (TypeError, ValueError):\\n                raise ControlError("Invalid event cursor", status=422)\\n            with self._lock:\\n                rows = self._db.execute("SELECT sequence,record FROM events WHERE sequence>? ORDER BY sequence LIMIT ?", (after, limit)).fetchall()\\n            events = [{**json.loads(encoded), "sequence": seq} for seq, encoded in rows]\\n            return {"events": events, "next_cursor": events[-1]["sequence"] if events else after}\\n        raise ControlError("Unknown manager operation", status=404)\\n\\n    def shutdown(self):\\n        with self._control_lock:\\n            self._shutdown_started, self._draining, self._phase = True, True, "stopping"\\n            with self._lock:\\n                self._set_api_expectation(False)\\n                self._api_intent_epoch += 1\\n                self._api_lifecycle = "stopping"\\n                self._api_ready = self._public_api_ready = False\\n            self._cancel.set()\\n            self._register_process_tree()\\n            names = [name for name in ("tunnel", "api", "model") if name in self.owners]\\n            results = stop_many([self.owners[name] for name in names], timeout=15)\\n            for name, observed in zip(names, results):\\n                self._record_tree_exit(name, observed)\\n            alive, errors = self._liveness()\\n            self._api_lifecycle = "stopped" if not alive["api"] and "api" not in errors else "stop_unconfirmed"\\n            if not alive["model"] and "model" not in errors:\\n                self._fail_jobs_after_exit("manager_stopped")\\n            with self._lock:\\n                historical = self._historical_ownership()\\n            self._phase = "failed" if any(alive.values()) or errors or historical else "stopped"\\n            if any(alive.values()) or errors or historical:\\n                self._error = "cleanup_incomplete; owned processes remain"\\n            elif self._error in {"cleanup_incomplete; owned processes remain", "audit_export_failed_during_shutdown"}:\\n                self._error = None\\n            try:\\n                self._checkpoint()\\n            except (OSError, sqlite3.Error):\\n                self._error = "audit_export_failed_during_shutdown"\\n                LOG.exception("Could not export shutdown audit")\\n            return self.snapshot()\\n\\n    def close(self):\\n        with self._lock:\\n            if self._closed:\\n                return\\n            self._control_admission_closed = True\\n        self.shutdown()\\n        if self._thread and self._thread is not threading.current_thread() and self._thread.ident is not None:\\n            self._thread.join(timeout=3 * self.probe_timeout + self.poll_interval + 2)\\n            if self._thread.is_alive():\\n                raise RuntimeError("Cannot finalize shutdown with a live supervisor")\\n        for thread in self._control_threads:\\n            if thread is not threading.current_thread() and thread.ident is not None:\\n                thread.join(timeout=20)\\n                if thread.is_alive():\\n                    raise RuntimeError("Cannot finalize shutdown with a live control operation")\\n        self._probe_pool.shutdown(wait=True, cancel_futures=True)\\n        alive, errors = self._liveness()\\n        if any(value is not False for value in alive.values()) or errors or self._historical_ownership():\\n            raise RuntimeError("Cannot release root ownership with live children")\\n        with self._lock:\\n            self._shutdown_complete = True\\n            try:\\n                with self._db:\\n                    self._event("manager.shutdown_complete")\\n                self._checkpoint()\\n            except BaseException:\\n                self._shutdown_complete = False\\n                raise\\n            self._db.close()\\n            self._owner.close()\\n            self._closed = True\\n        for output in self._files:\\n            output.close()\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    for argument in ("port", "api-port", "model-port"):\\n        parser.add_argument("--" + argument, required=True, type=int)\\n    parser.add_argument("--directory", required=True, type=Path)\\n    parser.add_argument("--root", required=True, type=Path)\\n    parser.add_argument("--instance-id", required=True)\\n    parser.add_argument("--staged", action="store_true")\\n    parser.add_argument("--public", action="store_true")\\n    parser.add_argument("--install-cloudflared", action="store_true")\\n    parser.add_argument("--no-model-autostart", action="store_true",\\n                        help="Start API/public transport only; start_model control can launch the model later")\\n    parser.add_argument("--parent-pid", type=int)\\n    parser.add_argument("--startup-timeout", type=float, default=3600)\\n    parser.add_argument("--generation-timeout", type=float, default=5400)\\n    parser.add_argument("--poll-interval", type=float, default=1)\\n    parser.add_argument("--probe-timeout", type=float, default=2)\\n    parser.add_argument("--tunnel-timeout", type=float, default=180)\\n    args = parser.parse_args()\\n    manager = GlobalManager(args.directory, root=args.root, instance_id=args.instance_id,\\n                            port=args.port, api_port=args.api_port, model_port=args.model_port,\\n                            staged=args.staged, public=args.public, install_cloudflared=args.install_cloudflared,\\n                            start_model=not args.no_model_autostart,\\n                            startup_timeout=args.startup_timeout, generation_timeout=args.generation_timeout,\\n                            poll_interval=args.poll_interval, probe_timeout=args.probe_timeout, tunnel_timeout=args.tunnel_timeout)\\n    server = make_server("127.0.0.1", args.port, manager.key, manager.dispatch)\\n    serve_thread = threading.Thread(target=server.serve_forever, name="manager-control-http", daemon=True)\\n    stop_requested = False\\n\\n    def request_stop(*_):\\n        # Python signal handlers can interrupt/re-enter Event.set while its\\n        # non-reentrant Condition lock is held. Only assign a flag here.\\n        nonlocal stop_requested\\n        stop_requested = True\\n    parent_identity = None\\n    if args.parent_pid:\\n        try:\\n            parent_identity = float(os.environ.get("H3_PARENT_CREATED_AT") or psutil.Process(args.parent_pid).create_time())\\n        except (psutil.Error, ValueError):\\n            stop_requested = True\\n    signal.signal(signal.SIGTERM, request_stop)\\n    signal.signal(signal.SIGINT, request_stop)\\n    try:\\n        serve_thread.start()\\n        if stop_requested:\\n            raise RuntimeError("Manager owner identity is unavailable")\\n        manager.start()\\n        while not stop_requested and not manager._exit_requested:\\n            time.sleep(.1)\\n            if args.parent_pid:\\n                try:\\n                    parent = psutil.Process(args.parent_pid)\\n                    if parent_identity is None or abs(parent.create_time() - parent_identity) >= .001 or not parent.is_running():\\n                        break\\n                except psutil.Error:\\n                    break\\n    finally:\\n        manager.close()\\n        server.shutdown()\\n        server.server_close()\\n        serve_thread.join(timeout=3)\\n\\n\\nif __name__ == "__main__":\\n    logging.basicConfig(level=logging.INFO)\\n    main()\\n\', \'model_container.py\': \'"""Private, independently supervised MiniMax-H3 model runtime.\\n\\nThe HTTP controller never imports torch or loads weights. The supervised owner\\nloads after binding its socket. Budgeted CPU stages overlap across requests and\\nshare one exclusive GPU lease. Production backends retain native BF16; injected\\nbackends are only a test seam.\\n"""\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport copy\\nfrom contextlib import contextmanager\\nfrom fractions import Fraction\\nimport gc\\nimport hashlib\\nimport json\\nimport logging\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport queue\\nimport re\\nimport shutil\\nimport signal\\nimport sqlite3\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nimport uuid\\n\\nfrom control_protocol import ControlError, make_server\\nfrom condition_guard import evaluate_gate, negate_observation, observation_state\\nfrom download_progress import read_progress\\nfrom pipeline_scheduler import BudgetUnavailable, PipelineBudget\\n\\nLOG = logging.getLogger("h3.model_container")\\nIDENTIFIER = re.compile(r"[a-f0-9]{32}\\\\Z")\\nTERMINAL = {"succeeded", "failed", "cancelled"}\\nSOURCE_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\nEXECUTE_CONDITIONS = ("instance_matches", "request_validated", "model_loaded", "worker_alive",\\n    "capability_available", "not_draining", "not_stopping", "attempt_available", "job_unowned",\\n    "artifact_absent", "generation_options_valid", "resources_reserved")\\nSTAGE_CONDITIONS = ("instance_matches", "attempt_owned", "attempt_live", "model_loaded", "worker_alive",\\n    "capability_available", "admission_confirmed", "predecessor_completed", "stage_not_started",\\n    "cancel_flag_clear", "cancel_event_clear", "reservation_owned", "budget_grant_held")\\n\\n\\nclass ExecutionCancelled(Exception):\\n    pass\\n\\n\\nclass ConditionGateBlocked(RuntimeError):\\n    """Missing or contradictory evidence is a failed execution, not cancellation."""\\n    pass\\n\\n\\ndef _all_observed(values):\\n    """Conjoin distinct obligations; a missing obligation is not a spare witness."""\\n    states = [observation_state(value) for value in values]\\n    if "conflict" in states:\\n        return "conflict"\\n    if "unknown" in states:\\n        return None\\n    return all(state == "true" for state in states)\\n\\n\\ndef parent_identity(parent_pid):\\n    """Bind ownership to one OS process generation, including on Windows."""\\n    import psutil\\n    try:\\n        process = psutil.Process(parent_pid)\\n        # A Windows venv redirector may be the OS parent. The authenticated\\n        # launcher specifies the actual owner; bind to that process generation.\\n        if not process.is_running() or process.status() == psutil.STATUS_ZOMBIE:\\n            raise RuntimeError("The requested model owner is not alive")\\n        return process.create_time()\\n    except psutil.Error as error:\\n        raise RuntimeError("Cannot establish the model parent process identity") from error\\n\\n\\ndef parent_is_alive(parent_pid, created_at):\\n    import psutil\\n    try:\\n        process = psutil.Process(parent_pid)\\n        return (process.is_running() and process.create_time() == created_at\\n                and process.status() != psutil.STATUS_ZOMBIE)\\n    except psutil.Error:\\n        # Unverified ownership must not retain a costly unmanaged model worker.\\n        return False\\n\\n\\ndef verify_artifact(path: Path, parameters: dict, *, cpu_threads=1) -> dict:\\n    """Validate the entire audiovisual artifact before publishing success."""\\n    if path.is_symlink() or not path.is_file() or not path.stat().st_size:\\n        raise ValueError("Missing model artifact")\\n    probe = subprocess.run(\\n        ["ffprobe", "-v", "error", "-threads", str(cpu_threads), "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)],\\n        capture_output=True, text=True, timeout=120, check=True,\\n    )\\n    data = json.loads(probe.stdout)\\n    streams = data.get("streams", [])\\n    video = next((x for x in streams if x.get("codec_type") == "video"), None)\\n    audio = next((x for x in streams if x.get("codec_type") == "audio"), None)\\n    if video is None or audio is None:\\n        raise ValueError("Artifact must contain video and audio")\\n    frames = int(video.get("nb_read_frames") or video.get("nb_frames") or 0)\\n    fps = float(Fraction(video.get("avg_frame_rate", "0")))\\n    duration = float(video.get("duration") or data.get("format", {}).get("duration") or 0)\\n    expected_frames = 5 + 17 * math.ceil((round(parameters["seconds"] * 24) - 5) / 17)\\n    if ((video.get("width"), video.get("height")) != (parameters["width"], parameters["height"])\\n            or video.get("codec_name") != "h264" or frames != expected_frames\\n            or abs(fps - 24) > .01 or not math.isfinite(duration)\\n            or abs(duration - expected_frames / 24) > .1 or int(audio.get("channels", 0)) != 2):\\n        raise ValueError("Artifact media does not match the admitted request")\\n    subprocess.run(\\n        ["ffmpeg", "-nostdin", "-v", "error", "-xerror", "-threads", str(cpu_threads),\\n         "-filter_threads", "1", "-filter_complex_threads", "1", "-i", str(path),\\n         "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"],\\n        capture_output=True, text=True, timeout=300, check=True,\\n    )\\n    digest = hashlib.sha256()\\n    with path.open("rb") as source:\\n        for block in iter(lambda: source.read(1024 * 1024), b""):\\n            digest.update(block)\\n    return {"path": str(path), "bytes": path.stat().st_size, "sha256": digest.hexdigest(),\\n            "media": {"video_codec": video["codec_name"], "width": video["width"], "height": video["height"],\\n                      "decoded_frames": frames, "fps": fps, "duration_seconds": duration,\\n                      "audio_codec": audio.get("codec_name"), "audio_channels": audio["channels"],\\n                      "audio_sample_rate": audio.get("sample_rate"), "full_decode_passed": True}}\\n\\n\\nclass ModelContainer:\\n    def __init__(self, root, instance_id, *, backend=None, staged=False,\\n                 artifact_validator=verify_artifact, resource_limits=None, progress_path=None):\\n        if not isinstance(instance_id, str) or not IDENTIFIER.fullmatch(instance_id):\\n            raise ValueError("Model instance must be a 32-character hexadecimal ID")\\n        self.root = Path(root).resolve()\\n        self.root.mkdir(parents=True, exist_ok=True)\\n        self.instance_id, self.staged = instance_id, bool(staged)\\n        self.backend = backend\\n        self._injected_backend = backend is not None\\n        self._resource_limits = resource_limits\\n        self._pipeline = False\\n        self._budget = None\\n        self._stage_workers = {}\\n        self._results = {}\\n        self.artifact_validator = artifact_validator\\n        self.event_directory = self.root / "_model_runtime" / instance_id\\n        if self.event_directory.resolve() != self.event_directory:\\n            raise ValueError("Model audit directory must not traverse a symlink")\\n        self.event_directory.mkdir(parents=True, exist_ok=True)\\n        self.events_path = self.event_directory / "events.jsonl"\\n        self.progress_path = Path(progress_path) if progress_path else self.event_directory / "downloads.json"\\n        self._lock = threading.RLock()\\n        self._queue = queue.Queue()\\n        self._worker = None\\n        self._done = threading.Event()\\n        self._stop_requested = False\\n        self._draining = False\\n        self._loaded = False\\n        self._phase = "loading"\\n        self._error = None\\n        self._active = None\\n        self._last_result = None\\n        self._attempts = {}\\n        self._sequence = 0\\n        self._consecutive_failures = 0\\n        self._worker_cleanup_complete = False\\n        self._last_condition_gate = None\\n        self._ledger_path = self.event_directory / "attempts.sqlite3"\\n        ledger = sqlite3.connect(self._ledger_path)\\n        try:\\n            with ledger:\\n                ledger.execute("CREATE TABLE IF NOT EXISTS acknowledged (attempt_id TEXT PRIMARY KEY, record TEXT NOT NULL)")\\n        finally:\\n            ledger.close()\\n\\n    def _previous_attempt(self, attempt_id):\\n        if attempt_id in self._attempts:\\n            return self._attempts[attempt_id]\\n        # Open briefly so acknowledgement does not retain a lifetime DB handle.\\n        ledger = sqlite3.connect(self._ledger_path)\\n        try:\\n            row = ledger.execute("SELECT record FROM acknowledged WHERE attempt_id=?", (attempt_id,)).fetchone()\\n            return json.loads(row[0]) if row else None\\n        finally:\\n            ledger.close()\\n\\n    def _live_attempts(self):\\n        return [value for value in self._attempts.values() if value["status"] not in TERMINAL]\\n\\n    def _event(self, kind, **data):\\n        with self._lock:\\n            self._sequence += 1\\n            event = {"sequence": self._sequence, "event_id": uuid.uuid4().hex,\\n                     "instance_id": self.instance_id, "type": kind, "observed_at": time.time(),\\n                     "monotonic_ns": time.monotonic_ns(), **data}\\n            try:\\n                with self.events_path.open("a", encoding="utf-8") as output:\\n                    output.write(json.dumps(event, ensure_ascii=False) + "\\\\n")\\n                    output.flush()\\n                    os.fsync(output.fileno())\\n            except OSError as error:\\n                self._loaded = False\\n                self._phase = "failed"\\n                self._error = "model_audit_unavailable"\\n                raise ControlError("Model audit storage unavailable", status=503) from error\\n\\n    def _safe_event(self, kind, **data):\\n        try:\\n            self._event(kind, **data)\\n        except BaseException:\\n            LOG.exception("Could not persist model event %s", kind)\\n\\n    def _gate(self, gate_id, required, observations, *, job_id, attempt_id, attempt=None, **context):\\n        """Caller owns _lock; persist the decision before permitting its effect."""\\n        receipt = evaluate_gate(gate_id, required, observations, {\\n            "instance_id": self.instance_id, "job_id": job_id, "attempt_id": attempt_id,\\n            "source_file": "model_container.py", "source_sha256": SOURCE_SHA256, **context})\\n        self._event("condition.gate", job_id=job_id, attempt_id=attempt_id,\\n                    gate_id=gate_id, receipt=receipt)\\n        self._last_condition_gate = receipt\\n        if attempt is not None:\\n            attempt.setdefault("condition_gates", {})[gate_id] = receipt\\n        return receipt\\n\\n    def _capability_observation(self, *, admitted=False):\\n        if type(self._phase) is not str:\\n            return None\\n        allowed = {"ready", "running", "draining"} if admitted else {"ready", "running"}\\n        if self._phase in allowed:\\n            return True\\n        if self._phase in {"loading", "starting", "failed", "stopped", "draining"}:\\n            return False\\n        return None\\n\\n    def start(self):\\n        with self._lock:\\n            if self._worker is not None:\\n                raise ControlError("Model container has already started", status=409)\\n            self._event("model.loading")\\n            self._worker = threading.Thread(target=self._work, name="h3-model-worker", daemon=False)\\n            try:\\n                self._worker.start()\\n            except BaseException:\\n                self._phase, self._error = "failed", "worker_start_failed"\\n                self._loaded = False\\n                self._done.set()\\n                self._safe_event("model.failed", error=self._error)\\n                raise\\n        return self\\n\\n    def status(self):\\n        with self._lock:\\n            alive = self._worker is not None and self._worker.is_alive()\\n            if self._worker is not None and not alive and self._phase not in {"failed", "stopped"}:\\n                self._phase, self._error, self._loaded = "failed", "worker_exited", False\\n            ready = self._loaded is True and alive and self._capability_observation() is True\\n            downloads = read_progress(self.progress_path, expected_instance_id=self.instance_id)\\n            downloads.update(model_ready=ready, lifecycle=self._phase,\\n                             observation_stale=bool(downloads.get("observation_stale") or not alive))\\n            phase = None\\n            snapshot = getattr(self.backend, "phase_snapshot", None)\\n            if snapshot:\\n                try:\\n                    phase = snapshot()\\n                except Exception:\\n                    phase = None\\n            active = self._active\\n            live = self._live_attempts()\\n            # Keep each reply below the private protocol\\\'s bounded body. Results\\n            # remain owned until ack; additional pages surface after that ack.\\n            results, result_bytes = [], 0\\n            for result in self._results.values():\\n                size = len(json.dumps(result).encode())\\n                if results and result_bytes + size > 1024 * 1024:\\n                    break\\n                results.append(result)\\n                result_bytes += size\\n            return copy.deepcopy({\\n                "instance_id": self.instance_id, "phase": self._phase, "ready": ready,\\n                "accepting": ready and self._draining is False and self._stop_requested is False and (\\n                    self._budget.has_capacity() if self._pipeline else active is None),\\n                "pipeline": self._pipeline,\\n                "downloads": downloads,\\n                "budget": self._budget.snapshot() if self._budget else None,\\n                "attempts": [{**self._accepted(item), "phase": item.get("stage", item["status"]),\\n                              "generation_options": item.get("generation_options"),\\n                              "condition_gates": item.get("condition_gates", {}),\\n                              "stage_completions": item.get("stage_completions", {}),\\n                              "cancel_requested": item["cancel_requested"]} for item in live],\\n                "results": results, "pending_results": len(self._results),\\n                "active_job_id": active["job_id"] if active else None,\\n                "active_attempt_id": active["attempt_id"] if active else None,\\n                "cancel_requested": bool(active and active["cancel_requested"]),\\n                "active_phase": phase, "last_result": self._last_result,\\n                "worker_alive": alive, "cleanup_complete": False,\\n                "worker_cleanup_complete": self._worker_cleanup_complete,\\n                "cleanup_scope": "Process exit must be confirmed by the owner before GPU ownership is released",\\n                "error": self._error, "observed_at": time.time(), "event_sequence": self._sequence,\\n                "last_condition_gate": self._last_condition_gate,\\n            })\\n\\n    def _check_instance(self, payload, required=False):\\n        instance = payload.get("instance_id")\\n        if instance != self.instance_id and (required or instance is not None):\\n            raise ControlError("Model instance does not match this execution generation", status=409)\\n\\n    def _request(self, payload):\\n        self._check_instance(payload, required=True)\\n        job_id, attempt_id = payload.get("job_id"), payload.get("attempt_id")\\n        if any(not isinstance(x, str) or not IDENTIFIER.fullmatch(x) for x in (job_id, attempt_id)):\\n            raise ControlError("Invalid job or attempt identity", status=422)\\n        parameters = payload.get("parameters")\\n        names = {"prompt", "references", "seconds", "width", "height", "steps", "seed"}\\n        optional = {"attention_backend", "acceleration_lora"}\\n        if not isinstance(parameters, dict) or not names <= set(parameters) or set(parameters) - names - optional:\\n            raise ControlError("Invalid generation parameters", status=422)\\n        p = copy.deepcopy(parameters)\\n        try:\\n            from adapter_config import load_adapter_settings\\n            from generation_options import requested_options, validate_generation_options\\n            validate_generation_options(requested_options(p), load_adapter_settings())\\n        except (TypeError, ValueError) as error:\\n            raise ControlError(str(error), status=422) from None\\n        if not isinstance(p["prompt"], str) or not p["prompt"].strip() or len(p["prompt"]) > 4000:\\n            raise ControlError("Invalid prompt", status=422)\\n        if any(isinstance(p[k], bool) or not isinstance(p[k], int) for k in ("width", "height", "steps", "seed")):\\n            raise ControlError("Generation dimensions, steps and seed must be integers", status=422)\\n        if (not 256 <= p["width"] <= 1344 or not 256 <= p["height"] <= 1344\\n                or p["width"] % 32 or p["height"] % 32 or p["width"] * p["height"] > 1344 * 768\\n                or not 2 <= p["steps"] <= 50 or not 0 <= p["seed"] <= 2147483647):\\n            raise ControlError("Generation parameters outside resource limits", status=422)\\n        seconds = p["seconds"]\\n        if (isinstance(seconds, bool) or not isinstance(seconds, (int, float)) or not math.isfinite(seconds)\\n                or not 5 <= seconds <= 15 or 5 + 17 * math.ceil((round(seconds * 24) - 5) / 17) > 360):\\n            raise ControlError("Invalid generation duration", status=422)\\n        refs = p["references"]\\n        if not isinstance(refs, list) or not 1 <= len(refs) <= 12:\\n            raise ControlError("Supply 1-12 references", status=422)\\n        directory = self.root / job_id\\n        inputs = directory / "inputs"\\n        if directory.resolve() != directory or inputs.resolve() != inputs:\\n            raise ControlError("Reference directory must stay within its assigned job", status=422)\\n        kinds = []\\n        total = 0\\n        for ref in refs:\\n            if not isinstance(ref, dict) or set(ref) != {"kind", "path"} or ref["kind"] not in {"image", "video", "audio"}:\\n                raise ControlError("Invalid reference metadata", status=422)\\n            try:\\n                path = Path(ref["path"])\\n                resolved = path.resolve(strict=True)\\n                size = resolved.stat().st_size\\n                valid = (path.is_absolute() and resolved.is_relative_to(inputs) and path == resolved\\n                         and resolved.is_file() and not path.is_symlink() and 0 < size <= 32 * 1024 * 1024)\\n            except (OSError, TypeError, ValueError):\\n                valid = False\\n            if not valid:\\n                raise ControlError("Reference path is missing, oversized or outside its assigned inputs", status=422)\\n            total += size\\n            ref["path"] = str(resolved)\\n            kinds.append(ref["kind"])\\n        if (total > 64 * 1024 * 1024 or kinds.count("image") > 9 or kinds.count("video") > 3\\n                or kinds.count("audio") > 3 or all(k == "audio" for k in kinds)):\\n            raise ControlError("Reference combination outside resource limits", status=422)\\n        fingerprint = hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()\\n        return job_id, attempt_id, p, fingerprint\\n\\n    def execute(self, payload):\\n        # Idempotent replies must remain possible after inputs have been cleaned.\\n        self._check_instance(payload, required=True)\\n        attempt_id = payload.get("attempt_id")\\n        with self._lock:\\n            old = self._previous_attempt(attempt_id) if isinstance(attempt_id, str) else None\\n            if old:\\n                candidate = hashlib.sha256(json.dumps(payload.get("parameters"), sort_keys=True).encode()).hexdigest()\\n                if old["job_id"] != payload.get("job_id") or old["fingerprint"] != candidate:\\n                    raise ControlError("Attempt identity was reused with different parameters", status=409)\\n                return self._accepted(old)\\n        try:\\n            job_id, attempt_id, parameters, fingerprint = self._request(payload)\\n        except ControlError as error:\\n            if error.status == 422:\\n                raise ControlError(json.dumps({"guarantees_not_accepted": True,\\n                                               "reason": "invalid_request", "detail": str(error)}), 422) from None\\n            raise\\n        # Header probes can perform bounded I/O. Do not hold the controller\\\'s\\n        # state lock while estimating an otherwise unadmitted request.\\n        resources = None\\n        with self._lock:\\n            estimator = self.backend if self._pipeline and self._loaded is True and self._draining is False else None\\n        if estimator is not None:\\n            try:\\n                resources = estimator.estimate_resources(parameters)\\n            except ValueError:\\n                raise ControlError(json.dumps({"guarantees_not_accepted": True,\\n                                               "reason": "invalid_reference_media"}), status=422) from None\\n        with self._lock:\\n            old = self._previous_attempt(attempt_id)\\n            if old:\\n                if old["job_id"] != job_id or old["fingerprint"] != fingerprint:\\n                    raise ControlError("Attempt identity conflict", status=409)\\n                return self._accepted(old)\\n            output = self.root / job_id / "result.mp4"\\n            validator = getattr(self.backend, "validate_request", None)\\n            options_valid = True\\n            if validator:\\n                try:\\n                    validator(steps=parameters["steps"], **{name: parameters[name]\\n                        for name in ("attention_backend", "acceleration_lora") if name in parameters})\\n                except ValueError:\\n                    options_valid = False\\n            observations = {\\n                "instance_matches": payload.get("instance_id") == self.instance_id,\\n                "request_validated": True,\\n                "model_loaded": self._loaded,\\n                "worker_alive": self._worker.is_alive() if self._worker is not None else False,\\n                "capability_available": self._capability_observation(),\\n                "not_draining": negate_observation(self._draining),\\n                "not_stopping": negate_observation(self._stop_requested),\\n                "attempt_available": old is None,\\n                "job_unowned": not any(item["job_id"] == job_id for item in self._live_attempts()),\\n                "artifact_absent": not (output.exists() or output.is_symlink()),\\n                "generation_options_valid": options_valid,\\n                "resources_reserved": None,\\n            }\\n            context = {"pipeline": self._pipeline, "budget_scope": "stage_budget" if self._pipeline else "legacy_single_active"}\\n            if any(observation_state(value) != "true" for key, value in observations.items() if key != "resources_reserved"):\\n                receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\\n                    job_id=job_id, attempt_id=attempt_id, **context)\\n                raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": "model_conditions_blocked",\\n                    "decision_id": receipt["decision_id"], "blocked": receipt["blocked"]}), 409 if options_valid else 422)\\n            attempt = {"job_id": job_id, "attempt_id": attempt_id, "instance_id": self.instance_id,\\n                       "status": "accepted", "cancel_requested": False, "parameters": parameters,\\n                       "fingerprint": fingerprint, "accepted_at": time.time(), "condition_gates": {},\\n                       "stage_started": {"prepare": False, "gpu": False, "encode_validate": False},\\n                       "stage_completions": {"prepare": None, "gpu": None, "encode_validate": None}}\\n            reserved = False\\n            if self._pipeline:\\n                if resources is None:\\n                    receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\\n                        job_id=job_id, attempt_id=attempt_id, **context)\\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True,\\n                                                   "reason": "model_not_accepting", "decision_id": receipt["decision_id"]}), status=409)\\n                try:\\n                    self._budget.reserve(attempt_id, resources)\\n                except BudgetUnavailable as error:\\n                    self._event("reservation.blocked", job_id=job_id, attempt_id=attempt_id,\\n                                reservation_witness=self._budget.reservation_witness(attempt_id),\\n                                reservation_receipt=self._budget.gate_receipt(attempt_id, "reserve"))\\n                    observations["resources_reserved"] = False\\n                    receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\\n                        job_id=job_id, attempt_id=attempt_id, budget_reason=str(error), **context)\\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": str(error),\\n                                                   "decision_id": receipt["decision_id"]}),\\n                                       status=507 if error.permanent else 429) from None\\n                reserved = True\\n                attempt["resources"] = resources\\n                attempt["_cancel_event"] = threading.Event()\\n            try:\\n                witness = self._budget.reservation_witness(attempt_id) if self._pipeline else None\\n                if self._pipeline:\\n                    self._event("reservation.acquired", job_id=job_id, attempt_id=attempt_id,\\n                                reservation_witness=witness,\\n                                reservation_receipt=self._budget.gate_receipt(attempt_id, "reserve"))\\n                observations["resources_reserved"] = (_all_observed([\\n                    witness.get("owned"), witness.get("attempt_id") == attempt_id,\\n                    witness.get("budget_instance_id") == self._budget.instance_id,\\n                    type(witness.get("reservation_id")) is str and bool(witness["reservation_id"])\\n                ]) if witness is not None else self._active is None if not self._pipeline else None)\\n                receipt = self._gate("model.execute", EXECUTE_CONDITIONS, observations,\\n                    job_id=job_id, attempt_id=attempt_id, attempt=attempt,\\n                    budget_instance_id=witness.get("budget_instance_id") if witness else None,\\n                    reservation_id=witness.get("reservation_id") if witness else None, **context)\\n                if receipt["allowed"] is not True:\\n                    raise ControlError(json.dumps({"guarantees_not_accepted": True, "reason": "model_conditions_blocked",\\n                        "decision_id": receipt["decision_id"], "blocked": receipt["blocked"]}), 409)\\n            except BaseException:\\n                if reserved:\\n                    self._budget.finish(attempt_id)\\n                raise\\n            self._attempts[attempt_id] = attempt\\n            if not self._pipeline:\\n                self._active = attempt\\n            self._phase = "running"\\n            try:\\n                self._event("execution.accepted", job_id=job_id, attempt_id=attempt_id,\\n                            resources=attempt.get("resources"), condition_decision_id=receipt["decision_id"])\\n                self._queue.put_nowait(attempt)\\n            except BaseException:\\n                attempt.update(status="failed", error="dispatch_failed")\\n                self._active = None\\n                if self._budget:\\n                    self._budget.finish(attempt_id)\\n                self._loaded, self._phase, self._error = False, "failed", "dispatch_failed"\\n                raise\\n            return self._accepted(attempt)\\n\\n    def _accepted(self, attempt):\\n        return copy.deepcopy({key: attempt[key] for key in ("job_id", "attempt_id", "instance_id", "status")}\\n                             | {"accepted": True, "result": attempt.get("result")})\\n\\n    def cancel(self, payload):\\n        self._check_instance(payload, required=True)\\n        if not isinstance(payload.get("attempt_id"), str):\\n            raise ControlError("Invalid execution attempt", status=422)\\n        with self._lock:\\n            attempt = self._previous_attempt(payload.get("attempt_id"))\\n            if attempt is None or attempt["job_id"] != payload.get("job_id"):\\n                raise ControlError("Execution attempt not found", status=404)\\n            if attempt["status"] in TERMINAL:\\n                return self._accepted(attempt)\\n            if not attempt["cancel_requested"]:\\n                self._event("execution.cancel_requested", job_id=attempt["job_id"], attempt_id=attempt["attempt_id"])\\n                attempt.update(cancel_requested=True, status="cancel_requested")\\n                if "_cancel_event" in attempt:\\n                    attempt["_cancel_event"].set()\\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\\n            return self._accepted(attempt)\\n\\n    def drain(self, payload=None):\\n        self._check_instance(payload or {})\\n        with self._lock:\\n            if not self._draining:\\n                self._event("model.drain_requested")\\n                self._draining = True\\n                if self._phase not in {"failed", "stopped"}:\\n                    self._phase = "draining"\\n        return self.status()\\n\\n    def stop(self, payload=None):\\n        self._check_instance(payload or {})\\n        with self._lock:\\n            if not self._stop_requested:\\n                self._event("model.stop_requested")\\n                self._stop_requested, self._draining = True, True\\n                if self._phase not in {"failed", "stopped"}:\\n                    self._phase = "draining"\\n        return self.status()\\n\\n    def dispatch(self, operation, payload=None):\\n        payload = payload or {}\\n        if operation == "status":\\n            return self.status()\\n        if operation == "execute":\\n            return self.execute(payload)\\n        if operation == "cancel":\\n            return self.cancel(payload)\\n        if operation == "drain":\\n            return self.drain(payload)\\n        if operation == "stop":\\n            return self.stop(payload)\\n        if operation == "ack_results":\\n            return self.ack_results(payload)\\n        raise ControlError("Unknown model operation", status=404)\\n\\n    def ack_results(self, payload):\\n        self._check_instance(payload, required=True)\\n        identities = payload.get("attempt_ids")\\n        if not isinstance(identities, list) or any(not isinstance(x, str) or not IDENTIFIER.fullmatch(x) for x in identities):\\n            raise ControlError("Invalid result acknowledgement", status=422)\\n        with self._lock:\\n            acknowledged = []\\n            for identity in identities:\\n                attempt = self._attempts.get(identity)\\n                if attempt and attempt["status"] not in TERMINAL:\\n                    raise ControlError("Cannot acknowledge an unfinished execution", status=409)\\n                if identity not in self._results:\\n                    if self._previous_attempt(identity):\\n                        acknowledged.append(identity)\\n                    continue\\n                record = {key: value for key, value in attempt.items() if not key.startswith("_")}\\n                ledger = sqlite3.connect(self._ledger_path)\\n                try:\\n                    with ledger:\\n                        ledger.execute("INSERT OR REPLACE INTO acknowledged VALUES (?,?)", (identity, json.dumps(record)))\\n                finally:\\n                    ledger.close()\\n                self._results.pop(identity)\\n                self._attempts.pop(identity)\\n                if self._budget:\\n                    self._budget.acknowledge(identity)\\n                acknowledged.append(identity)\\n            return {"instance_id": self.instance_id, "acknowledged": acknowledged}\\n\\n    def _revoke_pipeline(self, error, *, current=None):\\n        with self._lock:\\n            self._loaded, self._phase, self._error = False, "failed", error\\n            for attempt in self._live_attempts():\\n                if attempt is not current:\\n                    attempt["cancel_requested"] = True\\n                    attempt["_cancel_event"].set()\\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\\n\\n    @contextmanager\\n    def _pipeline_stage(self, attempt, name, lease):\\n        if name not in {"prepare", "gpu", "encode_validate"} or lease != ("gpu" if name == "gpu" else "cpu"):\\n            raise ValueError("Unknown pipeline stage or resource lease")\\n        cancellation = attempt["_cancel_event"]\\n        identity, job_id = attempt["attempt_id"], attempt["job_id"]\\n        witness = None\\n        # Never hold the model lock while waiting for a budget grant. Budget\\n        # callbacks read Event only and never acquire this model\\\'s lock.\\n        try:\\n            with self._budget.stage(identity, lease, cpu_threads=attempt["resources"]["cpu_threads"],\\n                                    cancellation_event=cancellation,\\n                                    context={"model_instance_id": self.instance_id, "job_id": job_id,\\n                                             "stage": name}) as acquired:\\n                with self._lock:\\n                    witness = self._budget.lease_witness(identity, lease)\\n                    reservation = self._budget.reservation_witness(identity)\\n                    if witness is not None:\\n                        self._event("lease.acquired", job_id=job_id, attempt_id=identity, stage=name,\\n                                    lease_id=witness.get("lease_id"), budget_instance_id=witness.get("budget_instance_id"),\\n                                    lease_witness=witness)\\n                    admission = attempt.get("condition_gates", {}).get("model.execute", {})\\n                    admission_context = admission.get("context", {})\\n                    previous = {"prepare": None, "gpu": "prepare", "encode_validate": "gpu"}[name]\\n                    live_state = attempt.get("status")\\n                    live = (None if type(live_state) is not str else False if live_state in TERMINAL else True if live_state in {\\n                        "accepted", "preparing", "gpu_wait", "gpu_running", "encoding", "validating", "cancel_requested"} else None)\\n                    observations = {\\n                        "instance_matches": attempt.get("instance_id") == self.instance_id,\\n                        "attempt_owned": (self._attempts.get(identity) is attempt\\n                            and admission_context.get("instance_id") == self.instance_id\\n                            and admission_context.get("job_id") == job_id and admission_context.get("attempt_id") == identity),\\n                        "attempt_live": live, "model_loaded": self._loaded,\\n                        "worker_alive": self._worker.is_alive() if self._worker is not None else False,\\n                        "capability_available": self._capability_observation(admitted=True),\\n                        "admission_confirmed": admission.get("allowed"),\\n                        "predecessor_completed": admission.get("allowed") if previous is None else attempt.get("stage_completions", {}).get(previous),\\n                        "stage_not_started": negate_observation(attempt.get("stage_started", {}).get(name)),\\n                        "cancel_flag_clear": negate_observation(attempt.get("cancel_requested")),\\n                        "cancel_event_clear": negate_observation(cancellation.is_set()),\\n                        "reservation_owned": _all_observed([\\n                            reservation.get("owned"), reservation.get("attempt_id") == identity,\\n                            reservation.get("budget_instance_id") == self._budget.instance_id,\\n                            type(reservation.get("reservation_id")) is str and bool(reservation["reservation_id"])\\n                        ]) if reservation else None,\\n                        "budget_grant_held": False if acquired is False else _all_observed([acquired,\\n                            witness.get("owned") if witness else (False if acquired is False else None),\\n                            witness.get("attempt_id") == identity if witness else None,\\n                            witness.get("kind") == lease if witness else None,\\n                            witness.get("budget_instance_id") == self._budget.instance_id if witness else None,\\n                            witness.get("reservation_id") == reservation.get("reservation_id") if witness and reservation else None]),\\n                    }\\n                    receipt = self._gate("model.stage." + name, STAGE_CONDITIONS, observations,\\n                        job_id=job_id, attempt_id=identity, attempt=attempt, stage=name,\\n                        lease_id=witness.get("lease_id") if witness else None,\\n                        budget_instance_id=witness.get("budget_instance_id") if witness else None,\\n                        budget_decision_id=witness.get("decision_id") if witness else None)\\n                    if receipt["allowed"] is not True:\\n                        # A confirmed cancellation is distinct from unknown or\\n                        # contradictory ownership/capability/precedence evidence.\\n                        non_cancel_unknown = any(state in {"unknown", "conflict"} for state in receipt["conditions"].values())\\n                        if (not non_cancel_unknown and attempt.get("cancel_requested") is True\\n                                and cancellation.is_set() is True):\\n                            raise ExecutionCancelled()\\n                        raise ConditionGateBlocked("Pipeline stage condition evidence is blocked")\\n                    attempt["stage"] = name\\n                    attempt["stage_started"][name] = True\\n                    attempt["status"] = {"prepare": "preparing", "gpu": "gpu_running", "encode_validate": "encoding"}[name]\\n                    stage_resources = dict(attempt["resources"], cpu_threads=(\\n                        self._budget.gpu_host_cpu_threads if lease == "gpu" else attempt["resources"]["cpu_threads"]))\\n                    event_context = {"condition_decision_id": receipt["decision_id"], "lease_id": witness["lease_id"],\\n                                     "budget_instance_id": witness["budget_instance_id"], "lease_witness": witness}\\n                    self._event("stage.started", job_id=job_id, attempt_id=identity, stage=name,\\n                                resources=stage_resources, **event_context)\\n                succeeded = False\\n                try:\\n                    yield\\n                    succeeded = True\\n                except BaseException:\\n                    if lease == "gpu":\\n                        self._revoke_pipeline("gpu_execution_failed", current=attempt)\\n                    raise\\n                finally:\\n                    with self._lock:\\n                        # A successfully persisted finish precedes the completion\\n                        # witness. An audit failure cannot unlock the next stage.\\n                        self._event("stage.finished", job_id=job_id, attempt_id=identity,\\n                                    stage=name, succeeded=succeeded, resources=stage_resources, **event_context)\\n                        attempt["stage_completions"][name] = succeeded\\n        finally:\\n            if witness is not None:\\n                with self._lock:\\n                    released = self._budget.release_witness(witness["lease_id"])\\n                    self._event("lease.released", job_id=job_id, attempt_id=identity, stage=name,\\n                                lease_id=witness["lease_id"], budget_instance_id=witness["budget_instance_id"],\\n                                lease_witness=released)\\n                    if released is None:\\n                        raise ConditionGateBlocked("Lease release evidence is unavailable")\\n\\n    def _run_pipeline_attempt(self, attempt):\\n        identity = attempt["attempt_id"]\\n        path = self.root / attempt["job_id"] / "result.mp4"\\n        result = {"job_id": attempt["job_id"], "attempt_id": identity, "instance_id": self.instance_id,\\n                  "status": "failed", "error": None, "artifact": None, "metrics": None,\\n                  "started_at": time.time(), "cleanup_complete": False}\\n        prepared = cpu_result = None\\n        try:\\n            parameters = copy.deepcopy(attempt["parameters"])\\n            for reference in parameters["references"]:\\n                reference["path"] = Path(reference["path"])\\n            with self._pipeline_stage(attempt, "prepare", "cpu"):\\n                prepared = self.backend.prepare(cpu_threads=attempt["resources"]["cpu_threads"], **parameters)\\n                data = getattr(getattr(prepared, "metrics", None), "data", {})\\n                if isinstance(data.get("generation_options"), dict):\\n                    with self._lock:\\n                        attempt["generation_options"] = data["generation_options"]\\n            with self._lock:\\n                if not attempt["cancel_requested"]:\\n                    attempt["status"], attempt["stage"] = "gpu_wait", "gpu_wait"\\n            with self._pipeline_stage(attempt, "gpu", "gpu"):\\n                cpu_result = self.backend.infer(prepared)\\n                if getattr(cpu_result, "cpu_handoff_complete", False) is not True:\\n                    raise RuntimeError("Backend did not confirm CPU handoff and CUDA synchronization")\\n                if isinstance(attempt.get("generation_options"), dict):\\n                    self._event("generation.configured", job_id=attempt["job_id"], attempt_id=identity,\\n                                generation_options=copy.deepcopy(attempt["generation_options"]))\\n            # CPU encoding and full-file decoding consume their CPU budget, but\\n            # no GPU lease. Another admitted job may now perform its inference.\\n            with self._pipeline_stage(attempt, "encode_validate", "cpu"):\\n                metrics = self.backend.encode(cpu_result, output_path=path)\\n                if isinstance(metrics, dict):\\n                    result["metrics"] = metrics\\n                with self._lock:\\n                    if attempt["cancel_requested"]:\\n                        raise ExecutionCancelled()\\n                    attempt["status"] = "validating"\\n                if self.artifact_validator is verify_artifact:\\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"],\\n                        cpu_threads=attempt["resources"]["cpu_threads"])\\n                else:\\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"])\\n            result["status"] = "succeeded"\\n            with self._lock:\\n                self._consecutive_failures = 0\\n        except ExecutionCancelled:\\n            result["status"] = "cancelled"\\n        except BaseException as error:\\n            LOG.exception("Pipeline attempt failed: %s/%s", attempt["job_id"], identity)\\n            result["error"] = ("condition_gate_blocked" if isinstance(error, ConditionGateBlocked)\\n                               else "generation_failed; inspect model log")\\n            with self._lock:\\n                if not isinstance(error, ValueError):\\n                    self._consecutive_failures += 1\\n                fatal = not isinstance(error, Exception) or self._consecutive_failures >= 2\\n            if fatal:\\n                self._revoke_pipeline("model_execution_unavailable", current=attempt)\\n        finally:\\n            try:\\n                if isinstance(attempt.get("generation_options"), dict):\\n                    result["generation_options"] = copy.deepcopy(attempt["generation_options"])\\n                result["condition_gates"] = copy.deepcopy(attempt.get("condition_gates", {}))\\n                result["stage_completions"] = copy.deepcopy(attempt.get("stage_completions", {}))\\n                # This operation is strictly CPU cleanup; it must not empty the\\n                # CUDA cache while the next attempt owns the GPU lease.\\n                self.backend.cleanup(prepared=prepared, result=cpu_result)\\n                prepared = cpu_result = None\\n                if result["metrics"] is not None:\\n                    try:\\n                        encoded = json.dumps(result["metrics"], allow_nan=False)\\n                        result["metrics"] = json.loads(encoded) if len(encoded.encode()) <= 128 * 1024 else None\\n                    except (TypeError, ValueError):\\n                        result["metrics"] = None\\n                        LOG.warning("Pipeline metrics were not finite JSON; omitting diagnostics")\\n                inputs = self.root / attempt["job_id"] / "inputs"\\n                if inputs.is_symlink() or inputs.resolve() != inputs:\\n                    raise ValueError("Input ownership changed during pipeline execution")\\n                if inputs.exists():\\n                    shutil.rmtree(inputs)\\n                result["cleanup_complete"] = True\\n            except BaseException:\\n                LOG.exception("Pipeline attempt cleanup failed")\\n                result.update(status="failed", artifact=None, cleanup_complete=False,\\n                              error="execution_cleanup_failed; inspect model log")\\n                self._revoke_pipeline("execution_cleanup_failed", current=attempt)\\n            with self._lock:\\n                if attempt["cancel_requested"] and result["status"] == "succeeded":\\n                    result["status"] = "cancelled"\\n                if result["status"] != "succeeded":\\n                    result["artifact"] = None\\n                    try:\\n                        path.unlink(missing_ok=True)\\n                        path.with_suffix(".metrics.json").unlink(missing_ok=True)\\n                        path.with_suffix(".metrics.json.tmp").unlink(missing_ok=True)\\n                    except OSError:\\n                        result.update(status="failed", cleanup_complete=False, error="cancel_cleanup_failed")\\n                        self._revoke_pipeline("cancel_cleanup_failed", current=attempt)\\n                result["finished_at"] = time.time()\\n                attempt.update(status=result["status"], result=result)\\n                attempt.pop("parameters", None)\\n                self._results[identity] = self._last_result = result\\n                if result["cleanup_complete"]:\\n                    self._budget.finish(identity, len(json.dumps(result).encode()))\\n                if self._phase != "failed":\\n                    self._phase = "draining" if self._draining else ("running" if self._live_attempts() else "ready")\\n                self._safe_event("execution." + result["status"], job_id=attempt["job_id"],\\n                                 attempt_id=identity, result=result)\\n\\n    def _run_attempt(self, attempt):\\n        path = self.root / attempt["job_id"] / "result.mp4"\\n        result = {"job_id": attempt["job_id"], "attempt_id": attempt["attempt_id"],\\n                  "instance_id": self.instance_id, "status": "failed", "error": None,\\n                  "artifact": None, "metrics": None, "started_at": time.time(),\\n                  "cleanup_complete": False}\\n        fatal = None\\n        generated = False\\n        try:\\n            with self._lock:\\n                if not attempt["cancel_requested"]:\\n                    attempt["status"] = "running"\\n                self._event("execution.running", job_id=attempt["job_id"], attempt_id=attempt["attempt_id"])\\n            if not attempt["cancel_requested"]:\\n                params = copy.deepcopy(attempt["parameters"])\\n                for reference in params["references"]:\\n                    reference["path"] = Path(reference["path"])\\n                self.backend.generate(output_path=path, **params)\\n                generated = True\\n                if not attempt["cancel_requested"]:\\n                    result["artifact"] = self.artifact_validator(path, attempt["parameters"])\\n            with self._lock:\\n                result["status"] = "cancelled" if attempt["cancel_requested"] else "succeeded"\\n            self._consecutive_failures = 0\\n        except BaseException as error:\\n            LOG.exception("Model attempt failed: %s/%s", attempt["job_id"], attempt["attempt_id"])\\n            result["error"] = "generation_failed; inspect model log"\\n            # Bad input media may be a request error. Invalid generated output\\n            # is an execution failure even when the validator raises ValueError.\\n            if generated or not isinstance(error, ValueError):\\n                self._consecutive_failures += 1\\n            if not isinstance(error, Exception) or self._consecutive_failures >= 2:\\n                fatal = error\\n                with self._lock:\\n                    self._loaded, self._phase, self._error = False, "failed", "model_execution_unavailable"\\n        finally:\\n            try:\\n                # Keep the lease until execution and its local cleanup have ended.\\n                metrics = getattr(self.backend, "last_metrics", None)\\n                if isinstance(metrics, dict):\\n                    try:\\n                        encoded = json.dumps(metrics, allow_nan=False)\\n                        if len(encoded.encode("utf-8")) <= 128 * 1024:\\n                            result["metrics"] = json.loads(encoded)\\n                    except (TypeError, ValueError):\\n                        LOG.warning("Model metrics were not finite JSON; omitting diagnostics")\\n                if result["status"] != "succeeded":\\n                    result["artifact"] = None\\n                    path.unlink(missing_ok=True)\\n                inputs = self.root / attempt["job_id"] / "inputs"\\n                if inputs.is_symlink() or inputs.resolve() != inputs:\\n                    raise ValueError("Input directory ownership changed during execution")\\n                if inputs.exists():\\n                    shutil.rmtree(inputs)\\n                # Attempt-local files are reclaimed; this does not release the\\n                # model process\\\'s persistent CPU/GPU resource ownership.\\n                result["cleanup_complete"] = True\\n            except BaseException as error:\\n                LOG.exception("Attempt cleanup failed")\\n                result.update(status="failed", artifact=None, cleanup_complete=False,\\n                              error="execution_cleanup_failed; inspect model log")\\n                fatal = error\\n                with self._lock:\\n                    self._loaded, self._phase, self._error = False, "failed", "execution_cleanup_failed"\\n            with self._lock:\\n                # A cancel racing with validation/cleanup still wins before commit.\\n                if attempt["cancel_requested"] and result["status"] == "succeeded":\\n                    result.update(status="cancelled", artifact=None)\\n                    try:\\n                        path.unlink(missing_ok=True)\\n                    except OSError as error:\\n                        result.update(status="failed", error="cancel_cleanup_failed", cleanup_complete=False)\\n                        fatal = error\\n                        self._loaded, self._phase, self._error = False, "failed", "cancel_cleanup_failed"\\n                result["finished_at"] = time.time()\\n                attempt.update(status=result["status"], result=result)\\n                attempt.pop("parameters", None)\\n                self._last_result = result\\n                self._results[attempt["attempt_id"]] = result\\n                self._active = None\\n                if self._phase != "failed":\\n                    self._phase = "draining" if self._draining else "ready"\\n                self._event("execution." + result["status"], job_id=attempt["job_id"],\\n                            attempt_id=attempt["attempt_id"], result=result)\\n        if fatal is not None:\\n            raise fatal\\n\\n    def _work(self):\\n        try:\\n            if self.backend is None:\\n                if self.staged:\\n                    from staged_backend import StagedMiniMaxBackend\\n                    self.backend = StagedMiniMaxBackend()\\n                else:\\n                    from model_backend import MiniMaxBackend\\n                    self.backend = MiniMaxBackend()\\n            self._pipeline = all(callable(getattr(self.backend, name, None)) for name in\\n                                 ("prepare", "infer", "encode", "cleanup", "estimate_resources"))\\n            if not self._pipeline and not self._injected_backend:\\n                raise RuntimeError("Production backend must implement the staged pipeline contract")\\n            if self._pipeline:\\n                self._budget = PipelineBudget(self.root, limits=self._resource_limits,\\n                    model_peak_ram_bytes=getattr(self.backend, "pipeline_model_peak_ram_bytes", 144 * 1024 ** 3))\\n                configure = getattr(self.backend, "configure_cpu_budget", None)\\n                if callable(configure):\\n                    self._budget.native_cpu_status = configure(self._budget.gpu_host_cpu_threads)\\n                elif not self._injected_backend:\\n                    raise RuntimeError("Production backend must configure its native CPU thread budget before loading")\\n                else:\\n                    self._budget.native_cpu_status = {"configured": False, "scope": "injected CPU test backend"}\\n            self.backend.load()\\n            if self._pipeline:\\n                native_snapshot = getattr(self.backend, "cpu_budget_snapshot", None)\\n                if callable(native_snapshot):\\n                    # Model loading can import additional BLAS/OpenMP runtimes.\\n                    # Read back their limits before publishing readiness; never\\n                    # reconfigure process-wide pools while jobs are running.\\n                    self._budget.native_cpu_status = native_snapshot()\\n                elif not self._injected_backend:\\n                    raise RuntimeError("Production backend must verify native CPU budgets after loading")\\n            with self._lock:\\n                self._loaded = True\\n                self._phase = "draining" if self._draining else "ready"\\n                self._event("model.ready", accepting=not self._draining)\\n            while True:\\n                with self._lock:\\n                    if self._pipeline:\\n                        for identity, worker in list(self._stage_workers.items()):\\n                            if not worker.is_alive():\\n                                worker.join()\\n                                del self._stage_workers[identity]\\n                        if self._phase == "failed":\\n                            self._revoke_pipeline(self._error or "pipeline_unavailable")\\n                        if (self._stop_requested or self._phase == "failed") and not self._live_attempts() and not self._stage_workers:\\n                            break\\n                    else:\\n                        if self._stop_requested and self._active is None:\\n                            break\\n                        if self._phase == "failed":\\n                            break\\n                try:\\n                    attempt = self._queue.get(timeout=.1)\\n                except queue.Empty:\\n                    continue\\n                try:\\n                    if self._pipeline:\\n                        worker = threading.Thread(target=self._run_pipeline_attempt, args=(attempt,),\\n                                                  name="h3-stage-" + attempt["attempt_id"][:8], daemon=False)\\n                        with self._lock:\\n                            self._stage_workers[attempt["attempt_id"]] = worker\\n                            try:\\n                                worker.start()\\n                            except BaseException:\\n                                self._stage_workers.pop(attempt["attempt_id"], None)\\n                                self._revoke_pipeline("pipeline_worker_start_failed")\\n                                self._run_pipeline_attempt(attempt)\\n                                raise\\n                    else:\\n                        self._run_attempt(attempt)\\n                finally:\\n                    self._queue.task_done()\\n        except BaseException as error:\\n            LOG.exception("Model worker exited")\\n            with self._lock:\\n                self._loaded, self._phase = False, "failed"\\n                self._error = self._error or "model_load_or_worker_failed"\\n                if self._active is not None:\\n                    attempt = self._active\\n                    result = {"job_id": attempt["job_id"], "attempt_id": attempt["attempt_id"],\\n                              "instance_id": self.instance_id, "status": "failed", "artifact": None,\\n                              "metrics": None, "error": "model_worker_exited", "finished_at": time.time(),\\n                              "cleanup_complete": False}\\n                    attempt.update(status="failed", result=result)\\n                    self._results[attempt["attempt_id"]] = result\\n                    self._last_result, self._active = result, None\\n                self._safe_event("model.failed", error=self._error, error_type=type(error).__name__)\\n        finally:\\n            with self._lock:\\n                self._loaded = False\\n            if self._pipeline:\\n                # Never unload model state while prepare, CUDA inference, CPU\\n                # encoding, validation or local cleanup still refers to it.\\n                for worker in list(self._stage_workers.values()):\\n                    if worker.ident is not None:\\n                        worker.join()\\n                for attempt in self._live_attempts():\\n                    attempt["cancel_requested"] = True\\n                    attempt["_cancel_event"].set()\\n                    self._budget.set_cancelled(attempt["attempt_id"], True)\\n                    self._run_pipeline_attempt(attempt)\\n            try:\\n                backend, self.backend = self.backend, None\\n                unload = getattr(backend, "unload", None)\\n                if unload:\\n                    unload()\\n                if hasattr(backend, "pipe"):\\n                    backend.pipe = None\\n                del backend\\n                gc.collect()\\n                torch = sys.modules.get("torch")\\n                if torch is not None and torch.cuda.is_initialized():\\n                    torch.cuda.empty_cache()\\n                self._worker_cleanup_complete = True\\n            except BaseException:\\n                LOG.exception("Model worker cleanup failed")\\n                self._phase, self._error = "failed", "model_unload_failed"\\n            with self._lock:\\n                if self._phase != "failed":\\n                    self._phase = "stopped"\\n                self._safe_event("model.worker_exited", phase=self._phase,\\n                                 worker_cleanup_complete=self._worker_cleanup_complete,\\n                                 budget_snapshot=self._budget.snapshot() if self._budget is not None else None)\\n            self._done.set()\\n\\n    def join(self, timeout=None):\\n        if self._worker is not None and self._worker.ident is not None:\\n            self._worker.join(timeout)\\n        return self._done.is_set()\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--port", type=int, required=True)\\n    parser.add_argument("--instance-id", required=True)\\n    parser.add_argument("--root", type=Path, required=True)\\n    parser.add_argument("--staged", action="store_true")\\n    parser.add_argument("--parent-pid", type=int)\\n    args = parser.parse_args()\\n    owner_created_at = parent_identity(args.parent_pid) if args.parent_pid else None\\n    key = os.environ.get("H3_CONTROL_KEY", "")\\n    if len(key) < 32:\\n        raise RuntimeError("A private H3_CONTROL_KEY is required")\\n    container = ModelContainer(args.root, args.instance_id, staged=args.staged,\\n                               progress_path=os.environ.get("H3_DOWNLOAD_PROGRESS_PATH"))\\n    server = make_server("127.0.0.1", args.port, key, container.dispatch)\\n    http = threading.Thread(target=server.serve_forever, name="h3-model-control", daemon=True)\\n    http.start()\\n    stop_pending = False\\n    stop_applied = False\\n\\n    def request_stop(*_):\\n        # Python signal handlers may interrupt this thread while it holds an\\n        # RLock or is writing an event. Only record intent here; even Event.set\\n        # takes a lock and must not run in a reentrant signal callback.\\n        nonlocal stop_pending\\n        stop_pending = True\\n\\n    def apply_stop():\\n        nonlocal stop_applied\\n        if stop_applied:\\n            return\\n        stop_applied = True\\n        try:\\n            container.stop()\\n        except BaseException:\\n            LOG.exception("Could not record model stop")\\n            with container._lock:\\n                container._stop_requested = container._draining = True\\n                container._loaded = False\\n\\n    signal.signal(signal.SIGTERM, request_stop)\\n    signal.signal(signal.SIGINT, request_stop)\\n    try:\\n        container.start()\\n        while True:\\n            if args.parent_pid and not parent_is_alive(args.parent_pid, owner_created_at):\\n                # An ownerless CUDA worker must not outlive its resource lease.\\n                apply_stop()\\n                container.join(2)\\n                server.shutdown()\\n                server.server_close()\\n                os._exit(70)\\n            if stop_pending:\\n                apply_stop()\\n            if container._stop_requested:\\n                if container.join(.2):\\n                    break\\n            else:\\n                # Failed loading leaves the private controller available for diagnosis.\\n                time.sleep(.2)\\n    finally:\\n        apply_stop()\\n        container.join(2)\\n        server.shutdown()\\n        server.server_close()\\n\\n\\nif __name__ == "__main__":\\n    logging.basicConfig(level=logging.INFO)\\n    main()\\n\', \'tunnel.py\': \'"""Prepare an explicit temporary tunnel independently of model readiness."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport re\\nimport signal\\nimport shutil\\nimport subprocess\\nimport sys\\nimport time\\nimport threading\\n\\n\\ndef watch_owner(parent_pid):\\n    """Exit through the signal handler so the connector is also reclaimed."""\\n    if not parent_pid:\\n        return\\n    import psutil\\n    try:\\n        owner = psutil.Process(int(parent_pid))\\n        created = owner.create_time()\\n    except psutil.Error:\\n        raise RuntimeError("Tunnel owner is already unavailable")\\n\\n    def observe():\\n        while True:\\n            time.sleep(1)\\n            try:\\n                alive = (owner.is_running() and owner.create_time() == created\\n                         and owner.status() != psutil.STATUS_ZOMBIE)\\n            except psutil.Error:\\n                alive = False\\n            if not alive:\\n                if os.name == "nt":\\n                    import _thread\\n                    _thread.interrupt_main()\\n                else:\\n                    os.kill(os.getpid(), signal.SIGTERM)\\n                return\\n    threading.Thread(target=observe, name="tunnel-owner-watch", daemon=True).start()\\nimport urllib.request\\n\\n\\ndef ready(port: int) -> bool:\\n    try:\\n        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as response:\\n            return json.load(response).get("ready") is True\\n    except Exception:\\n        return False\\n\\n\\ndef cloudflared_binary(install: bool = False) -> str:\\n    found = shutil.which("cloudflared")\\n    if found:\\n        return found\\n    cached = Path(__file__).resolve().parent / ".cache" / "cloudflared"\\n    if cached.is_file():\\n        return str(cached)\\n    if not install:\\n        raise RuntimeError("cloudflared is not installed. Use --install to fetch an official release with SHA-256 verification.")\\n    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "amd64"}:\\n        raise RuntimeError("Automatic cloudflared install supports Linux x86_64 only; otherwise install it from Cloudflare first.")\\n    request = urllib.request.Request(\\n        "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",\\n        headers={"User-Agent": "minimax-h3-ref-api", "Accept": "application/vnd.github+json"},\\n    )\\n    with urllib.request.urlopen(request, timeout=30) as response:\\n        release = json.load(response)\\n    asset = next(a for a in release["assets"] if a["name"] == "cloudflared-linux-amd64")\\n    digest = asset.get("digest", "")\\n    if not digest.startswith("sha256:"):\\n        raise RuntimeError("Official release metadata lacks a SHA-256 digest; install cloudflared manually from Cloudflare.")\\n    url = asset["browser_download_url"]\\n    if not url.startswith("https://github.com/cloudflare/cloudflared/releases/download/"):\\n        raise RuntimeError("Unexpected cloudflared release origin")\\n    cached.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = cached.with_suffix(".download")\\n    hasher = hashlib.sha256()\\n    try:\\n        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as output:\\n            while chunk := response.read(1024 * 1024):\\n                hasher.update(chunk)\\n                output.write(chunk)\\n        if hasher.hexdigest() != digest.split(":", 1)[1]:\\n            raise RuntimeError("cloudflared SHA-256 mismatch; binary was not installed")\\n        temporary.chmod(0o700)\\n        temporary.replace(cached)\\n    finally:\\n        temporary.unlink(missing_ok=True)\\n    print(f"Installed official cloudflared {release[\\\'tag_name\\\']} (SHA-256 verified).", flush=True)\\n    return str(cached)\\n\\n\\ndef run_tunnel(port: int = 8000, install: bool = False, *, status_file: Path | None = None,\\n               instance_id: str | None = None) -> None:\\n    """Register the transport; a candidate URL never implies application readiness."""\\n    if status_file and (not instance_id or not re.fullmatch(r"[a-f0-9]{32}", instance_id)):\\n        raise ValueError("Managed tunnel status requires its owner\\\'s 32-character instance ID")\\n    state = {"instance_id": instance_id, "phase": "preparing", "url": None,\\n             "connected": False, "pid": None, "error": None}\\n\\n    def publish(**changes):\\n        state.update(changes, observed_at=time.time())\\n        if status_file:\\n            path = Path(status_file)\\n            path.parent.mkdir(parents=True, exist_ok=True)\\n            temporary = path.with_suffix(".tmp")\\n            temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")\\n            temporary.replace(path)\\n\\n    process = None\\n    registered = set()\\n    publish()\\n    try:\\n        # A loading origin may not be listening yet. Transport installation and\\n        # registration do not depend on its /health response.\\n        binary = cloudflared_binary(install)\\n        process = subprocess.Popen(\\n            [binary, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],\\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,\\n        )\\n        publish(pid=process.pid)\\n        for line in process.stdout:\\n            match = re.search(r"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com", line)\\n            if match and not state["url"]:\\n                publish(url=match.group(0))\\n                print(f"PUBLIC_API_URL={match.group(0)}", flush=True)\\n                print("Candidate URL allocated; application readiness still requires the live join.", flush=True)\\n            connection = re.search(r"connIndex=(\\\\d+)", line)\\n            connection_id = connection.group(1) if connection else "default"\\n            if "Registered tunnel connection" in line:\\n                registered.add(connection_id)\\n                publish(phase="registered", connected=True)\\n            elif any(marker in line for marker in (\\n                "Unregistered tunnel connection", "Connection terminated",\\n                "failed to serve tunnel connection", "Serve tunnel error",\\n            )):\\n                if connection:\\n                    registered.discard(connection_id)\\n                else:\\n                    registered.clear()  # No connection identity: revoke conservatively.\\n                publish(phase="registered" if registered else "reconnecting", connected=bool(registered))\\n            print(line, end="", flush=True)\\n        if process.wait() != 0:\\n            raise RuntimeError("cloudflared stopped; see the tunnel log above")\\n    except KeyboardInterrupt:\\n        publish(phase="stopping", connected=False)\\n        raise\\n    except BaseException as error:\\n        publish(phase="failed", connected=False, error=type(error).__name__)\\n        raise\\n    finally:\\n        if process is not None and process.poll() is None:\\n            process.terminate()\\n            try:\\n                process.wait(timeout=10)\\n            except subprocess.TimeoutExpired:\\n                process.kill()\\n                process.wait()\\n        publish(phase="failed" if state["phase"] == "failed" else "stopped", connected=False)\\n\\n\\nif __name__ == "__main__":\\n    signal.signal(signal.SIGTERM, lambda *_: (_ for _ in ()).throw(KeyboardInterrupt()))\\n    watch_owner(os.environ.get("H3_PARENT_PID"))\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--port", type=int, default=8000)\\n    parser.add_argument("--install", action="store_true", help="Download official cloudflared if missing, checking GitHub release SHA-256")\\n    parser.add_argument("--status-file", type=Path, help="Atomic transport state checkpoint owned by the startup supervisor")\\n    parser.add_argument("--instance-id", help="Owning startup session ID; never an API credential")\\n    args = parser.parse_args()\\n    try:\\n        run_tunnel(args.port, args.install, status_file=args.status_file, instance_id=args.instance_id)\\n    except KeyboardInterrupt:\\n        print("Tunnel stopped. Local GPU server remains running until you stop it.")\\n\', \'client.py\': \'"""Submit ordered references to the ref2va API, poll, and download the MP4."""\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom contextlib import ExitStack\\nimport json\\nimport mimetypes\\nimport os\\nfrom pathlib import Path\\nimport time\\nimport requests\\n\\n\\nclass ReferenceAction(argparse.Action):\\n    def __call__(self, parser, namespace, value, option_string=None):\\n        items = list(getattr(namespace, self.dest, None) or [])\\n        items.append((option_string.removeprefix("--"), Path(value)))\\n        setattr(namespace, self.dest, items)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("url", help="API base URL, e.g. https://example.trycloudflare.com")\\n    parser.add_argument("--prompt", required=True)\\n    for kind in ("image", "video", "audio"):\\n        parser.add_argument("--" + kind, dest="references", action=ReferenceAction, metavar="FILE")\\n    parser.add_argument("--output", type=Path, default=Path("result.mp4"))\\n    parser.add_argument("--seconds", type=float, default=5)\\n    parser.add_argument("--width", type=int, default=960)\\n    parser.add_argument("--height", type=int, default=544)\\n    parser.add_argument("--steps", type=int, default=50)\\n    parser.add_argument("--seed", type=int, default=42)\\n    parser.add_argument("--timeout", type=int, default=7200, help="Maximum poll time in seconds")\\n    args = parser.parse_args()\\n    key = os.environ.get("H3_API_KEY")\\n    if not key:\\n        parser.error("Set H3_API_KEY in the caller\\\'s environment first")\\n    if not args.references:\\n        parser.error("At least one --image or --video reference is required")\\n    if args.output.exists():\\n        parser.error("Output file already exists; choose another --output path")\\n    base = args.url.rstrip("/")\\n    if not (base.startswith("https://") or base.startswith("http://127.0.0.1:")):\\n        parser.error("Use HTTPS for public endpoints, or HTTP on 127.0.0.1 for local testing")\\n    headers = {"Authorization": "Bearer " + key}\\n    with ExitStack() as stack:\\n        files = [("files", (path.name, stack.enter_context(path.open("rb")), mimetypes.guess_type(path.name)[0] or "application/octet-stream"))\\n                 for _, path in args.references]\\n        response = requests.post(\\n            base + "/v1/jobs", headers=headers, files=files,\\n            data={"kinds": json.dumps([kind for kind, _ in args.references]), "prompt": args.prompt,\\n                  "seconds": args.seconds, "width": args.width, "height": args.height,\\n                  "steps": args.steps, "seed": args.seed}, timeout=120,\\n        )\\n    response.raise_for_status()\\n    job_id = response.json()["id"]\\n    print("Job:", job_id, flush=True)\\n    deadline = time.monotonic() + args.timeout\\n    while True:\\n        response = requests.get(base + f"/v1/jobs/{job_id}", headers=headers, timeout=30)\\n        response.raise_for_status()\\n        job = response.json()\\n        print(job["status"], flush=True)\\n        if job["status"] == "failed":\\n            raise RuntimeError(json.dumps(job, ensure_ascii=False))\\n        if job["status"] == "succeeded":\\n            break\\n        if time.monotonic() >= deadline:\\n            raise TimeoutError(f"Polling timed out. Job {job_id} may still be running; query it later.")\\n        time.sleep(min(30, max(0, deadline - time.monotonic())))\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    partial = args.output.with_suffix(args.output.suffix + ".part")\\n    created_partial = False\\n    try:\\n        with requests.get(base + f"/v1/jobs/{job_id}/file", headers=headers, timeout=120, stream=True) as response:\\n            response.raise_for_status()\\n            with partial.open("xb") as output:\\n                created_partial = True\\n                for chunk in response.iter_content(1024 * 1024):\\n                    output.write(chunk)\\n        partial.replace(args.output)\\n    finally:\\n        if created_partial:\\n            partial.unlink(missing_ok=True)\\n    print("Saved:", args.output.resolve())\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'README.md\': \'# MiniMax-H3 ref2va 参考运行时 · API 0.4.0\\n\\n这里保存已验收的专用实现：总管理、API 后台、大模型容器三个受管进程，独立的可选公网隧道与 Colab 控制台。模型容器是模型运行时进程，不把 GPU 硬件本身称为容器。\\n\\n- 当前模型：MiniMax-H3，仅 ref2va；至少图片或视频参考，可附加音频。\\n- A100 原生 BF16 主干与 LoRA，保留官方 FP32 例外，不启用权重量化。\\n- 任务分为 CPU 预处理、独占 GPU 推理、CPU 编码校验；按真实依赖及资源预算调度。\\n- 队列无固定条数上限，可指定 ID 取消；资源不足、未知提交和运行中取消各自有真实状态。\\n- API 可先于模型就绪接收排队。生成、产物完整校验、任务清理、服务进程退出和云平台释放分别确认。\\n- 每任务控制 attention_backend、acceleration_lora、steps；Turbo 要显式 steps=5，对应4次模型评估。预加载不等于当前任务启用。\\n\\n仓库内的[运行说明](../../docs/operations.md)、[架构](../../docs/architecture.md)、[验证边界](../../docs/validation.md)提供当前入口。此 README 进入自包含 notebook 时，仓库相对链接仅用于源码导航；云端实际操作使用面板与日志。\\n\\n```bash\\npython run.py --help\\npython smoke_test.py --help\\n```\\n\\n历史 A100 实测不是当前机器的一次新运行。源码来源与精度/配置/任务/释放证据见仓库 evidence。真实生成验收每次必须是全新内容；未知结果按原 ID 对账，不自动重新生成。\\n\', \'notebook_ui.py\': \'"""Read-only dashboard and separate diagnostics for the notebook control panel."""\\nfrom __future__ import annotations\\n\\nimport html\\nimport json\\nfrom collections import deque\\nfrom datetime import datetime, timedelta, timezone\\nfrom pathlib import Path\\nimport re\\nimport subprocess\\nimport time\\n\\nSTYLE = """\\n<style>\\n.h3-card{--bg:#f8fafc;--ink:#172033;--muted:#627086;--line:#dbe3ec;--accent:#187c72;box-sizing:border-box;background:var(--bg);color:var(--ink);border:1px solid var(--line);border-left:4px solid var(--accent);border-radius:12px;padding:17px 20px;margin:8px 0;font:14px/1.55 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;max-width:980px}\\n.h3-card *{box-sizing:border-box}.h3-card[data-tone="error"]{--accent:#ba4242}.h3-card[data-tone="idle"]{--accent:#7b8797}.h3-card[data-tone="running"]{--accent:#bd7c16}\\n.h3-head{display:flex;gap:12px;justify-content:space-between;align-items:center;flex-wrap:wrap}.h3-title{font-size:17px;font-weight:650;letter-spacing:-.2px}.h3-badge{font-size:11px;font-weight:650;letter-spacing:.3px;color:var(--accent);border:1px solid var(--line);padding:3px 9px;border-radius:99px;white-space:nowrap}.h3-note{color:var(--muted);margin:5px 0 0}\\n.h3-fields{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px 18px;margin-top:13px}.h3-label{display:block;color:var(--muted);font-size:11px;letter-spacing:.25px;margin-bottom:2px}.h3-value{font-size:14px;font-weight:550;overflow-wrap:anywhere}.h3-card details{margin-top:12px;border-top:1px solid var(--line);padding-top:10px}.h3-card summary{cursor:pointer;color:var(--muted);font-size:12px}.h3-card pre{white-space:pre-wrap;overflow-wrap:anywhere;font:12px/1.55 ui-monospace,SFMono-Regular,Consolas,monospace;margin:9px 0 0;max-height:280px;overflow:auto}.h3-card a{color:var(--accent);text-decoration:none}.h3-card a:hover{text-decoration:underline}\\n.h3-board{border-left-width:1px;padding:22px 24px}.h3-board h2{font-size:23px;line-height:1.3;margin:4px 0 8px}.h3-kicker{font-size:12px;color:var(--muted)}.h3-next{margin:12px 0 21px;padding:11px 14px;border-radius:8px;background:color-mix(in srgb,var(--accent) 8%,var(--bg));font-size:15px}.h3-modules{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:12px 0 19px}.h3-module{padding:12px 14px;border:1px solid var(--line);border-radius:8px}.h3-module strong{font-weight:650;display:block;font-size:16px;margin-top:4px}.h3-subtle{color:var(--muted);font-size:12px;margin-top:4px}.h3-summary{display:flex;flex-wrap:wrap;gap:8px 22px;margin:12px 0}.h3-activity{border-top:1px solid var(--line);margin-top:18px;padding-top:13px}.h3-activity .h3-title{font-size:14px}.h3-activity .h3-fields{margin-top:8px;gap:8px 18px}.h3-stamp{font-size:11px;color:var(--muted);margin-top:17px}.h3-logs{border-left-width:1px}.h3-logs pre{max-height:360px;margin:14px 0 0}.h3-logs pre:focus{outline:2px solid var(--accent);outline-offset:3px}\\n@media(max-width:620px){.h3-board{padding:17px 15px}.h3-modules{grid-template-columns:1fr;gap:8px}.h3-module{padding:9px 12px;display:grid;grid-template-columns:95px 1fr;align-items:center}.h3-module strong{margin:0}.h3-module .h3-subtle{grid-column:2}.h3-board h2{font-size:21px}.h3-fields{grid-template-columns:1fr}}\\n@media(prefers-color-scheme:dark){.h3-card{--bg:#17202d;--ink:#e4ebf4;--muted:#a5b3c6;--line:#344358;--accent:#67cdbb}.h3-card[data-tone="error"]{--accent:#ff9696}.h3-card[data-tone="running"]{--accent:#e4ba6d}.h3-card[data-tone="idle"]{--accent:#b5bfcb}}\\n</style>\\n"""\\n\\n_dashboard = None\\n_diagnostics = deque(maxlen=20)\\n_known_secrets = ()\\n_process_observer = None\\n_DIAGNOSTIC_LIMIT = 12000\\n\\n\\ndef redact_text(text, secrets=()):\\n    """Redact known credentials and common authorization/key assignments before display."""\\n    text = str(text)\\n    for secret in sorted({str(s) for s in (*_known_secrets, *secrets) if s}, key=len, reverse=True):\\n        text = text.replace(secret, \\\'[已隐藏]\\\')\\n    text = re.sub(r\\\'(?i)(\\\\bBearer\\\\s+)[^\\\\s\\\\"\\\\\\\'<>;,]+\\\', r\\\'\\\\1[已隐藏]\\\', text)\\n    text = re.sub(r\\\'\\\'\\\'(?ix)(["\\\']?(?:H3_API_KEY|API_KEY|api[-_]?key|access_token|authorization)["\\\']?\\\\s*[:=]\\\\s*)(["\\\'])(.*?)(\\\\2)\\\'\\\'\\\',\\n                  lambda match: match.group(1) + match.group(2) + \\\'[已隐藏]\\\' + match.group(2), text)\\n    text = re.sub(r\\\'(?i)(\\\\b(?:H3_API_KEY|API_KEY|api[-_]?key|access_token)\\\\s*[:=]\\\\s*)[^\\\\s\\\\"\\\\\\\'<>;,]+\\\', r\\\'\\\\1[已隐藏]\\\', text)\\n    return text\\n\\n\\ndef _remember(title, text):\\n    if text:\\n        _diagnostics.append(redact_text(str(title) + \\\'\\\\n\\\' + str(text))[-_DIAGNOSTIC_LIMIT:])\\n\\n\\ndef last_diagnostics():\\n    return \\\'\\\\n\\\\n\\\'.join(_diagnostics)[-_DIAGNOSTIC_LIMIT:]\\n\\n\\ndef diagnostics_html(title, text, *, secrets=()):\\n    """A bounded, keyboard-scrollable log view; never embedded in dashboard cards."""\\n    value = redact_text(text, secrets=secrets)\\n    truncated = len(value) > _DIAGNOSTIC_LIMIT\\n    value = value[-_DIAGNOSTIC_LIMIT:]\\n    note = \\\'仅显示最后 12,000 个字符；完整记录保留在日志文件。\\\' if truncated else \\\'只读日志 · 敏感凭据已隐藏\\\'\\n    return (STYLE + \\\'<section class="h3-card h3-logs" aria-label="日志"><div class="h3-title">\\\'\\n            + html.escape(redact_text(title), quote=True) + \\\'</div><p class="h3-note">\\\' + note\\n            + \\\'</p><pre tabindex="0" aria-label="日志内容">\\\' + html.escape(value or \\\'暂无记录。\\\', quote=True)\\n            + \\\'</pre></section>\\\')\\n\\n\\ndef _short(value, maximum=160):\\n    text = redact_text(value).replace(\\\'\\\\n\\\', \\\' \\\').replace(\\\'\\\\r\\\', \\\' \\\')\\n    return text if len(text) <= maximum else text[:maximum - 1] + \\\'…\\\'\\n\\n\\ndef _activity(title, kwargs):\\n    _remember(title, kwargs.get(\\\'details\\\'))\\n    fields = [(label, value) for label, value in kwargs.get(\\\'fields\\\', ())\\n              if not any(word in str(label) for word in (\\\'日志\\\', \\\'路径\\\', \\\'密钥\\\', \\\'当前记录\\\'))]\\n    return {\\\'title\\\': _short(title, 70), \\\'badge\\\': _short(kwargs.get(\\\'badge\\\', \\\'\\\'), 35),\\n            \\\'note\\\': _short(kwargs.get(\\\'note\\\', \\\'\\\')), \\\'tone\\\': kwargs.get(\\\'tone\\\', \\\'idle\\\'),\\n            \\\'fields\\\': [(str(k), _short(v, 95)) for k, v in fields[:4]]}\\n\\n\\ndef card_html(title, *, badge="", note="", fields=(), tone="idle", details=None, details_label="查看详情"):\\n    escape = lambda value: html.escape(_short(value), quote=True)\\n    if tone not in {"idle", "running", "success", "error"}:\\n        tone = "idle"\\n    values = "".join(\\n        \\\'<div><span class="h3-label">\\\' + escape(label) + \\\'</span><span class="h3-value">\\\'\\n        + escape(value) + \\\'</span></div>\\\' for label, value in fields\\n    )\\n    return (STYLE + \\\'<section class="h3-card" data-tone="\\\' + tone + \\\'"><div class="h3-head">\\\'\\n            + \\\'<div class="h3-title">\\\' + escape(title) + \\\'</div><span class="h3-badge">\\\'\\n            + escape(badge) + \\\'</span></div>\\\' + (\\\'<p class="h3-note">\\\' + escape(note) + \\\'</p>\\\' if note else "")\\n            + (\\\'<div class="h3-fields">\\\' + values + \\\'</div>\\\' if fields else "") + "</section>")\\n\\n\\ndef render_dashboard(context, activity=None):\\n    """Render supplied observations only. This pure function performs no host I/O."""\\n    escape = lambda value: html.escape(_short(value), quote=True)\\n    state = context.get(\\\'state\\\') or {}\\n    stale = bool(state.get(\\\'observation_stale\\\'))\\n    failed_read = bool(context.get(\\\'observation_error\\\'))\\n    phase = state.get(\\\'phase\\\')\\n    if context.get(\\\'placeholder\\\'):\\n        headline, next_step, tone = \\\'等待首次读取\\\', \\\'运行「刷新看板」查看本次状态；此处是界面占位，没有实时观察。\\\', \\\'idle\\\'\\n    elif failed_read:\\n        headline, next_step, tone = \\\'状态读取失败\\\', \\\'重新刷新看板；如仍失败，在日志区查看原因。\\\', \\\'error\\\'\\n    elif stale:\\n        headline, next_step, tone = \\\'状态需要刷新\\\', \\\'当前观察已过期；刷新后再判断可用能力。\\\', \\\'running\\\'\\n    elif phase == \\\'stopped\\\' and state.get(\\\'cleanup_complete\\\'):\\n        headline, next_step, tone = \\\'本次服务已停止\\\', \\\'确认结果已保存后，可在控制面板释放 Colab 实例。\\\', \\\'idle\\\'\\n    elif phase == \\\'stopping\\\':\\n        headline, next_step, tone = \\\'正在停止与回收\\\', \\\'等待受管进程退出并确认清理完成；此时不要重复启动。\\\', \\\'running\\\'\\n    elif state.get(\\\'error\\\') or phase in {\\\'failed\\\', \\\'degraded\\\'}:\\n        headline, next_step, tone = \\\'有模块需要处理\\\', \\\'先看下方可用能力；具体原因在日志区，控制面板仍可查询或停止。\\\', \\\'error\\\'\\n    elif state.get(\\\'draining\\\'):\\n        headline, next_step, tone = \\\'已暂停接收新任务\\\', \\\'已接收的队列继续执行；可恢复接收，也可停止服务。\\\', \\\'idle\\\'\\n    elif state.get(\\\'ready\\\'):\\n        headline, next_step, tone = \\\'可以生成\\\', \\\'在控制面板提交新内容，或按任务 ID 查询已有任务。\\\', \\\'success\\\'\\n    elif state.get(\\\'api_ready\\\'):\\n        accepting = (state.get(\\\'queue\\\') or {}).get(\\\'admission_open\\\') is True\\n        headline, next_step, tone = (\\\'API 可用 · 生成尚未就绪\\\',\\n            (\\\'可先查询或排队；\\\' if accepting else \\\'可先查询；\\\') + \\\'模型、公网的当前能力分别见下方。\\\', \\\'running\\\')\\n    elif state:\\n        headline, next_step, tone = \\\'服务正在准备\\\', \\\'刷新看板观察进展；不要重复启动。\\\', \\\'running\\\'\\n    elif not context.get(\\\'preflight_ok\\\'):\\n        headline, next_step, tone = \\\'尚未检查资源\\\', \\\'在控制面板先运行「资源与驻留方案」。\\\', \\\'idle\\\'\\n    elif not context.get(\\\'install_ok\\\'):\\n        headline, next_step, tone = \\\'资源检查通过\\\', \\\'在控制面板运行「安装与检查」。\\\', \\\'idle\\\'\\n    else:\\n        headline, next_step, tone = \\\'准备工作已完成\\\', \\\'在控制面板选择是否开放公网，再启动服务。\\\', \\\'idle\\\'\\n\\n    def module(name, alive_key, ready_key=None):\\n        alive, ready = state.get(alive_key), state.get(ready_key) if ready_key else None\\n        if not state:\\n            status, detail = (\\\'尚未读取\\\' if context.get(\\\'placeholder\\\') else \\\'未启动\\\'), \\\'尚无运行观察\\\'\\n        elif ready is True:\\n            status, detail = \\\'可用\\\', \\\'进程运行中\\\' if alive is True else \\\'存活状态未确认\\\'\\n        elif alive is True:\\n            status, detail = (\\\'运行中\\\', \\\'调度与控制\\\') if ready_key is None else (\\\'准备中\\\', \\\'进程运行，能力未就绪\\\')\\n        elif alive is False:\\n            status, detail = \\\'已停止\\\', \\\'进程未运行\\\'\\n        else:\\n            status, detail = \\\'未确认\\\', \\\'缺少当前观察\\\'\\n        if stale or failed_read:\\n            status, detail = \\\'待刷新\\\', \\\'不据旧状态判断就绪\\\'\\n        return (\\\'<div class="h3-module"><span class="h3-label">\\\' + escape(name) + \\\'</span><strong>\\\'\\n                + escape(status) + \\\'</strong><div class="h3-subtle">\\\' + escape(detail) + \\\'</div></div>\\\')\\n\\n    if stale or failed_read:\\n        public = \\\'待刷新\\\'\\n    elif not state:\\n        public = \\\'尚未观察\\\'\\n    elif not state.get(\\\'public_requested\\\'):\\n        public = \\\'未启用\\\'\\n    elif state.get(\\\'public_api_ready\\\'):\\n        public = \\\'API 探测通过\\\'\\n    elif state.get(\\\'tunnel_connected\\\'):\\n        public = \\\'隧道已连接 · API 待验证\\\'\\n    else:\\n        public = \\\'准备中\\\'\\n    queue = state.get(\\\'queue\\\') or {}\\n    summary = [(\\\'公网\\\', public)]\\n    if queue and not stale and not failed_read:\\n        summary += [(\\\'任务\\\', f"排队 {queue.get(\\\'queued\\\', \\\'—\\\')} · 执行 {queue.get(\\\'running\\\', \\\'—\\\')}")]\\n        if queue.get(\\\'admission_open\\\') is False:\\n            summary.append((\\\'接收任务\\\', \\\'已暂停\\\'))\\n    resources = context.get(\\\'resource_report\\\') or {}\\n    if resources:\\n        summary += [(\\\'GPU\\\', resources.get(\\\'gpu\\\', \\\'未记录\\\')),\\n                    (\\\'可用内存\\\', str(resources.get(\\\'available_ram_gib\\\', \\\'—\\\')) + \\\' GiB（预检）\\\')]\\n    resource_text = \\\'\\\'.join(\\\'<span><span class="h3-label">\\\' + escape(k) + \\\'</span>\\\' + escape(v) + \\\'</span>\\\' for k, v in summary)\\n    stamp = state.get(\\\'observed_at\\\')\\n    if isinstance(stamp, (int, float)):\\n        try:\\n            stamp = datetime.fromtimestamp(stamp, timezone(timedelta(hours=8))).strftime(\\\'%m-%d %H:%M:%S 北京时间\\\')\\n        except (OverflowError, OSError, ValueError):\\n            stamp = \\\'未记录\\\'\\n    else:\\n        stamp = \\\'尚无服务快照\\\'\\n    body = (STYLE + \\\'<section class="h3-card h3-board" data-tone="\\\' + tone + \\\'" aria-label="运行看板">\\\'\\n            \\\'<div class="h3-kicker">当前状态</div><h2>\\\' + escape(headline) + \\\'</h2>\\\'\\n            \\\'<div class="h3-next"><b>下一步</b>\\u3000\\\' + escape(next_step) + \\\'</div><div class="h3-modules">\\\'\\n            + module(\\\'总管理\\\', \\\'manager_alive\\\') + module(\\\'API 后台\\\', \\\'server_alive\\\', \\\'api_ready\\\')\\n            + module(\\\'大模型容器\\\', \\\'model_alive\\\', \\\'model_ready\\\') + \\\'</div>\\\'\\n            \\\'<div class="h3-summary">\\\' + resource_text + \\\'</div>\\\')\\n    if activity:\\n        activity_tone = activity.get(\\\'tone\\\', \\\'idle\\\')\\n        if activity_tone not in {\\\'idle\\\', \\\'running\\\', \\\'success\\\', \\\'error\\\'}:\\n            activity_tone = \\\'idle\\\'\\n        body += (\\\'<div class="h3-activity" data-tone="\\\' + activity_tone + \\\'"><div class="h3-label">最近操作</div>\\\'\\n                 \\\'<div class="h3-head"><span class="h3-title">\\\' + escape(activity.get(\\\'title\\\', \\\'\\\'))\\n                 + \\\'</span><span class="h3-badge">\\\' + escape(activity.get(\\\'badge\\\', \\\'\\\')) + \\\'</span></div>\\\')\\n        if activity.get(\\\'note\\\'):\\n            body += \\\'<div class="h3-note">\\\' + escape(activity[\\\'note\\\']) + \\\'</div>\\\'\\n        fields = activity.get(\\\'fields\\\') or ()\\n        if fields:\\n            body += \\\'<div class="h3-fields">\\\' + \\\'\\\'.join(\\\'<div><span class="h3-label">\\\' + escape(k)\\n                + \\\'</span><span class="h3-value">\\\' + escape(v) + \\\'</span></div>\\\' for k, v in fields[:4]) + \\\'</div>\\\'\\n        body += \\\'</div>\\\'\\n    return body + \\\'<div class="h3-stamp">服务观察：\\\' + escape(stamp) + \\\' · 按需刷新，不执行控制命令</div></section>\\\'\\n\\n\\nclass Dashboard:\\n    def __init__(self, context_reader):\\n        self.context_reader = context_reader\\n        self.activity = None\\n        self._handle = None\\n\\n    def refresh(self):\\n        from IPython.display import HTML, display\\n        global _known_secrets\\n        context = dict(self.context_reader())\\n        host = context.pop(\\\'host\\\', None)\\n        if host is not None:\\n            key = getattr(host, \\\'api_key\\\', None)\\n            if key:\\n                _known_secrets = tuple(set((*_known_secrets[-3:], str(key))))\\n            try:\\n                context[\\\'state\\\'] = host.snapshot()\\n            except Exception as error:\\n                context[\\\'state\\\'] = {}\\n                context[\\\'observation_error\\\'] = True\\n                _remember(\\\'看板状态读取失败\\\', str(error))\\n        rendered = HTML(render_dashboard(context, self.activity))\\n        if self._handle is None:\\n            self._handle = display(rendered, display_id=\\\'h3-dashboard\\\')\\n        else:\\n            self._handle.update(rendered)\\n        return context\\n\\n\\ndef mount_dashboard(context_reader):\\n    global _dashboard\\n    _dashboard = Dashboard(context_reader)\\n    _dashboard.refresh()\\n    return _dashboard\\n\\n\\nclass StatusCard:\\n    def __init__(self, title, **kwargs):\\n        self._handle = None\\n        self.update(title, **kwargs)\\n\\n    def update(self, title, **kwargs):\\n        from IPython.display import HTML, display\\n        activity = _activity(title, kwargs)\\n        if _dashboard is not None:\\n            _dashboard.activity = activity\\n            _dashboard.refresh()\\n        elif self._handle is None:\\n            self._handle = display(HTML(card_html(**activity)), display_id=True)\\n        else:\\n            self._handle.update(HTML(card_html(**activity)))\\n\\n\\ndef elapsed_text(seconds):\\n    seconds = max(0, int(seconds))\\n    return f"{seconds // 60}分{seconds % 60:02d}秒" if seconds >= 60 else f"{seconds}秒"\\n\\n\\ndef log_tail(path, limit=6000):\\n    path = Path(path)\\n    if not path.is_file():\\n        return ""\\n    with path.open("rb") as source:\\n        source.seek(max(0, path.stat().st_size - limit))\\n        text = source.read().decode("utf-8", errors="replace")\\n    return re.sub(r"\\\\x1b\\\\[[0-?]*[ -/]*[@-~]", "", text).replace("\\\\r", "\\\\n")\\n\\n\\ndef latest_stage(path):\\n    lines = [line.strip() for line in log_tail(path).splitlines() if line.strip()]\\n    for line in reversed(lines):\\n        if "BF16 staged:" in line:\\n            return line.split("BF16 staged:", 1)[1].strip()[:145]\\n        if "status=" in line or "API_SMOKE_TEST_PASSED" in line:\\n            return line[:145]\\n    return lines[-1][:145] if lines else "进程已启动，等待首条日志"\\n\\n\\ndef run_logged(command, log_path, *, title, cwd=None, env=None, card=None, status_reader=None):\\n    """Run a real child process; persist logs and update a bounded status card."""\\n    path = Path(log_path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    status = card or StatusCard(title, badge="执行中", tone="running", note="完整输出写入日志。")\\n    started = time.monotonic()\\n    with path.open("a", encoding="utf-8") as output:\\n        output.write("\\\\n--- " + title + " ---\\\\n")\\n        output.flush()\\n        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)\\n        observer = _process_observer\\n        if observer is not None:\\n            observer.process_started(process)\\n        try:\\n            while process.poll() is None:\\n                if observer is not None:\\n                    observer.process_progress(process)\\n                current = []\\n                if status_reader is not None:\\n                    try:\\n                        current = status_reader()\\n                    except (OSError, ValueError):\\n                        pass\\n                status.update(title, badge="执行中", tone="running", fields=\\n                    [("已用时间", elapsed_text(time.monotonic() - started))]\\n                    + current)\\n                time.sleep(2)\\n        except BaseException:\\n            process.terminate()\\n            try:\\n                process.wait(timeout=15)\\n            except subprocess.TimeoutExpired:\\n                process.kill()\\n                process.wait()\\n            status.update(title, badge="已中断", tone="error", note="操作进程已停止；已提交的远程任务可能仍在执行。", fields=[("日志", str(path))])\\n            raise\\n        finally:\\n            if observer is not None:\\n                observer.process_finished(process)\\n    if process.returncode:\\n        status.update(title, badge="未完成", tone="error", note="查看日志中的实际错误后再重试。",\\n                      fields=[("日志", str(path))], details=log_tail(path), details_label="展开最后的错误日志")\\n        raise RuntimeError(f"{title}退出码 {process.returncode}；完整日志：{path}")\\n    status.update(title, badge="完成", tone="success", fields=[\\n        ("已用时间", elapsed_text(time.monotonic() - started)), ("完整日志", str(path)),\\n    ])\\n    return process.returncode\\n\\n\\ndef json_probe(command, log_path, *, cwd=None, env=None, card=None):\\n    """Capture a short JSON-producing check without dumping full notebook output."""\\n    process = subprocess.Popen(command, cwd=cwd, env=env, text=True,\\n                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)\\n    observer = _process_observer\\n    try:\\n        if observer is not None:\\n            observer.process_started(process)\\n        stdout, stderr = process.communicate(timeout=180)\\n    except BaseException:\\n        if process.poll() is None:\\n            process.kill()\\n            process.communicate(timeout=10)\\n        raise\\n    finally:\\n        if observer is not None:\\n            observer.process_finished(process)\\n    with Path(log_path).open("a", encoding="utf-8") as log:\\n        log.write(stdout + "\\\\n" + stderr + "\\\\n")\\n    if process.returncode:\\n        if card is not None:\\n            card.update("依赖检查未通过", badge="需要处理", tone="error", note="检查日志中的版本或导入错误后再继续。",\\n                        fields=[("日志", str(log_path))], details=stderr[-4000:], details_label="展开检查错误")\\n        raise RuntimeError(f"依赖检查失败；请在日志区查看：{log_path}")\\n    return json.loads(stdout.strip().splitlines()[-1])\\n\', \'pipeline_backend.py\': \'"""CPU prepare -> exclusive GPU inference -> CPU encoding, with per-job ownership.\\n\\nReference/codec behavior follows Diffusers v0.43 references.py/export_utils.py.\\nIts encode_video has no thread-budget argument; this local PyAV writer sets\\ncodec thread_count explicitly and avoids stacking every output frame again.\\nResource numbers are conservative admission estimates, not calibrated optima.\\n"""\\nfrom __future__ import annotations\\n\\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\\nfrom contextlib import contextmanager, nullcontext\\nfrom dataclasses import dataclass\\nfrom fractions import Fraction\\nimport json\\nimport gc\\nimport math\\nimport os\\nfrom pathlib import Path\\n\\nfrom runtime_options import RuntimeSupport, measured_generation\\nfrom generation_options import requested_options, validate_generation_options, effective_options\\nfrom adapter_config import select_reference_setup, select_loaded_adapters\\n\\nMIB, GIB = 1024**2, 1024**3\\n\\n\\n@dataclass\\nclass Prepared:\\n    parameters: dict\\n    references: list\\n    normalized: list\\n    metrics: object\\n    cpu_threads: int = 1\\n    cleaned: bool = False\\n\\n\\n@dataclass\\nclass CPUResult:\\n    video: list\\n    audio: object\\n    sampling_rate: int\\n    metrics: object\\n    cpu_threads: int\\n    cpu_handoff_complete: bool = False\\n    cleaned: bool = False\\n\\n\\ndef _audio_from_container(av, container, stream):\\n    import numpy as np\\n    import torch\\n    stream.codec_context.thread_count = 1\\n    rate = int(stream.codec_context.sample_rate)\\n    if not 0 < rate <= 96000 or len(stream.layout.channels) > 2:\\n        raise ValueError("Reference soundtrack exceeds 96 kHz / stereo")\\n    resampler = av.AudioResampler(format="fltp", layout=stream.layout, rate=rate)\\n    chunks, samples = [], 0\\n    def append(frames):\\n        nonlocal samples\\n        for item in frames:\\n            value = item.to_ndarray()\\n            samples += value.shape[-1]\\n            if samples > math.ceil(15.1 * rate):\\n                raise ValueError("Decoded reference soundtrack exceeds 15 seconds")\\n            chunks.append(value)\\n    for frame in container.decode(stream):\\n        append(resampler.resample(frame))\\n    append(resampler.resample(None))\\n    if not chunks:\\n        raise ValueError("Reference soundtrack is empty")\\n    return torch.from_numpy(np.concatenate(chunks, axis=-1).astype(np.float32, copy=False)), rate\\n\\n\\ndef decode_reference(kind, path, classes):\\n    """CPU-only decoding with one native codec thread and decoded-size guards."""\\n    if kind == "image":\\n        return classes[kind].from_file(str(path))\\n    import av\\n    import numpy as np\\n    with av.open(str(path)) as container:\\n        if kind == "audio":\\n            audio, rate = _audio_from_container(av, container, container.streams.audio[0])\\n            return classes[kind](audio=audio, sample_rate=rate)\\n        stream = container.streams.video[0]\\n        stream.codec_context.thread_count = 1\\n        frames = []\\n        for frame in container.decode(stream):\\n            if len(frames) >= 362 or frame.width * frame.height > 960 * 544:\\n                raise ValueError("Normalized reference exceeds bounded decoded geometry")\\n            frames.append(frame.to_ndarray(format="rgb24"))\\n        if not frames:\\n            raise ValueError("Reference video is empty")\\n        fps = float(stream.average_rate or stream.guessed_rate)\\n        audio, rate = None, None\\n        if container.streams.audio:\\n            container.seek(0)\\n            audio, rate = _audio_from_container(av, container, container.streams.audio[0])\\n        return classes[kind](frames=np.stack(frames), fps=fps, audio=audio, sample_rate=rate)\\n\\n\\ndef cpu_media(video, audio, sampling_rate, torch):\\n    """Copy outputs into CPU-owned numpy buffers; never retain PipelineState."""\\n    import numpy as np\\n    def array(value):\\n        if isinstance(value, torch.Tensor):\\n            value = value.detach().to(device="cpu", copy=True)\\n            if value.is_floating_point():\\n                value = value.float()\\n            value = value.numpy()\\n        return np.array(value, copy=True)\\n    frames = []\\n    for frame in video:\\n        pixels = array(frame)\\n        if pixels.ndim != 3 or pixels.shape[-1] != 3:\\n            raise ValueError("Expected HWC RGB video output")\\n        if pixels.dtype != np.uint8:\\n            if not np.isfinite(pixels).all() or pixels.min() < 0 or pixels.max() > 1:\\n                raise ValueError("Float video output must be finite and normalized")\\n            pixels = (pixels * 255).round().astype(np.uint8)\\n        frames.append(pixels)\\n    waveform = array(audio).astype(np.float32, copy=False)\\n    if waveform.ndim != 2 or waveform.shape[0] != 2 or not np.isfinite(waveform).all():\\n        raise ValueError("Expected finite stereo audio output")\\n    if not frames or not 0 < int(sampling_rate) <= 96000:\\n        raise ValueError("Invalid output media")\\n    return frames, waveform, int(sampling_rate)\\n\\n\\ndef encode_cpu_media(result, output_path):\\n    """Same H.264/AAC formats as official exporter, explicitly bounded threads."""\\n    import av\\n    import numpy as np\\n    if not result.cpu_handoff_complete or result.cleaned:\\n        raise RuntimeError("Encoding requires a completed CPU ownership handoff")\\n    with av.open(str(output_path), mode="w") as container:\\n        stream = container.add_stream("libx264", rate=24)\\n        stream.width, stream.height = result.video[0].shape[1], result.video[0].shape[0]\\n        stream.pix_fmt = "yuv420p"\\n        stream.codec_context.thread_count = result.cpu_threads\\n        audio_stream = container.add_stream("aac", rate=result.sampling_rate)\\n        cc = audio_stream.codec_context\\n        cc.sample_rate, cc.layout = result.sampling_rate, "stereo"\\n        cc.time_base, cc.thread_count = Fraction(1, result.sampling_rate), 1\\n        for pixels in result.video:\\n            frame = av.VideoFrame.from_ndarray(pixels, format="rgb24")\\n            for packet in stream.encode(frame):\\n                container.mux(packet)\\n        for packet in stream.encode():\\n            container.mux(packet)\\n        samples = (np.clip(result.audio.T, -1, 1) * 32767).astype(np.int16)\\n        audio_frame = av.AudioFrame.from_ndarray(np.ascontiguousarray(samples).reshape(1, -1), format="s16", layout="stereo")\\n        audio_frame.sample_rate = result.sampling_rate\\n        resampler = av.AudioResampler(format=cc.format or "fltp", layout=cc.layout or "stereo", rate=cc.sample_rate)\\n        next_pts = 0\\n        for frame in resampler.resample(audio_frame) + resampler.resample(None):\\n            if frame.pts is None:\\n                frame.pts = next_pts\\n            next_pts += frame.samples\\n            frame.sample_rate = result.sampling_rate\\n            for packet in audio_stream.encode(frame):\\n                container.mux(packet)\\n        for packet in audio_stream.encode():\\n            container.mux(packet)\\n    if not Path(output_path).is_file() or not Path(output_path).stat().st_size:\\n        raise RuntimeError("Encoder produced no video")\\n\\n\\nclass PipelineBackendSupport(RuntimeSupport):\\n    pipeline_model_peak_ram_bytes = 144 * GIB\\n    resource_residency = "resident"\\n\\n    def validate_generation_request(self, *, steps, attention_backend=None, acceleration_lora=None):\\n        values = {"steps": steps}\\n        if attention_backend is not None:\\n            values["attention_backend"] = attention_backend\\n        if acceleration_lora is not None:\\n            values["acceleration_lora"] = acceleration_lora\\n        return validate_generation_options(requested_options(values), self.adapter_settings)\\n\\n    def _apply_generation_transformer(self, effective, *, audit=True):\\n        transformer = getattr(self.pipe, "transformer_ref", None)\\n        if transformer is None:\\n            raise RuntimeError("The generation transformer is unavailable")\\n        # Always set native explicitly, so forced Flash never leaks into auto.\\n        transformer.set_attention_backend(effective["attention_backend"])\\n        select_loaded_adapters(self.pipe, self.adapter_settings, effective["acceleration_lora"])\\n        if audit and self._active_metrics is not None:\\n            self._active_metrics.data["generation_options"]["effective"] = dict(effective)\\n\\n    @contextmanager\\n    def generation_context(self, prepared):\\n        """Mutate and restore shared model state only inside infer\\\'s GPU lease.\\n\\n        CPU preparation can overlap other jobs; it must not change adapters,\\n        attention processors, reference setup, or scheduler state. Staged\\n        residency applies the same choices after its transformer is loaded.\\n        """\\n        if getattr(self, "_active_generation_options", None) is not None:\\n            raise RuntimeError("Generation configuration already owns the GPU lease")\\n        requested = requested_options(prepared.parameters)\\n        validate_generation_options(requested, self.adapter_settings)\\n        effective = effective_options(requested, self.adapter_settings, self.runtime_options)\\n        prepared.metrics.data.setdefault("generation_options", {"requested": dict(requested), "effective": None})\\n        original = self.pipe.blocks.sub_blocks["before_encode"]\\n        self._active_generation_options = effective\\n        try:\\n            select_reference_setup(self.pipe, requested["acceleration_lora"])\\n            if getattr(self.pipe, "transformer_ref", None) is not None:\\n                self._apply_generation_transformer(effective)\\n            yield\\n        finally:\\n            try:\\n                self.pipe.blocks.sub_blocks["before_encode"] = original\\n                if getattr(self.pipe, "transformer_ref", None) is not None:\\n                    defaults = effective_options({"attention_backend": "auto",\\n                        "acceleration_lora": self.adapter_settings.acceleration, "steps": requested["steps"]},\\n                        self.adapter_settings, self.runtime_options)\\n                    self._apply_generation_transformer(defaults, audit=False)\\n            finally:\\n                self._active_generation_options = None\\n\\n    def configure_cpu_budget(self, gpu_threads):\\n        """Configure native CPU pools exactly once, before any model is loaded.\\n\\n        GPU work also occupies host CPU capacity. CPU preparation/encoding use\\n        bounded codecs and numpy copies, not multithreaded torch/BLAS operators.\\n        Environment defaults cover libraries imported later; the caller can\\n        audit again after load without changing any process-wide setting.\\n        """\\n        if isinstance(gpu_threads, bool) or not isinstance(gpu_threads, int) or gpu_threads < 1:\\n            raise ValueError("GPU host CPU reservation must be a positive integer")\\n        previous = getattr(self, "_native_cpu_requested", None)\\n        if previous is not None:\\n            if previous != gpu_threads:\\n                raise RuntimeError("Native CPU pools cannot be reconfigured during this model lifetime")\\n            return self.cpu_budget_snapshot()\\n        if self._loaded or self.pipe is not None:\\n            raise RuntimeError("Native CPU pools must be configured before model loading")\\n        # Keep the hardware gate before torch imports/downloads, including the\\n        # real Windows CPU integration\\\'s explicit Linux/NVIDIA rejection.\\n        from preflight import check_resources\\n        check_resources(residency=self.resource_residency)\\n        environment = {name: str(gpu_threads) for name in (\\n            "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",\\n            "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS")}\\n        environment.update(OMP_MAX_ACTIVE_LEVELS="1", OMP_DYNAMIC="FALSE", MKL_DYNAMIC="FALSE",\\n                           TOKENIZERS_PARALLELISM="false", RAYON_NUM_THREADS=str(gpu_threads))\\n        os.environ.update(environment)\\n        import numpy  # Load the CPU pools we use before applying/readback.\\n        import torch\\n        from threadpoolctl import threadpool_limits\\n        torch.set_num_threads(gpu_threads)\\n        interop_note = "set_before_load"\\n        try:\\n            torch.set_num_interop_threads(1)\\n        except RuntimeError:\\n            if torch.get_num_interop_threads() != 1:\\n                raise RuntimeError("Cannot enforce one torch interop thread before model loading") from None\\n            interop_note = "already_initialized_at_one"\\n        self._native_cpu_limiter = threadpool_limits(limits=gpu_threads)\\n        self._native_cpu_requested = gpu_threads\\n        self._native_cpu_environment = environment\\n        self._native_cpu_interop_note = interop_note\\n        try:\\n            return self.cpu_budget_snapshot()\\n        except BaseException:\\n            self._native_cpu_requested = None\\n            raise\\n\\n    def cpu_budget_snapshot(self):\\n        """Read and validate real pool limits; never change them in running jobs."""\\n        requested = getattr(self, "_native_cpu_requested", None)\\n        if requested is None:\\n            raise RuntimeError("Native CPU budget has not been configured")\\n        import torch\\n        from threadpoolctl import threadpool_info\\n        pools = [{key: pool.get(key) for key in ("user_api", "internal_api", "prefix", "version", "num_threads")}\\n                 for pool in threadpool_info()]\\n        actual = torch.get_num_threads()\\n        interop = torch.get_num_interop_threads()\\n        if actual != requested or interop != 1:\\n            raise RuntimeError(f"Torch CPU budget differs from reservation: intra={actual}, interop={interop}, reserved={requested}")\\n        for pool in pools:\\n            threads = pool["num_threads"]\\n            if isinstance(threads, bool) or not isinstance(threads, int) or not 1 <= threads <= requested:\\n                raise RuntimeError(f"Native pool CPU limit is not confirmed: {pool[\\\'internal_api\\\']}={threads}, reserved={requested}")\\n        return {"configured": True, "configured_before_load": True, "gpu_host_cpu_threads": requested,\\n                "torch_num_threads": actual, "torch_num_interop_threads": interop,\\n                "interop_configuration": self._native_cpu_interop_note,\\n                "native_pools": pools, "environment_defaults": dict(self._native_cpu_environment),\\n                "scope": "observed native CPU pools; codec lanes have separate explicit thread budgets"}\\n\\n    @property\\n    def pipeline_model_headroom_bytes(self):\\n        import psutil\\n        return max(0, self.pipeline_model_peak_ram_bytes - psutil.Process().memory_info().rss)\\n\\n    def estimate_resources(self, parameters):\\n        """Bounds retained media + decode/transfer/encode copies; excludes model."""\\n        from PIL import Image\\n        references = parameters["references"]\\n        durations, retained, source_bytes, normalized_bytes, pipeline_media = [], 0, 0, 0, 0\\n        frames = self._aligned_frames(parameters["seconds"])\\n        for item in references:\\n            path, kind = Path(item["path"]), item["kind"]\\n            duration = self._reference_probe(path, kind)\\n            durations.append((kind, duration))\\n            source_bytes += path.stat().st_size\\n            if kind == "image":\\n                with Image.open(path) as image:\\n                    retained += image.width * image.height * 4\\n                    # Official conditioning upscales even tiny images to a 2048\\n                    # short edge. Reserve resized RGB plus float/temporary copies.\\n                    ratio = max(image.width, image.height) / min(image.width, image.height)\\n                    pipeline_media += 2048 * (math.ceil(2048 * ratio / 32) * 32) * 20\\n            elif kind == "video":\\n                size = (math.ceil(duration * 24) + 2) * 960 * 544 * 3 + math.ceil(duration * 32000) * 2 * 4\\n                retained += size\\n                normalized_bytes += size + MIB\\n                # The GPU workflow can resize reference video again to the\\n                # official 1344*768 canvas; it is not a CPU preparation stage.\\n                pipeline_media += min(frames, math.ceil(duration * 24) + 2) * 1344 * 768 * 3 * 8\\n            elif kind == "audio":\\n                retained += math.ceil(duration * 96000) * 2 * 4\\n            else:\\n                raise ValueError("Unknown reference modality")\\n        for kind in ("video", "audio"):\\n            if sum(d for k, d in durations if k == kind) > 15:\\n                raise ValueError("Total reference duration per modality must be <=15 seconds")\\n        output = frames * parameters["width"] * parameters["height"] * 3 + math.ceil(frames / 24 * 96000) * 2 * 4\\n        return {"ram_bytes": 3 * retained + 4 * output + pipeline_media + 256 * MIB,\\n                "cpu_threads": 1, "disk_bytes": source_bytes + normalized_bytes + output + 4 * MIB,\\n                "reference_retained_bytes": retained, "output_retained_bytes": output,\\n                "pipeline_reference_workspace_bytes": pipeline_media,\\n                "estimate_kind": "conservative decoded-media admission estimate; not calibrated physical optimum"}\\n\\n    def prepare(self, *, prompt, references, seconds, width, height, steps, seed, cpu_threads=1,\\n                attention_backend=None, acceleration_lora=None):\\n        options = {}\\n        if attention_backend is not None:\\n            options["attention_backend"] = attention_backend\\n        if acceleration_lora is not None:\\n            options["acceleration_lora"] = acceleration_lora\\n        self.validate_request(steps=steps, **options)\\n        if self.pipe is None:\\n            raise RuntimeError("The model has not loaded")\\n        if isinstance(cpu_threads, bool) or not isinstance(cpu_threads, int) or cpu_threads < 1:\\n            raise ValueError("cpu_threads must be a positive reserved CPU budget")\\n        from diffusers.modular_pipelines.minimax_h3 import MiniMaxH3ImageReference, MiniMaxH3VideoReference, MiniMaxH3AudioReference\\n        classes = {"image": MiniMaxH3ImageReference, "video": MiniMaxH3VideoReference, "audio": MiniMaxH3AudioReference}\\n        parameters = dict(prompt=prompt, seconds=seconds, width=width, height=height, steps=steps, seed=seed, **options)\\n        metrics = self._active_metrics or self._new_metrics("generation")\\n        metrics.data["timing"] = "host_wall_seconds; inference synchronizes at CPU handoff"\\n        metrics.data["pipeline_stage_contract"] = "cpu_prepare / exclusive_gpu_infer / cpu_encode_validate"\\n        metrics.data["generation_options"] = {"requested": requested_options(parameters), "effective": None}\\n        prepared = Prepared(parameters, [], [], metrics, cpu_threads)\\n        try:\\n            with self.metrics_context(metrics), self._stage("references.prepare"):\\n                durations = [self._reference_probe(Path(item["path"]), item["kind"]) for item in references]\\n                for kind in ("video", "audio"):\\n                    if sum(d for item, d in zip(references, durations) if item["kind"] == kind) > 15:\\n                        raise ValueError("Total reference duration per modality must be <=15 seconds")\\n                normalized_targets = {}\\n                for index, item in enumerate(references):\\n                    if item["kind"] == "video":\\n                        path = Path(item["path"])\\n                        target = path.with_name(path.stem + f"_ref{index}_24fps.mp4")\\n                        prepared.normalized.append(target)\\n                        normalized_targets[index] = target\\n                def decode(index):\\n                    item = references[index]\\n                    with metrics.stage(f"child.prepare.{index}"):\\n                        path = Path(item["path"])\\n                        if item["kind"] == "video":\\n                            path = self._normalize_reference(path, target=normalized_targets[index])\\n                        return decode_reference(item["kind"], path, classes)\\n                prepared.references = [None] * len(references)\\n                lanes = min(cpu_threads, len(references))\\n                metrics.record_schedule({"phase": "prepare", "workers": lanes, "child_count": len(references),\\n                                         "reason": "explicit CPU budget; each native decoder/normalizer uses one thread"})\\n                if lanes <= 1:\\n                    for index in range(len(references)):\\n                        prepared.references[index] = decode(index)\\n                else:\\n                    with ThreadPoolExecutor(max_workers=lanes, thread_name_prefix="h3-reference") as pool:\\n                        futures = {pool.submit(decode, i): i for i in range(len(references))}\\n                        for future in as_completed(futures):\\n                            prepared.references[futures[future]] = future.result()\\n            return prepared\\n        except BaseException as error:\\n            metrics.finish(error)\\n            self.cleanup(prepared=prepared)\\n            raise\\n\\n    def infer(self, prepared):\\n        import torch\\n        if prepared.cleaned:\\n            raise RuntimeError("Prepared references have already been released")\\n        raw = None\\n        try:\\n            with self.metrics_context(prepared.metrics), self._stage("pipeline"), torch.inference_mode():\\n                configuration = self.generation_context(prepared) if hasattr(self, "adapter_settings") else nullcontext()\\n                with configuration:\\n                    try:\\n                        raw = self._infer_raw(prepared)\\n                        video, audio, rate = cpu_media(raw["videos"][0], raw["audio"][0], raw["sampling_rate"], torch)\\n                        result = CPUResult(video, audio, rate, prepared.metrics, prepared.cpu_threads)\\n                    finally:\\n                        raw = None\\n                        prepared.references.clear()\\n                        # Synchronize before restoring adapters and attention.\\n                        # CPU encoding never changes model or CUDA state.\\n                        gc.collect()\\n                        torch.cuda.synchronize()\\n                        torch.cuda.empty_cache()\\n                result.cpu_handoff_complete = True\\n                return result\\n        except BaseException as error:\\n            prepared.metrics.finish(error)\\n            raise\\n        finally:\\n            raw = None\\n\\n    def encode(self, result, output_path):\\n        path = Path(output_path)\\n        error = None\\n        try:\\n            with self.metrics_context(result.metrics), self._stage("encode_mp4"):\\n                encode_cpu_media(result, path)\\n        except BaseException as caught:\\n            error = caught\\n            raise\\n        finally:\\n            metrics = result.metrics.finish(error)\\n            self.last_metrics = metrics\\n            sidecar = path.with_suffix(".metrics.json")\\n            temporary = sidecar.with_suffix(sidecar.suffix + ".tmp")\\n            try:\\n                temporary.write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + "\\\\n", encoding="utf-8")\\n                temporary.replace(sidecar)\\n            except OSError as write_error:\\n                metrics["metrics_write_error"] = type(write_error).__name__\\n        return metrics\\n\\n    def cleanup(self, prepared=None, result=None):\\n        """CPU ownership only: never collect globally or touch CUDA from this lane."""\\n        if prepared is not None and not prepared.cleaned:\\n            prepared.references.clear()\\n            for path in prepared.normalized:\\n                Path(path).unlink(missing_ok=True)\\n            prepared.normalized.clear()\\n            prepared.cleaned = True\\n        if result is not None and not result.cleaned:\\n            result.video.clear()\\n            result.audio = None\\n            result.cleaned = True\\n\\n    @measured_generation\\n    def generate(self, prompt, references, output_path, seconds, width, height, steps, seed,\\n                 attention_backend=None, acceleration_lora=None):\\n        """Standalone compatibility: the same stages, called sequentially."""\\n        prepared = result = None\\n        try:\\n            prepared = self.prepare(prompt=prompt, references=references, seconds=seconds,\\n                                    width=width, height=height, steps=steps, seed=seed,\\n                                    attention_backend=attention_backend, acceleration_lora=acceleration_lora)\\n            result = self.infer(prepared)\\n            return self.encode(result, output_path)\\n        finally:\\n            self.cleanup(prepared=prepared, result=result)\\n\', \'pipeline_scheduler.py\': \'"""Per-job working-set admission and stage leases; no global job-count cap."""\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nfrom collections import OrderedDict, deque\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport shutil\\nimport sys\\nimport threading\\nimport time\\nimport uuid\\n\\nimport psutil\\n\\nfrom condition_guard import evaluate_gate, negate_observation\\n\\nGiB = 1024 ** 3\\n\\n\\ndef cgroup_cpu_quotas(root=Path(\\\'/sys/fs/cgroup\\\'), membership=Path(\\\'/proc/self/cgroup\\\')):\\n    """Read v1/v2 quota limits, including stricter visible parent cgroups."""\\n    root = Path(root).resolve()\\n    candidates = {(root, \\\'v2\\\'), (root / \\\'cpu\\\', \\\'v1\\\'), (root / \\\'cpu,cpuacct\\\', \\\'v1\\\')}\\n    try:\\n        records = Path(membership).read_text().splitlines()\\n    except OSError:\\n        records = []\\n    for record in records:\\n        parts = record.split(\\\':\\\', 2)\\n        if len(parts) != 3:\\n            continue\\n        controllers, relative = parts[1], parts[2].lstrip(\\\'/\\\')\\n        bases = [(root, \\\'v2\\\')] if not controllers else (\\n            [(root / \\\'cpu\\\', \\\'v1\\\'), (root / \\\'cpu,cpuacct\\\', \\\'v1\\\')] if \\\'cpu\\\' in controllers.split(\\\',\\\') else [])\\n        for base, version in bases:\\n            leaf = (base / relative).resolve()\\n            if not leaf.is_relative_to(base):\\n                continue\\n            for directory in (leaf, *leaf.parents):\\n                if not directory.is_relative_to(base):\\n                    break\\n                candidates.add((directory, version))\\n    limits = []\\n    for directory, version in sorted(candidates):\\n        try:\\n            if version == \\\'v2\\\':\\n                filename = directory / \\\'cpu.max\\\'\\n                quota, period = filename.read_text().split()\\n                if quota == \\\'max\\\':\\n                    continue\\n            else:\\n                filename = directory / \\\'cpu.cfs_quota_us\\\'\\n                quota = filename.read_text().strip()\\n                period = (directory / \\\'cpu.cfs_period_us\\\').read_text().strip()\\n            quota, period = int(quota), int(period)\\n            if quota > 0 and period > 0:\\n                limits.append({\\\'source\\\': str(filename), \\\'cores\\\': quota / period})\\n        except (OSError, ValueError):\\n            continue\\n    return limits\\n\\n\\ndef detect_cpu_capacity():\\n    logical = max(1, os.cpu_count() or 1)\\n    affinity = None\\n    try:\\n        affinity = len(os.sched_getaffinity(0))\\n    except (AttributeError, OSError):\\n        try:\\n            affinity = len(psutil.Process().cpu_affinity())\\n        except (AttributeError, psutil.Error):\\n            pass\\n    quota_limits = cgroup_cpu_quotas() if sys.platform.startswith(\\\'linux\\\') else []\\n    effective = min([logical] + ([affinity] if affinity else []) + [value[\\\'cores\\\'] for value in quota_limits])\\n    return {\\\'logical_cpus\\\': logical, \\\'affinity_cpus\\\': affinity, \\\'cgroup_quotas\\\': quota_limits,\\n            \\\'effective_cpu_cores\\\': effective, \\\'usable_cpu_threads\\\': max(1, math.floor(effective)),\\n            \\\'fractional_quota_note\\\': \\\'A quota below one CPU uses one throttled lane, not one full CPU\\\'}\\n\\n\\nclass BudgetUnavailable(Exception):\\n    def __init__(self, reason, permanent=False):\\n        super().__init__(reason)\\n        self.permanent = permanent\\n\\n\\nclass PipelineBudget:\\n    def __init__(self, root, *, limits=None, model_peak_ram_bytes=0):\\n        self.root = Path(root)\\n        self._condition = threading.Condition()\\n        self._reservations = {}\\n        self._reservation_ids = {}\\n        self._cancellations = {}\\n        self._result_bytes = {}\\n        self._cpu_used = 0\\n        self._gpu_owner = None\\n        self._model_peak = int(model_peak_ram_bytes)\\n        self._system_checks = limits is None\\n        self._cpu_capacity = detect_cpu_capacity()\\n        self._os_reserve = int(os.environ.get("H3_PIPELINE_OS_RESERVE_MIB", "2048")) * 1024 ** 2\\n        defaults = {\\n            "ram_bytes": int(os.environ.get("H3_PIPELINE_RAM_MIB", "8192")) * 1024 ** 2,\\n            "disk_bytes": max(0, shutil.disk_usage(self.root).free - GiB),\\n            "cpu_threads": int(os.environ.get("H3_PIPELINE_CPU_THREADS", str(self._cpu_capacity[\\\'usable_cpu_threads\\\']))),\\n        }\\n        self.limits = defaults if limits is None else {**defaults, **limits}\\n        if any(type(value) is not int or value <= 0 for value in self.limits.values()):\\n            raise ValueError("Pipeline budgets must be positive integers")\\n        self._requested_cpu_threads = self.limits[\\\'cpu_threads\\\']\\n        self.limits[\\\'cpu_threads\\\'] = min(self.limits[\\\'cpu_threads\\\'], self._cpu_capacity[\\\'usable_cpu_threads\\\'])\\n        self._requested_gpu_cpu_threads = int(os.environ.get(\\\'H3_PIPELINE_GPU_CPU_THREADS\\\', \\\'1\\\'))\\n        if self._requested_gpu_cpu_threads < 1:\\n            raise ValueError(\\\'GPU host CPU reservation must be positive\\\')\\n        self.gpu_host_cpu_threads = min(self._requested_gpu_cpu_threads, self.limits[\\\'cpu_threads\\\'])\\n        self.native_cpu_status = {\\\'configured\\\': False, \\\'reason\\\': \\\'Backend native CPU configuration pending\\\'}\\n        self.instance_id = uuid.uuid4().hex\\n        self._source_sha256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n        self._last_gates = {}\\n        self._attempt_gates = OrderedDict()\\n        self._active_leases = {}\\n        self._released_leases = OrderedDict()\\n        self._lease_history = deque(maxlen=512)\\n        self._lease_event_count = 0\\n\\n    def _capacity_version(self):\\n        payload = {\\\'limits\\\': self.limits, \\\'gpu_host_cpu_threads\\\': self.gpu_host_cpu_threads}\\n        return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\\n\\n    def _gate(self, identity, kind, observations, context=None):\\n        # Called with our Condition held. No callbacks into other ownership domains.\\n        receipt = evaluate_gate(\\\'pipeline.\\\' + kind, list(observations), observations,\\n                                {**(context or {}), \\\'budget_instance_id\\\': self.instance_id,\\n                                 \\\'attempt_id\\\': identity, \\\'resource_kind\\\': kind,\\n                                 \\\'reservation_id\\\': self._reservation_ids.get(identity),\\n                                 \\\'capacity_version\\\': self._capacity_version(),\\n                                 \\\'source_file\\\': \\\'pipeline_scheduler.py\\\',\\n                                 \\\'source_sha256\\\': self._source_sha256})\\n        self._last_gates[receipt[\\\'gate_id\\\']] = receipt\\n        key = (identity, kind)\\n        self._attempt_gates[key] = receipt\\n        self._attempt_gates.move_to_end(key)\\n        while len(self._attempt_gates) > 128:\\n            self._attempt_gates.popitem(last=False)\\n        return receipt\\n\\n    def gate_receipt(self, identity, kind):\\n        with self._condition:\\n            return copy.deepcopy(self._attempt_gates.get((identity, kind)))\\n\\n    def reservation_witness(self, identity):\\n        with self._condition:\\n            return {\\\'owned\\\': identity in self._reservations,\\n                    \\\'budget_instance_id\\\': self.instance_id, \\\'attempt_id\\\': identity,\\n                    \\\'reservation_id\\\': self._reservation_ids.get(identity),\\n                    \\\'resources\\\': copy.deepcopy(self._reservations.get(identity)),\\n                    \\\'capacity_version\\\': self._capacity_version()}\\n\\n    def lease_witness(self, identity, kind):\\n        with self._condition:\\n            return copy.deepcopy(next((lease for lease in self._active_leases.values()\\n                                       if lease[\\\'attempt_id\\\'] == identity and lease[\\\'kind\\\'] == kind), None))\\n\\n    def release_witness(self, lease_id):\\n        with self._condition:\\n            return copy.deepcopy(self._released_leases.get(lease_id))\\n\\n    def set_cancelled(self, identity, observation):\\n        """Update cancellation evidence in the same domain as budget grants.\\n\\n        Model cancellation also owns its Event and rechecks under its own lock.\\n        Updating this ledger does not interrupt a lease already granted.\\n        """\\n        with self._condition:\\n            if identity not in self._reservations:\\n                return False\\n            self._cancellations[identity] = copy.deepcopy(observation)\\n            self._condition.notify_all()\\n            return True\\n\\n    def _lease_event(self, event_type, lease):\\n        self._lease_event_count += 1\\n        event = {key: copy.deepcopy(lease[key]) for key in\\n                 (\\\'lease_id\\\', \\\'budget_instance_id\\\', \\\'attempt_id\\\', \\\'kind\\\', \\\'cpu_threads\\\',\\n                  \\\'capacity_version\\\', \\\'decision_id\\\', \\\'reservation_id\\\')}\\n        event.update(event_id=uuid.uuid4().hex, sequence=self._lease_event_count,\\n                     type=event_type, observed_at=time.time(), monotonic_ns=time.monotonic_ns(),\\n                     cpu_threads_in_use=self._cpu_used, gpu_attempt_id=self._gpu_owner,\\n                     limits=dict(self.limits))\\n        self._lease_history.append(event)\\n        return event\\n\\n    def _used(self, name):\\n        return sum(value[name] for value in self._reservations.values()) + (\\n            sum(self._result_bytes.values()) if name == "ram_bytes" else 0)\\n\\n    def reserve(self, identity, resources):\\n        if type(identity) is not str or not 1 <= len(identity) <= 256:\\n            raise BudgetUnavailable(\\\'invalid_attempt_identity\\\', permanent=True)\\n        charge = {name: resources[name] for name in ("ram_bytes", "disk_bytes", "cpu_threads")}\\n        if any(type(value) is not int or value <= 0 for value in charge.values()):\\n            raise BudgetUnavailable("invalid_backend_resource_estimate", permanent=True)\\n        with self._condition:\\n            if identity in self._reservations:\\n                same = self._reservations[identity] == charge\\n                self._gate(identity, \\\'reserve.existing\\\', {\\\'reservation_matches\\\': same})\\n                if not same:\\n                    raise BudgetUnavailable(\\\'reservation_identity_conflict\\\', permanent=True)\\n                return\\n            fits = all(charge[name] <= self.limits[name] for name in charge)\\n            room = all(self._used(name) + charge[name] <= self.limits[name]\\n                       for name in (\\\'ram_bytes\\\', \\\'disk_bytes\\\'))\\n            host_available = disk_available = True\\n            if self._system_checks:\\n                try:\\n                    # Existing task memory is not part of the resident model.\\n                    rss = psutil.Process().memory_info().rss\\n                    resident_model = max(0, rss - self._used(\\\'ram_bytes\\\'))\\n                    future_model = max(0, self._model_peak - resident_model)\\n                    host_available = psutil.virtual_memory().available >= (\\n                        future_model + self._os_reserve + self._used(\\\'ram_bytes\\\') + charge[\\\'ram_bytes\\\'])\\n                    disk_available = shutil.disk_usage(self.root).free >= GiB + self._used(\\\'disk_bytes\\\') + charge[\\\'disk_bytes\\\']\\n                except (OSError, psutil.Error):\\n                    host_available = disk_available = None\\n            receipt = self._gate(identity, \\\'reserve\\\', {\\n                \\\'estimate_within_limits\\\': fits, \\\'working_set_available\\\': room,\\n                \\\'host_memory_available\\\': host_available, \\\'disk_available\\\': disk_available,\\n                \\\'result_identity_unused\\\': identity not in self._result_bytes})\\n            if not fits:\\n                raise BudgetUnavailable("request_exceeds_working_set_budget", permanent=True)\\n            if not room:\\n                raise BudgetUnavailable("resource_budget_exhausted")\\n            if host_available is False:\\n                raise BudgetUnavailable(\\\'live_host_memory_pressure\\\')\\n            if disk_available is False:\\n                raise BudgetUnavailable(\\\'live_disk_pressure\\\')\\n            if not receipt[\\\'allowed\\\']:\\n                raise BudgetUnavailable(\\\'reservation_conditions_unconfirmed\\\')\\n            self._reservations[identity] = charge\\n            self._reservation_ids[identity] = uuid.uuid4().hex\\n            self._cancellations[identity] = False\\n\\n    def finish(self, identity, result_bytes=0):\\n        with self._condition:\\n            if any(lease[\\\'attempt_id\\\'] == identity for lease in self._active_leases.values()):\\n                raise BudgetUnavailable(\\\'attempt_still_holds_stage_lease\\\')\\n            self._reservations.pop(identity, None)\\n            self._reservation_ids.pop(identity, None)\\n            self._cancellations.pop(identity, None)\\n            if result_bytes:\\n                self._result_bytes[identity] = result_bytes\\n            self._condition.notify_all()\\n\\n    def acknowledge(self, identity):\\n        with self._condition:\\n            self._result_bytes.pop(identity, None)\\n            self._condition.notify_all()\\n\\n    @contextmanager\\n    def stage(self, identity, kind, *, cpu_threads=1, cancelled=lambda: False,\\n              cancellation_event=None, context=None):\\n        if kind not in {\\\'cpu\\\', \\\'gpu\\\'}:\\n            raise ValueError(\\\'Unknown resource stage\\\')\\n        charge = self.gpu_host_cpu_threads if kind == \\\'gpu\\\' else cpu_threads\\n        if type(charge) is not int or not 1 <= charge <= self.limits[\\\'cpu_threads\\\']:\\n            raise ValueError(\\\'Stage CPU requirement exceeds the available CPU budget\\\')\\n        acquired = False\\n        lease_id = None\\n        with self._condition:\\n            reservation_id = self._reservation_ids.get(identity)\\n        while True:\\n            # Legacy arbitrary callbacks may take a model lock, so never invoke\\n            # them while holding the budget lock. Their sampled state does not\\n            # claim global atomicity; the model rechecks before actual effects.\\n            cancellation = None\\n            if cancellation_event is None:\\n                try:\\n                    cancellation = cancelled()\\n                except Exception:\\n                    pass\\n            with self._condition:\\n                if cancellation_event is not None:\\n                    # Event.is_set is a local flag read, never a foreign callback.\\n                    cancellation = (threading.Event.is_set(cancellation_event)\\n                                    if type(cancellation_event) is threading.Event else None)\\n                observations = {\\n                    \\\'reservation_owned\\\': reservation_id is not None and self._reservation_ids.get(identity) == reservation_id,\\n                    \\\'cancellation_clear\\\': negate_observation(self._cancellations.get(identity)),\\n                    \\\'not_cancelled\\\': negate_observation(cancellation),\\n                    \\\'attempt_stage_idle\\\': not any(lease[\\\'attempt_id\\\'] == identity for lease in self._active_leases.values()),\\n                    \\\'cpu_grant_available\\\': self._cpu_used + charge <= self.limits[\\\'cpu_threads\\\'],\\n                }\\n                if kind == \\\'gpu\\\':\\n                    observations[\\\'gpu_grant_available\\\'] = self._gpu_owner is None\\n                receipt = self._gate(identity, \\\'stage.\\\' + kind, observations, context)\\n                if receipt[\\\'allowed\\\']:\\n                    # The declaration and its resource effect share this lock.\\n                    self._cpu_used += charge\\n                    if kind == \\\'gpu\\\':\\n                        self._gpu_owner = identity\\n                    lease_id = uuid.uuid4().hex\\n                    lease = {\\\'owned\\\': True, \\\'lease_id\\\': lease_id, \\\'attempt_id\\\': identity, \\\'kind\\\': kind,\\n                             \\\'budget_instance_id\\\': self.instance_id, \\\'reservation_id\\\': reservation_id,\\n                             \\\'cpu_threads\\\': charge, \\\'capacity_version\\\': receipt[\\\'context\\\'][\\\'capacity_version\\\'],\\n                             \\\'decision_id\\\': receipt[\\\'decision_id\\\'], \\\'grant_receipt\\\': receipt,\\n                             \\\'granted_at\\\': time.time(), \\\'granted_monotonic_ns\\\': time.monotonic_ns()}\\n                    self._active_leases[lease_id] = lease\\n                    self._lease_event(\\\'lease.acquired\\\', lease)\\n                    acquired = True\\n                    break\\n                if any(receipt[\\\'conditions\\\'][name] != \\\'true\\\' for name in\\n                       (\\\'reservation_owned\\\', \\\'cancellation_clear\\\', \\\'not_cancelled\\\')):\\n                    break\\n                # Only actual contention waits; wait releases the lock and thus\\n                # cannot serialize unrelated CPU/GPU releases or admissions.\\n                self._condition.wait(.05)\\n        try:\\n            yield acquired\\n        finally:\\n            if acquired:\\n                with self._condition:\\n                    lease = self._active_leases.get(lease_id)\\n                    if lease is None or lease[\\\'attempt_id\\\'] != identity or lease[\\\'kind\\\'] != kind:\\n                        raise RuntimeError(\\\'Stage lease ownership was lost\\\')\\n                    if kind == "gpu":\\n                        if self._gpu_owner != identity:\\n                            raise RuntimeError(\\\'GPU lease ownership was lost\\\')\\n                        self._gpu_owner = None\\n                    self._cpu_used -= charge\\n                    self._active_leases.pop(lease_id)\\n                    lease = {**lease, \\\'owned\\\': False, \\\'released_at\\\': time.time(),\\n                             \\\'released_monotonic_ns\\\': time.monotonic_ns()}\\n                    self._released_leases[lease_id] = lease\\n                    while len(self._released_leases) > 256:\\n                        self._released_leases.popitem(last=False)\\n                    self._lease_event(\\\'lease.released\\\', lease)\\n                    self._condition.notify_all()\\n\\n    def snapshot(self):\\n        with self._condition:\\n            used = {name: self._used(name) for name in ("ram_bytes", "disk_bytes")}\\n            return {"limits": dict(self.limits), "reserved": used,\\n                    \\\'budget_instance_id\\\': self.instance_id,\\n                    \\\'capacity_version\\\': self._capacity_version(),\\n                    \\\'last_gate_receipts\\\': copy.deepcopy(self._last_gates),\\n                    \\\'active_leases\\\': copy.deepcopy(list(self._active_leases.values())),\\n                    \\\'lease_history\\\': {\\\'capacity\\\': self._lease_history.maxlen,\\n                                      \\\'events\\\': copy.deepcopy(list(self._lease_history)),\\n                                      \\\'count\\\': self._lease_event_count,\\n                                      \\\'dropped\\\': self._lease_event_count - len(self._lease_history),\\n                                      \\\'history_complete\\\': self._lease_event_count == len(self._lease_history)},\\n                    "free": {name: self.limits[name] - used[name] for name in used},\\n                    "cpu_threads_in_use": self._cpu_used, "gpu_attempt_id": self._gpu_owner,\\n                    "gpu_host_cpu_threads_in_use": self.gpu_host_cpu_threads if self._gpu_owner else 0,\\n                    "cpu_stage_threads_in_use": self._cpu_used - (self.gpu_host_cpu_threads if self._gpu_owner else 0),\\n                    "cpu_count_scope": "Reserved native execution lanes; not sampled CPU utilization",\\n                    "gpu_host_cpu_threads": self.gpu_host_cpu_threads,\\n                    "requested_cpu_threads": self._requested_cpu_threads,\\n                    "requested_gpu_host_cpu_threads": self._requested_gpu_cpu_threads,\\n                    "cpu_capacity": self._cpu_capacity, \\\'native_cpu\\\': self.native_cpu_status,\\n                    "io_budget": {"scope": "disk_capacity_only", "bandwidth_calibrated": False,\\n                                  "bytes_per_second_limit": None,\\n                                  "optimality": "Conservative admission estimates; no mathematical optimum claimed"},\\n                    "admitted_attempts": len(self._reservations),\\n                    "unacknowledged_result_bytes": sum(self._result_bytes.values()),\\n                    "model_peak_ram_bytes": self._model_peak,\\n                    "os_reserve_bytes": self._os_reserve,\\n                    "live_pressure_checks": self._system_checks}\\n\\n    def has_capacity(self):\\n        state = self.snapshot()\\n        return all(value > 0 for value in state["free"].values())\\n\', \'pipeline_smoke_test.py\': \'"""Real public-API queue/cancel/two-content acceptance. No model or fake video.\\n\\nStart while the model is loading: accept eight jobs, cancel six queued jobs,\\nthen fetch the two distinct survivors. --resume only queries existing identities.\\nStage overlap and GPU-release proofs are checked separately from the cloud audit.\\n"""\\nfrom __future__ import annotations\\nimport argparse\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom datetime import datetime, timezone\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport time\\nimport uuid\\nfrom urllib.parse import urlsplit\\n\\n\\ndef require(ok, message):\\n    if not ok:\\n        raise RuntimeError(message)\\n\\n\\ndef now():\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef digest(path):\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef checkpoint(folder, report):\\n    temporary = folder / \\\'pipeline_report.json.tmp\\\'\\n    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n    temporary.replace(folder / \\\'pipeline_report.json\\\')\\n\\n\\ndef download_ranges(base, key, job, folder):\\n    import requests\\n    artifact = job[\\\'artifact\\\']\\n    size = artifact[\\\'bytes\\\']\\n    require(isinstance(size, int) and 0 < size <= 128 * 1024**2, \\\'Unexpected output size\\\')\\n    require(artifact.get(\\\'media\\\', {}).get(\\\'full_decode_passed\\\') is True, \\\'Server did not validate the complete video\\\')\\n    url = base + \\\'/v1/jobs/\\\' + job[\\\'id\\\'] + \\\'/file\\\'\\n    def part(bounds):\\n        start, end = bounds\\n        records = []\\n        for number in range(1, 4):\\n            record = {\\\'attempt\\\': number, \\\'start\\\': start, \\\'end\\\': end}\\n            records.append(record)\\n            try:\\n                with requests.get(url, headers={\\\'Authorization\\\': \\\'Bearer \\\'+key,\\n                        \\\'Range\\\': f\\\'bytes={start}-{end}\\\', \\\'Accept-Encoding\\\': \\\'identity\\\'},\\n                        timeout=(15, 25), allow_redirects=False, stream=True) as response:\\n                    record.update(http_status=response.status_code,\\n                                  content_range=response.headers.get(\\\'Content-Range\\\'))\\n                    require(response.status_code == 206, \\\'Range request was not accepted\\\')\\n                    require(record[\\\'content_range\\\'] == f\\\'bytes {start}-{end}/{size}\\\', \\\'Incorrect Content-Range\\\')\\n                    data = response.raw.read(end-start+2)\\n                    record[\\\'bytes\\\'] = len(data)\\n                    require(len(data) == end-start+1, \\\'Truncated range\\\')\\n                record[\\\'passed\\\'] = True\\n                return data, records\\n            except Exception as error:\\n                record.update(passed=False, error_type=type(error).__name__)\\n        return None, records\\n    bounds = [(start, min(size-1, start+65535)) for start in range(0, size, 65536)]\\n    with ThreadPoolExecutor(max_workers=3) as pool:\\n        parts = list(pool.map(part, bounds))\\n    result = {\\\'job_id\\\': job[\\\'id\\\'], \\\'ranges\\\': [records for _, records in parts], \\\'passed\\\': False}\\n    (folder / (job[\\\'id\\\']+\\\'-download.json\\\')).write_text(json.dumps(result,indent=2),encoding=\\\'utf-8\\\')\\n    require(all(data is not None for data,_ in parts), \\\'File transfer exhausted bounded range retries; resume without generating again\\\')\\n    content = b\\\'\\\'.join(data for data,_ in parts)\\n    actual = hashlib.sha256(content).hexdigest()\\n    require(len(content)==size and actual==artifact[\\\'sha256\\\'], \\\'External file differs from committed artifact\\\')\\n    path = folder / (job[\\\'id\\\']+\\\'.mp4\\\')\\n    path.write_bytes(content)\\n    result.update(passed=True, bytes=size, sha256=actual, path=str(path),\\n                  full_decode_basis=\\\'Server full audiovisual decode, with exact external byte count and SHA256 match\\\')\\n    (folder / (job[\\\'id\\\']+\\\'-download.json\\\')).write_text(json.dumps(result,indent=2),encoding=\\\'utf-8\\\')\\n    return result\\n\\n\\ndef get_json(base, key, route, *, authenticated=True, records=None):\\n    """Bounded read retries never submit or replace an existing generation."""\\n    import requests\\n    for attempt in range(1, 4):\\n        record = {\\\'route\\\': route, \\\'attempt\\\': attempt, \\\'observed_at\\\': now()}\\n        if records is not None:\\n            records.append(record)\\n        try:\\n            response = requests.get(base+route,\\n                headers={\\\'Authorization\\\': \\\'Bearer \\\'+key} if authenticated else {},\\n                timeout=(10,20), allow_redirects=False)\\n            record[\\\'http_status\\\'] = response.status_code\\n            if response.status_code == 200:\\n                return response.json()\\n            require(response.status_code in (429,502,503,504),\\n                    \\\'GET \\\'+route+\\\' returned HTTP \\\'+str(response.status_code))\\n        except requests.RequestException as error:\\n            record[\\\'error_type\\\'] = type(error).__name__\\n        if attempt < 3:\\n            time.sleep(attempt)\\n    raise RuntimeError(\\\'GET \\\'+route+\\\' exhausted bounded read retries; resume existing task identities\\\')\\n\\n\\ndef _run(args):\\n    import requests\\n    key = os.environ.get(\\\'H3_API_KEY\\\', \\\'\\\')\\n    require(len(key)>=32, \\\'Set H3_API_KEY without placing it in command-line arguments\\\')\\n    base = args.url.rstrip(\\\'/\\\')\\n    parsed = urlsplit(base)\\n    require(parsed.scheme in {\\\'http\\\',\\\'https\\\'} and parsed.hostname and not any(\\n        (parsed.username, parsed.password, parsed.query, parsed.fragment)), \\\'Invalid API base URL\\\')\\n    args.outputdir.mkdir(parents=True, exist_ok=True)\\n    target = args.outputdir/\\\'pipeline_report.json\\\'\\n    headers = {\\\'Authorization\\\':\\\'Bearer \\\'+key}\\n    reads = []\\n    def get(route, authenticated=True):\\n        return get_json(base,key,route,authenticated=authenticated,records=reads)\\n    if args.resume:\\n        report = json.loads(target.read_text(encoding=\\\'utf-8\\\'))\\n        require(report[\\\'api_base_url\\\']==base and len(report.get(\\\'jobs\\\',[]))==8,\\n                \\\'Resume needs the original API URL and all eight recorded job identities\\\')\\n        require(all(job.get(\\\'id\\\') for job in report[\\\'jobs\\\']), \\\'Resolve uncertain submission identities before resume\\\')\\n        reads = report.setdefault(\\\'read_attempts\\\', [])\\n        report[\\\'resumed_at\\\'] = now()\\n        args._active_report = True\\n    else:\\n        require(not target.exists(), \\\'Existing test report: use --resume; never resubmit an existing run\\\')\\n        require(args.reference_a and args.reference_b and args.prompt_a and args.prompt_b,\\n                \\\'Two fresh reference images and distinct explicit prompts are required\\\')\\n        refs = [args.reference_a.resolve(),args.reference_b.resolve()]\\n        require(all(p.is_file() for p in refs), \\\'Reference files do not exist\\\')\\n        hashes = [digest(p) for p in refs]\\n        require(hashes[0]!=hashes[1] and args.prompt_a.strip()!=args.prompt_b.strip(),\\n                \\\'The two survivors must use different references and prompts\\\')\\n        report = {\\\'schema_version\\\':1, \\\'run_id\\\':uuid.uuid4().hex, \\\'started_at\\\':now(), \\\'api_base_url\\\':base,\\n                  \\\'passed\\\':False, \\\'jobs\\\':[], \\\'downloads\\\':[], \\\'observations\\\':[], \\\'read_attempts\\\':reads,\\n                  \\\'scope\\\':\\\'Eight durable queued jobs; six queued cancellations; two new-content real generations. Stage overlap and shutdown are independently audited.\\\'}\\n        checkpoint(args.outputdir,report)\\n        args._active_report = True\\n        deadline=time.monotonic()+args.startup_timeout\\n        while True:\\n            try:\\n                system=get(\\\'/v1/system\\\')\\n                if system.get(\\\'api_ready\\\') and not system.get(\\\'observation_stale\\\'):\\n                    break\\n            except Exception:\\n                pass\\n            require(time.monotonic()<deadline, \\\'API did not become queryable\\\')\\n            time.sleep(3)\\n        require(system.get(\\\'model_ready\\\') is False, \\\'Run queue admission/cancel test during cold loading, before any GPU job can start\\\')\\n        report[\\\'instance_id\\\']=system[\\\'instance_id\\\']\\n        report[\\\'system_before_submission\\\']=system\\n        from smoke_test import register_content\\n        registry=args.content_history or args.outputdir.parent/\\\'content-history.json\\\'\\n        report[\\\'content_reservations\\\']=[]\\n        for index,prompt in enumerate((args.prompt_a,args.prompt_b)):\\n            report[\\\'content_reservations\\\'].append(register_content(registry,hashes[index],prompt,\\n                outputdir=args.outputdir,api_base_url=base))\\n            checkpoint(args.outputdir,report)\\n        unauth=requests.get(base+\\\'/v1/system\\\',timeout=15,allow_redirects=False)\\n        report[\\\'unauthorized_system_http_status\\\']=unauth.status_code\\n        require(unauth.status_code==401,\\\'Unauthorized query was not rejected\\\')\\n        for index in range(8):\\n            survivor=index in (0,2)\\n            content_index=1 if index==2 else 0\\n            prompt=(args.prompt_a,args.prompt_b)[content_index]\\n            if not survivor:\\n                prompt+=f\\\' Alternate cancelled queue test {index}: a new viewpoint with slow lateral camera motion.\\\'\\n            entry={\\\'index\\\':index,\\\'survivor\\\':survivor,\\\'reference\\\':str(refs[content_index]),\\n                   \\\'reference_sha256\\\':hashes[content_index], \\\'prompt\\\':prompt,\\n                   \\\'idempotency_key\\\':report[\\\'run_id\\\']+\\\'-\\\'+str(index), \\\'submission_attempts\\\':[]}\\n            report[\\\'jobs\\\'].append(entry)\\n            checkpoint(args.outputdir,report)\\n            for attempt in range(1,4):\\n                try:\\n                    with refs[content_index].open(\\\'rb\\\') as media:\\n                        response=requests.post(base+\\\'/v1/jobs\\\',headers={**headers,\\\'Idempotency-Key\\\':entry[\\\'idempotency_key\\\']},\\n                            data={\\\'prompt\\\':prompt,\\\'kinds\\\':\\\'["image"]\\\',\\\'seconds\\\':5,\\\'width\\\':512,\\\'height\\\':288,\\\'steps\\\':8,\\\'seed\\\':42+index},\\n                            files=[(\\\'files\\\',(refs[content_index].name,media,\\\'image/png\\\'))],\\n                            timeout=(15,90),allow_redirects=False)\\n                    entry[\\\'submission_attempts\\\'].append({\\\'attempt\\\':attempt,\\\'http_status\\\':response.status_code})\\n                    if response.status_code==202:\\n                        value=response.json()\\n                        entry.update(id=value[\\\'id\\\'],submitted_status=value[\\\'status\\\'],receipt=value)\\n                        break\\n                    require(response.status_code in (429,502,503,504), \\\'Submission definitively rejected: HTTP \\\'+str(response.status_code))\\n                except requests.RequestException as error:\\n                    entry[\\\'submission_attempts\\\'].append({\\\'attempt\\\':attempt,\\\'error_type\\\':type(error).__name__})\\n                finally:\\n                    checkpoint(args.outputdir,report)\\n                time.sleep(1)\\n            require(entry.get(\\\'id\\\'), \\\'Submission uncertain: keep this report and idempotency key; do not start a new run\\\')\\n            require(entry[\\\'submitted_status\\\']==\\\'queued\\\',\\\'Model started before queue cancellation acceptance window\\\')\\n            print(\\\'accepted\\\',index,entry[\\\'id\\\'],flush=True)\\n        report[\\\'all_eight_accepted_before_cancellation\\\']=True\\n        checkpoint(args.outputdir,report)\\n    try:\\n        require(get(\\\'/v1/system\\\')[\\\'instance_id\\\']==report[\\\'instance_id\\\'],\\\'Manager instance changed; do not replay generation\\\')\\n        # Repeating cancellation for the recorded identities is idempotent, including on resume.\\n        for entry in report[\\\'jobs\\\']:\\n            if not entry[\\\'survivor\\\']:\\n                before=get(\\\'/v1/jobs/\\\'+entry[\\\'id\\\'])\\n                require(before[\\\'status\\\'] in {\\\'queued\\\',\\\'cancelled\\\'},\\\'Cancellation target already began executing; preserve failure evidence\\\')\\n                response=requests.post(base+\\\'/v1/jobs/\\\'+entry[\\\'id\\\']+\\\'/cancel\\\',headers=headers,timeout=30,allow_redirects=False)\\n                require(response.status_code==202,\\\'Cancellation request was not accepted\\\')\\n                after=get(\\\'/v1/jobs/\\\'+entry[\\\'id\\\'])\\n                entry[\\\'cancel_receipt\\\']=response.json()\\n                entry[\\\'terminal\\\']=after\\n                require(after[\\\'status\\\']==\\\'cancelled\\\' and after.get(\\\'started_at\\\') is None,\\n                        \\\'Queued cancellation did not complete before dispatch\\\')\\n                checkpoint(args.outputdir,report)\\n        deadline=time.monotonic()+args.generation_timeout\\n        previous=None\\n        while True:\\n            survivors=[entry for entry in report[\\\'jobs\\\'] if entry[\\\'survivor\\\']]\\n            for entry in survivors:\\n                entry[\\\'terminal\\\']=get(\\\'/v1/jobs/\\\'+entry[\\\'id\\\'])\\n            summary=[{\\\'id\\\':entry[\\\'id\\\'],\\\'status\\\':entry[\\\'terminal\\\'][\\\'status\\\'],\\n                      \\\'stage\\\':entry[\\\'terminal\\\'].get(\\\'active_phase\\\')} for entry in survivors]\\n            signature=json.dumps(summary,sort_keys=True)\\n            if signature!=previous:\\n                observation={\\\'observed_at\\\':now(),\\\'jobs\\\':summary}\\n                report[\\\'observations\\\'].append(observation)\\n                print(json.dumps(observation),flush=True)\\n                previous=signature\\n            checkpoint(args.outputdir,report)\\n            require(all(entry[\\\'terminal\\\'][\\\'status\\\'] not in {\\\'failed\\\',\\\'cancelled\\\'} for entry in survivors),\\n                    \\\'A generation did not succeed; inspect the retained report and model log\\\')\\n            if all(entry[\\\'terminal\\\'][\\\'status\\\']==\\\'succeeded\\\' for entry in survivors):\\n                break\\n            require(time.monotonic()<deadline,\\\'Generation deadline exceeded; query or cancel existing IDs without resubmitting\\\')\\n            time.sleep(args.poll_seconds)\\n        for entry in survivors:\\n            report[\\\'downloads\\\'].append(download_ranges(base,key,entry[\\\'terminal\\\'],args.outputdir))\\n            checkpoint(args.outputdir,report)\\n        report.update(passed=True,finished_at=now())\\n        checkpoint(args.outputdir,report)\\n        print(\\\'QUEUE_CANCEL_TWO_GENERATIONS_PASSED\\\',flush=True)\\n        return report\\n    except Exception as error:\\n        report.update(passed=False,error_type=type(error).__name__,error=str(error),last_updated_at=now())\\n        checkpoint(args.outputdir,report)\\n        raise\\n\\n\\ndef run(args):\\n    """Checkpoint failures during admission as well as during generation polling."""\\n    args._active_report = False\\n    try:\\n        return _run(args)\\n    except Exception as error:\\n        target = args.outputdir/\\\'pipeline_report.json\\\'\\n        if args._active_report and target.is_file():\\n            try:\\n                report = json.loads(target.read_text(encoding=\\\'utf-8\\\'))\\n                message = str(error)\\n                key = os.environ.get(\\\'H3_API_KEY\\\')\\n                if key:\\n                    message = message.replace(key, \\\'[REDACTED]\\\')\\n                report.update(passed=False,error_type=type(error).__name__,error=message,last_updated_at=now())\\n                checkpoint(args.outputdir,report)\\n            except (OSError, ValueError, TypeError):\\n                pass  # Preserve the original error and any earlier durable identities.\\n        raise\\n\\n\\ndef main():\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\'--url\\\',required=True)\\n    parser.add_argument(\\\'--outputdir\\\',type=Path,required=True)\\n    parser.add_argument(\\\'--reference-a\\\',type=Path)\\n    parser.add_argument(\\\'--reference-b\\\',type=Path)\\n    parser.add_argument(\\\'--prompt-a\\\')\\n    parser.add_argument(\\\'--prompt-b\\\')\\n    parser.add_argument(\\\'--resume\\\',action=\\\'store_true\\\')\\n    parser.add_argument(\\\'--content-history\\\',type=Path)\\n    parser.add_argument(\\\'--startup-timeout\\\',type=float,default=300)\\n    parser.add_argument(\\\'--generation-timeout\\\',type=float,default=5400)\\n    parser.add_argument(\\\'--poll-seconds\\\',type=float,default=10)\\n    run(parser.parse_args())\\n\\n\\nif __name__==\\\'__main__\\\':\\n    main()\\n\', \'notebook_console.py\': \'"""Owned notebook UI bridge. It observes the manager; it is not a GPU scheduler."""\\nfrom __future__ import annotations\\n\\nimport ast\\nimport base64\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nimport secrets\\nfrom pathlib import Path\\nimport threading\\nimport sys\\nimport time\\nimport traceback\\nimport uuid\\n\\nimport notebook_ui as ui\\n\\n\\nACTION_TITLES = {\\\'bootstrap\\\': \\\'一键启动\\\', \\\'preflight\\\': \\\'资源检查\\\', \\\'install\\\': \\\'安装依赖\\\', \\\'start\\\': \\\'启动服务\\\',\\n                 \\\'generate\\\': \\\'生成验收\\\', \\\'stop\\\': \\\'停止服务\\\', \\\'release\\\': \\\'释放实例\\\', \\\'control\\\': \\\'组件控制\\\'}\\nSTARTUP_LABELS = {\\n    \\\'preflight\\\': \\\'检查资源\\\',\\n    \\\'install\\\': \\\'安装依赖\\\',\\n    \\\'start\\\': \\\'启动独立服务\\\',\\n    \\\'awaiting_api\\\': \\\'等待 API 及所选公网可达\\\',\\n    \\\'awaiting_model\\\': \\\'API 已可接收排队任务；等待模型完整加载\\\',\\n    \\\'ready\\\': \\\'模型与 API 已就绪；排队任务由总管理自动调度\\\',\\n    \\\'failed\\\': \\\'启动未完成，查看实际错误；已启动的后台仍可查询和停止\\\',\\n    \\\'cancelled\\\': \\\'启动已取消\\\',\\n}\\nLOG_NAMES = {\\\'model\\\': \\\'server_log\\\', \\\'api\\\': \\\'api_log\\\', \\\'manager\\\': \\\'manager_log\\\', \\\'tunnel\\\': \\\'tunnel_log\\\'}\\nBOOLS = {\\\'staged\\\', \\\'cached\\\', \\\'public\\\', \\\'force\\\'}\\nSTRINGS = {\\\'reference_file\\\', \\\'prompt\\\', \\\'resume_job_id\\\', \\\'endpoint\\\', \\\'acceleration\\\', \\\'style\\\',\\n           \\\'attention\\\', \\\'offload\\\', \\\'action\\\', \\\'command_id\\\', \\\'request_id\\\', \\\'source\\\', \\\'credential_id\\\'}\\nCHOICES = {\\\'endpoint\\\': {\\\'public\\\', \\\'local\\\'}, \\\'acceleration\\\': {\\\'none\\\', \\\'lightx2v-ref2va-4step\\\'},\\n           \\\'style\\\': {\\\'none\\\', \\\'fal-realism-people\\\'}, \\\'attention\\\': {\\\'auto\\\', \\\'native\\\', \\\'_native_flash\\\'},\\n           \\\'offload\\\': {\\\'stream\\\', \\\'sync\\\'}}\\nSAFE_STATE = {\\\'instance_id\\\', \\\'phase\\\', \\\'ready\\\', \\\'api_ready\\\', \\\'model_ready\\\', \\\'public_requested\\\',\\n              \\\'public_api_ready\\\', \\\'public_ready\\\', \\\'tunnel_connected\\\', \\\'server_alive\\\', \\\'model_alive\\\',\\n              \\\'manager_alive\\\', \\\'tunnel_alive\\\', \\\'cleanup_complete\\\', \\\'observed_at\\\', \\\'observation_stale\\\',\\n              \\\'elapsed_seconds\\\', \\\'draining\\\', \\\'queue\\\'}\\n\\n\\nclass OperationCancelled(RuntimeError):\\n    pass\\n\\n\\ndef validated(payload):\\n    if not isinstance(payload, dict) or len(json.dumps(payload)) > 65536:\\n        raise ValueError(\\\'控制参数必须是有限的对象。\\\')\\n    result = {}\\n    for key, value in payload.items():\\n        if key in BOOLS:\\n            if type(value) is not bool:\\n                raise ValueError(f\\\'{key} 必须为布尔值。\\\')\\n        elif key in STRINGS:\\n            if not isinstance(value, str) or len(value) > (6000 if key == \\\'prompt\\\' else 2048):\\n                raise ValueError(f\\\'{key} 不是有效的短文本。\\\')\\n        elif key == \\\'style_scale\\\':\\n            if type(value) not in (int, float) or not .1 <= value <= 1:\\n                raise ValueError(\\\'风格强度应在 0.1 与 1 之间。\\\')\\n        elif key in {\\\'cursor\\\'}:\\n            continue\\n        else:\\n            raise ValueError(f\\\'未知控制字段：{key}\\\')\\n        if key in CHOICES and value not in CHOICES[key]:\\n            raise ValueError(f\\\'{key} 选项无效。\\\')\\n        result[key] = value\\n    return result\\n\\n\\ndef action_code(source, parameters):\\n    """Replace only known top-level notebook defaults, never execute browser-supplied code."""\\n    tree = ast.parse(source)\\n    found = set()\\n    for node in tree.body:\\n        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):\\n            name = node.targets[0].id\\n            if name in parameters:\\n                node.value = ast.Constant(parameters[name])\\n                found.add(name)\\n    if found != set(parameters):\\n        raise ValueError(\\\'操作参数与冻结的操作代码不匹配。\\\')\\n\\n    class NoAutomaticDownloads(ast.NodeTransformer):\\n        def visit_Expr(self, node):\\n            call = node.value\\n            if (isinstance(call, ast.Call) and isinstance(call.func, ast.Attribute)\\n                    and isinstance(call.func.value, ast.Name) and call.func.value.id == \\\'files\\\'\\n                    and call.func.attr == \\\'download\\\'):\\n                return ast.copy_location(ast.Pass(), node)\\n            return self.generic_visit(node)\\n    tree = NoAutomaticDownloads().visit(tree)\\n    return compile(ast.fix_missing_locations(tree), \\\'owned-notebook-operation\\\', \\\'exec\\\')\\n\\n\\nclass ConsoleController:\\n    def __init__(self, directory, programs, scope=None, release_callback=None):\\n        self.directory = Path(directory)\\n        self.directory.mkdir(parents=True, exist_ok=True)\\n        self.programs = dict(programs)\\n        self.scope = scope if scope is not None else {}\\n        self.scope[\\\'PROJECT_DIR\\\'] = self.directory\\n        self.scope[\\\'dashboard\\\'] = self\\n        self.scope[\\\'print\\\'] = self._print\\n        self.scope.update(StatusCard=ui.StatusCard, elapsed_text=ui.elapsed_text, log_tail=ui.log_tail,\\n                          run_logged=ui.run_logged, json_probe=ui.json_probe, os=os, sys=sys, json=json, Path=Path)\\n        self.lock = threading.RLock()\\n        self.executor = ThreadPoolExecutor(max_workers=3, thread_name_prefix=\\\'h3-console\\\')\\n        self.operations = {}\\n        self.active = {}\\n        self.finished = {}\\n        self.cancel_startup = threading.Event()\\n        self.startup_stage = None\\n        self.startup_delivery = None\\n        self._bootstrap_hosts = {}\\n        self.requested_options = {}\\n        self.processes = {}\\n        self.process_owners = {}\\n        self.closed = False\\n        self.epoch = None\\n        self.activity = None\\n        self._last_activity = None\\n        self.notice = None\\n        self.release_callback = release_callback\\n        self.audit_path = self.directory / \\\'console-events.jsonl\\\'\\n        self.log_path = self.directory / \\\'console.log\\\'\\n        ui._dashboard = self\\n        ui._process_observer = self\\n\\n    def _secrets(self):\\n        host = self.scope.get(\\\'startup_host\\\')\\n        return [getattr(host, \\\'api_key\\\', \\\'\\\'), getattr(host, \\\'_control_key\\\', \\\'\\\'), self.scope.get(\\\'API_KEY\\\', \\\'\\\') or \\\'\\\',\\n                self.scope.get(\\\'BOOTSTRAP_API_KEY\\\', \\\'\\\') or \\\'\\\']\\n\\n    def _clean(self, value):\\n        return ui.redact_text(str(value), secrets=self._secrets())\\n\\n    def _sanitize(self, value):\\n        if isinstance(value, str):\\n            return self._clean(value)\\n        if isinstance(value, dict):\\n            return {key: self._sanitize(item) for key, item in value.items()}\\n        if isinstance(value, (list, tuple)):\\n            return [self._sanitize(item) for item in value]\\n        return value\\n\\n    def _event(self, kind, **values):\\n        record = {\\\'type\\\': kind, \\\'observed_at\\\': time.time(), **values}\\n        with self.lock:\\n            with self.audit_path.open(\\\'a\\\', encoding=\\\'utf-8\\\') as stream:\\n                stream.write(self._clean(json.dumps(record, ensure_ascii=False)) + \\\'\\\\n\\\')\\n\\n    def _print(self, *values, sep=\\\' \\\', end=\\\'\\\\n\\\', **kwargs):\\n        text = self._clean(sep.join(str(v) for v in values))\\n        stamp = time.strftime(\\\'%m-%d %H:%M:%S\\\', time.gmtime(time.time() + 8 * 3600))\\n        with self.lock:\\n            with self.log_path.open(\\\'a\\\', encoding=\\\'utf-8\\\') as stream:\\n                stream.write(\\\'[\\\' + stamp + \\\' 北京时间] \\\' + text + end)\\n\\n    def refresh(self):\\n        """StatusCard sink: update data and log, never create a notebook output."""\\n        item = self.activity or {}\\n        marker = json.dumps(item, sort_keys=True, ensure_ascii=False)\\n        if marker != self._last_activity:\\n            self._last_activity = marker\\n            self._print(item.get(\\\'title\\\', \\\'\\\'), item.get(\\\'badge\\\', \\\'\\\'), item.get(\\\'note\\\', \\\'\\\'))\\n\\n    def process_started(self, process):\\n        owner = None\\n        if type(getattr(process, \\\'pid\\\', None)) is int:\\n            from process_owner import ProcessOwner\\n            owner = ProcessOwner(process)\\n        with self.lock:\\n            self.processes[threading.get_ident()] = process\\n            if owner is not None:\\n                self.process_owners[id(process)] = (owner, threading.get_ident())\\n            cancelled = any(record.get(\\\'thread_id\\\') == threading.get_ident()\\n                            and (record.get(\\\'cancel_monitor\\\') or record.get(\\\'cancel_requested\\\'))\\n                            for record in self.active.values())\\n        if cancelled:\\n            self._terminate_client_process(process)\\n\\n    def process_progress(self, process):\\n        with self.lock:\\n            entry = self.process_owners.get(id(process))\\n        if entry:\\n            entry[0].observe()  # Capture descendants before wrappers exit.\\n\\n    def process_finished(self, process):\\n        with self.lock:\\n            entry = self.process_owners.get(id(process))\\n        if entry:\\n            state = entry[0].observe()\\n            if not state[\\\'cleanup_complete\\\']:\\n                state = entry[0].terminate_all(timeout=10)\\n            self._event(\\\'client.process_cleanup\\\', **state)\\n            if not state[\\\'cleanup_complete\\\']:\\n                raise RuntimeError(\\\'安装或客户端的子进程树尚未确认退出；请停止后检查日志。\\\')\\n        with self.lock:\\n            self.process_owners.pop(id(process), None)\\n            if self.processes.get(threading.get_ident()) is process:\\n                self.processes.pop(threading.get_ident(), None)\\n\\n    def _terminate_client_process(self, process):\\n        with self.lock:\\n            entry = self.process_owners.get(id(process))\\n        if entry:\\n            state = entry[0].terminate_all(timeout=10)\\n            self._event(\\\'client.process_cleanup\\\', **state)\\n            if not state[\\\'cleanup_complete\\\']:\\n                raise RuntimeError(\\\'子进程树仍未退出；未声明停止或释放完成。\\\')\\n        elif process.poll() is None:\\n            process.terminate()\\n            try:\\n                process.wait(timeout=10)\\n            except Exception:\\n                process.kill()\\n                process.wait(timeout=5)\\n\\n    def _stop_client_monitor(self):\\n        with self.lock:\\n            active = self.active.get(\\\'generate\\\')\\n            if active:\\n                active[\\\'cancel_monitor\\\'] = True\\n            process = self.processes.get(active.get(\\\'thread_id\\\')) if active else None\\n        if process is not None and process.poll() is None:\\n            self._terminate_client_process(process)\\n            self._event(\\\'client.monitor_stopped\\\', request_id=active[\\\'request_id\\\'])\\n\\n    def _execute(self, name, parameters=None):\\n        exec(action_code(self.programs[name], parameters or {}), self.scope)\\n\\n    def _configuration(self, payload):\\n        self.requested_options = {name: payload.get(name, \\\'none\\\') for name in (\\\'acceleration\\\', \\\'style\\\')}\\n        self._execute(\\\'advanced\\\', {\\\'ACCELERATION_LORA\\\': payload.get(\\\'acceleration\\\', \\\'none\\\'),\\n            \\\'STYLE_LORA\\\': payload.get(\\\'style\\\', \\\'none\\\'), \\\'STYLE_LORA_SCALE\\\': payload.get(\\\'style_scale\\\', 1.0),\\n            \\\'ATTENTION_BACKEND\\\': payload.get(\\\'attention\\\', \\\'auto\\\'), \\\'DENOISER_OFFLOAD\\\': payload.get(\\\'offload\\\', \\\'stream\\\')})\\n\\n    def _check_startup_cancelled(self):\\n        if self.cancel_startup.is_set():\\n            raise OperationCancelled(\\\'启动已取消；不再进入后续阶段。\\\')\\n\\n    def _startup_stage(self, stage):\\n        if self.startup_stage != stage:\\n            self.startup_stage = stage\\n            self._event(\\\'startup.stage\\\', stage=stage)\\n            self._print(STARTUP_LABELS.get(stage, stage))\\n\\n    def _await_bootstrap(self, host, public):\\n        deadline = time.monotonic() + float(getattr(host, \\\'startup_timeout\\\', 3600))\\n        interval = max(.01, min(2, float(getattr(host, \\\'poll_interval\\\', 1))))\\n        while True:\\n            self._check_startup_cancelled()\\n            state = host.snapshot()\\n            fresh = not state.get(\\\'observation_stale\\\', False)\\n            if state.get(\\\'phase\\\') in {\\\'failed\\\', \\\'stopped\\\', \\\'stopping\\\'} or (fresh and state.get(\\\'error\\\')):\\n                raise RuntimeError(state.get(\\\'error\\\') or \\\'服务准备未完成；API 可用时仍可查询与取消任务。\\\')\\n            api = fresh and state.get(\\\'api_ready\\\') is True and (not public or state.get(\\\'public_api_ready\\\') is True)\\n            if api and state.get(\\\'model_ready\\\') is True and state.get(\\\'ready\\\') is True:\\n                self._startup_stage(\\\'ready\\\')\\n                return\\n            self._startup_stage(\\\'awaiting_model\\\' if api else \\\'awaiting_api\\\')\\n            if time.monotonic() >= deadline:\\n                raise TimeoutError(\\\'等待服务就绪超时；请查看日志。已有任务和后台保留，可查询、取消或停止。\\\')\\n            self.cancel_startup.wait(interval)\\n\\n    def _bootstrap(self, payload):\\n        payload = dict(payload, public=payload.get(\\\'public\\\', True))\\n        self._check_startup_cancelled()\\n        host = self.scope.get(\\\'startup_host\\\')\\n        state = host.snapshot() if host else {}\\n        public = payload.get(\\\'public\\\', True)\\n        if host and any(state.get(name) for name in (\\\'manager_alive\\\', \\\'server_alive\\\', \\\'model_alive\\\', \\\'tunnel_alive\\\')):\\n            # A repeated start observes the owned instance; it cannot reinstall or rotate its token.\\n            if bool(getattr(host, \\\'public\\\', False)) != public or bool(getattr(host, \\\'staged\\\', False)) != payload.get(\\\'staged\\\', False):\\n                raise RuntimeError(\\\'现有实例配置不同；请先停止，再以新配置启动。\\\')\\n            self._bind_bootstrap_host(payload[\\\'request_id\\\'], host)\\n            return self._await_bootstrap(host, public)\\n        clean_terminal = (state.get(\\\'phase\\\') in {\\\'failed\\\', \\\'stopped\\\'} and state.get(\\\'cleanup_complete\\\') is True\\n                          and all(state.get(name) is False for name in (\\\'manager_alive\\\', \\\'server_alive\\\', \\\'model_alive\\\', \\\'tunnel_alive\\\'))\\n                          and not state.get(\\\'process_observation_errors\\\'))\\n        if host and not clean_terminal and (state.get(\\\'observation_stale\\\') or state.get(\\\'phase\\\') != \\\'new\\\'):\\n            raise RuntimeError(\\\'旧生命周期尚未确认清理完成；先停止并检查状态。\\\')\\n        self.scope.pop(\\\'startup_host\\\', None)\\n        self.scope.update(PREFLIGHT_OK=False, INSTALL_OK=False, RESOURCE_REPORT={})\\n        self.requested_options = {name: payload.get(name, \\\'none\\\') for name in (\\\'acceleration\\\', \\\'style\\\')}\\n        self.scope[\\\'BOOTSTRAP_API_KEY\\\'] = secrets.token_urlsafe(32)\\n        self.scope[\\\'API_KEY\\\'] = self.scope[\\\'BOOTSTRAP_API_KEY\\\']\\n        self.scope[\\\'BOOTSTRAP_PUBLIC\\\'] = public\\n        self._event(\\\'credential.generated\\\', purpose=\\\'public_api\\\')\\n        for stage in (\\\'preflight\\\', \\\'install\\\', \\\'start\\\'):\\n            self._check_startup_cancelled()\\n            self._startup_stage(stage)\\n            self._operate(stage, payload)\\n            self._check_startup_cancelled()\\n            if stage == \\\'preflight\\\' and self.scope.get(\\\'PREFLIGHT_OK\\\') is not True:\\n                raise RuntimeError(\\\'资源检查没有提供通过凭据，停止启动。\\\')\\n            if stage == \\\'install\\\' and self.scope.get(\\\'INSTALL_OK\\\') is not True:\\n                raise RuntimeError(\\\'依赖检查没有提供通过凭据，停止启动。\\\')\\n        host = self.scope.get(\\\'startup_host\\\')\\n        if not host:\\n            raise RuntimeError(\\\'总管理未创建，停止启动。\\\')\\n        self._bind_bootstrap_host(payload[\\\'request_id\\\'], host)\\n        return self._await_bootstrap(host, public)\\n\\n    def _bind_bootstrap_host(self, request_id, host):\\n        """Bind observations to the owner accepted by this bootstrap request."""\\n        instance_id = getattr(host, \\\'instance_id\\\', None)\\n        with self.lock:\\n            operation = self.operations.get(request_id)\\n            if not instance_id or not operation or operation.get(\\\'action\\\') != \\\'bootstrap\\\':\\n                raise RuntimeError(\\\'启动请求与实例绑定无效。\\\')\\n            previous = self._bootstrap_hosts.get(request_id)\\n            if previous and (previous[0] is not host or previous[1] != instance_id):\\n                raise RuntimeError(\\\'启动请求不能更换已绑定实例。\\\')\\n            self._bootstrap_hosts[request_id] = (host, instance_id)\\n            operation[\\\'instance_id\\\'] = instance_id\\n        self._event(\\\'startup.host_bound\\\', request_id=request_id, instance_id=instance_id)\\n\\n    def _cancel_bootstrap_and_wait(self):\\n        with self.lock:\\n            record = self.active.get(\\\'setup\\\')\\n            if not record or record[\\\'action\\\'] != \\\'bootstrap\\\':\\n                return\\n            record[\\\'cancel_requested\\\'] = True\\n            self.cancel_startup.set()\\n            process = self.processes.get(record.get(\\\'thread_id\\\'))\\n            finished = self.finished[record[\\\'request_id\\\']]\\n        self._event(\\\'startup.cancel_requested\\\', request_id=record[\\\'request_id\\\'])\\n        if process is not None:\\n            self._terminate_client_process(process)\\n        # Join before stopping the owner: a late start must not resurrect a released service.\\n        if not finished.wait(60):\\n            raise TimeoutError(\\\'启动步骤尚未退出；未声明停止或释放完成，请检查日志后重试。\\\')\\n\\n    def begin_startup_delivery(self, request_id, timeout):\\n        """Observe delivery on the calling cell; never schedule dependent work."""\\n        if isinstance(timeout, bool) or not isinstance(timeout, (int, float)) or not math.isfinite(timeout) or not 0 < timeout <= 600:\\n            raise ValueError(\\\'启动交付观察时间须为 0 到 600 秒之间的有限数值。\\\')\\n        with self.lock:\\n            self.startup_delivery = {\\\'request_id\\\': request_id, \\\'status\\\': \\\'observing\\\', \\\'active\\\': True,\\n                \\\'started_at\\\': time.time(), \\\'deadline_at\\\': time.time() + timeout, \\\'timeout_seconds\\\': timeout,\\n                \\\'instance_id\\\': None, \\\'last_observed_at\\\': None, \\\'api_ready\\\': False, \\\'model_ready\\\': False,\\n                \\\'cleanup_complete\\\': None}\\n        self._event(\\\'startup.delivery_started\\\', request_id=request_id, timeout_seconds=timeout,\\n                    dependency=\\\'current_instance_api_and_selected_public_api; model_ready_not_required\\\')\\n        self._print(\\\'正在交付独立 API；最多观察\\\', timeout,\\n                    \\\'秒，不等待模型。观察期间请用 Colab 本单元的停止按钮取消，程序会回收已启动进程。\\\')\\n\\n    def wait_startup_delivery(self, request_id, *, publish=None):\\n        """A bounded foreground handoff, independent from model preparation.\\n\\n        Expiry only ends observation and retains the worker/owner. Interrupting\\n        the cell explicitly cancels startup, joins it, then reclaims its owned\\n        processes through the same audited stop path as the control panel.\\n        """\\n        delivery = self.startup_delivery\\n        if not delivery or delivery[\\\'request_id\\\'] != request_id or not delivery[\\\'active\\\']:\\n            raise RuntimeError(\\\'启动交付观察身份不匹配。\\\')\\n        deadline = time.monotonic() + delivery[\\\'timeout_seconds\\\']\\n        status, error, last_publish = \\\'observation_failed\\\', None, -float(\\\'inf\\\')\\n        def update():\\n            nonlocal last_publish\\n            if publish:\\n                try:\\n                    publish()\\n                except Exception as caught:\\n                    self._print(\\\'启动快照显示未更新：\\\', type(caught).__name__)\\n            last_publish = time.monotonic()\\n        try:\\n            while True:\\n                with self.lock:\\n                    record = dict(self.operations.get(request_id) or {})\\n                    binding = self._bootstrap_hosts.get(request_id)\\n                if self.cancel_startup.is_set() or record.get(\\\'status\\\') == \\\'cancelled\\\':\\n                    status = \\\'cancelled\\\'\\n                    break\\n                host, expected = binding if binding else (None, None)\\n                # An upgraded controller can still hold a clean, stopped host\\n                # while its new worker validates and replaces it. That previous\\n                # owner\\\'s terminal state cannot finish this request\\\'s handoff.\\n                if (host is not None and self.scope.get(\\\'startup_host\\\') is host\\n                        and getattr(host, \\\'instance_id\\\', None) == expected):\\n                    state = host.snapshot()\\n                    same = bool(expected and state.get(\\\'instance_id\\\') == expected)\\n                    fresh = same and state.get(\\\'observation_stale\\\') is False\\n                    selected_public = bool(getattr(host, \\\'public\\\', False))\\n                    with self.lock:\\n                        delivery.update(instance_id=expected, last_observed_at=state.get(\\\'observed_at\\\'),\\n                                        api_ready=bool(fresh and state.get(\\\'api_ready\\\') is True),\\n                                        model_ready=bool(fresh and state.get(\\\'model_ready\\\') is True))\\n                    if fresh and state.get(\\\'api_ready\\\') is True and (not selected_public or state.get(\\\'public_api_ready\\\') is True):\\n                        status = \\\'api_ready\\\'\\n                        break\\n                    if same and state.get(\\\'phase\\\') in {\\\'failed\\\', \\\'stopped\\\', \\\'stopping\\\'}:\\n                        status, error = \\\'failed\\\', state.get(\\\'error\\\') or \\\'启动生命周期已结束\\\'\\n                        break\\n                if record.get(\\\'status\\\') == \\\'failed\\\':\\n                    status, error = \\\'failed\\\', record.get(\\\'error\\\') or \\\'启动未完成\\\'\\n                    break\\n                if time.monotonic() >= deadline:\\n                    status = \\\'observation_expired\\\'\\n                    break\\n                if time.monotonic() - last_publish >= 2:\\n                    update()\\n                self.cancel_startup.wait(min(.2, max(0, deadline - time.monotonic())))\\n        except KeyboardInterrupt:\\n            status, error = self._interrupt_delivery(delivery, publish=update)\\n            raise\\n        except Exception as caught:\\n            error = str(caught)\\n            self._print(\\\'启动交付观察未完成；后台所有权保留：\\\', caught)\\n        finally:\\n            with self.lock:\\n                delivery.update(status=status, active=False, finished_at=time.time())\\n                if error:\\n                    delivery[\\\'error\\\'] = self._clean(error)[:500]\\n            self._event(\\\'startup.delivery_finished\\\', **dict(delivery))\\n            messages = {\\\'api_ready\\\': \\\'API 交付已确认；模型可继续下载，任务按实际就绪条件处理。\\\',\\n                        \\\'observation_expired\\\': \\\'前台观察到期；后台仍受管，面板可继续查询、取消启动或停止。\\\',\\n                        \\\'cancelled\\\': \\\'启动交付观察已取消；进程清理以停止回执为准。\\\',\\n                        \\\'failed\\\': \\\'启动未完成；控制器及已有后台保留，查看日志后处理。\\\',\\n                        \\\'interrupted\\\': \\\'启动已中断，所拥有进程清理已确认。\\\',\\n                        \\\'interrupted_cleanup_unconfirmed\\\': \\\'启动已中断，但清理未确认；继续保留所有权。\\\'}\\n            self._print(messages.get(status, \\\'前台观察已结束；继续通过面板查询实际状态。\\\'))\\n            update()\\n        return dict(delivery)\\n\\n    def _interrupt_delivery(self, delivery, *, publish=None):\\n        self.cancel_startup.set()\\n        with self.lock:\\n            delivery.update(status=\\\'cancelling\\\', deadline_at=time.time() + 90)\\n        self._event(\\\'startup.delivery_interrupted\\\', request_id=delivery[\\\'request_id\\\'])\\n        self._print(\\\'收到停止本单元请求，正在取消启动并回收已拥有的进程。\\\')\\n        try:\\n            if publish:\\n                publish()\\n        except BaseException:\\n            pass  # Another interrupt/display failure must not skip cleanup.\\n        try:\\n            self._operate(\\\'stop\\\', {})\\n            with self.lock:\\n                delivery[\\\'cleanup_complete\\\'] = True\\n            return \\\'interrupted\\\', None\\n        except BaseException as caught:\\n            with self.lock:\\n                delivery[\\\'cleanup_complete\\\'] = False\\n            self._print(\\\'启动中断后的清理尚未确认；保留控制器，请继续查询或停止：\\\', caught)\\n            return \\\'interrupted_cleanup_unconfirmed\\\', str(caught)\\n\\n    def _operate(self, action, payload):\\n        if action == \\\'bootstrap\\\':\\n            return self._bootstrap(payload)\\n        if action == \\\'preflight\\\':\\n            return self._execute(\\\'preflight\\\', {\\\'STAGED_BF16\\\': payload.get(\\\'staged\\\', False),\\n                                             \\\'MODEL_ALREADY_CACHED\\\': payload.get(\\\'cached\\\', False)})\\n        if action == \\\'install\\\':\\n            self._configuration(payload)\\n            return self._execute(\\\'install\\\')\\n        if action == \\\'start\\\':\\n            self._configuration(payload)\\n            return self._execute(\\\'start\\\', {\\\'START_GPU_API\\\': True, \\\'OPEN_PUBLIC_API\\\': payload.get(\\\'public\\\', False)})\\n        if action == \\\'generate\\\':\\n            host = self.scope.get(\\\'startup_host\\\')\\n            state = host.snapshot() if host else {}\\n            self.scope[\\\'PUBLIC_API_URL\\\'] = state.get(\\\'public_url\\\') if state.get(\\\'public_api_ready\\\') else None\\n            return self._execute(\\\'generate\\\', {\\\'RUN_SMOKE_TEST\\\': True,\\n                \\\'验收接口\\\': \\\'本机API\\\' if payload.get(\\\'endpoint\\\') == \\\'local\\\' else \\\'临时公网\\\',\\n                \\\'RESUME_JOB_ID\\\': payload.get(\\\'resume_job_id\\\', \\\'\\\').strip(),\\n                \\\'TEST_REFERENCE_FILE\\\': payload.get(\\\'reference_file\\\', \\\'\\\'), \\\'TEST_PROMPT\\\': payload.get(\\\'prompt\\\', \\\'\\\')})\\n        if action in {\\\'stop\\\', \\\'release\\\'}:\\n            self._cancel_bootstrap_and_wait()\\n            self._stop_client_monitor()\\n            with self.lock:\\n                remaining = list(self.process_owners.values())\\n            for owner, _ in remaining:\\n                state = owner.terminate_all(timeout=10)\\n                self._event(\\\'client.process_cleanup\\\', **state)\\n                if not state[\\\'cleanup_complete\\\']:\\n                    raise RuntimeError(\\\'仍有客户端子进程，不能声明停止或释放完成。\\\')\\n                with self.lock:\\n                    self.process_owners.pop(id(owner.process), None)\\n            self._execute(\\\'stop\\\', {\\\'STOP_SERVER\\\': True, \\\'RELEASE_COLAB_RUNTIME\\\': False})\\n            if action == \\\'release\\\':\\n                self._event(\\\'runtime.release_requested\\\')\\n                if self.release_callback:\\n                    self.release_callback()\\n                else:\\n                    from google.colab import runtime\\n                    runtime.unassign()\\n            return\\n        if action == \\\'control\\\':\\n            host = self.scope.get(\\\'startup_host\\\')\\n            if not host:\\n                raise RuntimeError(\\\'尚未启动总管理。\\\')\\n            command = payload.get(\\\'action\\\')\\n            if command == \\\'query\\\':\\n                if not payload.get(\\\'command_id\\\', \\\'\\\').strip():\\n                    raise ValueError(\\\'先填写命令 ID。\\\')\\n                result = host.command(payload[\\\'command_id\\\'].strip())\\n            elif command in {\\\'drain\\\', \\\'resume\\\', \\\'stop_model\\\', \\\'start_model\\\', \\\'stop_api\\\', \\\'start_api\\\'}:\\n                result = host.control(command, force=payload.get(\\\'force\\\', False))\\n            else:\\n                raise ValueError(\\\'组件控制动作无效。\\\')\\n            self._print(json.dumps(result, ensure_ascii=False))\\n            return result\\n        raise ValueError(\\\'未知操作。\\\')\\n\\n    def _worker(self, record, lane, action, payload):\\n        with self.lock:\\n            record.update(status=\\\'running\\\', started_at=time.time(), thread_id=threading.get_ident())\\n        self._event(\\\'operation.started\\\', action=action, request_id=record[\\\'request_id\\\'])\\n        try:\\n            result = self._operate(action, payload)\\n            if action == \\\'bootstrap\\\':\\n                self._check_startup_cancelled()\\n            with self.lock:\\n                record.update(status=\\\'succeeded\\\', finished_at=time.time())\\n                if isinstance(result, dict):\\n                    record[\\\'command\\\'] = {k: result[k] for k in (\\\'command_id\\\', \\\'status\\\') if k in result}\\n                message = (\\\'已发起平台释放；是否完成以 Colab 断开为准。\\\' if action == \\\'release\\\'\\n                           else \\\'操作已完成。\\\' if action != \\\'control\\\' else \\\'命令回执已记录；接收不等于执行完成。\\\')\\n                self.notice = None\\n                self._print(ACTION_TITLES[action], message)\\n        except BaseException as error:\\n            cancelled = action == \\\'bootstrap\\\' and self.cancel_startup.is_set()\\n            if action == \\\'bootstrap\\\':\\n                self._startup_stage(\\\'cancelled\\\' if cancelled else \\\'failed\\\')\\n            self._print(str(error) if cancelled else traceback.format_exc())\\n            with self.lock:\\n                record.update(status=\\\'cancelled\\\' if cancelled else \\\'failed\\\', finished_at=time.time(), error=self._clean(error)[:500])\\n                if not cancelled:\\n                    self.notice = {\\\'id\\\': record[\\\'request_id\\\'], \\\'kind\\\': \\\'error\\\', \\\'title\\\': ACTION_TITLES[action] + \\\'未完成\\\',\\n                                   \\\'message\\\': self._clean(error)[:500]}\\n        finally:\\n            self._event(\\\'operation.finished\\\', action=action, request_id=record[\\\'request_id\\\'], status=record[\\\'status\\\'])\\n            with self.lock:\\n                self.active.pop(lane, None)\\n                self.processes.pop(threading.get_ident(), None)\\n                self.finished[record[\\\'request_id\\\']].set()\\n\\n    def submit(self, action, payload):\\n        request_id = payload.get(\\\'request_id\\\')\\n        if not request_id or len(request_id) > 128:\\n            raise ValueError(\\\'操作必须携带唯一 request_id。\\\')\\n        fingerprint = hashlib.sha256(json.dumps([action, payload], sort_keys=True).encode()).hexdigest()\\n        lane = \\\'generate\\\' if action == \\\'generate\\\' else \\\'control\\\' if action in {\\\'stop\\\', \\\'release\\\', \\\'control\\\'} else \\\'setup\\\'\\n        with self.lock:\\n            if self.closed:\\n                raise RuntimeError(\\\'控制台已关闭。\\\')\\n            previous = self.operations.get(request_id)\\n            if previous:\\n                if previous[\\\'fingerprint\\\'] != fingerprint:\\n                    raise ValueError(\\\'同一请求 ID 的内容发生变化。\\\')\\n                return dict(previous)\\n            setup = self.active.get(\\\'setup\\\')\\n            setup_allows_control = setup and setup[\\\'action\\\'] == \\\'bootstrap\\\' and (\\n                action in {\\\'stop\\\', \\\'release\\\'} or self.startup_stage in {\\\'awaiting_api\\\', \\\'awaiting_model\\\', \\\'ready\\\'})\\n            if lane in self.active or (setup and not setup_allows_control) or (lane == \\\'setup\\\' and self.active):\\n                raise RuntimeError(\\\'相关操作尚在执行；可继续观察日志，不重复提交。\\\')\\n            if action == \\\'bootstrap\\\':\\n                self.cancel_startup.clear()\\n                self.notice = None\\n            record = {\\\'request_id\\\': request_id, \\\'action\\\': action, \\\'status\\\': \\\'accepted\\\',\\n                      \\\'accepted_at\\\': time.time(), \\\'fingerprint\\\': fingerprint}\\n            self.operations[request_id] = record\\n            self.active[lane] = record\\n            self.finished[request_id] = threading.Event()\\n            self._event(\\\'operation.accepted\\\', action=action, request_id=request_id, fingerprint=fingerprint)\\n            self.executor.submit(self._worker, record, lane, action, payload)\\n            return dict(record)\\n\\n    def _connection_context(self, state=None):\\n        host = self.scope.get(\\\'startup_host\\\')\\n        with self.lock:\\n            preparing = (self.startup_stage in {\\\'preflight\\\', \\\'install\\\', \\\'start\\\'}\\n                         and any(r[\\\'action\\\'] == \\\'bootstrap\\\' for r in self.active.values()))\\n        if preparing:\\n            host, state = None, {}\\n        if state is None:\\n            state = host.snapshot() if host else {}\\n        if state.get(\\\'instance_id\\\') and state[\\\'instance_id\\\'] != getattr(host, \\\'instance_id\\\', None):\\n            state = {}  # A read raced with replacement of the owned lifecycle.\\n        key = getattr(host, \\\'api_key\\\', None) or self.scope.get(\\\'BOOTSTRAP_API_KEY\\\')\\n        public = bool(state.get(\\\'public_requested\\\', self.scope.get(\\\'BOOTSTRAP_PUBLIC\\\', False)))\\n        fresh = not state.get(\\\'observation_stale\\\', False)\\n        api_ready = state.get(\\\'api_ready\\\') is True and fresh\\n        public_ready = api_ready and state.get(\\\'public_api_ready\\\') is True\\n        local = \\\'http://127.0.0.1:\\\' + str(getattr(host, \\\'port\\\', 8000)) if api_ready else None\\n        candidate = state.get(\\\'public_url\\\') if public else None\\n        url = candidate if public_ready else local if not public else None\\n        connection = {\\\'url\\\': url, \\\'candidate_url\\\': candidate, \\\'local_url\\\': local,\\n                      \\\'api_ready\\\': api_ready, \\\'public_requested\\\': public,\\n                      \\\'public_api_ready\\\': public_ready,\\n                      \\\'model_ready\\\': state.get(\\\'model_ready\\\') is True and fresh,\\n                      \\\'observation_stale\\\': not fresh, \\\'key_available\\\': bool(key),\\n                      \\\'credential_id\\\': hashlib.sha256(key.encode()).hexdigest()[:16] if key else None,\\n                      \\\'instance_id\\\': getattr(host, \\\'instance_id\\\', None)}\\n        return connection, key\\n\\n    def _downloads(self, state):\\n        progress = state.get(\\\'downloads\\\')\\n        if isinstance(progress, dict) and progress.get(\\\'schema_version\\\') == 1:\\n            return progress\\n        items = {}\\n        for name, variable in ((\\\'model\\\', None), (\\\'acceleration\\\', \\\'ACCELERATION_LORA\\\'), (\\\'style\\\', \\\'STYLE_LORA\\\')):\\n            selection = self.requested_options.get(name, self.scope.get(variable, \\\'none\\\')) if variable else None\\n            phase = \\\'waiting\\\' if name == \\\'model\\\' or selection != \\\'none\\\' else \\\'disabled\\\'\\n            items[name] = {\\\'phase\\\': phase, \\\'bytes_completed\\\': 0, \\\'bytes_total\\\': None, \\\'percent\\\': None,\\n                           \\\'files_completed\\\': 0, \\\'files_total\\\': None, \\\'active_files\\\': 0,\\n                           \\\'scope\\\': \\\'observed_transfers\\\' if name == \\\'model\\\' else \\\'asset\\\',\\n                           \\\'detail\\\': \\\'等待模型容器提供真实进度\\\' if phase == \\\'waiting\\\' else \\\'未启用\\\'}\\n        return {\\\'schema_version\\\': 1, \\\'model_instance_id\\\': None, \\\'observed_at\\\': None,\\n                \\\'observation_stale\\\': False, \\\'model_ready\\\': False, \\\'lifecycle\\\': \\\'not_started\\\', \\\'items\\\': items}\\n\\n    def snapshot(self, source=\\\'diagnostics\\\', *, observe_host=True):\\n        host = self.scope.get(\\\'startup_host\\\')\\n        for key in self._secrets():\\n            if key and key not in ui._known_secrets:\\n                ui._known_secrets = (*ui._known_secrets[-4:], key)\\n        try:\\n            # Mounting the panel must not depend on an API/model RPC response.\\n            # The browser requests a fresh observation after the cell returns.\\n            state = (host.snapshot() if observe_host else {\\\'observation_stale\\\': True}) if host else {}\\n        except Exception as error:\\n            state = {\\\'observation_stale\\\': True, \\\'phase\\\': \\\'unavailable\\\'}\\n            self._print(\\\'状态读取失败：\\\', error)\\n        safe = {k: state[k] for k in SAFE_STATE if k in state}\\n        if state.get(\\\'error\\\'):\\n            safe[\\\'error\\\'] = self._clean(state[\\\'error\\\'])[:300]\\n        if source == \\\'events\\\':\\n            text = json.dumps(host.events(limit=35), ensure_ascii=False, indent=2) if host else \\\'尚无运行实例。\\\'\\n        elif source in LOG_NAMES:\\n            text = ui.log_tail(getattr(host, LOG_NAMES[source]), 16000) if host else \\\'尚无运行实例。\\\'\\n        elif source == \\\'install\\\':\\n            text = ui.log_tail(self.directory / \\\'install.log\\\', 16000)\\n        elif source == \\\'preflight\\\':\\n            text = ui.log_tail(self.directory / \\\'preflight.log\\\', 16000)\\n        elif source == \\\'state\\\':\\n            text = json.dumps(self._sanitize(safe), ensure_ascii=False, indent=2, sort_keys=True) if host else \\\'尚无运行实例。\\\'\\n        elif source == \\\'smoke\\\':\\n            smoke_log = self.scope.get(\\\'smoke_log\\\')\\n            text = (ui.log_tail(smoke_log, 16000) if smoke_log else \\\'\\\') or \\\'暂无生成验收日志。\\\'\\n        else:\\n            text = ui.log_tail(self.log_path, 10000) + \\\'\\\\n\\\' + ui.last_diagnostics()[-4000:]\\n        with self.lock:\\n            active = [{k: v for k, v in record.items() if k not in {\\\'fingerprint\\\', \\\'thread_id\\\'}} for record in self.active.values()]\\n            notice = dict(self.notice) if self.notice else None\\n        resource = self.scope.get(\\\'RESOURCE_REPORT\\\') or {}\\n        bundle = self.scope.get(\\\'bundle\\\')\\n        connection, _ = self._connection_context(state)\\n        return self._sanitize({\\\'state\\\': safe, \\\'resources\\\': {k: resource[k] for k in (\\\'gpu\\\', \\\'available_ram_gib\\\', \\\'residency\\\') if k in resource},\\n                \\\'connection\\\': connection,\\n                \\\'downloads\\\': self._downloads(state),\\n                \\\'ui\\\': {\\\'connected\\\': not self.closed, \\\'closed\\\': self.closed, \\\'busy\\\': bool(active),\\n                       \\\'operation\\\': \\\', \\\'.join(ACTION_TITLES.get(r[\\\'action\\\'], r[\\\'action\\\']) for r in active),\\n                       \\\'operations\\\': active, \\\'preflight_ok\\\': self.scope.get(\\\'PREFLIGHT_OK\\\', False),\\n                       \\\'install_ok\\\': self.scope.get(\\\'INSTALL_OK\\\', False),\\n                       \\\'startup_stage\\\': self.startup_stage,\\n                       \\\'startup_delivery\\\': dict(self.startup_delivery) if self.startup_delivery else None,\\n                       \\\'bootstrap_running\\\': any(r[\\\'action\\\'] == \\\'bootstrap\\\' for r in active),\\n                       \\\'connection_available\\\': connection[\\\'key_available\\\']},\\n                \\\'logs\\\': {\\\'text\\\': self._clean(text)[-16000:], \\\'source\\\': source, \\\'observed_at\\\': time.time()},\\n                \\\'notice\\\': notice, \\\'result\\\': {\\\'available\\\': bool(bundle and Path(bundle).is_file())}})\\n\\n    def _download(self, action):\\n        if action == \\\'export_connection\\\':\\n            connection, key = self._connection_context()\\n            if not key:\\n                raise RuntimeError(\\\'密钥尚未生成；先执行一键启动。\\\')\\n            data = json.dumps({**connection, \\\'api_key\\\': key,\\n                               \\\'note\\\': \\\'url 为空时尚不可连接；API 可达后重新导出。模型未就绪时任务只排队。\\\'}, ensure_ascii=False).encode()\\n            name, mime = \\\'api-connection.private.json\\\', \\\'application/json\\\'\\n        else:\\n            path = self.scope.get(\\\'bundle\\\')\\n            if not path or not Path(path).is_file():\\n                raise RuntimeError(\\\'本次尚无已验证结果包。\\\')\\n            path = Path(path)\\n            if path.stat().st_size > 32 * 1024**2:\\n                raise ValueError(\\\'结果包超过内联下载上限；请使用 Colab 文件栏下载。\\\')\\n            data, name, mime = path.read_bytes(), path.name, \\\'application/zip\\\'\\n        return {\\\'name\\\': name, \\\'mime\\\': mime, \\\'data_base64\\\': base64.b64encode(data).decode(\\\'ascii\\\')}\\n\\n    def dispatch(self, action, payload=None):\\n        try:\\n            payload = validated(payload or {})\\n            if self.closed:\\n                return {\\\'ui\\\': {\\\'connected\\\': False, \\\'closed\\\': True}, \\\'error\\\': \\\'控制台已关闭。\\\'}\\n            if action in {\\\'snapshot\\\', \\\'refresh_board\\\'}:\\n                return self.snapshot(payload.get(\\\'source\\\', \\\'diagnostics\\\'))\\n            if action == \\\'reveal_connection\\\':\\n                result = self.snapshot(payload.get(\\\'source\\\', \\\'diagnostics\\\'))\\n                connection, key = self._connection_context()\\n                if not key:\\n                    raise RuntimeError(\\\'密钥尚未生成；先执行一键启动。\\\')\\n                if payload.get(\\\'credential_id\\\') != connection[\\\'credential_id\\\']:\\n                    raise RuntimeError(\\\'连接凭据已改变；请刷新看板后重新显示或复制。\\\')\\n                result[\\\'connection\\\'] = self._sanitize(connection)\\n                # This dedicated user-triggered response is never embedded into saved HTML or logs.\\n                result[\\\'connection_secret\\\'] = {\\\'api_key\\\': key, \\\'credential_id\\\': connection[\\\'credential_id\\\']}\\n                self._event(\\\'credential.revealed\\\', credential_id=connection[\\\'credential_id\\\'])\\n                return result\\n            if action in {\\\'export_connection\\\', \\\'download_result\\\'}:\\n                result = self.snapshot(payload.get(\\\'source\\\', \\\'diagnostics\\\'))\\n                result[\\\'download\\\'] = self._download(action)\\n                self._event(\\\'download.prepared\\\', action=action)\\n                return result\\n            if action not in ACTION_TITLES:\\n                raise ValueError(\\\'未知控制动作。\\\')\\n            receipt = self.submit(action, payload)\\n            return {\\\'accepted\\\': True, \\\'operation\\\': {k: v for k, v in receipt.items() if k not in {\\\'fingerprint\\\', \\\'thread_id\\\'}}}\\n        except Exception as error:\\n            return {\\\'error\\\': self._clean(error)[:500], \\\'notice\\\': {\\\'id\\\': uuid.uuid4().hex,\\n                \\\'kind\\\': \\\'error\\\', \\\'title\\\': \\\'操作未执行\\\', \\\'message\\\': self._clean(error)[:500]}}\\n\\n    def close(self):\\n        with self.lock:\\n            if self.active or self.process_owners:\\n                raise RuntimeError(\\\'仍有受管操作，不能丢弃控制器。\\\')\\n            self.closed = True\\n            self.epoch = None\\n        self.executor.shutdown(wait=True, cancel_futures=True)\\n        if ui._dashboard is self:\\n            ui._dashboard = None\\n        if getattr(ui, \\\'_process_observer\\\', None) is self:\\n            ui._process_observer = None\\n        self._event(\\\'console.closed\\\')\\n\\n\\ndef render_console(template, callback_name=\\\'\\\', initial=None):\\n    state = initial or {\\\'ui\\\': {\\\'connected\\\': False, \\\'offline\\\': True}, \\\'state\\\': {}, \\\'resources\\\': {},\\n                        \\\'logs\\\': {\\\'text\\\': \\\'尚未连接控制台。运行本单元后，日志在这里持续更新。\\\', \\\'source\\\': \\\'diagnostics\\\'}}\\n    safe_json = lambda data: json.dumps(data, ensure_ascii=False).replace(\\\'<\\\', \\\'\\\\\\\\u003c\\\').replace(\\\'>\\\', \\\'\\\\\\\\u003e\\\').replace(\\\'&\\\', \\\'\\\\\\\\u0026\\\')\\n    return template.replace(\\\'__CALLBACK_NAME__\\\', safe_json(callback_name)).replace(\\\'__INITIAL_STATE__\\\', safe_json(state))\\n\\n\\ndef mount_console(directory, programs, scope=None, *, auto_start=False, bootstrap_payload=None,\\n                  foreground_bootstrap_timeout=None):\\n    """Display the unchanged panel, optionally holding the cell until API delivery.\\n\\n    The owner still runs independent startup branches concurrently. The finite\\n    foreground observer never waits for the model or treats expiry as failure.\\n    """\\n    from google.colab import output\\n    from IPython.display import HTML, JSON, display\\n    scope = scope if scope is not None else {}\\n    controller = scope.get(\\\'h3_console\\\')\\n    created = False\\n    if not isinstance(controller, ConsoleController) or controller.closed:\\n        controller = ConsoleController(directory, programs, scope)\\n        scope[\\\'h3_console\\\'] = controller\\n        created = True\\n    controller.epoch = uuid.uuid4().hex\\n    epoch = controller.epoch\\n    name = \\\'h3.console.\\\' + epoch\\n    def callback(action, payload=None):\\n        if controller.epoch != epoch:\\n            return JSON({\\\'ui\\\': {\\\'connected\\\': False, \\\'closed\\\': True}, \\\'error\\\': \\\'旧面板已停用。\\\'})\\n        return JSON(controller.dispatch(action, payload))\\n    output.register_callback(name, callback)\\n    template = (Path(directory) / \\\'console.html\\\').read_text(encoding=\\\'utf-8\\\')\\n    request_id = \\\'mount-\\\' + epoch\\n    foreground = created and auto_start and foreground_bootstrap_timeout is not None\\n    active_delivery = lambda: foreground and bool(controller.startup_delivery and controller.startup_delivery.get(\\\'active\\\'))\\n    try:\\n        if foreground:\\n            controller.begin_startup_delivery(request_id, foreground_bootstrap_timeout)\\n        initial = HTML(render_console(template, name, controller.snapshot(observe_host=False)))\\n        handle = display(initial, display_id=True) if foreground else display(initial)\\n        if created and auto_start:\\n            payload = {\\\'public\\\': True, **(bootstrap_payload or {}), \\\'request_id\\\': request_id}\\n            receipt = controller.dispatch(\\\'bootstrap\\\', payload)\\n            if foreground:\\n                def publish():\\n                    if callable(getattr(handle, \\\'update\\\', None)):\\n                        handle.update(HTML(render_console(template, name, controller.snapshot())))\\n                if receipt.get(\\\'accepted\\\') is not True:\\n                    with controller.lock:\\n                        controller.startup_delivery.update(active=False, status=\\\'failed\\\',\\n                            error=receipt.get(\\\'error\\\', \\\'启动未被接收\\\'), finished_at=time.time())\\n                    publish()\\n                else:\\n                    controller.wait_startup_delivery(request_id, publish=publish)\\n    except KeyboardInterrupt:\\n        # Cover the display/dispatch window before wait_startup_delivery owns\\n        # interrupts. Its own handler sets active=False, preventing double stop.\\n        if active_delivery():\\n            status, error = controller._interrupt_delivery(controller.startup_delivery)\\n            controller.startup_delivery.update(status=status, error=error, active=False, finished_at=time.time())\\n            controller._event(\\\'startup.delivery_finished\\\', **dict(controller.startup_delivery))\\n        raise\\n    finally:\\n        if active_delivery():\\n            controller.startup_delivery.update(status=\\\'observation_failed\\\', active=False, finished_at=time.time())\\n            controller._event(\\\'startup.delivery_finished\\\', **dict(controller.startup_delivery))\\n    return controller\\n\', \'console.html\': \'<!doctype html>\\n<html lang="zh-CN">\\n<head>\\n<meta charset="utf-8">\\n<meta name="viewport" content="width=device-width,initial-scale=1">\\n<title>MiniMax-H3 控制台</title>\\n<style>\\n:root {\\n  color-scheme: light dark;\\n  --canvas: #f1f2f5;\\n  --panel: #fff;\\n  --surface: #f5f5f7;\\n  --input: #f7f7f9;\\n  --toolbar: #fafafc;\\n  --ink: #1d1d1f;\\n  --muted: #5f636d;\\n  --line: #dedfe5;\\n  --edge: #2226341a;\\n  --accent: #0065d8;\\n  --accent-hover: #005bc4;\\n  --tint: #eaf2ff;\\n  --ready: #187641;\\n  --danger: #b32634;\\n  --warn: #80540e;\\n  --log: #f7f8fb;\\n  --scroll-thumb: #bfc1ca;\\n  --shadow: 0 1px 2px #20283a08, 0 5px 14px #20283a0b;\\n  --mono: ui-monospace, "SFMono-Regular", Consolas, "Microsoft YaHei", monospace;\\n}\\n\\n@media (prefers-color-scheme: dark) {\\n  :root {\\n    --canvas: #16171b;\\n    --panel: #222327;\\n    --surface: #2b2c31;\\n    --input: #292a30;\\n    --toolbar: #25262b;\\n    --ink: #f2f2f5;\\n    --muted: #b3b5bf;\\n    --line: #3c3e46;\\n    --edge: #ffffff20;\\n    --accent: #78b6ff;\\n    --accent-hover: #9ac9ff;\\n    --tint: #263953;\\n    --ready: #73d69a;\\n    --danger: #ff9398;\\n    --warn: #edc278;\\n    --log: #1c1d22;\\n    --scroll-thumb: #646773;\\n    --shadow: 0 1px 2px #0004, 0 5px 14px #0003;\\n  }\\n}\\n\\n* { box-sizing: border-box; }\\nbody {\\n  margin: 0;\\n  color: var(--ink);\\n  font: 13px/1.5 -apple-system, BlinkMacSystemFont, "Segoe UI Variable Text", "Segoe UI", "PingFang SC", "Microsoft YaHei", system-ui, sans-serif;\\n}\\nh1, h2, p { margin: 0; }\\nbutton, input, textarea, select { font: inherit; }\\nbutton, input[type="checkbox"], select, summary { cursor: pointer; }\\nbutton {\\n  min-height: 33px;\\n  padding: 5px 10px;\\n  border: 1px solid var(--edge);\\n  border-radius: 9px;\\n  background: var(--panel);\\n  color: var(--ink);\\n  font-weight: 600;\\n  overflow-wrap: anywhere;\\n}\\nbutton:hover:not(:disabled) { background: var(--surface); }\\nbutton:active:not(:disabled) { box-shadow: inset 0 1px 3px #0002; }\\nbutton:disabled { cursor: default; color: var(--muted); opacity: .8; }\\nbutton.primary { border-color: transparent; background: var(--accent); color: light-dark(#fff, #102640); }\\nbutton.primary:hover:not(:disabled) { background: var(--accent-hover); }\\nbutton.primary:disabled { background: var(--tint); color: var(--accent); opacity: 1; }\\nbutton.danger { color: var(--danger); }\\n:is(button, input, textarea, select, summary, a, .board, .control, .log-viewport):focus-visible {\\n  outline: 2px solid var(--accent);\\n  outline-offset: 2px;\\n}\\n:is(.board, .control, .log-viewport):focus-visible { outline-offset: -3px; }\\ninput:not([type="checkbox"]), textarea, select {\\n  width: 100%;\\n  min-width: 0;\\n  min-height: 34px;\\n  padding: 5px 8px;\\n  border: 1px solid var(--line);\\n  border-radius: 8px;\\n  background: var(--input);\\n  color: var(--ink);\\n}\\ninput::placeholder, textarea::placeholder { color: var(--muted); opacity: 1; }\\ntextarea { min-height: 52px; max-height: 112px; resize: vertical; }\\ninput[type="checkbox"] {\\n  width: 15px;\\n  height: 15px;\\n  flex: 0 0 auto;\\n  margin: 0;\\n  accent-color: var(--accent);\\n}\\nlabel { display: block; color: var(--muted); font-size: 12px; }\\nlabel > :is(input, textarea, select) { margin-top: 4px; }\\n\\n.app {\\n  display: grid;\\n  grid-template-columns: repeat(2, minmax(0, 1fr));\\n  gap: 12px;\\n  width: 100%;\\n  height: 686px;\\n  padding: 10px;\\n  border-radius: 22px;\\n  background: var(--canvas);\\n}\\n.left {\\n  display: grid;\\n  grid-template-rows: 392px minmax(0, 1fr);\\n  gap: 12px;\\n  min-width: 0;\\n  min-height: 0;\\n}\\n.panel {\\n  min-width: 0;\\n  min-height: 0;\\n  overflow: hidden;\\n  border: 1px solid var(--edge);\\n  border-radius: 16px;\\n  background: var(--panel);\\n  box-shadow: var(--shadow);\\n}\\n.panel-head {\\n  display: flex;\\n  align-items: center;\\n  justify-content: space-between;\\n  gap: 8px;\\n  flex-shrink: 0;\\n  padding: 11px 15px;\\n  border-bottom: 1px solid var(--line);\\n}\\n.panel-head h2 { font-size: 16px; line-height: 1.3; font-weight: 650; }\\n.eyebrow { color: var(--muted); font-size: 11px; font-weight: 550; }\\n:is(.board, .control, .log-viewport, .notice-zone) {\\n  scrollbar-width: thin;\\n  scrollbar-color: var(--scroll-thumb) transparent;\\n}\\n\\n.board {\\n  display: flex;\\n  flex-direction: column;\\n  padding: 12px 15px 10px;\\n  overflow-y: auto;\\n  overscroll-behavior: contain;\\n  scrollbar-gutter: stable;\\n}\\n.board-top { display: flex; align-items: flex-start; justify-content: space-between; gap: 8px; }\\n.board-top > div { min-width: 0; }\\n.board h1 { margin-top: 1px; font-size: 22px; line-height: 1.25; font-weight: 650; overflow-wrap: anywhere; }\\n.board[data-tone="error"] h1 { color: var(--danger); }\\n.board[data-tone="warn"] h1 { color: var(--warn); }\\n.board[data-tone="ready"] h1 { color: var(--ready); }\\n.board-refresh { min-height: 26px; padding: 3px 8px; flex-shrink: 0; font-size: 11px; }\\n.board-message { color: var(--muted); font-size: 11px; overflow-wrap: anywhere; }\\n.capabilities {\\n  display: grid;\\n  grid-template-columns: repeat(4, minmax(0, 1fr));\\n  gap: 1px;\\n  flex-shrink: 0;\\n  margin: 9px 0;\\n  overflow: hidden;\\n  border: 1px solid var(--line);\\n  border-radius: 10px;\\n  background: var(--line);\\n}\\n.capability { min-width: 0; padding: 5px 8px; background: var(--surface); overflow-wrap: anywhere; }\\n.capability span { display: block; color: var(--muted); font-size: 10px; line-height: 1.4; }\\n.capability strong { display: block; font-size: 12px; line-height: 1.4; font-weight: 650; }\\n.connection-box { flex-shrink: 0; }\\n.connection-row {\\n  display: grid;\\n  grid-template-columns: 53px minmax(0, 1fr) auto;\\n  gap: 5px;\\n  align-items: center;\\n  margin-bottom: 4px;\\n}\\n.connection-row.key-row { grid-template-columns: 53px minmax(0, 1fr) auto auto; }\\n.connection-row label { font-size: 11px; white-space: nowrap; }\\n.connection-row input { height: 27px; min-height: 27px; padding: 3px 7px; font: 11px/1.3 var(--mono); }\\n.connection-row button { min-height: 27px; padding: 3px 7px; font-size: 11px; }\\n.connection-hint { color: var(--muted); font-size: 10px; overflow-wrap: anywhere; }\\n.transfers { flex-shrink: 0; margin: 7px 0 6px; border-top: 1px solid var(--line); }\\n.transfer-row {\\n  position: relative;\\n  display: grid;\\n  grid-template-columns: 69px minmax(0, 1fr) 40px;\\n  gap: 0 5px;\\n  min-height: 34px;\\n  padding: 4px 0;\\n  border-bottom: 1px solid var(--line);\\n  overflow-wrap: anywhere;\\n}\\n.transfer-name { grid-row: 1 / 3; align-self: center; font-size: 11px; font-weight: 650; }\\n.transfer-phase { font-size: 11px; line-height: 1.35; }\\n.transfer-number { color: var(--muted); font-size: 10px; text-align: right; font-variant-numeric: tabular-nums; }\\n.transfer-detail { grid-column: 2 / 4; color: var(--muted); font-size: 10px; line-height: 1.35; }\\n.transfer-row progress {\\n  appearance: none;\\n  position: absolute;\\n  inset: auto 0 0;\\n  width: 100%;\\n  height: 3px;\\n  overflow: hidden;\\n  border: 0;\\n  border-radius: 99px;\\n  background: var(--line);\\n}\\n.transfer-row progress::-webkit-progress-bar { background: var(--line); }\\n.transfer-row progress::-webkit-progress-value { background: var(--accent); }\\n.transfer-row progress::-moz-progress-bar { background: var(--accent); }\\n.transfer-row[data-phase="failed"] .transfer-phase { color: var(--danger); }\\n.transfer-row[data-phase="disabled"] .transfer-phase { color: var(--muted); }\\n#resources { font-size: 10px; overflow-wrap: anywhere; }\\n.board-bottom {\\n  display: flex;\\n  justify-content: space-between;\\n  align-items: center;\\n  flex: 0 0 auto;\\n  flex-wrap: wrap;\\n  gap: 4px 8px;\\n  margin-top: auto;\\n  padding-top: 5px;\\n  font-size: 10px;\\n}\\n.queue { padding: 2px 7px; border-radius: 99px; background: var(--surface); font-variant-numeric: tabular-nums; }\\n.stamp { color: var(--muted); }\\n\\n.control {\\n  overflow: auto;\\n  overscroll-behavior: contain;\\n  scrollbar-gutter: stable;\\n}\\n.control-actions {\\n  display: grid;\\n  gap: 6px;\\n  padding: 8px 12px;\\n}\\n.control-actions button { min-height: 31px; padding: 4px 6px; font-size: 12px; }\\n.control-body { padding: 4px 12px 13px; }\\n.button-row { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 6px; }\\n.button-row.two { grid-template-columns: repeat(2, minmax(0, 1fr)); }\\n.checkline { display: flex; align-items: center; gap: 8px; margin: 3px 0 10px; color: var(--ink); }\\n.checkline input { margin: 0; }\\n#public {\\n  appearance: none;\\n  position: relative;\\n  width: 31px;\\n  height: 18px;\\n  border: 1px solid var(--line);\\n  border-radius: 99px;\\n  background: var(--surface);\\n}\\n#public::before {\\n  position: absolute;\\n  inset: 2px auto 2px 2px;\\n  aspect-ratio: 1;\\n  border-radius: 50%;\\n  background: #fff;\\n  box-shadow: 0 1px 3px #0004;\\n  content: "";\\n}\\n#public:checked { border-color: transparent; background: var(--accent); }\\n#public:checked::before { inset-inline: auto 2px; }\\n@media (forced-colors: active) {\\n  #public { appearance: auto; width: 15px; height: 15px; }\\n  #public::before { content: none; }\\n}\\n.form-grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 8px; }\\n.full { grid-column: 1 / -1; }\\n.field-note { margin-top: 5px; color: var(--muted); font-size: 11px; overflow-wrap: anywhere; }\\n#download-result { margin-top: 9px; }\\n.advanced { margin-top: 10px; border-top: 1px solid var(--line); }\\n.advanced summary { padding: 8px 0 4px; font-size: 12px; font-weight: 650; }\\n.advanced h3 { margin: 10px 0 6px; font-size: 12px; font-weight: 650; }\\n.advanced .checkline { margin: 8px 0; }\\n\\n.right { display: flex; flex-direction: column; }\\n.right .panel-head > span { padding: 3px 8px; border-radius: 99px; background: var(--surface); }\\n.status-dot { display: inline-block; width: 7px; height: 7px; margin-right: 6px; border-radius: 50%; background: var(--muted); }\\n.status-dot.live { background: var(--ready); }\\n.log-controls {\\n  display: flex;\\n  flex: 0 0 auto;\\n  flex-wrap: wrap;\\n  align-items: center;\\n  gap: 6px;\\n  padding: 8px 12px;\\n  border-bottom: 1px solid var(--line);\\n  background: var(--toolbar);\\n}\\n.log-controls :is(button, select) { min-height: 30px; padding: 4px 8px; font-size: 12px; }\\n.log-controls .log-source { flex: 1 1 9rem; min-width: 0; max-width: 16rem; }\\n.log-controls .log-lines { flex: 0 1 auto; width: auto; max-width: 100%; }\\n.follow { display: flex; align-items: center; gap: 5px; margin-left: auto; white-space: nowrap; font-size: 11px; }\\n.follow input { margin: 0; }\\n.notice-zone { flex: 0 0 auto; max-height: 150px; padding: 0 12px; overflow: auto; overscroll-behavior: contain; }\\n.notice { margin-top: 9px; padding: 9px 11px; border: 1px solid var(--line); border-left: 3px solid var(--accent); border-radius: 9px; background: var(--tint); }\\n.notice[data-kind="error"] { border-left-color: var(--danger); background: var(--surface); }\\n.notice[data-kind="warning"] { border-left-color: var(--warn); background: var(--surface); }\\n.notice-title { font-size: 12px; font-weight: 650; }\\n.notice-message { margin-top: 4px; color: var(--muted); font-size: 12px; white-space: pre-wrap; overflow-wrap: anywhere; }\\n.notice-actions { display: flex; align-items: center; flex-wrap: wrap; gap: 7px; margin-top: 7px; }\\n.notice-actions :is(button, a) { min-height: 30px; padding: 4px 9px; font-size: 12px; }\\n.notice-actions a { border: 1px solid var(--line); border-radius: 8px; color: var(--accent); background: var(--panel); text-decoration: none; }\\n.notice-actions a:hover { text-decoration: underline; }\\n.log-viewport {\\n  flex: 1;\\n  min-height: 0;\\n  margin: 10px 12px 0;\\n  overflow: auto;\\n  overscroll-behavior: contain;\\n  scrollbar-gutter: stable;\\n  border: 1px solid var(--line);\\n  border-radius: 11px;\\n  background: var(--log);\\n}\\n.log-intro { padding: 12px 13px; border-bottom: 1px solid var(--line); background: var(--toolbar); font-size: 12px; overflow-wrap: anywhere; }\\n.log-intro h3 { margin: 0 0 4px; font-size: 14px; font-weight: 650; }\\n.log-intro p { color: var(--muted); }\\n.log-viewport pre { margin: 0; padding: 11px 13px; color: var(--muted); font: 12px/1.65 var(--mono); white-space: pre-wrap; overflow-wrap: anywhere; }\\n.log-footer { display: flex; justify-content: space-between; flex: 0 0 auto; flex-wrap: wrap; gap: 4px 8px; padding: 8px 13px; color: var(--muted); font-size: 10px; }\\n.sr-only { position: absolute; width: 1px; height: 1px; padding: 0; margin: -1px; overflow: hidden; clip: rect(0, 0, 0, 0); white-space: nowrap; border: 0; }\\n\\n@media (max-width: 920px) {\\n  .follow { margin-left: 0; }\\n  .capability { padding-inline: 5px; }\\n}\\n@media (max-width: 759px) {\\n  .app { height: auto; grid-template-columns: minmax(0, 1fr); padding: 8px; gap: 10px; }\\n  .left { grid-template-rows: auto 390px; gap: 10px; }\\n  .board { scrollbar-gutter: auto; }\\n  .right { height: 440px; }\\n  .log-controls .log-source { max-width: none; }\\n}\\n@media (max-width: 379px) {\\n  .capabilities, .control-actions .button-row, .advanced .button-row { grid-template-columns: repeat(2, minmax(0, 1fr)); }\\n  .form-grid { grid-template-columns: minmax(0, 1fr); }\\n}\\n</style>\\n</head>\\n<body>\\n<main class="app" id="h3-console" aria-label="MiniMax-H3 控制台">\\n  <div class="left">\\n    <section class="panel board" id="board" aria-labelledby="headline" tabindex="0">\\n      <div class="board-top"><div><p class="eyebrow">消息看板</p><h1 id="headline">尚未读取状态</h1></div><button type="button" class="board-refresh" id="refresh-board" title="只刷新状态，不执行控制命令">刷新</button></div>\\n      <p class="board-message" id="next-step">运行控制台单元后显示本次观察。</p>\\n      <div class="capabilities" aria-label="独立能力">\\n        <div class="capability"><span>总管理</span><strong id="manager-state">尚未读取</strong></div>\\n        <div class="capability"><span>API 后台</span><strong id="api-state">尚未读取</strong></div>\\n        <div class="capability"><span>模型容器</span><strong id="model-state">尚未读取</strong></div>\\n        <div class="capability"><span>公网 API</span><strong id="public-state">尚未读取</strong></div>\\n      </div>\\n      <div class="connection-box" aria-label="API 连接信息">\\n        <div class="connection-row"><label for="api-url">API URL</label><input id="api-url" readonly value="尚未分配" autocomplete="off" spellcheck="false" aria-describedby="connection-hint"><button type="button" id="copy-url" disabled aria-label="复制 API URL">复制</button></div>\\n        <div class="connection-row key-row"><label for="api-key">API 密钥</label><input id="api-key" readonly value="尚未生成" autocomplete="off" spellcheck="false" aria-label="API 密钥"><button type="button" id="toggle-key" disabled aria-label="显示 API 密钥">显示</button><button type="button" id="copy-key" disabled aria-label="复制 API 密钥">复制</button></div>\\n        <p class="connection-hint" id="connection-hint">API 可达后可复制地址；密钥按需显示。</p>\\n      </div>\\n      <div class="transfers" aria-label="模型与 LoRA 下载进度">\\n        <div class="transfer-row" id="transfer-model"><span class="transfer-name">模型</span><span class="transfer-phase" id="model-download-phase">尚未观察</span><span class="transfer-number" id="model-download-number">—</span><span class="transfer-detail" id="model-download-detail">下载与加载分别显示</span><progress id="model-download-progress" max="100" aria-label="模型下载进度"></progress></div>\\n        <div class="transfer-row" id="transfer-acceleration"><span class="transfer-name">加速 LoRA</span><span class="transfer-phase" id="acceleration-download-phase">尚未观察</span><span class="transfer-number" id="acceleration-download-number">—</span><span class="transfer-detail" id="acceleration-download-detail">未启用时不下载</span><progress id="acceleration-download-progress" max="100" aria-label="加速 LoRA 下载进度"></progress></div>\\n        <div class="transfer-row" id="transfer-style"><span class="transfer-name">风格 LoRA</span><span class="transfer-phase" id="style-download-phase">尚未观察</span><span class="transfer-number" id="style-download-number">—</span><span class="transfer-detail" id="style-download-detail">未启用时不下载</span><progress id="style-download-progress" max="100" aria-label="风格 LoRA 下载进度"></progress></div>\\n      </div>\\n      <p class="eyebrow" id="resources">配置：A100 · 原生 BF16</p>\\n      <div class="board-bottom"><span class="queue" id="queue-state">排队 —\\u3000执行 —</span><span class="stamp" id="board-stamp">尚无服务观察</span></div>\\n    </section>\\n    <section class="panel control" id="control-scroll" aria-labelledby="control-heading" tabindex="0">\\n      <header class="panel-head"><h2 id="control-heading">常用控制</h2><span class="eyebrow">启动后自动推进</span></header>\\n      <div class="control-actions">\\n        <div class="button-row two"><button type="button" data-action="bootstrap" class="primary">一键启动</button><button type="button" data-action="export_connection">导出 API 连接</button></div>\\n        <div class="button-row"><button type="button" data-action="generate">提交 / 查询任务</button><button type="button" data-action="stop">停止服务</button><button type="button" class="danger" data-action="release">释放云实例</button></div>\\n      </div>\\n      <div class="control-body">\\n        <label class="checkline"><input id="public" type="checkbox" checked>开放临时公网 API</label>\\n        <div class="form-grid">\\n          <label class="full">本次参考图路径<input id="reference_file" placeholder="/content/本次新参考图.png" autocomplete="off"></label>\\n          <label class="full">本次视频描述<textarea id="prompt" rows="2" placeholder="描述本次全新内容；按已有任务 ID 恢复时可留空"></textarea></label>\\n          <label>已有任务 ID<input id="resume_job_id" placeholder="留空则创建新任务" autocomplete="off"></label>\\n          <label>验收接口<select id="endpoint"><option value="public">临时公网</option><option value="local">本机 API</option></select></label>\\n        </div>\\n        <button type="button" data-action="download_result" id="download-result" disabled>下载最近结果</button>\\n        <details class="advanced" id="advanced-controls"><summary>高级控制 · 不常用时收起</summary><div class="advanced-body">\\n          <h3>分步维护</h3>\\n          <div class="button-row"><button type="button" data-action="preflight">检查资源</button><button type="button" data-action="install">安装依赖</button><button type="button" data-action="start">单独启动</button></div>\\n          <p class="field-note">正常使用只需一键启动；检查、安装或启动失败时在日志区处理。</p>\\n          <label class="checkline"><input id="staged" type="checkbox">使用 40GB 分阶段驻留</label>\\n          <label class="checkline"><input id="cached" type="checkbox">权重已完整缓存</label>\\n          <h3>性能与 LoRA</h3>\\n          <div class="form-grid">\\n            <label>加速 LoRA<select id="acceleration"><option value="none">不启用</option><option value="lightx2v-ref2va-4step">Ref2VA Turbo · 实验</option></select></label>\\n            <label>风格 LoRA<select id="style"><option value="none">不启用</option><option value="fal-realism-people">人物写实 · 实验</option></select></label>\\n            <label>风格强度<input id="style_scale" type="number" min="0.1" max="1" step="0.1" value="1"></label>\\n            <label>注意力后端<select id="attention"><option value="auto">自动</option><option value="native">原生</option><option value="_native_flash">原生 Flash · 实验</option></select></label>\\n            <label>去噪卸载<select id="offload"><option value="stream">流式</option><option value="sync">同步 · 实验</option></select></label>\\n          </div><p class="field-note">高级参数在安装与启动前设置；保持 BF16，LoRA 默认关闭。</p>\\n          <h3>组件控制</h3>\\n          <div class="form-grid"><label>操作<select id="component-action"><option value="drain">暂停接收新任务</option><option value="resume">恢复接收新任务</option><option value="stop_model">停止模型</option><option value="start_model">启动模型</option><option value="stop_api">停止 API</option><option value="start_api">启动 API</option></select></label><label>命令 ID<input id="command_id" placeholder="查询已有命令" autocomplete="off"></label></div>\\n          <label class="checkline"><input id="force" type="checkbox">强制停止模型</label>\\n          <div class="button-row two"><button type="button" id="send-control">执行组件操作</button><button type="button" id="query-command">查询命令</button></div>\\n          <p class="field-note">暂停接收时，已有队列继续执行。命令接收不等于执行完成。</p>\\n          <h3>客户端连接</h3><p class="field-note">启动时自动生成 API 密钥，可用上方按钮导出。地址为空表示尚不可连接；API 可达后重新导出，密钥不变。模型准备期间接受排队，完整就绪后自动处理。</p>\\n        </div></details>\\n      </div>\\n    </section>\\n  </div>\\n  <section class="panel right" aria-labelledby="logs-heading">\\n    <header class="panel-head"><h2 id="logs-heading">动态日志</h2><span class="eyebrow"><i class="status-dot" id="live-dot" aria-hidden="true"></i><span id="connection-state">等待连接</span></span></header>\\n    <div class="log-controls">\\n      <label class="sr-only" for="log-source">日志来源</label>\\n      <select id="log-source" class="log-source">\\n        <option value="diagnostics">操作与诊断</option><option value="preflight">资源检查</option>\\n        <option value="install">安装依赖</option><option value="model">模型容器</option>\\n        <option value="api">API 后台</option><option value="manager">全局管理</option>\\n        <option value="tunnel">公网隧道</option><option value="events">调度事件</option>\\n        <option value="state">状态详情</option><option value="smoke">最近验收</option>\\n      </select>\\n      <label class="sr-only" for="log-lines">最新日志行数</label>\\n      <select id="log-lines" class="log-lines">\\n        <option value="20">20 行</option><option value="40">40 行</option><option value="60">60 行</option>\\n        <option value="80">80 行</option><option value="100" selected>100 行</option><option value="120">120 行</option>\\n        <option value="140">140 行</option><option value="160">160 行</option><option value="180">180 行</option>\\n        <option value="200">200 行</option>\\n      </select>\\n      <button type="button" id="toggle-logs">暂停</button><button type="button" id="refresh-logs">刷新</button>\\n      <label class="follow"><input type="checkbox" id="follow-logs" checked>跟随新日志</label>\\n    </div>\\n    <div class="notice-zone" id="notice-zone" aria-live="polite" aria-atomic="false"></div>\\n    <div class="log-viewport" id="log-viewport" tabindex="0" aria-label="日志内容，可独立滚动">\\n      <header class="log-intro" id="log-intro">\\n        <h3>MiniMax-H3 · 参考生视频</h3>\\n        <p>A100 原生 BF16 · 仅 ref2va。点击 Colab「一键启动」单元的运行箭头后，自动检查资源、安装依赖并启动 API、公网和模型。API 可达后即可排队，模型就绪后自动处理。</p>\\n      </header>\\n      <pre id="log-text">尚无本次运行日志。</pre>\\n    </div>\\n    <footer class="log-footer"><span id="log-stamp">只读 · 有限尾部 · 凭据脱敏</span><span id="log-mode">每 2 秒读取</span></footer>\\n  </section>\\n</main>\\n<script>\\n(() => {\\n  \\\'use strict\\\';\\n  const CALLBACK_NAME = __CALLBACK_NAME__;\\n  const INITIAL_STATE = __INITIAL_STATE__;\\n  const $ = id => document.getElementById(id);\\n  const actionButtons = [...document.querySelectorAll(\\\'[data-action]\\\')];\\n  const activeActions = new Set();\\n  const uncertainRequests = new Map();\\n  const knownNotices = new Set();\\n  const blobUrls = new Set();\\n  let current = INITIAL_STATE && typeof INITIAL_STATE === \\\'object\\\' ? INITIAL_STATE : {};\\n  let timer = null, polling = false, paused = false, disposed = false;\\n  let connectionLost = false, localNotice = null, shownNoticeId = null, failures = 0;\\n  let lastText = \\\'\\\', lastSource = \\\'\\\', lastLogs = null, pendingConfirmation = null;\\n  let revealedKey = \\\'\\\', keyVisible = false, secretBusy = false, secretEpoch = 0;\\n  let observedCredentialId = null, observedInstanceId = null;\\n  const hasTransport = () => typeof window.H3_TRANSPORT === \\\'function\\\' || typeof window.google?.colab?.kernel?.invokeFunction === \\\'function\\\';\\n  const offline = () => current.ui?.offline === true || !hasTransport();\\n  const closed = () => current.ui?.closed === true;\\n  const delivering = () => current.ui?.startup_delivery?.active === true &&\\n    Date.now() < Number(current.ui.startup_delivery.deadline_at || 0) * 1000 + 5000;\\n  const text = (id, value) => { $(id).textContent = String(value ?? \\\'—\\\'); };\\n  const nowText = value => {\\n    if (!value) return \\\'未记录\\\';\\n    const date = typeof value === \\\'number\\\' ? new Date(value * 1000) : new Date(value);\\n    if (!Number.isFinite(date.getTime())) return \\\'未记录\\\';\\n    return new Intl.DateTimeFormat(\\\'zh-CN\\\', {timeZone:\\\'Asia/Shanghai\\\', hour:\\\'2-digit\\\', minute:\\\'2-digit\\\', second:\\\'2-digit\\\', hour12:false}).format(date) + \\\' 北京时间\\\';\\n  };\\n  const safeText = value => String(value ?? \\\'\\\').replace(/\\\\x1b\\\\[[0-?]*[ -/]*[@-~]/g, \\\'\\\').replace(/\\\\bBearer\\\\s+[^\\\\s"\\\'<>;,]+/gi,\\\'Bearer [已隐藏]\\\').replace(/(["\\\']?(?:H3_API_KEY|API_KEY|api[-_]?key|access_token)["\\\']?\\\\s*[:=]\\\\s*)(["\\\'])(.*?)\\\\2/gi,\\\'$1$2[已隐藏]$2\\\');\\n  const payload = () => ({staged:$(\\\'staged\\\').checked,cached:$(\\\'cached\\\').checked,public:$(\\\'public\\\').checked,\\n    reference_file:$(\\\'reference_file\\\').value.trim(),prompt:$(\\\'prompt\\\').value.trim(),resume_job_id:$(\\\'resume_job_id\\\').value.trim(),\\n    endpoint:$(\\\'endpoint\\\').value,acceleration:$(\\\'acceleration\\\').value,style:$(\\\'style\\\').value,style_scale:Number($(\\\'style_scale\\\').value),\\n    attention:$(\\\'attention\\\').value,offload:$(\\\'offload\\\').value});\\n  function uuid() {\\n    if (window.crypto?.randomUUID) return window.crypto.randomUUID();\\n    const bytes = new Uint8Array(16); window.crypto.getRandomValues(bytes);\\n    return [...bytes].map(value=>value.toString(16).padStart(2,\\\'0\\\')).join(\\\'\\\');\\n  }\\n  function setButtons() {\\n    const unavailable = offline() || closed() || delivering();\\n    const busy = current.ui?.busy === true;\\n    const operations = current.ui?.operations || [];\\n    const generationBusy = operations.some(item=>item.action===\\\'generate\\\');\\n    const stopping = current.state?.phase===\\\'stopping\\\' || operations.some(item=>[\\\'stop\\\',\\\'release\\\'].includes(item.action));\\n    const apiUsable = current.state?.api_ready===true && current.state?.observation_stale!==true && !connectionLost;\\n    const connectionAvailable = current.ui?.connection_available===true || apiUsable;\\n    for (const button of actionButtons) {\\n      const action = button.dataset.action;\\n      const setupBlocked = (busy || current.ui?.bootstrap_running===true) && [\\\'bootstrap\\\',\\\'preflight\\\',\\\'install\\\',\\\'start\\\'].includes(action);\\n      const generationBlocked = action===\\\'generate\\\' && (stopping || generationBusy || (busy && !apiUsable));\\n      button.disabled = unavailable || activeActions.has(action) || setupBlocked || generationBlocked || (action===\\\'export_connection\\\' && !connectionAvailable) || (action===\\\'download_result\\\' && !current.result?.available);\\n    }\\n    $(\\\'send-control\\\').disabled = unavailable || activeActions.has(\\\'control\\\');\\n    $(\\\'query-command\\\').disabled = unavailable || activeActions.has(\\\'control\\\');\\n    $(\\\'refresh-board\\\').disabled = unavailable || polling;\\n    $(\\\'refresh-logs\\\').disabled = unavailable || polling;\\n    $(\\\'toggle-logs\\\').disabled = unavailable;\\n    const connection=current.connection||{};\\n    const secretUnavailable=unavailable || connectionLost || document.hidden || !connection.key_available || !connection.credential_id;\\n    $(\\\'toggle-key\\\').disabled=secretUnavailable || secretBusy;\\n    $(\\\'copy-key\\\').disabled=secretUnavailable || secretBusy;\\n    $(\\\'copy-url\\\').disabled=unavailable || connectionLost || current.state?.observation_stale===true || !usableUrl(connection.url);\\n  }\\n  function usableUrl(value) {\\n    if(typeof value!==\\\'string\\\'||!value)return null;\\n    try {const url=new URL(value);return [\\\'http:\\\',\\\'https:\\\'].includes(url.protocol)&&!url.username&&!url.password?value:null;}catch{return null;}\\n  }\\n  function clearSecret() {\\n    revealedKey=\\\'\\\';keyVisible=false;secretEpoch++;\\n    $(\\\'api-key\\\').value=current.connection?.key_available?\\\'••••••••••••••••\\\':\\\'尚未生成\\\';\\n    text(\\\'toggle-key\\\',\\\'显示\\\');$(\\\'toggle-key\\\').setAttribute(\\\'aria-label\\\',\\\'显示 API 密钥\\\');\\n  }\\n  function renderConnection() {\\n    const connection=current.connection||{};\\n    const credentialId=connection.credential_id||null, instanceId=current.state?.instance_id||null;\\n    if(credentialId!==observedCredentialId || instanceId!==observedInstanceId || offline() || closed() || connectionLost || document.hidden){\\n      clearSecret();observedCredentialId=credentialId;observedInstanceId=instanceId;\\n    }\\n    const stale=current.state?.observation_stale===true;\\n    const readyUrl=stale?null:usableUrl(connection.url), candidate=usableUrl(connection.candidate_url)||(stale?usableUrl(connection.url):null);\\n    const local=usableUrl(connection.local_url);\\n    $(\\\'api-url\\\').value=readyUrl||candidate||(offline()?\\\'尚未连接运行时\\\':\\\'等待分配地址\\\');\\n    $(\\\'api-url\\\').title=readyUrl||candidate||\\\'\\\';\\n    let hint=\\\'密钥在启动初期生成；按需显示或复制。\\\';\\n    if(offline()||closed())hint=\\\'当前为离线界面，没有本次连接或密钥。\\\';\\n    else if(connectionLost)hint=\\\'连接中断；上次地址不代表当前可用，密钥已隐藏。\\\';\\n    else if(readyUrl)hint=(connection.public_requested?\\\'公网 API 已通过探测\\\':\\\'本机 API 已可达\\\')+\\\' · 密钥默认隐藏\\\';\\n    else if(candidate)hint=\\\'地址已分配 · 尚未验证／当前不可用\\\'+(local?\\\' · 本机 API 可用\\\':\\\'\\\');\\n    else if(connection.public_requested&&local)hint=\\\'公网地址准备中 · 本机 API 已可用\\\';\\n    else if(current.ui?.bootstrap_running)hint=\\\'正在准备 API；密钥生成后即可显示或复制。\\\';\\n    text(\\\'connection-hint\\\',hint);$(\\\'connection-hint\\\').title=hint+(local?\\\'\\\\n本机 API：\\\'+local:\\\'\\\');\\n    $(\\\'api-key\\\').value=keyVisible&&revealedKey?revealedKey:connection.key_available?\\\'••••••••••••••••\\\':\\\'尚未生成\\\';\\n    text(\\\'toggle-key\\\',keyVisible?\\\'隐藏\\\':\\\'显示\\\');\\n    $(\\\'toggle-key\\\').setAttribute(\\\'aria-label\\\',keyVisible?\\\'隐藏 API 密钥\\\':\\\'显示 API 密钥\\\');\\n  }\\n  function bytesText(value) {\\n    if(typeof value!==\\\'number\\\'||!Number.isFinite(value)||value<0)return \\\'—\\\';\\n    const units=[\\\'B\\\',\\\'KiB\\\',\\\'MiB\\\',\\\'GiB\\\',\\\'TiB\\\'];let amount=value,index=0;\\n    while(amount>=1024&&index<units.length-1){amount/=1024;index++;}\\n    return (index?amount.toFixed(1):Math.round(amount))+\\\' \\\'+units[index];\\n  }\\n  function renderTransfers() {\\n    const downloads=current.downloads||{}, items=downloads.items||{};\\n    const stale=downloads.observation_stale===true || connectionLost || current.state?.observation_stale===true;\\n    const phases={waiting:\\\'等待下载\\\',downloading:\\\'下载中\\\',verifying:\\\'校验中\\\',loading:\\\'加载中\\\',loaded:\\\'已加载\\\',disabled:\\\'未启用\\\',failed:\\\'准备失败\\\'};\\n    for(const name of [\\\'model\\\',\\\'acceleration\\\',\\\'style\\\']) {\\n      const item=items[name], row=$(\\\'transfer-\\\'+name), progress=$(name+\\\'-download-progress\\\');\\n      let phase=\\\'尚未观察\\\', detail=name===\\\'model\\\'?\\\'下载与加载分别显示\\\':\\\'未启用时不下载\\\', percent=null;\\n      if(item&&typeof item===\\\'object\\\') {\\n        phase=phases[item.phase]||\\\'等待可测进度\\\';\\n        if(name===\\\'model\\\'&&item.phase===\\\'loaded\\\')phase=\\\'已加载 · 等待就绪检查\\\';\\n        if(name===\\\'model\\\'&&current.state?.model_ready===true&&!stale)phase=\\\'可推理\\\';\\n        const bytesKnown=typeof item.bytes_completed===\\\'number\\\'&&Number.isFinite(item.bytes_completed)&&item.bytes_completed>=0;\\n        const totalKnown=typeof item.bytes_total===\\\'number\\\'&&Number.isFinite(item.bytes_total)&&item.bytes_total>0;\\n        const parts=[];\\n        if(bytesKnown)parts.push((item.scope===\\\'observed_transfers\\\'?\\\'已观察 \\\':\\\'\\\')+bytesText(item.bytes_completed)+(totalKnown?\\\' / \\\'+bytesText(item.bytes_total):\\\' · 总量未知\\\'));\\n        if(Number.isInteger(item.files_completed)&&item.files_completed>=0)parts.push(\\\'文件 \\\'+item.files_completed+\\\' / \\\'+(Number.isInteger(item.files_total)&&item.files_total>=0?item.files_total:\\\'?\\\'));\\n        detail=parts.join(\\\' · \\\')||item.detail||\\\'等待可测进度\\\';\\n        if(bytesKnown&&item.scope===\\\'observed_transfers\\\')detail+=\\\'（含续传）\\\';\\n        if(item.phase===\\\'disabled\\\')detail=\\\'不下载，不占用加载资源\\\';\\n        else if(item.detail&&[\\\'loaded\\\',\\\'loading\\\',\\\'verifying\\\',\\\'failed\\\'].includes(item.phase))detail=item.detail;\\n        if(!stale&&totalKnown&&typeof item.percent===\\\'number\\\'&&Number.isFinite(item.percent)&&item.percent>=0&&item.percent<=100)percent=item.percent;\\n        if([\\\'stopped\\\',\\\'stopping\\\',\\\'failed\\\'].includes(downloads.lifecycle)&&item.phase!==\\\'disabled\\\'){\\n          phase=({stopped:\\\'已停止\\\',stopping:\\\'正在停止\\\',failed:\\\'已失败\\\'})[downloads.lifecycle]+\\\' · 上次\\\'+phase;\\n          detail=\\\'保留记录：\\\'+detail;percent=null;\\n        } else if(stale&&item.phase!==\\\'disabled\\\'){phase=\\\'待刷新 · 上次\\\'+phase;detail=\\\'上次观察：\\\'+detail;}\\n        row.dataset.phase=item.phase||\\\'waiting\\\';\\n      } else {row.dataset.phase=\\\'waiting\\\';if(!offline()&&current.ui?.bootstrap_running)detail=\\\'下载观察尚未开始\\\';}\\n      if(offline()||closed()){phase=\\\'尚未观察\\\';detail=name===\\\'model\\\'?\\\'连接后显示真实下载与加载状态\\\':\\\'连接后显示实际启用状态\\\';percent=null;}\\n      text(name+\\\'-download-phase\\\',phase);$(name+\\\'-download-phase\\\').title=phase;\\n      text(name+\\\'-download-detail\\\',safeText(detail));\\n      $(name+\\\'-download-detail\\\').title=safeText([detail,item?.detail].filter(Boolean).join(\\\'\\\\n\\\'));\\n      text(name+\\\'-download-number\\\',percent===null?\\\'—\\\':percent.toFixed(percent===100?0:1)+\\\'%\\\');\\n      progress.hidden=percent===null;\\n      if(percent===null)progress.removeAttribute(\\\'value\\\');else progress.value=percent;\\n    }\\n  }\\n  function renderBoard() {\\n    const state=current.state||{}, ui=current.ui||{}, resources=current.resources||{};\\n    const cleanStop=!ui.bootstrap_running && state.phase===\\\'stopped\\\' && state.cleanup_complete===true && !state.manager_alive && !state.server_alive && !state.model_alive && !state.tunnel_alive;\\n    const stale=(state.observation_stale===true || connectionLost || ui.observation_error===true) && !cleanStop;\\n    let headline=\\\'等待一键启动\\\', next=\\\'一键完成检查、安装与启动；API 就绪后可排队，模型就绪后自动处理。\\\', tone=\\\'\\\';\\n    if (closed()) {headline=\\\'控制台连接已关闭\\\';next=\\\'需要继续操作时，重新运行控制台单元。\\\';}\\n    else if (offline()) {headline=\\\'未连接运行时\\\';next=\\\'运行本格即自动启动；正常无需再次点击。此处尚无实时状态。\\\';}\\n    else if(delivering()){headline=\\\'一键启动 · 交付 API\\\';next=\\\'主单元正有限观察，状态由宿主推送；如需取消，请停止本单元，程序会回收已启动进程。\\\';}\\n    else if(cleanStop){headline=\\\'服务已停止\\\';next=\\\'进程清理已确认；释放云实例需另行操作。\\\';}\\n    else if(stale){headline=\\\'状态需要刷新\\\';next=\\\'当前观察无法确认；查看右侧日志，不据旧状态判断可用能力。\\\';tone=\\\'warn\\\';}\\n    else if(state.phase===\\\'stopping\\\'){headline=\\\'正在停止与回收\\\';next=\\\'等待进程退出与清理确认，请勿重复启动。\\\';tone=\\\'warn\\\';}\\n    else if(state.error || [\\\'failed\\\',\\\'degraded\\\'].includes(state.phase)){headline=\\\'有模块需要处理\\\';next=\\\'仍可使用已就绪的能力；原因与处理提示在右侧日志。\\\';tone=\\\'error\\\';}\\n    else if(ui.startup_stage===\\\'failed\\\'){headline=\\\'启动未完成\\\';next=\\\'原因在右侧日志；处理后可再次一键启动，已就绪的模块仍可查询。\\\';tone=\\\'error\\\';}\\n    else if(ui.startup_stage===\\\'cancelled\\\'){headline=\\\'自动启动已取消\\\';next=\\\'查看各模块的真实状态；需要继续时可再次一键启动。\\\';}\\n    else if(state.draining){headline=\\\'已暂停接收新任务\\\';next=\\\'已有队列继续执行；高级控制中可以恢复接收。\\\';}\\n    else if(state.ready){headline=\\\'可以生成\\\';next=\\\'排队任务自动处理；API 可查询进度、失败原因和视频结果。\\\';tone=\\\'ready\\\';}\\n    else if(state.api_ready && state.model_ready){headline=\\\'模型已就绪\\\';next=state.public_requested&&!state.public_api_ready?\\\'本机 API 已可处理任务；公网仍在准备，请以公网探测结果为准。\\\':\\\'任务自动处理；API 返回进度与视频结果。\\\';tone=\\\'ready\\\';}\\n    else if(state.api_ready){headline=state.queue?.admission_open?\\\'可排队 · 等待模型\\\':\\\'API 可查询 · 等待模型\\\';next=state.queue?.admission_open?\\\'已接收任务保持排队；模型完成加载并通过就绪检查后自动处理。\\\':\\\'API 可查询；任务接收状态以当前后台为准。\\\';}\\n    else if(ui.bootstrap_running){const stages={preflight:\\\'检查资源\\\',install:\\\'安装依赖\\\',start:\\\'启动后台\\\',awaiting_api:\\\'等待 API\\\',awaiting_model:\\\'等待模型\\\'};headline=\\\'一键启动 · \\\'+(stages[ui.startup_stage]||\\\'准备中\\\');next=\\\'正在自动推进；API 密钥可用后可导出，异常会在右侧日志显示。\\\';}\\n    else if(ui.busy){headline=ui.operation || \\\'操作进行中\\\';next=\\\'详细过程只在右侧日志更新；不重复提交当前操作。\\\';}\\n    else if(state.phase && state.phase!==\\\'new\\\'){headline=\\\'服务正在准备\\\';next=\\\'右侧日志持续更新；无需等待模型就绪才观察 API。\\\';}\\n    else if(ui.install_ok){headline=\\\'依赖已就绪\\\';next=\\\'点击一键启动继续；需要分步维护时再展开高级控制。\\\';}\\n    else if(ui.preflight_ok){headline=\\\'资源检查通过\\\';next=\\\'点击一键启动继续安装和启动，无需逐步点击。\\\';}\\n    text(\\\'headline\\\',headline);text(\\\'next-step\\\',next);$(\\\'board\\\').dataset.tone=tone;\\n    const module=(alive,ready)=>{\\n      if(offline() || closed() || !Object.keys(state).length) return \\\'尚未读取\\\';\\n      if(cleanStop) return \\\'已停止\\\';\\n      if(stale) return \\\'待刷新\\\';\\n      if(ready===true) return \\\'可用\\\';\\n      if(alive===true) return ready===undefined?\\\'运行中\\\':\\\'准备中\\\';\\n      return alive===false?\\\'已停止\\\':\\\'未确认\\\';\\n    };\\n    text(\\\'manager-state\\\',module(state.manager_alive));text(\\\'api-state\\\',module(state.server_alive,state.api_ready));text(\\\'model-state\\\',module(state.model_alive,state.model_ready));\\n    text(\\\'public-state\\\', offline()||closed()||!Object.keys(state).length?\\\'尚未读取\\\':cleanStop?\\\'已关闭\\\':stale?\\\'待刷新\\\':!state.public_requested?\\\'未启用\\\':state.public_api_ready?\\\'探测通过\\\':state.tunnel_connected?\\\'等待探测\\\':\\\'准备中\\\');\\n    const queue=state.queue||{};text(\\\'queue-state\\\',stale||offline()?\\\'排队 —\\u3000执行 —\\\':`排队 ${queue.queued??\\\'—\\\'}\\u3000执行 ${queue.running??\\\'—\\\'}`);\\n    const gpu=resources.gpu?String(resources.gpu).replace(/^NVIDIA\\\\s+/,\\\'\\\'):\\\'A100（配置）\\\';\\n    text(\\\'resources\\\',`${gpu} · 原生 BF16${resources.available_ram_gib!==undefined?\\\' · 可用内存 \\\'+resources.available_ram_gib+\\\' GiB（预检）\\\':\\\'\\\'}`);\\n    text(\\\'board-stamp\\\',state.observed_at?\\\'观察 \\\'+nowText(state.observed_at):\\\'尚无服务观察\\\');\\n    renderConnection();renderTransfers();\\n  }\\n  function connectionLabel() {\\n    const value=closed()?\\\'连接已关闭\\\':offline()?\\\'离线占位\\\':delivering()?\\\'启动交付 · 宿主推送\\\':connectionLost?\\\'连接待确认\\\':paused?\\\'已暂停读取\\\':document.hidden?\\\'页面隐藏 · 暂停\\\':\\\'持续读取\\\';\\n    text(\\\'connection-state\\\',value);$(\\\'live-dot\\\').classList.toggle(\\\'live\\\',!offline()&&!closed()&&!connectionLost&&!paused&&!document.hidden);\\n    text(\\\'log-mode\\\',delivering()?\\\'宿主推送 · 停止本单元可取消\\\':paused?\\\'日志保持静止\\\':offline()||closed()?\\\'未自动读取\\\':\\\'每 2 秒读取\\\');\\n  }\\n  function clearNotice() { $(\\\'notice-zone\\\').replaceChildren(); shownNoticeId=null; }\\n  function notice(value,buttons=[]) {\\n    if(!value){clearNotice();return;}\\n    clearNotice();\\n    const card=document.createElement(\\\'section\\\');card.className=\\\'notice\\\';card.dataset.kind=value.kind||\\\'info\\\';\\n    const heading=document.createElement(\\\'div\\\');heading.className=\\\'notice-title\\\';heading.textContent=safeText(value.title||\\\'提示\\\');\\n    const message=document.createElement(\\\'div\\\');message.className=\\\'notice-message\\\';message.textContent=safeText(value.message||\\\'\\\').slice(0,3000);\\n    card.append(heading,message);\\n    const actions=document.createElement(\\\'div\\\');actions.className=\\\'notice-actions\\\';\\n    for(const item of buttons){const button=document.createElement(\\\'button\\\');button.type=\\\'button\\\';button.textContent=item.label;if(item.primary)button.className=\\\'primary\\\';button.addEventListener(\\\'click\\\',item.click);actions.append(button);}\\n    if(!buttons.length){const dismiss=document.createElement(\\\'button\\\');dismiss.type=\\\'button\\\';dismiss.textContent=\\\'收起\\\';dismiss.addEventListener(\\\'click\\\',()=>{if(value.id)knownNotices.add(String(value.id));localNotice=null;clearNotice();});actions.append(dismiss);}\\n    card.append(actions);$(\\\'notice-zone\\\').append(card);shownNoticeId=value.id?String(value.id):null;\\n    return {card,actions};\\n  }\\n  function showLocal(value){localNotice=value;notice(value);}\\n  function renderLogs(logs) {\\n    if(!logs || typeof logs.text!==\\\'string\\\')return;\\n    const selectedSource=$(\\\'log-source\\\').value;\\n    const source=logs.source||selectedSource;\\n    if(source!==selectedSource)return;\\n    lastLogs=logs;\\n    const viewport=$(\\\'log-viewport\\\');const nearBottom=viewport.scrollHeight-viewport.scrollTop-viewport.clientHeight<38;\\n    const scrollTop=viewport.scrollTop;const changedSource=source!==lastSource;\\n    const fullText=safeText(logs.text).slice(-64000);\\n    const tail=source===\\\'state\\\'?fullText:fullText.replace(/\\\\r?\\\\n$/, \\\'\\\').split(/\\\\r?\\\\n/).slice(-Number($(\\\'log-lines\\\').value)).join(\\\'\\\\n\\\');\\n    const next=tail||\\\'该来源暂无日志。\\\';\\n    if(next!==lastText || changedSource){$(\\\'log-text\\\').textContent=next;lastText=next;lastSource=source;\\n      if($(\\\'follow-logs\\\').checked&&(nearBottom||changedSource))viewport.scrollTop=viewport.scrollHeight;\\n      else viewport.scrollTop=scrollTop;\\n    }\\n    text(\\\'log-stamp\\\',`只读 · 已脱敏 · ${logs.observed_at?nowText(logs.observed_at):\\\'最新读取\\\'}`);\\n  }\\n  function apply(data) {\\n    if(!data || typeof data!==\\\'object\\\')return;\\n    for(const key of [\\\'state\\\',\\\'resources\\\',\\\'ui\\\',\\\'result\\\',\\\'connection\\\',\\\'downloads\\\'])if(data[key]!==undefined)current[key]=data[key];\\n    renderBoard();renderLogs(data.logs);connectionLabel();setButtons();\\n    if(data.notice&&!pendingConfirmation&&!localNotice){const id=String(data.notice.id||JSON.stringify(data.notice));if(!knownNotices.has(id)&&shownNoticeId!==id)notice({...data.notice,id});}\\n    if(closed()){clearTimeout(timer);timer=null;}\\n  }\\n  async function rpc(action,data) {\\n    if(offline()||closed())throw new Error(\\\'当前未连接到控制台后台。\\\');\\n    if(delivering())throw new Error(\\\'启动交付观察期间由宿主推送状态；如需取消，请停止本单元。\\\');\\n    const request=typeof window.H3_TRANSPORT===\\\'function\\\'?window.H3_TRANSPORT(action,data):window.google.colab.kernel.invokeFunction(CALLBACK_NAME,[action,data],{});\\n    let timeout;\\n    try {\\n      const raw=await Promise.race([Promise.resolve(request),new Promise((_,reject)=>{timeout=setTimeout(()=>reject(new Error(\\\'连接超时，操作结果尚未确认。\\\')),20000);})]);\\n      let value=raw?.data?.[\\\'application/json\\\']??raw;\\n      if(typeof value===\\\'string\\\')value=JSON.parse(value);\\n      if(!value||typeof value!==\\\'object\\\')throw new Error(\\\'后台没有返回可识别的结果。\\\');\\n      if(value.accepted===false || value.ok===false || (value.error && ![\\\'snapshot\\\',\\\'refresh_board\\\'].includes(action)) || (value.error && !value.state)){\\n        const error=new Error(typeof value.error===\\\'string\\\'?value.error:\\\'后台拒绝了本次操作。\\\');error.confirmed=true;throw error;\\n      }\\n      return value;\\n    } finally {clearTimeout(timeout);}\\n  }\\n  async function requestSecret() {\\n    const credentialId=current.connection?.credential_id, epoch=secretEpoch;\\n    if(!credentialId||!current.connection?.key_available||connectionLost||document.hidden)throw new Error(\\\'当前密钥不可读取；先刷新连接状态。\\\');\\n    if(keyVisible&&revealedKey&&credentialId===observedCredentialId)return revealedKey;\\n    const data=await rpc(\\\'reveal_connection\\\',{credential_id:credentialId});\\n    const secret=data.connection_secret;\\n    if(epoch!==secretEpoch||credentialId!==current.connection?.credential_id||secret?.credential_id!==credentialId||offline()||closed()||connectionLost||document.hidden){\\n      throw new Error(\\\'连接或页面状态已变化，密钥未显示；请重新读取。\\\');\\n    }\\n    if(typeof secret.api_key!==\\\'string\\\'||secret.api_key.length<32||secret.api_key.length>4096)throw new Error(\\\'没有收到有效的 API 密钥。\\\');\\n    return secret.api_key;\\n  }\\n  async function toggleSecret() {\\n    if(secretBusy)return;\\n    if(keyVisible){clearSecret();renderConnection();setButtons();return;}\\n    secretBusy=true;setButtons();\\n    try {revealedKey=await requestSecret();keyVisible=true;renderConnection();}\\n    catch {clearSecret();showLocal({kind:\\\'warning\\\',title:\\\'密钥未显示\\\',message:\\\'当前连接无法确认或密钥已变化。刷新看板后可重新显示；密钥不会写入日志。\\\'});}\\n    finally{secretBusy=false;setButtons();}\\n  }\\n  async function copyConnection(kind) {\\n    if(kind===\\\'key\\\'&&secretBusy)return;\\n    let value=\\\'\\\';const button=kind===\\\'key\\\'?$(\\\'copy-key\\\'):$(\\\'copy-url\\\');\\n    if(kind===\\\'key\\\'){secretBusy=true;setButtons();}\\n    try {\\n      value=kind===\\\'key\\\'?await requestSecret():usableUrl(current.connection?.url);\\n      if(!value)throw new Error(\\\'No confirmed value\\\');\\n      if(typeof navigator.clipboard?.writeText!==\\\'function\\\')throw new Error(\\\'Clipboard unavailable\\\');\\n      await navigator.clipboard.writeText(value);\\n      text(button.id,\\\'已复制\\\');setTimeout(()=>{if(!disposed)text(button.id,\\\'复制\\\');},1500);\\n    } catch {\\n      showLocal({kind:\\\'warning\\\',title:\\\'暂时无法复制\\\',message:kind===\\\'key\\\'?\\\'浏览器未允许复制或连接已变化。可先点击「显示」，再手动复制密钥。\\\':\\\'尚无已验证地址或浏览器未允许复制。确认 API 可用后，可从地址栏手动复制。\\\'});\\n    } finally {value=\\\'\\\';if(kind===\\\'key\\\')secretBusy=false;setButtons();}\\n  }\\n  function schedule() {\\n    clearTimeout(timer);timer=null;\\n    if(disposed||offline()||closed()||paused||document.hidden)return;\\n    if(delivering()){\\n      const remaining=Number(current.ui.startup_delivery.deadline_at)*1000+5100-Date.now();\\n      timer=setTimeout(()=>{connectionLabel();setButtons();readSnapshot(false);},Math.max(100,remaining));return;\\n    }\\n    timer=setTimeout(()=>readSnapshot(false),Math.min(10000,2000*Math.max(1,failures)));\\n  }\\n  async function readSnapshot(manual=false,action=\\\'snapshot\\\') {\\n    if(polling||disposed||offline()||closed()||delivering())return;\\n    if(!manual&&(paused||document.hidden))return;\\n    const source=$(\\\'log-source\\\').value;\\n    clearTimeout(timer);timer=null;polling=true;setButtons();\\n    try {const data=await rpc(action,{source});connectionLost=false;failures=0;if(localNotice?.id===\\\'connection-lost\\\'){localNotice=null;clearNotice();}apply(data);}\\n    catch(error){connectionLost=true;failures++;renderBoard();if(manual||failures===1)showLocal({id:\\\'connection-lost\\\',kind:\\\'warning\\\',title:\\\'暂时无法读取状态\\\',message:safeText(error.message)+\\\'\\\\n日志保留在原位；不会自动重发业务操作。\\\'});}\\n    finally{polling=false;connectionLabel();setButtons();if(source!==$(\\\'log-source\\\').value)readSnapshot(true);else schedule();}\\n  }\\n  function offerDownload(download) {\\n    if(!download || typeof download.data_base64!==\\\'string\\\')return;\\n    if(download.data_base64.length>Math.ceil(32*1024*1024*4/3)+8)throw new Error(\\\'下载文件超过控制台的 32 MiB 上限。\\\');\\n    const binary=atob(download.data_base64);const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);\\n    const url=URL.createObjectURL(new Blob([bytes],{type:download.mime||\\\'application/octet-stream\\\'}));blobUrls.add(url);\\n    localNotice={kind:\\\'info\\\',title:\\\'文件已准备\\\',message:\\\'点击保存文件。若为私密连接文件，请单独保管。\\\'};\\n    const view=notice(localNotice);const link=document.createElement(\\\'a\\\');link.href=url;link.download=String(download.name||\\\'result.bin\\\').replace(/[\\\\\\\\/]/g,\\\'_\\\');link.textContent=\\\'保存文件\\\';view.actions.prepend(link);\\n    link.addEventListener(\\\'click\\\',()=>{setTimeout(()=>{URL.revokeObjectURL(url);blobUrls.delete(url);},60000);});\\n  }\\n  async function dispatch(action,values) {\\n    if(activeActions.has(action))return;\\n    if(delivering()){showLocal({kind:\\\'info\\\',title:\\\'启动交付观察中\\\',message:\\\'当前单元仍在执行，后台保持并行准备。如需取消，请使用 Colab 本单元的停止按钮，程序会回收已启动进程。\\\'});return;}\\n    if(offline()||closed()){showLocal({kind:\\\'info\\\',title:\\\'尚未连接\\\',message:\\\'连接运行时并运行控制台单元后再操作。\\\'});return;}\\n    const fingerprint=action+\\\':\\\'+JSON.stringify(values);const requestId=uncertainRequests.get(fingerprint)||uuid();\\n    activeActions.add(action);setButtons();localNotice=null;pendingConfirmation=null;clearNotice();\\n    try {\\n      const data=await rpc(action,{...values,request_id:requestId});uncertainRequests.delete(fingerprint);connectionLost=false;apply(data);\\n      if(data.download)offerDownload(data.download);\\n      else if(data.notice){localNotice=data.notice;notice(data.notice);}\\n      // Ordinary accepted operations update the log and board without another notice card.\\n    } catch(error) {\\n      if(error.confirmed)uncertainRequests.delete(fingerprint);else uncertainRequests.set(fingerprint,requestId);\\n      showLocal({kind:\\\'error\\\',title:error.confirmed?\\\'操作未完成\\\':\\\'操作结果尚未确认\\\',message:safeText(error.message)+(error.confirmed?\\\'\\\\n请根据日志处理后再操作。\\\':\\\'\\\\n未自动重发。先查看状态；再次提交相同操作时会沿用本次请求编号。\\\')});\\n    } finally {activeActions.delete(action);setButtons();schedule();}\\n  }\\n  function requestAction(action,values) {\\n    if(action===\\\'release\\\'||(action===\\\'control\\\'&&values.action===\\\'stop_model\\\'&&values.force)){\\n      pendingConfirmation={action,values};localNotice=null;\\n      notice({kind:\\\'warning\\\',title:action===\\\'release\\\'?\\\'释放当前云实例？\\\':\\\'强制停止模型？\\\',message:action===\\\'release\\\'?\\\'释放后，Colab 临时文件将丢失。请确认已保存结果。\\\':\\\'这会中断模型进程中的任务。排队取消不需要强制停止。\\\'},[\\n        {label:\\\'取消\\\',click:()=>{pendingConfirmation=null;clearNotice();}},\\n        {label:action===\\\'release\\\'?\\\'确认释放\\\':\\\'确认强制停止\\\',primary:true,click:()=>{const pending=pendingConfirmation;if(pending){pendingConfirmation=null;dispatch(pending.action,pending.values);}}}\\n      ]);return;\\n    }\\n    dispatch(action,values);\\n  }\\n  for(const button of actionButtons)button.addEventListener(\\\'click\\\',()=>requestAction(button.dataset.action,payload()));\\n  $(\\\'send-control\\\').addEventListener(\\\'click\\\',()=>requestAction(\\\'control\\\',{action:$(\\\'component-action\\\').value,force:$(\\\'force\\\').checked}));\\n  $(\\\'query-command\\\').addEventListener(\\\'click\\\',()=>{const id=$(\\\'command_id\\\').value.trim();if(!id){showLocal({kind:\\\'info\\\',title:\\\'填写命令 ID\\\',message:\\\'在高级控制中填写需要查询的命令 ID。\\\'});return;}dispatch(\\\'control\\\',{action:\\\'query\\\',command_id:id});});\\n  $(\\\'refresh-board\\\').addEventListener(\\\'click\\\',()=>readSnapshot(true,\\\'refresh_board\\\'));\\n  $(\\\'toggle-key\\\').addEventListener(\\\'click\\\',toggleSecret);\\n  $(\\\'copy-key\\\').addEventListener(\\\'click\\\',()=>copyConnection(\\\'key\\\'));\\n  $(\\\'copy-url\\\').addEventListener(\\\'click\\\',()=>copyConnection(\\\'url\\\'));\\n  $(\\\'refresh-logs\\\').addEventListener(\\\'click\\\',()=>readSnapshot(true));\\n  $(\\\'log-source\\\').addEventListener(\\\'change\\\',()=>{\\n    lastSource=\\\'\\\';\\n    $(\\\'log-lines\\\').disabled=$(\\\'log-source\\\').value===\\\'state\\\';\\n    readSnapshot(true);\\n  });\\n  $(\\\'log-lines\\\').addEventListener(\\\'change\\\',()=>{if(!offline())renderLogs(lastLogs);});\\n  $(\\\'toggle-logs\\\').addEventListener(\\\'click\\\',()=>{paused=!paused;clearTimeout(timer);timer=null;text(\\\'toggle-logs\\\',paused?\\\'继续\\\':\\\'暂停\\\');connectionLabel();if(!paused)readSnapshot(true);});\\n  $(\\\'follow-logs\\\').addEventListener(\\\'change\\\',()=>{if($(\\\'follow-logs\\\').checked)$(\\\'log-viewport\\\').scrollTop=$(\\\'log-viewport\\\').scrollHeight;});\\n  document.addEventListener(\\\'visibilitychange\\\',()=>{clearTimeout(timer);timer=null;if(document.hidden)clearSecret();connectionLabel();setButtons();if(!document.hidden&&!paused)readSnapshot(false);});\\n  function dispose(){disposed=true;clearSecret();clearTimeout(timer);timer=null;for(const url of blobUrls)URL.revokeObjectURL(url);blobUrls.clear();}\\n  window.addEventListener(\\\'pagehide\\\',dispose,{once:true});window.addEventListener(\\\'beforeunload\\\',dispose,{once:true});\\n  apply(current);if(offline())text(\\\'log-text\\\',\\\'尚未连接运行时，没有实时日志。\\\\n连接后，提示、确认和下载入口只在此区域出现。\\\');else schedule();\\n})();\\n</script>\\n</body>\\n</html>\\n\', \'download_progress.py\': \'"""Observed Hugging Face transfer progress, separate from model readiness.\\n\\nNo extra downloads, repository scans, or scheduling are introduced. An owned\\nload temporarily supplies a progress-bar observer to HF\\\'s existing HTTP/Xet\\ntransfer functions. Unsupported signatures degrade to unknown progress. The\\ndownload algorithm, retries, validation, caching and worker counts stay in HF.\\n"""\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nfrom copy import deepcopy\\nimport functools\\nimport hashlib\\nimport importlib\\nimport inspect\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport threading\\nimport time\\nimport uuid\\n\\nMAX_STATE_BYTES = 256 * 1024\\nPHASES = {\\\'waiting\\\', \\\'downloading\\\', \\\'verifying\\\', \\\'loading\\\', \\\'loaded\\\', \\\'disabled\\\', \\\'failed\\\'}\\n_ACTIVE = None\\n_RECORDER = None\\n_HOOK_LOCK = threading.RLock()\\n\\n\\ndef _number(value):\\n    return type(value) in (int, float) and math.isfinite(value) and value >= 0\\n\\n\\ndef read_progress(path, expected_instance_id=None):\\n    unavailable = {\\\'schema_version\\\': 1, \\\'model_instance_id\\\': expected_instance_id,\\n                   \\\'observed_at\\\': None, \\\'observation_stale\\\': True, \\\'items\\\': {}}\\n    try:\\n        path = Path(path)\\n        if path.stat().st_size > MAX_STATE_BYTES:\\n            raise ValueError(\\\'oversized\\\')\\n        with path.open(\\\'r\\\', encoding=\\\'utf-8\\\') as stream:\\n            raw = stream.read(MAX_STATE_BYTES + 1)\\n        if len(raw.encode(\\\'utf-8\\\')) > MAX_STATE_BYTES:\\n            raise ValueError(\\\'oversized\\\')\\n        data = json.loads(raw)\\n        if data.get(\\\'schema_version\\\') != 1 or not isinstance(data.get(\\\'items\\\'), dict):\\n            raise ValueError(\\\'invalid_schema\\\')\\n        if expected_instance_id is not None and data.get(\\\'model_instance_id\\\') != expected_instance_id:\\n            raise ValueError(\\\'generation_mismatch\\\')\\n        if not _number(data.get(\\\'observed_at\\\')):\\n            raise ValueError(\\\'invalid_timestamp\\\')\\n        items = {}\\n        for key in (\\\'model\\\', \\\'acceleration\\\', \\\'style\\\'):\\n            item = data[\\\'items\\\'].get(key)\\n            if item is None:\\n                continue\\n            if not isinstance(item, dict) or item.get(\\\'phase\\\') not in PHASES:\\n                raise ValueError(\\\'invalid_item\\\')\\n            clean = {\\\'phase\\\': item[\\\'phase\\\']}\\n            for field in (\\\'bytes_completed\\\', \\\'bytes_total\\\', \\\'percent\\\', \\\'files_completed\\\', \\\'files_total\\\', \\\'active_files\\\'):\\n                value = item.get(field)\\n                if value is not None and not _number(value):\\n                    raise ValueError(\\\'invalid_counter\\\')\\n                if field == \\\'percent\\\' and value is not None and value > 100:\\n                    raise ValueError(\\\'invalid_percent\\\')\\n                clean[field] = value\\n            for field in (\\\'scope\\\', \\\'detail\\\'):\\n                clean[field] = str(item.get(field, \\\'\\\'))[:240]\\n            clean[\\\'cache_hit\\\'] = item.get(\\\'cache_hit\\\') if type(item.get(\\\'cache_hit\\\')) is bool else None\\n            items[key] = clean\\n        return {\\\'schema_version\\\': 1, \\\'model_instance_id\\\': data.get(\\\'model_instance_id\\\'),\\n                \\\'observed_at\\\': data[\\\'observed_at\\\'], \\\'observation_stale\\\': False, \\\'items\\\': items}\\n    except (OSError, ValueError, TypeError, AttributeError) as error:\\n        return dict(unavailable, error=str(error) if isinstance(error, ValueError) and str(error) in {\\n            \\\'oversized\\\', \\\'invalid_schema\\\', \\\'generation_mismatch\\\', \\\'invalid_timestamp\\\', \\\'invalid_item\\\',\\n            \\\'invalid_counter\\\', \\\'invalid_percent\\\'} else \\\'progress_unavailable\\\')\\n\\n\\nclass DownloadProgress:\\n    def __init__(self, path=None, *, instance_id=None, settings=None, interval=.5, clock=time.monotonic):\\n        self.path = Path(path) if path else None\\n        self.instance_id = instance_id\\n        self.lock = threading.RLock()\\n        self.interval, self.clock, self.last_write = interval, clock, -math.inf\\n        self.category = \\\'model\\\'\\n        self.transfers = {}\\n        self.transfer_identities = {}\\n        self.measurement_available = None\\n        self.items = {}\\n        self._phase_details = {}\\n        for name in (\\\'model\\\', \\\'acceleration\\\', \\\'style\\\'):\\n            enabled = name == \\\'model\\\' or (settings is not None and getattr(settings, name, \\\'none\\\') != \\\'none\\\')\\n            self.items[name] = dict(phase=\\\'waiting\\\' if enabled else \\\'disabled\\\', bytes_completed=0,\\n                bytes_total=None, percent=None, files_completed=0, files_total=None,\\n                active_files=0, scope=\\\'observed_transfers\\\' if name == \\\'model\\\' else \\\'asset\\\',\\n                detail=\\\'尚未开始\\\' if enabled else \\\'未启用\\\', cache_hit=None)\\n        self._write(force=True)\\n\\n    def _document(self):\\n        return {\\\'schema_version\\\': 1, \\\'model_instance_id\\\': self.instance_id,\\n                \\\'observed_at\\\': time.time(), \\\'observation_stale\\\': False, \\\'items\\\': deepcopy(self.items)}\\n\\n    def snapshot(self):\\n        with self.lock:\\n            return self._document()\\n\\n    def _write(self, *, force=False, event=None):\\n        if not self.path:\\n            return\\n        now = self.clock()\\n        if not force and now - self.last_write < self.interval:\\n            return\\n        try:\\n            self.path.parent.mkdir(parents=True, exist_ok=True)\\n            temporary = self.path.with_name(self.path.name + \\\'.\\\' + uuid.uuid4().hex + \\\'.tmp\\\')\\n            try:\\n                temporary.write_text(json.dumps(self._document(), ensure_ascii=False), encoding=\\\'utf-8\\\')\\n                os.replace(temporary, self.path)\\n            finally:\\n                temporary.unlink(missing_ok=True)\\n            if event:\\n                with self.path.with_suffix(\\\'.events.jsonl\\\').open(\\\'a\\\', encoding=\\\'utf-8\\\') as stream:\\n                    stream.write(json.dumps({\\\'observed_at\\\': time.time(), \\\'model_instance_id\\\': self.instance_id,\\n                                             **event}, ensure_ascii=False) + \\\'\\\\n\\\')\\n            self.last_write = now\\n        except OSError:\\n            # UI telemetry must never abort or retry an otherwise valid download.\\n            pass\\n\\n    def phase(self, category, phase, detail=\\\'\\\', *, total=None, cache_hit=None):\\n        if phase not in PHASES or category not in self.items:\\n            raise ValueError(\\\'Invalid progress phase/category\\\')\\n        with self.lock:\\n            item = self.items[category]\\n            if self.measurement_available is False and phase in {\\\'loading\\\', \\\'loaded\\\'}:\\n                detail += \\\'；当前下载库未提供兼容进度回调\\\'\\n            item.update(phase=phase, detail=detail[:240])\\n            self._phase_details[category] = (phase, detail[:240])\\n            if total is not None and category != \\\'model\\\':\\n                item.update(bytes_total=int(total), files_total=1)\\n            if cache_hit is not None:\\n                item[\\\'cache_hit\\\'] = bool(cache_hit)\\n            if phase in {\\\'verifying\\\', \\\'loaded\\\'} and total is not None and category != \\\'model\\\':\\n                item.update(bytes_completed=int(total), percent=100, files_completed=1)\\n            self._write(force=True, event={\\\'type\\\': \\\'phase\\\', \\\'category\\\': category, \\\'phase\\\': phase})\\n\\n    def begin_transfer(self, category, total=None, initial=0, identity=None):\\n        with self.lock:\\n            identity = (category, identity) if identity is not None else None\\n            transfer = self.transfer_identities.get(identity) if identity is not None else None\\n            if transfer is None:\\n                transfer = uuid.uuid4().hex\\n                self.transfers[transfer] = {\\\'category\\\': category, \\\'n\\\': max(0, initial),\\n                                           \\\'total\\\': total, \\\'finished\\\': False, \\\'active\\\': 0}\\n                if identity is not None:\\n                    self.transfer_identities[identity] = transfer\\n            entry = self.transfers[transfer]\\n            entry.update(n=max(entry[\\\'n\\\'], initial), finished=False, active=entry[\\\'active\\\'] + 1)\\n            self._update_item(category)\\n            self._write(force=True, event={\\\'type\\\': \\\'transfer.started\\\', \\\'category\\\': category,\\n                                          \\\'transfer_id\\\': transfer, \\\'total\\\': total, \\\'initial\\\': initial})\\n        return transfer\\n\\n    def update_transfer(self, transfer, completed):\\n        with self.lock:\\n            entry = self.transfers[transfer]\\n            # Callback retries cannot decrease or double-count a transfer\\\'s absolute progress.\\n            entry[\\\'n\\\'] = max(entry[\\\'n\\\'], completed)\\n            self._update_item(entry[\\\'category\\\'])\\n            self._write()\\n\\n    def finish_transfer(self, transfer, *, success):\\n        with self.lock:\\n            entry = self.transfers[transfer]\\n            entry[\\\'active\\\'] = max(0, entry[\\\'active\\\'] - 1)\\n            entry.update(finished=entry[\\\'active\\\'] == 0, success=success)\\n            self._update_item(entry[\\\'category\\\'])\\n            self._write(force=True, event={\\\'type\\\': \\\'transfer.finished\\\', \\\'category\\\': entry[\\\'category\\\'],\\n                                          \\\'transfer_id\\\': transfer, \\\'success\\\': success})\\n\\n    def _update_item(self, category):\\n        entries = [e for e in self.transfers.values() if e[\\\'category\\\'] == category]\\n        item = self.items[category]\\n        item[\\\'active_files\\\'] = sum(not e[\\\'finished\\\'] for e in entries)\\n        item[\\\'files_completed\\\'] = sum(e.get(\\\'success\\\') is True and e[\\\'finished\\\'] for e in entries)\\n        value = sum(e[\\\'n\\\'] for e in entries)\\n        item[\\\'bytes_completed\\\'] = max(item[\\\'bytes_completed\\\'], value)\\n        if category != \\\'model\\\' and item[\\\'bytes_total\\\']:\\n            item[\\\'bytes_completed\\\'] = min(item[\\\'bytes_total\\\'], item[\\\'bytes_completed\\\'])\\n            item[\\\'percent\\\'] = round(100 * item[\\\'bytes_completed\\\'] / item[\\\'bytes_total\\\'], 1)\\n        if item[\\\'active_files\\\']:\\n            item.update(phase=\\\'downloading\\\', detail=\\\'已观察文件量（含续传与重建）；整模型总量未知\\\' if category == \\\'model\\\'\\n                        else \\\'正在下载/重建已选 LoRA 文件\\\')\\n        elif item[\\\'phase\\\'] == \\\'downloading\\\':\\n            phase, detail = self._phase_details.get(category, (\\\'loading\\\', \\\'继续加载；下载完成不代表模型就绪\\\'))\\n            item.update(phase=phase, detail=detail)\\n\\n\\nclass _ProgressBar:\\n    """Protocol proxy used by HF HTTP and Xet reconstruction callbacks."""\\n    def __init__(self, recorder, transfer, *, initial=0, total=None, delegate=None):\\n        self.recorder, self.transfer, self.delegate = recorder, transfer, delegate\\n        self.n, self.total = initial, total\\n        self.disable = False\\n        self.lock = threading.RLock()\\n\\n    def update(self, amount=1):\\n        with self.lock:\\n            self.n = max(0, self.n + (amount or 0))\\n            try:\\n                self.recorder.update_transfer(self.transfer, self.n)\\n            except Exception:\\n                pass  # Observation cannot fail an otherwise valid payload transfer.\\n            if self.delegate is not None:\\n                return self.delegate.update(amount)\\n\\n    def __getattr__(self, name):\\n        if self.delegate is not None:\\n            return getattr(self.delegate, name)\\n        if name == \\\'format_dict\\\':\\n            return {\\\'n\\\': self.n, \\\'total\\\': self.total, \\\'rate\\\': None}\\n        if name in {\\\'refresh\\\', \\\'set_description\\\', \\\'set_description_str\\\', \\\'set_postfix_str\\\', \\\'close\\\'}:\\n            return lambda *a, **kw: None\\n        raise AttributeError(name)\\n\\n    def __enter__(self):\\n        return self\\n\\n    def __exit__(self, *args):\\n        return False\\n\\n\\ndef _wrap_transfer(original, recorder):\\n    try:\\n        signature = inspect.signature(original)\\n    except (ValueError, TypeError):\\n        return original\\n    if \\\'_tqdm_bar\\\' not in signature.parameters:\\n        return original\\n\\n    @functools.wraps(original)\\n    def observed(*args, **kwargs):\\n        bound = signature.bind_partial(*args, **kwargs)\\n        existing = bound.arguments.get(\\\'_tqdm_bar\\\')\\n        if isinstance(existing, _ProgressBar):\\n            return original(*args, **kwargs)  # HF\\\'s nested HTTP retry owns the same byte counter.\\n        total, initial = bound.arguments.get(\\\'expected_size\\\'), bound.arguments.get(\\\'resume_size\\\', 0)\\n        total = int(total) if _number(total) else None\\n        initial = int(initial) if _number(initial) else 0\\n        # Hash an owned temporary-file identity, never persist a signed URL or local path.\\n        path = bound.arguments.get(\\\'incomplete_path\\\') or bound.arguments.get(\\\'downloaded_file\\\')\\n        if path is None:\\n            path = getattr(bound.arguments.get(\\\'temp_file\\\'), \\\'name\\\', None)\\n        identity = hashlib.sha256(os.fsencode(path)).hexdigest() if isinstance(path, (str, bytes, os.PathLike)) else None\\n        transfer = recorder.begin_transfer(recorder.category, total, initial, identity=identity)\\n        bound.arguments[\\\'_tqdm_bar\\\'] = _ProgressBar(recorder, transfer, initial=initial, total=total, delegate=existing)\\n        try:\\n            result = original(*bound.args, **bound.kwargs)\\n        except BaseException:\\n            recorder.finish_transfer(transfer, success=False)\\n            raise\\n        recorder.finish_transfer(transfer, success=True)\\n        return result\\n    return observed\\n\\n\\n@contextmanager\\ndef observe_hub_downloads(recorder):\\n    """A model process owns these hooks; no thread pool or concurrency limit changes."""\\n    global _ACTIVE\\n    with _HOOK_LOCK:\\n        if _ACTIVE is recorder:\\n            yield recorder\\n            return\\n        previous, patched = _ACTIVE, []\\n        _ACTIVE = recorder\\n        try:\\n            try:\\n                module = importlib.import_module(\\\'huggingface_hub.file_download\\\')\\n                for name in (\\\'http_get\\\', \\\'xet_get\\\'):\\n                    original = getattr(module, name, None)\\n                    if callable(original):\\n                        wrapper = _wrap_transfer(original, recorder)\\n                        if wrapper is not original:\\n                            setattr(module, name, wrapper)\\n                            patched.append((module, name, original, wrapper))\\n            except (ImportError, AttributeError, TypeError, ValueError):\\n                pass\\n            recorder.measurement_available = bool(patched)\\n            yield recorder\\n        finally:\\n            for module, name, original, wrapper in reversed(patched):\\n                if getattr(module, name) is wrapper:\\n                    setattr(module, name, original)\\n            _ACTIVE = previous\\n\\n\\ndef current_progress():\\n    return _ACTIVE or _RECORDER\\n\\n\\n@contextmanager\\ndef adapter_progress(category, spec):\\n    recorder = current_progress()\\n    if recorder is None:\\n        yield None\\n        return\\n    with observe_hub_downloads(recorder):\\n        previous = recorder.category\\n        recorder.category = category\\n        try:\\n            recorder.phase(category, \\\'loading\\\', \\\'准备所选 LoRA\\\', total=spec.size)\\n            yield recorder\\n        except BaseException:\\n            recorder.phase(category, \\\'failed\\\', \\\'LoRA 下载、校验或加载失败；详见模型日志\\\')\\n            raise\\n        finally:\\n            recorder.category = previous\\n\\n\\ndef tracked_model_load(function):\\n    @functools.wraps(function)\\n    def tracked(self, *args, **kwargs):\\n        global _RECORDER\\n        path = os.environ.get(\\\'H3_DOWNLOAD_PROGRESS_PATH\\\')\\n        if not path:\\n            return function(self, *args, **kwargs)\\n        recorder = DownloadProgress(path, instance_id=os.environ.get(\\\'H3_MODEL_INSTANCE_ID\\\'),\\n                                    settings=getattr(self, \\\'adapter_settings\\\', None))\\n        _RECORDER = recorder\\n        with observe_hub_downloads(recorder):\\n            recorder.phase(\\\'model\\\', \\\'loading\\\', \\\'检查缓存并加载所需组件；整模型总量未知\\\')\\n            try:\\n                result = function(self, *args, **kwargs)\\n            except BaseException:\\n                recorder.phase(\\\'model\\\', \\\'failed\\\', \\\'下载或加载失败；详见模型日志\\\')\\n                raise\\n            no_transfers = not any(e[\\\'category\\\'] == \\\'model\\\' for e in recorder.transfers.values())\\n            recorder.phase(\\\'model\\\', \\\'loaded\\\', \\\'组件加载完成；等待容器就绪检查\\\' +\\n                           (\\\'。未观察到传输：缓存命中或当前库未提供回调\\\' if no_transfers else \\\'\\\'))\\n            return result\\n    return tracked\\n\\n\\ndef tracked_adapter_application(function):\\n    @functools.wraps(function)\\n    def tracked(pipe, settings, *args, **kwargs):\\n        try:\\n            return function(pipe, settings, *args, **kwargs)\\n        except BaseException:\\n            recorder = current_progress()\\n            if recorder:\\n                for category in (\\\'acceleration\\\', \\\'style\\\'):\\n                    if getattr(settings, category, \\\'none\\\') != \\\'none\\\':\\n                        recorder.phase(category, \\\'failed\\\', \\\'LoRA 下载、验证或组合加载失败；详见模型日志\\\')\\n            raise\\n    return tracked\\n\', \'generation_options.py\': \'"""Auditable per-job BF16 choices; no arbitrary kernels or adapter downloads.\\n\\nConfiguration permits queue admission before readiness. Only the model\\\'s GPU\\nlease may apply these choices; CPU preparation only validates immutable data.\\n"""\\nfrom __future__ import annotations\\n\\nfrom adapter_config import AdapterSettings, TURBO, load_adapter_settings, validate_steps\\nfrom runtime_options import RuntimeOptions\\n\\nATTENTION_BACKENDS = ("auto", "native", "_native_flash")\\nACCELERATION_LORAS = ("none", TURBO.key)\\n\\n\\ndef requested_options(parameters):\\n    """The same defaults apply at public API, queue, and model boundaries."""\\n    return {\\n        "attention_backend": parameters.get("attention_backend", "auto"),\\n        "acceleration_lora": parameters.get("acceleration_lora", "none"),\\n        "steps": parameters["steps"],\\n    }\\n\\n\\ndef validate_generation_options(requested, settings):\\n    if requested["attention_backend"] not in ATTENTION_BACKENDS:\\n        raise ValueError("attention_backend must be auto, native or _native_flash; quantized/external attention kernels are unsupported")\\n    selected = requested["acceleration_lora"]\\n    if selected not in ACCELERATION_LORAS:\\n        raise ValueError("acceleration_lora must be none or lightx2v-ref2va-4step; arbitrary adapters are unsupported")\\n    steps = requested["steps"]\\n    if isinstance(steps, bool) or not isinstance(steps, int) or not 2 <= steps <= 50:\\n        raise ValueError("steps must be an integer between 2 and 50")\\n    if selected != "none" and selected != settings.acceleration:\\n        raise ValueError("Requested acceleration LoRA is not configured for preloading; restart with H3_ACCELERATION_LORA=lightx2v-ref2va-4step")\\n    validate_steps(AdapterSettings(acceleration=selected), steps)\\n    return requested\\n\\n\\ndef effective_options(requested, settings, runtime_options):\\n    attention = requested["attention_backend"]\\n    if attention == "auto":\\n        attention = runtime_options.attention_backend\\n    # Explicitly restore PyTorch dispatch after a previous forced-Flash job.\\n    if attention == "auto":\\n        attention = "native"\\n    return {**requested, "attention_backend": attention, "attention_scope": "transformer_ref",\\n            "precision": "bf16", "denoiser_evaluations": requested["steps"] - 1,\\n            "style_lora": settings.style, "style_scale": settings.style_scale}\\n\\n\\ndef generation_capabilities(*, model_ready=False, settings=None, runtime_options=None):\\n    settings = settings or load_adapter_settings()\\n    runtime_options = runtime_options or RuntimeOptions.from_env()\\n    default_attention = runtime_options.attention_backend\\n    attention = []\\n    for identity, label in (("auto", "自动 · PyTorch SDPA"), ("native", "PyTorch SDPA"),\\n                            ("_native_flash", "PyTorch Flash SDPA")):\\n        value = {"id": identity, "label": label, "supported": True, "configured": True,\\n                 "ready": bool(model_ready), "scope": "transformer_ref", "precision": "bf16"}\\n        if identity == "auto":\\n            value["resolves_to"] = "native" if default_attention == "auto" else default_attention\\n        if identity == "_native_flash":\\n            value["reason"] = "强制 PyTorch Flash SDPA；内核不兼容时明确失败，不静默回退"\\n        attention.append(value)\\n    configured = settings.acceleration == TURBO.key\\n    return {"schema_version": 1, "workflow": "ref2va", "precision": "bf16",\\n            "model_ready": bool(model_ready),\\n            "defaults": {"attention_backend": "auto", "acceleration_lora": "none", "steps": 50},\\n            "options": {"attention_backends": attention, "acceleration_loras": [\\n                {"id": "none", "label": "关闭", "supported": True, "configured": True,\\n                 "ready": bool(model_ready), "steps": {"min": 2, "max": 50, "default": 50}},\\n                {"id": TURBO.key, "label": "LightX2V Ref2VA Turbo · 4 次评估", "supported": True,\\n                 "configured": configured, "ready": bool(configured and model_ready),\\n                 "steps": {"min": 5, "max": 5, "required": 5, "denoiser_evaluations": 4},\\n                 "reason": "已配置预加载；模型就绪前可排队" if configured else "未在启动配置中预加载",\\n                 "repository": TURBO.repository, "revision": TURBO.revision, "sha256": TURBO.sha256,\\n                 "precision": "bf16", "validation": "GPU acceptance is reported separately; configuration is not a speed claim"},\\n            ]},\\n            "queue_semantics": "supported && configured permits admission before model_ready; readiness never means completion",\\n            "step_semantics": "steps counts sigma grid points; actual denoiser evaluations = steps - 1",\\n            "style_lora": {"id": settings.style, "scale": settings.style_scale, "control": "deployment"},\\n            "quantization": False}\\n\', \'lora_preflight.py\': \'"""Fresh-process LoRA dependency check before model downloads or GPU loading.\\n\\nPEFT 0.21.2 visits its optional torchao dispatcher even for ordinary BF16\\nLinear layers. An installed torchao < 0.16 raises before the normal dispatcher.\\nThe dedicated Colab runtime may explicitly remove that unused optional package;\\nthis helper never installs or upgrades packages, or changes the PyTorch stack.\\n"""\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom importlib import metadata\\nimport importlib.util\\nimport json\\nimport subprocess\\nimport sys\\n\\nfrom packaging.version import Version\\n\\n\\nSTACK_PACKAGES = ("torch", "torchvision", "torchaudio", "psutil")\\n\\n\\ndef distribution_version(name):\\n    try:\\n        return metadata.version(name)\\n    except metadata.PackageNotFoundError:\\n        return None\\n\\n\\ndef prepare_optional_torchao(*, repair=False):\\n    """Run before importing torch/PEFT; only the explicit old optional package changes."""\\n    before = {name: distribution_version(name) for name in STACK_PACKAGES}\\n    old_version = distribution_version("torchao")\\n    removed = None\\n    if old_version is not None and Version(old_version) < Version("0.16.0"):\\n        if not repair:\\n            raise RuntimeError(\\n                f"Unused torchao {old_version} blocks PEFT BF16 LoRA dispatch. "\\n                "In this dedicated runtime run lora_preflight.py --repair-optional-torchao; "\\n                "do not upgrade the existing PyTorch stack."\\n            )\\n        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],\\n                       check=True, capture_output=True, text=True)\\n        importlib.invalidate_caches()\\n        removed = old_version\\n    after = {name: distribution_version(name) for name in STACK_PACKAGES}\\n    if before != after:\\n        raise RuntimeError("LoRA optional dependency preparation changed the protected PyTorch/psutil stack")\\n    current = distribution_version("torchao")\\n    if removed is not None and (current is not None or importlib.util.find_spec("torchao") is not None):\\n        raise RuntimeError("torchao remains importable after uninstall; use a clean model process before loading LoRA")\\n    return {"optional_torchao": {"before": old_version, "after": current, "removed_version": removed},\\n            "stack_before": before, "stack_after": after, "stack_unchanged": True}\\n\\n\\ndef bf16_lora_smoke():\\n    """Exercise PEFT\\\'s actual dispatch and injection, which import-only checks miss.\\n\\n    Tiny CPU tensors avoid GPU initialization, large model downloads, and changes\\n    to the model container. Native BF16 GPU support is checked by normal preflight.\\n    """\\n    import torch\\n    import peft\\n    from peft import LoraConfig, get_peft_model\\n    from peft.import_utils import is_torchao_available\\n\\n    base = torch.nn.Sequential(torch.nn.Linear(4, 4, bias=False, dtype=torch.bfloat16))\\n    model = get_peft_model(base, LoraConfig(r=2, lora_alpha=2, target_modules=["0"]),\\n                           autocast_adapter_dtype=False)\\n    model.requires_grad_(False)\\n    model.eval()\\n    value = torch.ones((1, 4), dtype=torch.bfloat16)\\n    with torch.no_grad():\\n        enabled = model(value)\\n        with model.disable_adapter():\\n            disabled = model(value)\\n        restored = model(value)\\n    if any(parameter.dtype != torch.bfloat16 for parameter in model.parameters()):\\n        raise RuntimeError("Tiny LoRA injection changed BF16 parameter precision")\\n    if any(output.dtype != torch.bfloat16 or not torch.isfinite(output).all().item()\\n           for output in (enabled, disabled, restored)):\\n        raise RuntimeError("Tiny BF16 LoRA enable/disable/restore forward check failed")\\n    if not any("lora_A" in name for name, _ in model.named_parameters()):\\n        raise RuntimeError("Tiny LoRA test did not inject an adapter")\\n    return {"peft": peft.__version__, "torch": torch.__version__,\\n            "torchao_available": is_torchao_available(), "device": "cpu", "precision": "bf16",\\n            "injected": True, "enable_disable_restore_forward": True,\\n            "parameter_elements": sum(parameter.numel() for parameter in model.parameters()),\\n            "real_model_loaded": False, "gpu_generation_verified": False}\\n\\n\\ndef run(*, repair=False):\\n    report = prepare_optional_torchao(repair=repair)\\n    report["smoke"] = bf16_lora_smoke()\\n    report["status"] = "passed"\\n    return report\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--repair-optional-torchao", action="store_true")\\n    args = parser.parse_args()\\n    print(json.dumps(run(repair=args.repair_optional_torchao), ensure_ascii=False))\\n\', \'condition_guard.py\': \'"""Bounded, side-effect-free four-valued admission decisions.\\n\\nThe caller owns observation identity, freshness and the lock that makes a decision\\nactionable. This helper performs no I/O and acquires no locks. Its timestamp is\\nthe decision time, not a claim that an external observation was refreshed then.\\n"""\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport time\\nimport uuid\\n\\nSCHEMA_VERSION = "1.0"\\nGATE_VERSION = "1.0"\\nMAX_CONDITIONS = 64\\nMAX_WITNESSES = 64\\nMAX_CONTEXT_BYTES = 8192\\n\\n\\ndef observation_state(value):\\n    """Only native booleans assert facts; missing witnesses do not negate facts."""\\n    if type(value) is bool:\\n        return "true" if value else "false"\\n    if type(value) is str and value in {"unknown", "conflict"}:\\n        return value\\n    if type(value) is list:\\n        if len(value) > MAX_WITNESSES:\\n            return "unknown"\\n        # Lists are one level only: nested structures cannot introduce assertions.\\n        states = {observation_state(item) if type(item) is not list else "unknown"\\n                  for item in value}\\n        if "conflict" in states or {"true", "false"} <= states:\\n            return "conflict"\\n        if "true" in states:\\n            return "true"\\n        if "false" in states:\\n            return "false"\\n    return "unknown"\\n\\n\\ndef negate_observation(value):\\n    """Negate known booleans without turning unknown/conflict into a true fact."""\\n    state = observation_state(value)\\n    return {"true": False, "false": True, "unknown": None, "conflict": "conflict"}[state]\\n\\n\\ndef _context_copy(context):\\n    if type(context) is not dict:\\n        raise ValueError("Gate context must be a JSON object")\\n    def check(value, depth=0):\\n        if depth > 5:\\n            raise ValueError("Gate context is too deeply nested")\\n        if value is None or type(value) in {bool, int, str}:\\n            return\\n        if type(value) is float and math.isfinite(value):\\n            return\\n        if type(value) in {list, dict} and len(value) <= 64:\\n            if type(value) is dict:\\n                if any(type(key) is not str for key in value):\\n                    raise ValueError("Gate context keys must be strings")\\n                value = value.values()\\n            for item in value:\\n                check(item, depth + 1)\\n            return\\n        raise ValueError("Gate context must contain bounded finite JSON values")\\n    check(context)\\n    encoded = json.dumps(context, allow_nan=False, separators=(",", ":"), ensure_ascii=True)\\n    if len(encoded.encode("utf-8")) > MAX_CONTEXT_BYTES:\\n        raise ValueError("Gate context exceeds the receipt budget")\\n    return json.loads(encoded)\\n\\n\\ndef evaluate_gate(gate_id, required, observations, context):\\n    """Return a decision receipt; malformed declarations fail before admission.\\n\\nExplicit false facts differ from unknown and contradictory facts. In a witness\\nlist, a native boolean supplies evidence even if another witness is missing;\\nopposite boolean witnesses produce conflict. No string \\\'true\\\' or numeric 1 is a\\ntrue assertion. Inputs outside the declared required set cannot enable a gate.\\n"""\\n    if type(gate_id) is not str or not 1 <= len(gate_id) <= 128:\\n        raise ValueError("Gate identity must be a bounded nonempty string")\\n    if (type(required) not in {list, tuple} or not 1 <= len(required) <= MAX_CONDITIONS\\n            or any(type(name) is not str or not 1 <= len(name) <= 128 for name in required)\\n            or len(required) != len(set(required))):\\n        raise ValueError("Gate requires a bounded, nonempty set of unique condition names")\\n    if type(observations) is not dict:\\n        raise ValueError("Gate observations must be a mapping")\\n    context = _context_copy(context)\\n    conditions = {name: observation_state(observations.get(name)) for name in required}\\n    blocked = [name for name, state in conditions.items() if state != "true"]\\n    return {"schema_version": SCHEMA_VERSION, "gate_version": GATE_VERSION,\\n            "gate_id": gate_id, "decision_id": uuid.uuid4().hex,\\n            "observed_at": time.time(), "monotonic_ns": time.monotonic_ns(),\\n            "context": context, "conditions": conditions,\\n            "allowed": not blocked, "blocked": blocked}\\n\'}\nfor name, content in PACKAGE_FILES.items():\n    (PROJECT_DIR / name).write_text(content, encoding="utf-8")\ninstall_log = PROJECT_DIR / "install.log"\ninstall_card = StatusCard("检查预装依赖", badge="检查中", tone="running")\nstack_before = json_probe([sys.executable, "-c", PYTORCH_STACK_CHECK], install_log, card=install_card)\nconstraints = PROJECT_DIR / "colab-runtime-constraints.txt"\nprotected_versions = {**stack_before["pytorch_stack"], **stack_before["host_runtime"]}\nconstraints.write_text("\\n".join(name + "==" + version for name, version in protected_versions.items()) + "\\n", encoding="utf-8")\nif not shutil.which("ffprobe"):\n    run_logged(["apt-get", "update", "-qq"], install_log, title="准备系统依赖", card=install_card)\n    run_logged(["apt-get", "install", "-y", "-qq", "ffmpeg"], install_log, title="安装 FFmpeg", card=install_card)\nrun_logged([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-colab.txt"), "-c", str(constraints)],\n           install_log, title="安装 Python 依赖", card=install_card)\nfrom adapter_config import load_adapter_settings\nadapter_settings = load_adapter_settings()\nLORA_PREFLIGHT_REPORT = None\nif adapter_settings.acceleration != "none" or adapter_settings.style != "none":\n    run_logged([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-lora.txt"), "-c", str(constraints)],\n               install_log, title="安装 LoRA 依赖", card=install_card)\n    LORA_PREFLIGHT_REPORT = json_probe([sys.executable, str(PROJECT_DIR / "lora_preflight.py"), "--repair-optional-torchao"],\n                                      install_log, cwd=PROJECT_DIR, card=install_card)\n    if LORA_PREFLIGHT_REPORT.get("status") != "passed" or LORA_PREFLIGHT_REPORT.get("stack_unchanged") is not True:\n        raise RuntimeError("LoRA 实际 BF16 注入检查未通过；停止模型下载和加载。")\nstack_after = json_probe([sys.executable, "-c", PYTORCH_STACK_CHECK], install_log, card=install_card)\nif stack_after["pytorch_stack"] != stack_before["pytorch_stack"]:\n    install_card.update("预装 PyTorch 栈已改变", badge="停止启动", tone="error", note="版本比对不一致；请检查 install.log。")\n    raise RuntimeError("预装 PyTorch 栈发生变化，停止启动；检查安装日志。")\nif stack_after["host_runtime"] != stack_before["host_runtime"]:\n    install_card.update("宿主进程依赖已改变", badge="停止启动", tone="error", note="psutil 版本比对不一致；请检查 install.log。")\n    raise RuntimeError("宿主进程依赖 psutil 发生变化，停止启动；检查安装日志。")\nDEPENDENCY_REPORT = json_probe([sys.executable, "-c", DEPENDENCY_CHECK], install_log, cwd=PROJECT_DIR, card=install_card)\nDEPENDENCY_REPORT["lora_preflight"] = LORA_PREFLIGHT_REPORT\nINSTALL_OK = True\ninstall_card.update("依赖已就绪", badge="检查通过", tone="success", note="预装 PyTorch 栈与宿主 psutil 保持不变；新进程导入检查通过。",\n    fields=[("Python", DEPENDENCY_REPORT["python"]), ("PyTorch", DEPENDENCY_REPORT["torch"]),\n            ("Diffusers", DEPENDENCY_REPORT["diffusers"]), ("原生 BF16", "支持")],\n    details="完整安装日志：" + str(install_log) + "\\n" + json.dumps(DEPENDENCY_REPORT, ensure_ascii=False, indent=2),\n    details_label="展开版本与日志位置")\n', 'start': '\x23@title 启动总管理、API与模型容器\n\x23@markdown 总管理独立运行并持久化事件。API、模型与已启用公网并行准备；此格返回不代表生成已就绪。\n启动服务 = False \x23@param {type:"boolean"}\nSTART_GPU_API = 启动服务\n开放临时公网 = False \x23@param {type:"boolean"}\nOPEN_PUBLIC_API = 开放临时公网\nif START_GPU_API:\n    import json, socket\n    from startup import StartupSession\n    if not globals().get("INSTALL_OK"):\n        raise RuntimeError("先完成资源检查和安装依赖。")\n    previous_host = globals().get("startup_host")\n    if previous_host:\n        previous_state = previous_host.snapshot()\n        if (previous_state["server_alive"] or previous_state["tunnel_alive"] or\n                previous_state.get("model_alive") or previous_state.get("manager_alive") or\n                (previous_state["phase"] != "new" and\n                 (previous_state["phase"] not in {"failed", "stopped"} or\n                  not previous_state.get("cleanup_complete", False)))):\n            raise RuntimeError("旧生命周期仍在运行或尚未完成清理；就绪检查查询或停止与释放停止并确认子进程已退出，不要重复启动。")\n    startup_card = StatusCard("检查启动条件", badge="检查中", tone="running")\n    DEPENDENCY_REPORT = json_probe([sys.executable, "-c", DEPENDENCY_CHECK], PROJECT_DIR / "install.log", cwd=PROJECT_DIR, card=startup_card)\n    with socket.socket() as probe:\n        if probe.connect_ex(("127.0.0.1", 8000)) == 0:\n            raise RuntimeError("8000 端口已占用；先停止原服务。")\n    server_environment = os.environ.copy()\n    if globals().get("BOOTSTRAP_API_KEY"):\n        server_environment["H3_API_KEY"] = BOOTSTRAP_API_KEY\n    server_environment.setdefault("HF_XET_FIXED_DOWNLOAD_CONCURRENCY", "4")\n    ACTIVE_SERVER_OPTIONS = {name: server_environment.get(name, default) for name, default in {\n        "H3_ACCELERATION_LORA": "none", "H3_STYLE_LORA": "none", "H3_STYLE_LORA_SCALE": "1.0",\n        "H3_ATTENTION_BACKEND": "auto", "H3_DENOISER_OFFLOAD": "stream", "H3_MEDIA_WORKERS": "auto"}.items()}\n    PUBLIC_API_URL = None\n    API_KEY = None\n    startup_host = StartupSession(PROJECT_DIR, staged=RESIDENCY == "staged", public=OPEN_PUBLIC_API,\n                                  install_cloudflared=OPEN_PUBLIC_API, env=server_environment)\n    startup_host.start()\n    API_KEY = startup_host.api_key\n    server_log = startup_host.server_log\n    startup_card.update("准备分支已启动", badge="准备中", tone="running",\n        note="API与模型容器独立推进；刷新顶部看板观察当前能力。后台可在模型未就绪时查询与控制。",\n        fields=[("实例", startup_host.instance_id), ("驻留方案", RESIDENCY),\n                ("公网准备", "已并行启动" if OPEN_PUBLIC_API else "未启用")],\n        details="状态：" + str(startup_host.state_path)\n                + "\\n模型日志：" + str(startup_host.server_log) + "\\n隧道日志：" + str(startup_host.tunnel_log), details_label="诊断记录")\nelse:\n    StatusCard("本次未启动", badge="等待开启", tone="idle",\n               note="勾选启动服务；需要公网时同时勾选开放临时公网。")\n', 'generate': '\x23@title 生成验收与恢复查询\n\x23@markdown 单图512×288、8格点；每次新生成必须提供新的参考图和提示词。填写已有任务ID只查询下载。多任务验收使用pipeline_smoke_test.py。\n执行本次验收 = False \x23@param {type:"boolean"}\nRUN_SMOKE_TEST = 执行本次验收\n验收接口 = "临时公网" \x23@param ["临时公网", "本机API"]\nTEST_API_URL = 验收接口\nTEST_API_URL = {"临时公网": "PUBLIC_API_URL", "本机API": "http://127.0.0.1:8000"}[TEST_API_URL]\n已有任务ID = "" \x23@param {type:"string"}\nRESUME_JOB_ID = 已有任务ID\n新参考图路径 = "" \x23@param {type:"string"}\nTEST_REFERENCE_FILE = 新参考图路径\n新视频描述 = "" \x23@param {type:"string"}\nTEST_PROMPT = 新视频描述\nif RUN_SMOKE_TEST:\n    import json, re, subprocess, time, uuid, zipfile\n    from pathlib import Path\n    from IPython.display import display, Video\n    from google.colab import files\n    live_host = globals().get("startup_host")\n    live_state = live_host.snapshot() if live_host else None\n    if not RESUME_JOB_ID.strip() and (not live_state or live_state.get("observation_stale")\n                                      or live_state.get("api_ready") is not True):\n        raise RuntimeError("API 当前尚不可达；等待后台就绪后即可提交排队，无需等待模型加载。")\n    if TEST_API_URL == "PUBLIC_API_URL" and not RESUME_JOB_ID.strip():\n        if live_state.get("public_api_ready") is not True:\n            raise RuntimeError("公网 API 尚未通过实际探测；可继续等待或选择本机 API。")\n        PUBLIC_API_URL = live_state["public_url"]\n    test_base_url = globals().get("PUBLIC_API_URL") if TEST_API_URL == "PUBLIC_API_URL" else TEST_API_URL\n    if not test_base_url:\n        raise RuntimeError("尚无可达的公网地址；等待公网 API 就绪，或选择本机 API。")\n    if not globals().get("API_KEY"):\n        raise RuntimeError("缺少启动服务的API密钥；先启动本笔记本的模型服务。")\n    result_dir = PROJECT_DIR / ("smoke_" + time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])\n    smoke_environment = os.environ.copy()\n    smoke_environment["H3_API_KEY"] = API_KEY\n    result_dir.mkdir(parents=True, exist_ok=True)\n    smoke_log = result_dir / "smoke.log"\n    report_path = result_dir / "test_report.json"\n    resume_id = RESUME_JOB_ID.strip()\n    command = [sys.executable, "-u", str(PROJECT_DIR / "smoke_test.py"), "--url", test_base_url,\n               "--outputdir", str(result_dir), "--generation-timeout", "5400", "--poll-seconds", "30"]\n    if resume_id:\n        if not re.fullmatch(r"[a-f0-9]{32}", resume_id):\n            raise ValueError("已有任务ID必须是32位小写十六进制字符。")\n        command += ["--resume-job", resume_id]\n        action_title = "继续查询与下载"\n    else:\n        if not TEST_REFERENCE_FILE.strip() or not Path(TEST_REFERENCE_FILE).is_file() or not TEST_PROMPT.strip():\n            raise ValueError("新生成必须提供本次全新参考图的路径和提示词；不自动复用历史素材。")\n        active_options = globals().get("ACTIVE_SERVER_OPTIONS", {})\n        turbo = active_options.get("H3_ACCELERATION_LORA", "none") == "lightx2v-ref2va-4step"\n        command += ["--steps", "5" if turbo else "8"]\n        command += ["--reference", TEST_REFERENCE_FILE, "--prompt", TEST_PROMPT,\n                    "--content-history", str(PROJECT_DIR / "content-history.json")]\n        action_title = "真实 API 验收"\n    def current_job_status():\n        if not report_path.is_file():\n            return []\n        checkpoint = json.loads(report_path.read_text(encoding="utf-8"))\n        history = checkpoint.get("status_history") or []\n        latest = history[-1] if history else {}\n        phase = latest.get("active_phase")\n        if isinstance(phase, dict):\n            phase_name = phase.get("phase")\n            phase_seconds = phase.get("elapsed_seconds")\n            phase = str(phase_name) if phase_name is not None else None\n            if phase is not None and isinstance(phase_seconds, (int, float)):\n                phase += " · " + str(round(phase_seconds, 1)) + " 秒"\n        return [(label, value) for label, value in [\n            ("任务 ID", checkpoint.get("job_id")), ("真实状态", latest.get("status")),\n            ("当前阶段", phase)] if value is not None]\n    result_card = StatusCard(action_title, badge="执行中", tone="running",\n                             note="读取真实任务状态；无百分比估算。填写旧任务ID时不创建新生成任务。")\n    returncode = run_logged(command, smoke_log, cwd=PROJECT_DIR, env=smoke_environment,\n                           title=action_title, card=result_card, status_reader=current_job_status)\n    if not report_path.is_file():\n        raise RuntimeError("验收脚本未生成报告；展开底部日志查看错误。")\n    test_report = json.loads(report_path.read_text(encoding="utf-8"))\n    if returncode != 0 or test_report.get("passed") is not True:\n        raise RuntimeError("真实验收未通过。超时/中断后服务端任务可能仍在运行，请先查询任务再决定是否重试。")\n    video_path = Path(test_report["video_path"]).resolve()\n    if video_path.parent != result_dir.resolve() or not video_path.is_file():\n        raise RuntimeError("报告中的成片路径无效。")\n    video_info = test_report["video"]\n    result_card.update("视频已生成并通过解码检查", badge="验收通过", tone="success", note="真实链路验收通过；画质仍以预览为准。",\n        fields=[("分辨率", str(video_info["width"]) + " × " + str(video_info["height"])),\n                ("时长", str(round(video_info["duration_seconds"], 3)) + " 秒"),\n                ("帧数 / 帧率", str(video_info["decoded_frames"]) + " / " + str(video_info["fps"]) + " fps"),\n                ("本次验收耗时", elapsed_text(test_report["elapsed_seconds"]) if "elapsed_seconds" in test_report else "未记录")],\n        details=json.dumps({key: test_report[key] for key in ("job_id", "terminal_status", "video", "parameters", "test_scope", "metrics", "lifecycle") if key in test_report}, ensure_ascii=False, indent=2)\n                + "\\n完整报告：" + str(report_path) + "\\n日志：" + str(smoke_log),\n        details_label="展开参数、校验与报告位置")\n    # Media preview belongs to the dashboard, never to the command output.\n    bundle = result_dir.with_suffix(".zip")\n    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for artifact in (report_path, video_path):\n            if artifact.is_file():\n                archive.write(artifact, arcname=artifact.name)\n        if not resume_id:\n            archive.write(TEST_REFERENCE_FILE, arcname="reference" + Path(TEST_REFERENCE_FILE).suffix)\n    print("结果包已准备：报告、成片、参考图；不包含 API 密钥。")\n    files.download(str(bundle))\nelse:\n    StatusCard("本次未提交验收", badge="等待开启", tone="idle", note="勾选执行本次验收；通过后下载结果包，看板可预览成片。")\n', 'stop': '\x23@title 关闭选项\n\x23@markdown 停止服务保留实例；释放实例会丢失临时文件。先保存结果，再执行释放。\n停止服务 = False \x23@param {type:"boolean"}\nSTOP_SERVER = 停止服务\n释放云实例 = False \x23@param {type:"boolean"}\nRELEASE_COLAB_RUNTIME = 释放云实例\nif STOP_SERVER or RELEASE_COLAB_RUNTIME:\n    if globals().get("startup_host"):\n        stopped = startup_host.stop()\n        from notebook_ui import redact_text\n        STOP_RECEIPT = {key: stopped.get(key) for key in (\n            "phase", "error", "cleanup_complete", "manager_alive", "server_alive",\n            "model_alive", "tunnel_alive", "process_observation_errors", "cleanup_errors")}\n        stop_detail = redact_text(json.dumps(STOP_RECEIPT, ensure_ascii=False), secrets=[\n            getattr(startup_host, "api_key", ""), getattr(startup_host, "_control_key", "")])\n        print("停止回执：" + stop_detail)\n        process_names = {"manager_alive": "总管理", "server_alive": "API",\n                         "model_alive": "模型", "tunnel_alive": "公网隧道"}\n        live_processes = [label for key, label in process_names.items() if stopped.get(key) is True]\n        unknown_processes = [label for key, label in process_names.items()\n                             if stopped.get(key) is not True and stopped.get(key) is not False]\n        stop_issue = None\n        if live_processes:\n            stop_issue = "仍有进程存活：" + "、".join(live_processes)\n        elif unknown_processes or stopped.get("process_observation_errors"):\n            stop_issue = "进程退出状态尚未确认" + ("：" + "、".join(unknown_processes) if unknown_processes else "")\n        elif stopped.get("cleanup_complete") is not True or stopped.get("phase") not in {"stopped", "failed"}:\n            stop_issue = "进程已退出，但最终清理审计尚未确认"\n        if stop_issue:\n            raise RuntimeError(stop_issue + "；不声明关闭完成或释放实例。停止回执：" + stop_detail)\n    PUBLIC_API_URL = None\n    StatusCard("本笔记本的进程已停止", badge="已停止", tone="success", note="停止进程不等于释放云 GPU；实例仍可能计费。")\nif RELEASE_COLAB_RUNTIME:\n    from google.colab import runtime\n    StatusCard("已发起实例释放", badge="释放中", tone="running", note="等待 Colab 断开；临时文件将丢失。此消息不代表平台已确认释放。")\n    runtime.unassign()\nelif not STOP_SERVER:\n    StatusCard("本次未执行关闭操作", badge="等待选择", tone="idle", note="完成后勾选释放云实例。")\n', 'advanced': '\x23@title 性能与 LoRA\n\x23@markdown 两类LoRA独立选择，可组合；新选项尚未做GPU验收。默认none保持原有模型。\n加速LoRA = "none" \x23@param ["none", "lightx2v-ref2va-4step"]\nACCELERATION_LORA = 加速LoRA\n风格LoRA = "none" \x23@param ["none", "fal-realism-people"]\nSTYLE_LORA = 风格LoRA\n风格强度 = 1.0 \x23@param {type:"slider", min:0.1, max:1.0, step:0.1}\nSTYLE_LORA_SCALE = 风格强度\n\x23@markdown Turbo使用5个sigma格点=4次去噪评估，验收会自动匹配；它不保证当前硬件的实际速度。\n注意力后端 = "auto" \x23@param ["auto", "native", "_native_flash"]\nATTENTION_BACKEND = 注意力后端\n去噪卸载 = "stream" \x23@param ["stream", "sync"]\nDENOISER_OFFLOAD = 去噪卸载\n\x23@markdown auto+stream保持默认。_native_flash和sync属于实验；三阶段执行由资源预算调度。\nimport os\nos.environ["H3_ACCELERATION_LORA"] = ACCELERATION_LORA\nos.environ["H3_STYLE_LORA"] = STYLE_LORA\nos.environ["H3_STYLE_LORA_SCALE"] = str(STYLE_LORA_SCALE)\nos.environ["H3_ATTENTION_BACKEND"] = ATTENTION_BACKEND\nos.environ["H3_DENOISER_OFFLOAD"] = DENOISER_OFFLOAD\nos.environ["H3_MEDIA_WORKERS"] = "auto"\nif "StatusCard" in globals():\n    StatusCard("高级配置已记录", badge="尚未启动", tone="idle",\n               fields=[("加速 LoRA", ACCELERATION_LORA), ("风格 LoRA", STYLE_LORA),\n                       ("注意力", ATTENTION_BACKEND), ("去噪卸载", DENOISER_OFFLOAD)],\n               note="下一次启动时生效；选择LoRA后请执行安装依赖安装依赖。已运行服务不受本次修改影响。")\nelse:\n    print("高级配置已记录；未启动模型。先运行看板和资源检查，再继续安装。")\n', 'wait': '\x23@title 等待就绪或重新查询\n\x23@markdown 仅检查已启动分支。API可用、模型就绪、公网可达分别显示；模型失败时保留后台查询和控制。\nif not globals().get("startup_host"):\n    StatusCard("尚无运行实例", badge="未启动", tone="idle", note="先在启动服务启动本次生命周期。")\nelse:\n    join_card = StatusCard("检查汇合条件", badge="检查中", tone="running")\n    def show_readiness(state):\n        join_card.update("等待当前能力就绪", badge=state["phase"], tone="running",\n            note="候选URL或某一步完成，不等于公网API可用。每次读取当前状态。",\n            fields=[("已用时间", elapsed_text(state["elapsed_seconds"])),\n                    ("API后台", "可用" if state.get("api_ready") else "准备中"),\n                    ("模型容器", "就绪" if state["model_ready"] else "未就绪"),\n                    ("隧道连接", "已连接" if state["tunnel_connected"] else ("等待中" if state["public_requested"] else "未启用")),\n                    ("公网探测", "通过" if state.get("public_api_ready") else ("未通过" if state["public_requested"] else "不需要"))],\n            details="状态：" + str(startup_host.state_path) + "\\n模型日志：" + str(startup_host.server_log)\n                    + "\\n隧道日志：" + str(startup_host.tunnel_log), details_label="诊断记录")\n    try:\n        live_state = startup_host.wait_ready(on_update=show_readiness)\n    except BaseException as error:\n        join_card.update("生成能力尚未就绪", badge="可查询控制", tone="error", note=str(error)[:400],\n                         details="状态：" + str(startup_host.state_path), details_label="查看失败记录")\n        raise\n    PUBLIC_API_URL = live_state["public_url"] if live_state["public_ready"] else None\n    API_KEY = startup_host.api_key\n    join_card.update("公网 API 已就绪" if live_state["public_ready"] else "本地 API 已就绪",\n        badge="ready", tone="success", note="当前实例探测通过；生成验收另行验证真实视频生成。宿主持续监测就绪失效。",\n        fields=[("API 地址", PUBLIC_API_URL or "http://127.0.0.1:8000"),\n                ("实例", live_state["instance_id"]), ("驻留方案", RESIDENCY)],\n        details="状态：" + str(startup_host.state_path),\n        details_label="宿主状态位置")\n', 'control': '\x23@title 查询、事件与模型控制\n\x23@markdown 仅在需要单独控制组件时使用。命令接收不代表完成；原始状态和事件在日志区查看。\n组件操作 = "不发送命令" \x23@param ["不发送命令", "暂停接收", "恢复接收", "停止模型", "启动模型", "停止API", "启动API"]\nCONTROL_ACTION = 组件操作\n强制停止模型 = False \x23@param {type:"boolean"}\nFORCE_MODEL_STOP = 强制停止模型\n查询命令ID = "" \x23@param {type:"string"}\nCOMMAND_ID = 查询命令ID\nif not globals().get("startup_host"):\n    StatusCard("尚无运行实例", badge="未发送", tone="idle", note="先启动服务。")\nelse:\n    action_names = {"暂停接收": "drain", "恢复接收": "resume", "停止模型": "stop_model",\n                    "启动模型": "start_model", "停止API": "stop_api", "启动API": "start_api"}\n    if CONTROL_ACTION in action_names:\n        control_result = startup_host.control(action_names[CONTROL_ACTION], force=FORCE_MODEL_STOP)\n    elif COMMAND_ID.strip():\n        control_result = startup_host.command(COMMAND_ID.strip())\n    else:\n        control_result = {"status": "未发送", "command_id": "—"}\n    control_card = StatusCard("组件控制", badge=control_result.get("status", "已接收"), tone="idle",\n        note="填写命令 ID 并选择不发送命令，可查询执行结果。",\n        fields=[("命令 ID", control_result.get("command_id", "—"))],\n        details=json.dumps(control_result, ensure_ascii=False, indent=2), details_label="展开真实状态或命令回执")\n', 'export': '\x23@title 导出 API 连接信息\n\x23@markdown 需要外部客户端时再导出。文件包含密钥，请单独保存；密钥不显示在看板或日志里。\n导出私密连接文件 = False \x23@param {type:"boolean"}\nEXPORT_CONNECTION = 导出私密连接文件\nif EXPORT_CONNECTION:\n    from google.colab import files\n    host = globals().get("startup_host")\n    state = host.snapshot() if host else {}\n    if not state.get("api_ready"):\n        raise RuntimeError("API 尚不可用；刷新看板查看当前状态。")\n    base_url = state.get("public_url") if state.get("public_api_ready") else "http://127.0.0.1:8000"\n    connection_path = PROJECT_DIR / "api-connection.private.json"\n    connection_path.write_text(json.dumps({"url": base_url, "api_key": host.api_key,\n                                          "instance_id": host.instance_id}), encoding="utf-8")\n    connection_path.chmod(0o600)\n    files.download(str(connection_path))\n    StatusCard("连接文件已准备", badge="已导出", tone="success", note="文件包含密钥，请单独保存。")\n', 'logs': '\x23@title 查看日志\n\x23@markdown 只读取选择的来源；不自动滚动、不发送控制命令。显示有限尾部并隐藏密钥。\n日志来源 = "最近诊断" \x23@param ["最近诊断", "资源检查", "安装依赖", "API后台", "模型容器", "全局管理", "公网隧道", "调度事件", "完整状态", "最近验收"]\nLOG_SOURCE = 日志来源\n显示行数 = 100 \x23@param {type:"slider", min:20, max:200, step:20}\nLOG_LINES = 显示行数\nfrom IPython.display import HTML, display\nfrom notebook_ui import diagnostics_html, last_diagnostics\nhost = globals().get("startup_host")\nlog_files = {"资源检查": PROJECT_DIR / "preflight.log", "安装依赖": PROJECT_DIR / "install.log"}\nif host:\n    log_files.update({"API后台": host.api_log, "模型容器": host.server_log,\n                      "全局管理": host.manager_log, "公网隧道": host.tunnel_log})\nif globals().get("smoke_log"):\n    log_files["最近验收"] = smoke_log\nif LOG_SOURCE == "最近诊断":\n    log_text = last_diagnostics() or "暂无诊断记录。"\nelif LOG_SOURCE in {"调度事件", "完整状态"}:\n    value = (host.events(limit=50) if LOG_SOURCE == "调度事件" else host.snapshot()) if host else {"状态": "尚无运行实例"}\n    log_text = json.dumps(value, ensure_ascii=False, indent=2)\nelse:\n    selected_path = log_files.get(LOG_SOURCE)\n    log_text = log_tail(selected_path, limit=32768) if selected_path else ""\n    log_text = log_text or "该来源暂无日志。"\nlog_text = "\\n".join(log_text.splitlines()[-max(20, min(200, LOG_LINES)):])\ndisplay(HTML(diagnostics_html(LOG_SOURCE, log_text, secrets=[getattr(host, "api_key", ""), globals().get("API_KEY", "") or ""])))\n'}
H3_CONSOLE_BUNDLE_HASH = '3b187cf50163937821bf2d661559100c5fb09538de87a36342ad49a56cf7dc75'
H3_BOOTSTRAP_OPTIONS = {'public': True, 'acceleration': 'lightx2v-ref2va-4step'}
_h3_previous = globals().get("h3_console")
_h3_reuse = (_h3_previous is not None and not getattr(_h3_previous, "closed", False)
             and globals().get("H3_LOADED_BUNDLE_HASH") == H3_CONSOLE_BUNDLE_HASH)
if not _h3_reuse:
    # Never replace code or discard ownership while old work may still be active.
    if _h3_previous is not None and getattr(_h3_previous, "active", None) != {}:
        raise RuntimeError("旧控制台仍有操作或无法确认空闲；先在旧控制台停止并等待完成，再运行本格升级。")
    _h3_old_host = globals().get("startup_host")
    if _h3_old_host is None and _h3_previous is not None:
        _h3_old_host = getattr(_h3_previous, "scope", {}).get("startup_host")
    if _h3_old_host is not None:
        _h3_old_state = _h3_old_host.snapshot()
        _h3_no_processes = all(_h3_old_state.get(name) is False
                               for name in ("manager_alive", "server_alive", "model_alive", "tunnel_alive"))
        _h3_clean_terminal = (_h3_old_state.get("phase") in {"failed", "stopped"}
                              and _h3_old_state.get("cleanup_complete") is True)
        _h3_never_started = (_h3_old_state.get("phase") == "new"
                             and not _h3_old_state.get("observation_stale"))
        if (not _h3_no_processes or _h3_old_state.get("process_observation_errors")
                or not (_h3_clean_terminal or _h3_never_started)):
            raise RuntimeError("旧生命周期仍在运行或清理未确认；保留旧控制台和文件。先停止服务并确认资源释放，再运行本格升级。")
    if _h3_previous is not None and not getattr(_h3_previous, "closed", False):
        _h3_previous.close()
    globals().pop("h3_console", None)
    for name, content in PACKAGE_FILES.items():
        (PROJECT_DIR / name).write_text(content, encoding="utf-8")
    # Python's import cache must not silently retain the old controller signature.
    _h3_project_root = PROJECT_DIR.resolve()
    for _h3_name, _h3_module in list(sys.modules.items()):
        _h3_module_file = getattr(_h3_module, "__file__", None)
        if _h3_name != "__main__" and _h3_module_file:
            try:
                _h3_is_project_module = Path(_h3_module_file).resolve().is_relative_to(_h3_project_root)
            except (OSError, TypeError, ValueError):
                _h3_is_project_module = False
            if _h3_is_project_module:
                sys.modules.pop(_h3_name, None)
    importlib.invalidate_caches()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from notebook_console import mount_console
h3_console = mount_console(PROJECT_DIR, CONSOLE_PROGRAMS, globals(), auto_start=True,
                           bootstrap_payload=H3_BOOTSTRAP_OPTIONS, foreground_bootstrap_timeout=180)
H3_LOADED_BUNDLE_HASH = H3_CONSOLE_BUNDLE_HASH


控制台尚未挂载。首次运行本单元即自动开始检查、安装、API 和模型准备。